# Modified Determinat

In [ ]:
# @title
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
from io import StringIO
import hashlib

# 1. Load data from GitHub
def load_data_from_github():
    base_url = "https://raw.githubusercontent.com/kboroz/MSFE_Capstone_Project/main/01_Data/Streamlined/"
    files = [
        "streamlined_asx_50.csv",
        "streamlined_euro_stoxx_50.csv",
        "streamlined_ftse_100.csv",
        "streamlined_ibovespa.csv",
        "streamlined_jse_top_40.csv",
        "streamlined_macro.csv",
        "streamlined_nikkei_225.csv",
        "streamlined_s&p_500.csv"
    ]
    all_data = {}
    for file in files:
        url = base_url + file
        try:
            response = requests.get(url)
            response.raise_for_status()
            data = pd.read_csv(StringIO(response.text), index_col=0, parse_dates=True)
            data.columns = data.columns.str.replace('[^a-zA-Z0-9]', '_', regex=True)
            all_data[file.replace("streamlined_", "").replace(".csv", "").upper()] = data
        except Exception as e:
            print(f"Error loading {file}: {str(e)}")
            continue
    if not all_data:
        raise ValueError("No data files could be loaded.")
    return all_data

# 2. Configuration
DET_WINDOWS = [3, 7, 28]
WINDOW_COLORS = {
    3: 'green',
    7: 'blue',
    28: 'red'
}

# Index colors for consistent visualization
INDEX_COLORS = {
    'SP500': '#1f77b4',  # Blue
    'BOVESPA': '#ff7f0e',  # Orange
    'EUROSTOXX50': '#2ca02c',  # Green
    'FTSE100': '#d62728',  # Red
    'JSE40': '#9467bd',  # Purple
    'NIKKEI225': '#8c564b',  # Brown
    'ASX50': '#e377c2',  # Pink
    'Yields': '#7f7f7f',  # Gray
    'FX': '#bcbd22',  # Olive
    'Risk Factors': '#17becf'  # Cyan
}

# 3. Core Functions
def calculate_modified_determinant(corr_matrix):
    """Calculates det(R - I) to measure systemic coupling."""
    if corr_matrix.empty or corr_matrix.isnull().values.any():
        return np.nan
    clean_matrix = corr_matrix.dropna(axis=0, how='all').dropna(axis=1, how='all')
    if clean_matrix.shape[0] < 2:
        return np.nan
    mat = clean_matrix.to_numpy()
    identity = np.eye(mat.shape[0])
    try:
        return np.linalg.det(mat - identity)
    except np.linalg.LinAlgError:
        return np.nan

# 4. Main Analysis
def run_systemic_risk_analysis():
    # Load data
    all_data = load_data_from_github()
    all_assets = pd.concat(all_data.values(), axis=1)
    all_assets = all_assets.loc[:, ~all_assets.columns.duplicated()]

    # Calculate returns
    simple_returns = all_assets.pct_change().dropna()

    # Group assets
    asset_groups = {
        'Equity Indices': ['SP500', 'BOVESPA', 'EUROSTOXX50', 'FTSE100', 'JSE40', 'NIKKEI225', 'ASX50'],
        'Yields': [c for c in all_assets.columns if '10Y' in c],
        'FX': [c for c in all_assets.columns if any(x in c for x in ['USD', 'EUR', 'GBP'])],
        'Risk Factors': ['Gold', 'Oil', 'VIX', 'Bitcoin']
    }

    # Analyze each group
    for group_name, cols in asset_groups.items():
        valid_cols = [c for c in cols if c in simple_returns.columns]
        if len(valid_cols) < 2: continue

        plt.figure(figsize=(16, 7))
        group_returns = simple_returns[valid_cols]

        # Get group color
        color_key = 'SP500' if group_name == 'Equity Indices' else group_name
        group_color = INDEX_COLORS.get(color_key, '#ffffff')

        # Calculate and plot determinants
        for window in DET_WINDOWS:
            det_history = []
            for start in range(0, len(group_returns) - window, window):
                window_slice = group_returns.iloc[start:start+window]
                if window_slice.isnull().values.any():
                    continue
                corr_mat = window_slice.corr(method='spearman')
                det_val = calculate_modified_determinant(corr_mat)
                if not np.isnan(det_val):
                    det_history.append({'Date': window_slice.index[-1], 'Det': det_val})

            if det_history:
                df_det = pd.DataFrame(det_history).set_index('Date')
                plt.plot(df_det.index, df_det['Det'],
                        label=f'{window}-Day Window',
                        color=WINDOW_COLORS[window],
                        alpha=0.8,
                        linewidth=2.5 if window == 28 else 1.8)

        # Format plot
        plt.yscale('symlog', linthresh=1e-10)
        plt.axhline(0, color='white', linestyle='--', alpha=0.3)
        plt.title(f'Systemic Risk: Modified Determinant $|R - I|$ — {group_name}', loc='left', color=group_color)
        plt.ylabel('Determinant (Log Scale)')
        plt.legend(loc='center left', bbox_to_anchor=(1, 0.5), fontsize=9)
        plt.grid(True, alpha=0.1)
        plt.tight_layout()
        plt.show()

# Run the analysis
run_systemic_risk_analysis()


In [ ]:
# @title
all_data = load_data_from_github()

for name, df in all_data.items():
    print(f"{name}: {df.shape[1]} columns → {df.columns.tolist()[:4]}")

In [ ]:
# @title
for name, df in all_data.items():
    if name != 'MACRO':
        print(f"\n{name}")
        print(f"  Shape: {df.shape}")
        print(f"  Dtypes sample: {df.dtypes.value_counts().to_dict()}")
        print(f"  NaN %: {(df.isnull().sum().sum() / df.size * 100):.1f}%")
        print(f"  First 3 cols: {df.columns[:3].tolist()}")


In [ ]:
# @title
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ============================================================
# CONFIGURATION
# ============================================================

DET_WINDOWS = [3, 7, 28]

WINDOW_COLORS = {
    3:  'green',
    7:  'blue',
    28: 'red'
}

INDEX_FILES = ['ASX_50', 'EURO_STOXX_50', 'FTSE_100',
               'IBOVESPA', 'JSE_TOP_40', 'NIKKEI_225', 'S&P_500']

# ============================================================
# CORE FUNCTION
# ============================================================

def calculate_modified_determinant(corr_matrix: pd.DataFrame) -> float:
    """
    Computes det(R - I) on a correlation matrix R.
    - Large negative → high systemic coupling
    - Near zero     → diversified / low coupling
    - Positive      → mixed correlation structure
    """
    if corr_matrix.empty or corr_matrix.isnull().values.any():
        return np.nan
    mat = corr_matrix.to_numpy()
    identity = np.eye(mat.shape[0])
    try:
        return np.linalg.det(mat - identity)
    except np.linalg.LinAlgError:
        return np.nan


# ============================================================
# DETERMINANT CALCULATION PER INDEX
# ============================================================

def compute_determinants(all_data: dict,
                         windows: list = DET_WINDOWS) -> dict:
    """
    For each index, compute rolling det(R - I) over non-overlapping
    windows of 3, 7, 28 days using Spearman correlation across stocks.

    Returns:
        det_results: {
            index_name: {
                window: pd.Series (Date → determinant value)
            }
        }
    """
    det_results = {}

    for index_name in INDEX_FILES:
        if index_name not in all_data:
            print(f"⚠️  {index_name} not found in loaded data. Skipping.")
            continue

        price_df = all_data[index_name]

        # Log returns — more stable than pct_change for large matrices
        log_returns = np.log(price_df / price_df.shift(1)).dropna()

        det_results[index_name] = {}

        for window in windows:
            det_history = []

            # Non-overlapping windows
            for start in range(0, len(log_returns) - window, window):
                window_slice = log_returns.iloc[start : start + window]

                # Skip if any NaN survived
                if window_slice.isnull().values.any():
                    continue

                # Need at least 2 rows to compute correlation
                if window_slice.shape[0] < 2:
                    continue

                corr_mat = window_slice.corr(method='spearman')
                det_val  = calculate_modified_determinant(corr_mat)

                if not np.isnan(det_val):
                    det_history.append({
                        'Date': window_slice.index[-1],
                        'Det':  det_val
                    })

            if det_history:
                series = (pd.DataFrame(det_history)
                            .set_index('Date')['Det'])
                det_results[index_name][window] = series
                print(f"  ✅ {index_name} | {window}-day window "
                      f"→ {len(series)} observations")
            else:
                print(f"  ⚠️  {index_name} | {window}-day window → no valid observations")

    return det_results


# ============================================================
# RUN
# ============================================================

print("Computing modified determinants...\n")
det_results = compute_determinants(all_data)
print("\nDone.")


# Sign Detection

det(R-I) 	Interpretation 	Portfolio Action

R - I removes the diagonal (self-correlation)

det(R - I) captures purely the off-diagonal structure

Large negative 	High systemic coupling 	Reduce exposure

Near zero 	Low coupling / diversified 	Neutral

Positive 	Mixed correlation structure 	Increase exposure

# Signal Generation Function

In [ ]:
# @title
# ============================================================
# STEP 2 — SIGN SIGNAL EXTRACTION
# ============================================================

def extract_sign_signals(det_results: dict) -> dict:
    """
    For each index and each window, extract:
      - sign       : +1 (positive det) or -1 (negative det)
      - sign_change: transition events (+2, -2, 0)
      - regime     : RISK_ON / RISK_OFF label

    Convention:
      det > 0  → low systemic coupling → RISK_OFF (safe to invest)
      det < 0  → high systemic coupling → RISK_ON  (reduce exposure)

    Returns:
        signal_results: {
            index_name: {
                window: pd.DataFrame with columns
                        [Det, Sign, Sign_Change, Regime]
            }
        }
    """
    signal_results = {}

    for index_name, windows in det_results.items():
        signal_results[index_name] = {}

        for window, det_series in windows.items():

            df = det_series.to_frame(name='Det')

            # Sign: +1 or -1 (zero det is treated as RISK_ON conservatively)
            df['Sign'] = np.where(df['Det'] > 0, 1, -1)

            # Sign change: captures regime transitions
            # +2 = moved from -1 to +1 (RISK_ON → RISK_OFF)
            # -2 = moved from +1 to -1 (RISK_OFF → RISK_ON)
            #  0 = no change
            df['Sign_Change'] = df['Sign'].diff().fillna(0)

            # Regime label
            df['Regime'] = np.where(df['Sign'] == 1, 'RISK_OFF', 'RISK_ON')

            signal_results[index_name][window] = df

    return signal_results


# ============================================================
# RUN + QUICK DIAGNOSTIC
# ============================================================

signal_results = extract_sign_signals(det_results)

print("=" * 60)
print("SIGN SIGNAL SUMMARY — Last 5 observations per index/window")
print("=" * 60)

for index_name, windows in signal_results.items():
    print(f"\n{'─'*50}")
    print(f"  {index_name}")
    print(f"{'─'*50}")

    for window, df in windows.items():
        n_risk_on  = (df['Regime'] == 'RISK_ON').sum()
        n_risk_off = (df['Regime'] == 'RISK_OFF').sum()
        n_trans    = (df['Sign_Change'] != 0).sum()

        print(f"\n  {window}-day window | "
              f"RISK_ON: {n_risk_on} | "
              f"RISK_OFF: {n_risk_off} | "
              f"Transitions: {n_trans}")
        print(df[['Det', 'Sign', 'Regime']].tail(3).to_string())


In [ ]:
# @title
# ============================================================
# STEP 3A — VISUALIZATION
# ============================================================

def plot_determinant_signals(det_results: dict,
                              signal_results: dict) -> None:
    """
    For each index:
      - Top panel: raw determinant values (log scale) for all 3 windows
      - Bottom panel: regime signal (RISK_ON = -1, RISK_OFF = +1)
    """

    for index_name in INDEX_FILES:
        if index_name not in det_results:
            continue

        fig, axes = plt.subplots(3, 2, figsize=(16, 12))
        fig.suptitle(f'{index_name} — Determinant Signals',
                     fontsize=14, fontweight='bold')

        for row, window in enumerate(DET_WINDOWS):
            if window not in det_results[index_name]:
                continue

            det_series = det_results[index_name][window]
            sig_df     = signal_results[index_name][window]

            # ── Left panel: raw determinant ──────────────────
            ax_det = axes[row, 0]
            ax_det.plot(det_series.index,
                        det_series.values,
                        color=WINDOW_COLORS[window],
                        linewidth=0.8,
                        label=f'{window}-day det')
            ax_det.axhline(0, color='black', linewidth=0.6, linestyle='--')
            ax_det.set_title(f'{window}-day | Raw Determinant det(R-I)')
            ax_det.set_ylabel('det(R - I)')
            ax_det.legend(fontsize=8)
            ax_det.grid(True, alpha=0.3)

            # Shade negative regions (RISK_ON)
            ax_det.fill_between(det_series.index,
                                 det_series.values,
                                 0,
                                 where=(det_series.values < 0),
                                 alpha=0.15,
                                 color='red',
                                 label='RISK_ON zone')

            # ── Right panel: regime signal ────────────────────
            ax_sig = axes[row, 1]

            colors = sig_df['Regime'].map({'RISK_ON': 'red',
                                            'RISK_OFF': 'green'})

            ax_sig.bar(sig_df.index,
                       sig_df['Sign'],
                       color=colors,
                       width=window * 0.9,
                       label='Regime')

            ax_sig.axhline(0, color='black', linewidth=0.6)
            ax_sig.set_title(f'{window}-day | Regime Signal')
            ax_sig.set_ylabel('+1 RISK_OFF / -1 RISK_ON')
            ax_sig.set_ylim(-1.5, 1.5)
            ax_sig.grid(True, alpha=0.3)

            # Mark transitions
            transitions = sig_df[sig_df['Sign_Change'] != 0]
            ax_sig.scatter(transitions.index,
                           transitions['Sign'],
                           color='black',
                           zorder=5,
                           s=20,
                           label='Transition')
            ax_sig.legend(fontsize=8)

        plt.tight_layout()
        plt.savefig(f'{index_name}_determinant_signals.png',
                    dpi=150, bbox_inches='tight')
        plt.show()
        print(f"  ✅ Saved: {index_name}_determinant_signals.png")


# ============================================================
# RUN VISUALIZATION
# ============================================================

plot_determinant_signals(det_results, signal_results)


In [ ]:
# @title
# ============================================================
# STEP 3B — SIGNAL PACKAGING FOR PORTFOLIO OPTIMIZER
# ============================================================

def package_signals_for_optimizer(signal_results: dict,
                                   macro_dates: pd.Index) -> pd.DataFrame:
    """
    Packages all index signals into a single aligned DataFrame
    ready for the portfolio optimizer.

    For each index, we produce 3 columns:
        {INDEX}_{window}d_signal  →  +1 (RISK_OFF) or -1 (RISK_ON)

    All signals are forward-filled to match macro_dates frequency.

    Returns:
        signal_df: pd.DataFrame (macro_dates × n_signals)
    """
    signal_df = pd.DataFrame(index=macro_dates)

    for index_name, windows in signal_results.items():
        for window, df in windows.items():
            col_name = f"{index_name}_{window}d_signal"

            # Reindex to macro dates, forward fill regime signal
            aligned = (df['Sign']
                         .reindex(macro_dates)
                         .ffill()
                         .bfill())

            signal_df[col_name] = aligned

    return signal_df


# ============================================================
# RUN PACKAGING
# ============================================================

macro_dates   = all_data['MACRO'].index
packaged_signals = package_signals_for_optimizer(signal_results, macro_dates)

print("Packaged Signal DataFrame")
print(f"Shape : {packaged_signals.shape}")
print(f"NaNs  : {packaged_signals.isnull().sum().sum()}")
print(f"\nColumns:\n{packaged_signals.columns.tolist()}")
print(f"\nSample (last 5 rows):\n{packaged_signals.tail()}")


In [ ]:
# @title
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap

# ============================================================
# CONFIGURATION
# ============================================================

DET_WINDOWS   = [3, 7, 28]
WINDOW_WEIGHTS = {3: 0.20, 7: 0.35, 28: 0.45}   # longer window = more weight

INDEX_FILES = ['ASX_50', 'EURO_STOXX_50', 'FTSE_100',
               'IBOVESPA', 'JSE_TOP_40', 'NIKKEI_225', 'S&P_500']

INDEX_LABELS = {
    'ASX_50':        'ASX 50',
    'EURO_STOXX_50': 'Euro Stoxx 50',
    'FTSE_100':      'FTSE 100',
    'IBOVESPA':      'Ibovespa',
    'JSE_TOP_40':    'JSE Top 40',
    'NIKKEI_225':    'Nikkei 225',
    'S&P_500':       'S&P 500',
}

# ============================================================
# SIMULATE packaged_signals (replace with real data when running
# inside the full pipeline)
# ============================================================

np.random.seed(42)
dates = pd.date_range('2015-01-05', '2025-12-30', freq='B')

rows = {}
for idx in INDEX_FILES:
    for w in DET_WINDOWS:
        col = f"{idx}_{w}d_signal"
        # mostly +1 for short, mostly -1 for long (mimics real output)
        if w == 28:
            vals = np.where(np.random.rand(len(dates)) > 0.85, 1, -1)
        else:
            vals = np.where(np.random.rand(len(dates)) > 0.30, 1, -1)
        rows[col] = vals.astype(float)

packaged_signals = pd.DataFrame(rows, index=dates)

# ============================================================
# STEP 4A — COMPOSITE SCORE PER INDEX
# ============================================================

def compute_composite_score(packaged_signals: pd.DataFrame,
                             index_files: list,
                             window_weights: dict) -> pd.DataFrame:
    """
    Weighted average of the 3 window signals per index.
    Score ∈ [-1, +1]:
      +1  → all windows RISK_OFF (strong bullish)
      -1  → all windows RISK_ON  (strong bearish)
       0  → mixed / neutral

    Returns composite_df: (dates × n_indices)
    """
    total_weight = sum(window_weights.values())
    composite = {}

    for idx in index_files:
        score = pd.Series(0.0, index=packaged_signals.index)
        for w, wt in window_weights.items():
            col = f"{idx}_{w}d_signal"
            if col in packaged_signals.columns:
                score += (wt / total_weight) * packaged_signals[col]
        composite[idx] = score

    return pd.DataFrame(composite)


composite_df = compute_composite_score(packaged_signals, INDEX_FILES, WINDOW_WEIGHTS)

print("Composite Score DataFrame")
print(f"Shape : {composite_df.shape}")
print(f"NaNs  : {composite_df.isnull().sum().sum()}")
print(f"\nSample (last 5 rows):\n{composite_df.tail()}")
print(f"\nComposite Score Stats:\n{composite_df.describe().round(3)}")

# ============================================================
# STEP 4B — DISCRETE REGIME LABEL
# ============================================================

def label_regime(score: float) -> str:
    if score > 0.33:
        return 'RISK_OFF'
    elif score < -0.33:
        return 'RISK_ON'
    else:
        return 'NEUTRAL'


regime_df = composite_df.applymap(label_regime)

print(f"\nRegime Distribution per Index:")
for col in regime_df.columns:
    counts = regime_df[col].value_counts()
    print(f"  {INDEX_LABELS[col]:20s} → "
          f"RISK_OFF: {counts.get('RISK_OFF',0):4d} | "
          f"NEUTRAL: {counts.get('NEUTRAL',0):4d} | "
          f"RISK_ON: {counts.get('RISK_ON',0):4d}")

# ============================================================
# STEP 4C — GLOBAL RISK BAROMETER
# ============================================================

global_score = composite_df.mean(axis=1)

# ============================================================
# STEP 4D — VISUALIZATION
# ============================================================

REGIME_COLOR = {
    'RISK_OFF': '#2ecc71',
    'NEUTRAL':  '#f39c12',
    'RISK_ON':  '#e74c3c',
}

SCORE_CMAP = LinearSegmentedColormap.from_list(
    'risk', ['#e74c3c', '#f39c12', '#2ecc71'], N=256
)

fig = plt.figure(figsize=(20, 26))
fig.patch.set_facecolor('#0f0f1a')

gs = gridspec.GridSpec(
    len(INDEX_FILES) + 2, 1,
    figure=fig,
    hspace=0.55
)

# ── Per-index composite score ─────────────────────────────────
for i, idx in enumerate(INDEX_FILES):
    ax = fig.add_subplot(gs[i])
    ax.set_facecolor('#1a1a2e')

    series = composite_df[idx]

    # Colour each bar by regime
    bar_colors = series.apply(
        lambda s: '#2ecc71' if s > 0.33 else ('#e74c3c' if s < -0.33 else '#f39c12')
    )

    ax.bar(series.index, series.values,
           color=bar_colors, width=1.5, alpha=0.85)

    ax.axhline(0.33,  color='#2ecc71', linewidth=0.6, linestyle='--', alpha=0.6)
    ax.axhline(-0.33, color='#e74c3c', linewidth=0.6, linestyle='--', alpha=0.6)
    ax.axhline(0,     color='white',   linewidth=0.4, linestyle='-',  alpha=0.3)

    ax.set_ylim(-1.2, 1.2)
    ax.set_xlim(series.index[0], series.index[-1])
    ax.set_ylabel('Score', color='white', fontsize=7)
    ax.set_title(f'{INDEX_LABELS[idx]}  —  Composite Risk Score',
                 color='white', fontsize=9, fontweight='bold', pad=4)

    ax.tick_params(colors='#aaaaaa', labelsize=7)
    for spine in ax.spines.values():
        spine.set_edgecolor('#333355')

    # Regime % annotation
    n = len(series)
    pct_off  = (series > 0.33).sum()  / n * 100
    pct_on   = (series < -0.33).sum() / n * 100
    pct_neut = 100 - pct_off - pct_on
    ax.text(0.01, 0.92,
            f'RISK_OFF {pct_off:.0f}%  |  NEUTRAL {pct_neut:.0f}%  |  RISK_ON {pct_on:.0f}%',
            transform=ax.transAxes, color='#cccccc',
            fontsize=7, va='top', ha='left',
            bbox=dict(boxstyle='round,pad=0.2', facecolor='#0f0f1a', alpha=0.6))

# ── Global barometer ─────────────────────────────────────────
ax_g = fig.add_subplot(gs[len(INDEX_FILES)])
ax_g.set_facecolor('#1a1a2e')

global_colors = global_score.apply(
    lambda s: '#2ecc71' if s > 0.33 else ('#e74c3c' if s < -0.33 else '#f39c12')
)

ax_g.bar(global_score.index, global_score.values,
         color=global_colors, width=1.5, alpha=0.9)

ax_g.fill_between(global_score.index, global_score.values, 0,
                   where=(global_score > 0.33),
                   color='#2ecc71', alpha=0.15)
ax_g.fill_between(global_score.index, global_score.values, 0,
                   where=(global_score < -0.33),
                   color='#e74c3c', alpha=0.15)

ax_g.axhline(0.33,  color='#2ecc71', linewidth=0.8, linestyle='--', alpha=0.7)
ax_g.axhline(-0.33, color='#e74c3c', linewidth=0.8, linestyle='--', alpha=0.7)
ax_g.axhline(0,     color='white',   linewidth=0.5, alpha=0.4)

ax_g.set_ylim(-1.2, 1.2)
ax_g.set_xlim(global_score.index[0], global_score.index[-1])
ax_g.set_title('GLOBAL RISK BAROMETER  —  Mean Composite Score Across All Indices',
               color='white', fontsize=10, fontweight='bold', pad=5)
ax_g.set_ylabel('Global Score', color='white', fontsize=8)
ax_g.tick_params(colors='#aaaaaa', labelsize=7)
for spine in ax_g.spines.values():
    spine.set_edgecolor('#333355')

# ── Heatmap: regime matrix ────────────────────────────────────
ax_h = fig.add_subplot(gs[len(INDEX_FILES) + 1])
ax_h.set_facecolor('#1a1a2e')

# Resample to monthly for readability
monthly_score = composite_df.resample('ME').mean()
monthly_score.columns = [INDEX_LABELS[c] for c in monthly_score.columns]

im = ax_h.imshow(
    monthly_score.T.values,
    aspect='auto',
    cmap=SCORE_CMAP,
    vmin=-1, vmax=1,
    interpolation='nearest'
)

ax_h.set_title('REGIME HEATMAP  —  Monthly Average Composite Score',
               color='white', fontsize=10, fontweight='bold', pad=5)

ax_h.set_yticks(range(len(monthly_score.columns)))
ax_h.set_yticklabels(monthly_score.columns, color='white', fontsize=8)

# X-axis: show year labels
year_ticks = []
year_labels = []
for i, dt in enumerate(monthly_score.index):
    if dt.month == 1:
        year_ticks.append(i)
        year_labels.append(str(dt.year))

ax_h.set_xticks(year_ticks)
ax_h.set_xticklabels(year_labels, color='#aaaaaa', fontsize=8)
for spine in ax_h.spines.values():
    spine.set_edgecolor('#333355')

cbar = fig.colorbar(im, ax=ax_h, orientation='vertical', pad=0.01, shrink=0.9)
cbar.ax.tick_params(colors='white', labelsize=7)
cbar.set_label('Score', color='white', fontsize=8)
cbar.ax.yaxis.label.set_color('white')

# ── Legend ────────────────────────────────────────────────────
legend_elements = [
    plt.Rectangle((0,0), 1, 1, color='#2ecc71', label='RISK_OFF  (score > +0.33)'),
    plt.Rectangle((0,0), 1, 1, color='#f39c12', label='NEUTRAL   (-0.33 to +0.33)'),
    plt.Rectangle((0,0), 1, 1, color='#e74c3c', label='RISK_ON   (score < -0.33)'),
]
fig.legend(handles=legend_elements,
           loc='lower center',
           ncol=3,
           fontsize=9,
           facecolor='#1a1a2e',
           edgecolor='#333355',
           labelcolor='white',
           bbox_to_anchor=(0.5, 0.002))

fig.suptitle(
    'STEP 4 — Composite Risk Score & Regime Classification\n'
    'Window Weights: 3-day 20% | 7-day 35% | 28-day 45%',
    color='white', fontsize=13, fontweight='bold', y=0.995
)

plt.savefig('step4_composite_signal.png', dpi=150,
            bbox_inches='tight', facecolor=fig.get_facecolor())
plt.close()
print("Saved: step4_composite_signal.png")

# ============================================================
# STEP 4E — EXPORT PACKAGED OPTIMIZER INPUT
# ============================================================

optimizer_input = pd.concat([
    composite_df.add_suffix('_composite'),
    global_score.rename('GLOBAL_score')
], axis=1)

print(f"\nOptimizer Input Shape : {optimizer_input.shape}")
print(f"Columns               : {optimizer_input.columns.tolist()}")
print(f"\nLast 3 rows:\n{optimizer_input.tail(3).round(3)}")


In [ ]:
# First, inspect the structure
sample_index = INDEX_FILES[0]
sample_window = 7
sig_df = signal_results[sample_index][sample_window]
print(sig_df.dtypes)
print(sig_df.head())


In [ ]:
# @title
fig, axes = plt.subplots(3, 1, figsize=(20, 10))

for ax, target_window in zip(axes, [3, 7, 28]):
    window_scores = {}
    for index_name in INDEX_FILES:
        if index_name in signal_results and target_window in signal_results[index_name]:
            sig_df = signal_results[index_name][target_window]
            window_scores[index_name] = sig_df['Sign']

    if not window_scores:
        print(f"No data for window={target_window}")
        continue

    window_df    = pd.DataFrame(window_scores)
    heatmap_data = window_df.resample('ME').mean()

    sns.heatmap(
        heatmap_data.T,
        ax=ax,
        cmap=cmap_discrete,
        norm=norm,
        linewidths=0.8,
        linecolor='#111111',
        annot=False,
        cbar_kws={
            'label'  : 'Regime',
            'shrink' : 0.8,
            'ticks'  : [-0.67, 0, 0.67],
        }
    )

    cbar = ax.collections[0].colorbar
    cbar.set_ticklabels(['RISK_ON', 'NEUTRAL', 'RISK_OFF'])

    # ── Clean x-axis ─────────────────────────────────────────────
    n_cols        = heatmap_data.shape[0]
    tick_labels   = [d.strftime('%Y-%m') for d in heatmap_data.index]
    ax.set_xticks([i + 0.5 for i in range(n_cols)[::6]])
    ax.set_xticklabels(tick_labels[::6], rotation=45, ha='right', fontsize=8)

    ax.set_title(f'Window = {target_window}d', fontsize=11, fontweight='bold', pad=6)
    ax.set_xlabel('')
    ax.set_ylabel('Index', fontsize=9)

axes[2].set_xlabel('Month', fontsize=10)

fig.suptitle('Monthly Regime Heatmap — All Indices (3d / 7d / 28d)',
             fontsize=14, fontweight='bold', y=1.01)

plt.tight_layout()
plt.savefig('regime_heatmap_all_windows.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved: regime_heatmap_all_windows.png")


# Regime-Adjusted EW Portfolio

In [ ]:
# @title
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from io import StringIO

# ── 1. Load data from GitHub ─────────────────────────────────────────────────
def load_data_from_github():
    base_url = "https://raw.githubusercontent.com/kboroz/MSFE_Capstone_Project/main/01_Data/Streamlined/"
    files = [
        "streamlined_asx_50.csv",
        "streamlined_euro_stoxx_50.csv",
        "streamlined_ftse_100.csv",
        "streamlined_ibovespa.csv",
        "streamlined_jse_top_40.csv",
        "streamlined_macro.csv",
        "streamlined_nikkei_225.csv",
        "streamlined_s&p_500.csv"
    ]
    all_data = {}
    for file in files:
        url = base_url + file
        try:
            response = requests.get(url)
            response.raise_for_status()
            data = pd.read_csv(StringIO(response.text), index_col=0, parse_dates=True)
            data.columns = data.columns.str.replace('[^a-zA-Z0-9]', '_', regex=True)
            key = file.replace("streamlined_", "").replace(".csv", "").upper()
            all_data[key] = data
        except Exception as e:
            print(f"Error loading {file}: {str(e)}")
            continue
    if not all_data:
        raise ValueError("No data files could be loaded.")
    return all_data

# ── 2. Load and extract macro ────────────────────────────────────────────────
all_data = load_data_from_github()

macro_df = all_data['MACRO'].copy()
macro_df = macro_df.sort_index()
macro_df = macro_df.dropna(how='all')

print(f"Macro file shape : {macro_df.shape}")
print(f"Date range       : {macro_df.index[0].date()} → {macro_df.index[-1].date()}")
print(f"Assets           : {list(macro_df.columns)}")


In [ ]:
# @title
# ── Diagnose the problem ─────────────────────────────────────────────────────

# 1. Check for extreme daily returns
print("=== EXTREME DAILY RETURNS (>50% or <-50%) ===")
for col in returns_raw.columns:
    extremes = returns_raw[col][np.abs(returns_raw[col]) > 0.5]
    if len(extremes) > 0:
        print(f"{col:20s}: {len(extremes)} extreme days | min={extremes.min():.2%} max={extremes.max():.2%}")

# 2. Check where portfolio value goes NaN
print("\n=== CHECKING WHERE NaN FIRST APPEARS ===")
test_series = ew_backtest(returns_raw, ALL_ASSETS, 'ME', 0.005, 1_000_000_000)
first_nan = test_series[test_series.isna()].index
if len(first_nan) > 0:
    print(f"First NaN at: {first_nan[0].date()}")
    print(f"Returns on that day:\n{returns_raw.loc[first_nan[0]]}")
else:
    print("No NaN found")

# 3. Check what negative/zero weight sums look like
print("\n=== CHECKING WEIGHT SUM COLLAPSE ===")
rets_test = returns_raw[ALL_ASSETS].fillna(0)
weights = np.ones(len(ALL_ASSETS)) / len(ALL_ASSETS)
for date, row in rets_test.iterrows():
    new_w = weights * (1 + row.values)
    s = new_w.sum()
    if s <= 0 or np.isnan(s):
        print(f"Weight sum collapsed at {date.date()}: sum={s:.6f}")
        print(f"Worst assets: {row.nsmallest(5)}")
        break
    weights = new_w / s


In [ ]:
# @title
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from io import StringIO

# ── 1. Load data ──────────────────────────────────────────────────────────────
def load_macro():
    url = "https://raw.githubusercontent.com/kboroz/MSFE_Capstone_Project/main/01_Data/Streamlined/streamlined_macro.csv"
    response = requests.get(url)
    df = pd.read_csv(StringIO(response.text), index_col=0, parse_dates=True)
    df.columns = df.columns.str.replace('[^a-zA-Z0-9]', '_', regex=True)
    return df.sort_index().dropna(how='all')

macro_df = load_macro()

# ── 2. Asset classification ───────────────────────────────────────────────────
EQUITY_INDICES = ['ASX50', 'Ibovespa', 'FTSE100', 'SP500', 'JSE40', 'NIKKEI225', 'EUROSTOXX50']
FX             = ['AUDUSD', 'BRLUSD', 'EURUSD', 'GBPUSD', 'JPYUSD', 'ZARUSD']
COMMODITIES    = ['Oil', 'Gold', 'Bitcoin']
BONDS          = ['US_10Y', 'DE_10Y', 'UK_10Y', 'JP_10Y', 'AU_10Y', 'ZA_10Y']
# VIX excluded — not directly investable as a price series

# Problem assets:
# JP_10Y, DE_10Y → near-zero/negative yields → infinite pct_change → EXCLUDE
# Oil            → went negative April 2020 → pct_change explodes → HANDLE
# VIX            → not investable → EXCLUDE

SAFE_BONDS     = ['US_10Y', 'UK_10Y', 'AU_10Y', 'ZA_10Y']  # JP & DE excluded
COMMODITIES_NO_OIL = ['Gold', 'Bitcoin']

ALL_ASSETS     = EQUITY_INDICES + FX + ['Oil', 'Gold', 'Bitcoin'] + SAFE_BONDS
NO_BTC_ASSETS  = [a for a in ALL_ASSETS if a != 'Bitcoin']

print(f"All Assets  ({len(ALL_ASSETS)}): {ALL_ASSETS}")
print(f"No Bitcoin  ({len(NO_BTC_ASSETS)}): {NO_BTC_ASSETS}")

# ── 3. Build clean returns ────────────────────────────────────────────────────
# Step 1: raw pct_change
returns_raw = macro_df[ALL_ASSETS].pct_change(fill_method=None)

# Step 2: Invert bond yields (price proxy — rising yield = falling price)
for col in SAFE_BONDS:
    returns_raw[col] = -returns_raw[col]

# Step 3: Fix Oil — set return to -99% on the day it went negative (Apr 2020)
# Any day where Oil price in raw data <= 0 makes pct_change unreliable
oil_prices = macro_df['Oil']
bad_oil_days = oil_prices[oil_prices <= 0].index
if len(bad_oil_days) > 0:
    print(f"\nOil negative/zero price days: {bad_oil_days.tolist()}")
    # Set return on those days and surrounding days to 0
    for d in bad_oil_days:
        loc = returns_raw.index.get_loc(d)
        returns_raw.iloc[max(0,loc-1):loc+2, returns_raw.columns.get_loc('Oil')] = 0.0

# Step 4: Clip all returns to [-50%, +100%] — removes inf/-inf and extreme outliers
returns_raw = returns_raw.clip(lower=-0.50, upper=1.00)

# Step 5: Drop first row (NaN from pct_change)
returns_raw = returns_raw.iloc[1:]

# ── 4. Verify no inf/nan remains ──────────────────────────────────────────────
print(f"\nInf values  : {np.isinf(returns_raw).sum().sum()}")
print(f"NaN values  : {returns_raw.isna().sum().sum()}")
print(f"Returns shape: {returns_raw.shape}")
print(f"Date range   : {returns_raw.index[0].date()} → {returns_raw.index[-1].date()}")

# Check extremes after cleaning
print("\n=== Extremes after cleaning (>40% or <-40%) ===")
for col in returns_raw.columns:
    extremes = returns_raw[col][np.abs(returns_raw[col]) > 0.40]
    if len(extremes) > 0:
        print(f"  {col:15s}: {len(extremes)} days | min={extremes.min():.2%} max={extremes.max():.2%}")

# ── 5. Backtest function ──────────────────────────────────────────────────────
def ew_backtest(returns_df, assets, freq_code, tc_rate, initial_capital):
    """
    Equally-weighted backtest with transaction costs.

    Parameters
    ----------
    returns_df    : daily returns DataFrame (clean, no inf/nan)
    assets        : list of asset column names
    freq_code     : 'ME' = month-end, 'QE' = quarter-end
    tc_rate       : one-way transaction cost (e.g. 0.01 = 1%)
    initial_capital: starting capital in USD
    """
    rets = returns_df[assets].fillna(0.0).copy()
    n    = len(assets)

    # Rebalance dates
    rebal_dates = set(rets.resample(freq_code).last().index)

    # Initialise
    portfolio_value = initial_capital
    weights         = np.ones(n) / n          # equal weight at start
    values          = []

    for date, row in rets.iterrows():

        # ── Rebalance ────────────────────────────────────────────────────────
        if date in rebal_dates:
            target_weights = np.ones(n) / n
            turnover       = np.abs(weights - target_weights).sum() / 2
            tc_cost        = turnover * tc_rate
            portfolio_value *= (1 - tc_cost)
            weights         = target_weights.copy()

        # ── Daily return ─────────────────────────────────────────────────────
        daily_ret    = np.dot(weights, row.values)
        portfolio_value *= (1 + daily_ret)

        # ── Update weights (drift) ────────────────────────────────────────────
        new_weights = weights * (1 + row.values)
        weight_sum  = new_weights.sum()

        if weight_sum <= 0 or np.isnan(weight_sum) or np.isinf(weight_sum):
            # Safety: reset to equal weight (shouldn't happen after cleaning)
            weights = np.ones(n) / n
        else:
            weights = new_weights / weight_sum

        values.append(portfolio_value)

    return pd.Series(values, index=rets.index, name=f"EW_{freq_code}_TC{tc_rate}")

# ── 6. Run all combinations ───────────────────────────────────────────────────
UNIVERSES  = {'All_Assets': ALL_ASSETS, 'No_Bitcoin': NO_BTC_ASSETS}
FREQS      = {'Monthly': 'ME', 'Quarterly': 'QE'}
TC_LEVELS  = [0.005, 0.01, 0.02]
INITIAL    = 1_000_000_000

results = {}
for univ_name, assets in UNIVERSES.items():
    for freq_name, freq_code in FREQS.items():
        for tc in TC_LEVELS:
            label = f"{univ_name} | {freq_name} | TC={int(tc*100)}%"
            series = ew_backtest(returns_raw, assets, freq_code, tc, INITIAL)
            results[label] = series
            final_b = series.iloc[-1] / 1e9
            print(f"✓ {label:45s} → Final: ${final_b:.3f}B")

# ── 7. Performance metrics ────────────────────────────────────────────────────
def performance_metrics(series, initial=INITIAL, rf=0.03):
    daily_rets = series.pct_change(fill_method=None).dropna()
    n_years    = len(daily_rets) / 252

    total_ret  = (series.iloc[-1] / initial) - 1
    cagr       = (series.iloc[-1] / initial) ** (1 / n_years) - 1
    ann_vol    = daily_rets.std() * np.sqrt(252)
    sharpe     = (cagr - rf) / ann_vol if ann_vol > 0 else np.nan

    rolling_max = series.cummax()
    drawdown    = (series - rolling_max) / rolling_max
    max_dd      = drawdown.min()
    calmar      = cagr / abs(max_dd) if max_dd != 0 else np.nan

    var_95      = daily_rets.quantile(0.05)
    final_val   = series.iloc[-1]

    return {
        'Total Return (%)':      round(total_ret * 100, 2),
        'CAGR (%)':              round(cagr * 100, 2),
        'Ann. Volatility (%)':   round(ann_vol * 100, 2),
        'Sharpe Ratio':          round(sharpe, 3),
        'Max Drawdown (%)':      round(max_dd * 100, 2),
        'Calmar Ratio':          round(calmar, 3),
        'VaR 95% (daily %)':     round(var_95 * 100, 3),
        'Final Value ($B)':      round(final_val / 1e9, 3),
    }

metrics_df = pd.DataFrame({k: performance_metrics(v) for k, v in results.items()}).T
metrics_df.index.name = 'Strategy'

print("\n" + "="*100)
print("EQUALLY-WEIGHTED BENCHMARK — PERFORMANCE SUMMARY")
print("="*100)
print(metrics_df.to_string())

# ── 8. Portfolio Value Plot ───────────────────────────────────────────────────
colors_tc  = {0.005: 'green', 0.01: 'orange', 0.02: 'red'}
tc_labels  = {0.005: 'TC=0.5%', 0.01: 'TC=1%', 0.02: 'TC=2%'}

fig, axes = plt.subplots(2, 2, figsize=(18, 11))
fig.suptitle("Equally-Weighted Benchmark Portfolio\n$1,000,000,000 Initial Capital",
             fontsize=15, fontweight='bold')

for ax_idx, (univ_name, _) in enumerate(UNIVERSES.items()):
    for freq_idx, (freq_name, _) in enumerate(FREQS.items()):
        ax = axes[ax_idx][freq_idx]
        for tc in TC_LEVELS:
            label  = f"{univ_name} | {freq_name} | TC={int(tc*100)}%"
            series = results[label]
            ax.plot(series.index, series.values / 1e9,
                    label=tc_labels[tc], color=colors_tc[tc], linewidth=1.5)

        # Add $1B reference line
        ax.axhline(1.0, color='black', linestyle='--', linewidth=0.8, alpha=0.5, label='Start ($1B)')
        ax.set_title(f"{univ_name.replace('_',' ')} — {freq_name} Rebalancing",
                     fontsize=11, fontweight='bold')
        ax.set_ylabel("Portfolio Value ($B)")
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))
        ax.legend(fontsize=9)
        ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("ew_benchmark_fixed.png", dpi=150, bbox_inches='tight')
plt.show()
print("\nPlot saved as ew_benchmark_fixed.png")

# ── 9. Drawdown Plot ──────────────────────────────────────────────────────────
fig2, axes2 = plt.subplots(2, 2, figsize=(18, 10))
fig2.suptitle("Equally-Weighted Benchmark — Drawdown Analysis",
              fontsize=15, fontweight='bold')

for ax_idx, (univ_name, _) in enumerate(UNIVERSES.items()):
    for freq_idx, (freq_name, _) in enumerate(FREQS.items()):
        ax = axes2[ax_idx][freq_idx]
        for tc in TC_LEVELS:
            label  = f"{univ_name} | {freq_name} | TC={int(tc*100)}%"
            series = results[label]
            dd     = (series - series.cummax()) / series.cummax() * 100
            ax.fill_between(dd.index, dd.values, 0,
                            alpha=0.25, color=colors_tc[tc])
            ax.plot(dd.index, dd.values,
                    color=colors_tc[tc], linewidth=1.0, label=tc_labels[tc])
        ax.set_title(f"{univ_name.replace('_',' ')} — {freq_name} Rebalancing",
                     fontsize=11, fontweight='bold')
        ax.set_ylabel("Drawdown (%)")
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.0f}%"))
        ax.legend(fontsize=9)
        ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("ew_benchmark_drawdown_fixed.png", dpi=150, bbox_inches='tight')
plt.show()
print("Drawdown plot saved as ew_benchmark_drawdown_fixed.png")

# Run All Strategies Together

In [ ]:
# @title
# ── DIAGNOSTIC ────────────────────────────────────────────────────────────────
# Check 1: What assets are actually in macro_df vs signal files
print("macro_df columns:", macro_df.columns.tolist())
print("\nEquity assets in signal_files:", equity_assets)
print("Matched in macro_df:", [a for a in equity_assets if a in macro_df.columns])
print("NOT matched in macro_df:", [a for a in equity_assets if a not in macro_df.columns])

# Check 2: Signal values
print("\nSignal_df sample:")
print(signal_df.head(10))
print("\nSignal unique values per asset:")
print(signal_df.apply(lambda x: x.dropna().unique()))

# Check 3: Returns sanity
print("\nReturns describe (first 5 cols):")
print(returns.iloc[:, :5].describe())

# Check 4: Trace portfolio value for first 30 days
print("\n── Tracing first rebalance ──")
N = len(all_assets)
base_weight = 1.0 / N
print(f"N assets: {N}, base_weight: {base_weight:.4f}")

signal_df_copy = signal_df.copy()
signal_df_copy.index = pd.to_datetime(signal_df_copy.index)

first_reb = returns.index[0]
past_signals = signal_df_copy[signal_df_copy.index <= first_reb]
sig_today = past_signals.iloc[-1] if len(past_signals) > 0 else None

raw_weights = {}
for a in non_equity_assets:
    raw_weights[a] = base_weight
for a in equity_assets:
    if sig_today is not None and a in sig_today.index:
        sig = sig_today[a]
        raw_weights[a] = base_weight if sig >= 0 else -base_weight
    else:
        raw_weights[a] = base_weight

print(f"\nRaw weights before reinvestment:")
for k, v in raw_weights.items():
    print(f"  {k}: {v:.4f}")

short_proceeds = sum(abs(w) for w in raw_weights.values() if w < 0)
long_assets    = [a for a, w in raw_weights.items() if w > 0]
n_long         = len(long_assets)

print(f"\nShort proceeds: {short_proceeds:.4f}")
print(f"Num long assets: {n_long}")
print(f"Extra per long: {short_proceeds/n_long if n_long > 0 else 0:.4f}")

for a in long_assets:
    raw_weights[a] += short_proceeds / n_long if n_long > 0 else 0

total_weight = sum(raw_weights.values())
print(f"\nTotal weight after reinvestment: {total_weight:.4f}")
print(f"Sum of |weights|: {sum(abs(v) for v in raw_weights.values()):.4f}")

# Check 5: First 30 days portfolio value
pf = 1e9
cw = raw_weights.copy()
pf *= (1 - 0.005)  # initial TC
print(f"\nPortfolio value after initial TC: {pf:.0f}")

for i, date in enumerate(returns.index[:30]):
    dr = sum(cw.get(a, 0) * returns.loc[date, a] for a in all_assets)
    pf *= (1 + dr)
    if i < 5 or pf < 0:
        print(f"  Day {i}: date={date.date()}, daily_ret={dr:.4%}, pf={pf:,.0f}")
    if pf <= 0:
        print("  *** PORTFOLIO WENT NEGATIVE ***")
        break


In [ ]:
# @title
# ============================================================
# SIGNAL PORTFOLIO — FULL SELF-CONTAINED COLAB VERSION
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import requests
from io import StringIO

# ── 1. LOAD MACRO DATA ────────────────────────────────────────────────────────
GITHUB_BASE = "https://raw.githubusercontent.com/kboroz/MSFE_Capstone_Project/main/01_Data/Streamlined/"

def load_macro():
    url      = GITHUB_BASE + "streamlined_macro.csv"
    response = requests.get(url)
    df       = pd.read_csv(StringIO(response.text), index_col=0, parse_dates=True)
    df.columns = df.columns.str.replace('[^a-zA-Z0-9]', '_', regex=True)
    return df.sort_index()

macro_df = load_macro()
macro_df = macro_df[~macro_df.index.duplicated(keep='first')].sort_index()

print(f"macro_df shape : {macro_df.shape}")
print(f"Date range     : {macro_df.index[0].date()} → {macro_df.index[-1].date()}")
print(f"Columns        : {macro_df.columns.tolist()}")

# ── 2. DEFINE ASSET UNIVERSE ──────────────────────────────────────────────────
# Map from macro_df column names to portfolio names
# Adjust these to match your actual macro_df column names
EQUITY_INDICES = ['ASX50', 'Ibovespa', 'FTSE100', 'SP500',
                  'JSE40', 'NIKKEI225', 'EUROSTOXX50']
FX             = ['AUDUSD', 'BRLUSD', 'EURUSD', 'GBPUSD', 'JPYUSD', 'ZARUSD']
COMMODITIES    = ['Oil', 'Gold', 'Bitcoin']
SAFE_BONDS     = ['US_10Y', 'UK_10Y', 'AU_10Y', 'ZA_10Y']  # JP & DE excluded

ALL_ASSETS = EQUITY_INDICES + FX + COMMODITIES + SAFE_BONDS

# Keep only those actually in macro_df
ALL_ASSETS = [a for a in ALL_ASSETS if a in macro_df.columns]

print(f"\nAssets in universe ({len(ALL_ASSETS)}): {ALL_ASSETS}")
missing = [a for a in EQUITY_INDICES + FX + COMMODITIES + SAFE_BONDS
           if a not in macro_df.columns]
if missing:
    print(f"WARNING — Not found in macro_df: {missing}")

# ── 3. BUILD CLEAN RETURNS ────────────────────────────────────────────────────
returns_raw = macro_df[ALL_ASSETS].pct_change(fill_method=None)

# Invert bond yields (rising yield = falling price)
for col in SAFE_BONDS:
    if col in returns_raw.columns:
        returns_raw[col] = -returns_raw[col]

# Fix Oil negative price days (April 2020)
if 'Oil' in macro_df.columns:
    bad_oil = macro_df['Oil'][macro_df['Oil'] <= 0].index
    if len(bad_oil) > 0:
        print(f"\nOil negative price days found: {bad_oil.tolist()}")
        for d in bad_oil:
            loc = returns_raw.index.get_loc(d)
            returns_raw.iloc[
                max(0, loc - 1): loc + 2,
                returns_raw.columns.get_loc('Oil')
            ] = 0.0

# Clip extremes, fill NaN, drop first row
returns = returns_raw.clip(lower=-0.50, upper=1.00).fillna(0).iloc[1:]

print(f"\nReturns shape : {returns.shape}")
print(f"Date range    : {returns.index[0].date()} → {returns.index[-1].date()}")

# Quick sanity check — no inf/nan
assert not returns.isnull().any().any(), "NaNs remain in returns"
assert not np.isinf(returns.values).any(), "Inf values remain in returns"
print("✓ Returns clean — no NaN or Inf")

# ── 4. LOAD SIGNAL FILES ──────────────────────────────────────────────────────
# These are the equity indices for which you computed modified-determinant signals
# Signal files must contain a column with 'signal' in the name, values +1 or -1

SIGNAL_FILE_MAP = {
    'ASX50':   'ASX50_signals.csv',
    'FTSE100': 'FTSE100_signals.csv',
    'SP500':   'SP500_signals.csv',
}

signals = {}
for asset, fname in SIGNAL_FILE_MAP.items():
    if asset not in ALL_ASSETS:
        print(f"Skipping {asset} — not in universe")
        continue
    try:
        url = GITHUB_BASE + fname
        df  = pd.read_csv(url, index_col=0, parse_dates=True)
        df.index = pd.to_datetime(df.index)

        # Find signal column
        sig_col = next(
            (c for c in df.columns if 'signal' in c.lower()),
            df.columns[-1]
        )
        s = df[sig_col].rename(asset)

        # Standardise: any positive → +1, any negative → -1, zero/nan → +1 (default long)
        s = s.apply(lambda x: 1 if (pd.isna(x) or x >= 0) else -1)

        signals[asset] = s
        print(f"✓ {asset} | rows={len(s)} | "
              f"RISK_ON(+1)={( s==1).sum()} | RISK_OFF(-1)={(s==-1).sum()}")

    except Exception as e:
        print(f"✗ {asset}: {e}")

signal_df = pd.DataFrame(signals).sort_index()
signal_df.index = pd.to_datetime(signal_df.index)

print(f"\nsignal_df shape   : {signal_df.shape}")
print(f"signal_df columns : {signal_df.columns.tolist()}")
print(signal_df.tail())

# ── 5. ASSET GROUPS ───────────────────────────────────────────────────────────
# Equity assets that have signals
equity_assets     = [a for a in SIGNAL_FILE_MAP.keys()
                     if a in ALL_ASSETS and a in signal_df.columns]
# Everything else — always held long
non_equity_assets = [a for a in ALL_ASSETS if a not in equity_assets]

N           = len(ALL_ASSETS)
base_weight = 1.0 / N

print(f"\nEquity (signal-driven)  : {equity_assets}")
print(f"Non-equity (always long): {non_equity_assets}")
print(f"N={N}, base_weight={base_weight:.4f}, "
      f"equal-weight sum={N*base_weight:.4f}")

# ── 6. WEIGHT CONSTRUCTION LOGIC ─────────────────────────────────────────────
def build_weights(sig_row, equity_assets, non_equity_assets, base_weight, date):
    """
    Rules:
      - Non-equity: always +base_weight (long)
      - Equity with signal +1 (RISK_ON) : +base_weight (long)
      - Equity with signal -1 (RISK_OFF): -base_weight (short)
      - Short proceeds reinvested equally across all long positions
      - Final weights must sum to exactly 1.0
    """
    raw = {}

    # Non-equity always long
    for a in non_equity_assets:
        raw[a] = base_weight

    # Equity — signal driven
    for a in equity_assets:
        if sig_row is not None and a in sig_row.index and not pd.isna(sig_row[a]):
            raw[a] = base_weight if sig_row[a] >= 0 else -base_weight
        else:
            raw[a] = base_weight  # default long if no signal

    # Reinvest short proceeds equally into long positions
    short_proceeds = sum(abs(w) for w in raw.values() if w < 0)
    long_assets    = [a for a, w in raw.items() if w > 0]
    n_long         = len(long_assets)

    if short_proceeds > 0 and n_long > 0:
        extra = short_proceeds / n_long
        for a in long_assets:
            raw[a] += extra

    # Hard check
    total = sum(raw.values())
    if abs(total - 1.0) > 1e-6:
        raise ValueError(
            f"Weights sum to {total:.8f} on {date} — "
            f"short_proceeds={short_proceeds:.4f}, n_long={n_long}"
        )

    return raw

# ── 7. EQUAL-WEIGHT BENCHMARK ─────────────────────────────────────────────────
def run_equal_weight(returns, all_assets, tc_rate=0.005,
                     rebal_freq='monthly', initial_capital=1e9):

    freq_code = 'MS' if rebal_freq == 'monthly' else 'QS'
    reb_dates_raw = returns.resample(freq_code).first().index
    actual_dates  = returns.index

    reb_set = set()
    for d in reb_dates_raw:
        future = actual_dates[actual_dates >= d]
        if len(future) > 0:
            reb_set.add(future[0])

    N           = len(all_assets)
    base        = 1.0 / N
    pv          = float(initial_capital)
    cur_weights = {a: base for a in all_assets}
    records     = []

    for date in actual_dates:
        if date in reb_set:
            target   = {a: base for a in all_assets}
            turnover = sum(abs(target.get(a, 0) - cur_weights.get(a, 0))
                           for a in all_assets) / 2.0
            pv          *= (1 - turnover * tc_rate)
            cur_weights  = target.copy()

        dr  = sum(cur_weights.get(a, 0) * returns.loc[date, a]
                  for a in all_assets)
        pv *= (1 + dr)
        records.append({'date': date, 'portfolio_value': pv})

    return pd.DataFrame(records).set_index('date')

# ── 8. SIGNAL-DRIVEN PORTFOLIO ────────────────────────────────────────────────
def run_signal_portfolio(returns, signal_df, equity_assets, non_equity_assets,
                          all_assets, freq='monthly', tc_rate=0.005,
                          initial_capital=1e9):

    N           = len(all_assets)
    base_weight = 1.0 / N

    freq_code     = 'MS' if freq == 'monthly' else 'QS'
    reb_dates_raw = returns.resample(freq_code).first().index
    actual_dates  = returns.index

    reb_set = set()
    for d in reb_dates_raw:
        future = actual_dates[actual_dates >= d]
        if len(future) > 0:
            reb_set.add(future[0])

    sig = signal_df.copy()
    sig.index = pd.to_datetime(sig.index)

    pv              = float(initial_capital)
    current_weights = None
    records         = []

    for date in actual_dates:

        # ── Rebalance ──────────────────────────────────────────────────────
        if date in reb_set:
            past     = sig[sig.index <= date]
            sig_row  = past.iloc[-1] if len(past) > 0 else None

            raw = build_weights(
                sig_row, equity_assets, non_equity_assets, base_weight, date
            )

            # Transaction cost on turnover
            if current_weights is not None:
                all_keys = set(raw) | set(current_weights)
                turnover = sum(
                    abs(raw.get(a, 0) - current_weights.get(a, 0))
                    for a in all_keys
                ) / 2.0
                tc = turnover * tc_rate
            else:
                # First rebalance — assume full portfolio turnover
                tc = tc_rate

            pv             *= (1 - tc)
            current_weights = raw.copy()

        # ── Daily P&L ──────────────────────────────────────────────────────
        if current_weights is not None:
            dr  = sum(current_weights.get(a, 0) * returns.loc[date, a]
                      for a in all_assets)
            pv *= (1 + dr)

        records.append({'date': date, 'portfolio_value': pv})

    return pd.DataFrame(records).set_index('date')

# ── 9. RUN ALL PORTFOLIOS ─────────────────────────────────────────────────────
TC_LEVELS  = {'0%': 0.00, '0.5%': 0.005, '1%': 0.01, '2%': 0.02}
FREQS      = ['monthly', 'quarterly']

# Benchmarks
ew_results = {}
for tc_label, tc_rate in TC_LEVELS.items():
    for freq in FREQS:
        label = f"EW_{freq}_TC{tc_label}"
        ew_results[label] = run_equal_weight(
            returns, ALL_ASSETS, tc_rate=tc_rate,
            rebal_freq=freq, initial_capital=1e9
        )
        print(f"✓ {label}")

# Signal portfolios
signal_results = {}
for freq in FREQS:
    for tc_label, tc_rate in TC_LEVELS.items():
        label = f"Signal_{freq}_TC{tc_label}"
        signal_results[label] = run_signal_portfolio(
            returns, signal_df, equity_assets, non_equity_assets,
            ALL_ASSETS, freq=freq, tc_rate=tc_rate, initial_capital=1e9
        )
        print(f"✓ {label}")

print("\n✓ All portfolios computed")

# ── 10. PERFORMANCE METRICS ───────────────────────────────────────────────────
def compute_metrics(pv_series):
    pv        = pv_series['portfolio_value']
    daily_ret = pv.pct_change().dropna()
    n_years   = len(daily_ret) / 252
    total_ret = (pv.iloc[-1] / pv.iloc[0]) - 1
    cagr      = (1 + total_ret) ** (1 / n_years) - 1 if n_years > 0 else np.nan
    vol       = daily_ret.std() * np.sqrt(252)
    sharpe    = cagr / vol if vol > 0 else np.nan
    roll_max  = pv.cummax()
    max_dd    = ((pv - roll_max) / roll_max).min()
    calmar    = cagr / abs(max_dd) if max_dd != 0 else np.nan
    var_95    = daily_ret.quantile(0.05)

    return {
        'CAGR':          f"{cagr*100:.2f}%",
        'Volatility':    f"{vol*100:.2f}%",
        'Sharpe':        f"{sharpe:.2f}",
        'Max Drawdown':  f"{max_dd*100:.2f}%",
        'Calmar':        f"{calmar:.2f}",
        'VaR 95%':       f"{var_95*100:.2f}%",
        'Total Return':  f"{total_ret*100:.2f}%",
        'Final ($B)':    f"{pv.iloc[-1]/1e9:.3f}",
    }

print("\n" + "="*80)
print("EQUALLY-WEIGHTED BENCHMARK")
print("="*80)
ew_metrics  = {k: compute_metrics(v) for k, v in ew_results.items()}
print(pd.DataFrame(ew_metrics).T.to_string())

print("\n" + "="*80)
print("SIGNAL-DRIVEN PORTFOLIO")
print("="*80)
sig_metrics = {k: compute_metrics(v) for k, v in signal_results.items()}
print(pd.DataFrame(sig_metrics).T.to_string())

# ── 11. PLOT ──────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(18, 12))
fig.suptitle("Portfolio Performance: EW Benchmark vs Signal-Driven",
             fontsize=15, fontweight='bold')

colors = {'0%': 'black', '0.5%': 'steelblue', '1%': 'darkorange', '2%': 'crimson'}

for col_idx, freq in enumerate(FREQS):
    # EW
    ax = axes[0][col_idx]
    for tc_label in TC_LEVELS:
        label = f"EW_{freq}_TC{tc_label}"
        pv    = ew_results[label]['portfolio_value']
        ax.plot(pv.index, pv / 1e9, label=f"TC={tc_label}",
                color=colors[tc_label], lw=1.5)
    ax.axhline(1.0, color='grey', ls='--', lw=0.8)
    ax.set_title(f"EW Benchmark — {freq.capitalize()}", fontsize=12)
    ax.set_ylabel("Portfolio Value ($B)")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

    # Signal
    ax = axes[1][col_idx]
    for tc_label in TC_LEVELS:
        label = f"Signal_{freq}_TC{tc_label}"
        pv    = signal_results[label]['portfolio_value']
        ax.plot(pv.index, pv / 1e9, label=f"TC={tc_label}",
                color=colors[tc_label], lw=1.5)
    ax.axhline(1.0, color='grey', ls='--', lw=0.8)
    ax.set_title(f"Signal Portfolio — {freq.capitalize()}", fontsize=12)
    ax.set_ylabel("Portfolio Value ($B)")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("portfolio_comparison.png", dpi=150, bbox_inches='tight')
plt.show()
print("✓ Plot saved.")

# ── 12. SAVE ──────────────────────────────────────────────────────────────────
for label, df in {**ew_results, **signal_results}.items():
    df.to_csv(f"{label}_portfolio.csv")

all_metrics = pd.DataFrame({**ew_metrics, **sig_metrics}).T
all_metrics.to_csv("all_portfolio_metrics.csv")
print("✓ All CSVs saved.")


In [ ]:
# @title
# ============================================================
# SIGNAL PORTFOLIO — FULL SELF-CONTAINED VERSION
# Signals generated from Modified Determinant Model
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import requests
from io import StringIO

# ── 1. LOAD ALL DATA FROM GITHUB ─────────────────────────────────────────────
GITHUB_BASE = "https://raw.githubusercontent.com/kboroz/MSFE_Capstone_Project/main/01_Data/Streamlined/"

INDEX_FILE_MAP = {
    'ASX50':       'streamlined_asx_50.csv',
    'EUROSTOXX50': 'streamlined_euro_stoxx_50.csv',
    'FTSE100':     'streamlined_ftse_100.csv',
    'Ibovespa':    'streamlined_ibovespa.csv',
    'JSE40':       'streamlined_jse_top_40.csv',
    'NIKKEI225':   'streamlined_nikkei_225.csv',
    'SP500':       'streamlined_s&p_500.csv',
}

MACRO_FILE = 'streamlined_macro.csv'

def load_csv(url):
    r = requests.get(url)
    r.raise_for_status()
    df = pd.read_csv(StringIO(r.text), index_col=0, parse_dates=True)
    df.columns = df.columns.str.replace('[^a-zA-Z0-9]', '_', regex=True)
    return df.sort_index()

# Load macro
macro_df = load_csv(GITHUB_BASE + MACRO_FILE)
macro_df = macro_df[~macro_df.index.duplicated(keep='first')].sort_index()
print(f"macro_df shape : {macro_df.shape}")
print(f"Columns        : {macro_df.columns.tolist()}")

# Load individual index files (multi-column — OHLCV style)
index_data = {}
for name, fname in INDEX_FILE_MAP.items():
    try:
        df = load_csv(GITHUB_BASE + fname)
        index_data[name] = df
        print(f"✓ {name:15s} | shape={df.shape} | cols={df.columns.tolist()[:4]}")
    except Exception as e:
        print(f"✗ {name}: {e}")

# ── 2. EXTRACT PRICE SERIES FOR SIGNAL GENERATION ────────────────────────────
PRICE_COLS = ['Close', 'close', 'Price', 'price', 'Adj_Close', 'Last']

def get_price_series(df, name):
    col = next((c for c in PRICE_COLS if c in df.columns), df.columns[0])
    s   = pd.to_numeric(df[col], errors='coerce').dropna()
    print(f"  {name}: using column '{col}'")
    return s

index_prices = {}
for name, df in index_data.items():
    index_prices[name] = get_price_series(df, name)

# ── 3. MODIFIED DETERMINANT SIGNAL MODEL ─────────────────────────────────────
# Uses rolling correlation matrix of returns
# Signal = sign of det(R - I)
# RISK_ON  (+1) : det < 0  (correlations below identity — diversified)
# RISK_OFF (-1) : det >= 0 (correlations elevated — systemic risk)

DET_WINDOWS    = [3, 7, 28]
WINDOW_WEIGHTS = {3: 0.20, 7: 0.35, 28: 0.45}

def compute_modified_determinant(price_series_dict, windows=DET_WINDOWS):
    """
    For each window, compute rolling det(R - I) across all indices.
    Returns dict: {window: pd.DataFrame with columns per index and det column}
    """
    # Build returns dataframe from all indices
    prices = pd.DataFrame(price_series_dict).sort_index()
    rets   = prices.pct_change().dropna()

    det_results = {}

    for window in windows:
        det_series = []

        for i in range(window, len(rets)):
            window_rets = rets.iloc[i - window: i]

            # Drop columns with zero variance
            valid_cols = window_rets.columns[window_rets.std() > 1e-10]
            w = window_rets[valid_cols]

            if len(valid_cols) < 2:
                det_series.append(np.nan)
                continue

            try:
                R   = w.corr().values
                I   = np.eye(len(R))
                M   = R - I
                det = np.linalg.det(M)
            except Exception:
                det = np.nan

            det_series.append(det)

        idx = rets.index[window:]
        det_results[window] = pd.Series(det_series, index=idx, name=f'det_{window}d')
        print(f"  Window {window:2d}d | computed {len(det_series)} observations")

    return det_results

print("\n── Computing Modified Determinant ──")
det_results = compute_modified_determinant(index_prices)

# ── 4. GENERATE COMPOSITE SIGNAL PER INDEX ───────────────────────────────────
# For portfolio use we need per-index signals
# We compute each index's contribution by leaving it out (LOO) OR
# simply use the global composite score to drive all equity signals uniformly

def compute_composite_signal(det_results, window_weights=WINDOW_WEIGHTS):
    """
    Weighted average of sign across windows.
    Returns a Series of composite scores (between -1 and +1).
    Score > 0  → RISK_ON  → go long  (+1)
    Score <= 0 → RISK_OFF → go short (-1)
    """
    # Align all windows to common dates
    all_series = []
    for window, series in det_results.items():
        sign = np.sign(series).rename(f'sign_{window}d')
        all_series.append(sign)

    sign_df = pd.concat(all_series, axis=1).dropna()

    # Weighted composite
    composite = sum(
        sign_df[f'sign_{w}d'] * window_weights[w]
        for w in window_weights if f'sign_{w}d' in sign_df.columns
    )
    composite = composite / sum(window_weights.values())

    return composite, sign_df

composite_score, sign_df = compute_composite_signal(det_results)

print(f"\nComposite signal shape : {composite_score.shape}")
print(f"Date range             : {composite_score.index[0].date()} → "
      f"{composite_score.index[-1].date()}")
print(f"Score range            : {composite_score.min():.3f} → "
      f"{composite_score.max():.3f}")

# ── 5. BUILD SIGNAL DATAFRAME FOR PORTFOLIO ───────────────────────────────────
# Convert composite score to +1 / -1 per equity index
# We apply the SAME global signal to all equity indices
# (You can replace this with per-index LOO signals later)

EQUITY_ASSETS = list(index_prices.keys())  # indices we have signal data for

def build_signal_df(composite_score, equity_assets, macro_dates,
                    risk_on_threshold=0.0):
    """
    composite_score > threshold  → RISK_ON  → +1 (long)
    composite_score <= threshold → RISK_OFF → -1 (short)
    Reindexed to macro_dates with forward fill.
    """
    binary_signal = composite_score.apply(
        lambda x: 1 if x > risk_on_threshold else -1
    )

    signal_df = pd.DataFrame(index=macro_dates)
    for asset in equity_assets:
        aligned = binary_signal.reindex(macro_dates).ffill().bfill()
        signal_df[asset] = aligned

    return signal_df

signal_df = build_signal_df(composite_score, EQUITY_ASSETS, macro_df.index)

print(f"\nsignal_df shape : {signal_df.shape}")
print(f"Columns         : {signal_df.columns.tolist()}")
print(f"\nSignal distribution:")
for col in signal_df.columns:
    n_long  = (signal_df[col] == 1).sum()
    n_short = (signal_df[col] == -1).sum()
    print(f"  {col:15s} | LONG(+1): {n_long:4d} | SHORT(-1): {n_short:4d}")

# ── 6. DEFINE ASSET UNIVERSE ──────────────────────────────────────────────────
SAFE_BONDS = ['US_10Y', 'UK_10Y', 'AU_10Y', 'ZA_10Y']
FX         = ['AUDUSD', 'BRLUSD', 'EURUSD', 'GBPUSD', 'JPYUSD', 'ZARUSD']
COMMODITIES = ['Oil', 'Gold', 'Bitcoin']

ALL_ASSETS = EQUITY_ASSETS + FX + COMMODITIES + SAFE_BONDS
ALL_ASSETS = [a for a in ALL_ASSETS if a in macro_df.columns]

# Equity assets = those with signals AND in macro_df
equity_assets     = [a for a in EQUITY_ASSETS if a in ALL_ASSETS
                     and a in signal_df.columns]
non_equity_assets = [a for a in ALL_ASSETS if a not in equity_assets]

N           = len(ALL_ASSETS)
base_weight = 1.0 / N

print(f"\nTotal assets              : {N}")
print(f"Equity (signal-driven)    : {equity_assets}")
print(f"Non-equity (always long)  : {non_equity_assets}")
print(f"base_weight = 1/{N} = {base_weight:.4f}")

# ── 7. BUILD RETURNS ──────────────────────────────────────────────────────────
returns_raw = macro_df[ALL_ASSETS].pct_change(fill_method=None)

# Invert bond yields
for col in SAFE_BONDS:
    if col in returns_raw.columns:
        returns_raw[col] = -returns_raw[col]

# Fix Oil negative price days
if 'Oil' in macro_df.columns:
    bad_oil = macro_df['Oil'][macro_df['Oil'] <= 0].index
    for d in bad_oil:
        loc = returns_raw.index.get_loc(d)
        returns_raw.iloc[
            max(0, loc - 1): loc + 2,
            returns_raw.columns.get_loc('Oil')
        ] = 0.0

# Clip, fill, drop first row
returns = returns_raw.clip(lower=-0.50, upper=1.00).fillna(0).iloc[1:]

assert not returns.isnull().any().any(), "NaNs in returns"
assert not np.isinf(returns.values).any(), "Inf in returns"
print(f"\nReturns shape : {returns.shape}")
print("✓ Returns clean")

# ── 8. WEIGHT BUILDER ─────────────────────────────────────────────────────────
def build_weights(sig_row, equity_assets, non_equity_assets, base_weight, date):
    raw = {}

    for a in non_equity_assets:
        raw[a] = base_weight

    for a in equity_assets:
        if sig_row is not None and a in sig_row.index and not pd.isna(sig_row[a]):
            raw[a] = base_weight if sig_row[a] >= 0 else -base_weight
        else:
            raw[a] = base_weight

    # Reinvest short proceeds into long positions
    short_proceeds = sum(abs(w) for w in raw.values() if w < 0)
    long_assets    = [a for a, w in raw.items() if w > 0]
    n_long         = len(long_assets)

    if short_proceeds > 0 and n_long > 0:
        extra = short_proceeds / n_long
        for a in long_assets:
            raw[a] += extra

    total = sum(raw.values())
    if abs(total - 1.0) > 1e-6:
        raise ValueError(f"Weights sum to {total:.8f} on {date}")

    return raw

# ── 9. EQUAL-WEIGHT BENCHMARK ─────────────────────────────────────────────────
def build_weights(sig_row, equity_assets, non_equity_assets, base_weight, date):
    raw = {}

    for a in non_equity_assets:
        raw[a] = base_weight

    for a in equity_assets:
        if sig_row is not None and a in sig_row.index and not pd.isna(sig_row[a]):
            raw[a] = base_weight if sig_row[a] >= 0 else -base_weight
        else:
            raw[a] = base_weight

    # Reinvest short proceeds into long positions
    short_proceeds = sum(abs(w) for w in raw.values() if w < 0)
    long_assets    = [a for a, w in raw.items() if w > 0]
    n_long         = len(long_assets)

    if short_proceeds > 0 and n_long > 0:
        extra = short_proceeds / n_long
        for a in long_assets:
            raw[a] += extra
    elif short_proceeds > 0 and n_long == 0:
        # Edge case: everything is short — force all to zero weight (cash)
        for a in raw:
            raw[a] = 0.0
        # Redistribute equally
        all_keys = list(raw.keys())
        for a in all_keys:
            raw[a] = 1.0 / len(all_keys)

    # Final normalise — handles any floating point drift
    total = sum(raw.values())
    if total <= 0:
        # Full cash fallback
        for a in raw:
            raw[a] = 1.0 / len(raw)
    else:
        for a in raw:
            raw[a] = raw[a] / total

    return raw


# ── 10. SIGNAL-DRIVEN PORTFOLIO ───────────────────────────────────────────────
def run_signal_portfolio(returns, signal_df, equity_assets, non_equity_assets,
                          all_assets, freq='monthly', tc_rate=0.005,
                          initial_capital=1e9):

    N           = len(all_assets)
    base_weight = 1.0 / N

    freq_code     = 'MS' if freq == 'monthly' else 'QS'
    reb_dates_raw = returns.resample(freq_code).first().index
    actual_dates  = returns.index

    reb_set = set()
    for d in reb_dates_raw:
        future = actual_dates[actual_dates >= d]
        if len(future) > 0:
            reb_set.add(future[0])

    sig = signal_df.copy()
    sig.index = pd.to_datetime(sig.index)

    pv              = float(initial_capital)
    current_weights = None
    records         = []

    for date in actual_dates:
        if date in reb_set:
            past    = sig[sig.index <= date]
            sig_row = past.iloc[-1] if len(past) > 0 else None

            raw = build_weights(
                sig_row, equity_assets, non_equity_assets, base_weight, date
            )

            if current_weights is not None:
                all_keys = set(raw) | set(current_weights)
                turnover = sum(
                    abs(raw.get(a, 0) - current_weights.get(a, 0))
                    for a in all_keys
                ) / 2.0
                tc = turnover * tc_rate
            else:
                tc = tc_rate

            pv             *= (1 - tc)
            current_weights = raw.copy()

        if current_weights is not None:
            dr  = sum(current_weights.get(a, 0) * returns.loc[date, a]
                      for a in all_assets)
            pv *= (1 + dr)

        records.append({'date': date, 'portfolio_value': pv})

    return pd.DataFrame(records).set_index('date')

# ── 11. RUN ALL PORTFOLIOS ────────────────────────────────────────────────────
TC_LEVELS = {'0%': 0.00, '0.5%': 0.005, '1%': 0.01, '2%': 0.02}
FREQS     = ['monthly', 'quarterly']

ew_results = {}
for freq in FREQS:
    for tc_label, tc_rate in TC_LEVELS.items():
        label = f"EW_{freq}_TC{tc_label}"
        ew_results[label] = run_equal_weight(
            returns, ALL_ASSETS, tc_rate=tc_rate,
            rebal_freq=freq, initial_capital=1e9
        )
        print(f"✓ {label}")

signal_results = {}
for freq in FREQS:
    for tc_label, tc_rate in TC_LEVELS.items():
        label = f"Signal_{freq}_TC{tc_label}"
        signal_results[label] = run_signal_portfolio(
            returns, signal_df, equity_assets, non_equity_assets,
            ALL_ASSETS, freq=freq, tc_rate=tc_rate, initial_capital=1e9
        )
        print(f"✓ {label}")

print("\n✓ All portfolios computed")

# ── 12. PERFORMANCE METRICS ───────────────────────────────────────────────────
def compute_metrics(pv_series):
    pv        = pv_series['portfolio_value']
    daily_ret = pv.pct_change().dropna()
    n_years   = len(daily_ret) / 252
    total_ret = (pv.iloc[-1] / pv.iloc[0]) - 1
    cagr      = (1 + total_ret) ** (1 / n_years) - 1 if n_years > 0 else np.nan
    vol       = daily_ret.std() * np.sqrt(252)
    sharpe    = cagr / vol if vol > 0 else np.nan
    roll_max  = pv.cummax()
    max_dd    = ((pv - roll_max) / roll_max).min()
    calmar    = cagr / abs(max_dd) if max_dd != 0 else np.nan
    var_95    = daily_ret.quantile(0.05)

    return {
        'CAGR':         f"{cagr*100:.2f}%",
        'Volatility':   f"{vol*100:.2f}%",
        'Sharpe':       f"{sharpe:.2f}",
        'Max Drawdown': f"{max_dd*100:.2f}%",
        'Calmar':       f"{calmar:.2f}",
        'VaR 95%':      f"{var_95*100:.2f}%",
        'Total Return': f"{total_ret*100:.2f}%",
        'Final ($B)':   f"{pv.iloc[-1]/1e9:.3f}",
    }

print("\n" + "="*80)
print("EQUALLY-WEIGHTED BENCHMARK")
print("="*80)
ew_metrics = {k: compute_metrics(v) for k, v in ew_results.items()}
print(pd.DataFrame(ew_metrics).T.to_string())

print("\n" + "="*80)
print("SIGNAL-DRIVEN PORTFOLIO")
print("="*80)
sig_metrics = {k: compute_metrics(v) for k, v in signal_results.items()}
print(pd.DataFrame(sig_metrics).T.to_string())

# ── 13. DIAGNOSTIC: SIGNAL ACTIVITY ──────────────────────────────────────────
print("\n" + "="*80)
print("SIGNAL ACTIVITY SUMMARY")
print("="*80)
print(f"\nComposite score stats:\n{composite_score.describe()}")
print(f"\nSign distribution:")
print(f"  RISK_ON  (+1): {(composite_score > 0).sum()} days")
print(f"  RISK_OFF (-1): {(composite_score <= 0).sum()} days")

# ── 14. PLOTS ─────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(3, 2, figsize=(18, 16))
fig.suptitle("Portfolio Analysis: EW Benchmark vs Signal-Driven",
             fontsize=15, fontweight='bold')

colors = {'0%': 'black', '0.5%': 'steelblue', '1%': 'darkorange', '2%': 'crimson'}

# Row 1: Portfolio values
for col_idx, freq in enumerate(FREQS):
    ax = axes[0][col_idx]
    for tc_label in TC_LEVELS:
        ew_pv  = ew_results[f"EW_{freq}_TC{tc_label}"]['portfolio_value']
        sig_pv = signal_results[f"Signal_{freq}_TC{tc_label}"]['portfolio_value']
        ax.plot(ew_pv.index,  ew_pv  / 1e9, '--',
                color=colors[tc_label], lw=1.2, alpha=0.7, label=f"EW TC={tc_label}")
        ax.plot(sig_pv.index, sig_pv / 1e9,
                color=colors[tc_label], lw=1.5, label=f"Signal TC={tc_label}")
    ax.axhline(1.0, color='grey', ls=':', lw=0.8)
    ax.set_title(f"Portfolio Value — {freq.capitalize()}", fontsize=12)
    ax.set_ylabel("Value ($B)")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))
    ax.legend(fontsize=7, ncol=2)
    ax.grid(True, alpha=0.3)

# Row 2: Drawdowns
for col_idx, freq in enumerate(FREQS):
    ax = axes[1][col_idx]
    for tc_label in TC_LEVELS:
        sig_pv = signal_results[f"Signal_{freq}_TC{tc_label}"]['portfolio_value']
        dd     = (sig_pv - sig_pv.cummax()) / sig_pv.cummax() * 100
        ax.fill_between(dd.index, dd.values, 0,
                        alpha=0.2, color=colors[tc_label])
        ax.plot(dd.index, dd.values,
                color=colors[tc_label], lw=1.0, label=f"TC={tc_label}")
    ax.set_title(f"Signal Drawdown — {freq.capitalize()}", fontsize=12)
    ax.set_ylabel("Drawdown (%)")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

# Row 3: Composite signal over time
ax = axes[2][0]
ax.fill_between(composite_score.index, composite_score.values, 0,
                where=(composite_score > 0), color='green', alpha=0.4, label='RISK_ON')
ax.fill_between(composite_score.index, composite_score.values, 0,
                where=(composite_score <= 0), color='red', alpha=0.4, label='RISK_OFF')
ax.axhline(0, color='black', lw=0.8)
ax.set_title("Composite Determinant Signal", fontsize=12)
ax.set_ylabel("Composite Score")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Row 3: Signal transitions per window
ax = axes[2][1]
for window, series in det_results.items():
    sign = np.sign(series)
    ax.plot(sign.index, sign.values, alpha=0.6,
            label=f"{window}d window", lw=1.0)
ax.axhline(0, color='black', lw=0.8)
ax.set_title("Sign of Det(R-I) per Window", fontsize=12)
ax.set_ylabel("+1 RISK_ON / -1 RISK_OFF")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("signal_vs_ew_portfolio.png", dpi=150, bbox_inches='tight')
plt.show()
print("✓ Plot saved.")

# ── 15. SAVE ──────────────────────────────────────────────────────────────────
for label, df in {**ew_results, **signal_results}.items():
    df.to_csv(f"{label}_portfolio.csv")

all_metrics = pd.DataFrame({**ew_metrics, **sig_metrics}).T
all_metrics.to_csv("all_portfolio_metrics.csv")

composite_score.to_csv("composite_signal.csv")
signal_df.to_csv("signal_df_packaged.csv")

print("✓ All files saved.")


In [ ]:
# @title
# ============================================================
# SIGNAL PORTFOLIO — FULL SELF-CONTAINED VERSION
# Signals generated from Modified Determinant Model
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import requests
from io import StringIO

# ── 1. LOAD ALL DATA FROM GITHUB ─────────────────────────────────────────────
GITHUB_BASE = "https://raw.githubusercontent.com/kboroz/MSFE_Capstone_Project/main/01_Data/Streamlined/"

INDEX_FILE_MAP = {
    'ASX50':       'streamlined_asx_50.csv',
    'EUROSTOXX50': 'streamlined_euro_stoxx_50.csv',
    'FTSE100':     'streamlined_ftse_100.csv',
    'Ibovespa':    'streamlined_ibovespa.csv',
    'JSE40':       'streamlined_jse_top_40.csv',
    'NIKKEI225':   'streamlined_nikkei_225.csv',
    'SP500':       'streamlined_s&p_500.csv',
}

MACRO_FILE = 'streamlined_macro.csv'

def load_csv(url):
    r = requests.get(url)
    r.raise_for_status()
    df = pd.read_csv(StringIO(r.text), index_col=0, parse_dates=True)
    df.columns = df.columns.str.replace('[^a-zA-Z0-9]', '_', regex=True)
    return df.sort_index()

# Load macro
macro_df = load_csv(GITHUB_BASE + MACRO_FILE)
macro_df = macro_df[~macro_df.index.duplicated(keep='first')].sort_index()
print(f"macro_df shape : {macro_df.shape}")
print(f"Columns        : {macro_df.columns.tolist()}")

# Load individual index files (multi-column — OHLCV style)
index_data = {}
for name, fname in INDEX_FILE_MAP.items():
    try:
        df = load_csv(GITHUB_BASE + fname)
        index_data[name] = df
        print(f"✓ {name:15s} | shape={df.shape} | cols={df.columns.tolist()[:4]}")
    except Exception as e:
        print(f"✗ {name}: {e}")

# ── 2. EXTRACT PRICE SERIES FOR SIGNAL GENERATION ────────────────────────────
PRICE_COLS = ['Close', 'close', 'Price', 'price', 'Adj_Close', 'Last']

def get_price_series(df, name):
    col = next((c for c in PRICE_COLS if c in df.columns), df.columns[0])
    s   = pd.to_numeric(df[col], errors='coerce').dropna()
    print(f"  {name}: using column '{col}'")
    return s

index_prices = {}
for name, df in index_data.items():
    index_prices[name] = get_price_series(df, name)

# ── 3. MODIFIED DETERMINANT SIGNAL MODEL ─────────────────────────────────────
# Uses rolling correlation matrix of returns
# Signal = sign of det(R - I)
# RISK_ON  (+1) : det < 0  (correlations below identity — diversified)
# RISK_OFF (-1) : det >= 0 (correlations elevated — systemic risk)

DET_WINDOWS    = [3, 7, 28]
WINDOW_WEIGHTS = {3: 0.20, 7: 0.35, 28: 0.45}

def compute_modified_determinant(price_series_dict, windows=DET_WINDOWS):
    """
    For each window, compute rolling det(R - I) across all indices.
    Returns dict: {window: pd.DataFrame with columns per index and det column}
    """
    # Build returns dataframe from all indices
    prices = pd.DataFrame(price_series_dict).sort_index()
    rets   = prices.pct_change().dropna()

    det_results = {}

    for window in windows:
        det_series = []

        for i in range(window, len(rets)):
            window_rets = rets.iloc[i - window: i]

            # Drop columns with zero variance
            valid_cols = window_rets.columns[window_rets.std() > 1e-10]
            w = window_rets[valid_cols]

            if len(valid_cols) < 2:
                det_series.append(np.nan)
                continue

            try:
                R   = w.corr().values
                I   = np.eye(len(R))
                M   = R - I
                det = np.linalg.det(M)
            except Exception:
                det = np.nan

            det_series.append(det)

        idx = rets.index[window:]
        det_results[window] = pd.Series(det_series, index=idx, name=f'det_{window}d')
        print(f"  Window {window:2d}d | computed {len(det_series)} observations")

    return det_results

print("\n── Computing Modified Determinant ──")
det_results = compute_modified_determinant(index_prices)

# ── 4. GENERATE COMPOSITE SIGNAL PER INDEX ───────────────────────────────────
# For portfolio use we need per-index signals
# We compute each index's contribution by leaving it out (LOO) OR
# simply use the global composite score to drive all equity signals uniformly

def compute_composite_signal(det_results, window_weights=WINDOW_WEIGHTS):
    """
    Weighted average of sign across windows.
    Returns a Series of composite scores (between -1 and +1).
    Score > 0  → RISK_ON  → go long  (+1)
    Score <= 0 → RISK_OFF → go short (-1)
    """
    # Align all windows to common dates
    all_series = []
    for window, series in det_results.items():
        sign = np.sign(series).rename(f'sign_{window}d')
        all_series.append(sign)

    sign_df = pd.concat(all_series, axis=1).dropna()

    # Weighted composite
    composite = sum(
        sign_df[f'sign_{w}d'] * window_weights[w]
        for w in window_weights if f'sign_{w}d' in sign_df.columns
    )
    composite = composite / sum(window_weights.values())

    return composite, sign_df

composite_score, sign_df = compute_composite_signal(det_results)

print(f"\nComposite signal shape : {composite_score.shape}")
print(f"Date range             : {composite_score.index[0].date()} → "
      f"{composite_score.index[-1].date()}")
print(f"Score range            : {composite_score.min():.3f} → "
      f"{composite_score.max():.3f}")

# ── 5. BUILD SIGNAL DATAFRAME FOR PORTFOLIO ───────────────────────────────────
# Convert composite score to +1 / -1 per equity index
# We apply the SAME global signal to all equity indices
# (You can replace this with per-index LOO signals later)

EQUITY_ASSETS = list(index_prices.keys())  # indices we have signal data for

def build_signal_df(composite_score, equity_assets, macro_dates,
                    risk_on_threshold=0.0):
    """
    composite_score > threshold  → RISK_ON  → +1 (long)
    composite_score <= threshold → RISK_OFF → -1 (short)
    Reindexed to macro_dates with forward fill.
    """
    binary_signal = composite_score.apply(
        lambda x: 1 if x > risk_on_threshold else -1
    )

    signal_df = pd.DataFrame(index=macro_dates)
    for asset in equity_assets:
        aligned = binary_signal.reindex(macro_dates).ffill().bfill()
        signal_df[asset] = aligned

    return signal_df

signal_df = build_signal_df(composite_score, EQUITY_ASSETS, macro_df.index)

print(f"\nsignal_df shape : {signal_df.shape}")
print(f"Columns         : {signal_df.columns.tolist()}")
print(f"\nSignal distribution:")
for col in signal_df.columns:
    n_long  = (signal_df[col] == 1).sum()
    n_short = (signal_df[col] == -1).sum()
    print(f"  {col:15s} | LONG(+1): {n_long:4d} | SHORT(-1): {n_short:4d}")

# ── 6. DEFINE ASSET UNIVERSE ──────────────────────────────────────────────────
SAFE_BONDS = ['US_10Y', 'UK_10Y', 'AU_10Y', 'ZA_10Y']
FX         = ['AUDUSD', 'BRLUSD', 'EURUSD', 'GBPUSD', 'JPYUSD', 'ZARUSD']
COMMODITIES = ['Oil', 'Gold', 'Bitcoin']

ALL_ASSETS = EQUITY_ASSETS + FX + COMMODITIES + SAFE_BONDS
ALL_ASSETS = [a for a in ALL_ASSETS if a in macro_df.columns]

# Equity assets = those with signals AND in macro_df
equity_assets     = [a for a in EQUITY_ASSETS if a in ALL_ASSETS
                     and a in signal_df.columns]
non_equity_assets = [a for a in ALL_ASSETS if a not in equity_assets]

N           = len(ALL_ASSETS)
base_weight = 1.0 / N

print(f"\nTotal assets              : {N}")
print(f"Equity (signal-driven)    : {equity_assets}")
print(f"Non-equity (always long)  : {non_equity_assets}")
print(f"base_weight = 1/{N} = {base_weight:.4f}")

# ── 7. BUILD RETURNS ──────────────────────────────────────────────────────────
returns_raw = macro_df[ALL_ASSETS].pct_change(fill_method=None)

# Invert bond yields
for col in SAFE_BONDS:
    if col in returns_raw.columns:
        returns_raw[col] = -returns_raw[col]

# Fix Oil negative price days
if 'Oil' in macro_df.columns:
    bad_oil = macro_df['Oil'][macro_df['Oil'] <= 0].index
    for d in bad_oil:
        loc = returns_raw.index.get_loc(d)
        returns_raw.iloc[
            max(0, loc - 1): loc + 2,
            returns_raw.columns.get_loc('Oil')
        ] = 0.0

# Clip, fill, drop first row
returns = returns_raw.clip(lower=-0.50, upper=1.00).fillna(0).iloc[1:]

assert not returns.isnull().any().any(), "NaNs in returns"
assert not np.isinf(returns.values).any(), "Inf in returns"
print(f"\nReturns shape : {returns.shape}")
print("✓ Returns clean")

# ── 8. WEIGHT BUILDER ─────────────────────────────────────────────────────────
def build_weights(sig_row, equity_assets, non_equity_assets, base_weight, date):
    raw = {}

    for a in non_equity_assets:
        raw[a] = base_weight

    for a in equity_assets:
        if sig_row is not None and a in sig_row.index and not pd.isna(sig_row[a]):
            raw[a] = base_weight if sig_row[a] >= 0 else -base_weight
        else:
            raw[a] = base_weight

    # Reinvest short proceeds into long positions
    short_proceeds = sum(abs(w) for w in raw.values() if w < 0)
    long_assets    = [a for a, w in raw.items() if w > 0]
    n_long         = len(long_assets)

    if short_proceeds > 0 and n_long > 0:
        extra = short_proceeds / n_long
        for a in long_assets:
            raw[a] += extra

    total = sum(raw.values())
    if abs(total - 1.0) > 1e-6:
        raise ValueError(f"Weights sum to {total:.8f} on {date}")

    return raw

# ── 9. EQUAL-WEIGHT BENCHMARK ─────────────────────────────────────────────────
def build_weights(sig_row, equity_assets, non_equity_assets, base_weight, date):
    raw = {}

    for a in non_equity_assets:
        raw[a] = base_weight

    for a in equity_assets:
        if sig_row is not None and a in sig_row.index and not pd.isna(sig_row[a]):
            # FLIPPED: signal +1 (was RISK_ON) → now SHORT, signal -1 (was RISK_OFF) → now LONG
            raw[a] = base_weight if sig_row[a] < 0 else -base_weight
        else:
            raw[a] = base_weight

    # Reinvest short proceeds into long positions
    short_proceeds = sum(abs(w) for w in raw.values() if w < 0)
    long_assets    = [a for a, w in raw.items() if w > 0]
    n_long         = len(long_assets)

    if short_proceeds > 0 and n_long > 0:
        extra = short_proceeds / n_long
        for a in long_assets:
            raw[a] += extra
    elif short_proceeds > 0 and n_long == 0:
        for a in raw:
            raw[a] = 1.0 / len(raw)

    total = sum(raw.values())
    if total <= 0:
        for a in raw:
            raw[a] = 1.0 / len(raw)
    else:
        for a in raw:
            raw[a] = raw[a] / total

    return raw



# ── 10. SIGNAL-DRIVEN PORTFOLIO ───────────────────────────────────────────────
def run_signal_portfolio(returns, signal_df, equity_assets, non_equity_assets,
                          all_assets, freq='monthly', tc_rate=0.005,
                          initial_capital=1e9):

    N           = len(all_assets)
    base_weight = 1.0 / N

    freq_code     = 'MS' if freq == 'monthly' else 'QS'
    reb_dates_raw = returns.resample(freq_code).first().index
    actual_dates  = returns.index

    reb_set = set()
    for d in reb_dates_raw:
        future = actual_dates[actual_dates >= d]
        if len(future) > 0:
            reb_set.add(future[0])

    sig = signal_df.copy()
    sig.index = pd.to_datetime(sig.index)

    pv              = float(initial_capital)
    current_weights = None
    records         = []

    for date in actual_dates:
        if date in reb_set:
            past    = sig[sig.index <= date]
            sig_row = past.iloc[-1] if len(past) > 0 else None

            raw = build_weights(
                sig_row, equity_assets, non_equity_assets, base_weight, date
            )

            if current_weights is not None:
                all_keys = set(raw) | set(current_weights)
                turnover = sum(
                    abs(raw.get(a, 0) - current_weights.get(a, 0))
                    for a in all_keys
                ) / 2.0
                tc = turnover * tc_rate
            else:
                tc = tc_rate

            pv             *= (1 - tc)
            current_weights = raw.copy()

        if current_weights is not None:
            dr  = sum(current_weights.get(a, 0) * returns.loc[date, a]
                      for a in all_assets)
            pv *= (1 + dr)

        records.append({'date': date, 'portfolio_value': pv})

    return pd.DataFrame(records).set_index('date')

# ── 11. RUN ALL PORTFOLIOS ────────────────────────────────────────────────────
TC_LEVELS = {'0%': 0.00, '0.5%': 0.005, '1%': 0.01, '2%': 0.02}
FREQS     = ['monthly', 'quarterly']

ew_results = {}
for freq in FREQS:
    for tc_label, tc_rate in TC_LEVELS.items():
        label = f"EW_{freq}_TC{tc_label}"
        ew_results[label] = run_equal_weight(
            returns, ALL_ASSETS, tc_rate=tc_rate,
            rebal_freq=freq, initial_capital=1e9
        )
        print(f"✓ {label}")

signal_results = {}
for freq in FREQS:
    for tc_label, tc_rate in TC_LEVELS.items():
        label = f"Signal_{freq}_TC{tc_label}"
        signal_results[label] = run_signal_portfolio(
            returns, signal_df, equity_assets, non_equity_assets,
            ALL_ASSETS, freq=freq, tc_rate=tc_rate, initial_capital=1e9
        )
        print(f"✓ {label}")

print("\n✓ All portfolios computed")

# ── 12. PERFORMANCE METRICS ───────────────────────────────────────────────────
def compute_metrics(pv_series):
    pv        = pv_series['portfolio_value']
    daily_ret = pv.pct_change().dropna()
    n_years   = len(daily_ret) / 252
    total_ret = (pv.iloc[-1] / pv.iloc[0]) - 1
    cagr      = (1 + total_ret) ** (1 / n_years) - 1 if n_years > 0 else np.nan
    vol       = daily_ret.std() * np.sqrt(252)
    sharpe    = cagr / vol if vol > 0 else np.nan
    roll_max  = pv.cummax()
    max_dd    = ((pv - roll_max) / roll_max).min()
    calmar    = cagr / abs(max_dd) if max_dd != 0 else np.nan
    var_95    = daily_ret.quantile(0.05)

    return {
        'CAGR':         f"{cagr*100:.2f}%",
        'Volatility':   f"{vol*100:.2f}%",
        'Sharpe':       f"{sharpe:.2f}",
        'Max Drawdown': f"{max_dd*100:.2f}%",
        'Calmar':       f"{calmar:.2f}",
        'VaR 95%':      f"{var_95*100:.2f}%",
        'Total Return': f"{total_ret*100:.2f}%",
        'Final ($B)':   f"{pv.iloc[-1]/1e9:.3f}",
    }

print("\n" + "="*80)
print("EQUALLY-WEIGHTED BENCHMARK")
print("="*80)
ew_metrics = {k: compute_metrics(v) for k, v in ew_results.items()}
print(pd.DataFrame(ew_metrics).T.to_string())

print("\n" + "="*80)
print("SIGNAL-DRIVEN PORTFOLIO")
print("="*80)
sig_metrics = {k: compute_metrics(v) for k, v in signal_results.items()}
print(pd.DataFrame(sig_metrics).T.to_string())

# ── 13. DIAGNOSTIC: SIGNAL ACTIVITY ──────────────────────────────────────────
print("\n" + "="*80)
print("SIGNAL ACTIVITY SUMMARY")
print("="*80)
print(f"\nComposite score stats:\n{composite_score.describe()}")
print(f"\nSign distribution:")
print(f"  RISK_ON  (+1): {(composite_score > 0).sum()} days")
print(f"  RISK_OFF (-1): {(composite_score <= 0).sum()} days")

# ── 14. PLOTS ─────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(3, 2, figsize=(18, 16))
fig.suptitle("Portfolio Analysis: EW Benchmark vs Signal-Driven",
             fontsize=15, fontweight='bold')

colors = {'0%': 'black', '0.5%': 'steelblue', '1%': 'darkorange', '2%': 'crimson'}

# Row 1: Portfolio values
for col_idx, freq in enumerate(FREQS):
    ax = axes[0][col_idx]
    for tc_label in TC_LEVELS:
        ew_pv  = ew_results[f"EW_{freq}_TC{tc_label}"]['portfolio_value']
        sig_pv = signal_results[f"Signal_{freq}_TC{tc_label}"]['portfolio_value']
        ax.plot(ew_pv.index,  ew_pv  / 1e9, '--',
                color=colors[tc_label], lw=1.2, alpha=0.7, label=f"EW TC={tc_label}")
        ax.plot(sig_pv.index, sig_pv / 1e9,
                color=colors[tc_label], lw=1.5, label=f"Signal TC={tc_label}")
    ax.axhline(1.0, color='grey', ls=':', lw=0.8)
    ax.set_title(f"Portfolio Value — {freq.capitalize()}", fontsize=12)
    ax.set_ylabel("Value ($B)")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))
    ax.legend(fontsize=7, ncol=2)
    ax.grid(True, alpha=0.3)

# Row 2: Drawdowns
for col_idx, freq in enumerate(FREQS):
    ax = axes[1][col_idx]
    for tc_label in TC_LEVELS:
        sig_pv = signal_results[f"Signal_{freq}_TC{tc_label}"]['portfolio_value']
        dd     = (sig_pv - sig_pv.cummax()) / sig_pv.cummax() * 100
        ax.fill_between(dd.index, dd.values, 0,
                        alpha=0.2, color=colors[tc_label])
        ax.plot(dd.index, dd.values,
                color=colors[tc_label], lw=1.0, label=f"TC={tc_label}")
    ax.set_title(f"Signal Drawdown — {freq.capitalize()}", fontsize=12)
    ax.set_ylabel("Drawdown (%)")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

# Row 3: Composite signal over time
ax = axes[2][0]
ax.fill_between(composite_score.index, composite_score.values, 0,
                where=(composite_score > 0), color='green', alpha=0.4, label='RISK_ON')
ax.fill_between(composite_score.index, composite_score.values, 0,
                where=(composite_score <= 0), color='red', alpha=0.4, label='RISK_OFF')
ax.axhline(0, color='black', lw=0.8)
ax.set_title("Composite Determinant Signal", fontsize=12)
ax.set_ylabel("Composite Score")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Row 3: Signal transitions per window
ax = axes[2][1]
for window, series in det_results.items():
    sign = np.sign(series)
    ax.plot(sign.index, sign.values, alpha=0.6,
            label=f"{window}d window", lw=1.0)
ax.axhline(0, color='black', lw=0.8)
ax.set_title("Sign of Det(R-I) per Window", fontsize=12)
ax.set_ylabel("+1 RISK_ON / -1 RISK_OFF")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("signal_vs_ew_portfolio.png", dpi=150, bbox_inches='tight')
plt.show()
print("✓ Plot saved.")

# ── 15. SAVE ──────────────────────────────────────────────────────────────────
for label, df in {**ew_results, **signal_results}.items():
    df.to_csv(f"{label}_portfolio.csv")

all_metrics = pd.DataFrame({**ew_metrics, **sig_metrics}).T
all_metrics.to_csv("all_portfolio_metrics.csv")

composite_score.to_csv("composite_signal.csv")
signal_df.to_csv("signal_df_packaged.csv")

print("✓ All files saved.")


In [ ]:
# @title
# ============================================================
# SIGNAL PORTFOLIO — THREE-STAGE THRESHOLD VERSION
# Signals generated from Modified Determinant Model
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import requests
from io import StringIO

# ── 1. LOAD ALL DATA FROM GITHUB ─────────────────────────────────────────────
GITHUB_BASE = "https://raw.githubusercontent.com/kboroz/MSFE_Capstone_Project/main/01_Data/Streamlined/"

INDEX_FILE_MAP = {
    'ASX50':       'streamlined_asx_50.csv',
    'EUROSTOXX50': 'streamlined_euro_stoxx_50.csv',
    'FTSE100':     'streamlined_ftse_100.csv',
    'Ibovespa':    'streamlined_ibovespa.csv',
    'JSE40':       'streamlined_jse_top_40.csv',
    'NIKKEI225':   'streamlined_nikkei_225.csv',
    'SP500':       'streamlined_s&p_500.csv',
}

MACRO_FILE = 'streamlined_macro.csv'

def load_csv(url):
    r = requests.get(url)
    r.raise_for_status()
    df = pd.read_csv(StringIO(r.text), index_col=0, parse_dates=True)
    df.columns = df.columns.str.replace('[^a-zA-Z0-9]', '_', regex=True)
    return df.sort_index()

# Load macro
macro_df = load_csv(GITHUB_BASE + MACRO_FILE)
macro_df = macro_df[~macro_df.index.duplicated(keep='first')].sort_index()
print(f"macro_df shape : {macro_df.shape}")
print(f"Columns        : {macro_df.columns.tolist()}")

# Load individual index files
index_data = {}
for name, fname in INDEX_FILE_MAP.items():
    try:
        df = load_csv(GITHUB_BASE + fname)
        index_data[name] = df
        print(f"✓ {name:15s} | shape={df.shape} | cols={df.columns.tolist()[:4]}")
    except Exception as e:
        print(f"✗ {name}: {e}")

# ── 2. EXTRACT PRICE SERIES ───────────────────────────────────────────────────
PRICE_COLS = ['Close', 'close', 'Price', 'price', 'Adj_Close', 'Last']

def get_price_series(df, name):
    col = next((c for c in PRICE_COLS if c in df.columns), df.columns[0])
    s   = pd.to_numeric(df[col], errors='coerce').dropna()
    print(f"  {name}: using column '{col}'")
    return s

index_prices = {}
for name, df in index_data.items():
    index_prices[name] = get_price_series(df, name)

# ── 3. MODIFIED DETERMINANT SIGNAL MODEL ─────────────────────────────────────
DET_WINDOWS    = [3, 7, 28]
WINDOW_WEIGHTS = {3: 0.20, 7: 0.35, 28: 0.45}

def compute_modified_determinant(price_series_dict, windows=DET_WINDOWS):
    prices = pd.DataFrame(price_series_dict).sort_index()
    rets   = prices.pct_change().dropna()
    det_results = {}

    for window in windows:
        det_series = []
        for i in range(window, len(rets)):
            window_rets = rets.iloc[i - window: i]
            valid_cols  = window_rets.columns[window_rets.std() > 1e-10]
            w           = window_rets[valid_cols]

            if len(valid_cols) < 2:
                det_series.append(np.nan)
                continue
            try:
                R   = w.corr().values
                I   = np.eye(len(R))
                M   = R - I
                det = np.linalg.det(M)
            except Exception:
                det = np.nan
            det_series.append(det)

        idx = rets.index[window:]
        det_results[window] = pd.Series(det_series, index=idx, name=f'det_{window}d')
        print(f"  Window {window:2d}d | computed {len(det_series)} observations")

    return det_results

print("\n── Computing Modified Determinant ──")
det_results = compute_modified_determinant(index_prices)

# ── 4. COMPOSITE SCORE ────────────────────────────────────────────────────────
def compute_composite_score(det_results, window_weights=WINDOW_WEIGHTS):
    """
    Returns raw weighted-average of normalised determinant values (not signs).
    Normalised per window via rolling z-score so windows are comparable.
    """
    all_series = []
    for window, series in det_results.items():
        # z-score normalise so scale is comparable across windows
        mu    = series.rolling(252, min_periods=60).mean()
        sigma = series.rolling(252, min_periods=60).std()
        z     = (series - mu) / sigma.replace(0, np.nan)
        z.name = f'z_{window}d'
        all_series.append(z)

    z_df      = pd.concat(all_series, axis=1).dropna()
    composite = sum(
        z_df[f'z_{w}d'] * window_weights[w]
        for w in window_weights if f'z_{w}d' in z_df.columns
    )
    composite = composite / sum(window_weights.values())
    return composite, z_df

composite_score, z_df = compute_composite_score(det_results)

print(f"\nComposite score shape  : {composite_score.shape}")
print(f"Date range             : {composite_score.index[0].date()} → "
      f"{composite_score.index[-1].date()}")
print(f"Score range            : {composite_score.min():.3f} → "
      f"{composite_score.max():.3f}")

# ── 5. THREE-STAGE SIGNAL CLASSIFICATION ─────────────────────────────────────
# |score| > threshold  → RISK_ON  (+1): high det magnitude = abnormal regime
#                         → short equity, long country bonds + FX
# |score| <= threshold → RISK_OFF (-1): low det magnitude  = calm regime
#                         → long equity

THRESHOLDS = [0.3, 0.5, 0.7]   # tune here

def classify_signal(score, threshold):
    """
    Returns +1 (RISK_ON / defensive) or -1 (RISK_OFF / long equity)
    """
    if abs(score) > threshold:
        return +1   # high |det| → abnormal → defensive
    else:
        return -1   # low  |det| → calm     → long equity

def build_signal_df(composite_score, equity_assets, macro_dates, threshold=0.5):
    """
    Apply three-stage classification and reindex to portfolio dates.
    """
    binary = composite_score.apply(lambda x: classify_signal(x, threshold))
    signal_df = pd.DataFrame(index=macro_dates)
    for asset in equity_assets:
        signal_df[asset] = binary.reindex(macro_dates).ffill().bfill()
    return signal_df

# ── 6. ASSET UNIVERSE ─────────────────────────────────────────────────────────
EQUITY_ASSETS = list(index_prices.keys())
SAFE_BONDS    = ['US_10Y', 'UK_10Y', 'AU_10Y', 'ZA_10Y']
FX            = ['AUDUSD', 'BRLUSD', 'EURUSD', 'GBPUSD', 'JPYUSD', 'ZARUSD']
COMMODITIES   = ['Oil', 'Gold', 'Bitcoin']

ALL_ASSETS        = EQUITY_ASSETS + FX + COMMODITIES + SAFE_BONDS
ALL_ASSETS        = [a for a in ALL_ASSETS if a in macro_df.columns]
equity_assets     = [a for a in EQUITY_ASSETS if a in ALL_ASSETS]
non_equity_assets = [a for a in ALL_ASSETS if a not in equity_assets]

N           = len(ALL_ASSETS)
base_weight = 1.0 / N

print(f"\nTotal assets              : {N}")
print(f"Equity (signal-driven)    : {equity_assets}")
print(f"Non-equity (always long)  : {non_equity_assets}")
print(f"base_weight = 1/{N} = {base_weight:.4f}")

# ── 7. COUNTRY SAFE-HAVEN MAP ─────────────────────────────────────────────────
# When an equity index is shorted (RISK_ON), proceeds are redistributed
# to that country's bond + FX where available in the universe

EQUITY_SAFE_HAVEN_MAP = {
    'ASX50':       ['AU_10Y', 'AUDUSD'],
    'EUROSTOXX50': ['EURUSD'],            # no EU bond in universe
    'FTSE100':     ['UK_10Y', 'GBPUSD'],
    'Ibovespa':    ['BRLUSD'],            # no Brazil bond in universe
    'JSE40':       ['ZA_10Y', 'ZARUSD'],
    'NIKKEI225':   ['JPYUSD'],            # no JP bond in universe
    'SP500':       ['US_10Y'],            # USD is base currency
}

# Filter to assets actually in universe
UNIVERSE_SET = set(ALL_ASSETS)
for eq, havens in EQUITY_SAFE_HAVEN_MAP.items():
    EQUITY_SAFE_HAVEN_MAP[eq] = [h for h in havens if h in UNIVERSE_SET]

print("\nSafe-haven routing (filtered to universe):")
for eq, havens in EQUITY_SAFE_HAVEN_MAP.items():
    print(f"  {eq:15s} → {havens}")

# ── 8. BUILD RETURNS ──────────────────────────────────────────────────────────
returns_raw = macro_df[ALL_ASSETS].pct_change(fill_method=None)

for col in SAFE_BONDS:
    if col in returns_raw.columns:
        returns_raw[col] = -returns_raw[col]   # invert yield → price

if 'Oil' in macro_df.columns:
    bad_oil = macro_df['Oil'][macro_df['Oil'] <= 0].index
    for d in bad_oil:
        loc = returns_raw.index.get_loc(d)
        returns_raw.iloc[
            max(0, loc - 1): loc + 2,
            returns_raw.columns.get_loc('Oil')
        ] = 0.0

returns = returns_raw.clip(lower=-0.50, upper=1.00).fillna(0).iloc[1:]

assert not returns.isnull().any().any(), "NaNs in returns"
assert not np.isinf(returns.values).any(),  "Inf in returns"
print(f"\nReturns shape : {returns.shape}")
print("✓ Returns clean")

# ── 9. THREE-STAGE WEIGHT BUILDER ────────────────────────────────────────────
def build_weights_3stage(sig_row, equity_assets, non_equity_assets,
                         base_weight, date, safe_haven_map):
    """
    RISK_OFF (-1) : long equity at base_weight
    RISK_ON  (+1) : short equity, redistribute proceeds to country safe havens
                    or equally across all non-equity if no safe haven available
    Non-equity assets always start at base_weight; safe-haven assets receive
    additional weight from shorted equities.
    """
    raw = {a: base_weight for a in non_equity_assets}

    safe_haven_extra = {}   # accumulate extra weight per safe-haven asset

    for a in equity_assets:
        if sig_row is not None and a in sig_row.index and not pd.isna(sig_row[a]):
            sig = sig_row[a]
        else:
            sig = -1   # default: long equity

        if sig > 0:
            # RISK_ON: short equity, redirect base_weight to safe havens
            raw[a]   = -base_weight
            havens   = safe_haven_map.get(a, [])
            havens   = [h for h in havens if h in raw]   # only if in universe

            if havens:
                per_haven = base_weight / len(havens)
                for h in havens:
                    safe_haven_extra[h] = safe_haven_extra.get(h, 0) + per_haven
            else:
                # No safe haven available → spread across all non-equity
                per_ne = base_weight / len(non_equity_assets)
                for ne in non_equity_assets:
                    safe_haven_extra[ne] = safe_haven_extra.get(ne, 0) + per_ne
        else:
            # RISK_OFF: long equity
            raw[a] = base_weight

    # Apply safe-haven extras
    for asset, extra in safe_haven_extra.items():
        raw[asset] = raw.get(asset, 0) + extra

    # Normalise to sum = 1 (handles edge cases)
    total = sum(raw.values())
    if total <= 0:
        for a in raw:
            raw[a] = 1.0 / len(raw)
    else:
        for a in raw:
            raw[a] /= total

    return raw

# ── 10. EQUAL-WEIGHT BENCHMARK ────────────────────────────────────────────────
def run_equal_weight(returns, all_assets, tc_rate=0.005,
                     rebal_freq='monthly', initial_capital=1e9):
    N           = len(all_assets)
    base_weight = 1.0 / N
    ew          = {a: base_weight for a in all_assets}

    freq_code     = 'MS' if rebal_freq == 'monthly' else 'QS'
    reb_dates_raw = returns.resample(freq_code).first().index
    actual_dates  = returns.index

    reb_set = set()
    for d in reb_dates_raw:
        future = actual_dates[actual_dates >= d]
        if len(future) > 0:
            reb_set.add(future[0])

    pv              = float(initial_capital)
    current_weights = None
    records         = []

    for date in actual_dates:
        if date in reb_set:
            if current_weights is not None:
                turnover = sum(abs(ew.get(a, 0) - current_weights.get(a, 0))
                               for a in set(ew) | set(current_weights)) / 2.0
                pv *= (1 - turnover * tc_rate)
            else:
                pv *= (1 - tc_rate)
            current_weights = ew.copy()

        if current_weights is not None:
            dr  = sum(current_weights.get(a, 0) * returns.loc[date, a]
                      for a in all_assets)
            pv *= (1 + dr)

        records.append({'date': date, 'portfolio_value': pv})

    return pd.DataFrame(records).set_index('date')

# ── 11. THREE-STAGE SIGNAL PORTFOLIO RUNNER ───────────────────────────────────
def run_signal_portfolio_3stage(returns, signal_df, equity_assets,
                                non_equity_assets, all_assets,
                                freq='monthly', tc_rate=0.005,
                                safe_haven_map=None,
                                initial_capital=1e9):

    if safe_haven_map is None:
        safe_haven_map = {}

    N           = len(all_assets)
    base_weight = 1.0 / N

    freq_code     = 'MS' if freq == 'monthly' else 'QS'
    reb_dates_raw = returns.resample(freq_code).first().index
    actual_dates  = returns.index

    reb_set = set()
    for d in reb_dates_raw:
        future = actual_dates[actual_dates >= d]
        if len(future) > 0:
            reb_set.add(future[0])

    sig = signal_df.copy()
    sig.index = pd.to_datetime(sig.index)

    pv              = float(initial_capital)
    current_weights = None
    records         = []

    for date in actual_dates:
        if date in reb_set:
            past    = sig[sig.index <= date]
            sig_row = past.iloc[-1] if len(past) > 0 else None

            raw = build_weights_3stage(
                sig_row, equity_assets, non_equity_assets,
                base_weight, date, safe_haven_map
            )

            if current_weights is not None:
                all_keys = set(raw) | set(current_weights)
                turnover = sum(
                    abs(raw.get(a, 0) - current_weights.get(a, 0))
                    for a in all_keys
                ) / 2.0
                pv *= (1 - turnover * tc_rate)
            else:
                pv *= (1 - tc_rate)

            current_weights = raw.copy()

        if current_weights is not None:
            dr  = sum(current_weights.get(a, 0) * returns.loc[date, a]
                      for a in all_assets)
            pv *= (1 + dr)

        records.append({'date': date, 'portfolio_value': pv})

    return pd.DataFrame(records).set_index('date')

# ── 12. RUN ALL PORTFOLIOS ────────────────────────────────────────────────────
TC_LEVELS = {'0%': 0.00, '0.5%': 0.005, '1%': 0.01, '2%': 0.02}
FREQS     = ['monthly', 'quarterly']

# Equal-weight benchmarks
ew_results = {}
for freq in FREQS:
    for tc_label, tc_rate in TC_LEVELS.items():
        label = f"EW_{freq}_TC{tc_label}"
        ew_results[label] = run_equal_weight(
            returns, ALL_ASSETS, tc_rate=tc_rate,
            rebal_freq=freq, initial_capital=1e9
        )
        print(f"✓ {label}")

# Three-stage signal portfolios across all thresholds
signal_results = {}
signal_distributions = {}

for thresh in THRESHOLDS:
    sig_df_t = build_signal_df(composite_score, equity_assets,
                               macro_df.index, threshold=thresh)

    # Print signal distribution for this threshold
    n_risk_on  = (sig_df_t[equity_assets[0]] == +1).sum()
    n_risk_off = (sig_df_t[equity_assets[0]] == -1).sum()
    total      = len(sig_df_t)
    print(f"\nThreshold={thresh} | RISK_ON(+1): {n_risk_on} days "
          f"({n_risk_on/total*100:.1f}%) | "
          f"RISK_OFF(-1): {n_risk_off} days ({n_risk_off/total*100:.1f}%)")

    signal_distributions[thresh] = {'RISK_ON': n_risk_on, 'RISK_OFF': n_risk_off}

    for freq in FREQS:
        for tc_label, tc_rate in TC_LEVELS.items():
            label = f"Signal3S_t{thresh}_{freq}_TC{tc_label}"
            signal_results[label] = run_signal_portfolio_3stage(
                returns, sig_df_t, equity_assets, non_equity_assets,
                ALL_ASSETS, freq=freq, tc_rate=tc_rate,
                safe_haven_map=EQUITY_SAFE_HAVEN_MAP,
                initial_capital=1e9
            )
            print(f"  ✓ {label}")

print("\n✓ All portfolios computed")

# ── 13. PERFORMANCE METRICS ───────────────────────────────────────────────────
def compute_metrics(pv_df, label=''):
    pv        = pv_df['portfolio_value']
    daily_ret = pv.pct_change().dropna()
    n_years   = len(daily_ret) / 252
    total_ret = (pv.iloc[-1] / pv.iloc[0]) - 1
    cagr      = (1 + total_ret) ** (1 / n_years) - 1 if n_years > 0 else np.nan
    vol       = daily_ret.std() * np.sqrt(252)
    sharpe    = cagr / vol if vol > 0 else np.nan
    roll_max  = pv.cummax()
    max_dd    = ((pv - roll_max) / roll_max).min()
    calmar    = cagr / abs(max_dd) if max_dd != 0 else np.nan
    var_95    = daily_ret.quantile(0.05)

    return {
        'CAGR':         f"{cagr*100:.2f}%",
        'Volatility':   f"{vol*100:.2f}%",
        'Sharpe':       f"{sharpe:.2f}",
        'Max Drawdown': f"{max_dd*100:.2f}%",
        'Calmar':       f"{calmar:.2f}",
        'VaR 95%':      f"{var_95*100:.2f}%",
        'Total Return': f"{total_ret*100:.2f}%",
        'Final ($B)':   f"{pv.iloc[-1]/1e9:.3f}",
    }

# Print results grouped by threshold
ew_metrics = {k: compute_metrics(v, k) for k, v in ew_results.items()}

print("\n" + "="*90)
print("EQUAL-WEIGHT BENCHMARK")
print("="*90)
print(pd.DataFrame(ew_metrics).T.to_string())

sig_metrics = {k: compute_metrics(v, k) for k, v in signal_results.items()}

for thresh in THRESHOLDS:
    sub = {k: v for k, v in sig_metrics.items() if f"_t{thresh}_" in k}
    print(f"\n{'='*90}")
    print(f"THREE-STAGE SIGNAL — Threshold = {thresh} | "
          f"RISK_ON: {signal_distributions[thresh]['RISK_ON']} days | "
          f"RISK_OFF: {signal_distributions[thresh]['RISK_OFF']} days")
    print("="*90)
    print(pd.DataFrame(sub).T.to_string())

# ── 14. PLOTS ─────────────────────────────────────────────────────────────────
# One figure per threshold + one overview figure

colors_tc   = {'0%': 'black', '0.5%': 'steelblue', '1%': 'darkorange', '2%': 'crimson'}
colors_freq = {'monthly': 'steelblue', 'quarterly': 'darkorange'}

# ── Figure 1: Composite signal + safe-haven routing overview ──────────────────
fig, axes = plt.subplots(2, 1, figsize=(16, 10))
fig.suptitle("Modified Determinant — Composite Score & Threshold Signals",
             fontsize=14, fontweight='bold')

ax = axes[0]
ax.plot(composite_score.index, composite_score.values,
        color='navy', lw=1.0, alpha=0.8, label='Composite Z-score')
for thresh, col in zip(THRESHOLDS, ['green', 'orange', 'red']):
    ax.axhline( thresh, color=col, ls='--', lw=1.0, label=f'+{thresh}')
    ax.axhline(-thresh, color=col, ls='--', lw=1.0, label=f'-{thresh}')
ax.axhline(0, color='black', lw=0.6)
ax.fill_between(composite_score.index, composite_score.values, 0,
                where=(composite_score > 0), color='green', alpha=0.1)
ax.fill_between(composite_score.index, composite_score.values, 0,
                where=(composite_score <= 0), color='red', alpha=0.1)
ax.set_title("Composite Determinant Score with Thresholds", fontsize=11)
ax.set_ylabel("Z-score")
ax.legend(fontsize=8, ncol=4)
ax.grid(True, alpha=0.3)

ax = axes[1]
for window, series in det_results.items():
    ax.plot(series.index, np.sign(series.values), alpha=0.6,
            label=f"{window}d window", lw=1.0)
ax.axhline(0, color='black', lw=0.8)
ax.set_title("Sign of Det(R-I) per Window", fontsize=11)
ax.set_ylabel("+1 / -1")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("composite_signal_thresholds.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Figure 2: Portfolio values per threshold (TC=0.5%, monthly) ───────────────
fig, axes = plt.subplots(1, 3, figsize=(20, 6), sharey=False)
fig.suptitle("Three-Stage Signal Portfolio — Value by Threshold (TC=0.5%, Monthly)",
             fontsize=13, fontweight='bold')

for ax, thresh in zip(axes, THRESHOLDS):
    # EW reference
    ew_pv = ew_results['EW_monthly_TC0.5%']['portfolio_value']
    ax.plot(ew_pv.index, ew_pv / 1e9, '--', color='grey',
            lw=1.5, label='EW Benchmark')

    label = f"Signal3S_t{thresh}_monthly_TC0.5%"
    pv    = signal_results[label]['portfolio_value']
    ax.plot(pv.index, pv / 1e9, color='steelblue', lw=1.8,
            label=f'Signal (t={thresh})')

    ax.axhline(1.0, color='black', ls=':', lw=0.8)
    ax.set_title(f"Threshold = {thresh}", fontsize=11)
    ax.set_ylabel("Value ($B)")
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("signal_3stage_by_threshold.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Figure 3: Drawdowns per threshold ────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(20, 5), sharey=True)
fig.suptitle("Three-Stage Signal Portfolio — Drawdowns by Threshold (TC=0.5%, Monthly)",
             fontsize=13, fontweight='bold')

for ax, thresh in zip(axes, THRESHOLDS):
    label  = f"Signal3S_t{thresh}_monthly_TC0.5%"
    pv     = signal_results[label]['portfolio_value']
    dd     = (pv - pv.cummax()) / pv.cummax() * 100
    ew_pv  = ew_results['EW_monthly_TC0.5%']['portfolio_value']
    ew_dd  = (ew_pv - ew_pv.cummax()) / ew_pv.cummax() * 100

    ax.fill_between(dd.index,  dd.values,  0, alpha=0.3, color='steelblue')
    ax.fill_between(ew_dd.index, ew_dd.values, 0, alpha=0.2, color='grey')
    ax.plot(dd.index,    dd.values,    color='steelblue', lw=1.0,
            label=f'Signal (t={thresh})')
    ax.plot(ew_dd.index, ew_dd.values, color='grey',      lw=1.0, ls='--',
            label='EW Benchmark')
    ax.set_title(f"Threshold = {thresh}", fontsize=11)
    ax.set_ylabel("Drawdown (%)")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("signal_3stage_drawdowns.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Figure 4: TC sensitivity per threshold ────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.suptitle("Three-Stage Signal — TC Sensitivity (Monthly)",
             fontsize=13, fontweight='bold')

for ax, thresh in zip(axes, THRESHOLDS):
    for tc_label, col in colors_tc.items():
        label = f"Signal3S_t{thresh}_monthly_TC{tc_label}"
        pv    = signal_results[label]['portfolio_value']
        ax.plot(pv.index, pv / 1e9, color=col, lw=1.3,
                label=f'TC={tc_label}')
    ew_pv = ew_results['EW_monthly_TC0.5%']['portfolio_value']
    ax.plot(ew_pv.index, ew_pv / 1e9, '--', color='grey',
            lw=1.5, label='EW Benchmark')
    ax.set_title(f"Threshold = {thresh}", fontsize=11)
    ax.set_ylabel("Value ($B)")
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("signal_3stage_tc_sensitivity.png", dpi=150, bbox_inches='tight')
plt.show()

# ── 15. SAVE ──────────────────────────────────────────────────────────────────
for label, df in {**ew_results, **signal_results}.items():
    df.to_csv(f"{label}_portfolio.csv")

all_metrics = pd.DataFrame({**ew_metrics, **sig_metrics}).T
all_metrics.to_csv("all_portfolio_metrics.csv")

composite_score.to_csv("composite_signal.csv")

for thresh in THRESHOLDS:
    sig_df_t = build_signal_df(composite_score, equity_assets,
                               macro_df.index, threshold=thresh)
    sig_df_t.to_csv(f"signal_df_threshold_{thresh}.csv")

print("✓ All files saved.")


In [ ]:
# @title
# ============================================================
# SIGNAL PORTFOLIO — THREE-STAGE THRESHOLD VERSION
# Signals generated from Modified Determinant Model
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import requests
from io import StringIO

# ── 1. LOAD ALL DATA FROM GITHUB ─────────────────────────────────────────────
GITHUB_BASE = "https://raw.githubusercontent.com/kboroz/MSFE_Capstone_Project/main/01_Data/Streamlined/"

INDEX_FILE_MAP = {
    'ASX50':       'streamlined_asx_50.csv',
    'EUROSTOXX50': 'streamlined_euro_stoxx_50.csv',
    'FTSE100':     'streamlined_ftse_100.csv',
    'Ibovespa':    'streamlined_ibovespa.csv',
    'JSE40':       'streamlined_jse_top_40.csv',
    'NIKKEI225':   'streamlined_nikkei_225.csv',
    'SP500':       'streamlined_s&p_500.csv',
}

MACRO_FILE = 'streamlined_macro.csv'

def load_csv(url):
    r = requests.get(url)
    r.raise_for_status()
    df = pd.read_csv(StringIO(r.text), index_col=0, parse_dates=True)
    df.columns = df.columns.str.replace('[^a-zA-Z0-9]', '_', regex=True)
    return df.sort_index()

# Load macro
macro_df = load_csv(GITHUB_BASE + MACRO_FILE)
macro_df = macro_df[~macro_df.index.duplicated(keep='first')].sort_index()
print(f"macro_df shape : {macro_df.shape}")
print(f"Columns        : {macro_df.columns.tolist()}")

# Load individual index files
index_data = {}
for name, fname in INDEX_FILE_MAP.items():
    try:
        df = load_csv(GITHUB_BASE + fname)
        index_data[name] = df
        print(f"✓ {name:15s} | shape={df.shape} | cols={df.columns.tolist()[:4]}")
    except Exception as e:
        print(f"✗ {name}: {e}")

# ── 2. EXTRACT PRICE SERIES ───────────────────────────────────────────────────
PRICE_COLS = ['Close', 'close', 'Price', 'price', 'Adj_Close', 'Last']

def get_price_series(df, name):
    col = next((c for c in PRICE_COLS if c in df.columns), df.columns[0])
    s   = pd.to_numeric(df[col], errors='coerce').dropna()
    print(f"  {name}: using column '{col}'")
    return s

index_prices = {}
for name, df in index_data.items():
    index_prices[name] = get_price_series(df, name)

# ── 3. MODIFIED DETERMINANT SIGNAL MODEL ─────────────────────────────────────
DET_WINDOWS    = [3, 7, 28]
WINDOW_WEIGHTS = {3: 0.98, 7: 0.01, 28: 0.01}

def compute_modified_determinant(price_series_dict, windows=DET_WINDOWS):
    prices = pd.DataFrame(price_series_dict).sort_index()
    rets   = prices.pct_change().dropna()
    det_results = {}

    for window in windows:
        det_series = []
        for i in range(window, len(rets)):
            window_rets = rets.iloc[i - window: i]
            valid_cols  = window_rets.columns[window_rets.std() > 1e-10]
            w           = window_rets[valid_cols]

            if len(valid_cols) < 2:
                det_series.append(np.nan)
                continue
            try:
                R   = w.corr().values
                I   = np.eye(len(R))
                M   = R - I
                det = np.linalg.det(M)
            except Exception:
                det = np.nan
            det_series.append(det)

        idx = rets.index[window:]
        det_results[window] = pd.Series(det_series, index=idx, name=f'det_{window}d')
        print(f"  Window {window:2d}d | computed {len(det_series)} observations")

    return det_results

print("\n── Computing Modified Determinant ──")
det_results = compute_modified_determinant(index_prices)

# ── 4. COMPOSITE SCORE ────────────────────────────────────────────────────────
def compute_composite_score(det_results, window_weights=WINDOW_WEIGHTS):
    """
    Returns raw weighted-average of normalised determinant values (not signs).
    Normalised per window via rolling z-score so windows are comparable.
    """
    all_series = []
    for window, series in det_results.items():
        # z-score normalise so scale is comparable across windows
        mu    = series.rolling(252, min_periods=60).mean()
        sigma = series.rolling(252, min_periods=60).std()
        z     = (series - mu) / sigma.replace(0, np.nan)
        z.name = f'z_{window}d'
        all_series.append(z)

    z_df      = pd.concat(all_series, axis=1).dropna()
    composite = sum(
        z_df[f'z_{w}d'] * window_weights[w]
        for w in window_weights if f'z_{w}d' in z_df.columns
    )
    composite = composite / sum(window_weights.values())
    return composite, z_df

composite_score, z_df = compute_composite_score(det_results)

print(f"\nComposite score shape  : {composite_score.shape}")
print(f"Date range             : {composite_score.index[0].date()} → "
      f"{composite_score.index[-1].date()}")
print(f"Score range            : {composite_score.min():.3f} → "
      f"{composite_score.max():.3f}")

# ── 5. THREE-STAGE SIGNAL CLASSIFICATION ─────────────────────────────────────
# |score| > threshold  → RISK_ON  (+1): high det magnitude = abnormal regime
#                         → short equity, long country bonds + FX
# |score| <= threshold → RISK_OFF (-1): low det magnitude  = calm regime
#                         → long equity

THRESHOLDS = [0.3, 0.5, 0.7]   # tune here

def classify_signal(score, threshold):
    """
    Returns +1 (RISK_ON / defensive) or -1 (RISK_OFF / long equity)
    """
    if abs(score) > threshold:
        return +1   # high |det| → abnormal → defensive
    else:
        return -1   # low  |det| → calm     → long equity

def build_signal_df(composite_score, equity_assets, macro_dates, threshold=0.5):
    """
    Apply three-stage classification and reindex to portfolio dates.
    """
    binary = composite_score.apply(lambda x: classify_signal(x, threshold))
    signal_df = pd.DataFrame(index=macro_dates)
    for asset in equity_assets:
        signal_df[asset] = binary.reindex(macro_dates).ffill().bfill()
    return signal_df

# ── 6. ASSET UNIVERSE ─────────────────────────────────────────────────────────
EQUITY_ASSETS = list(index_prices.keys())
SAFE_BONDS    = ['US_10Y', 'UK_10Y', 'AU_10Y', 'ZA_10Y']
FX            = ['AUDUSD', 'BRLUSD', 'EURUSD', 'GBPUSD', 'JPYUSD', 'ZARUSD']
COMMODITIES   = ['Oil', 'Gold', 'Bitcoin']

ALL_ASSETS        = EQUITY_ASSETS + FX + COMMODITIES + SAFE_BONDS
ALL_ASSETS        = [a for a in ALL_ASSETS if a in macro_df.columns]
equity_assets     = [a for a in EQUITY_ASSETS if a in ALL_ASSETS]
non_equity_assets = [a for a in ALL_ASSETS if a not in equity_assets]

N           = len(ALL_ASSETS)
base_weight = 1.0 / N

print(f"\nTotal assets              : {N}")
print(f"Equity (signal-driven)    : {equity_assets}")
print(f"Non-equity (always long)  : {non_equity_assets}")
print(f"base_weight = 1/{N} = {base_weight:.4f}")

# ── 7. COUNTRY SAFE-HAVEN MAP ─────────────────────────────────────────────────
# When an equity index is shorted (RISK_ON), proceeds are redistributed
# to that country's bond + FX where available in the universe

EQUITY_SAFE_HAVEN_MAP = {
    'ASX50':       ['AU_10Y', 'AUDUSD'],
    'EUROSTOXX50': ['EURUSD'],            # no EU bond in universe
    'FTSE100':     ['UK_10Y', 'GBPUSD'],
    'Ibovespa':    ['BRLUSD'],            # no Brazil bond in universe
    'JSE40':       ['ZA_10Y', 'ZARUSD'],
    'NIKKEI225':   ['JPYUSD'],            # no JP bond in universe
    'SP500':       ['US_10Y'],            # USD is base currency
}

# Filter to assets actually in universe
UNIVERSE_SET = set(ALL_ASSETS)
for eq, havens in EQUITY_SAFE_HAVEN_MAP.items():
    EQUITY_SAFE_HAVEN_MAP[eq] = [h for h in havens if h in UNIVERSE_SET]

print("\nSafe-haven routing (filtered to universe):")
for eq, havens in EQUITY_SAFE_HAVEN_MAP.items():
    print(f"  {eq:15s} → {havens}")

# ── 8. BUILD RETURNS ──────────────────────────────────────────────────────────
returns_raw = macro_df[ALL_ASSETS].pct_change(fill_method=None)

for col in SAFE_BONDS:
    if col in returns_raw.columns:
        returns_raw[col] = -returns_raw[col]   # invert yield → price

if 'Oil' in macro_df.columns:
    bad_oil = macro_df['Oil'][macro_df['Oil'] <= 0].index
    for d in bad_oil:
        loc = returns_raw.index.get_loc(d)
        returns_raw.iloc[
            max(0, loc - 1): loc + 2,
            returns_raw.columns.get_loc('Oil')
        ] = 0.0

returns = returns_raw.clip(lower=-0.50, upper=1.00).fillna(0).iloc[1:]

assert not returns.isnull().any().any(), "NaNs in returns"
assert not np.isinf(returns.values).any(),  "Inf in returns"
print(f"\nReturns shape : {returns.shape}")
print("✓ Returns clean")

# ── 9. THREE-STAGE WEIGHT BUILDER ────────────────────────────────────────────
def build_weights_3stage(sig_row, equity_assets, non_equity_assets,
                         base_weight, date, safe_haven_map):
    """
    RISK_OFF (-1) : long equity at base_weight
    RISK_ON  (+1) : short equity, redistribute proceeds to country safe havens
                    or equally across all non-equity if no safe haven available
    Non-equity assets always start at base_weight; safe-haven assets receive
    additional weight from shorted equities.
    """
    raw = {a: base_weight for a in non_equity_assets}

    safe_haven_extra = {}   # accumulate extra weight per safe-haven asset

    for a in equity_assets:
        if sig_row is not None and a in sig_row.index and not pd.isna(sig_row[a]):
            sig = sig_row[a]
        else:
            sig = -1   # default: long equity

        if sig > 0:
            # RISK_ON: short equity, redirect base_weight to safe havens
            raw[a]   = -base_weight
            havens   = safe_haven_map.get(a, [])
            havens   = [h for h in havens if h in raw]   # only if in universe

            if havens:
                per_haven = base_weight / len(havens)
                for h in havens:
                    safe_haven_extra[h] = safe_haven_extra.get(h, 0) + per_haven
            else:
                # No safe haven available → spread across all non-equity
                per_ne = base_weight / len(non_equity_assets)
                for ne in non_equity_assets:
                    safe_haven_extra[ne] = safe_haven_extra.get(ne, 0) + per_ne
        else:
            # RISK_OFF: long equity
            raw[a] = base_weight

    # Apply safe-haven extras
    for asset, extra in safe_haven_extra.items():
        raw[asset] = raw.get(asset, 0) + extra

    # Normalise to sum = 1 (handles edge cases)
    total = sum(raw.values())
    if total <= 0:
        for a in raw:
            raw[a] = 1.0 / len(raw)
    else:
        for a in raw:
            raw[a] /= total

    return raw

# ── 10. EQUAL-WEIGHT BENCHMARK ────────────────────────────────────────────────
def run_equal_weight(returns, all_assets, tc_rate=0.005,
                     rebal_freq='monthly', initial_capital=1e9):
    N           = len(all_assets)
    base_weight = 1.0 / N
    ew          = {a: base_weight for a in all_assets}

    freq_code     = 'MS' if rebal_freq == 'monthly' else 'QS'
    reb_dates_raw = returns.resample(freq_code).first().index
    actual_dates  = returns.index

    reb_set = set()
    for d in reb_dates_raw:
        future = actual_dates[actual_dates >= d]
        if len(future) > 0:
            reb_set.add(future[0])

    pv              = float(initial_capital)
    current_weights = None
    records         = []

    for date in actual_dates:
        if date in reb_set:
            if current_weights is not None:
                turnover = sum(abs(ew.get(a, 0) - current_weights.get(a, 0))
                               for a in set(ew) | set(current_weights)) / 2.0
                pv *= (1 - turnover * tc_rate)
            else:
                pv *= (1 - tc_rate)
            current_weights = ew.copy()

        if current_weights is not None:
            dr  = sum(current_weights.get(a, 0) * returns.loc[date, a]
                      for a in all_assets)
            pv *= (1 + dr)

        records.append({'date': date, 'portfolio_value': pv})

    return pd.DataFrame(records).set_index('date')

# ── 11. THREE-STAGE SIGNAL PORTFOLIO RUNNER ───────────────────────────────────
def run_signal_portfolio_3stage(returns, signal_df, equity_assets,
                                non_equity_assets, all_assets,
                                freq='monthly', tc_rate=0.005,
                                safe_haven_map=None,
                                initial_capital=1e9):

    if safe_haven_map is None:
        safe_haven_map = {}

    N           = len(all_assets)
    base_weight = 1.0 / N

    freq_code     = 'MS' if freq == 'monthly' else 'QS'
    reb_dates_raw = returns.resample(freq_code).first().index
    actual_dates  = returns.index

    reb_set = set()
    for d in reb_dates_raw:
        future = actual_dates[actual_dates >= d]
        if len(future) > 0:
            reb_set.add(future[0])

    sig = signal_df.copy()
    sig.index = pd.to_datetime(sig.index)

    pv              = float(initial_capital)
    current_weights = None
    records         = []

    for date in actual_dates:
        if date in reb_set:
            past    = sig[sig.index <= date]
            sig_row = past.iloc[-1] if len(past) > 0 else None

            raw = build_weights_3stage(
                sig_row, equity_assets, non_equity_assets,
                base_weight, date, safe_haven_map
            )

            if current_weights is not None:
                all_keys = set(raw) | set(current_weights)
                turnover = sum(
                    abs(raw.get(a, 0) - current_weights.get(a, 0))
                    for a in all_keys
                ) / 2.0
                pv *= (1 - turnover * tc_rate)
            else:
                pv *= (1 - tc_rate)

            current_weights = raw.copy()

        if current_weights is not None:
            dr  = sum(current_weights.get(a, 0) * returns.loc[date, a]
                      for a in all_assets)
            pv *= (1 + dr)

        records.append({'date': date, 'portfolio_value': pv})

    return pd.DataFrame(records).set_index('date')

# ── 12. RUN ALL PORTFOLIOS ────────────────────────────────────────────────────
TC_LEVELS = {'0%': 0.00, '0.5%': 0.005, '1%': 0.01, '2%': 0.02}
FREQS     = ['monthly', 'quarterly']

# Equal-weight benchmarks
ew_results = {}
for freq in FREQS:
    for tc_label, tc_rate in TC_LEVELS.items():
        label = f"EW_{freq}_TC{tc_label}"
        ew_results[label] = run_equal_weight(
            returns, ALL_ASSETS, tc_rate=tc_rate,
            rebal_freq=freq, initial_capital=1e9
        )
        print(f"✓ {label}")

# Three-stage signal portfolios across all thresholds
signal_results = {}
signal_distributions = {}

for thresh in THRESHOLDS:
    sig_df_t = build_signal_df(composite_score, equity_assets,
                               macro_df.index, threshold=thresh)

    # Print signal distribution for this threshold
    n_risk_on  = (sig_df_t[equity_assets[0]] == +1).sum()
    n_risk_off = (sig_df_t[equity_assets[0]] == -1).sum()
    total      = len(sig_df_t)
    print(f"\nThreshold={thresh} | RISK_ON(+1): {n_risk_on} days "
          f"({n_risk_on/total*100:.1f}%) | "
          f"RISK_OFF(-1): {n_risk_off} days ({n_risk_off/total*100:.1f}%)")

    signal_distributions[thresh] = {'RISK_ON': n_risk_on, 'RISK_OFF': n_risk_off}

    for freq in FREQS:
        for tc_label, tc_rate in TC_LEVELS.items():
            label = f"Signal3S_t{thresh}_{freq}_TC{tc_label}"
            signal_results[label] = run_signal_portfolio_3stage(
                returns, sig_df_t, equity_assets, non_equity_assets,
                ALL_ASSETS, freq=freq, tc_rate=tc_rate,
                safe_haven_map=EQUITY_SAFE_HAVEN_MAP,
                initial_capital=1e9
            )
            print(f"  ✓ {label}")

print("\n✓ All portfolios computed")

# ── 13. PERFORMANCE METRICS ───────────────────────────────────────────────────
def compute_metrics(pv_df, label=''):
    pv        = pv_df['portfolio_value']
    daily_ret = pv.pct_change().dropna()
    n_years   = len(daily_ret) / 252
    total_ret = (pv.iloc[-1] / pv.iloc[0]) - 1
    cagr      = (1 + total_ret) ** (1 / n_years) - 1 if n_years > 0 else np.nan
    vol       = daily_ret.std() * np.sqrt(252)
    sharpe    = cagr / vol if vol > 0 else np.nan
    roll_max  = pv.cummax()
    max_dd    = ((pv - roll_max) / roll_max).min()
    calmar    = cagr / abs(max_dd) if max_dd != 0 else np.nan
    var_95    = daily_ret.quantile(0.05)

    return {
        'CAGR':         f"{cagr*100:.2f}%",
        'Volatility':   f"{vol*100:.2f}%",
        'Sharpe':       f"{sharpe:.2f}",
        'Max Drawdown': f"{max_dd*100:.2f}%",
        'Calmar':       f"{calmar:.2f}",
        'VaR 95%':      f"{var_95*100:.2f}%",
        'Total Return': f"{total_ret*100:.2f}%",
        'Final ($B)':   f"{pv.iloc[-1]/1e9:.3f}",
    }

# Print results grouped by threshold
ew_metrics = {k: compute_metrics(v, k) for k, v in ew_results.items()}

print("\n" + "="*90)
print("EQUAL-WEIGHT BENCHMARK")
print("="*90)
print(pd.DataFrame(ew_metrics).T.to_string())

sig_metrics = {k: compute_metrics(v, k) for k, v in signal_results.items()}

for thresh in THRESHOLDS:
    sub = {k: v for k, v in sig_metrics.items() if f"_t{thresh}_" in k}
    print(f"\n{'='*90}")
    print(f"THREE-STAGE SIGNAL — Threshold = {thresh} | "
          f"RISK_ON: {signal_distributions[thresh]['RISK_ON']} days | "
          f"RISK_OFF: {signal_distributions[thresh]['RISK_OFF']} days")
    print("="*90)
    print(pd.DataFrame(sub).T.to_string())

# ── 14. PLOTS ─────────────────────────────────────────────────────────────────
# One figure per threshold + one overview figure

colors_tc   = {'0%': 'black', '0.5%': 'steelblue', '1%': 'darkorange', '2%': 'crimson'}
colors_freq = {'monthly': 'steelblue', 'quarterly': 'darkorange'}

# ── Figure 1: Composite signal + safe-haven routing overview ──────────────────
fig, axes = plt.subplots(2, 1, figsize=(16, 10))
fig.suptitle("Modified Determinant — Composite Score & Threshold Signals",
             fontsize=14, fontweight='bold')

ax = axes[0]
ax.plot(composite_score.index, composite_score.values,
        color='navy', lw=1.0, alpha=0.8, label='Composite Z-score')
for thresh, col in zip(THRESHOLDS, ['green', 'orange', 'red']):
    ax.axhline( thresh, color=col, ls='--', lw=1.0, label=f'+{thresh}')
    ax.axhline(-thresh, color=col, ls='--', lw=1.0, label=f'-{thresh}')
ax.axhline(0, color='black', lw=0.6)
ax.fill_between(composite_score.index, composite_score.values, 0,
                where=(composite_score > 0), color='green', alpha=0.1)
ax.fill_between(composite_score.index, composite_score.values, 0,
                where=(composite_score <= 0), color='red', alpha=0.1)
ax.set_title("Composite Determinant Score with Thresholds", fontsize=11)
ax.set_ylabel("Z-score")
ax.legend(fontsize=8, ncol=4)
ax.grid(True, alpha=0.3)

ax = axes[1]
for window, series in det_results.items():
    ax.plot(series.index, np.sign(series.values), alpha=0.6,
            label=f"{window}d window", lw=1.0)
ax.axhline(0, color='black', lw=0.8)
ax.set_title("Sign of Det(R-I) per Window", fontsize=11)
ax.set_ylabel("+1 / -1")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("composite_signal_thresholds.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Figure 2: Portfolio values per threshold (TC=0.5%, monthly) ───────────────
fig, axes = plt.subplots(1, 3, figsize=(20, 6), sharey=False)
fig.suptitle("Three-Stage Signal Portfolio — Value by Threshold (TC=0.5%, Monthly)",
             fontsize=13, fontweight='bold')

for ax, thresh in zip(axes, THRESHOLDS):
    # EW reference
    ew_pv = ew_results['EW_monthly_TC0.5%']['portfolio_value']
    ax.plot(ew_pv.index, ew_pv / 1e9, '--', color='grey',
            lw=1.5, label='EW Benchmark')

    label = f"Signal3S_t{thresh}_monthly_TC0.5%"
    pv    = signal_results[label]['portfolio_value']
    ax.plot(pv.index, pv / 1e9, color='steelblue', lw=1.8,
            label=f'Signal (t={thresh})')

    ax.axhline(1.0, color='black', ls=':', lw=0.8)
    ax.set_title(f"Threshold = {thresh}", fontsize=11)
    ax.set_ylabel("Value ($B)")
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("signal_3stage_by_threshold.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Figure 3: Drawdowns per threshold ────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(20, 5), sharey=True)
fig.suptitle("Three-Stage Signal Portfolio — Drawdowns by Threshold (TC=0.5%, Monthly)",
             fontsize=13, fontweight='bold')

for ax, thresh in zip(axes, THRESHOLDS):
    label  = f"Signal3S_t{thresh}_monthly_TC0.5%"
    pv     = signal_results[label]['portfolio_value']
    dd     = (pv - pv.cummax()) / pv.cummax() * 100
    ew_pv  = ew_results['EW_monthly_TC0.5%']['portfolio_value']
    ew_dd  = (ew_pv - ew_pv.cummax()) / ew_pv.cummax() * 100

    ax.fill_between(dd.index,  dd.values,  0, alpha=0.3, color='steelblue')
    ax.fill_between(ew_dd.index, ew_dd.values, 0, alpha=0.2, color='grey')
    ax.plot(dd.index,    dd.values,    color='steelblue', lw=1.0,
            label=f'Signal (t={thresh})')
    ax.plot(ew_dd.index, ew_dd.values, color='grey',      lw=1.0, ls='--',
            label='EW Benchmark')
    ax.set_title(f"Threshold = {thresh}", fontsize=11)
    ax.set_ylabel("Drawdown (%)")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("signal_3stage_drawdowns.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Figure 4: TC sensitivity per threshold ────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.suptitle("Three-Stage Signal — TC Sensitivity (Monthly)",
             fontsize=13, fontweight='bold')

for ax, thresh in zip(axes, THRESHOLDS):
    for tc_label, col in colors_tc.items():
        label = f"Signal3S_t{thresh}_monthly_TC{tc_label}"
        pv    = signal_results[label]['portfolio_value']
        ax.plot(pv.index, pv / 1e9, color=col, lw=1.3,
                label=f'TC={tc_label}')
    ew_pv = ew_results['EW_monthly_TC0.5%']['portfolio_value']
    ax.plot(ew_pv.index, ew_pv / 1e9, '--', color='grey',
            lw=1.5, label='EW Benchmark')
    ax.set_title(f"Threshold = {thresh}", fontsize=11)
    ax.set_ylabel("Value ($B)")
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("signal_3stage_tc_sensitivity.png", dpi=150, bbox_inches='tight')
plt.show()

# ── 15. SAVE ──────────────────────────────────────────────────────────────────
for label, df in {**ew_results, **signal_results}.items():
    df.to_csv(f"{label}_portfolio.csv")

all_metrics = pd.DataFrame({**ew_metrics, **sig_metrics}).T
all_metrics.to_csv("all_portfolio_metrics.csv")

composite_score.to_csv("composite_signal.csv")

for thresh in THRESHOLDS:
    sig_df_t = build_signal_df(composite_score, equity_assets,
                               macro_df.index, threshold=thresh)
    sig_df_t.to_csv(f"signal_df_threshold_{thresh}.csv")

print("✓ All files saved.")


In [ ]:
# @title
# ============================================================
# SIGNAL PORTFOLIO — THREE-STAGE THRESHOLD VERSION
# Signals generated from Modified Determinant Model
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import requests
from io import StringIO

# ── 1. LOAD ALL DATA FROM GITHUB ─────────────────────────────────────────────
GITHUB_BASE = "https://raw.githubusercontent.com/kboroz/MSFE_Capstone_Project/main/01_Data/Streamlined/"

INDEX_FILE_MAP = {
    'ASX50':       'streamlined_asx_50.csv',
    'EUROSTOXX50': 'streamlined_euro_stoxx_50.csv',
    'FTSE100':     'streamlined_ftse_100.csv',
    'Ibovespa':    'streamlined_ibovespa.csv',
    'JSE40':       'streamlined_jse_top_40.csv',
    'NIKKEI225':   'streamlined_nikkei_225.csv',
    'SP500':       'streamlined_s&p_500.csv',
}

MACRO_FILE = 'streamlined_macro.csv'

def load_csv(url):
    r = requests.get(url)
    r.raise_for_status()
    df = pd.read_csv(StringIO(r.text), index_col=0, parse_dates=True)
    df.columns = df.columns.str.replace('[^a-zA-Z0-9]', '_', regex=True)
    return df.sort_index()

# Load macro
macro_df = load_csv(GITHUB_BASE + MACRO_FILE)
macro_df = macro_df[~macro_df.index.duplicated(keep='first')].sort_index()
print(f"macro_df shape : {macro_df.shape}")
print(f"Columns        : {macro_df.columns.tolist()}")

# Load individual index files
index_data = {}
for name, fname in INDEX_FILE_MAP.items():
    try:
        df = load_csv(GITHUB_BASE + fname)
        index_data[name] = df
        print(f"✓ {name:15s} | shape={df.shape} | cols={df.columns.tolist()[:4]}")
    except Exception as e:
        print(f"✗ {name}: {e}")

# ── 2. EXTRACT PRICE SERIES ───────────────────────────────────────────────────
PRICE_COLS = ['Close', 'close', 'Price', 'price', 'Adj_Close', 'Last']

def get_price_series(df, name):
    col = next((c for c in PRICE_COLS if c in df.columns), df.columns[0])
    s   = pd.to_numeric(df[col], errors='coerce').dropna()
    print(f"  {name}: using column '{col}'")
    return s

index_prices = {}
for name, df in index_data.items():
    index_prices[name] = get_price_series(df, name)

# ── 3. MODIFIED DETERMINANT SIGNAL MODEL ─────────────────────────────────────
DET_WINDOWS    = [3, 7, 28]
WINDOW_WEIGHTS = {3: 0.01, 7: 0.98, 28: 0.01}

def compute_modified_determinant(price_series_dict, windows=DET_WINDOWS):
    prices = pd.DataFrame(price_series_dict).sort_index()
    rets   = prices.pct_change().dropna()
    det_results = {}

    for window in windows:
        det_series = []
        for i in range(window, len(rets)):
            window_rets = rets.iloc[i - window: i]
            valid_cols  = window_rets.columns[window_rets.std() > 1e-10]
            w           = window_rets[valid_cols]

            if len(valid_cols) < 2:
                det_series.append(np.nan)
                continue
            try:
                R   = w.corr().values
                I   = np.eye(len(R))
                M   = R - I
                det = np.linalg.det(M)
            except Exception:
                det = np.nan
            det_series.append(det)

        idx = rets.index[window:]
        det_results[window] = pd.Series(det_series, index=idx, name=f'det_{window}d')
        print(f"  Window {window:2d}d | computed {len(det_series)} observations")

    return det_results

print("\n── Computing Modified Determinant ──")
det_results = compute_modified_determinant(index_prices)

# ── 4. COMPOSITE SCORE ────────────────────────────────────────────────────────
def compute_composite_score(det_results, window_weights=WINDOW_WEIGHTS):
    """
    Returns raw weighted-average of normalised determinant values (not signs).
    Normalised per window via rolling z-score so windows are comparable.
    """
    all_series = []
    for window, series in det_results.items():
        # z-score normalise so scale is comparable across windows
        mu    = series.rolling(252, min_periods=60).mean()
        sigma = series.rolling(252, min_periods=60).std()
        z     = (series - mu) / sigma.replace(0, np.nan)
        z.name = f'z_{window}d'
        all_series.append(z)

    z_df      = pd.concat(all_series, axis=1).dropna()
    composite = sum(
        z_df[f'z_{w}d'] * window_weights[w]
        for w in window_weights if f'z_{w}d' in z_df.columns
    )
    composite = composite / sum(window_weights.values())
    return composite, z_df

composite_score, z_df = compute_composite_score(det_results)

print(f"\nComposite score shape  : {composite_score.shape}")
print(f"Date range             : {composite_score.index[0].date()} → "
      f"{composite_score.index[-1].date()}")
print(f"Score range            : {composite_score.min():.3f} → "
      f"{composite_score.max():.3f}")

# ── 5. THREE-STAGE SIGNAL CLASSIFICATION ─────────────────────────────────────
# |score| > threshold  → RISK_ON  (+1): high det magnitude = abnormal regime
#                         → short equity, long country bonds + FX
# |score| <= threshold → RISK_OFF (-1): low det magnitude  = calm regime
#                         → long equity

THRESHOLDS = [0.3, 0.5, 0.7]   # tune here

def classify_signal(score, threshold):
    """
    Returns +1 (RISK_ON / defensive) or -1 (RISK_OFF / long equity)
    """
    if abs(score) > threshold:
        return +1   # high |det| → abnormal → defensive
    else:
        return -1   # low  |det| → calm     → long equity

def build_signal_df(composite_score, equity_assets, macro_dates, threshold=0.5):
    """
    Apply three-stage classification and reindex to portfolio dates.
    """
    binary = composite_score.apply(lambda x: classify_signal(x, threshold))
    signal_df = pd.DataFrame(index=macro_dates)
    for asset in equity_assets:
        signal_df[asset] = binary.reindex(macro_dates).ffill().bfill()
    return signal_df

# ── 6. ASSET UNIVERSE ─────────────────────────────────────────────────────────
EQUITY_ASSETS = list(index_prices.keys())
SAFE_BONDS    = ['US_10Y', 'UK_10Y', 'AU_10Y', 'ZA_10Y']
FX            = ['AUDUSD', 'BRLUSD', 'EURUSD', 'GBPUSD', 'JPYUSD', 'ZARUSD']
COMMODITIES   = ['Oil', 'Gold', 'Bitcoin']

ALL_ASSETS        = EQUITY_ASSETS + FX + COMMODITIES + SAFE_BONDS
ALL_ASSETS        = [a for a in ALL_ASSETS if a in macro_df.columns]
equity_assets     = [a for a in EQUITY_ASSETS if a in ALL_ASSETS]
non_equity_assets = [a for a in ALL_ASSETS if a not in equity_assets]

N           = len(ALL_ASSETS)
base_weight = 1.0 / N

print(f"\nTotal assets              : {N}")
print(f"Equity (signal-driven)    : {equity_assets}")
print(f"Non-equity (always long)  : {non_equity_assets}")
print(f"base_weight = 1/{N} = {base_weight:.4f}")

# ── 7. COUNTRY SAFE-HAVEN MAP ─────────────────────────────────────────────────
# When an equity index is shorted (RISK_ON), proceeds are redistributed
# to that country's bond + FX where available in the universe

EQUITY_SAFE_HAVEN_MAP = {
    'ASX50':       ['AU_10Y', 'AUDUSD'],
    'EUROSTOXX50': ['EURUSD'],            # no EU bond in universe
    'FTSE100':     ['UK_10Y', 'GBPUSD'],
    'Ibovespa':    ['BRLUSD'],            # no Brazil bond in universe
    'JSE40':       ['ZA_10Y', 'ZARUSD'],
    'NIKKEI225':   ['JPYUSD'],            # no JP bond in universe
    'SP500':       ['US_10Y'],            # USD is base currency
}

# Filter to assets actually in universe
UNIVERSE_SET = set(ALL_ASSETS)
for eq, havens in EQUITY_SAFE_HAVEN_MAP.items():
    EQUITY_SAFE_HAVEN_MAP[eq] = [h for h in havens if h in UNIVERSE_SET]

print("\nSafe-haven routing (filtered to universe):")
for eq, havens in EQUITY_SAFE_HAVEN_MAP.items():
    print(f"  {eq:15s} → {havens}")

# ── 8. BUILD RETURNS ──────────────────────────────────────────────────────────
returns_raw = macro_df[ALL_ASSETS].pct_change(fill_method=None)

for col in SAFE_BONDS:
    if col in returns_raw.columns:
        returns_raw[col] = -returns_raw[col]   # invert yield → price

if 'Oil' in macro_df.columns:
    bad_oil = macro_df['Oil'][macro_df['Oil'] <= 0].index
    for d in bad_oil:
        loc = returns_raw.index.get_loc(d)
        returns_raw.iloc[
            max(0, loc - 1): loc + 2,
            returns_raw.columns.get_loc('Oil')
        ] = 0.0

returns = returns_raw.clip(lower=-0.50, upper=1.00).fillna(0).iloc[1:]

assert not returns.isnull().any().any(), "NaNs in returns"
assert not np.isinf(returns.values).any(),  "Inf in returns"
print(f"\nReturns shape : {returns.shape}")
print("✓ Returns clean")

# ── 9. THREE-STAGE WEIGHT BUILDER ────────────────────────────────────────────
def build_weights_3stage(sig_row, equity_assets, non_equity_assets,
                         base_weight, date, safe_haven_map):
    """
    RISK_OFF (-1) : long equity at base_weight
    RISK_ON  (+1) : short equity, redistribute proceeds to country safe havens
                    or equally across all non-equity if no safe haven available
    Non-equity assets always start at base_weight; safe-haven assets receive
    additional weight from shorted equities.
    """
    raw = {a: base_weight for a in non_equity_assets}

    safe_haven_extra = {}   # accumulate extra weight per safe-haven asset

    for a in equity_assets:
        if sig_row is not None and a in sig_row.index and not pd.isna(sig_row[a]):
            sig = sig_row[a]
        else:
            sig = -1   # default: long equity

        if sig > 0:
            # RISK_ON: short equity, redirect base_weight to safe havens
            raw[a]   = -base_weight
            havens   = safe_haven_map.get(a, [])
            havens   = [h for h in havens if h in raw]   # only if in universe

            if havens:
                per_haven = base_weight / len(havens)
                for h in havens:
                    safe_haven_extra[h] = safe_haven_extra.get(h, 0) + per_haven
            else:
                # No safe haven available → spread across all non-equity
                per_ne = base_weight / len(non_equity_assets)
                for ne in non_equity_assets:
                    safe_haven_extra[ne] = safe_haven_extra.get(ne, 0) + per_ne
        else:
            # RISK_OFF: long equity
            raw[a] = base_weight

    # Apply safe-haven extras
    for asset, extra in safe_haven_extra.items():
        raw[asset] = raw.get(asset, 0) + extra

    # Normalise to sum = 1 (handles edge cases)
    total = sum(raw.values())
    if total <= 0:
        for a in raw:
            raw[a] = 1.0 / len(raw)
    else:
        for a in raw:
            raw[a] /= total

    return raw

# ── 10. EQUAL-WEIGHT BENCHMARK ────────────────────────────────────────────────
def run_equal_weight(returns, all_assets, tc_rate=0.005,
                     rebal_freq='monthly', initial_capital=1e9):
    N           = len(all_assets)
    base_weight = 1.0 / N
    ew          = {a: base_weight for a in all_assets}

    freq_code     = 'MS' if rebal_freq == 'monthly' else 'QS'
    reb_dates_raw = returns.resample(freq_code).first().index
    actual_dates  = returns.index

    reb_set = set()
    for d in reb_dates_raw:
        future = actual_dates[actual_dates >= d]
        if len(future) > 0:
            reb_set.add(future[0])

    pv              = float(initial_capital)
    current_weights = None
    records         = []

    for date in actual_dates:
        if date in reb_set:
            if current_weights is not None:
                turnover = sum(abs(ew.get(a, 0) - current_weights.get(a, 0))
                               for a in set(ew) | set(current_weights)) / 2.0
                pv *= (1 - turnover * tc_rate)
            else:
                pv *= (1 - tc_rate)
            current_weights = ew.copy()

        if current_weights is not None:
            dr  = sum(current_weights.get(a, 0) * returns.loc[date, a]
                      for a in all_assets)
            pv *= (1 + dr)

        records.append({'date': date, 'portfolio_value': pv})

    return pd.DataFrame(records).set_index('date')

# ── 11. THREE-STAGE SIGNAL PORTFOLIO RUNNER ───────────────────────────────────
def run_signal_portfolio_3stage(returns, signal_df, equity_assets,
                                non_equity_assets, all_assets,
                                freq='monthly', tc_rate=0.005,
                                safe_haven_map=None,
                                initial_capital=1e9):

    if safe_haven_map is None:
        safe_haven_map = {}

    N           = len(all_assets)
    base_weight = 1.0 / N

    freq_code     = 'MS' if freq == 'monthly' else 'QS'
    reb_dates_raw = returns.resample(freq_code).first().index
    actual_dates  = returns.index

    reb_set = set()
    for d in reb_dates_raw:
        future = actual_dates[actual_dates >= d]
        if len(future) > 0:
            reb_set.add(future[0])

    sig = signal_df.copy()
    sig.index = pd.to_datetime(sig.index)

    pv              = float(initial_capital)
    current_weights = None
    records         = []

    for date in actual_dates:
        if date in reb_set:
            past    = sig[sig.index <= date]
            sig_row = past.iloc[-1] if len(past) > 0 else None

            raw = build_weights_3stage(
                sig_row, equity_assets, non_equity_assets,
                base_weight, date, safe_haven_map
            )

            if current_weights is not None:
                all_keys = set(raw) | set(current_weights)
                turnover = sum(
                    abs(raw.get(a, 0) - current_weights.get(a, 0))
                    for a in all_keys
                ) / 2.0
                pv *= (1 - turnover * tc_rate)
            else:
                pv *= (1 - tc_rate)

            current_weights = raw.copy()

        if current_weights is not None:
            dr  = sum(current_weights.get(a, 0) * returns.loc[date, a]
                      for a in all_assets)
            pv *= (1 + dr)

        records.append({'date': date, 'portfolio_value': pv})

    return pd.DataFrame(records).set_index('date')

# ── 12. RUN ALL PORTFOLIOS ────────────────────────────────────────────────────
TC_LEVELS = {'0%': 0.00, '0.5%': 0.005, '1%': 0.01, '2%': 0.02}
FREQS     = ['monthly', 'quarterly']

# Equal-weight benchmarks
ew_results = {}
for freq in FREQS:
    for tc_label, tc_rate in TC_LEVELS.items():
        label = f"EW_{freq}_TC{tc_label}"
        ew_results[label] = run_equal_weight(
            returns, ALL_ASSETS, tc_rate=tc_rate,
            rebal_freq=freq, initial_capital=1e9
        )
        print(f"✓ {label}")

# Three-stage signal portfolios across all thresholds
signal_results = {}
signal_distributions = {}

for thresh in THRESHOLDS:
    sig_df_t = build_signal_df(composite_score, equity_assets,
                               macro_df.index, threshold=thresh)

    # Print signal distribution for this threshold
    n_risk_on  = (sig_df_t[equity_assets[0]] == +1).sum()
    n_risk_off = (sig_df_t[equity_assets[0]] == -1).sum()
    total      = len(sig_df_t)
    print(f"\nThreshold={thresh} | RISK_ON(+1): {n_risk_on} days "
          f"({n_risk_on/total*100:.1f}%) | "
          f"RISK_OFF(-1): {n_risk_off} days ({n_risk_off/total*100:.1f}%)")

    signal_distributions[thresh] = {'RISK_ON': n_risk_on, 'RISK_OFF': n_risk_off}

    for freq in FREQS:
        for tc_label, tc_rate in TC_LEVELS.items():
            label = f"Signal3S_t{thresh}_{freq}_TC{tc_label}"
            signal_results[label] = run_signal_portfolio_3stage(
                returns, sig_df_t, equity_assets, non_equity_assets,
                ALL_ASSETS, freq=freq, tc_rate=tc_rate,
                safe_haven_map=EQUITY_SAFE_HAVEN_MAP,
                initial_capital=1e9
            )
            print(f"  ✓ {label}")

print("\n✓ All portfolios computed")

# ── 13. PERFORMANCE METRICS ───────────────────────────────────────────────────
def compute_metrics(pv_df, label=''):
    pv        = pv_df['portfolio_value']
    daily_ret = pv.pct_change().dropna()
    n_years   = len(daily_ret) / 252
    total_ret = (pv.iloc[-1] / pv.iloc[0]) - 1
    cagr      = (1 + total_ret) ** (1 / n_years) - 1 if n_years > 0 else np.nan
    vol       = daily_ret.std() * np.sqrt(252)
    sharpe    = cagr / vol if vol > 0 else np.nan
    roll_max  = pv.cummax()
    max_dd    = ((pv - roll_max) / roll_max).min()
    calmar    = cagr / abs(max_dd) if max_dd != 0 else np.nan
    var_95    = daily_ret.quantile(0.05)

    return {
        'CAGR':         f"{cagr*100:.2f}%",
        'Volatility':   f"{vol*100:.2f}%",
        'Sharpe':       f"{sharpe:.2f}",
        'Max Drawdown': f"{max_dd*100:.2f}%",
        'Calmar':       f"{calmar:.2f}",
        'VaR 95%':      f"{var_95*100:.2f}%",
        'Total Return': f"{total_ret*100:.2f}%",
        'Final ($B)':   f"{pv.iloc[-1]/1e9:.3f}",
    }

# Print results grouped by threshold
ew_metrics = {k: compute_metrics(v, k) for k, v in ew_results.items()}

print("\n" + "="*90)
print("EQUAL-WEIGHT BENCHMARK")
print("="*90)
print(pd.DataFrame(ew_metrics).T.to_string())

sig_metrics = {k: compute_metrics(v, k) for k, v in signal_results.items()}

for thresh in THRESHOLDS:
    sub = {k: v for k, v in sig_metrics.items() if f"_t{thresh}_" in k}
    print(f"\n{'='*90}")
    print(f"THREE-STAGE SIGNAL — Threshold = {thresh} | "
          f"RISK_ON: {signal_distributions[thresh]['RISK_ON']} days | "
          f"RISK_OFF: {signal_distributions[thresh]['RISK_OFF']} days")
    print("="*90)
    print(pd.DataFrame(sub).T.to_string())

# ── 14. PLOTS ─────────────────────────────────────────────────────────────────
# One figure per threshold + one overview figure

colors_tc   = {'0%': 'black', '0.5%': 'steelblue', '1%': 'darkorange', '2%': 'crimson'}
colors_freq = {'monthly': 'steelblue', 'quarterly': 'darkorange'}

# ── Figure 1: Composite signal + safe-haven routing overview ──────────────────
fig, axes = plt.subplots(2, 1, figsize=(16, 10))
fig.suptitle("Modified Determinant — Composite Score & Threshold Signals",
             fontsize=14, fontweight='bold')

ax = axes[0]
ax.plot(composite_score.index, composite_score.values,
        color='navy', lw=1.0, alpha=0.8, label='Composite Z-score')
for thresh, col in zip(THRESHOLDS, ['green', 'orange', 'red']):
    ax.axhline( thresh, color=col, ls='--', lw=1.0, label=f'+{thresh}')
    ax.axhline(-thresh, color=col, ls='--', lw=1.0, label=f'-{thresh}')
ax.axhline(0, color='black', lw=0.6)
ax.fill_between(composite_score.index, composite_score.values, 0,
                where=(composite_score > 0), color='green', alpha=0.1)
ax.fill_between(composite_score.index, composite_score.values, 0,
                where=(composite_score <= 0), color='red', alpha=0.1)
ax.set_title("Composite Determinant Score with Thresholds", fontsize=11)
ax.set_ylabel("Z-score")
ax.legend(fontsize=8, ncol=4)
ax.grid(True, alpha=0.3)

ax = axes[1]
for window, series in det_results.items():
    ax.plot(series.index, np.sign(series.values), alpha=0.6,
            label=f"{window}d window", lw=1.0)
ax.axhline(0, color='black', lw=0.8)
ax.set_title("Sign of Det(R-I) per Window", fontsize=11)
ax.set_ylabel("+1 / -1")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("composite_signal_thresholds.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Figure 2: Portfolio values per threshold (TC=0.5%, monthly) ───────────────
fig, axes = plt.subplots(1, 3, figsize=(20, 6), sharey=False)
fig.suptitle("Three-Stage Signal Portfolio — Value by Threshold (TC=0.5%, Monthly)",
             fontsize=13, fontweight='bold')

for ax, thresh in zip(axes, THRESHOLDS):
    # EW reference
    ew_pv = ew_results['EW_monthly_TC0.5%']['portfolio_value']
    ax.plot(ew_pv.index, ew_pv / 1e9, '--', color='grey',
            lw=1.5, label='EW Benchmark')

    label = f"Signal3S_t{thresh}_monthly_TC0.5%"
    pv    = signal_results[label]['portfolio_value']
    ax.plot(pv.index, pv / 1e9, color='steelblue', lw=1.8,
            label=f'Signal (t={thresh})')

    ax.axhline(1.0, color='black', ls=':', lw=0.8)
    ax.set_title(f"Threshold = {thresh}", fontsize=11)
    ax.set_ylabel("Value ($B)")
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("signal_3stage_by_threshold.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Figure 3: Drawdowns per threshold ────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(20, 5), sharey=True)
fig.suptitle("Three-Stage Signal Portfolio — Drawdowns by Threshold (TC=0.5%, Monthly)",
             fontsize=13, fontweight='bold')

for ax, thresh in zip(axes, THRESHOLDS):
    label  = f"Signal3S_t{thresh}_monthly_TC0.5%"
    pv     = signal_results[label]['portfolio_value']
    dd     = (pv - pv.cummax()) / pv.cummax() * 100
    ew_pv  = ew_results['EW_monthly_TC0.5%']['portfolio_value']
    ew_dd  = (ew_pv - ew_pv.cummax()) / ew_pv.cummax() * 100

    ax.fill_between(dd.index,  dd.values,  0, alpha=0.3, color='steelblue')
    ax.fill_between(ew_dd.index, ew_dd.values, 0, alpha=0.2, color='grey')
    ax.plot(dd.index,    dd.values,    color='steelblue', lw=1.0,
            label=f'Signal (t={thresh})')
    ax.plot(ew_dd.index, ew_dd.values, color='grey',      lw=1.0, ls='--',
            label='EW Benchmark')
    ax.set_title(f"Threshold = {thresh}", fontsize=11)
    ax.set_ylabel("Drawdown (%)")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("signal_3stage_drawdowns.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Figure 4: TC sensitivity per threshold ────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.suptitle("Three-Stage Signal — TC Sensitivity (Monthly)",
             fontsize=13, fontweight='bold')

for ax, thresh in zip(axes, THRESHOLDS):
    for tc_label, col in colors_tc.items():
        label = f"Signal3S_t{thresh}_monthly_TC{tc_label}"
        pv    = signal_results[label]['portfolio_value']
        ax.plot(pv.index, pv / 1e9, color=col, lw=1.3,
                label=f'TC={tc_label}')
    ew_pv = ew_results['EW_monthly_TC0.5%']['portfolio_value']
    ax.plot(ew_pv.index, ew_pv / 1e9, '--', color='grey',
            lw=1.5, label='EW Benchmark')
    ax.set_title(f"Threshold = {thresh}", fontsize=11)
    ax.set_ylabel("Value ($B)")
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("signal_3stage_tc_sensitivity.png", dpi=150, bbox_inches='tight')
plt.show()

# ── 15. SAVE ──────────────────────────────────────────────────────────────────
for label, df in {**ew_results, **signal_results}.items():
    df.to_csv(f"{label}_portfolio.csv")

all_metrics = pd.DataFrame({**ew_metrics, **sig_metrics}).T
all_metrics.to_csv("all_portfolio_metrics.csv")

composite_score.to_csv("composite_signal.csv")

for thresh in THRESHOLDS:
    sig_df_t = build_signal_df(composite_score, equity_assets,
                               macro_df.index, threshold=thresh)
    sig_df_t.to_csv(f"signal_df_threshold_{thresh}.csv")

print("✓ All files saved.")


In [ ]:
# @title
# ============================================================
# SIGNAL PORTFOLIO — THREE-STAGE THRESHOLD VERSION
# Signals generated from Modified Determinant Model
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import requests
from io import StringIO

# ── 1. LOAD ALL DATA FROM GITHUB ─────────────────────────────────────────────
GITHUB_BASE = "https://raw.githubusercontent.com/kboroz/MSFE_Capstone_Project/main/01_Data/Streamlined/"

INDEX_FILE_MAP = {
    'ASX50':       'streamlined_asx_50.csv',
    'EUROSTOXX50': 'streamlined_euro_stoxx_50.csv',
    'FTSE100':     'streamlined_ftse_100.csv',
    'Ibovespa':    'streamlined_ibovespa.csv',
    'JSE40':       'streamlined_jse_top_40.csv',
    'NIKKEI225':   'streamlined_nikkei_225.csv',
    'SP500':       'streamlined_s&p_500.csv',
}

MACRO_FILE = 'streamlined_macro.csv'

def load_csv(url):
    r = requests.get(url)
    r.raise_for_status()
    df = pd.read_csv(StringIO(r.text), index_col=0, parse_dates=True)
    df.columns = df.columns.str.replace('[^a-zA-Z0-9]', '_', regex=True)
    return df.sort_index()

# Load macro
macro_df = load_csv(GITHUB_BASE + MACRO_FILE)
macro_df = macro_df[~macro_df.index.duplicated(keep='first')].sort_index()
print(f"macro_df shape : {macro_df.shape}")
print(f"Columns        : {macro_df.columns.tolist()}")

# Load individual index files
index_data = {}
for name, fname in INDEX_FILE_MAP.items():
    try:
        df = load_csv(GITHUB_BASE + fname)
        index_data[name] = df
        print(f"✓ {name:15s} | shape={df.shape} | cols={df.columns.tolist()[:4]}")
    except Exception as e:
        print(f"✗ {name}: {e}")

# ── 2. EXTRACT PRICE SERIES ───────────────────────────────────────────────────
PRICE_COLS = ['Close', 'close', 'Price', 'price', 'Adj_Close', 'Last']

def get_price_series(df, name):
    col = next((c for c in PRICE_COLS if c in df.columns), df.columns[0])
    s   = pd.to_numeric(df[col], errors='coerce').dropna()
    print(f"  {name}: using column '{col}'")
    return s

index_prices = {}
for name, df in index_data.items():
    index_prices[name] = get_price_series(df, name)

# ── 3. MODIFIED DETERMINANT SIGNAL MODEL ─────────────────────────────────────
DET_WINDOWS    = [3, 7, 28]
WINDOW_WEIGHTS = {3: 0.01, 7: 0.01, 28: 0.98}

def compute_modified_determinant(price_series_dict, windows=DET_WINDOWS):
    prices = pd.DataFrame(price_series_dict).sort_index()
    rets   = prices.pct_change().dropna()
    det_results = {}

    for window in windows:
        det_series = []
        for i in range(window, len(rets)):
            window_rets = rets.iloc[i - window: i]
            valid_cols  = window_rets.columns[window_rets.std() > 1e-10]
            w           = window_rets[valid_cols]

            if len(valid_cols) < 2:
                det_series.append(np.nan)
                continue
            try:
                R   = w.corr().values
                I   = np.eye(len(R))
                M   = R - I
                det = np.linalg.det(M)
            except Exception:
                det = np.nan
            det_series.append(det)

        idx = rets.index[window:]
        det_results[window] = pd.Series(det_series, index=idx, name=f'det_{window}d')
        print(f"  Window {window:2d}d | computed {len(det_series)} observations")

    return det_results

print("\n── Computing Modified Determinant ──")
det_results = compute_modified_determinant(index_prices)

# ── 4. COMPOSITE SCORE ────────────────────────────────────────────────────────
def compute_composite_score(det_results, window_weights=WINDOW_WEIGHTS):
    """
    Returns raw weighted-average of normalised determinant values (not signs).
    Normalised per window via rolling z-score so windows are comparable.
    """
    all_series = []
    for window, series in det_results.items():
        # z-score normalise so scale is comparable across windows
        mu    = series.rolling(252, min_periods=60).mean()
        sigma = series.rolling(252, min_periods=60).std()
        z     = (series - mu) / sigma.replace(0, np.nan)
        z.name = f'z_{window}d'
        all_series.append(z)

    z_df      = pd.concat(all_series, axis=1).dropna()
    composite = sum(
        z_df[f'z_{w}d'] * window_weights[w]
        for w in window_weights if f'z_{w}d' in z_df.columns
    )
    composite = composite / sum(window_weights.values())
    return composite, z_df

composite_score, z_df = compute_composite_score(det_results)

print(f"\nComposite score shape  : {composite_score.shape}")
print(f"Date range             : {composite_score.index[0].date()} → "
      f"{composite_score.index[-1].date()}")
print(f"Score range            : {composite_score.min():.3f} → "
      f"{composite_score.max():.3f}")

# ── 5. THREE-STAGE SIGNAL CLASSIFICATION ─────────────────────────────────────
# |score| > threshold  → RISK_ON  (+1): high det magnitude = abnormal regime
#                         → short equity, long country bonds + FX
# |score| <= threshold → RISK_OFF (-1): low det magnitude  = calm regime
#                         → long equity

THRESHOLDS = [0.3, 0.5, 0.7]   # tune here

def classify_signal(score, threshold):
    """
    Returns +1 (RISK_ON / defensive) or -1 (RISK_OFF / long equity)
    """
    if abs(score) > threshold:
        return +1   # high |det| → abnormal → defensive
    else:
        return -1   # low  |det| → calm     → long equity

def build_signal_df(composite_score, equity_assets, macro_dates, threshold=0.5):
    """
    Apply three-stage classification and reindex to portfolio dates.
    """
    binary = composite_score.apply(lambda x: classify_signal(x, threshold))
    signal_df = pd.DataFrame(index=macro_dates)
    for asset in equity_assets:
        signal_df[asset] = binary.reindex(macro_dates).ffill().bfill()
    return signal_df

# ── 6. ASSET UNIVERSE ─────────────────────────────────────────────────────────
EQUITY_ASSETS = list(index_prices.keys())
SAFE_BONDS    = ['US_10Y', 'UK_10Y', 'AU_10Y', 'ZA_10Y']
FX            = ['AUDUSD', 'BRLUSD', 'EURUSD', 'GBPUSD', 'JPYUSD', 'ZARUSD']
COMMODITIES   = ['Oil', 'Gold', 'Bitcoin']

ALL_ASSETS        = EQUITY_ASSETS + FX + COMMODITIES + SAFE_BONDS
ALL_ASSETS        = [a for a in ALL_ASSETS if a in macro_df.columns]
equity_assets     = [a for a in EQUITY_ASSETS if a in ALL_ASSETS]
non_equity_assets = [a for a in ALL_ASSETS if a not in equity_assets]

N           = len(ALL_ASSETS)
base_weight = 1.0 / N

print(f"\nTotal assets              : {N}")
print(f"Equity (signal-driven)    : {equity_assets}")
print(f"Non-equity (always long)  : {non_equity_assets}")
print(f"base_weight = 1/{N} = {base_weight:.4f}")

# ── 7. COUNTRY SAFE-HAVEN MAP ─────────────────────────────────────────────────
# When an equity index is shorted (RISK_ON), proceeds are redistributed
# to that country's bond + FX where available in the universe

EQUITY_SAFE_HAVEN_MAP = {
    'ASX50':       ['AU_10Y', 'AUDUSD'],
    'EUROSTOXX50': ['EURUSD'],            # no EU bond in universe
    'FTSE100':     ['UK_10Y', 'GBPUSD'],
    'Ibovespa':    ['BRLUSD'],            # no Brazil bond in universe
    'JSE40':       ['ZA_10Y', 'ZARUSD'],
    'NIKKEI225':   ['JPYUSD'],            # no JP bond in universe
    'SP500':       ['US_10Y'],            # USD is base currency
}

# Filter to assets actually in universe
UNIVERSE_SET = set(ALL_ASSETS)
for eq, havens in EQUITY_SAFE_HAVEN_MAP.items():
    EQUITY_SAFE_HAVEN_MAP[eq] = [h for h in havens if h in UNIVERSE_SET]

print("\nSafe-haven routing (filtered to universe):")
for eq, havens in EQUITY_SAFE_HAVEN_MAP.items():
    print(f"  {eq:15s} → {havens}")

# ── 8. BUILD RETURNS ──────────────────────────────────────────────────────────
returns_raw = macro_df[ALL_ASSETS].pct_change(fill_method=None)

for col in SAFE_BONDS:
    if col in returns_raw.columns:
        returns_raw[col] = -returns_raw[col]   # invert yield → price

if 'Oil' in macro_df.columns:
    bad_oil = macro_df['Oil'][macro_df['Oil'] <= 0].index
    for d in bad_oil:
        loc = returns_raw.index.get_loc(d)
        returns_raw.iloc[
            max(0, loc - 1): loc + 2,
            returns_raw.columns.get_loc('Oil')
        ] = 0.0

returns = returns_raw.clip(lower=-0.50, upper=1.00).fillna(0).iloc[1:]

assert not returns.isnull().any().any(), "NaNs in returns"
assert not np.isinf(returns.values).any(),  "Inf in returns"
print(f"\nReturns shape : {returns.shape}")
print("✓ Returns clean")

# ── 9. THREE-STAGE WEIGHT BUILDER ────────────────────────────────────────────
def build_weights_3stage(sig_row, equity_assets, non_equity_assets,
                         base_weight, date, safe_haven_map):
    """
    RISK_OFF (-1) : long equity at base_weight
    RISK_ON  (+1) : short equity, redistribute proceeds to country safe havens
                    or equally across all non-equity if no safe haven available
    Non-equity assets always start at base_weight; safe-haven assets receive
    additional weight from shorted equities.
    """
    raw = {a: base_weight for a in non_equity_assets}

    safe_haven_extra = {}   # accumulate extra weight per safe-haven asset

    for a in equity_assets:
        if sig_row is not None and a in sig_row.index and not pd.isna(sig_row[a]):
            sig = sig_row[a]
        else:
            sig = -1   # default: long equity

        if sig > 0:
            # RISK_ON: short equity, redirect base_weight to safe havens
            raw[a]   = -base_weight
            havens   = safe_haven_map.get(a, [])
            havens   = [h for h in havens if h in raw]   # only if in universe

            if havens:
                per_haven = base_weight / len(havens)
                for h in havens:
                    safe_haven_extra[h] = safe_haven_extra.get(h, 0) + per_haven
            else:
                # No safe haven available → spread across all non-equity
                per_ne = base_weight / len(non_equity_assets)
                for ne in non_equity_assets:
                    safe_haven_extra[ne] = safe_haven_extra.get(ne, 0) + per_ne
        else:
            # RISK_OFF: long equity
            raw[a] = base_weight

    # Apply safe-haven extras
    for asset, extra in safe_haven_extra.items():
        raw[asset] = raw.get(asset, 0) + extra

    # Normalise to sum = 1 (handles edge cases)
    total = sum(raw.values())
    if total <= 0:
        for a in raw:
            raw[a] = 1.0 / len(raw)
    else:
        for a in raw:
            raw[a] /= total

    return raw

# ── 10. EQUAL-WEIGHT BENCHMARK ────────────────────────────────────────────────
def run_equal_weight(returns, all_assets, tc_rate=0.005,
                     rebal_freq='monthly', initial_capital=1e9):
    N           = len(all_assets)
    base_weight = 1.0 / N
    ew          = {a: base_weight for a in all_assets}

    freq_code     = 'MS' if rebal_freq == 'monthly' else 'QS'
    reb_dates_raw = returns.resample(freq_code).first().index
    actual_dates  = returns.index

    reb_set = set()
    for d in reb_dates_raw:
        future = actual_dates[actual_dates >= d]
        if len(future) > 0:
            reb_set.add(future[0])

    pv              = float(initial_capital)
    current_weights = None
    records         = []

    for date in actual_dates:
        if date in reb_set:
            if current_weights is not None:
                turnover = sum(abs(ew.get(a, 0) - current_weights.get(a, 0))
                               for a in set(ew) | set(current_weights)) / 2.0
                pv *= (1 - turnover * tc_rate)
            else:
                pv *= (1 - tc_rate)
            current_weights = ew.copy()

        if current_weights is not None:
            dr  = sum(current_weights.get(a, 0) * returns.loc[date, a]
                      for a in all_assets)
            pv *= (1 + dr)

        records.append({'date': date, 'portfolio_value': pv})

    return pd.DataFrame(records).set_index('date')

# ── 11. THREE-STAGE SIGNAL PORTFOLIO RUNNER ───────────────────────────────────
def run_signal_portfolio_3stage(returns, signal_df, equity_assets,
                                non_equity_assets, all_assets,
                                freq='monthly', tc_rate=0.005,
                                safe_haven_map=None,
                                initial_capital=1e9):

    if safe_haven_map is None:
        safe_haven_map = {}

    N           = len(all_assets)
    base_weight = 1.0 / N

    freq_code     = 'MS' if freq == 'monthly' else 'QS'
    reb_dates_raw = returns.resample(freq_code).first().index
    actual_dates  = returns.index

    reb_set = set()
    for d in reb_dates_raw:
        future = actual_dates[actual_dates >= d]
        if len(future) > 0:
            reb_set.add(future[0])

    sig = signal_df.copy()
    sig.index = pd.to_datetime(sig.index)

    pv              = float(initial_capital)
    current_weights = None
    records         = []

    for date in actual_dates:
        if date in reb_set:
            past    = sig[sig.index <= date]
            sig_row = past.iloc[-1] if len(past) > 0 else None

            raw = build_weights_3stage(
                sig_row, equity_assets, non_equity_assets,
                base_weight, date, safe_haven_map
            )

            if current_weights is not None:
                all_keys = set(raw) | set(current_weights)
                turnover = sum(
                    abs(raw.get(a, 0) - current_weights.get(a, 0))
                    for a in all_keys
                ) / 2.0
                pv *= (1 - turnover * tc_rate)
            else:
                pv *= (1 - tc_rate)

            current_weights = raw.copy()

        if current_weights is not None:
            dr  = sum(current_weights.get(a, 0) * returns.loc[date, a]
                      for a in all_assets)
            pv *= (1 + dr)

        records.append({'date': date, 'portfolio_value': pv})

    return pd.DataFrame(records).set_index('date')

# ── 12. RUN ALL PORTFOLIOS ────────────────────────────────────────────────────
TC_LEVELS = {'0%': 0.00, '0.5%': 0.005, '1%': 0.01, '2%': 0.02}
FREQS     = ['monthly', 'quarterly']

# Equal-weight benchmarks
ew_results = {}
for freq in FREQS:
    for tc_label, tc_rate in TC_LEVELS.items():
        label = f"EW_{freq}_TC{tc_label}"
        ew_results[label] = run_equal_weight(
            returns, ALL_ASSETS, tc_rate=tc_rate,
            rebal_freq=freq, initial_capital=1e9
        )
        print(f"✓ {label}")

# Three-stage signal portfolios across all thresholds
signal_results = {}
signal_distributions = {}

for thresh in THRESHOLDS:
    sig_df_t = build_signal_df(composite_score, equity_assets,
                               macro_df.index, threshold=thresh)

    # Print signal distribution for this threshold
    n_risk_on  = (sig_df_t[equity_assets[0]] == +1).sum()
    n_risk_off = (sig_df_t[equity_assets[0]] == -1).sum()
    total      = len(sig_df_t)
    print(f"\nThreshold={thresh} | RISK_ON(+1): {n_risk_on} days "
          f"({n_risk_on/total*100:.1f}%) | "
          f"RISK_OFF(-1): {n_risk_off} days ({n_risk_off/total*100:.1f}%)")

    signal_distributions[thresh] = {'RISK_ON': n_risk_on, 'RISK_OFF': n_risk_off}

    for freq in FREQS:
        for tc_label, tc_rate in TC_LEVELS.items():
            label = f"Signal3S_t{thresh}_{freq}_TC{tc_label}"
            signal_results[label] = run_signal_portfolio_3stage(
                returns, sig_df_t, equity_assets, non_equity_assets,
                ALL_ASSETS, freq=freq, tc_rate=tc_rate,
                safe_haven_map=EQUITY_SAFE_HAVEN_MAP,
                initial_capital=1e9
            )
            print(f"  ✓ {label}")

print("\n✓ All portfolios computed")

# ── 13. PERFORMANCE METRICS ───────────────────────────────────────────────────
def compute_metrics(pv_df, label=''):
    pv        = pv_df['portfolio_value']
    daily_ret = pv.pct_change().dropna()
    n_years   = len(daily_ret) / 252
    total_ret = (pv.iloc[-1] / pv.iloc[0]) - 1
    cagr      = (1 + total_ret) ** (1 / n_years) - 1 if n_years > 0 else np.nan
    vol       = daily_ret.std() * np.sqrt(252)
    sharpe    = cagr / vol if vol > 0 else np.nan
    roll_max  = pv.cummax()
    max_dd    = ((pv - roll_max) / roll_max).min()
    calmar    = cagr / abs(max_dd) if max_dd != 0 else np.nan
    var_95    = daily_ret.quantile(0.05)

    return {
        'CAGR':         f"{cagr*100:.2f}%",
        'Volatility':   f"{vol*100:.2f}%",
        'Sharpe':       f"{sharpe:.2f}",
        'Max Drawdown': f"{max_dd*100:.2f}%",
        'Calmar':       f"{calmar:.2f}",
        'VaR 95%':      f"{var_95*100:.2f}%",
        'Total Return': f"{total_ret*100:.2f}%",
        'Final ($B)':   f"{pv.iloc[-1]/1e9:.3f}",
    }

# Print results grouped by threshold
ew_metrics = {k: compute_metrics(v, k) for k, v in ew_results.items()}

print("\n" + "="*90)
print("EQUAL-WEIGHT BENCHMARK")
print("="*90)
print(pd.DataFrame(ew_metrics).T.to_string())

sig_metrics = {k: compute_metrics(v, k) for k, v in signal_results.items()}

for thresh in THRESHOLDS:
    sub = {k: v for k, v in sig_metrics.items() if f"_t{thresh}_" in k}
    print(f"\n{'='*90}")
    print(f"THREE-STAGE SIGNAL — Threshold = {thresh} | "
          f"RISK_ON: {signal_distributions[thresh]['RISK_ON']} days | "
          f"RISK_OFF: {signal_distributions[thresh]['RISK_OFF']} days")
    print("="*90)
    print(pd.DataFrame(sub).T.to_string())

# ── 14. PLOTS ─────────────────────────────────────────────────────────────────
# One figure per threshold + one overview figure

colors_tc   = {'0%': 'black', '0.5%': 'steelblue', '1%': 'darkorange', '2%': 'crimson'}
colors_freq = {'monthly': 'steelblue', 'quarterly': 'darkorange'}

# ── Figure 1: Composite signal + safe-haven routing overview ──────────────────
fig, axes = plt.subplots(2, 1, figsize=(16, 10))
fig.suptitle("Modified Determinant — Composite Score & Threshold Signals",
             fontsize=14, fontweight='bold')

ax = axes[0]
ax.plot(composite_score.index, composite_score.values,
        color='navy', lw=1.0, alpha=0.8, label='Composite Z-score')
for thresh, col in zip(THRESHOLDS, ['green', 'orange', 'red']):
    ax.axhline( thresh, color=col, ls='--', lw=1.0, label=f'+{thresh}')
    ax.axhline(-thresh, color=col, ls='--', lw=1.0, label=f'-{thresh}')
ax.axhline(0, color='black', lw=0.6)
ax.fill_between(composite_score.index, composite_score.values, 0,
                where=(composite_score > 0), color='green', alpha=0.1)
ax.fill_between(composite_score.index, composite_score.values, 0,
                where=(composite_score <= 0), color='red', alpha=0.1)
ax.set_title("Composite Determinant Score with Thresholds", fontsize=11)
ax.set_ylabel("Z-score")
ax.legend(fontsize=8, ncol=4)
ax.grid(True, alpha=0.3)

ax = axes[1]
for window, series in det_results.items():
    ax.plot(series.index, np.sign(series.values), alpha=0.6,
            label=f"{window}d window", lw=1.0)
ax.axhline(0, color='black', lw=0.8)
ax.set_title("Sign of Det(R-I) per Window", fontsize=11)
ax.set_ylabel("+1 / -1")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("composite_signal_thresholds.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Figure 2: Portfolio values per threshold (TC=0.5%, monthly) ───────────────
fig, axes = plt.subplots(1, 3, figsize=(20, 6), sharey=False)
fig.suptitle("Three-Stage Signal Portfolio — Value by Threshold (TC=0.5%, Monthly)",
             fontsize=13, fontweight='bold')

for ax, thresh in zip(axes, THRESHOLDS):
    # EW reference
    ew_pv = ew_results['EW_monthly_TC0.5%']['portfolio_value']
    ax.plot(ew_pv.index, ew_pv / 1e9, '--', color='grey',
            lw=1.5, label='EW Benchmark')

    label = f"Signal3S_t{thresh}_monthly_TC0.5%"
    pv    = signal_results[label]['portfolio_value']
    ax.plot(pv.index, pv / 1e9, color='steelblue', lw=1.8,
            label=f'Signal (t={thresh})')

    ax.axhline(1.0, color='black', ls=':', lw=0.8)
    ax.set_title(f"Threshold = {thresh}", fontsize=11)
    ax.set_ylabel("Value ($B)")
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("signal_3stage_by_threshold.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Figure 3: Drawdowns per threshold ────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(20, 5), sharey=True)
fig.suptitle("Three-Stage Signal Portfolio — Drawdowns by Threshold (TC=0.5%, Monthly)",
             fontsize=13, fontweight='bold')

for ax, thresh in zip(axes, THRESHOLDS):
    label  = f"Signal3S_t{thresh}_monthly_TC0.5%"
    pv     = signal_results[label]['portfolio_value']
    dd     = (pv - pv.cummax()) / pv.cummax() * 100
    ew_pv  = ew_results['EW_monthly_TC0.5%']['portfolio_value']
    ew_dd  = (ew_pv - ew_pv.cummax()) / ew_pv.cummax() * 100

    ax.fill_between(dd.index,  dd.values,  0, alpha=0.3, color='steelblue')
    ax.fill_between(ew_dd.index, ew_dd.values, 0, alpha=0.2, color='grey')
    ax.plot(dd.index,    dd.values,    color='steelblue', lw=1.0,
            label=f'Signal (t={thresh})')
    ax.plot(ew_dd.index, ew_dd.values, color='grey',      lw=1.0, ls='--',
            label='EW Benchmark')
    ax.set_title(f"Threshold = {thresh}", fontsize=11)
    ax.set_ylabel("Drawdown (%)")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("signal_3stage_drawdowns.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Figure 4: TC sensitivity per threshold ────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.suptitle("Three-Stage Signal — TC Sensitivity (Monthly)",
             fontsize=13, fontweight='bold')

for ax, thresh in zip(axes, THRESHOLDS):
    for tc_label, col in colors_tc.items():
        label = f"Signal3S_t{thresh}_monthly_TC{tc_label}"
        pv    = signal_results[label]['portfolio_value']
        ax.plot(pv.index, pv / 1e9, color=col, lw=1.3,
                label=f'TC={tc_label}')
    ew_pv = ew_results['EW_monthly_TC0.5%']['portfolio_value']
    ax.plot(ew_pv.index, ew_pv / 1e9, '--', color='grey',
            lw=1.5, label='EW Benchmark')
    ax.set_title(f"Threshold = {thresh}", fontsize=11)
    ax.set_ylabel("Value ($B)")
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("signal_3stage_tc_sensitivity.png", dpi=150, bbox_inches='tight')
plt.show()

# ── 15. SAVE ──────────────────────────────────────────────────────────────────
for label, df in {**ew_results, **signal_results}.items():
    df.to_csv(f"{label}_portfolio.csv")

all_metrics = pd.DataFrame({**ew_metrics, **sig_metrics}).T
all_metrics.to_csv("all_portfolio_metrics.csv")

composite_score.to_csv("composite_signal.csv")

for thresh in THRESHOLDS:
    sig_df_t = build_signal_df(composite_score, equity_assets,
                               macro_df.index, threshold=thresh)
    sig_df_t.to_csv(f"signal_df_threshold_{thresh}.csv")

print("✓ All files saved.")


In [ ]:
# @title
# ============================================================
# SIGNAL PORTFOLIO — THREE-STAGE THRESHOLD VERSION
# Signals generated from Modified Determinant Model
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import requests
from io import StringIO

# ── 1. LOAD ALL DATA FROM GITHUB ─────────────────────────────────────────────
GITHUB_BASE = "https://raw.githubusercontent.com/kboroz/MSFE_Capstone_Project/main/01_Data/Streamlined/"

INDEX_FILE_MAP = {
    'ASX50':       'streamlined_asx_50.csv',
    'EUROSTOXX50': 'streamlined_euro_stoxx_50.csv',
    'FTSE100':     'streamlined_ftse_100.csv',
    'Ibovespa':    'streamlined_ibovespa.csv',
    'JSE40':       'streamlined_jse_top_40.csv',
    'NIKKEI225':   'streamlined_nikkei_225.csv',
    'SP500':       'streamlined_s&p_500.csv',
}

MACRO_FILE = 'streamlined_macro.csv'

def load_csv(url):
    r = requests.get(url)
    r.raise_for_status()
    df = pd.read_csv(StringIO(r.text), index_col=0, parse_dates=True)
    df.columns = df.columns.str.replace('[^a-zA-Z0-9]', '_', regex=True)
    return df.sort_index()

# Load macro
macro_df = load_csv(GITHUB_BASE + MACRO_FILE)
macro_df = macro_df[~macro_df.index.duplicated(keep='first')].sort_index()
print(f"macro_df shape : {macro_df.shape}")
print(f"Columns        : {macro_df.columns.tolist()}")

# Load individual index files
index_data = {}
for name, fname in INDEX_FILE_MAP.items():
    try:
        df = load_csv(GITHUB_BASE + fname)
        index_data[name] = df
        print(f"✓ {name:15s} | shape={df.shape} | cols={df.columns.tolist()[:4]}")
    except Exception as e:
        print(f"✗ {name}: {e}")

# ── 2. EXTRACT PRICE SERIES ───────────────────────────────────────────────────
PRICE_COLS = ['Close', 'close', 'Price', 'price', 'Adj_Close', 'Last']

def get_price_series(df, name):
    col = next((c for c in PRICE_COLS if c in df.columns), df.columns[0])
    s   = pd.to_numeric(df[col], errors='coerce').dropna()
    print(f"  {name}: using column '{col}'")
    return s

index_prices = {}
for name, df in index_data.items():
    index_prices[name] = get_price_series(df, name)

# ── 3. MODIFIED DETERMINANT SIGNAL MODEL ─────────────────────────────────────
DET_WINDOWS    = [3, 7, 28]
WINDOW_WEIGHTS = {3: 0.33, 7: 0.34, 28: 0.33}

def compute_modified_determinant(price_series_dict, windows=DET_WINDOWS):
    prices = pd.DataFrame(price_series_dict).sort_index()
    rets   = prices.pct_change().dropna()
    det_results = {}

    for window in windows:
        det_series = []
        for i in range(window, len(rets)):
            window_rets = rets.iloc[i - window: i]
            valid_cols  = window_rets.columns[window_rets.std() > 1e-10]
            w           = window_rets[valid_cols]

            if len(valid_cols) < 2:
                det_series.append(np.nan)
                continue
            try:
                R   = w.corr().values
                I   = np.eye(len(R))
                M   = R - I
                det = np.linalg.det(M)
            except Exception:
                det = np.nan
            det_series.append(det)

        idx = rets.index[window:]
        det_results[window] = pd.Series(det_series, index=idx, name=f'det_{window}d')
        print(f"  Window {window:2d}d | computed {len(det_series)} observations")

    return det_results

print("\n── Computing Modified Determinant ──")
det_results = compute_modified_determinant(index_prices)

# ── 4. COMPOSITE SCORE ────────────────────────────────────────────────────────
def compute_composite_score(det_results, window_weights=WINDOW_WEIGHTS):
    """
    Returns raw weighted-average of normalised determinant values (not signs).
    Normalised per window via rolling z-score so windows are comparable.
    """
    all_series = []
    for window, series in det_results.items():
        # z-score normalise so scale is comparable across windows
        mu    = series.rolling(252, min_periods=60).mean()
        sigma = series.rolling(252, min_periods=60).std()
        z     = (series - mu) / sigma.replace(0, np.nan)
        z.name = f'z_{window}d'
        all_series.append(z)

    z_df      = pd.concat(all_series, axis=1).dropna()
    composite = sum(
        z_df[f'z_{w}d'] * window_weights[w]
        for w in window_weights if f'z_{w}d' in z_df.columns
    )
    composite = composite / sum(window_weights.values())
    return composite, z_df

composite_score, z_df = compute_composite_score(det_results)

print(f"\nComposite score shape  : {composite_score.shape}")
print(f"Date range             : {composite_score.index[0].date()} → "
      f"{composite_score.index[-1].date()}")
print(f"Score range            : {composite_score.min():.3f} → "
      f"{composite_score.max():.3f}")

# ── 5. THREE-STAGE SIGNAL CLASSIFICATION ─────────────────────────────────────
# |score| > threshold  → RISK_ON  (+1): high det magnitude = abnormal regime
#                         → short equity, long country bonds + FX
# |score| <= threshold → RISK_OFF (-1): low det magnitude  = calm regime
#                         → long equity

THRESHOLDS = [1.0, 2.0, 3.0]   # tune here

def classify_signal(score, threshold):
    """
    Returns +1 (RISK_ON / defensive) or -1 (RISK_OFF / long equity)
    """
    if abs(score) > threshold:
        return +1   # high |det| → abnormal → defensive
    else:
        return -1   # low  |det| → calm     → long equity

def build_signal_df(composite_score, equity_assets, macro_dates, threshold=0.5):
    """
    Apply three-stage classification and reindex to portfolio dates.
    """
    binary = composite_score.apply(lambda x: classify_signal(x, threshold))
    signal_df = pd.DataFrame(index=macro_dates)
    for asset in equity_assets:
        signal_df[asset] = binary.reindex(macro_dates).ffill().bfill()
    return signal_df

# ── 6. ASSET UNIVERSE ─────────────────────────────────────────────────────────
EQUITY_ASSETS = list(index_prices.keys())
SAFE_BONDS    = ['US_10Y', 'UK_10Y', 'AU_10Y', 'ZA_10Y']
FX            = ['AUDUSD', 'BRLUSD', 'EURUSD', 'GBPUSD', 'JPYUSD', 'ZARUSD']
COMMODITIES   = ['Oil', 'Gold', 'Bitcoin']

ALL_ASSETS        = EQUITY_ASSETS + FX + COMMODITIES + SAFE_BONDS
ALL_ASSETS        = [a for a in ALL_ASSETS if a in macro_df.columns]
equity_assets     = [a for a in EQUITY_ASSETS if a in ALL_ASSETS]
non_equity_assets = [a for a in ALL_ASSETS if a not in equity_assets]

N           = len(ALL_ASSETS)
base_weight = 1.0 / N

print(f"\nTotal assets              : {N}")
print(f"Equity (signal-driven)    : {equity_assets}")
print(f"Non-equity (always long)  : {non_equity_assets}")
print(f"base_weight = 1/{N} = {base_weight:.4f}")

# ── 7. COUNTRY SAFE-HAVEN MAP ─────────────────────────────────────────────────
# When an equity index is shorted (RISK_ON), proceeds are redistributed
# to that country's bond + FX where available in the universe

EQUITY_SAFE_HAVEN_MAP = {
    'ASX50':       ['AU_10Y', 'AUDUSD'],
    'EUROSTOXX50': ['EURUSD'],            # no EU bond in universe
    'FTSE100':     ['UK_10Y', 'GBPUSD'],
    'Ibovespa':    ['BRLUSD'],            # no Brazil bond in universe
    'JSE40':       ['ZA_10Y', 'ZARUSD'],
    'NIKKEI225':   ['JPYUSD'],            # no JP bond in universe
    'SP500':       ['US_10Y'],            # USD is base currency
}

# Filter to assets actually in universe
UNIVERSE_SET = set(ALL_ASSETS)
for eq, havens in EQUITY_SAFE_HAVEN_MAP.items():
    EQUITY_SAFE_HAVEN_MAP[eq] = [h for h in havens if h in UNIVERSE_SET]

print("\nSafe-haven routing (filtered to universe):")
for eq, havens in EQUITY_SAFE_HAVEN_MAP.items():
    print(f"  {eq:15s} → {havens}")

# ── 8. BUILD RETURNS ──────────────────────────────────────────────────────────
returns_raw = macro_df[ALL_ASSETS].pct_change(fill_method=None)

for col in SAFE_BONDS:
    if col in returns_raw.columns:
        returns_raw[col] = -returns_raw[col]   # invert yield → price

if 'Oil' in macro_df.columns:
    bad_oil = macro_df['Oil'][macro_df['Oil'] <= 0].index
    for d in bad_oil:
        loc = returns_raw.index.get_loc(d)
        returns_raw.iloc[
            max(0, loc - 1): loc + 2,
            returns_raw.columns.get_loc('Oil')
        ] = 0.0

returns = returns_raw.clip(lower=-0.50, upper=1.00).fillna(0).iloc[1:]

assert not returns.isnull().any().any(), "NaNs in returns"
assert not np.isinf(returns.values).any(),  "Inf in returns"
print(f"\nReturns shape : {returns.shape}")
print("✓ Returns clean")

# ── 9. THREE-STAGE WEIGHT BUILDER ────────────────────────────────────────────
def build_weights_3stage(sig_row, equity_assets, non_equity_assets,
                         base_weight, date, safe_haven_map):
    """
    RISK_OFF (-1) : long equity at base_weight
    RISK_ON  (+1) : short equity, redistribute proceeds to country safe havens
                    or equally across all non-equity if no safe haven available
    Non-equity assets always start at base_weight; safe-haven assets receive
    additional weight from shorted equities.
    """
    raw = {a: base_weight for a in non_equity_assets}

    safe_haven_extra = {}   # accumulate extra weight per safe-haven asset

    for a in equity_assets:
        if sig_row is not None and a in sig_row.index and not pd.isna(sig_row[a]):
            sig = sig_row[a]
        else:
            sig = -1   # default: long equity

        if sig > 0:
            # RISK_ON: short equity, redirect base_weight to safe havens
            raw[a]   = -base_weight
            havens   = safe_haven_map.get(a, [])
            havens   = [h for h in havens if h in raw]   # only if in universe

            if havens:
                per_haven = base_weight / len(havens)
                for h in havens:
                    safe_haven_extra[h] = safe_haven_extra.get(h, 0) + per_haven
            else:
                # No safe haven available → spread across all non-equity
                per_ne = base_weight / len(non_equity_assets)
                for ne in non_equity_assets:
                    safe_haven_extra[ne] = safe_haven_extra.get(ne, 0) + per_ne
        else:
            # RISK_OFF: long equity
            raw[a] = base_weight

    # Apply safe-haven extras
    for asset, extra in safe_haven_extra.items():
        raw[asset] = raw.get(asset, 0) + extra

    # Normalise to sum = 1 (handles edge cases)
    total = sum(raw.values())
    if total <= 0:
        for a in raw:
            raw[a] = 1.0 / len(raw)
    else:
        for a in raw:
            raw[a] /= total

    return raw

# ── 10. EQUAL-WEIGHT BENCHMARK ────────────────────────────────────────────────
def run_equal_weight(returns, all_assets, tc_rate=0.005,
                     rebal_freq='monthly', initial_capital=1e9):
    N           = len(all_assets)
    base_weight = 1.0 / N
    ew          = {a: base_weight for a in all_assets}

    freq_code     = 'MS' if rebal_freq == 'monthly' else 'QS'
    reb_dates_raw = returns.resample(freq_code).first().index
    actual_dates  = returns.index

    reb_set = set()
    for d in reb_dates_raw:
        future = actual_dates[actual_dates >= d]
        if len(future) > 0:
            reb_set.add(future[0])

    pv              = float(initial_capital)
    current_weights = None
    records         = []

    for date in actual_dates:
        if date in reb_set:
            if current_weights is not None:
                turnover = sum(abs(ew.get(a, 0) - current_weights.get(a, 0))
                               for a in set(ew) | set(current_weights)) / 2.0
                pv *= (1 - turnover * tc_rate)
            else:
                pv *= (1 - tc_rate)
            current_weights = ew.copy()

        if current_weights is not None:
            dr  = sum(current_weights.get(a, 0) * returns.loc[date, a]
                      for a in all_assets)
            pv *= (1 + dr)

        records.append({'date': date, 'portfolio_value': pv})

    return pd.DataFrame(records).set_index('date')

# ── 11. THREE-STAGE SIGNAL PORTFOLIO RUNNER ───────────────────────────────────
def run_signal_portfolio_3stage(returns, signal_df, equity_assets,
                                non_equity_assets, all_assets,
                                freq='monthly', tc_rate=0.005,
                                safe_haven_map=None,
                                initial_capital=1e9):

    if safe_haven_map is None:
        safe_haven_map = {}

    N           = len(all_assets)
    base_weight = 1.0 / N

    freq_code     = 'MS' if freq == 'monthly' else 'QS'
    reb_dates_raw = returns.resample(freq_code).first().index
    actual_dates  = returns.index

    reb_set = set()
    for d in reb_dates_raw:
        future = actual_dates[actual_dates >= d]
        if len(future) > 0:
            reb_set.add(future[0])

    sig = signal_df.copy()
    sig.index = pd.to_datetime(sig.index)

    pv              = float(initial_capital)
    current_weights = None
    records         = []

    for date in actual_dates:
        if date in reb_set:
            past    = sig[sig.index <= date]
            sig_row = past.iloc[-1] if len(past) > 0 else None

            raw = build_weights_3stage(
                sig_row, equity_assets, non_equity_assets,
                base_weight, date, safe_haven_map
            )

            if current_weights is not None:
                all_keys = set(raw) | set(current_weights)
                turnover = sum(
                    abs(raw.get(a, 0) - current_weights.get(a, 0))
                    for a in all_keys
                ) / 2.0
                pv *= (1 - turnover * tc_rate)
            else:
                pv *= (1 - tc_rate)

            current_weights = raw.copy()

        if current_weights is not None:
            dr  = sum(current_weights.get(a, 0) * returns.loc[date, a]
                      for a in all_assets)
            pv *= (1 + dr)

        records.append({'date': date, 'portfolio_value': pv})

    return pd.DataFrame(records).set_index('date')

# ── 12. RUN ALL PORTFOLIOS ────────────────────────────────────────────────────
TC_LEVELS = {'0%': 0.00, '0.5%': 0.005, '1%': 0.01, '2%': 0.02}
FREQS     = ['monthly', 'quarterly']

# Equal-weight benchmarks
ew_results = {}
for freq in FREQS:
    for tc_label, tc_rate in TC_LEVELS.items():
        label = f"EW_{freq}_TC{tc_label}"
        ew_results[label] = run_equal_weight(
            returns, ALL_ASSETS, tc_rate=tc_rate,
            rebal_freq=freq, initial_capital=1e9
        )
        print(f"✓ {label}")

# Three-stage signal portfolios across all thresholds
signal_results = {}
signal_distributions = {}

for thresh in THRESHOLDS:
    sig_df_t = build_signal_df(composite_score, equity_assets,
                               macro_df.index, threshold=thresh)

    # Print signal distribution for this threshold
    n_risk_on  = (sig_df_t[equity_assets[0]] == +1).sum()
    n_risk_off = (sig_df_t[equity_assets[0]] == -1).sum()
    total      = len(sig_df_t)
    print(f"\nThreshold={thresh} | RISK_ON(+1): {n_risk_on} days "
          f"({n_risk_on/total*100:.1f}%) | "
          f"RISK_OFF(-1): {n_risk_off} days ({n_risk_off/total*100:.1f}%)")

    signal_distributions[thresh] = {'RISK_ON': n_risk_on, 'RISK_OFF': n_risk_off}

    for freq in FREQS:
        for tc_label, tc_rate in TC_LEVELS.items():
            label = f"Signal3S_t{thresh}_{freq}_TC{tc_label}"
            signal_results[label] = run_signal_portfolio_3stage(
                returns, sig_df_t, equity_assets, non_equity_assets,
                ALL_ASSETS, freq=freq, tc_rate=tc_rate,
                safe_haven_map=EQUITY_SAFE_HAVEN_MAP,
                initial_capital=1e9
            )
            print(f"  ✓ {label}")

print("\n✓ All portfolios computed")

# ── 13. PERFORMANCE METRICS ───────────────────────────────────────────────────
def compute_metrics(pv_df, label=''):
    pv        = pv_df['portfolio_value']
    daily_ret = pv.pct_change().dropna()
    n_years   = len(daily_ret) / 252
    total_ret = (pv.iloc[-1] / pv.iloc[0]) - 1
    cagr      = (1 + total_ret) ** (1 / n_years) - 1 if n_years > 0 else np.nan
    vol       = daily_ret.std() * np.sqrt(252)
    sharpe    = cagr / vol if vol > 0 else np.nan
    roll_max  = pv.cummax()
    max_dd    = ((pv - roll_max) / roll_max).min()
    calmar    = cagr / abs(max_dd) if max_dd != 0 else np.nan
    var_95    = daily_ret.quantile(0.05)

    return {
        'CAGR':         f"{cagr*100:.2f}%",
        'Volatility':   f"{vol*100:.2f}%",
        'Sharpe':       f"{sharpe:.2f}",
        'Max Drawdown': f"{max_dd*100:.2f}%",
        'Calmar':       f"{calmar:.2f}",
        'VaR 95%':      f"{var_95*100:.2f}%",
        'Total Return': f"{total_ret*100:.2f}%",
        'Final ($B)':   f"{pv.iloc[-1]/1e9:.3f}",
    }

# Print results grouped by threshold
ew_metrics = {k: compute_metrics(v, k) for k, v in ew_results.items()}

print("\n" + "="*90)
print("EQUAL-WEIGHT BENCHMARK")
print("="*90)
print(pd.DataFrame(ew_metrics).T.to_string())

sig_metrics = {k: compute_metrics(v, k) for k, v in signal_results.items()}

for thresh in THRESHOLDS:
    sub = {k: v for k, v in sig_metrics.items() if f"_t{thresh}_" in k}
    print(f"\n{'='*90}")
    print(f"THREE-STAGE SIGNAL — Threshold = {thresh} | "
          f"RISK_ON: {signal_distributions[thresh]['RISK_ON']} days | "
          f"RISK_OFF: {signal_distributions[thresh]['RISK_OFF']} days")
    print("="*90)
    print(pd.DataFrame(sub).T.to_string())

# ── 14. PLOTS ─────────────────────────────────────────────────────────────────
# One figure per threshold + one overview figure

colors_tc   = {'0%': 'black', '0.5%': 'steelblue', '1%': 'darkorange', '2%': 'crimson'}
colors_freq = {'monthly': 'steelblue', 'quarterly': 'darkorange'}

# ── Figure 1: Composite signal + safe-haven routing overview ──────────────────
fig, axes = plt.subplots(2, 1, figsize=(16, 10))
fig.suptitle("Modified Determinant — Composite Score & Threshold Signals",
             fontsize=14, fontweight='bold')

ax = axes[0]
ax.plot(composite_score.index, composite_score.values,
        color='navy', lw=1.0, alpha=0.8, label='Composite Z-score')
for thresh, col in zip(THRESHOLDS, ['green', 'orange', 'red']):
    ax.axhline( thresh, color=col, ls='--', lw=1.0, label=f'+{thresh}')
    ax.axhline(-thresh, color=col, ls='--', lw=1.0, label=f'-{thresh}')
ax.axhline(0, color='black', lw=0.6)
ax.fill_between(composite_score.index, composite_score.values, 0,
                where=(composite_score > 0), color='green', alpha=0.1)
ax.fill_between(composite_score.index, composite_score.values, 0,
                where=(composite_score <= 0), color='red', alpha=0.1)
ax.set_title("Composite Determinant Score with Thresholds", fontsize=11)
ax.set_ylabel("Z-score")
ax.legend(fontsize=8, ncol=4)
ax.grid(True, alpha=0.3)

ax = axes[1]
for window, series in det_results.items():
    ax.plot(series.index, np.sign(series.values), alpha=0.6,
            label=f"{window}d window", lw=1.0)
ax.axhline(0, color='black', lw=0.8)
ax.set_title("Sign of Det(R-I) per Window", fontsize=11)
ax.set_ylabel("+1 / -1")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("composite_signal_thresholds.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Figure 2: Portfolio values per threshold (TC=0.5%, monthly) ───────────────
fig, axes = plt.subplots(1, 3, figsize=(20, 6), sharey=False)
fig.suptitle("Three-Stage Signal Portfolio — Value by Threshold (TC=0.5%, Monthly)",
             fontsize=13, fontweight='bold')

for ax, thresh in zip(axes, THRESHOLDS):
    # EW reference
    ew_pv = ew_results['EW_monthly_TC0.5%']['portfolio_value']
    ax.plot(ew_pv.index, ew_pv / 1e9, '--', color='grey',
            lw=1.5, label='EW Benchmark')

    label = f"Signal3S_t{thresh}_monthly_TC0.5%"
    pv    = signal_results[label]['portfolio_value']
    ax.plot(pv.index, pv / 1e9, color='steelblue', lw=1.8,
            label=f'Signal (t={thresh})')

    ax.axhline(1.0, color='black', ls=':', lw=0.8)
    ax.set_title(f"Threshold = {thresh}", fontsize=11)
    ax.set_ylabel("Value ($B)")
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("signal_3stage_by_threshold.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Figure 3: Drawdowns per threshold ────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(20, 5), sharey=True)
fig.suptitle("Three-Stage Signal Portfolio — Drawdowns by Threshold (TC=0.5%, Monthly)",
             fontsize=13, fontweight='bold')

for ax, thresh in zip(axes, THRESHOLDS):
    label  = f"Signal3S_t{thresh}_monthly_TC0.5%"
    pv     = signal_results[label]['portfolio_value']
    dd     = (pv - pv.cummax()) / pv.cummax() * 100
    ew_pv  = ew_results['EW_monthly_TC0.5%']['portfolio_value']
    ew_dd  = (ew_pv - ew_pv.cummax()) / ew_pv.cummax() * 100

    ax.fill_between(dd.index,  dd.values,  0, alpha=0.3, color='steelblue')
    ax.fill_between(ew_dd.index, ew_dd.values, 0, alpha=0.2, color='grey')
    ax.plot(dd.index,    dd.values,    color='steelblue', lw=1.0,
            label=f'Signal (t={thresh})')
    ax.plot(ew_dd.index, ew_dd.values, color='grey',      lw=1.0, ls='--',
            label='EW Benchmark')
    ax.set_title(f"Threshold = {thresh}", fontsize=11)
    ax.set_ylabel("Drawdown (%)")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("signal_3stage_drawdowns.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Figure 4: TC sensitivity per threshold ────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.suptitle("Three-Stage Signal — TC Sensitivity (Monthly)",
             fontsize=13, fontweight='bold')

for ax, thresh in zip(axes, THRESHOLDS):
    for tc_label, col in colors_tc.items():
        label = f"Signal3S_t{thresh}_monthly_TC{tc_label}"
        pv    = signal_results[label]['portfolio_value']
        ax.plot(pv.index, pv / 1e9, color=col, lw=1.3,
                label=f'TC={tc_label}')
    ew_pv = ew_results['EW_monthly_TC0.5%']['portfolio_value']
    ax.plot(ew_pv.index, ew_pv / 1e9, '--', color='grey',
            lw=1.5, label='EW Benchmark')
    ax.set_title(f"Threshold = {thresh}", fontsize=11)
    ax.set_ylabel("Value ($B)")
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("signal_3stage_tc_sensitivity.png", dpi=150, bbox_inches='tight')
plt.show()

# ── 15. SAVE ──────────────────────────────────────────────────────────────────
for label, df in {**ew_results, **signal_results}.items():
    df.to_csv(f"{label}_portfolio.csv")

all_metrics = pd.DataFrame({**ew_metrics, **sig_metrics}).T
all_metrics.to_csv("all_portfolio_metrics.csv")

composite_score.to_csv("composite_signal.csv")

for thresh in THRESHOLDS:
    sig_df_t = build_signal_df(composite_score, equity_assets,
                               macro_df.index, threshold=thresh)
    sig_df_t.to_csv(f"signal_df_threshold_{thresh}.csv")

print("✓ All files saved.")


In [ ]:
# @title
# ============================================================
# SIGNAL PORTFOLIO — THREE-STAGE THRESHOLD VERSION
# Signals generated from Modified Determinant Model
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import requests
from io import StringIO

# ── 1. LOAD ALL DATA FROM GITHUB ─────────────────────────────────────────────
GITHUB_BASE = "https://raw.githubusercontent.com/kboroz/MSFE_Capstone_Project/main/01_Data/Streamlined/"

INDEX_FILE_MAP = {
    'ASX50':       'streamlined_asx_50.csv',
    'EUROSTOXX50': 'streamlined_euro_stoxx_50.csv',
    'FTSE100':     'streamlined_ftse_100.csv',
    'Ibovespa':    'streamlined_ibovespa.csv',
    'JSE40':       'streamlined_jse_top_40.csv',
    'NIKKEI225':   'streamlined_nikkei_225.csv',
    'SP500':       'streamlined_s&p_500.csv',
}

MACRO_FILE = 'streamlined_macro.csv'

def load_csv(url):
    r = requests.get(url)
    r.raise_for_status()
    df = pd.read_csv(StringIO(r.text), index_col=0, parse_dates=True)
    df.columns = df.columns.str.replace('[^a-zA-Z0-9]', '_', regex=True)
    return df.sort_index()

# Load macro
macro_df = load_csv(GITHUB_BASE + MACRO_FILE)
macro_df = macro_df[~macro_df.index.duplicated(keep='first')].sort_index()
print(f"macro_df shape : {macro_df.shape}")
print(f"Columns        : {macro_df.columns.tolist()}")

# Load individual index files
index_data = {}
for name, fname in INDEX_FILE_MAP.items():
    try:
        df = load_csv(GITHUB_BASE + fname)
        index_data[name] = df
        print(f"✓ {name:15s} | shape={df.shape} | cols={df.columns.tolist()[:4]}")
    except Exception as e:
        print(f"✗ {name}: {e}")

# ── 2. EXTRACT PRICE SERIES ───────────────────────────────────────────────────
PRICE_COLS = ['Close', 'close', 'Price', 'price', 'Adj_Close', 'Last']

def get_price_series(df, name):
    col = next((c for c in PRICE_COLS if c in df.columns), df.columns[0])
    s   = pd.to_numeric(df[col], errors='coerce').dropna()
    print(f"  {name}: using column '{col}'")
    return s

index_prices = {}
for name, df in index_data.items():
    index_prices[name] = get_price_series(df, name)

# ── 3. MODIFIED DETERMINANT SIGNAL MODEL ─────────────────────────────────────
DET_WINDOWS    = [3, 7, 28]
WINDOW_WEIGHTS = {3: 0.33, 7: 0.34, 28: 0.33}

def compute_modified_determinant(price_series_dict, windows=DET_WINDOWS):
    prices = pd.DataFrame(price_series_dict).sort_index()
    rets   = prices.pct_change().dropna()
    det_results = {}

    for window in windows:
        det_series = []
        for i in range(window, len(rets)):
            window_rets = rets.iloc[i - window: i]
            valid_cols  = window_rets.columns[window_rets.std() > 1e-10]
            w           = window_rets[valid_cols]

            if len(valid_cols) < 2:
                det_series.append(np.nan)
                continue
            try:
                R   = w.corr().values
                I   = np.eye(len(R))
                M   = R - I
                det = np.linalg.det(M)
            except Exception:
                det = np.nan
            det_series.append(det)

        idx = rets.index[window:]
        det_results[window] = pd.Series(det_series, index=idx, name=f'det_{window}d')
        print(f"  Window {window:2d}d | computed {len(det_series)} observations")

    return det_results

print("\n── Computing Modified Determinant ──")
det_results = compute_modified_determinant(index_prices)

# ── 4. COMPOSITE SCORE ────────────────────────────────────────────────────────
def compute_composite_score(det_results, window_weights=WINDOW_WEIGHTS):
    """
    Returns raw weighted-average of normalised determinant values (not signs).
    Normalised per window via rolling z-score so windows are comparable.
    """
    all_series = []
    for window, series in det_results.items():
        # z-score normalise so scale is comparable across windows
        mu    = series.rolling(252, min_periods=60).mean()
        sigma = series.rolling(252, min_periods=60).std()
        z     = (series - mu) / sigma.replace(0, np.nan)
        z.name = f'z_{window}d'
        all_series.append(z)

    z_df      = pd.concat(all_series, axis=1).dropna()
    composite = sum(
        z_df[f'z_{w}d'] * window_weights[w]
        for w in window_weights if f'z_{w}d' in z_df.columns
    )
    composite = composite / sum(window_weights.values())
    return composite, z_df

composite_score, z_df = compute_composite_score(det_results)

print(f"\nComposite score shape  : {composite_score.shape}")
print(f"Date range             : {composite_score.index[0].date()} → "
      f"{composite_score.index[-1].date()}")
print(f"Score range            : {composite_score.min():.3f} → "
      f"{composite_score.max():.3f}")

# ── 5. THREE-STAGE SIGNAL CLASSIFICATION ─────────────────────────────────────
# |score| > threshold  → RISK_ON  (+1): high det magnitude = abnormal regime
#                         → short equity, long country bonds + FX
# |score| <= threshold → RISK_OFF (-1): low det magnitude  = calm regime
#                         → long equity

THRESHOLDS = [0.5, 1.0, 1.5]   # tune here

def classify_signal(score, threshold):
    """
    Returns +1 (RISK_ON / defensive) or -1 (RISK_OFF / long equity)
    """
    if abs(score) > threshold:
        return +1   # high |det| → abnormal → defensive
    else:
        return -1   # low  |det| → calm     → long equity

def build_signal_df(composite_score, equity_assets, macro_dates, threshold=0.5):
    """
    Apply three-stage classification and reindex to portfolio dates.
    """
    binary = composite_score.apply(lambda x: classify_signal(x, threshold))
    signal_df = pd.DataFrame(index=macro_dates)
    for asset in equity_assets:
        signal_df[asset] = binary.reindex(macro_dates).ffill().bfill()
    return signal_df

# ── 6. ASSET UNIVERSE ─────────────────────────────────────────────────────────
EQUITY_ASSETS = list(index_prices.keys())
SAFE_BONDS    = ['US_10Y', 'UK_10Y', 'AU_10Y', 'ZA_10Y']
FX            = ['AUDUSD', 'BRLUSD', 'EURUSD', 'GBPUSD', 'JPYUSD', 'ZARUSD']
COMMODITIES   = ['Oil', 'Gold', 'Bitcoin']

ALL_ASSETS        = EQUITY_ASSETS + FX + COMMODITIES + SAFE_BONDS
ALL_ASSETS        = [a for a in ALL_ASSETS if a in macro_df.columns]
equity_assets     = [a for a in EQUITY_ASSETS if a in ALL_ASSETS]
non_equity_assets = [a for a in ALL_ASSETS if a not in equity_assets]

N           = len(ALL_ASSETS)
base_weight = 1.0 / N

print(f"\nTotal assets              : {N}")
print(f"Equity (signal-driven)    : {equity_assets}")
print(f"Non-equity (always long)  : {non_equity_assets}")
print(f"base_weight = 1/{N} = {base_weight:.4f}")

# ── 7. COUNTRY SAFE-HAVEN MAP ─────────────────────────────────────────────────
# When an equity index is shorted (RISK_ON), proceeds are redistributed
# to that country's bond + FX where available in the universe

EQUITY_SAFE_HAVEN_MAP = {
    'ASX50':       ['AU_10Y', 'AUDUSD'],
    'EUROSTOXX50': ['EURUSD'],            # no EU bond in universe
    'FTSE100':     ['UK_10Y', 'GBPUSD'],
    'Ibovespa':    ['BRLUSD'],            # no Brazil bond in universe
    'JSE40':       ['ZA_10Y', 'ZARUSD'],
    'NIKKEI225':   ['JPYUSD'],            # no JP bond in universe
    'SP500':       ['US_10Y'],            # USD is base currency
}

# Filter to assets actually in universe
UNIVERSE_SET = set(ALL_ASSETS)
for eq, havens in EQUITY_SAFE_HAVEN_MAP.items():
    EQUITY_SAFE_HAVEN_MAP[eq] = [h for h in havens if h in UNIVERSE_SET]

print("\nSafe-haven routing (filtered to universe):")
for eq, havens in EQUITY_SAFE_HAVEN_MAP.items():
    print(f"  {eq:15s} → {havens}")

# ── 8. BUILD RETURNS ──────────────────────────────────────────────────────────
returns_raw = macro_df[ALL_ASSETS].pct_change(fill_method=None)

for col in SAFE_BONDS:
    if col in returns_raw.columns:
        returns_raw[col] = -returns_raw[col]   # invert yield → price

if 'Oil' in macro_df.columns:
    bad_oil = macro_df['Oil'][macro_df['Oil'] <= 0].index
    for d in bad_oil:
        loc = returns_raw.index.get_loc(d)
        returns_raw.iloc[
            max(0, loc - 1): loc + 2,
            returns_raw.columns.get_loc('Oil')
        ] = 0.0

returns = returns_raw.clip(lower=-0.50, upper=1.00).fillna(0).iloc[1:]

assert not returns.isnull().any().any(), "NaNs in returns"
assert not np.isinf(returns.values).any(),  "Inf in returns"
print(f"\nReturns shape : {returns.shape}")
print("✓ Returns clean")

# ── 9. THREE-STAGE WEIGHT BUILDER ────────────────────────────────────────────
def build_weights_3stage(sig_row, equity_assets, non_equity_assets,
                         base_weight, date, safe_haven_map):
    """
    RISK_OFF (-1) : long equity at base_weight
    RISK_ON  (+1) : short equity, redistribute proceeds to country safe havens
                    or equally across all non-equity if no safe haven available
    Non-equity assets always start at base_weight; safe-haven assets receive
    additional weight from shorted equities.
    """
    raw = {a: base_weight for a in non_equity_assets}

    safe_haven_extra = {}   # accumulate extra weight per safe-haven asset

    for a in equity_assets:
        if sig_row is not None and a in sig_row.index and not pd.isna(sig_row[a]):
            sig = sig_row[a]
        else:
            sig = -1   # default: long equity

        if sig > 0:
            # RISK_ON: short equity, redirect base_weight to safe havens
            raw[a]   = -base_weight
            havens   = safe_haven_map.get(a, [])
            havens   = [h for h in havens if h in raw]   # only if in universe

            if havens:
                per_haven = base_weight / len(havens)
                for h in havens:
                    safe_haven_extra[h] = safe_haven_extra.get(h, 0) + per_haven
            else:
                # No safe haven available → spread across all non-equity
                per_ne = base_weight / len(non_equity_assets)
                for ne in non_equity_assets:
                    safe_haven_extra[ne] = safe_haven_extra.get(ne, 0) + per_ne
        else:
            # RISK_OFF: long equity
            raw[a] = base_weight

    # Apply safe-haven extras
    for asset, extra in safe_haven_extra.items():
        raw[asset] = raw.get(asset, 0) + extra

    # Normalise to sum = 1 (handles edge cases)
    total = sum(raw.values())
    if total <= 0:
        for a in raw:
            raw[a] = 1.0 / len(raw)
    else:
        for a in raw:
            raw[a] /= total

    return raw

# ── 10. EQUAL-WEIGHT BENCHMARK ────────────────────────────────────────────────
def run_equal_weight(returns, all_assets, tc_rate=0.005,
                     rebal_freq='monthly', initial_capital=1e9):
    N           = len(all_assets)
    base_weight = 1.0 / N
    ew          = {a: base_weight for a in all_assets}

    freq_code     = 'MS' if rebal_freq == 'monthly' else 'QS'
    reb_dates_raw = returns.resample(freq_code).first().index
    actual_dates  = returns.index

    reb_set = set()
    for d in reb_dates_raw:
        future = actual_dates[actual_dates >= d]
        if len(future) > 0:
            reb_set.add(future[0])

    pv              = float(initial_capital)
    current_weights = None
    records         = []

    for date in actual_dates:
        if date in reb_set:
            if current_weights is not None:
                turnover = sum(abs(ew.get(a, 0) - current_weights.get(a, 0))
                               for a in set(ew) | set(current_weights)) / 2.0
                pv *= (1 - turnover * tc_rate)
            else:
                pv *= (1 - tc_rate)
            current_weights = ew.copy()

        if current_weights is not None:
            dr  = sum(current_weights.get(a, 0) * returns.loc[date, a]
                      for a in all_assets)
            pv *= (1 + dr)

        records.append({'date': date, 'portfolio_value': pv})

    return pd.DataFrame(records).set_index('date')

# ── 11. THREE-STAGE SIGNAL PORTFOLIO RUNNER ───────────────────────────────────
def run_signal_portfolio_3stage(returns, signal_df, equity_assets,
                                non_equity_assets, all_assets,
                                freq='monthly', tc_rate=0.005,
                                safe_haven_map=None,
                                initial_capital=1e9):

    if safe_haven_map is None:
        safe_haven_map = {}

    N           = len(all_assets)
    base_weight = 1.0 / N

    freq_code     = 'MS' if freq == 'monthly' else 'QS'
    reb_dates_raw = returns.resample(freq_code).first().index
    actual_dates  = returns.index

    reb_set = set()
    for d in reb_dates_raw:
        future = actual_dates[actual_dates >= d]
        if len(future) > 0:
            reb_set.add(future[0])

    sig = signal_df.copy()
    sig.index = pd.to_datetime(sig.index)

    pv              = float(initial_capital)
    current_weights = None
    records         = []

    for date in actual_dates:
        if date in reb_set:
            past    = sig[sig.index <= date]
            sig_row = past.iloc[-1] if len(past) > 0 else None

            raw = build_weights_3stage(
                sig_row, equity_assets, non_equity_assets,
                base_weight, date, safe_haven_map
            )

            if current_weights is not None:
                all_keys = set(raw) | set(current_weights)
                turnover = sum(
                    abs(raw.get(a, 0) - current_weights.get(a, 0))
                    for a in all_keys
                ) / 2.0
                pv *= (1 - turnover * tc_rate)
            else:
                pv *= (1 - tc_rate)

            current_weights = raw.copy()

        if current_weights is not None:
            dr  = sum(current_weights.get(a, 0) * returns.loc[date, a]
                      for a in all_assets)
            pv *= (1 + dr)

        records.append({'date': date, 'portfolio_value': pv})

    return pd.DataFrame(records).set_index('date')

# ── 12. RUN ALL PORTFOLIOS ────────────────────────────────────────────────────
TC_LEVELS = {'0%': 0.00, '0.5%': 0.005, '1%': 0.01, '2%': 0.02}
FREQS     = ['monthly', 'quarterly']

# Equal-weight benchmarks
ew_results = {}
for freq in FREQS:
    for tc_label, tc_rate in TC_LEVELS.items():
        label = f"EW_{freq}_TC{tc_label}"
        ew_results[label] = run_equal_weight(
            returns, ALL_ASSETS, tc_rate=tc_rate,
            rebal_freq=freq, initial_capital=1e9
        )
        print(f"✓ {label}")

# Three-stage signal portfolios across all thresholds
signal_results = {}
signal_distributions = {}

for thresh in THRESHOLDS:
    sig_df_t = build_signal_df(composite_score, equity_assets,
                               macro_df.index, threshold=thresh)

    # Print signal distribution for this threshold
    n_risk_on  = (sig_df_t[equity_assets[0]] == +1).sum()
    n_risk_off = (sig_df_t[equity_assets[0]] == -1).sum()
    total      = len(sig_df_t)
    print(f"\nThreshold={thresh} | RISK_ON(+1): {n_risk_on} days "
          f"({n_risk_on/total*100:.1f}%) | "
          f"RISK_OFF(-1): {n_risk_off} days ({n_risk_off/total*100:.1f}%)")

    signal_distributions[thresh] = {'RISK_ON': n_risk_on, 'RISK_OFF': n_risk_off}

    for freq in FREQS:
        for tc_label, tc_rate in TC_LEVELS.items():
            label = f"Signal3S_t{thresh}_{freq}_TC{tc_label}"
            signal_results[label] = run_signal_portfolio_3stage(
                returns, sig_df_t, equity_assets, non_equity_assets,
                ALL_ASSETS, freq=freq, tc_rate=tc_rate,
                safe_haven_map=EQUITY_SAFE_HAVEN_MAP,
                initial_capital=1e9
            )
            print(f"  ✓ {label}")

print("\n✓ All portfolios computed")

# ── 13. PERFORMANCE METRICS ───────────────────────────────────────────────────
def compute_metrics(pv_df, label=''):
    pv        = pv_df['portfolio_value']
    daily_ret = pv.pct_change().dropna()
    n_years   = len(daily_ret) / 252
    total_ret = (pv.iloc[-1] / pv.iloc[0]) - 1
    cagr      = (1 + total_ret) ** (1 / n_years) - 1 if n_years > 0 else np.nan
    vol       = daily_ret.std() * np.sqrt(252)
    sharpe    = cagr / vol if vol > 0 else np.nan
    roll_max  = pv.cummax()
    max_dd    = ((pv - roll_max) / roll_max).min()
    calmar    = cagr / abs(max_dd) if max_dd != 0 else np.nan
    var_95    = daily_ret.quantile(0.05)

    return {
        'CAGR':         f"{cagr*100:.2f}%",
        'Volatility':   f"{vol*100:.2f}%",
        'Sharpe':       f"{sharpe:.2f}",
        'Max Drawdown': f"{max_dd*100:.2f}%",
        'Calmar':       f"{calmar:.2f}",
        'VaR 95%':      f"{var_95*100:.2f}%",
        'Total Return': f"{total_ret*100:.2f}%",
        'Final ($B)':   f"{pv.iloc[-1]/1e9:.3f}",
    }

# Print results grouped by threshold
ew_metrics = {k: compute_metrics(v, k) for k, v in ew_results.items()}

print("\n" + "="*90)
print("EQUAL-WEIGHT BENCHMARK")
print("="*90)
print(pd.DataFrame(ew_metrics).T.to_string())

sig_metrics = {k: compute_metrics(v, k) for k, v in signal_results.items()}

for thresh in THRESHOLDS:
    sub = {k: v for k, v in sig_metrics.items() if f"_t{thresh}_" in k}
    print(f"\n{'='*90}")
    print(f"THREE-STAGE SIGNAL — Threshold = {thresh} | "
          f"RISK_ON: {signal_distributions[thresh]['RISK_ON']} days | "
          f"RISK_OFF: {signal_distributions[thresh]['RISK_OFF']} days")
    print("="*90)
    print(pd.DataFrame(sub).T.to_string())

# ── 14. PLOTS ─────────────────────────────────────────────────────────────────
# One figure per threshold + one overview figure

colors_tc   = {'0%': 'black', '0.5%': 'steelblue', '1%': 'darkorange', '2%': 'crimson'}
colors_freq = {'monthly': 'steelblue', 'quarterly': 'darkorange'}

# ── Figure 1: Composite signal + safe-haven routing overview ──────────────────
fig, axes = plt.subplots(2, 1, figsize=(16, 10))
fig.suptitle("Modified Determinant — Composite Score & Threshold Signals",
             fontsize=14, fontweight='bold')

ax = axes[0]
ax.plot(composite_score.index, composite_score.values,
        color='navy', lw=1.0, alpha=0.8, label='Composite Z-score')
for thresh, col in zip(THRESHOLDS, ['green', 'orange', 'red']):
    ax.axhline( thresh, color=col, ls='--', lw=1.0, label=f'+{thresh}')
    ax.axhline(-thresh, color=col, ls='--', lw=1.0, label=f'-{thresh}')
ax.axhline(0, color='black', lw=0.6)
ax.fill_between(composite_score.index, composite_score.values, 0,
                where=(composite_score > 0), color='green', alpha=0.1)
ax.fill_between(composite_score.index, composite_score.values, 0,
                where=(composite_score <= 0), color='red', alpha=0.1)
ax.set_title("Composite Determinant Score with Thresholds", fontsize=11)
ax.set_ylabel("Z-score")
ax.legend(fontsize=8, ncol=4)
ax.grid(True, alpha=0.3)

ax = axes[1]
for window, series in det_results.items():
    ax.plot(series.index, np.sign(series.values), alpha=0.6,
            label=f"{window}d window", lw=1.0)
ax.axhline(0, color='black', lw=0.8)
ax.set_title("Sign of Det(R-I) per Window", fontsize=11)
ax.set_ylabel("+1 / -1")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("composite_signal_thresholds.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Figure 2: Portfolio values per threshold (TC=0.5%, monthly) ───────────────
fig, axes = plt.subplots(1, 3, figsize=(20, 6), sharey=False)
fig.suptitle("Three-Stage Signal Portfolio — Value by Threshold (TC=0.5%, Monthly)",
             fontsize=13, fontweight='bold')

for ax, thresh in zip(axes, THRESHOLDS):
    # EW reference
    ew_pv = ew_results['EW_monthly_TC0.5%']['portfolio_value']
    ax.plot(ew_pv.index, ew_pv / 1e9, '--', color='grey',
            lw=1.5, label='EW Benchmark')

    label = f"Signal3S_t{thresh}_monthly_TC0.5%"
    pv    = signal_results[label]['portfolio_value']
    ax.plot(pv.index, pv / 1e9, color='steelblue', lw=1.8,
            label=f'Signal (t={thresh})')

    ax.axhline(1.0, color='black', ls=':', lw=0.8)
    ax.set_title(f"Threshold = {thresh}", fontsize=11)
    ax.set_ylabel("Value ($B)")
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("signal_3stage_by_threshold.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Figure 3: Drawdowns per threshold ────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(20, 5), sharey=True)
fig.suptitle("Three-Stage Signal Portfolio — Drawdowns by Threshold (TC=0.5%, Monthly)",
             fontsize=13, fontweight='bold')

for ax, thresh in zip(axes, THRESHOLDS):
    label  = f"Signal3S_t{thresh}_monthly_TC0.5%"
    pv     = signal_results[label]['portfolio_value']
    dd     = (pv - pv.cummax()) / pv.cummax() * 100
    ew_pv  = ew_results['EW_monthly_TC0.5%']['portfolio_value']
    ew_dd  = (ew_pv - ew_pv.cummax()) / ew_pv.cummax() * 100

    ax.fill_between(dd.index,  dd.values,  0, alpha=0.3, color='steelblue')
    ax.fill_between(ew_dd.index, ew_dd.values, 0, alpha=0.2, color='grey')
    ax.plot(dd.index,    dd.values,    color='steelblue', lw=1.0,
            label=f'Signal (t={thresh})')
    ax.plot(ew_dd.index, ew_dd.values, color='grey',      lw=1.0, ls='--',
            label='EW Benchmark')
    ax.set_title(f"Threshold = {thresh}", fontsize=11)
    ax.set_ylabel("Drawdown (%)")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("signal_3stage_drawdowns.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Figure 4: TC sensitivity per threshold ────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.suptitle("Three-Stage Signal — TC Sensitivity (Monthly)",
             fontsize=13, fontweight='bold')

for ax, thresh in zip(axes, THRESHOLDS):
    for tc_label, col in colors_tc.items():
        label = f"Signal3S_t{thresh}_monthly_TC{tc_label}"
        pv    = signal_results[label]['portfolio_value']
        ax.plot(pv.index, pv / 1e9, color=col, lw=1.3,
                label=f'TC={tc_label}')
    ew_pv = ew_results['EW_monthly_TC0.5%']['portfolio_value']
    ax.plot(ew_pv.index, ew_pv / 1e9, '--', color='grey',
            lw=1.5, label='EW Benchmark')
    ax.set_title(f"Threshold = {thresh}", fontsize=11)
    ax.set_ylabel("Value ($B)")
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("signal_3stage_tc_sensitivity.png", dpi=150, bbox_inches='tight')
plt.show()

# ── 15. SAVE ──────────────────────────────────────────────────────────────────
for label, df in {**ew_results, **signal_results}.items():
    df.to_csv(f"{label}_portfolio.csv")

all_metrics = pd.DataFrame({**ew_metrics, **sig_metrics}).T
all_metrics.to_csv("all_portfolio_metrics.csv")

composite_score.to_csv("composite_signal.csv")

for thresh in THRESHOLDS:
    sig_df_t = build_signal_df(composite_score, equity_assets,
                               macro_df.index, threshold=thresh)
    sig_df_t.to_csv(f"signal_df_threshold_{thresh}.csv")

print("✓ All files saved.")


In [ ]:
# @title
# ============================================================
# SIGNAL PORTFOLIO — THREE-STAGE THRESHOLD VERSION
# Signals generated from Modified Determinant Model
# + Buy-and-Hold Benchmark
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import requests
from io import StringIO

# ── 1. LOAD ALL DATA FROM GITHUB ─────────────────────────────────────────────
GITHUB_BASE = "https://raw.githubusercontent.com/kboroz/MSFE_Capstone_Project/main/01_Data/Streamlined/"

INDEX_FILE_MAP = {
    'ASX50':       'streamlined_asx_50.csv',
    'EUROSTOXX50': 'streamlined_euro_stoxx_50.csv',
    'FTSE100':     'streamlined_ftse_100.csv',
    'Ibovespa':    'streamlined_ibovespa.csv',
    'JSE40':       'streamlined_jse_top_40.csv',
    'NIKKEI225':   'streamlined_nikkei_225.csv',
    'SP500':       'streamlined_s&p_500.csv',
}

MACRO_FILE = 'streamlined_macro.csv'

def load_csv(url):
    r = requests.get(url)
    r.raise_for_status()
    df = pd.read_csv(StringIO(r.text), index_col=0, parse_dates=True)
    df.columns = df.columns.str.replace('[^a-zA-Z0-9]', '_', regex=True)
    return df.sort_index()

# Load macro
macro_df = load_csv(GITHUB_BASE + MACRO_FILE)
macro_df = macro_df[~macro_df.index.duplicated(keep='first')].sort_index()
print(f"macro_df shape : {macro_df.shape}")
print(f"Columns        : {macro_df.columns.tolist()}")

# Load individual index files
index_data = {}
for name, fname in INDEX_FILE_MAP.items():
    try:
        df = load_csv(GITHUB_BASE + fname)
        index_data[name] = df
        print(f"✓ {name:15s} | shape={df.shape} | cols={df.columns.tolist()[:4]}")
    except Exception as e:
        print(f"✗ {name}: {e}")

# ── 2. EXTRACT PRICE SERIES ───────────────────────────────────────────────────
PRICE_COLS = ['Close', 'close', 'Price', 'price', 'Adj_Close', 'Last']

def get_price_series(df, name):
    col = next((c for c in PRICE_COLS if c in df.columns), df.columns[0])
    s   = pd.to_numeric(df[col], errors='coerce').dropna()
    print(f"  {name}: using column '{col}'")
    return s

index_prices = {}
for name, df in index_data.items():
    index_prices[name] = get_price_series(df, name)

# ── 3. MODIFIED DETERMINANT SIGNAL MODEL ─────────────────────────────────────
DET_WINDOWS    = [3, 7, 28]
WINDOW_WEIGHTS = {3: 0.33, 7: 0.34, 28: 0.33}

def compute_modified_determinant(price_series_dict, windows=DET_WINDOWS):
    prices = pd.DataFrame(price_series_dict).sort_index()
    rets   = prices.pct_change().dropna()
    det_results = {}

    for window in windows:
        det_series = []
        for i in range(window, len(rets)):
            window_rets = rets.iloc[i - window: i]
            valid_cols  = window_rets.columns[window_rets.std() > 1e-10]
            w           = window_rets[valid_cols]

            if len(valid_cols) < 2:
                det_series.append(np.nan)
                continue
            try:
                R   = w.corr().values
                I   = np.eye(len(R))
                M   = R - I
                det = np.linalg.det(M)
            except Exception:
                det = np.nan
            det_series.append(det)

        idx = rets.index[window:]
        det_results[window] = pd.Series(det_series, index=idx, name=f'det_{window}d')
        print(f"  Window {window:2d}d | computed {len(det_series)} observations")

    return det_results

print("\n── Computing Modified Determinant ──")
det_results = compute_modified_determinant(index_prices)

# ── 4. COMPOSITE SCORE ────────────────────────────────────────────────────────
def compute_composite_score(det_results, window_weights=WINDOW_WEIGHTS):
    all_series = []
    for window, series in det_results.items():
        mu    = series.rolling(252, min_periods=60).mean()
        sigma = series.rolling(252, min_periods=60).std()
        z     = (series - mu) / sigma.replace(0, np.nan)
        z.name = f'z_{window}d'
        all_series.append(z)

    z_df      = pd.concat(all_series, axis=1).dropna()
    composite = sum(
        z_df[f'z_{w}d'] * window_weights[w]
        for w in window_weights if f'z_{w}d' in z_df.columns
    )
    composite = composite / sum(window_weights.values())
    return composite, z_df

composite_score, z_df = compute_composite_score(det_results)

print(f"\nComposite score shape  : {composite_score.shape}")
print(f"Date range             : {composite_score.index[0].date()} → "
      f"{composite_score.index[-1].date()}")
print(f"Score range            : {composite_score.min():.3f} → "
      f"{composite_score.max():.3f}")

# ── 5. THREE-STAGE SIGNAL CLASSIFICATION ─────────────────────────────────────
THRESHOLDS = [0.5, 1.0, 1.5]

def classify_signal(score, threshold):
    if abs(score) > threshold:
        return +1   # RISK_ON  → defensive
    else:
        return -1   # RISK_OFF → long equity

def build_signal_df(composite_score, equity_assets, macro_dates, threshold=0.5):
    binary    = composite_score.apply(lambda x: classify_signal(x, threshold))
    signal_df = pd.DataFrame(index=macro_dates)
    for asset in equity_assets:
        signal_df[asset] = binary.reindex(macro_dates).ffill().bfill()
    return signal_df

# ── 6. ASSET UNIVERSE ─────────────────────────────────────────────────────────
EQUITY_ASSETS = list(index_prices.keys())
SAFE_BONDS    = ['US_10Y', 'UK_10Y', 'AU_10Y', 'ZA_10Y']
FX            = ['AUDUSD', 'BRLUSD', 'EURUSD', 'GBPUSD', 'JPYUSD', 'ZARUSD']
COMMODITIES   = ['Oil', 'Gold', 'Bitcoin']

ALL_ASSETS        = EQUITY_ASSETS + FX + COMMODITIES + SAFE_BONDS
ALL_ASSETS        = [a for a in ALL_ASSETS if a in macro_df.columns]
equity_assets     = [a for a in EQUITY_ASSETS if a in ALL_ASSETS]
non_equity_assets = [a for a in ALL_ASSETS if a not in equity_assets]

N           = len(ALL_ASSETS)
base_weight = 1.0 / N

print(f"\nTotal assets              : {N}")
print(f"Equity (signal-driven)    : {equity_assets}")
print(f"Non-equity (always long)  : {non_equity_assets}")
print(f"base_weight = 1/{N} = {base_weight:.4f}")

# ── 7. COUNTRY SAFE-HAVEN MAP ─────────────────────────────────────────────────
EQUITY_SAFE_HAVEN_MAP = {
    'ASX50':       ['AU_10Y', 'AUDUSD'],
    'EUROSTOXX50': ['EURUSD'],
    'FTSE100':     ['UK_10Y', 'GBPUSD'],
    'Ibovespa':    ['BRLUSD'],
    'JSE40':       ['ZA_10Y', 'ZARUSD'],
    'NIKKEI225':   ['JPYUSD'],
    'SP500':       ['US_10Y'],
}

UNIVERSE_SET = set(ALL_ASSETS)
for eq, havens in EQUITY_SAFE_HAVEN_MAP.items():
    EQUITY_SAFE_HAVEN_MAP[eq] = [h for h in havens if h in UNIVERSE_SET]

print("\nSafe-haven routing (filtered to universe):")
for eq, havens in EQUITY_SAFE_HAVEN_MAP.items():
    print(f"  {eq:15s} → {havens}")

# ── 8. BUILD RETURNS ──────────────────────────────────────────────────────────
returns_raw = macro_df[ALL_ASSETS].pct_change(fill_method=None)

for col in SAFE_BONDS:
    if col in returns_raw.columns:
        returns_raw[col] = -returns_raw[col]

if 'Oil' in macro_df.columns:
    bad_oil = macro_df['Oil'][macro_df['Oil'] <= 0].index
    for d in bad_oil:
        loc = returns_raw.index.get_loc(d)
        returns_raw.iloc[
            max(0, loc - 1): loc + 2,
            returns_raw.columns.get_loc('Oil')
        ] = 0.0

returns = returns_raw.clip(lower=-0.50, upper=1.00).fillna(0).iloc[1:]

assert not returns.isnull().any().any(), "NaNs in returns"
assert not np.isinf(returns.values).any(),  "Inf in returns"
print(f"\nReturns shape : {returns.shape}")
print("✓ Returns clean")

# ── 9. BUY-AND-HOLD BENCHMARK ────────────────────────────────────────────────
def run_buy_and_hold(returns, all_assets, initial_capital=1e9):
    """
    Equally-weighted across all assets, set weights on day 1 and never rebalance.
    One-time entry transaction cost applied at inception only.
    No further TC since there is no trading.
    Weights drift with market moves — true passive holding.
    """
    N           = len(all_assets)
    base_weight = 1.0 / N

    # Initialise dollar holdings
    pv       = float(initial_capital)
    holdings = {a: pv * base_weight for a in all_assets}   # dollar amounts

    records = []
    first   = True

    for date in returns.index:
        if first:
            first = False
            # Record starting value (no return on day 0)
            records.append({'date': date, 'portfolio_value': pv})
            continue

        # Update each holding by its return
        new_holdings = {}
        for a in all_assets:
            new_holdings[a] = holdings[a] * (1 + returns.loc[date, a])

        holdings = new_holdings
        pv       = sum(holdings.values())
        records.append({'date': date, 'portfolio_value': pv})

    df = pd.DataFrame(records).set_index('date')
    print(f"  Buy-and-Hold | Start: ${initial_capital/1e9:.2f}B → "
          f"End: ${df['portfolio_value'].iloc[-1]/1e9:.3f}B")
    return df

# ── 10. THREE-STAGE WEIGHT BUILDER ───────────────────────────────────────────
def build_weights_3stage(sig_row, equity_assets, non_equity_assets,
                         base_weight, date, safe_haven_map):
    raw              = {a: base_weight for a in non_equity_assets}
    safe_haven_extra = {}

    for a in equity_assets:
        if sig_row is not None and a in sig_row.index and not pd.isna(sig_row[a]):
            sig = sig_row[a]
        else:
            sig = -1

        if sig > 0:
            raw[a]  = -base_weight
            havens  = safe_haven_map.get(a, [])
            havens  = [h for h in havens if h in raw]

            if havens:
                per_haven = base_weight / len(havens)
                for h in havens:
                    safe_haven_extra[h] = safe_haven_extra.get(h, 0) + per_haven
            else:
                per_ne = base_weight / len(non_equity_assets)
                for ne in non_equity_assets:
                    safe_haven_extra[ne] = safe_haven_extra.get(ne, 0) + per_ne
        else:
            raw[a] = base_weight

    for asset, extra in safe_haven_extra.items():
        raw[asset] = raw.get(asset, 0) + extra

    total = sum(raw.values())
    if total <= 0:
        for a in raw:
            raw[a] = 1.0 / len(raw)
    else:
        for a in raw:
            raw[a] /= total

    return raw

# ── 11. EQUAL-WEIGHT REBALANCED BENCHMARK ────────────────────────────────────
def run_equal_weight(returns, all_assets, tc_rate=0.005,
                     rebal_freq='monthly', initial_capital=1e9):
    N           = len(all_assets)
    base_weight = 1.0 / N
    ew          = {a: base_weight for a in all_assets}

    freq_code     = 'MS' if rebal_freq == 'monthly' else 'QS'
    reb_dates_raw = returns.resample(freq_code).first().index
    actual_dates  = returns.index

    reb_set = set()
    for d in reb_dates_raw:
        future = actual_dates[actual_dates >= d]
        if len(future) > 0:
            reb_set.add(future[0])

    pv              = float(initial_capital)
    current_weights = None
    records         = []

    for date in actual_dates:
        if date in reb_set:
            if current_weights is not None:
                turnover = sum(abs(ew.get(a, 0) - current_weights.get(a, 0))
                               for a in set(ew) | set(current_weights)) / 2.0
                pv *= (1 - turnover * tc_rate)
            else:
                pv *= (1 - tc_rate)
            current_weights = ew.copy()

        if current_weights is not None:
            dr  = sum(current_weights.get(a, 0) * returns.loc[date, a]
                      for a in all_assets)
            pv *= (1 + dr)

        records.append({'date': date, 'portfolio_value': pv})

    return pd.DataFrame(records).set_index('date')

# ── 12. THREE-STAGE SIGNAL PORTFOLIO RUNNER ───────────────────────────────────
def run_signal_portfolio_3stage(returns, signal_df, equity_assets,
                                non_equity_assets, all_assets,
                                freq='monthly', tc_rate=0.005,
                                safe_haven_map=None,
                                initial_capital=1e9):
    if safe_haven_map is None:
        safe_haven_map = {}

    N           = len(all_assets)
    base_weight = 1.0 / N

    freq_code     = 'MS' if freq == 'monthly' else 'QS'
    reb_dates_raw = returns.resample(freq_code).first().index
    actual_dates  = returns.index

    reb_set = set()
    for d in reb_dates_raw:
        future = actual_dates[actual_dates >= d]
        if len(future) > 0:
            reb_set.add(future[0])

    sig = signal_df.copy()
    sig.index = pd.to_datetime(sig.index)

    pv              = float(initial_capital)
    current_weights = None
    records         = []

    for date in actual_dates:
        if date in reb_set:
            past    = sig[sig.index <= date]
            sig_row = past.iloc[-1] if len(past) > 0 else None

            raw = build_weights_3stage(
                sig_row, equity_assets, non_equity_assets,
                base_weight, date, safe_haven_map
            )

            if current_weights is not None:
                all_keys = set(raw) | set(current_weights)
                turnover = sum(
                    abs(raw.get(a, 0) - current_weights.get(a, 0))
                    for a in all_keys
                ) / 2.0
                pv *= (1 - turnover * tc_rate)
            else:
                pv *= (1 - tc_rate)

            current_weights = raw.copy()

        if current_weights is not None:
            dr  = sum(current_weights.get(a, 0) * returns.loc[date, a]
                      for a in all_assets)
            pv *= (1 + dr)

        records.append({'date': date, 'portfolio_value': pv})

    return pd.DataFrame(records).set_index('date')

# ── 13. RUN ALL PORTFOLIOS ────────────────────────────────────────────────────
TC_LEVELS = {'0%': 0.00, '0.5%': 0.005, '1%': 0.01, '2%': 0.02}
FREQS     = ['monthly', 'quarterly']

# ── Buy-and-Hold (single run, no TC variants needed) ─────────────────────────
print("\n── Running Buy-and-Hold ──")
bh_result  = run_buy_and_hold(returns, ALL_ASSETS, initial_capital=1e9)
bh_results = {'BuyAndHold_EW': bh_result}   # dict for uniform handling

# ── Equal-weight rebalanced benchmarks ───────────────────────────────────────
print("\n── Running Equal-Weight Rebalanced ──")
ew_results = {}
for freq in FREQS:
    for tc_label, tc_rate in TC_LEVELS.items():
        label = f"EW_{freq}_TC{tc_label}"
        ew_results[label] = run_equal_weight(
            returns, ALL_ASSETS, tc_rate=tc_rate,
            rebal_freq=freq, initial_capital=1e9
        )
        print(f"  ✓ {label}")

# ── Three-stage signal portfolios ─────────────────────────────────────────────
print("\n── Running Signal Portfolios ──")
signal_results       = {}
signal_distributions = {}

for thresh in THRESHOLDS:
    sig_df_t   = build_signal_df(composite_score, equity_assets,
                                 macro_df.index, threshold=thresh)
    n_risk_on  = (sig_df_t[equity_assets[0]] == +1).sum()
    n_risk_off = (sig_df_t[equity_assets[0]] == -1).sum()
    total      = len(sig_df_t)

    print(f"\n  Threshold={thresh} | RISK_ON(+1): {n_risk_on} days "
          f"({n_risk_on/total*100:.1f}%) | "
          f"RISK_OFF(-1): {n_risk_off} days ({n_risk_off/total*100:.1f}%)")

    signal_distributions[thresh] = {'RISK_ON': n_risk_on, 'RISK_OFF': n_risk_off}

    for freq in FREQS:
        for tc_label, tc_rate in TC_LEVELS.items():
            label = f"Signal3S_t{thresh}_{freq}_TC{tc_label}"
            signal_results[label] = run_signal_portfolio_3stage(
                returns, sig_df_t, equity_assets, non_equity_assets,
                ALL_ASSETS, freq=freq, tc_rate=tc_rate,
                safe_haven_map=EQUITY_SAFE_HAVEN_MAP,
                initial_capital=1e9
            )
            print(f"    ✓ {label}")

print("\n✓ All portfolios computed")

# ── 14. PERFORMANCE METRICS ───────────────────────────────────────────────────
def compute_metrics(pv_df, label=''):
    pv        = pv_df['portfolio_value']
    daily_ret = pv.pct_change().dropna()
    n_years   = len(daily_ret) / 252
    total_ret = (pv.iloc[-1] / pv.iloc[0]) - 1
    cagr      = (1 + total_ret) ** (1 / n_years) - 1 if n_years > 0 else np.nan
    vol       = daily_ret.std() * np.sqrt(252)
    sharpe    = cagr / vol if vol > 0 else np.nan
    roll_max  = pv.cummax()
    max_dd    = ((pv - roll_max) / roll_max).min()
    calmar    = cagr / abs(max_dd) if max_dd != 0 else np.nan
    var_95    = daily_ret.quantile(0.05)

    return {
        'CAGR':         f"{cagr*100:.2f}%",
        'Volatility':   f"{vol*100:.2f}%",
        'Sharpe':       f"{sharpe:.2f}",
        'Max Drawdown': f"{max_dd*100:.2f}%",
        'Calmar':       f"{calmar:.2f}",
        'VaR 95%':      f"{var_95*100:.2f}%",
        'Total Return': f"{total_ret*100:.2f}%",
        'Final ($B)':   f"{pv.iloc[-1]/1e9:.3f}",
    }

# ── Print metrics ─────────────────────────────────────────────────────────────
bh_metrics  = {k: compute_metrics(v, k) for k, v in bh_results.items()}
ew_metrics  = {k: compute_metrics(v, k) for k, v in ew_results.items()}
sig_metrics = {k: compute_metrics(v, k) for k, v in signal_results.items()}

print("\n" + "="*90)
print("BUY-AND-HOLD BENCHMARK (EW, No Rebalancing)")
print("="*90)
print(pd.DataFrame(bh_metrics).T.to_string())

print("\n" + "="*90)
print("EQUAL-WEIGHT REBALANCED BENCHMARK")
print("="*90)
print(pd.DataFrame(ew_metrics).T.to_string())

for thresh in THRESHOLDS:
    sub = {k: v for k, v in sig_metrics.items() if f"_t{thresh}_" in k}
    print(f"\n{'='*90}")
    print(f"THREE-STAGE SIGNAL — Threshold = {thresh} | "
          f"RISK_ON: {signal_distributions[thresh]['RISK_ON']} days | "
          f"RISK_OFF: {signal_distributions[thresh]['RISK_OFF']} days")
    print("="*90)
    print(pd.DataFrame(sub).T.to_string())

# ── 15. PLOTS ─────────────────────────────────────────────────────────────────
colors_tc = {'0%': 'black', '0.5%': 'steelblue', '1%': 'darkorange', '2%': 'crimson'}

# Convenience references reused across figures
bh_pv      = bh_results['BuyAndHold_EW']['portfolio_value']
ew_ref_pv  = ew_results['EW_monthly_TC0.5%']['portfolio_value']

# ── Figure 1: Composite signal overview ──────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(16, 10))
fig.suptitle("Modified Determinant — Composite Score & Threshold Signals",
             fontsize=14, fontweight='bold')

ax = axes[0]
ax.plot(composite_score.index, composite_score.values,
        color='navy', lw=1.0, alpha=0.8, label='Composite Z-score')
for thresh, col in zip(THRESHOLDS, ['green', 'orange', 'red']):
    ax.axhline( thresh, color=col, ls='--', lw=1.0, label=f'+{thresh}')
    ax.axhline(-thresh, color=col, ls='--', lw=1.0, label=f'-{thresh}')
ax.axhline(0, color='black', lw=0.6)
ax.fill_between(composite_score.index, composite_score.values, 0,
                where=(composite_score > 0),  color='green', alpha=0.1)
ax.fill_between(composite_score.index, composite_score.values, 0,
                where=(composite_score <= 0), color='red',   alpha=0.1)
ax.set_title("Composite Determinant Score with Thresholds", fontsize=11)
ax.set_ylabel("Z-score")
ax.legend(fontsize=8, ncol=4)
ax.grid(True, alpha=0.3)

ax = axes[1]
for window, series in det_results.items():
    ax.plot(series.index, np.sign(series.values), alpha=0.6,
            label=f"{window}d window", lw=1.0)
ax.axhline(0, color='black', lw=0.8)
ax.set_title("Sign of Det(R-I) per Window", fontsize=11)
ax.set_ylabel("+1 / -1")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("composite_signal_thresholds.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Figure 2: Portfolio values per threshold vs both benchmarks ───────────────
fig, axes = plt.subplots(1, 3, figsize=(22, 6), sharey=False)
fig.suptitle(
    "Three-Stage Signal vs Benchmarks — Portfolio Value (TC=0.5%, Monthly)",
    fontsize=13, fontweight='bold')

for ax, thresh in zip(axes, THRESHOLDS):
    # Buy-and-hold
    ax.plot(bh_pv.index, bh_pv / 1e9,
            color='forestgreen', lw=1.8, ls='-.',
            label='Buy & Hold (EW)')
    # EW rebalanced
    ax.plot(ew_ref_pv.index, ew_ref_pv / 1e9,
            color='grey', lw=1.5, ls='--',
            label='EW Rebalanced')
    # Signal
    label = f"Signal3S_t{thresh}_monthly_TC0.5%"
    pv    = signal_results[label]['portfolio_value']
    ax.plot(pv.index, pv / 1e9,
            color='steelblue', lw=1.8,
            label=f'Signal (t={thresh})')

    ax.axhline(1.0, color='black', ls=':', lw=0.8)
    ax.set_title(f"Threshold = {thresh}", fontsize=11)
    ax.set_ylabel("Value ($B)")
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("signal_3stage_by_threshold.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Figure 3: Drawdowns per threshold vs both benchmarks ─────────────────────
fig, axes = plt.subplots(1, 3, figsize=(22, 5), sharey=True)
fig.suptitle(
    "Three-Stage Signal — Drawdowns vs Benchmarks (TC=0.5%, Monthly)",
    fontsize=13, fontweight='bold')

bh_dd  = (bh_pv  - bh_pv.cummax())  / bh_pv.cummax()  * 100
ew_dd  = (ew_ref_pv - ew_ref_pv.cummax()) / ew_ref_pv.cummax() * 100

for ax, thresh in zip(axes, THRESHOLDS):
    label  = f"Signal3S_t{thresh}_monthly_TC0.5%"
    pv     = signal_results[label]['portfolio_value']
    dd     = (pv - pv.cummax()) / pv.cummax() * 100

    ax.fill_between(dd.index,    dd.values,    0, alpha=0.25, color='steelblue')
    ax.fill_between(ew_dd.index, ew_dd.values, 0, alpha=0.15, color='grey')
    ax.fill_between(bh_dd.index, bh_dd.values, 0, alpha=0.15, color='forestgreen')

    ax.plot(dd.index,    dd.values,    color='steelblue',   lw=1.2,
            label=f'Signal (t={thresh})')
    ax.plot(ew_dd.index, ew_dd.values, color='grey',        lw=1.0, ls='--',
            label='EW Rebalanced')
    ax.plot(bh_dd.index, bh_dd.values, color='forestgreen', lw=1.0, ls='-.',
            label='Buy & Hold')

    ax.set_title(f"Threshold = {thresh}", fontsize=11)
    ax.set_ylabel("Drawdown (%)")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("signal_3stage_drawdowns.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Figure 4: TC sensitivity per threshold ────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(22, 6))
fig.suptitle("Three-Stage Signal — TC Sensitivity vs Benchmarks (Monthly)",
             fontsize=13, fontweight='bold')

for ax, thresh in zip(axes, THRESHOLDS):
    # Signal across TC levels
    for tc_label, col in colors_tc.items():
        label = f"Signal3S_t{thresh}_monthly_TC{tc_label}"
        pv    = signal_results[label]['portfolio_value']
        ax.plot(pv.index, pv / 1e9, color=col, lw=1.3,
                label=f'Signal TC={tc_label}')
    # Benchmarks
    ax.plot(ew_ref_pv.index, ew_ref_pv / 1e9,
            color='grey', lw=1.8, ls='--', label='EW Rebalanced')
    ax.plot(bh_pv.index, bh_pv / 1e9,
            color='forestgreen', lw=1.8, ls='-.', label='Buy & Hold')

    ax.set_title(f"Threshold = {thresh}", fontsize=11)
    ax.set_ylabel("Value ($B)")
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("signal_3stage_tc_sensitivity.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Figure 5: All three strategies on one panel (TC=0.5%, monthly) ────────────
fig, ax = plt.subplots(figsize=(16, 7))
fig.suptitle("Strategy Overview — Buy & Hold vs EW Rebalanced vs Signal (TC=0.5%)",
             fontsize=13, fontweight='bold')

ax.plot(bh_pv.index, bh_pv / 1e9,
        color='forestgreen', lw=2.0, ls='-.', label='Buy & Hold (EW)')
ax.plot(ew_ref_pv.index, ew_ref_pv / 1e9,
        color='grey', lw=2.0, ls='--', label='EW Rebalanced (monthly)')

thresh_colors = {0.5: 'steelblue', 1.0: 'darkorange', 1.5: 'crimson'}
for thresh, col in thresh_colors.items():
    label = f"Signal3S_t{thresh}_monthly_TC0.5%"
    pv    = signal_results[label]['portfolio_value']
    ax.plot(pv.index, pv / 1e9, color=col, lw=1.8,
            label=f'Signal t={thresh}')

ax.axhline(1.0, color='black', ls=':', lw=0.8)
ax.set_ylabel("Portfolio Value ($B)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("strategy_overview.png", dpi=150, bbox_inches='tight')
plt.show()

# ── 16. SAVE ──────────────────────────────────────────────────────────────────
for label, df in {**bh_results, **ew_results, **signal_results}.items():
    df.to_csv(f"{label}_portfolio.csv")

all_metrics = pd.DataFrame({**bh_metrics, **ew_metrics, **sig_metrics}).T
all_metrics.to_csv("all_portfolio_metrics.csv")

composite_score.to_csv("composite_signal.csv")

for thresh in THRESHOLDS:
    sig_df_t = build_signal_df(composite_score, equity_assets,
                               macro_df.index, threshold=thresh)
    sig_df_t.to_csv(f"signal_df_threshold_{thresh}.csv")

print("✓ All files saved.")


In [ ]:
# @title
# ============================================================
# SIGNAL PORTFOLIO — THREE-STAGE THRESHOLD VERSION
# Signals generated from Modified Determinant Model
# + Buy-and-Hold Benchmark
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import requests
from io import StringIO

# ── 1. LOAD ALL DATA FROM GITHUB ─────────────────────────────────────────────
GITHUB_BASE = "https://raw.githubusercontent.com/kboroz/MSFE_Capstone_Project/main/01_Data/Streamlined/"

INDEX_FILE_MAP = {
    'ASX50':       'streamlined_asx_50.csv',
    'EUROSTOXX50': 'streamlined_euro_stoxx_50.csv',
    'FTSE100':     'streamlined_ftse_100.csv',
    'Ibovespa':    'streamlined_ibovespa.csv',
    'JSE40':       'streamlined_jse_top_40.csv',
    'NIKKEI225':   'streamlined_nikkei_225.csv',
    'SP500':       'streamlined_s&p_500.csv',
}

MACRO_FILE = 'streamlined_macro.csv'

def load_csv(url):
    r = requests.get(url)
    r.raise_for_status()
    df = pd.read_csv(StringIO(r.text), index_col=0, parse_dates=True)
    df.columns = df.columns.str.replace('[^a-zA-Z0-9]', '_', regex=True)
    return df.sort_index()

# Load macro
macro_df = load_csv(GITHUB_BASE + MACRO_FILE)
macro_df = macro_df[~macro_df.index.duplicated(keep='first')].sort_index()
print(f"macro_df shape : {macro_df.shape}")
print(f"Columns        : {macro_df.columns.tolist()}")

# Load individual index files
index_data = {}
for name, fname in INDEX_FILE_MAP.items():
    try:
        df = load_csv(GITHUB_BASE + fname)
        index_data[name] = df
        print(f"✓ {name:15s} | shape={df.shape} | cols={df.columns.tolist()[:4]}")
    except Exception as e:
        print(f"✗ {name}: {e}")

# ── 2. EXTRACT PRICE SERIES ───────────────────────────────────────────────────
PRICE_COLS = ['Close', 'close', 'Price', 'price', 'Adj_Close', 'Last']

def get_price_series(df, name):
    col = next((c for c in PRICE_COLS if c in df.columns), df.columns[0])
    s   = pd.to_numeric(df[col], errors='coerce').dropna()
    print(f"  {name}: using column '{col}'")
    return s

index_prices = {}
for name, df in index_data.items():
    index_prices[name] = get_price_series(df, name)

# ── 3. MODIFIED DETERMINANT SIGNAL MODEL ─────────────────────────────────────
DET_WINDOWS    = [3, 7, 28]
WINDOW_WEIGHTS = {3: 0.33, 7: 0.34, 28: 0.33}

def compute_modified_determinant(price_series_dict, windows=DET_WINDOWS):
    prices = pd.DataFrame(price_series_dict).sort_index()
    rets   = prices.pct_change().dropna()
    det_results = {}

    for window in windows:
        det_series = []
        for i in range(window, len(rets)):
            window_rets = rets.iloc[i - window: i]
            valid_cols  = window_rets.columns[window_rets.std() > 1e-10]
            w           = window_rets[valid_cols]

            if len(valid_cols) < 2:
                det_series.append(np.nan)
                continue
            try:
                R   = w.corr().values
                I   = np.eye(len(R))
                M   = R - I
                det = np.linalg.det(M)
            except Exception:
                det = np.nan
            det_series.append(det)

        idx = rets.index[window:]
        det_results[window] = pd.Series(det_series, index=idx, name=f'det_{window}d')
        print(f"  Window {window:2d}d | computed {len(det_series)} observations")

    return det_results

print("\n── Computing Modified Determinant ──")
det_results = compute_modified_determinant(index_prices)

# ── 4. COMPOSITE SCORE ────────────────────────────────────────────────────────
def compute_composite_score(det_results, window_weights=WINDOW_WEIGHTS):
    all_series = []
    for window, series in det_results.items():
        mu    = series.rolling(252, min_periods=60).mean()
        sigma = series.rolling(252, min_periods=60).std()
        z     = (series - mu) / sigma.replace(0, np.nan)
        z.name = f'z_{window}d'
        all_series.append(z)

    z_df      = pd.concat(all_series, axis=1).dropna()
    composite = sum(
        z_df[f'z_{w}d'] * window_weights[w]
        for w in window_weights if f'z_{w}d' in z_df.columns
    )
    composite = composite / sum(window_weights.values())
    return composite, z_df

composite_score, z_df = compute_composite_score(det_results)

print(f"\nComposite score shape  : {composite_score.shape}")
print(f"Date range             : {composite_score.index[0].date()} → "
      f"{composite_score.index[-1].date()}")
print(f"Score range            : {composite_score.min():.3f} → "
      f"{composite_score.max():.3f}")

# ── 5. THREE-STAGE SIGNAL CLASSIFICATION ─────────────────────────────────────
THRESHOLDS = [0.5, 1.0, 1.5]

def classify_signal(score, threshold):
    if abs(score) > threshold:
        return +1   # RISK_ON  → defensive
    else:
        return -1   # RISK_OFF → long equity

def build_signal_df(composite_score, equity_assets, macro_dates, threshold=0.5):
    binary    = composite_score.apply(lambda x: classify_signal(x, threshold))
    signal_df = pd.DataFrame(index=macro_dates)
    for asset in equity_assets:
        signal_df[asset] = binary.reindex(macro_dates).ffill().bfill()
    return signal_df

# ── 6. ASSET UNIVERSE ─────────────────────────────────────────────────────────
EQUITY_ASSETS = list(index_prices.keys())
SAFE_BONDS    = ['US_10Y', 'UK_10Y', 'AU_10Y', 'ZA_10Y']
FX            = ['AUDUSD', 'BRLUSD', 'EURUSD', 'GBPUSD', 'JPYUSD', 'ZARUSD']
COMMODITIES   = ['Oil', 'Gold', 'Bitcoin']

EXCLUDE_ASSETS = ['Bitcoin']

ALL_ASSETS     = EQUITY_ASSETS + FX + COMMODITIES + SAFE_BONDS
ALL_ASSETS     = [a for a in ALL_ASSETS
                  if a in macro_df.columns and a not in EXCLUDE_ASSETS]

equity_assets     = [a for a in EQUITY_ASSETS if a in ALL_ASSETS]
non_equity_assets = [a for a in ALL_ASSETS if a not in equity_assets]

N           = len(ALL_ASSETS)
base_weight = 1.0 / N

print(f"\nTotal assets              : {N}")
print(f"Equity (signal-driven)    : {equity_assets}")
print(f"Non-equity (always long)  : {non_equity_assets}")
print(f"base_weight = 1/{N} = {base_weight:.4f}")

# ── 7. COUNTRY SAFE-HAVEN MAP ─────────────────────────────────────────────────
EQUITY_SAFE_HAVEN_MAP = {
    'ASX50':       ['AU_10Y', 'AUDUSD'],
    'EUROSTOXX50': ['EURUSD'],
    'FTSE100':     ['UK_10Y', 'GBPUSD'],
    'Ibovespa':    ['BRLUSD'],
    'JSE40':       ['ZA_10Y', 'ZARUSD'],
    'NIKKEI225':   ['JPYUSD'],
    'SP500':       ['US_10Y'],
}

UNIVERSE_SET = set(ALL_ASSETS)
for eq, havens in EQUITY_SAFE_HAVEN_MAP.items():
    EQUITY_SAFE_HAVEN_MAP[eq] = [h for h in havens if h in UNIVERSE_SET]

print("\nSafe-haven routing (filtered to universe):")
for eq, havens in EQUITY_SAFE_HAVEN_MAP.items():
    print(f"  {eq:15s} → {havens}")

# ── 8. BUILD RETURNS ──────────────────────────────────────────────────────────
returns_raw = macro_df[ALL_ASSETS].pct_change(fill_method=None)

for col in SAFE_BONDS:
    if col in returns_raw.columns:
        returns_raw[col] = -returns_raw[col]

if 'Oil' in macro_df.columns:
    bad_oil = macro_df['Oil'][macro_df['Oil'] <= 0].index
    for d in bad_oil:
        loc = returns_raw.index.get_loc(d)
        returns_raw.iloc[
            max(0, loc - 1): loc + 2,
            returns_raw.columns.get_loc('Oil')
        ] = 0.0

returns = returns_raw.clip(lower=-0.50, upper=1.00).fillna(0).iloc[1:]

assert not returns.isnull().any().any(), "NaNs in returns"
assert not np.isinf(returns.values).any(),  "Inf in returns"
print(f"\nReturns shape : {returns.shape}")
print("✓ Returns clean")

# ── 9. BUY-AND-HOLD BENCHMARK ────────────────────────────────────────────────
def run_buy_and_hold(returns, all_assets, initial_capital=1e9):
    """
    Equally-weighted across all assets, set weights on day 1 and never rebalance.
    One-time entry transaction cost applied at inception only.
    No further TC since there is no trading.
    Weights drift with market moves — true passive holding.
    """
    N           = len(all_assets)
    base_weight = 1.0 / N

    # Initialise dollar holdings
    pv       = float(initial_capital)
    holdings = {a: pv * base_weight for a in all_assets}   # dollar amounts

    records = []
    first   = True

    for date in returns.index:
        if first:
            first = False
            # Record starting value (no return on day 0)
            records.append({'date': date, 'portfolio_value': pv})
            continue

        # Update each holding by its return
        new_holdings = {}
        for a in all_assets:
            new_holdings[a] = holdings[a] * (1 + returns.loc[date, a])

        holdings = new_holdings
        pv       = sum(holdings.values())
        records.append({'date': date, 'portfolio_value': pv})

    df = pd.DataFrame(records).set_index('date')
    print(f"  Buy-and-Hold | Start: ${initial_capital/1e9:.2f}B → "
          f"End: ${df['portfolio_value'].iloc[-1]/1e9:.3f}B")
    return df

# ── 10. THREE-STAGE WEIGHT BUILDER ───────────────────────────────────────────
def build_weights_3stage(sig_row, equity_assets, non_equity_assets,
                         base_weight, date, safe_haven_map):
    raw              = {a: base_weight for a in non_equity_assets}
    safe_haven_extra = {}

    for a in equity_assets:
        if sig_row is not None and a in sig_row.index and not pd.isna(sig_row[a]):
            sig = sig_row[a]
        else:
            sig = -1

        if sig > 0:
            raw[a]  = -base_weight
            havens  = safe_haven_map.get(a, [])
            havens  = [h for h in havens if h in raw]

            if havens:
                per_haven = base_weight / len(havens)
                for h in havens:
                    safe_haven_extra[h] = safe_haven_extra.get(h, 0) + per_haven
            else:
                per_ne = base_weight / len(non_equity_assets)
                for ne in non_equity_assets:
                    safe_haven_extra[ne] = safe_haven_extra.get(ne, 0) + per_ne
        else:
            raw[a] = base_weight

    for asset, extra in safe_haven_extra.items():
        raw[asset] = raw.get(asset, 0) + extra

    total = sum(raw.values())
    if total <= 0:
        for a in raw:
            raw[a] = 1.0 / len(raw)
    else:
        for a in raw:
            raw[a] /= total

    return raw

# ── 11. EQUAL-WEIGHT REBALANCED BENCHMARK ────────────────────────────────────
def run_equal_weight(returns, all_assets, tc_rate=0.005,
                     rebal_freq='monthly', initial_capital=1e9):
    N           = len(all_assets)
    base_weight = 1.0 / N
    ew          = {a: base_weight for a in all_assets}

    freq_code     = 'MS' if rebal_freq == 'monthly' else 'QS'
    reb_dates_raw = returns.resample(freq_code).first().index
    actual_dates  = returns.index

    reb_set = set()
    for d in reb_dates_raw:
        future = actual_dates[actual_dates >= d]
        if len(future) > 0:
            reb_set.add(future[0])

    pv              = float(initial_capital)
    current_weights = None
    records         = []

    for date in actual_dates:
        if date in reb_set:
            if current_weights is not None:
                turnover = sum(abs(ew.get(a, 0) - current_weights.get(a, 0))
                               for a in set(ew) | set(current_weights)) / 2.0
                pv *= (1 - turnover * tc_rate)
            else:
                pv *= (1 - tc_rate)
            current_weights = ew.copy()

        if current_weights is not None:
            dr  = sum(current_weights.get(a, 0) * returns.loc[date, a]
                      for a in all_assets)
            pv *= (1 + dr)

        records.append({'date': date, 'portfolio_value': pv})

    return pd.DataFrame(records).set_index('date')

# ── 12. THREE-STAGE SIGNAL PORTFOLIO RUNNER ───────────────────────────────────
def run_signal_portfolio_3stage(returns, signal_df, equity_assets,
                                non_equity_assets, all_assets,
                                freq='monthly', tc_rate=0.005,
                                safe_haven_map=None,
                                initial_capital=1e9):
    if safe_haven_map is None:
        safe_haven_map = {}

    N           = len(all_assets)
    base_weight = 1.0 / N

    freq_code     = 'MS' if freq == 'monthly' else 'QS'
    reb_dates_raw = returns.resample(freq_code).first().index
    actual_dates  = returns.index

    reb_set = set()
    for d in reb_dates_raw:
        future = actual_dates[actual_dates >= d]
        if len(future) > 0:
            reb_set.add(future[0])

    sig = signal_df.copy()
    sig.index = pd.to_datetime(sig.index)

    pv              = float(initial_capital)
    current_weights = None
    records         = []

    for date in actual_dates:
        if date in reb_set:
            past    = sig[sig.index <= date]
            sig_row = past.iloc[-1] if len(past) > 0 else None

            raw = build_weights_3stage(
                sig_row, equity_assets, non_equity_assets,
                base_weight, date, safe_haven_map
            )

            if current_weights is not None:
                all_keys = set(raw) | set(current_weights)
                turnover = sum(
                    abs(raw.get(a, 0) - current_weights.get(a, 0))
                    for a in all_keys
                ) / 2.0
                pv *= (1 - turnover * tc_rate)
            else:
                pv *= (1 - tc_rate)

            current_weights = raw.copy()

        if current_weights is not None:
            dr  = sum(current_weights.get(a, 0) * returns.loc[date, a]
                      for a in all_assets)
            pv *= (1 + dr)

        records.append({'date': date, 'portfolio_value': pv})

    return pd.DataFrame(records).set_index('date')

# ── 13. RUN ALL PORTFOLIOS ────────────────────────────────────────────────────
TC_LEVELS = {'0%': 0.00, '0.5%': 0.005, '1%': 0.01, '2%': 0.02}
FREQS     = ['monthly', 'quarterly']

# ── Buy-and-Hold (single run, no TC variants needed) ─────────────────────────
print("\n── Running Buy-and-Hold ──")
bh_result  = run_buy_and_hold(returns, ALL_ASSETS, initial_capital=1e9)
bh_results = {'BuyAndHold_EW': bh_result}   # dict for uniform handling

# ── Equal-weight rebalanced benchmarks ───────────────────────────────────────
print("\n── Running Equal-Weight Rebalanced ──")
ew_results = {}
for freq in FREQS:
    for tc_label, tc_rate in TC_LEVELS.items():
        label = f"EW_{freq}_TC{tc_label}"
        ew_results[label] = run_equal_weight(
            returns, ALL_ASSETS, tc_rate=tc_rate,
            rebal_freq=freq, initial_capital=1e9
        )
        print(f"  ✓ {label}")

# ── Three-stage signal portfolios ─────────────────────────────────────────────
print("\n── Running Signal Portfolios ──")
signal_results       = {}
signal_distributions = {}

for thresh in THRESHOLDS:
    sig_df_t   = build_signal_df(composite_score, equity_assets,
                                 macro_df.index, threshold=thresh)
    n_risk_on  = (sig_df_t[equity_assets[0]] == +1).sum()
    n_risk_off = (sig_df_t[equity_assets[0]] == -1).sum()
    total      = len(sig_df_t)

    print(f"\n  Threshold={thresh} | RISK_ON(+1): {n_risk_on} days "
          f"({n_risk_on/total*100:.1f}%) | "
          f"RISK_OFF(-1): {n_risk_off} days ({n_risk_off/total*100:.1f}%)")

    signal_distributions[thresh] = {'RISK_ON': n_risk_on, 'RISK_OFF': n_risk_off}

    for freq in FREQS:
        for tc_label, tc_rate in TC_LEVELS.items():
            label = f"Signal3S_t{thresh}_{freq}_TC{tc_label}"
            signal_results[label] = run_signal_portfolio_3stage(
                returns, sig_df_t, equity_assets, non_equity_assets,
                ALL_ASSETS, freq=freq, tc_rate=tc_rate,
                safe_haven_map=EQUITY_SAFE_HAVEN_MAP,
                initial_capital=1e9
            )
            print(f"    ✓ {label}")

print("\n✓ All portfolios computed")

# ── 14. PERFORMANCE METRICS ───────────────────────────────────────────────────
def compute_metrics(pv_df, label=''):
    pv        = pv_df['portfolio_value']
    daily_ret = pv.pct_change().dropna()
    n_years   = len(daily_ret) / 252
    total_ret = (pv.iloc[-1] / pv.iloc[0]) - 1
    cagr      = (1 + total_ret) ** (1 / n_years) - 1 if n_years > 0 else np.nan
    vol       = daily_ret.std() * np.sqrt(252)
    sharpe    = cagr / vol if vol > 0 else np.nan
    roll_max  = pv.cummax()
    max_dd    = ((pv - roll_max) / roll_max).min()
    calmar    = cagr / abs(max_dd) if max_dd != 0 else np.nan
    var_95    = daily_ret.quantile(0.05)

    return {
        'CAGR':         f"{cagr*100:.2f}%",
        'Volatility':   f"{vol*100:.2f}%",
        'Sharpe':       f"{sharpe:.2f}",
        'Max Drawdown': f"{max_dd*100:.2f}%",
        'Calmar':       f"{calmar:.2f}",
        'VaR 95%':      f"{var_95*100:.2f}%",
        'Total Return': f"{total_ret*100:.2f}%",
        'Final ($B)':   f"{pv.iloc[-1]/1e9:.3f}",
    }

# ── Print metrics ─────────────────────────────────────────────────────────────
bh_metrics  = {k: compute_metrics(v, k) for k, v in bh_results.items()}
ew_metrics  = {k: compute_metrics(v, k) for k, v in ew_results.items()}
sig_metrics = {k: compute_metrics(v, k) for k, v in signal_results.items()}

print("\n" + "="*90)
print("BUY-AND-HOLD BENCHMARK (EW, No Rebalancing)")
print("="*90)
print(pd.DataFrame(bh_metrics).T.to_string())

print("\n" + "="*90)
print("EQUAL-WEIGHT REBALANCED BENCHMARK")
print("="*90)
print(pd.DataFrame(ew_metrics).T.to_string())

for thresh in THRESHOLDS:
    sub = {k: v for k, v in sig_metrics.items() if f"_t{thresh}_" in k}
    print(f"\n{'='*90}")
    print(f"THREE-STAGE SIGNAL — Threshold = {thresh} | "
          f"RISK_ON: {signal_distributions[thresh]['RISK_ON']} days | "
          f"RISK_OFF: {signal_distributions[thresh]['RISK_OFF']} days")
    print("="*90)
    print(pd.DataFrame(sub).T.to_string())

# ── 15. PLOTS ─────────────────────────────────────────────────────────────────
colors_tc = {'0%': 'black', '0.5%': 'steelblue', '1%': 'darkorange', '2%': 'crimson'}

# Convenience references reused across figures
bh_pv      = bh_results['BuyAndHold_EW']['portfolio_value']
ew_ref_pv  = ew_results['EW_monthly_TC0.5%']['portfolio_value']

# ── Figure 1: Composite signal overview ──────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(16, 10))
fig.suptitle("Modified Determinant — Composite Score & Threshold Signals",
             fontsize=14, fontweight='bold')

ax = axes[0]
ax.plot(composite_score.index, composite_score.values,
        color='navy', lw=1.0, alpha=0.8, label='Composite Z-score')
for thresh, col in zip(THRESHOLDS, ['green', 'orange', 'red']):
    ax.axhline( thresh, color=col, ls='--', lw=1.0, label=f'+{thresh}')
    ax.axhline(-thresh, color=col, ls='--', lw=1.0, label=f'-{thresh}')
ax.axhline(0, color='black', lw=0.6)
ax.fill_between(composite_score.index, composite_score.values, 0,
                where=(composite_score > 0),  color='green', alpha=0.1)
ax.fill_between(composite_score.index, composite_score.values, 0,
                where=(composite_score <= 0), color='red',   alpha=0.1)
ax.set_title("Composite Determinant Score with Thresholds", fontsize=11)
ax.set_ylabel("Z-score")
ax.legend(fontsize=8, ncol=4)
ax.grid(True, alpha=0.3)

ax = axes[1]
for window, series in det_results.items():
    ax.plot(series.index, np.sign(series.values), alpha=0.6,
            label=f"{window}d window", lw=1.0)
ax.axhline(0, color='black', lw=0.8)
ax.set_title("Sign of Det(R-I) per Window", fontsize=11)
ax.set_ylabel("+1 / -1")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("composite_signal_thresholds.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Figure 2: Portfolio values per threshold vs both benchmarks ───────────────
fig, axes = plt.subplots(1, 3, figsize=(22, 6), sharey=False)
fig.suptitle(
    "Three-Stage Signal vs Benchmarks — Portfolio Value (TC=0.5%, Monthly)",
    fontsize=13, fontweight='bold')

for ax, thresh in zip(axes, THRESHOLDS):
    # Buy-and-hold
    ax.plot(bh_pv.index, bh_pv / 1e9,
            color='forestgreen', lw=1.8, ls='-.',
            label='Buy & Hold (EW)')
    # EW rebalanced
    ax.plot(ew_ref_pv.index, ew_ref_pv / 1e9,
            color='grey', lw=1.5, ls='--',
            label='EW Rebalanced')
    # Signal
    label = f"Signal3S_t{thresh}_monthly_TC0.5%"
    pv    = signal_results[label]['portfolio_value']
    ax.plot(pv.index, pv / 1e9,
            color='steelblue', lw=1.8,
            label=f'Signal (t={thresh})')

    ax.axhline(1.0, color='black', ls=':', lw=0.8)
    ax.set_title(f"Threshold = {thresh}", fontsize=11)
    ax.set_ylabel("Value ($B)")
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("signal_3stage_by_threshold.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Figure 3: Drawdowns per threshold vs both benchmarks ─────────────────────
fig, axes = plt.subplots(1, 3, figsize=(22, 5), sharey=True)
fig.suptitle(
    "Three-Stage Signal — Drawdowns vs Benchmarks (TC=0.5%, Monthly)",
    fontsize=13, fontweight='bold')

bh_dd  = (bh_pv  - bh_pv.cummax())  / bh_pv.cummax()  * 100
ew_dd  = (ew_ref_pv - ew_ref_pv.cummax()) / ew_ref_pv.cummax() * 100

for ax, thresh in zip(axes, THRESHOLDS):
    label  = f"Signal3S_t{thresh}_monthly_TC0.5%"
    pv     = signal_results[label]['portfolio_value']
    dd     = (pv - pv.cummax()) / pv.cummax() * 100

    ax.fill_between(dd.index,    dd.values,    0, alpha=0.25, color='steelblue')
    ax.fill_between(ew_dd.index, ew_dd.values, 0, alpha=0.15, color='grey')
    ax.fill_between(bh_dd.index, bh_dd.values, 0, alpha=0.15, color='forestgreen')

    ax.plot(dd.index,    dd.values,    color='steelblue',   lw=1.2,
            label=f'Signal (t={thresh})')
    ax.plot(ew_dd.index, ew_dd.values, color='grey',        lw=1.0, ls='--',
            label='EW Rebalanced')
    ax.plot(bh_dd.index, bh_dd.values, color='forestgreen', lw=1.0, ls='-.',
            label='Buy & Hold')

    ax.set_title(f"Threshold = {thresh}", fontsize=11)
    ax.set_ylabel("Drawdown (%)")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("signal_3stage_drawdowns.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Figure 4: TC sensitivity per threshold ────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(22, 6))
fig.suptitle("Three-Stage Signal — TC Sensitivity vs Benchmarks (Monthly)",
             fontsize=13, fontweight='bold')

for ax, thresh in zip(axes, THRESHOLDS):
    # Signal across TC levels
    for tc_label, col in colors_tc.items():
        label = f"Signal3S_t{thresh}_monthly_TC{tc_label}"
        pv    = signal_results[label]['portfolio_value']
        ax.plot(pv.index, pv / 1e9, color=col, lw=1.3,
                label=f'Signal TC={tc_label}')
    # Benchmarks
    ax.plot(ew_ref_pv.index, ew_ref_pv / 1e9,
            color='grey', lw=1.8, ls='--', label='EW Rebalanced')
    ax.plot(bh_pv.index, bh_pv / 1e9,
            color='forestgreen', lw=1.8, ls='-.', label='Buy & Hold')

    ax.set_title(f"Threshold = {thresh}", fontsize=11)
    ax.set_ylabel("Value ($B)")
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("signal_3stage_tc_sensitivity.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Figure 5: All three strategies on one panel (TC=0.5%, monthly) ────────────
fig, ax = plt.subplots(figsize=(16, 7))
fig.suptitle("Strategy Overview — Buy & Hold vs EW Rebalanced vs Signal (TC=0.5%)",
             fontsize=13, fontweight='bold')

ax.plot(bh_pv.index, bh_pv / 1e9,
        color='forestgreen', lw=2.0, ls='-.', label='Buy & Hold (EW)')
ax.plot(ew_ref_pv.index, ew_ref_pv / 1e9,
        color='grey', lw=2.0, ls='--', label='EW Rebalanced (monthly)')

thresh_colors = {0.5: 'steelblue', 1.0: 'darkorange', 1.5: 'crimson'}
for thresh, col in thresh_colors.items():
    label = f"Signal3S_t{thresh}_monthly_TC0.5%"
    pv    = signal_results[label]['portfolio_value']
    ax.plot(pv.index, pv / 1e9, color=col, lw=1.8,
            label=f'Signal t={thresh}')

ax.axhline(1.0, color='black', ls=':', lw=0.8)
ax.set_ylabel("Portfolio Value ($B)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("strategy_overview.png", dpi=150, bbox_inches='tight')
plt.show()

# ── 16. SAVE ──────────────────────────────────────────────────────────────────
for label, df in {**bh_results, **ew_results, **signal_results}.items():
    df.to_csv(f"{label}_portfolio.csv")

all_metrics = pd.DataFrame({**bh_metrics, **ew_metrics, **sig_metrics}).T
all_metrics.to_csv("all_portfolio_metrics.csv")

composite_score.to_csv("composite_signal.csv")

for thresh in THRESHOLDS:
    sig_df_t = build_signal_df(composite_score, equity_assets,
                               macro_df.index, threshold=thresh)
    sig_df_t.to_csv(f"signal_df_threshold_{thresh}.csv")

print("✓ All files saved.")


In [ ]:
# @title
# ============================================================
# SIGNAL PORTFOLIO — THREE-STAGE THRESHOLD VERSION
# Signals generated from Modified Determinant Model
# + Buy-and-Hold Benchmark
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import requests
from io import StringIO

# ── 1. LOAD ALL DATA FROM GITHUB ─────────────────────────────────────────────
GITHUB_BASE = "https://raw.githubusercontent.com/kboroz/MSFE_Capstone_Project/main/01_Data/Streamlined/"

INDEX_FILE_MAP = {
    'ASX50':       'streamlined_asx_50.csv',
    'EUROSTOXX50': 'streamlined_euro_stoxx_50.csv',
    'FTSE100':     'streamlined_ftse_100.csv',
    'Ibovespa':    'streamlined_ibovespa.csv',
    'JSE40':       'streamlined_jse_top_40.csv',
    'NIKKEI225':   'streamlined_nikkei_225.csv',
    'SP500':       'streamlined_s&p_500.csv',
}

MACRO_FILE = 'streamlined_macro.csv'

def load_csv(url):
    r = requests.get(url)
    r.raise_for_status()
    df = pd.read_csv(StringIO(r.text), index_col=0, parse_dates=True)
    df.columns = df.columns.str.replace('[^a-zA-Z0-9]', '_', regex=True)
    return df.sort_index()

# Load macro
macro_df = load_csv(GITHUB_BASE + MACRO_FILE)
macro_df = macro_df[~macro_df.index.duplicated(keep='first')].sort_index()
print(f"macro_df shape : {macro_df.shape}")
print(f"Columns        : {macro_df.columns.tolist()}")

# Load individual index files
index_data = {}
for name, fname in INDEX_FILE_MAP.items():
    try:
        df = load_csv(GITHUB_BASE + fname)
        index_data[name] = df
        print(f"✓ {name:15s} | shape={df.shape} | cols={df.columns.tolist()[:4]}")
    except Exception as e:
        print(f"✗ {name}: {e}")

# ── 2. EXTRACT PRICE SERIES ───────────────────────────────────────────────────
PRICE_COLS = ['Close', 'close', 'Price', 'price', 'Adj_Close', 'Last']

def get_price_series(df, name):
    col = next((c for c in PRICE_COLS if c in df.columns), df.columns[0])
    s   = pd.to_numeric(df[col], errors='coerce').dropna()
    print(f"  {name}: using column '{col}'")
    return s

index_prices = {}
for name, df in index_data.items():
    index_prices[name] = get_price_series(df, name)

# ── 3. MODIFIED DETERMINANT SIGNAL MODEL ─────────────────────────────────────
DET_WINDOWS    = [3, 7, 28]
WINDOW_WEIGHTS = {3: 0.33, 7: 0.34, 28: 0.33}

def compute_modified_determinant(price_series_dict, windows=DET_WINDOWS):
    prices = pd.DataFrame(price_series_dict).sort_index()
    rets   = prices.pct_change().dropna()
    det_results = {}

    for window in windows:
        det_series = []
        for i in range(window, len(rets)):
            window_rets = rets.iloc[i - window: i]
            valid_cols  = window_rets.columns[window_rets.std() > 1e-10]
            w           = window_rets[valid_cols]

            if len(valid_cols) < 2:
                det_series.append(np.nan)
                continue
            try:
                R   = w.corr().values
                I   = np.eye(len(R))
                M   = R - I
                det = np.linalg.det(M)
            except Exception:
                det = np.nan
            det_series.append(det)

        idx = rets.index[window:]
        det_results[window] = pd.Series(det_series, index=idx, name=f'det_{window}d')
        print(f"  Window {window:2d}d | computed {len(det_series)} observations")

    return det_results

print("\n── Computing Modified Determinant ──")
det_results = compute_modified_determinant(index_prices)

# ── 4. COMPOSITE SCORE ────────────────────────────────────────────────────────
def compute_composite_score(det_results, window_weights=WINDOW_WEIGHTS):
    all_series = []
    for window, series in det_results.items():
        mu    = series.rolling(252, min_periods=60).mean()
        sigma = series.rolling(252, min_periods=60).std()
        z     = (series - mu) / sigma.replace(0, np.nan)
        z.name = f'z_{window}d'
        all_series.append(z)

    z_df      = pd.concat(all_series, axis=1).dropna()
    composite = sum(
        z_df[f'z_{w}d'] * window_weights[w]
        for w in window_weights if f'z_{w}d' in z_df.columns
    )
    composite = composite / sum(window_weights.values())
    return composite, z_df

composite_score, z_df = compute_composite_score(det_results)

print(f"\nComposite score shape  : {composite_score.shape}")
print(f"Date range             : {composite_score.index[0].date()} → "
      f"{composite_score.index[-1].date()}")
print(f"Score range            : {composite_score.min():.3f} → "
      f"{composite_score.max():.3f}")

# ── 5. THREE-STAGE SIGNAL CLASSIFICATION ─────────────────────────────────────
THRESHOLDS = [0.5, 1.0, 1.5]

def classify_signal(score, threshold):
    if abs(score) > threshold:
        return +1   # RISK_ON  → defensive
    else:
        return -1   # RISK_OFF → long equity

def build_signal_df(composite_score, equity_assets, macro_dates, threshold=0.5):
    binary    = composite_score.apply(lambda x: classify_signal(x, threshold))
    signal_df = pd.DataFrame(index=macro_dates)
    for asset in equity_assets:
        signal_df[asset] = binary.reindex(macro_dates).ffill().bfill()
    return signal_df

# ── 6. ASSET UNIVERSE ─────────────────────────────────────────────────────────
EQUITY_ASSETS = list(index_prices.keys())
SAFE_BONDS    = ['US_10Y', 'UK_10Y', 'AU_10Y', 'ZA_10Y']
FX            = ['AUDUSD', 'BRLUSD', 'EURUSD', 'GBPUSD', 'JPYUSD', 'ZARUSD']
COMMODITIES   = ['Oil', 'Gold', 'Bitcoin']

EXCLUDE_ASSETS = ['Bitcoin']

ALL_ASSETS     = EQUITY_ASSETS + FX + COMMODITIES + SAFE_BONDS
ALL_ASSETS     = [a for a in ALL_ASSETS
                  if a in macro_df.columns and a not in EXCLUDE_ASSETS]

equity_assets     = [a for a in EQUITY_ASSETS if a in ALL_ASSETS]
non_equity_assets = [a for a in ALL_ASSETS if a not in equity_assets]

N           = len(ALL_ASSETS)
base_weight = 1.0 / N

print(f"\nTotal assets              : {N}")
print(f"Equity (signal-driven)    : {equity_assets}")
print(f"Non-equity (always long)  : {non_equity_assets}")
print(f"base_weight = 1/{N} = {base_weight:.4f}")

# ── 7. COUNTRY SAFE-HAVEN MAP ─────────────────────────────────────────────────
EQUITY_SAFE_HAVEN_MAP = {
    'ASX50':       ['AU_10Y', 'AUDUSD'],
    'EUROSTOXX50': ['EURUSD'],
    'FTSE100':     ['UK_10Y', 'GBPUSD'],
    'Ibovespa':    ['BRLUSD'],
    'JSE40':       ['ZA_10Y', 'ZARUSD'],
    'NIKKEI225':   ['JPYUSD'],
    'SP500':       ['US_10Y'],
}

UNIVERSE_SET = set(ALL_ASSETS)
for eq, havens in EQUITY_SAFE_HAVEN_MAP.items():
    EQUITY_SAFE_HAVEN_MAP[eq] = [h for h in havens if h in UNIVERSE_SET]

print("\nSafe-haven routing (filtered to universe):")
for eq, havens in EQUITY_SAFE_HAVEN_MAP.items():
    print(f"  {eq:15s} → {havens}")

# ── 8. BUILD RETURNS ──────────────────────────────────────────────────────────
returns_raw = macro_df[ALL_ASSETS].pct_change(fill_method=None)

for col in SAFE_BONDS:
    if col in returns_raw.columns:
        returns_raw[col] = -returns_raw[col]

if 'Oil' in macro_df.columns:
    bad_oil = macro_df['Oil'][macro_df['Oil'] <= 0].index
    for d in bad_oil:
        loc = returns_raw.index.get_loc(d)
        returns_raw.iloc[
            max(0, loc - 1): loc + 2,
            returns_raw.columns.get_loc('Oil')
        ] = 0.0

returns = returns_raw.clip(lower=-0.50, upper=1.00).fillna(0).iloc[1:]

assert not returns.isnull().any().any(), "NaNs in returns"
assert not np.isinf(returns.values).any(),  "Inf in returns"
print(f"\nReturns shape : {returns.shape}")
print("✓ Returns clean")

# ── 9. BUY-AND-HOLD BENCHMARK ────────────────────────────────────────────────
def run_buy_and_hold(returns, all_assets, initial_capital=1e9):
    """
    Equally-weighted across all assets, set weights on day 1 and never rebalance.
    One-time entry transaction cost applied at inception only.
    No further TC since there is no trading.
    Weights drift with market moves — true passive holding.
    """
    N           = len(all_assets)
    base_weight = 1.0 / N

    # Initialise dollar holdings
    pv       = float(initial_capital)
    holdings = {a: pv * base_weight for a in all_assets}   # dollar amounts

    records = []
    first   = True

    for date in returns.index:
        if first:
            first = False
            # Record starting value (no return on day 0)
            records.append({'date': date, 'portfolio_value': pv})
            continue

        # Update each holding by its return
        new_holdings = {}
        for a in all_assets:
            new_holdings[a] = holdings[a] * (1 + returns.loc[date, a])

        holdings = new_holdings
        pv       = sum(holdings.values())
        records.append({'date': date, 'portfolio_value': pv})

    df = pd.DataFrame(records).set_index('date')
    print(f"  Buy-and-Hold | Start: ${initial_capital/1e9:.2f}B → "
          f"End: ${df['portfolio_value'].iloc[-1]/1e9:.3f}B")
    return df

# ── 10. THREE-STAGE WEIGHT BUILDER ───────────────────────────────────────────
def build_weights_3stage(sig_row, equity_assets, non_equity_assets,
                         base_weight, date, safe_haven_map):
    raw              = {a: base_weight for a in non_equity_assets}
    safe_haven_extra = {}

    for a in equity_assets:
        if sig_row is not None and a in sig_row.index and not pd.isna(sig_row[a]):
            sig = sig_row[a]
        else:
            sig = -1

        if sig > 0:
            raw[a]  = -base_weight
            havens  = safe_haven_map.get(a, [])
            havens  = [h for h in havens if h in raw]

            if havens:
                per_haven = base_weight / len(havens)
                for h in havens:
                    safe_haven_extra[h] = safe_haven_extra.get(h, 0) + per_haven
            else:
                per_ne = base_weight / len(non_equity_assets)
                for ne in non_equity_assets:
                    safe_haven_extra[ne] = safe_haven_extra.get(ne, 0) + per_ne
        else:
            raw[a] = base_weight

    for asset, extra in safe_haven_extra.items():
        raw[asset] = raw.get(asset, 0) + extra

    total = sum(raw.values())
    if total <= 0:
        for a in raw:
            raw[a] = 1.0 / len(raw)
    else:
        for a in raw:
            raw[a] /= total

    return raw

# ── 11. EQUAL-WEIGHT REBALANCED BENCHMARK ────────────────────────────────────
def run_equal_weight(returns, all_assets, tc_rate=0.005,
                     rebal_freq='monthly', initial_capital=1e9):
    N           = len(all_assets)
    base_weight = 1.0 / N
    ew          = {a: base_weight for a in all_assets}

    freq_code     = 'MS' if rebal_freq == 'monthly' else 'QS'
    reb_dates_raw = returns.resample(freq_code).first().index
    actual_dates  = returns.index

    reb_set = set()
    for d in reb_dates_raw:
        future = actual_dates[actual_dates >= d]
        if len(future) > 0:
            reb_set.add(future[0])

    pv              = float(initial_capital)
    current_weights = None
    records         = []

    for date in actual_dates:
        if date in reb_set:
            if current_weights is not None:
                turnover = sum(abs(ew.get(a, 0) - current_weights.get(a, 0))
                               for a in set(ew) | set(current_weights)) / 2.0
                pv *= (1 - turnover * tc_rate)
            else:
                pv *= (1 - tc_rate)
            current_weights = ew.copy()

        if current_weights is not None:
            dr  = sum(current_weights.get(a, 0) * returns.loc[date, a]
                      for a in all_assets)
            pv *= (1 + dr)

        records.append({'date': date, 'portfolio_value': pv})

    return pd.DataFrame(records).set_index('date')

# ── 12. THREE-STAGE SIGNAL PORTFOLIO RUNNER ───────────────────────────────────
def run_signal_portfolio_3stage(returns, signal_df, equity_assets,
                                non_equity_assets, all_assets,
                                freq='monthly', tc_rate=0.005,
                                safe_haven_map=None,
                                initial_capital=1e9):
    if safe_haven_map is None:
        safe_haven_map = {}

    N           = len(all_assets)
    base_weight = 1.0 / N

    freq_code     = 'MS' if freq == 'monthly' else 'QS'
    reb_dates_raw = returns.resample(freq_code).first().index
    actual_dates  = returns.index

    reb_set = set()
    for d in reb_dates_raw:
        future = actual_dates[actual_dates >= d]
        if len(future) > 0:
            reb_set.add(future[0])

    sig = signal_df.copy()
    sig.index = pd.to_datetime(sig.index)

    pv              = float(initial_capital)
    current_weights = None
    records         = []

    for date in actual_dates:
        if date in reb_set:
            past    = sig[sig.index <= date]
            sig_row = past.iloc[-1] if len(past) > 0 else None

            raw = build_weights_3stage(
                sig_row, equity_assets, non_equity_assets,
                base_weight, date, safe_haven_map
            )

            if current_weights is not None:
                all_keys = set(raw) | set(current_weights)
                turnover = sum(
                    abs(raw.get(a, 0) - current_weights.get(a, 0))
                    for a in all_keys
                ) / 2.0
                pv *= (1 - turnover * tc_rate)
            else:
                pv *= (1 - tc_rate)

            current_weights = raw.copy()

        if current_weights is not None:
            dr  = sum(current_weights.get(a, 0) * returns.loc[date, a]
                      for a in all_assets)
            pv *= (1 + dr)

        records.append({'date': date, 'portfolio_value': pv})

    return pd.DataFrame(records).set_index('date')

# ── 13. RUN ALL PORTFOLIOS ────────────────────────────────────────────────────
TC_LEVELS = {'0%': 0.00, '0.5%': 0.005, '1%': 0.01, '2%': 0.02}
FREQS     = ['monthly', 'quarterly']

# ── Buy-and-Hold (single run, no TC variants needed) ─────────────────────────
print("\n── Running Buy-and-Hold ──")
bh_result  = run_buy_and_hold(returns, ALL_ASSETS, initial_capital=1e9)
bh_results = {'BuyAndHold_EW': bh_result}   # dict for uniform handling

# ── Equal-weight rebalanced benchmarks ───────────────────────────────────────
print("\n── Running Equal-Weight Rebalanced ──")
ew_results = {}
for freq in FREQS:
    for tc_label, tc_rate in TC_LEVELS.items():
        label = f"EW_{freq}_TC{tc_label}"
        ew_results[label] = run_equal_weight(
            returns, ALL_ASSETS, tc_rate=tc_rate,
            rebal_freq=freq, initial_capital=1e9
        )
        print(f"  ✓ {label}")

# ── Three-stage signal portfolios ─────────────────────────────────────────────
print("\n── Running Signal Portfolios ──")
signal_results       = {}
signal_distributions = {}

for thresh in THRESHOLDS:
    sig_df_t   = build_signal_df(composite_score, equity_assets,
                                 macro_df.index, threshold=thresh)
    n_risk_on  = (sig_df_t[equity_assets[0]] == +1).sum()
    n_risk_off = (sig_df_t[equity_assets[0]] == -1).sum()
    total      = len(sig_df_t)

    print(f"\n  Threshold={thresh} | RISK_ON(+1): {n_risk_on} days "
          f"({n_risk_on/total*100:.1f}%) | "
          f"RISK_OFF(-1): {n_risk_off} days ({n_risk_off/total*100:.1f}%)")

    signal_distributions[thresh] = {'RISK_ON': n_risk_on, 'RISK_OFF': n_risk_off}

    for freq in FREQS:
        for tc_label, tc_rate in TC_LEVELS.items():
            label = f"Signal3S_t{thresh}_{freq}_TC{tc_label}"
            signal_results[label] = run_signal_portfolio_3stage(
                returns, sig_df_t, equity_assets, non_equity_assets,
                ALL_ASSETS, freq=freq, tc_rate=tc_rate,
                safe_haven_map=EQUITY_SAFE_HAVEN_MAP,
                initial_capital=1e9
            )
            print(f"    ✓ {label}")

print("\n✓ All portfolios computed")

# ── 14. PERFORMANCE METRICS ───────────────────────────────────────────────────
def compute_metrics(pv_df, label=''):
    pv        = pv_df['portfolio_value']
    daily_ret = pv.pct_change().dropna()
    n_years   = len(daily_ret) / 252
    total_ret = (pv.iloc[-1] / pv.iloc[0]) - 1
    cagr      = (1 + total_ret) ** (1 / n_years) - 1 if n_years > 0 else np.nan
    vol       = daily_ret.std() * np.sqrt(252)
    sharpe    = cagr / vol if vol > 0 else np.nan
    roll_max  = pv.cummax()
    max_dd    = ((pv - roll_max) / roll_max).min()
    calmar    = cagr / abs(max_dd) if max_dd != 0 else np.nan
    var_95    = daily_ret.quantile(0.05)

    return {
        'CAGR':         f"{cagr*100:.2f}%",
        'Volatility':   f"{vol*100:.2f}%",
        'Sharpe':       f"{sharpe:.2f}",
        'Max Drawdown': f"{max_dd*100:.2f}%",
        'Calmar':       f"{calmar:.2f}",
        'VaR 95%':      f"{var_95*100:.2f}%",
        'Total Return': f"{total_ret*100:.2f}%",
        'Final ($B)':   f"{pv.iloc[-1]/1e9:.3f}",
    }

# ── Print metrics ─────────────────────────────────────────────────────────────
bh_metrics  = {k: compute_metrics(v, k) for k, v in bh_results.items()}
ew_metrics  = {k: compute_metrics(v, k) for k, v in ew_results.items()}
sig_metrics = {k: compute_metrics(v, k) for k, v in signal_results.items()}

print("\n" + "="*90)
print("BUY-AND-HOLD BENCHMARK (EW, No Rebalancing)")
print("="*90)
print(pd.DataFrame(bh_metrics).T.to_string())

print("\n" + "="*90)
print("EQUAL-WEIGHT REBALANCED BENCHMARK")
print("="*90)
print(pd.DataFrame(ew_metrics).T.to_string())

for thresh in THRESHOLDS:
    sub = {k: v for k, v in sig_metrics.items() if f"_t{thresh}_" in k}
    print(f"\n{'='*90}")
    print(f"THREE-STAGE SIGNAL — Threshold = {thresh} | "
          f"RISK_ON: {signal_distributions[thresh]['RISK_ON']} days | "
          f"RISK_OFF: {signal_distributions[thresh]['RISK_OFF']} days")
    print("="*90)
    print(pd.DataFrame(sub).T.to_string())

# ── 15. PLOTS ─────────────────────────────────────────────────────────────────
colors_tc = {'0%': 'black', '0.5%': 'steelblue', '1%': 'darkorange', '2%': 'crimson'}

# Convenience references reused across figures
bh_pv      = bh_results['BuyAndHold_EW']['portfolio_value']
ew_ref_pv  = ew_results['EW_monthly_TC0.5%']['portfolio_value']

# ── Figure 1: Composite signal overview ──────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(16, 10))
fig.suptitle("Modified Determinant — Composite Score & Threshold Signals",
             fontsize=14, fontweight='bold')

ax = axes[0]
ax.plot(composite_score.index, composite_score.values,
        color='navy', lw=1.0, alpha=0.8, label='Composite Z-score')
for thresh, col in zip(THRESHOLDS, ['green', 'orange', 'red']):
    ax.axhline( thresh, color=col, ls='--', lw=1.0, label=f'+{thresh}')
    ax.axhline(-thresh, color=col, ls='--', lw=1.0, label=f'-{thresh}')
ax.axhline(0, color='black', lw=0.6)
ax.fill_between(composite_score.index, composite_score.values, 0,
                where=(composite_score > 0),  color='green', alpha=0.1)
ax.fill_between(composite_score.index, composite_score.values, 0,
                where=(composite_score <= 0), color='red',   alpha=0.1)
ax.set_title("Composite Determinant Score with Thresholds", fontsize=11)
ax.set_ylabel("Z-score")
ax.legend(fontsize=8, ncol=4)
ax.grid(True, alpha=0.3)

ax = axes[1]
for window, series in det_results.items():
    ax.plot(series.index, np.sign(series.values), alpha=0.6,
            label=f"{window}d window", lw=1.0)
ax.axhline(0, color='black', lw=0.8)
ax.set_title("Sign of Det(R-I) per Window", fontsize=11)
ax.set_ylabel("+1 / -1")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("composite_signal_thresholds.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Figure 2: Portfolio values per threshold vs both benchmarks ───────────────
fig, axes = plt.subplots(1, 3, figsize=(22, 6), sharey=False)
fig.suptitle(
    "Three-Stage Signal vs Benchmarks — Portfolio Value (TC=0.5%, Monthly)",
    fontsize=13, fontweight='bold')

for ax, thresh in zip(axes, THRESHOLDS):
    # Buy-and-hold
    ax.plot(bh_pv.index, bh_pv / 1e9,
            color='forestgreen', lw=1.8, ls='-.',
            label='Buy & Hold (EW)')
    # EW rebalanced
    ax.plot(ew_ref_pv.index, ew_ref_pv / 1e9,
            color='grey', lw=1.5, ls='--',
            label='EW Rebalanced')
    # Signal
    label = f"Signal3S_t{thresh}_monthly_TC0.5%"
    pv    = signal_results[label]['portfolio_value']
    ax.plot(pv.index, pv / 1e9,
            color='steelblue', lw=1.8,
            label=f'Signal (t={thresh})')

    ax.axhline(1.0, color='black', ls=':', lw=0.8)
    ax.set_title(f"Threshold = {thresh}", fontsize=11)
    ax.set_ylabel("Value ($B)")
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("signal_3stage_by_threshold.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Figure 3: Drawdowns per threshold vs both benchmarks ─────────────────────
fig, axes = plt.subplots(1, 3, figsize=(22, 5), sharey=True)
fig.suptitle(
    "Three-Stage Signal — Drawdowns vs Benchmarks (TC=0.5%, Monthly)",
    fontsize=13, fontweight='bold')

bh_dd  = (bh_pv  - bh_pv.cummax())  / bh_pv.cummax()  * 100
ew_dd  = (ew_ref_pv - ew_ref_pv.cummax()) / ew_ref_pv.cummax() * 100

for ax, thresh in zip(axes, THRESHOLDS):
    label  = f"Signal3S_t{thresh}_monthly_TC0.5%"
    pv     = signal_results[label]['portfolio_value']
    dd     = (pv - pv.cummax()) / pv.cummax() * 100

    ax.fill_between(dd.index,    dd.values,    0, alpha=0.25, color='steelblue')
    ax.fill_between(ew_dd.index, ew_dd.values, 0, alpha=0.15, color='grey')
    ax.fill_between(bh_dd.index, bh_dd.values, 0, alpha=0.15, color='forestgreen')

    ax.plot(dd.index,    dd.values,    color='steelblue',   lw=1.2,
            label=f'Signal (t={thresh})')
    ax.plot(ew_dd.index, ew_dd.values, color='grey',        lw=1.0, ls='--',
            label='EW Rebalanced')
    ax.plot(bh_dd.index, bh_dd.values, color='forestgreen', lw=1.0, ls='-.',
            label='Buy & Hold')

    ax.set_title(f"Threshold = {thresh}", fontsize=11)
    ax.set_ylabel("Drawdown (%)")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("signal_3stage_drawdowns.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Figure 4: TC sensitivity per threshold ────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(22, 6))
fig.suptitle("Three-Stage Signal — TC Sensitivity vs Benchmarks (Monthly)",
             fontsize=13, fontweight='bold')

for ax, thresh in zip(axes, THRESHOLDS):
    # Signal across TC levels
    for tc_label, col in colors_tc.items():
        label = f"Signal3S_t{thresh}_monthly_TC{tc_label}"
        pv    = signal_results[label]['portfolio_value']
        ax.plot(pv.index, pv / 1e9, color=col, lw=1.3,
                label=f'Signal TC={tc_label}')
    # Benchmarks
    ax.plot(ew_ref_pv.index, ew_ref_pv / 1e9,
            color='grey', lw=1.8, ls='--', label='EW Rebalanced')
    ax.plot(bh_pv.index, bh_pv / 1e9,
            color='forestgreen', lw=1.8, ls='-.', label='Buy & Hold')

    ax.set_title(f"Threshold = {thresh}", fontsize=11)
    ax.set_ylabel("Value ($B)")
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("signal_3stage_tc_sensitivity.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Figure 5: All three strategies on one panel (TC=0.5%, monthly) ────────────
fig, ax = plt.subplots(figsize=(16, 7))
fig.suptitle("Strategy Overview — Buy & Hold vs EW Rebalanced vs Signal (TC=0.5%)",
             fontsize=13, fontweight='bold')

ax.plot(bh_pv.index, bh_pv / 1e9,
        color='forestgreen', lw=2.0, ls='-.', label='Buy & Hold (EW)')
ax.plot(ew_ref_pv.index, ew_ref_pv / 1e9,
        color='grey', lw=2.0, ls='--', label='EW Rebalanced (monthly)')

thresh_colors = {0.5: 'steelblue', 1.0: 'darkorange', 1.5: 'crimson'}
for thresh, col in thresh_colors.items():
    label = f"Signal3S_t{thresh}_monthly_TC0.5%"
    pv    = signal_results[label]['portfolio_value']
    ax.plot(pv.index, pv / 1e9, color=col, lw=1.8,
            label=f'Signal t={thresh}')

ax.axhline(1.0, color='black', ls=':', lw=0.8)
ax.set_ylabel("Portfolio Value ($B)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("strategy_overview.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Figure Q1: Quarterly — Portfolio Value (all thresholds, TC=0.5%) ──────────
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.suptitle("Three-Stage Signal — Quarterly Portfolio Value (TC=0.5%)",
             fontsize=13, fontweight='bold')

for ax, thresh in zip(axes, THRESHOLDS):
    label = f"Signal3S_t{thresh}_quarterly_TC0.5%"
    pv    = signal_results[label]['portfolio_value']
    ew_pv = ew_results['EW_quarterly_TC0.5%']['portfolio_value']

    ax.plot(pv.index,    pv / 1e9,    color='steelblue', lw=1.8,
            label=f'Signal t={thresh}')
    ax.plot(ew_pv.index, ew_pv / 1e9, color='grey', lw=1.5, ls='--',
            label='EW Rebalanced')
    ax.plot(bh_pv.index, bh_pv / 1e9, color='forestgreen', lw=1.5, ls='-.',
            label='Buy & Hold')
    ax.axhline(1.0, color='black', ls=':', lw=0.8)
    ax.set_title(f"Threshold = {thresh}", fontsize=11)
    ax.set_ylabel("Value ($B)")
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("signal_quarterly_value.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Figure Q2: Quarterly — Drawdowns ─────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.suptitle("Three-Stage Signal — Quarterly Drawdowns (TC=0.5%)",
             fontsize=13, fontweight='bold')

for ax, thresh in zip(axes, THRESHOLDS):
    label  = f"Signal3S_t{thresh}_quarterly_TC0.5%"
    pv     = signal_results[label]['portfolio_value']
    dd     = (pv - pv.cummax()) / pv.cummax() * 100
    ew_pv  = ew_results['EW_quarterly_TC0.5%']['portfolio_value']
    ew_dd  = (ew_pv - ew_pv.cummax()) / ew_pv.cummax() * 100
    bh_dd  = (bh_pv - bh_pv.cummax()) / bh_pv.cummax() * 100

    ax.fill_between(dd.index,    dd.values,    0, alpha=0.3, color='steelblue')
    ax.fill_between(ew_dd.index, ew_dd.values, 0, alpha=0.2, color='grey')
    ax.fill_between(bh_dd.index, bh_dd.values, 0, alpha=0.15, color='forestgreen')
    ax.plot(dd.index,    dd.values,    color='steelblue',   lw=1.0,
            label=f'Signal t={thresh}')
    ax.plot(ew_dd.index, ew_dd.values, color='grey',        lw=1.0, ls='--',
            label='EW Rebalanced')
    ax.plot(bh_dd.index, bh_dd.values, color='forestgreen', lw=1.0, ls='-.',
            label='Buy & Hold')
    ax.set_title(f"Threshold = {thresh}", fontsize=11)
    ax.set_ylabel("Drawdown (%)")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("signal_quarterly_drawdowns.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Figure Q3: Quarterly — TC Sensitivity ────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.suptitle("Three-Stage Signal — Quarterly TC Sensitivity",
             fontsize=13, fontweight='bold')

for ax, thresh in zip(axes, THRESHOLDS):
    for tc_label, col in colors_tc.items():
        label = f"Signal3S_t{thresh}_quarterly_TC{tc_label}"
        pv    = signal_results[label]['portfolio_value']
        ax.plot(pv.index, pv / 1e9, color=col, lw=1.3,
                label=f'TC={tc_label}')
    ew_pv = ew_results['EW_quarterly_TC0.5%']['portfolio_value']
    ax.plot(ew_pv.index, ew_pv / 1e9, '--', color='grey',
            lw=1.5, label='EW Benchmark')
    ax.plot(bh_pv.index, bh_pv / 1e9, '-.', color='forestgreen',
            lw=1.5, label='Buy & Hold')
    ax.set_title(f"Threshold = {thresh}", fontsize=11)
    ax.set_ylabel("Value ($B)")
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("signal_quarterly_tc_sensitivity.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Figure Q4: Quarterly — Strategy Overview ─────────────────────────────────
fig, ax = plt.subplots(figsize=(16, 7))
fig.suptitle("Quarterly Strategy Overview — Buy & Hold vs EW vs Signal (TC=0.5%)",
             fontsize=13, fontweight='bold')

ax.plot(bh_pv.index, bh_pv / 1e9,
        color='forestgreen', lw=2.0, ls='-.', label='Buy & Hold (EW)')
ew_q_pv = ew_results['EW_quarterly_TC0.5%']['portfolio_value']
ax.plot(ew_q_pv.index, ew_q_pv / 1e9,
        color='grey', lw=2.0, ls='--', label='EW Rebalanced (quarterly)')

for thresh, col in thresh_colors.items():
    label = f"Signal3S_t{thresh}_quarterly_TC0.5%"
    pv    = signal_results[label]['portfolio_value']
    ax.plot(pv.index, pv / 1e9, color=col, lw=1.8,
            label=f'Signal t={thresh}')

ax.axhline(1.0, color='black', ls=':', lw=0.8)
ax.set_ylabel("Portfolio Value ($B)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("strategy_quarterly_overview.png", dpi=150, bbox_inches='tight')
plt.show()


# ── 16. SAVE ──────────────────────────────────────────────────────────────────
for label, df in {**bh_results, **ew_results, **signal_results}.items():
    df.to_csv(f"{label}_portfolio.csv")

all_metrics = pd.DataFrame({**bh_metrics, **ew_metrics, **sig_metrics}).T
all_metrics.to_csv("all_portfolio_metrics.csv")

composite_score.to_csv("composite_signal.csv")

for thresh in THRESHOLDS:
    sig_df_t = build_signal_df(composite_score, equity_assets,
                               macro_df.index, threshold=thresh)
    sig_df_t.to_csv(f"signal_df_threshold_{thresh}.csv")

print("✓ All files saved.")


In [ ]:
# @title
# ============================================================
# SIGNAL PORTFOLIO — THREE-STAGE THRESHOLD VERSION
# Signals generated from Modified Determinant Model
# + Buy-and-Hold Benchmark
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import requests
from io import StringIO

# ── 1. LOAD ALL DATA FROM GITHUB ─────────────────────────────────────────────
GITHUB_BASE = "https://raw.githubusercontent.com/kboroz/MSFE_Capstone_Project/main/01_Data/Streamlined/"

INDEX_FILE_MAP = {
    'ASX50':       'streamlined_asx_50.csv',
    'EUROSTOXX50': 'streamlined_euro_stoxx_50.csv',
    'FTSE100':     'streamlined_ftse_100.csv',
    'Ibovespa':    'streamlined_ibovespa.csv',
    'JSE40':       'streamlined_jse_top_40.csv',
    'NIKKEI225':   'streamlined_nikkei_225.csv',
    'SP500':       'streamlined_s&p_500.csv',
}

MACRO_FILE = 'streamlined_macro.csv'

def load_csv(url):
    r = requests.get(url)
    r.raise_for_status()
    df = pd.read_csv(StringIO(r.text), index_col=0, parse_dates=True)
    df.columns = df.columns.str.replace('[^a-zA-Z0-9]', '_', regex=True)
    return df.sort_index()

# Load macro
macro_df = load_csv(GITHUB_BASE + MACRO_FILE)
macro_df = macro_df[~macro_df.index.duplicated(keep='first')].sort_index()
print(f"macro_df shape : {macro_df.shape}")
print(f"Columns        : {macro_df.columns.tolist()}")

# Load individual index files
index_data = {}
for name, fname in INDEX_FILE_MAP.items():
    try:
        df = load_csv(GITHUB_BASE + fname)
        index_data[name] = df
        print(f"✓ {name:15s} | shape={df.shape} | cols={df.columns.tolist()[:4]}")
    except Exception as e:
        print(f"✗ {name}: {e}")

# ── 2. EXTRACT PRICE SERIES ───────────────────────────────────────────────────
PRICE_COLS = ['Close', 'close', 'Price', 'price', 'Adj_Close', 'Last']

def get_price_series(df, name):
    col = next((c for c in PRICE_COLS if c in df.columns), df.columns[0])
    s   = pd.to_numeric(df[col], errors='coerce').dropna()
    print(f"  {name}: using column '{col}'")
    return s

index_prices = {}
for name, df in index_data.items():
    index_prices[name] = get_price_series(df, name)

# ── 3. MODIFIED DETERMINANT SIGNAL MODEL ─────────────────────────────────────
DET_WINDOWS    = [3, 7, 28]
WINDOW_WEIGHTS = {3: 0.5, 7: 0.5, 28: 0.0}

def compute_modified_determinant(price_series_dict, windows=DET_WINDOWS):
    prices = pd.DataFrame(price_series_dict).sort_index()
    rets   = prices.pct_change().dropna()
    det_results = {}

    for window in windows:
        det_series = []
        for i in range(window, len(rets)):
            window_rets = rets.iloc[i - window: i]
            valid_cols  = window_rets.columns[window_rets.std() > 1e-10]
            w           = window_rets[valid_cols]

            if len(valid_cols) < 2:
                det_series.append(np.nan)
                continue
            try:
                R   = w.corr().values
                I   = np.eye(len(R))
                M   = R - I
                det = np.linalg.det(M)
            except Exception:
                det = np.nan
            det_series.append(det)

        idx = rets.index[window:]
        det_results[window] = pd.Series(det_series, index=idx, name=f'det_{window}d')
        print(f"  Window {window:2d}d | computed {len(det_series)} observations")

    return det_results

print("\n── Computing Modified Determinant ──")
det_results = compute_modified_determinant(index_prices)

# ── 4. COMPOSITE SCORE ────────────────────────────────────────────────────────
def compute_composite_score(det_results, window_weights=WINDOW_WEIGHTS):
    all_series = []
    for window, series in det_results.items():
        mu    = series.rolling(252, min_periods=60).mean()
        sigma = series.rolling(252, min_periods=60).std()
        z     = (series - mu) / sigma.replace(0, np.nan)
        z.name = f'z_{window}d'
        all_series.append(z)

    z_df      = pd.concat(all_series, axis=1).dropna()
    composite = sum(
        z_df[f'z_{w}d'] * window_weights[w]
        for w in window_weights if f'z_{w}d' in z_df.columns
    )
    composite = composite / sum(window_weights.values())
    return composite, z_df

composite_score, z_df = compute_composite_score(det_results)

print(f"\nComposite score shape  : {composite_score.shape}")
print(f"Date range             : {composite_score.index[0].date()} → "
      f"{composite_score.index[-1].date()}")
print(f"Score range            : {composite_score.min():.3f} → "
      f"{composite_score.max():.3f}")

# ── 5. THREE-STAGE SIGNAL CLASSIFICATION ─────────────────────────────────────
THRESHOLDS = [0.5, 1.0, 1.5]

def classify_signal(score, threshold):
    if abs(score) > threshold:
        return +1   # RISK_ON  → defensive
    else:
        return -1   # RISK_OFF → long equity

def build_signal_df(composite_score, equity_assets, macro_dates, threshold=0.5):
    binary    = composite_score.apply(lambda x: classify_signal(x, threshold))
    signal_df = pd.DataFrame(index=macro_dates)
    for asset in equity_assets:
        signal_df[asset] = binary.reindex(macro_dates).ffill().bfill()
    return signal_df

# ── 6. ASSET UNIVERSE ─────────────────────────────────────────────────────────
EQUITY_ASSETS = list(index_prices.keys())
SAFE_BONDS    = ['US_10Y', 'UK_10Y', 'AU_10Y', 'ZA_10Y']
FX            = ['AUDUSD', 'BRLUSD', 'EURUSD', 'GBPUSD', 'JPYUSD', 'ZARUSD']
COMMODITIES   = ['Oil', 'Gold', 'Bitcoin']

EXCLUDE_ASSETS = ['Bitcoin']

ALL_ASSETS     = EQUITY_ASSETS + FX + COMMODITIES + SAFE_BONDS
ALL_ASSETS     = [a for a in ALL_ASSETS
                  if a in macro_df.columns and a not in EXCLUDE_ASSETS]

equity_assets     = [a for a in EQUITY_ASSETS if a in ALL_ASSETS]
non_equity_assets = [a for a in ALL_ASSETS if a not in equity_assets]

N           = len(ALL_ASSETS)
base_weight = 1.0 / N

print(f"\nTotal assets              : {N}")
print(f"Equity (signal-driven)    : {equity_assets}")
print(f"Non-equity (always long)  : {non_equity_assets}")
print(f"base_weight = 1/{N} = {base_weight:.4f}")

# ── 7. COUNTRY SAFE-HAVEN MAP ─────────────────────────────────────────────────
EQUITY_SAFE_HAVEN_MAP = {
    'ASX50':       ['AU_10Y', 'AUDUSD'],
    'EUROSTOXX50': ['EURUSD'],
    'FTSE100':     ['UK_10Y', 'GBPUSD'],
    'Ibovespa':    ['BRLUSD'],
    'JSE40':       ['ZA_10Y', 'ZARUSD'],
    'NIKKEI225':   ['JPYUSD'],
    'SP500':       ['US_10Y'],
}

UNIVERSE_SET = set(ALL_ASSETS)
for eq, havens in EQUITY_SAFE_HAVEN_MAP.items():
    EQUITY_SAFE_HAVEN_MAP[eq] = [h for h in havens if h in UNIVERSE_SET]

print("\nSafe-haven routing (filtered to universe):")
for eq, havens in EQUITY_SAFE_HAVEN_MAP.items():
    print(f"  {eq:15s} → {havens}")

# ── 8. BUILD RETURNS ──────────────────────────────────────────────────────────
returns_raw = macro_df[ALL_ASSETS].pct_change(fill_method=None)

for col in SAFE_BONDS:
    if col in returns_raw.columns:
        returns_raw[col] = -returns_raw[col]

if 'Oil' in macro_df.columns:
    bad_oil = macro_df['Oil'][macro_df['Oil'] <= 0].index
    for d in bad_oil:
        loc = returns_raw.index.get_loc(d)
        returns_raw.iloc[
            max(0, loc - 1): loc + 2,
            returns_raw.columns.get_loc('Oil')
        ] = 0.0

returns = returns_raw.clip(lower=-0.50, upper=1.00).fillna(0).iloc[1:]

assert not returns.isnull().any().any(), "NaNs in returns"
assert not np.isinf(returns.values).any(),  "Inf in returns"
print(f"\nReturns shape : {returns.shape}")
print("✓ Returns clean")

# ── 9. BUY-AND-HOLD BENCHMARK ────────────────────────────────────────────────
def run_buy_and_hold(returns, all_assets, initial_capital=1e9):
    """
    Equally-weighted across all assets, set weights on day 1 and never rebalance.
    One-time entry transaction cost applied at inception only.
    No further TC since there is no trading.
    Weights drift with market moves — true passive holding.
    """
    N           = len(all_assets)
    base_weight = 1.0 / N

    # Initialise dollar holdings
    pv       = float(initial_capital)
    holdings = {a: pv * base_weight for a in all_assets}   # dollar amounts

    records = []
    first   = True

    for date in returns.index:
        if first:
            first = False
            # Record starting value (no return on day 0)
            records.append({'date': date, 'portfolio_value': pv})
            continue

        # Update each holding by its return
        new_holdings = {}
        for a in all_assets:
            new_holdings[a] = holdings[a] * (1 + returns.loc[date, a])

        holdings = new_holdings
        pv       = sum(holdings.values())
        records.append({'date': date, 'portfolio_value': pv})

    df = pd.DataFrame(records).set_index('date')
    print(f"  Buy-and-Hold | Start: ${initial_capital/1e9:.2f}B → "
          f"End: ${df['portfolio_value'].iloc[-1]/1e9:.3f}B")
    return df

# ── 10. THREE-STAGE WEIGHT BUILDER ───────────────────────────────────────────
def build_weights_3stage(sig_row, equity_assets, non_equity_assets,
                         base_weight, date, safe_haven_map):
    raw              = {a: base_weight for a in non_equity_assets}
    safe_haven_extra = {}

    for a in equity_assets:
        if sig_row is not None and a in sig_row.index and not pd.isna(sig_row[a]):
            sig = sig_row[a]
        else:
            sig = -1

        if sig > 0:
            raw[a]  = -base_weight
            havens  = safe_haven_map.get(a, [])
            havens  = [h for h in havens if h in raw]

            if havens:
                per_haven = base_weight / len(havens)
                for h in havens:
                    safe_haven_extra[h] = safe_haven_extra.get(h, 0) + per_haven
            else:
                per_ne = base_weight / len(non_equity_assets)
                for ne in non_equity_assets:
                    safe_haven_extra[ne] = safe_haven_extra.get(ne, 0) + per_ne
        else:
            raw[a] = base_weight

    for asset, extra in safe_haven_extra.items():
        raw[asset] = raw.get(asset, 0) + extra

    total = sum(raw.values())
    if total <= 0:
        for a in raw:
            raw[a] = 1.0 / len(raw)
    else:
        for a in raw:
            raw[a] /= total

    return raw

# ── 11. EQUAL-WEIGHT REBALANCED BENCHMARK ────────────────────────────────────
def run_equal_weight(returns, all_assets, tc_rate=0.005,
                     rebal_freq='monthly', initial_capital=1e9):
    N           = len(all_assets)
    base_weight = 1.0 / N
    ew          = {a: base_weight for a in all_assets}

    freq_code     = 'MS' if rebal_freq == 'monthly' else 'QS'
    reb_dates_raw = returns.resample(freq_code).first().index
    actual_dates  = returns.index

    reb_set = set()
    for d in reb_dates_raw:
        future = actual_dates[actual_dates >= d]
        if len(future) > 0:
            reb_set.add(future[0])

    pv              = float(initial_capital)
    current_weights = None
    records         = []

    for date in actual_dates:
        if date in reb_set:
            if current_weights is not None:
                turnover = sum(abs(ew.get(a, 0) - current_weights.get(a, 0))
                               for a in set(ew) | set(current_weights)) / 2.0
                pv *= (1 - turnover * tc_rate)
            else:
                pv *= (1 - tc_rate)
            current_weights = ew.copy()

        if current_weights is not None:
            dr  = sum(current_weights.get(a, 0) * returns.loc[date, a]
                      for a in all_assets)
            pv *= (1 + dr)

        records.append({'date': date, 'portfolio_value': pv})

    return pd.DataFrame(records).set_index('date')

# ── 12. THREE-STAGE SIGNAL PORTFOLIO RUNNER ───────────────────────────────────
def run_signal_portfolio_3stage(returns, signal_df, equity_assets,
                                non_equity_assets, all_assets,
                                freq='monthly', tc_rate=0.005,
                                safe_haven_map=None,
                                initial_capital=1e9):
    if safe_haven_map is None:
        safe_haven_map = {}

    N           = len(all_assets)
    base_weight = 1.0 / N

    freq_code     = 'MS' if freq == 'monthly' else 'QS'
    reb_dates_raw = returns.resample(freq_code).first().index
    actual_dates  = returns.index

    reb_set = set()
    for d in reb_dates_raw:
        future = actual_dates[actual_dates >= d]
        if len(future) > 0:
            reb_set.add(future[0])

    sig = signal_df.copy()
    sig.index = pd.to_datetime(sig.index)

    pv              = float(initial_capital)
    current_weights = None
    records         = []

    for date in actual_dates:
        if date in reb_set:
            past    = sig[sig.index <= date]
            sig_row = past.iloc[-1] if len(past) > 0 else None

            raw = build_weights_3stage(
                sig_row, equity_assets, non_equity_assets,
                base_weight, date, safe_haven_map
            )

            if current_weights is not None:
                all_keys = set(raw) | set(current_weights)
                turnover = sum(
                    abs(raw.get(a, 0) - current_weights.get(a, 0))
                    for a in all_keys
                ) / 2.0
                pv *= (1 - turnover * tc_rate)
            else:
                pv *= (1 - tc_rate)

            current_weights = raw.copy()

        if current_weights is not None:
            dr  = sum(current_weights.get(a, 0) * returns.loc[date, a]
                      for a in all_assets)
            pv *= (1 + dr)

        records.append({'date': date, 'portfolio_value': pv})

    return pd.DataFrame(records).set_index('date')

# ── 13. RUN ALL PORTFOLIOS ────────────────────────────────────────────────────
TC_LEVELS = {'0%': 0.00, '0.5%': 0.005, '1%': 0.01, '2%': 0.02}
FREQS     = ['monthly', 'quarterly']

# ── Buy-and-Hold (single run, no TC variants needed) ─────────────────────────
print("\n── Running Buy-and-Hold ──")
bh_result  = run_buy_and_hold(returns, ALL_ASSETS, initial_capital=1e9)
bh_results = {'BuyAndHold_EW': bh_result}   # dict for uniform handling

# ── Equal-weight rebalanced benchmarks ───────────────────────────────────────
print("\n── Running Equal-Weight Rebalanced ──")
ew_results = {}
for freq in FREQS:
    for tc_label, tc_rate in TC_LEVELS.items():
        label = f"EW_{freq}_TC{tc_label}"
        ew_results[label] = run_equal_weight(
            returns, ALL_ASSETS, tc_rate=tc_rate,
            rebal_freq=freq, initial_capital=1e9
        )
        print(f"  ✓ {label}")

# ── Three-stage signal portfolios ─────────────────────────────────────────────
print("\n── Running Signal Portfolios ──")
signal_results       = {}
signal_distributions = {}

for thresh in THRESHOLDS:
    sig_df_t   = build_signal_df(composite_score, equity_assets,
                                 macro_df.index, threshold=thresh)
    n_risk_on  = (sig_df_t[equity_assets[0]] == +1).sum()
    n_risk_off = (sig_df_t[equity_assets[0]] == -1).sum()
    total      = len(sig_df_t)

    print(f"\n  Threshold={thresh} | RISK_ON(+1): {n_risk_on} days "
          f"({n_risk_on/total*100:.1f}%) | "
          f"RISK_OFF(-1): {n_risk_off} days ({n_risk_off/total*100:.1f}%)")

    signal_distributions[thresh] = {'RISK_ON': n_risk_on, 'RISK_OFF': n_risk_off}

    for freq in FREQS:
        for tc_label, tc_rate in TC_LEVELS.items():
            label = f"Signal3S_t{thresh}_{freq}_TC{tc_label}"
            signal_results[label] = run_signal_portfolio_3stage(
                returns, sig_df_t, equity_assets, non_equity_assets,
                ALL_ASSETS, freq=freq, tc_rate=tc_rate,
                safe_haven_map=EQUITY_SAFE_HAVEN_MAP,
                initial_capital=1e9
            )
            print(f"    ✓ {label}")

print("\n✓ All portfolios computed")

# ── 14. PERFORMANCE METRICS ───────────────────────────────────────────────────
def compute_metrics(pv_df, label=''):
    pv        = pv_df['portfolio_value']
    daily_ret = pv.pct_change().dropna()
    n_years   = len(daily_ret) / 252
    total_ret = (pv.iloc[-1] / pv.iloc[0]) - 1
    cagr      = (1 + total_ret) ** (1 / n_years) - 1 if n_years > 0 else np.nan
    vol       = daily_ret.std() * np.sqrt(252)
    sharpe    = cagr / vol if vol > 0 else np.nan
    roll_max  = pv.cummax()
    max_dd    = ((pv - roll_max) / roll_max).min()
    calmar    = cagr / abs(max_dd) if max_dd != 0 else np.nan
    var_95    = daily_ret.quantile(0.05)

    return {
        'CAGR':         f"{cagr*100:.2f}%",
        'Volatility':   f"{vol*100:.2f}%",
        'Sharpe':       f"{sharpe:.2f}",
        'Max Drawdown': f"{max_dd*100:.2f}%",
        'Calmar':       f"{calmar:.2f}",
        'VaR 95%':      f"{var_95*100:.2f}%",
        'Total Return': f"{total_ret*100:.2f}%",
        'Final ($B)':   f"{pv.iloc[-1]/1e9:.3f}",
    }

# ── Print metrics ─────────────────────────────────────────────────────────────
bh_metrics  = {k: compute_metrics(v, k) for k, v in bh_results.items()}
ew_metrics  = {k: compute_metrics(v, k) for k, v in ew_results.items()}
sig_metrics = {k: compute_metrics(v, k) for k, v in signal_results.items()}

print("\n" + "="*90)
print("BUY-AND-HOLD BENCHMARK (EW, No Rebalancing)")
print("="*90)
print(pd.DataFrame(bh_metrics).T.to_string())

print("\n" + "="*90)
print("EQUAL-WEIGHT REBALANCED BENCHMARK")
print("="*90)
print(pd.DataFrame(ew_metrics).T.to_string())

for thresh in THRESHOLDS:
    sub = {k: v for k, v in sig_metrics.items() if f"_t{thresh}_" in k}
    print(f"\n{'='*90}")
    print(f"THREE-STAGE SIGNAL — Threshold = {thresh} | "
          f"RISK_ON: {signal_distributions[thresh]['RISK_ON']} days | "
          f"RISK_OFF: {signal_distributions[thresh]['RISK_OFF']} days")
    print("="*90)
    print(pd.DataFrame(sub).T.to_string())

# ── 15. PLOTS ─────────────────────────────────────────────────────────────────
colors_tc = {'0%': 'black', '0.5%': 'steelblue', '1%': 'darkorange', '2%': 'crimson'}

# Convenience references reused across figures
bh_pv      = bh_results['BuyAndHold_EW']['portfolio_value']
ew_ref_pv  = ew_results['EW_monthly_TC0.5%']['portfolio_value']

# ── Figure 1: Composite signal overview ──────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(16, 10))
fig.suptitle("Modified Determinant — Composite Score & Threshold Signals",
             fontsize=14, fontweight='bold')

ax = axes[0]
ax.plot(composite_score.index, composite_score.values,
        color='navy', lw=1.0, alpha=0.8, label='Composite Z-score')
for thresh, col in zip(THRESHOLDS, ['green', 'orange', 'red']):
    ax.axhline( thresh, color=col, ls='--', lw=1.0, label=f'+{thresh}')
    ax.axhline(-thresh, color=col, ls='--', lw=1.0, label=f'-{thresh}')
ax.axhline(0, color='black', lw=0.6)
ax.fill_between(composite_score.index, composite_score.values, 0,
                where=(composite_score > 0),  color='green', alpha=0.1)
ax.fill_between(composite_score.index, composite_score.values, 0,
                where=(composite_score <= 0), color='red',   alpha=0.1)
ax.set_title("Composite Determinant Score with Thresholds", fontsize=11)
ax.set_ylabel("Z-score")
ax.legend(fontsize=8, ncol=4)
ax.grid(True, alpha=0.3)

ax = axes[1]
for window, series in det_results.items():
    ax.plot(series.index, np.sign(series.values), alpha=0.6,
            label=f"{window}d window", lw=1.0)
ax.axhline(0, color='black', lw=0.8)
ax.set_title("Sign of Det(R-I) per Window", fontsize=11)
ax.set_ylabel("+1 / -1")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("composite_signal_thresholds.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Figure 2: Portfolio values per threshold vs both benchmarks ───────────────
fig, axes = plt.subplots(1, 3, figsize=(22, 6), sharey=False)
fig.suptitle(
    "Three-Stage Signal vs Benchmarks — Portfolio Value (TC=0.5%, Monthly)",
    fontsize=13, fontweight='bold')

for ax, thresh in zip(axes, THRESHOLDS):
    # Buy-and-hold
    ax.plot(bh_pv.index, bh_pv / 1e9,
            color='forestgreen', lw=1.8, ls='-.',
            label='Buy & Hold (EW)')
    # EW rebalanced
    ax.plot(ew_ref_pv.index, ew_ref_pv / 1e9,
            color='grey', lw=1.5, ls='--',
            label='EW Rebalanced')
    # Signal
    label = f"Signal3S_t{thresh}_monthly_TC0.5%"
    pv    = signal_results[label]['portfolio_value']
    ax.plot(pv.index, pv / 1e9,
            color='steelblue', lw=1.8,
            label=f'Signal (t={thresh})')

    ax.axhline(1.0, color='black', ls=':', lw=0.8)
    ax.set_title(f"Threshold = {thresh}", fontsize=11)
    ax.set_ylabel("Value ($B)")
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("signal_3stage_by_threshold.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Figure 3: Drawdowns per threshold vs both benchmarks ─────────────────────
fig, axes = plt.subplots(1, 3, figsize=(22, 5), sharey=True)
fig.suptitle(
    "Three-Stage Signal — Drawdowns vs Benchmarks (TC=0.5%, Monthly)",
    fontsize=13, fontweight='bold')

bh_dd  = (bh_pv  - bh_pv.cummax())  / bh_pv.cummax()  * 100
ew_dd  = (ew_ref_pv - ew_ref_pv.cummax()) / ew_ref_pv.cummax() * 100

for ax, thresh in zip(axes, THRESHOLDS):
    label  = f"Signal3S_t{thresh}_monthly_TC0.5%"
    pv     = signal_results[label]['portfolio_value']
    dd     = (pv - pv.cummax()) / pv.cummax() * 100

    ax.fill_between(dd.index,    dd.values,    0, alpha=0.25, color='steelblue')
    ax.fill_between(ew_dd.index, ew_dd.values, 0, alpha=0.15, color='grey')
    ax.fill_between(bh_dd.index, bh_dd.values, 0, alpha=0.15, color='forestgreen')

    ax.plot(dd.index,    dd.values,    color='steelblue',   lw=1.2,
            label=f'Signal (t={thresh})')
    ax.plot(ew_dd.index, ew_dd.values, color='grey',        lw=1.0, ls='--',
            label='EW Rebalanced')
    ax.plot(bh_dd.index, bh_dd.values, color='forestgreen', lw=1.0, ls='-.',
            label='Buy & Hold')

    ax.set_title(f"Threshold = {thresh}", fontsize=11)
    ax.set_ylabel("Drawdown (%)")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("signal_3stage_drawdowns.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Figure 4: TC sensitivity per threshold ────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(22, 6))
fig.suptitle("Three-Stage Signal — TC Sensitivity vs Benchmarks (Monthly)",
             fontsize=13, fontweight='bold')

for ax, thresh in zip(axes, THRESHOLDS):
    # Signal across TC levels
    for tc_label, col in colors_tc.items():
        label = f"Signal3S_t{thresh}_monthly_TC{tc_label}"
        pv    = signal_results[label]['portfolio_value']
        ax.plot(pv.index, pv / 1e9, color=col, lw=1.3,
                label=f'Signal TC={tc_label}')
    # Benchmarks
    ax.plot(ew_ref_pv.index, ew_ref_pv / 1e9,
            color='grey', lw=1.8, ls='--', label='EW Rebalanced')
    ax.plot(bh_pv.index, bh_pv / 1e9,
            color='forestgreen', lw=1.8, ls='-.', label='Buy & Hold')

    ax.set_title(f"Threshold = {thresh}", fontsize=11)
    ax.set_ylabel("Value ($B)")
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("signal_3stage_tc_sensitivity.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Figure 5: All three strategies on one panel (TC=0.5%, monthly) ────────────
fig, ax = plt.subplots(figsize=(16, 7))
fig.suptitle("Strategy Overview — Buy & Hold vs EW Rebalanced vs Signal (TC=0.5%)",
             fontsize=13, fontweight='bold')

ax.plot(bh_pv.index, bh_pv / 1e9,
        color='forestgreen', lw=2.0, ls='-.', label='Buy & Hold (EW)')
ax.plot(ew_ref_pv.index, ew_ref_pv / 1e9,
        color='grey', lw=2.0, ls='--', label='EW Rebalanced (monthly)')

thresh_colors = {0.5: 'steelblue', 1.0: 'darkorange', 1.5: 'crimson'}
for thresh, col in thresh_colors.items():
    label = f"Signal3S_t{thresh}_monthly_TC0.5%"
    pv    = signal_results[label]['portfolio_value']
    ax.plot(pv.index, pv / 1e9, color=col, lw=1.8,
            label=f'Signal t={thresh}')

ax.axhline(1.0, color='black', ls=':', lw=0.8)
ax.set_ylabel("Portfolio Value ($B)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("strategy_overview.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Figure Q1: Quarterly — Portfolio Value (all thresholds, TC=0.5%) ──────────
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.suptitle("Three-Stage Signal — Quarterly Portfolio Value (TC=0.5%)",
             fontsize=13, fontweight='bold')

for ax, thresh in zip(axes, THRESHOLDS):
    label = f"Signal3S_t{thresh}_quarterly_TC0.5%"
    pv    = signal_results[label]['portfolio_value']
    ew_pv = ew_results['EW_quarterly_TC0.5%']['portfolio_value']

    ax.plot(pv.index,    pv / 1e9,    color='steelblue', lw=1.8,
            label=f'Signal t={thresh}')
    ax.plot(ew_pv.index, ew_pv / 1e9, color='grey', lw=1.5, ls='--',
            label='EW Rebalanced')
    ax.plot(bh_pv.index, bh_pv / 1e9, color='forestgreen', lw=1.5, ls='-.',
            label='Buy & Hold')
    ax.axhline(1.0, color='black', ls=':', lw=0.8)
    ax.set_title(f"Threshold = {thresh}", fontsize=11)
    ax.set_ylabel("Value ($B)")
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("signal_quarterly_value.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Figure Q2: Quarterly — Drawdowns ─────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.suptitle("Three-Stage Signal — Quarterly Drawdowns (TC=0.5%)",
             fontsize=13, fontweight='bold')

for ax, thresh in zip(axes, THRESHOLDS):
    label  = f"Signal3S_t{thresh}_quarterly_TC0.5%"
    pv     = signal_results[label]['portfolio_value']
    dd     = (pv - pv.cummax()) / pv.cummax() * 100
    ew_pv  = ew_results['EW_quarterly_TC0.5%']['portfolio_value']
    ew_dd  = (ew_pv - ew_pv.cummax()) / ew_pv.cummax() * 100
    bh_dd  = (bh_pv - bh_pv.cummax()) / bh_pv.cummax() * 100

    ax.fill_between(dd.index,    dd.values,    0, alpha=0.3, color='steelblue')
    ax.fill_between(ew_dd.index, ew_dd.values, 0, alpha=0.2, color='grey')
    ax.fill_between(bh_dd.index, bh_dd.values, 0, alpha=0.15, color='forestgreen')
    ax.plot(dd.index,    dd.values,    color='steelblue',   lw=1.0,
            label=f'Signal t={thresh}')
    ax.plot(ew_dd.index, ew_dd.values, color='grey',        lw=1.0, ls='--',
            label='EW Rebalanced')
    ax.plot(bh_dd.index, bh_dd.values, color='forestgreen', lw=1.0, ls='-.',
            label='Buy & Hold')
    ax.set_title(f"Threshold = {thresh}", fontsize=11)
    ax.set_ylabel("Drawdown (%)")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("signal_quarterly_drawdowns.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Figure Q3: Quarterly — TC Sensitivity ────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.suptitle("Three-Stage Signal — Quarterly TC Sensitivity",
             fontsize=13, fontweight='bold')

for ax, thresh in zip(axes, THRESHOLDS):
    for tc_label, col in colors_tc.items():
        label = f"Signal3S_t{thresh}_quarterly_TC{tc_label}"
        pv    = signal_results[label]['portfolio_value']
        ax.plot(pv.index, pv / 1e9, color=col, lw=1.3,
                label=f'TC={tc_label}')
    ew_pv = ew_results['EW_quarterly_TC0.5%']['portfolio_value']
    ax.plot(ew_pv.index, ew_pv / 1e9, '--', color='grey',
            lw=1.5, label='EW Benchmark')
    ax.plot(bh_pv.index, bh_pv / 1e9, '-.', color='forestgreen',
            lw=1.5, label='Buy & Hold')
    ax.set_title(f"Threshold = {thresh}", fontsize=11)
    ax.set_ylabel("Value ($B)")
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("signal_quarterly_tc_sensitivity.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Figure Q4: Quarterly — Strategy Overview ─────────────────────────────────
fig, ax = plt.subplots(figsize=(16, 7))
fig.suptitle("Quarterly Strategy Overview — Buy & Hold vs EW vs Signal (TC=0.5%)",
             fontsize=13, fontweight='bold')

ax.plot(bh_pv.index, bh_pv / 1e9,
        color='forestgreen', lw=2.0, ls='-.', label='Buy & Hold (EW)')
ew_q_pv = ew_results['EW_quarterly_TC0.5%']['portfolio_value']
ax.plot(ew_q_pv.index, ew_q_pv / 1e9,
        color='grey', lw=2.0, ls='--', label='EW Rebalanced (quarterly)')

for thresh, col in thresh_colors.items():
    label = f"Signal3S_t{thresh}_quarterly_TC0.5%"
    pv    = signal_results[label]['portfolio_value']
    ax.plot(pv.index, pv / 1e9, color=col, lw=1.8,
            label=f'Signal t={thresh}')

ax.axhline(1.0, color='black', ls=':', lw=0.8)
ax.set_ylabel("Portfolio Value ($B)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("strategy_quarterly_overview.png", dpi=150, bbox_inches='tight')
plt.show()


# ── 16. SAVE ──────────────────────────────────────────────────────────────────
for label, df in {**bh_results, **ew_results, **signal_results}.items():
    df.to_csv(f"{label}_portfolio.csv")

all_metrics = pd.DataFrame({**bh_metrics, **ew_metrics, **sig_metrics}).T
all_metrics.to_csv("all_portfolio_metrics.csv")

composite_score.to_csv("composite_signal.csv")

for thresh in THRESHOLDS:
    sig_df_t = build_signal_df(composite_score, equity_assets,
                               macro_df.index, threshold=thresh)
    sig_df_t.to_csv(f"signal_df_threshold_{thresh}.csv")

print("✓ All files saved.")


In [ ]:
# @title
# ============================================================
# SIGNAL PORTFOLIO — THREE-STAGE THRESHOLD VERSION
# Signals generated from Modified Determinant Model
# + Buy-and-Hold Benchmark
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import requests
from io import StringIO

# ── 1. LOAD ALL DATA FROM GITHUB ─────────────────────────────────────────────
GITHUB_BASE = "https://raw.githubusercontent.com/kboroz/MSFE_Capstone_Project/main/01_Data/Streamlined/"

INDEX_FILE_MAP = {
    'ASX50':       'streamlined_asx_50.csv',
    'EUROSTOXX50': 'streamlined_euro_stoxx_50.csv',
    'FTSE100':     'streamlined_ftse_100.csv',
    'Ibovespa':    'streamlined_ibovespa.csv',
    'JSE40':       'streamlined_jse_top_40.csv',
    'NIKKEI225':   'streamlined_nikkei_225.csv',
    'SP500':       'streamlined_s&p_500.csv',
}

MACRO_FILE = 'streamlined_macro.csv'

def load_csv(url):
    r = requests.get(url)
    r.raise_for_status()
    df = pd.read_csv(StringIO(r.text), index_col=0, parse_dates=True)
    df.columns = df.columns.str.replace('[^a-zA-Z0-9]', '_', regex=True)
    return df.sort_index()

# Load macro
macro_df = load_csv(GITHUB_BASE + MACRO_FILE)
macro_df = macro_df[~macro_df.index.duplicated(keep='first')].sort_index()
print(f"macro_df shape : {macro_df.shape}")
print(f"Columns        : {macro_df.columns.tolist()}")

# Load individual index files
index_data = {}
for name, fname in INDEX_FILE_MAP.items():
    try:
        df = load_csv(GITHUB_BASE + fname)
        index_data[name] = df
        print(f"✓ {name:15s} | shape={df.shape} | cols={df.columns.tolist()[:4]}")
    except Exception as e:
        print(f"✗ {name}: {e}")

# ── 2. EXTRACT PRICE SERIES ───────────────────────────────────────────────────
PRICE_COLS = ['Close', 'close', 'Price', 'price', 'Adj_Close', 'Last']

def get_price_series(df, name):
    col = next((c for c in PRICE_COLS if c in df.columns), df.columns[0])
    s   = pd.to_numeric(df[col], errors='coerce').dropna()
    print(f"  {name}: using column '{col}'")
    return s

index_prices = {}
for name, df in index_data.items():
    index_prices[name] = get_price_series(df, name)

# ── 3. MODIFIED DETERMINANT SIGNAL MODEL ─────────────────────────────────────
DET_WINDOWS    = [3, 7, 28]
WINDOW_WEIGHTS = {3: 0.0, 7: 1.0, 28: 0.0}

def compute_modified_determinant(price_series_dict, windows=DET_WINDOWS):
    prices = pd.DataFrame(price_series_dict).sort_index()
    rets   = prices.pct_change().dropna()
    det_results = {}

    for window in windows:
        det_series = []
        for i in range(window, len(rets)):
            window_rets = rets.iloc[i - window: i]
            valid_cols  = window_rets.columns[window_rets.std() > 1e-10]
            w           = window_rets[valid_cols]

            if len(valid_cols) < 2:
                det_series.append(np.nan)
                continue
            try:
                R   = w.corr().values
                I   = np.eye(len(R))
                M   = R - I
                det = np.linalg.det(M)
            except Exception:
                det = np.nan
            det_series.append(det)

        idx = rets.index[window:]
        det_results[window] = pd.Series(det_series, index=idx, name=f'det_{window}d')
        print(f"  Window {window:2d}d | computed {len(det_series)} observations")

    return det_results

print("\n── Computing Modified Determinant ──")
det_results = compute_modified_determinant(index_prices)

# ── 4. COMPOSITE SCORE ────────────────────────────────────────────────────────
def compute_composite_score(det_results, window_weights=WINDOW_WEIGHTS):
    all_series = []
    for window, series in det_results.items():
        mu    = series.rolling(252, min_periods=60).mean()
        sigma = series.rolling(252, min_periods=60).std()
        z     = (series - mu) / sigma.replace(0, np.nan)
        z.name = f'z_{window}d'
        all_series.append(z)

    z_df      = pd.concat(all_series, axis=1).dropna()
    composite = sum(
        z_df[f'z_{w}d'] * window_weights[w]
        for w in window_weights if f'z_{w}d' in z_df.columns
    )
    composite = composite / sum(window_weights.values())
    return composite, z_df

composite_score, z_df = compute_composite_score(det_results)

print(f"\nComposite score shape  : {composite_score.shape}")
print(f"Date range             : {composite_score.index[0].date()} → "
      f"{composite_score.index[-1].date()}")
print(f"Score range            : {composite_score.min():.3f} → "
      f"{composite_score.max():.3f}")

# ── 5. THREE-STAGE SIGNAL CLASSIFICATION ─────────────────────────────────────
THRESHOLDS = [0.5, 1.0, 1.5]

def classify_signal(score, threshold):
    if abs(score) > threshold:
        return +1   # RISK_ON  → defensive
    else:
        return -1   # RISK_OFF → long equity

def build_signal_df(composite_score, equity_assets, macro_dates, threshold=0.5):
    binary    = composite_score.apply(lambda x: classify_signal(x, threshold))
    signal_df = pd.DataFrame(index=macro_dates)
    for asset in equity_assets:
        signal_df[asset] = binary.reindex(macro_dates).ffill().bfill()
    return signal_df

# ── 6. ASSET UNIVERSE ─────────────────────────────────────────────────────────
EQUITY_ASSETS = list(index_prices.keys())
SAFE_BONDS    = ['US_10Y', 'UK_10Y', 'AU_10Y', 'ZA_10Y']
FX            = ['AUDUSD', 'BRLUSD', 'EURUSD', 'GBPUSD', 'JPYUSD', 'ZARUSD']
COMMODITIES   = ['Oil', 'Gold', 'Bitcoin']

EXCLUDE_ASSETS = ['Bitcoin']

ALL_ASSETS     = EQUITY_ASSETS + FX + COMMODITIES + SAFE_BONDS
ALL_ASSETS     = [a for a in ALL_ASSETS
                  if a in macro_df.columns and a not in EXCLUDE_ASSETS]

equity_assets     = [a for a in EQUITY_ASSETS if a in ALL_ASSETS]
non_equity_assets = [a for a in ALL_ASSETS if a not in equity_assets]

N           = len(ALL_ASSETS)
base_weight = 1.0 / N

print(f"\nTotal assets              : {N}")
print(f"Equity (signal-driven)    : {equity_assets}")
print(f"Non-equity (always long)  : {non_equity_assets}")
print(f"base_weight = 1/{N} = {base_weight:.4f}")

# ── 7. COUNTRY SAFE-HAVEN MAP ─────────────────────────────────────────────────
EQUITY_SAFE_HAVEN_MAP = {
    'ASX50':       ['AU_10Y', 'AUDUSD'],
    'EUROSTOXX50': ['EURUSD'],
    'FTSE100':     ['UK_10Y', 'GBPUSD'],
    'Ibovespa':    ['BRLUSD'],
    'JSE40':       ['ZA_10Y', 'ZARUSD'],
    'NIKKEI225':   ['JPYUSD'],
    'SP500':       ['US_10Y'],
}

UNIVERSE_SET = set(ALL_ASSETS)
for eq, havens in EQUITY_SAFE_HAVEN_MAP.items():
    EQUITY_SAFE_HAVEN_MAP[eq] = [h for h in havens if h in UNIVERSE_SET]

print("\nSafe-haven routing (filtered to universe):")
for eq, havens in EQUITY_SAFE_HAVEN_MAP.items():
    print(f"  {eq:15s} → {havens}")

# ── 8. BUILD RETURNS ──────────────────────────────────────────────────────────
returns_raw = macro_df[ALL_ASSETS].pct_change(fill_method=None)

for col in SAFE_BONDS:
    if col in returns_raw.columns:
        returns_raw[col] = -returns_raw[col]

if 'Oil' in macro_df.columns:
    bad_oil = macro_df['Oil'][macro_df['Oil'] <= 0].index
    for d in bad_oil:
        loc = returns_raw.index.get_loc(d)
        returns_raw.iloc[
            max(0, loc - 1): loc + 2,
            returns_raw.columns.get_loc('Oil')
        ] = 0.0

returns = returns_raw.clip(lower=-0.50, upper=1.00).fillna(0).iloc[1:]

assert not returns.isnull().any().any(), "NaNs in returns"
assert not np.isinf(returns.values).any(),  "Inf in returns"
print(f"\nReturns shape : {returns.shape}")
print("✓ Returns clean")

# ── 9. BUY-AND-HOLD BENCHMARK ────────────────────────────────────────────────
def run_buy_and_hold(returns, all_assets, initial_capital=1e9):
    """
    Equally-weighted across all assets, set weights on day 1 and never rebalance.
    One-time entry transaction cost applied at inception only.
    No further TC since there is no trading.
    Weights drift with market moves — true passive holding.
    """
    N           = len(all_assets)
    base_weight = 1.0 / N

    # Initialise dollar holdings
    pv       = float(initial_capital)
    holdings = {a: pv * base_weight for a in all_assets}   # dollar amounts

    records = []
    first   = True

    for date in returns.index:
        if first:
            first = False
            # Record starting value (no return on day 0)
            records.append({'date': date, 'portfolio_value': pv})
            continue

        # Update each holding by its return
        new_holdings = {}
        for a in all_assets:
            new_holdings[a] = holdings[a] * (1 + returns.loc[date, a])

        holdings = new_holdings
        pv       = sum(holdings.values())
        records.append({'date': date, 'portfolio_value': pv})

    df = pd.DataFrame(records).set_index('date')
    print(f"  Buy-and-Hold | Start: ${initial_capital/1e9:.2f}B → "
          f"End: ${df['portfolio_value'].iloc[-1]/1e9:.3f}B")
    return df

# ── 10. THREE-STAGE WEIGHT BUILDER ───────────────────────────────────────────
def build_weights_3stage(sig_row, equity_assets, non_equity_assets,
                         base_weight, date, safe_haven_map):
    raw              = {a: base_weight for a in non_equity_assets}
    safe_haven_extra = {}

    for a in equity_assets:
        if sig_row is not None and a in sig_row.index and not pd.isna(sig_row[a]):
            sig = sig_row[a]
        else:
            sig = -1

        if sig > 0:
            raw[a]  = -base_weight
            havens  = safe_haven_map.get(a, [])
            havens  = [h for h in havens if h in raw]

            if havens:
                per_haven = base_weight / len(havens)
                for h in havens:
                    safe_haven_extra[h] = safe_haven_extra.get(h, 0) + per_haven
            else:
                per_ne = base_weight / len(non_equity_assets)
                for ne in non_equity_assets:
                    safe_haven_extra[ne] = safe_haven_extra.get(ne, 0) + per_ne
        else:
            raw[a] = base_weight

    for asset, extra in safe_haven_extra.items():
        raw[asset] = raw.get(asset, 0) + extra

    total = sum(raw.values())
    if total <= 0:
        for a in raw:
            raw[a] = 1.0 / len(raw)
    else:
        for a in raw:
            raw[a] /= total

    return raw

# ── 11. EQUAL-WEIGHT REBALANCED BENCHMARK ────────────────────────────────────
def run_equal_weight(returns, all_assets, tc_rate=0.005,
                     rebal_freq='monthly', initial_capital=1e9):
    N           = len(all_assets)
    base_weight = 1.0 / N
    ew          = {a: base_weight for a in all_assets}

    freq_code     = 'MS' if rebal_freq == 'monthly' else 'QS'
    reb_dates_raw = returns.resample(freq_code).first().index
    actual_dates  = returns.index

    reb_set = set()
    for d in reb_dates_raw:
        future = actual_dates[actual_dates >= d]
        if len(future) > 0:
            reb_set.add(future[0])

    pv              = float(initial_capital)
    current_weights = None
    records         = []

    for date in actual_dates:
        if date in reb_set:
            if current_weights is not None:
                turnover = sum(abs(ew.get(a, 0) - current_weights.get(a, 0))
                               for a in set(ew) | set(current_weights)) / 2.0
                pv *= (1 - turnover * tc_rate)
            else:
                pv *= (1 - tc_rate)
            current_weights = ew.copy()

        if current_weights is not None:
            dr  = sum(current_weights.get(a, 0) * returns.loc[date, a]
                      for a in all_assets)
            pv *= (1 + dr)

        records.append({'date': date, 'portfolio_value': pv})

    return pd.DataFrame(records).set_index('date')

# ── 12. THREE-STAGE SIGNAL PORTFOLIO RUNNER ───────────────────────────────────
def run_signal_portfolio_3stage(returns, signal_df, equity_assets,
                                non_equity_assets, all_assets,
                                freq='monthly', tc_rate=0.005,
                                safe_haven_map=None,
                                initial_capital=1e9):
    if safe_haven_map is None:
        safe_haven_map = {}

    N           = len(all_assets)
    base_weight = 1.0 / N

    freq_code     = 'MS' if freq == 'monthly' else 'QS'
    reb_dates_raw = returns.resample(freq_code).first().index
    actual_dates  = returns.index

    reb_set = set()
    for d in reb_dates_raw:
        future = actual_dates[actual_dates >= d]
        if len(future) > 0:
            reb_set.add(future[0])

    sig = signal_df.copy()
    sig.index = pd.to_datetime(sig.index)

    pv              = float(initial_capital)
    current_weights = None
    records         = []

    for date in actual_dates:
        if date in reb_set:
            past    = sig[sig.index <= date]
            sig_row = past.iloc[-1] if len(past) > 0 else None

            raw = build_weights_3stage(
                sig_row, equity_assets, non_equity_assets,
                base_weight, date, safe_haven_map
            )

            if current_weights is not None:
                all_keys = set(raw) | set(current_weights)
                turnover = sum(
                    abs(raw.get(a, 0) - current_weights.get(a, 0))
                    for a in all_keys
                ) / 2.0
                pv *= (1 - turnover * tc_rate)
            else:
                pv *= (1 - tc_rate)

            current_weights = raw.copy()

        if current_weights is not None:
            dr  = sum(current_weights.get(a, 0) * returns.loc[date, a]
                      for a in all_assets)
            pv *= (1 + dr)

        records.append({'date': date, 'portfolio_value': pv})

    return pd.DataFrame(records).set_index('date')

# ── 13. RUN ALL PORTFOLIOS ────────────────────────────────────────────────────
TC_LEVELS = {'0%': 0.00, '0.5%': 0.005, '1%': 0.01, '2%': 0.02}
FREQS     = ['monthly', 'quarterly']

# ── Buy-and-Hold (single run, no TC variants needed) ─────────────────────────
print("\n── Running Buy-and-Hold ──")
bh_result  = run_buy_and_hold(returns, ALL_ASSETS, initial_capital=1e9)
bh_results = {'BuyAndHold_EW': bh_result}   # dict for uniform handling

# ── Equal-weight rebalanced benchmarks ───────────────────────────────────────
print("\n── Running Equal-Weight Rebalanced ──")
ew_results = {}
for freq in FREQS:
    for tc_label, tc_rate in TC_LEVELS.items():
        label = f"EW_{freq}_TC{tc_label}"
        ew_results[label] = run_equal_weight(
            returns, ALL_ASSETS, tc_rate=tc_rate,
            rebal_freq=freq, initial_capital=1e9
        )
        print(f"  ✓ {label}")

# ── Three-stage signal portfolios ─────────────────────────────────────────────
print("\n── Running Signal Portfolios ──")
signal_results       = {}
signal_distributions = {}

for thresh in THRESHOLDS:
    sig_df_t   = build_signal_df(composite_score, equity_assets,
                                 macro_df.index, threshold=thresh)
    n_risk_on  = (sig_df_t[equity_assets[0]] == +1).sum()
    n_risk_off = (sig_df_t[equity_assets[0]] == -1).sum()
    total      = len(sig_df_t)

    print(f"\n  Threshold={thresh} | RISK_ON(+1): {n_risk_on} days "
          f"({n_risk_on/total*100:.1f}%) | "
          f"RISK_OFF(-1): {n_risk_off} days ({n_risk_off/total*100:.1f}%)")

    signal_distributions[thresh] = {'RISK_ON': n_risk_on, 'RISK_OFF': n_risk_off}

    for freq in FREQS:
        for tc_label, tc_rate in TC_LEVELS.items():
            label = f"Signal3S_t{thresh}_{freq}_TC{tc_label}"
            signal_results[label] = run_signal_portfolio_3stage(
                returns, sig_df_t, equity_assets, non_equity_assets,
                ALL_ASSETS, freq=freq, tc_rate=tc_rate,
                safe_haven_map=EQUITY_SAFE_HAVEN_MAP,
                initial_capital=1e9
            )
            print(f"    ✓ {label}")

print("\n✓ All portfolios computed")

# ── 14. PERFORMANCE METRICS ───────────────────────────────────────────────────
def compute_metrics(pv_df, label=''):
    pv        = pv_df['portfolio_value']
    daily_ret = pv.pct_change().dropna()
    n_years   = len(daily_ret) / 252
    total_ret = (pv.iloc[-1] / pv.iloc[0]) - 1
    cagr      = (1 + total_ret) ** (1 / n_years) - 1 if n_years > 0 else np.nan
    vol       = daily_ret.std() * np.sqrt(252)
    sharpe    = cagr / vol if vol > 0 else np.nan
    roll_max  = pv.cummax()
    max_dd    = ((pv - roll_max) / roll_max).min()
    calmar    = cagr / abs(max_dd) if max_dd != 0 else np.nan
    var_95    = daily_ret.quantile(0.05)

    return {
        'CAGR':         f"{cagr*100:.2f}%",
        'Volatility':   f"{vol*100:.2f}%",
        'Sharpe':       f"{sharpe:.2f}",
        'Max Drawdown': f"{max_dd*100:.2f}%",
        'Calmar':       f"{calmar:.2f}",
        'VaR 95%':      f"{var_95*100:.2f}%",
        'Total Return': f"{total_ret*100:.2f}%",
        'Final ($B)':   f"{pv.iloc[-1]/1e9:.3f}",
    }

# ── Print metrics ─────────────────────────────────────────────────────────────
bh_metrics  = {k: compute_metrics(v, k) for k, v in bh_results.items()}
ew_metrics  = {k: compute_metrics(v, k) for k, v in ew_results.items()}
sig_metrics = {k: compute_metrics(v, k) for k, v in signal_results.items()}

print("\n" + "="*90)
print("BUY-AND-HOLD BENCHMARK (EW, No Rebalancing)")
print("="*90)
print(pd.DataFrame(bh_metrics).T.to_string())

print("\n" + "="*90)
print("EQUAL-WEIGHT REBALANCED BENCHMARK")
print("="*90)
print(pd.DataFrame(ew_metrics).T.to_string())

for thresh in THRESHOLDS:
    sub = {k: v for k, v in sig_metrics.items() if f"_t{thresh}_" in k}
    print(f"\n{'='*90}")
    print(f"THREE-STAGE SIGNAL — Threshold = {thresh} | "
          f"RISK_ON: {signal_distributions[thresh]['RISK_ON']} days | "
          f"RISK_OFF: {signal_distributions[thresh]['RISK_OFF']} days")
    print("="*90)
    print(pd.DataFrame(sub).T.to_string())

# ── 15. PLOTS ─────────────────────────────────────────────────────────────────
colors_tc = {'0%': 'black', '0.5%': 'steelblue', '1%': 'darkorange', '2%': 'crimson'}

# Convenience references reused across figures
bh_pv      = bh_results['BuyAndHold_EW']['portfolio_value']
ew_ref_pv  = ew_results['EW_monthly_TC0.5%']['portfolio_value']

# ── Figure 1: Composite signal overview ──────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(16, 10))
fig.suptitle("Modified Determinant — Composite Score & Threshold Signals",
             fontsize=14, fontweight='bold')

ax = axes[0]
ax.plot(composite_score.index, composite_score.values,
        color='navy', lw=1.0, alpha=0.8, label='Composite Z-score')
for thresh, col in zip(THRESHOLDS, ['green', 'orange', 'red']):
    ax.axhline( thresh, color=col, ls='--', lw=1.0, label=f'+{thresh}')
    ax.axhline(-thresh, color=col, ls='--', lw=1.0, label=f'-{thresh}')
ax.axhline(0, color='black', lw=0.6)
ax.fill_between(composite_score.index, composite_score.values, 0,
                where=(composite_score > 0),  color='green', alpha=0.1)
ax.fill_between(composite_score.index, composite_score.values, 0,
                where=(composite_score <= 0), color='red',   alpha=0.1)
ax.set_title("Composite Determinant Score with Thresholds", fontsize=11)
ax.set_ylabel("Z-score")
ax.legend(fontsize=8, ncol=4)
ax.grid(True, alpha=0.3)

ax = axes[1]
for window, series in det_results.items():
    ax.plot(series.index, np.sign(series.values), alpha=0.6,
            label=f"{window}d window", lw=1.0)
ax.axhline(0, color='black', lw=0.8)
ax.set_title("Sign of Det(R-I) per Window", fontsize=11)
ax.set_ylabel("+1 / -1")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("composite_signal_thresholds.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Figure 2: Portfolio values per threshold vs both benchmarks ───────────────
fig, axes = plt.subplots(1, 3, figsize=(22, 6), sharey=False)
fig.suptitle(
    "Three-Stage Signal vs Benchmarks — Portfolio Value (TC=0.5%, Monthly)",
    fontsize=13, fontweight='bold')

for ax, thresh in zip(axes, THRESHOLDS):
    # Buy-and-hold
    ax.plot(bh_pv.index, bh_pv / 1e9,
            color='forestgreen', lw=1.8, ls='-.',
            label='Buy & Hold (EW)')
    # EW rebalanced
    ax.plot(ew_ref_pv.index, ew_ref_pv / 1e9,
            color='grey', lw=1.5, ls='--',
            label='EW Rebalanced')
    # Signal
    label = f"Signal3S_t{thresh}_monthly_TC0.5%"
    pv    = signal_results[label]['portfolio_value']
    ax.plot(pv.index, pv / 1e9,
            color='steelblue', lw=1.8,
            label=f'Signal (t={thresh})')

    ax.axhline(1.0, color='black', ls=':', lw=0.8)
    ax.set_title(f"Threshold = {thresh}", fontsize=11)
    ax.set_ylabel("Value ($B)")
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("signal_3stage_by_threshold.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Figure 3: Drawdowns per threshold vs both benchmarks ─────────────────────
fig, axes = plt.subplots(1, 3, figsize=(22, 5), sharey=True)
fig.suptitle(
    "Three-Stage Signal — Drawdowns vs Benchmarks (TC=0.5%, Monthly)",
    fontsize=13, fontweight='bold')

bh_dd  = (bh_pv  - bh_pv.cummax())  / bh_pv.cummax()  * 100
ew_dd  = (ew_ref_pv - ew_ref_pv.cummax()) / ew_ref_pv.cummax() * 100

for ax, thresh in zip(axes, THRESHOLDS):
    label  = f"Signal3S_t{thresh}_monthly_TC0.5%"
    pv     = signal_results[label]['portfolio_value']
    dd     = (pv - pv.cummax()) / pv.cummax() * 100

    ax.fill_between(dd.index,    dd.values,    0, alpha=0.25, color='steelblue')
    ax.fill_between(ew_dd.index, ew_dd.values, 0, alpha=0.15, color='grey')
    ax.fill_between(bh_dd.index, bh_dd.values, 0, alpha=0.15, color='forestgreen')

    ax.plot(dd.index,    dd.values,    color='steelblue',   lw=1.2,
            label=f'Signal (t={thresh})')
    ax.plot(ew_dd.index, ew_dd.values, color='grey',        lw=1.0, ls='--',
            label='EW Rebalanced')
    ax.plot(bh_dd.index, bh_dd.values, color='forestgreen', lw=1.0, ls='-.',
            label='Buy & Hold')

    ax.set_title(f"Threshold = {thresh}", fontsize=11)
    ax.set_ylabel("Drawdown (%)")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("signal_3stage_drawdowns.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Figure 4: TC sensitivity per threshold ────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(22, 6))
fig.suptitle("Three-Stage Signal — TC Sensitivity vs Benchmarks (Monthly)",
             fontsize=13, fontweight='bold')

for ax, thresh in zip(axes, THRESHOLDS):
    # Signal across TC levels
    for tc_label, col in colors_tc.items():
        label = f"Signal3S_t{thresh}_monthly_TC{tc_label}"
        pv    = signal_results[label]['portfolio_value']
        ax.plot(pv.index, pv / 1e9, color=col, lw=1.3,
                label=f'Signal TC={tc_label}')
    # Benchmarks
    ax.plot(ew_ref_pv.index, ew_ref_pv / 1e9,
            color='grey', lw=1.8, ls='--', label='EW Rebalanced')
    ax.plot(bh_pv.index, bh_pv / 1e9,
            color='forestgreen', lw=1.8, ls='-.', label='Buy & Hold')

    ax.set_title(f"Threshold = {thresh}", fontsize=11)
    ax.set_ylabel("Value ($B)")
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("signal_3stage_tc_sensitivity.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Figure 5: All three strategies on one panel (TC=0.5%, monthly) ────────────
fig, ax = plt.subplots(figsize=(16, 7))
fig.suptitle("Strategy Overview — Buy & Hold vs EW Rebalanced vs Signal (TC=0.5%)",
             fontsize=13, fontweight='bold')

ax.plot(bh_pv.index, bh_pv / 1e9,
        color='forestgreen', lw=2.0, ls='-.', label='Buy & Hold (EW)')
ax.plot(ew_ref_pv.index, ew_ref_pv / 1e9,
        color='grey', lw=2.0, ls='--', label='EW Rebalanced (monthly)')

thresh_colors = {0.5: 'steelblue', 1.0: 'darkorange', 1.5: 'crimson'}
for thresh, col in thresh_colors.items():
    label = f"Signal3S_t{thresh}_monthly_TC0.5%"
    pv    = signal_results[label]['portfolio_value']
    ax.plot(pv.index, pv / 1e9, color=col, lw=1.8,
            label=f'Signal t={thresh}')

ax.axhline(1.0, color='black', ls=':', lw=0.8)
ax.set_ylabel("Portfolio Value ($B)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("strategy_overview.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Figure Q1: Quarterly — Portfolio Value (all thresholds, TC=0.5%) ──────────
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.suptitle("Three-Stage Signal — Quarterly Portfolio Value (TC=0.5%)",
             fontsize=13, fontweight='bold')

for ax, thresh in zip(axes, THRESHOLDS):
    label = f"Signal3S_t{thresh}_quarterly_TC0.5%"
    pv    = signal_results[label]['portfolio_value']
    ew_pv = ew_results['EW_quarterly_TC0.5%']['portfolio_value']

    ax.plot(pv.index,    pv / 1e9,    color='steelblue', lw=1.8,
            label=f'Signal t={thresh}')
    ax.plot(ew_pv.index, ew_pv / 1e9, color='grey', lw=1.5, ls='--',
            label='EW Rebalanced')
    ax.plot(bh_pv.index, bh_pv / 1e9, color='forestgreen', lw=1.5, ls='-.',
            label='Buy & Hold')
    ax.axhline(1.0, color='black', ls=':', lw=0.8)
    ax.set_title(f"Threshold = {thresh}", fontsize=11)
    ax.set_ylabel("Value ($B)")
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("signal_quarterly_value.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Figure Q2: Quarterly — Drawdowns ─────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.suptitle("Three-Stage Signal — Quarterly Drawdowns (TC=0.5%)",
             fontsize=13, fontweight='bold')

for ax, thresh in zip(axes, THRESHOLDS):
    label  = f"Signal3S_t{thresh}_quarterly_TC0.5%"
    pv     = signal_results[label]['portfolio_value']
    dd     = (pv - pv.cummax()) / pv.cummax() * 100
    ew_pv  = ew_results['EW_quarterly_TC0.5%']['portfolio_value']
    ew_dd  = (ew_pv - ew_pv.cummax()) / ew_pv.cummax() * 100
    bh_dd  = (bh_pv - bh_pv.cummax()) / bh_pv.cummax() * 100

    ax.fill_between(dd.index,    dd.values,    0, alpha=0.3, color='steelblue')
    ax.fill_between(ew_dd.index, ew_dd.values, 0, alpha=0.2, color='grey')
    ax.fill_between(bh_dd.index, bh_dd.values, 0, alpha=0.15, color='forestgreen')
    ax.plot(dd.index,    dd.values,    color='steelblue',   lw=1.0,
            label=f'Signal t={thresh}')
    ax.plot(ew_dd.index, ew_dd.values, color='grey',        lw=1.0, ls='--',
            label='EW Rebalanced')
    ax.plot(bh_dd.index, bh_dd.values, color='forestgreen', lw=1.0, ls='-.',
            label='Buy & Hold')
    ax.set_title(f"Threshold = {thresh}", fontsize=11)
    ax.set_ylabel("Drawdown (%)")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("signal_quarterly_drawdowns.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Figure Q3: Quarterly — TC Sensitivity ────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.suptitle("Three-Stage Signal — Quarterly TC Sensitivity",
             fontsize=13, fontweight='bold')

for ax, thresh in zip(axes, THRESHOLDS):
    for tc_label, col in colors_tc.items():
        label = f"Signal3S_t{thresh}_quarterly_TC{tc_label}"
        pv    = signal_results[label]['portfolio_value']
        ax.plot(pv.index, pv / 1e9, color=col, lw=1.3,
                label=f'TC={tc_label}')
    ew_pv = ew_results['EW_quarterly_TC0.5%']['portfolio_value']
    ax.plot(ew_pv.index, ew_pv / 1e9, '--', color='grey',
            lw=1.5, label='EW Benchmark')
    ax.plot(bh_pv.index, bh_pv / 1e9, '-.', color='forestgreen',
            lw=1.5, label='Buy & Hold')
    ax.set_title(f"Threshold = {thresh}", fontsize=11)
    ax.set_ylabel("Value ($B)")
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("signal_quarterly_tc_sensitivity.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Figure Q4: Quarterly — Strategy Overview ─────────────────────────────────
fig, ax = plt.subplots(figsize=(16, 7))
fig.suptitle("Quarterly Strategy Overview — Buy & Hold vs EW vs Signal (TC=0.5%)",
             fontsize=13, fontweight='bold')

ax.plot(bh_pv.index, bh_pv / 1e9,
        color='forestgreen', lw=2.0, ls='-.', label='Buy & Hold (EW)')
ew_q_pv = ew_results['EW_quarterly_TC0.5%']['portfolio_value']
ax.plot(ew_q_pv.index, ew_q_pv / 1e9,
        color='grey', lw=2.0, ls='--', label='EW Rebalanced (quarterly)')

for thresh, col in thresh_colors.items():
    label = f"Signal3S_t{thresh}_quarterly_TC0.5%"
    pv    = signal_results[label]['portfolio_value']
    ax.plot(pv.index, pv / 1e9, color=col, lw=1.8,
            label=f'Signal t={thresh}')

ax.axhline(1.0, color='black', ls=':', lw=0.8)
ax.set_ylabel("Portfolio Value ($B)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("strategy_quarterly_overview.png", dpi=150, bbox_inches='tight')
plt.show()


# ── 16. SAVE ──────────────────────────────────────────────────────────────────
for label, df in {**bh_results, **ew_results, **signal_results}.items():
    df.to_csv(f"{label}_portfolio.csv")

all_metrics = pd.DataFrame({**bh_metrics, **ew_metrics, **sig_metrics}).T
all_metrics.to_csv("all_portfolio_metrics.csv")

composite_score.to_csv("composite_signal.csv")

for thresh in THRESHOLDS:
    sig_df_t = build_signal_df(composite_score, equity_assets,
                               macro_df.index, threshold=thresh)
    sig_df_t.to_csv(f"signal_df_threshold_{thresh}.csv")

print("✓ All files saved.")


In [ ]:
# @title
# ============================================================
# SIGNAL PORTFOLIO — THREE-STAGE THRESHOLD VERSION
# Signals generated from Modified Determinant Model
# + Buy-and-Hold Benchmark
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import requests
from io import StringIO

# ── 1. LOAD ALL DATA FROM GITHUB ─────────────────────────────────────────────
GITHUB_BASE = "https://raw.githubusercontent.com/kboroz/MSFE_Capstone_Project/main/01_Data/Streamlined/"

INDEX_FILE_MAP = {
    'ASX50':       'streamlined_asx_50.csv',
    'EUROSTOXX50': 'streamlined_euro_stoxx_50.csv',
    'FTSE100':     'streamlined_ftse_100.csv',
    'Ibovespa':    'streamlined_ibovespa.csv',
    'JSE40':       'streamlined_jse_top_40.csv',
    'NIKKEI225':   'streamlined_nikkei_225.csv',
    'SP500':       'streamlined_s&p_500.csv',
}

MACRO_FILE = 'streamlined_macro.csv'

def load_csv(url):
    r = requests.get(url)
    r.raise_for_status()
    df = pd.read_csv(StringIO(r.text), index_col=0, parse_dates=True)
    df.columns = df.columns.str.replace('[^a-zA-Z0-9]', '_', regex=True)
    return df.sort_index()

# Load macro
macro_df = load_csv(GITHUB_BASE + MACRO_FILE)
macro_df = macro_df[~macro_df.index.duplicated(keep='first')].sort_index()
print(f"macro_df shape : {macro_df.shape}")
print(f"Columns        : {macro_df.columns.tolist()}")

# Load individual index files
index_data = {}
for name, fname in INDEX_FILE_MAP.items():
    try:
        df = load_csv(GITHUB_BASE + fname)
        index_data[name] = df
        print(f"✓ {name:15s} | shape={df.shape} | cols={df.columns.tolist()[:4]}")
    except Exception as e:
        print(f"✗ {name}: {e}")

# ── 2. EXTRACT PRICE SERIES ───────────────────────────────────────────────────
PRICE_COLS = ['Close', 'close', 'Price', 'price', 'Adj_Close', 'Last']

def get_price_series(df, name):
    col = next((c for c in PRICE_COLS if c in df.columns), df.columns[0])
    s   = pd.to_numeric(df[col], errors='coerce').dropna()
    print(f"  {name}: using column '{col}'")
    return s

index_prices = {}
for name, df in index_data.items():
    index_prices[name] = get_price_series(df, name)

# ── 3. MODIFIED DETERMINANT SIGNAL MODEL ─────────────────────────────────────
DET_WINDOWS    = [3, 7, 28]
WINDOW_WEIGHTS = {3: 0.33, 7: 0.34, 28: 0.33}

def compute_modified_determinant(price_series_dict, windows=DET_WINDOWS):
    prices = pd.DataFrame(price_series_dict).sort_index()
    rets   = prices.pct_change().dropna()
    det_results = {}

    for window in windows:
        det_series = []
        for i in range(window, len(rets)):
            window_rets = rets.iloc[i - window: i]
            valid_cols  = window_rets.columns[window_rets.std() > 1e-10]
            w           = window_rets[valid_cols]

            if len(valid_cols) < 2:
                det_series.append(np.nan)
                continue
            try:
                R   = w.corr().values
                I   = np.eye(len(R))
                M   = R - I
                det = np.linalg.det(M)
            except Exception:
                det = np.nan
            det_series.append(det)

        idx = rets.index[window:]
        det_results[window] = pd.Series(det_series, index=idx, name=f'det_{window}d')
        print(f"  Window {window:2d}d | computed {len(det_series)} observations")

    return det_results

print("\n── Computing Modified Determinant ──")
det_results = compute_modified_determinant(index_prices)

# ── 4. COMPOSITE SCORE ────────────────────────────────────────────────────────
def compute_composite_score(det_results, window_weights=WINDOW_WEIGHTS):
    all_series = []
    for window, series in det_results.items():
        mu    = series.rolling(252, min_periods=60).mean()
        sigma = series.rolling(252, min_periods=60).std()
        z     = (series - mu) / sigma.replace(0, np.nan)
        z.name = f'z_{window}d'
        all_series.append(z)

    z_df      = pd.concat(all_series, axis=1).dropna()
    composite = sum(
        z_df[f'z_{w}d'] * window_weights[w]
        for w in window_weights if f'z_{w}d' in z_df.columns
    )
    composite = composite / sum(window_weights.values())
    return composite, z_df

composite_score, z_df = compute_composite_score(det_results)

print(f"\nComposite score shape  : {composite_score.shape}")
print(f"Date range             : {composite_score.index[0].date()} → "
      f"{composite_score.index[-1].date()}")
print(f"Score range            : {composite_score.min():.3f} → "
      f"{composite_score.max():.3f}")

# ── 5. THREE-STAGE SIGNAL CLASSIFICATION ─────────────────────────────────────
THRESHOLDS = [0.5, 1.0, 1.5]

def classify_signal(score, threshold):
    if abs(score) > threshold:
        return +1   # RISK_ON  → defensive
    else:
        return -1   # RISK_OFF → long equity

def build_signal_df(composite_score, equity_assets, macro_dates, threshold=0.5):
    binary    = composite_score.apply(lambda x: classify_signal(x, threshold))
    signal_df = pd.DataFrame(index=macro_dates)
    for asset in equity_assets:
        signal_df[asset] = binary.reindex(macro_dates).ffill().bfill()
    return signal_df

# ── 6. ASSET UNIVERSE ─────────────────────────────────────────────────────────
EQUITY_ASSETS = list(index_prices.keys())
SAFE_BONDS    = ['US_10Y', 'UK_10Y', 'AU_10Y', 'ZA_10Y']
FX            = ['AUDUSD', 'BRLUSD', 'EURUSD', 'GBPUSD', 'JPYUSD', 'ZARUSD']
COMMODITIES   = ['Oil', 'Gold', 'Bitcoin']

EXCLUDE_ASSETS = ['Bitcoin']

ALL_ASSETS     = EQUITY_ASSETS + FX + COMMODITIES + SAFE_BONDS
ALL_ASSETS     = [a for a in ALL_ASSETS
                  if a in macro_df.columns and a not in EXCLUDE_ASSETS]

equity_assets     = [a for a in EQUITY_ASSETS if a in ALL_ASSETS]
non_equity_assets = [a for a in ALL_ASSETS if a not in equity_assets]

N           = len(ALL_ASSETS)
base_weight = 1.0 / N

print(f"\nTotal assets              : {N}")
print(f"Equity (signal-driven)    : {equity_assets}")
print(f"Non-equity (always long)  : {non_equity_assets}")
print(f"base_weight = 1/{N} = {base_weight:.4f}")

# ── 7. COUNTRY SAFE-HAVEN MAP ─────────────────────────────────────────────────
EQUITY_SAFE_HAVEN_MAP = {
    'ASX50':       ['AU_10Y', 'AUDUSD'],
    'EUROSTOXX50': ['EURUSD'],
    'FTSE100':     ['UK_10Y', 'GBPUSD'],
    'Ibovespa':    ['BRLUSD'],
    'JSE40':       ['ZA_10Y', 'ZARUSD'],
    'NIKKEI225':   ['JPYUSD'],
    'SP500':       ['US_10Y'],
}

UNIVERSE_SET = set(ALL_ASSETS)
for eq, havens in EQUITY_SAFE_HAVEN_MAP.items():
    EQUITY_SAFE_HAVEN_MAP[eq] = [h for h in havens if h in UNIVERSE_SET]

print("\nSafe-haven routing (filtered to universe):")
for eq, havens in EQUITY_SAFE_HAVEN_MAP.items():
    print(f"  {eq:15s} → {havens}")

# ── 8. BUILD RETURNS ──────────────────────────────────────────────────────────
returns_raw = macro_df[ALL_ASSETS].pct_change(fill_method=None)

for col in SAFE_BONDS:
    if col in returns_raw.columns:
        returns_raw[col] = -returns_raw[col]

if 'Oil' in macro_df.columns:
    bad_oil = macro_df['Oil'][macro_df['Oil'] <= 0].index
    for d in bad_oil:
        loc = returns_raw.index.get_loc(d)
        returns_raw.iloc[
            max(0, loc - 1): loc + 2,
            returns_raw.columns.get_loc('Oil')
        ] = 0.0

returns = returns_raw.clip(lower=-0.50, upper=1.00).fillna(0).iloc[1:]

assert not returns.isnull().any().any(), "NaNs in returns"
assert not np.isinf(returns.values).any(),  "Inf in returns"
print(f"\nReturns shape : {returns.shape}")
print("✓ Returns clean")

# ── 9. BUY-AND-HOLD BENCHMARK ────────────────────────────────────────────────
def run_buy_and_hold(returns, all_assets, initial_capital=1e9):
    """
    Equally-weighted across all assets, set weights on day 1 and never rebalance.
    One-time entry transaction cost applied at inception only.
    No further TC since there is no trading.
    Weights drift with market moves — true passive holding.
    """
    N           = len(all_assets)
    base_weight = 1.0 / N

    # Initialise dollar holdings
    pv       = float(initial_capital)
    holdings = {a: pv * base_weight for a in all_assets}   # dollar amounts

    records = []
    first   = True

    for date in returns.index:
        if first:
            first = False
            # Record starting value (no return on day 0)
            records.append({'date': date, 'portfolio_value': pv})
            continue

        # Update each holding by its return
        new_holdings = {}
        for a in all_assets:
            new_holdings[a] = holdings[a] * (1 + returns.loc[date, a])

        holdings = new_holdings
        pv       = sum(holdings.values())
        records.append({'date': date, 'portfolio_value': pv})

    df = pd.DataFrame(records).set_index('date')
    print(f"  Buy-and-Hold | Start: ${initial_capital/1e9:.2f}B → "
          f"End: ${df['portfolio_value'].iloc[-1]/1e9:.3f}B")
    return df

# ── 10. THREE-STAGE WEIGHT BUILDER ───────────────────────────────────────────
def build_weights_3stage(sig_row, equity_assets, non_equity_assets,
                         base_weight, date, safe_haven_map):
    raw              = {a: base_weight for a in non_equity_assets}
    safe_haven_extra = {}

    for a in equity_assets:
        if sig_row is not None and a in sig_row.index and not pd.isna(sig_row[a]):
            sig = sig_row[a]
        else:
            sig = -1

        if sig > 0:
            raw[a]  = -base_weight
            havens  = safe_haven_map.get(a, [])
            havens  = [h for h in havens if h in raw]

            if havens:
                per_haven = base_weight / len(havens)
                for h in havens:
                    safe_haven_extra[h] = safe_haven_extra.get(h, 0) + per_haven
            else:
                per_ne = base_weight / len(non_equity_assets)
                for ne in non_equity_assets:
                    safe_haven_extra[ne] = safe_haven_extra.get(ne, 0) + per_ne
        else:
            raw[a] = base_weight

    for asset, extra in safe_haven_extra.items():
        raw[asset] = raw.get(asset, 0) + extra

    total = sum(raw.values())
    if total <= 0:
        for a in raw:
            raw[a] = 1.0 / len(raw)
    else:
        for a in raw:
            raw[a] /= total

    return raw

# ── 11. EQUAL-WEIGHT REBALANCED BENCHMARK ────────────────────────────────────
def run_equal_weight(returns, all_assets, tc_rate=0.005,
                     rebal_freq='monthly', initial_capital=1e9):
    N           = len(all_assets)
    base_weight = 1.0 / N
    ew          = {a: base_weight for a in all_assets}

    freq_code     = 'MS' if rebal_freq == 'monthly' else 'QS'
    reb_dates_raw = returns.resample(freq_code).first().index
    actual_dates  = returns.index

    reb_set = set()
    for d in reb_dates_raw:
        future = actual_dates[actual_dates >= d]
        if len(future) > 0:
            reb_set.add(future[0])

    pv              = float(initial_capital)
    current_weights = None
    records         = []

    for date in actual_dates:
        if date in reb_set:
            if current_weights is not None:
                turnover = sum(abs(ew.get(a, 0) - current_weights.get(a, 0))
                               for a in set(ew) | set(current_weights)) / 2.0
                pv *= (1 - turnover * tc_rate)
            else:
                pv *= (1 - tc_rate)
            current_weights = ew.copy()

        if current_weights is not None:
            dr  = sum(current_weights.get(a, 0) * returns.loc[date, a]
                      for a in all_assets)
            pv *= (1 + dr)

        records.append({'date': date, 'portfolio_value': pv})

    return pd.DataFrame(records).set_index('date')

# ── 12. THREE-STAGE SIGNAL PORTFOLIO RUNNER ───────────────────────────────────
def run_signal_portfolio_3stage(returns, signal_df, equity_assets,
                                non_equity_assets, all_assets,
                                freq='monthly', tc_rate=0.005,
                                safe_haven_map=None,
                                initial_capital=1e9):
    if safe_haven_map is None:
        safe_haven_map = {}

    N           = len(all_assets)
    base_weight = 1.0 / N

    freq_code     = 'MS' if freq == 'monthly' else 'QS'
    reb_dates_raw = returns.resample(freq_code).first().index
    actual_dates  = returns.index

    reb_set = set()
    for d in reb_dates_raw:
        future = actual_dates[actual_dates >= d]
        if len(future) > 0:
            reb_set.add(future[0])

    sig = signal_df.copy()
    sig.index = pd.to_datetime(sig.index)

    pv              = float(initial_capital)
    current_weights = None
    records         = []

    for date in actual_dates:
        if date in reb_set:
            past    = sig[sig.index <= date]
            sig_row = past.iloc[-1] if len(past) > 0 else None

            raw = build_weights_3stage(
                sig_row, equity_assets, non_equity_assets,
                base_weight, date, safe_haven_map
            )

            if current_weights is not None:
                all_keys = set(raw) | set(current_weights)
                turnover = sum(
                    abs(raw.get(a, 0) - current_weights.get(a, 0))
                    for a in all_keys
                ) / 2.0
                pv *= (1 - turnover * tc_rate)
            else:
                pv *= (1 - tc_rate)

            current_weights = raw.copy()

        if current_weights is not None:
            dr  = sum(current_weights.get(a, 0) * returns.loc[date, a]
                      for a in all_assets)
            pv *= (1 + dr)

        records.append({'date': date, 'portfolio_value': pv})

    return pd.DataFrame(records).set_index('date')

# ── 13. RUN ALL PORTFOLIOS ────────────────────────────────────────────────────
TC_LEVELS = {'0%': 0.00, '0.5%': 0.005, '1%': 0.01, '2%': 0.02}
FREQS     = ['monthly', 'quarterly']

# ── Buy-and-Hold (single run, no TC variants needed) ─────────────────────────
print("\n── Running Buy-and-Hold ──")
bh_result  = run_buy_and_hold(returns, ALL_ASSETS, initial_capital=1e9)
bh_results = {'BuyAndHold_EW': bh_result}   # dict for uniform handling

# ── Equal-weight rebalanced benchmarks ───────────────────────────────────────
print("\n── Running Equal-Weight Rebalanced ──")
ew_results = {}
for freq in FREQS:
    for tc_label, tc_rate in TC_LEVELS.items():
        label = f"EW_{freq}_TC{tc_label}"
        ew_results[label] = run_equal_weight(
            returns, ALL_ASSETS, tc_rate=tc_rate,
            rebal_freq=freq, initial_capital=1e9
        )
        print(f"  ✓ {label}")

# ── Three-stage signal portfolios ─────────────────────────────────────────────
print("\n── Running Signal Portfolios ──")
signal_results       = {}
signal_distributions = {}

for thresh in THRESHOLDS:
    sig_df_t   = build_signal_df(composite_score, equity_assets,
                                 macro_df.index, threshold=thresh)
    n_risk_on  = (sig_df_t[equity_assets[0]] == +1).sum()
    n_risk_off = (sig_df_t[equity_assets[0]] == -1).sum()
    total      = len(sig_df_t)

    print(f"\n  Threshold={thresh} | RISK_ON(+1): {n_risk_on} days "
          f"({n_risk_on/total*100:.1f}%) | "
          f"RISK_OFF(-1): {n_risk_off} days ({n_risk_off/total*100:.1f}%)")

    signal_distributions[thresh] = {'RISK_ON': n_risk_on, 'RISK_OFF': n_risk_off}

    for freq in FREQS:
        for tc_label, tc_rate in TC_LEVELS.items():
            label = f"Signal3S_t{thresh}_{freq}_TC{tc_label}"
            signal_results[label] = run_signal_portfolio_3stage(
                returns, sig_df_t, equity_assets, non_equity_assets,
                ALL_ASSETS, freq=freq, tc_rate=tc_rate,
                safe_haven_map=EQUITY_SAFE_HAVEN_MAP,
                initial_capital=1e9
            )
            print(f"    ✓ {label}")

print("\n✓ All portfolios computed")

# ── 14. PERFORMANCE METRICS ───────────────────────────────────────────────────
def compute_metrics(pv_df, label=''):
    pv        = pv_df['portfolio_value']
    daily_ret = pv.pct_change().dropna()
    n_years   = len(daily_ret) / 252
    total_ret = (pv.iloc[-1] / pv.iloc[0]) - 1
    cagr      = (1 + total_ret) ** (1 / n_years) - 1 if n_years > 0 else np.nan
    vol       = daily_ret.std() * np.sqrt(252)
    sharpe    = cagr / vol if vol > 0 else np.nan
    roll_max  = pv.cummax()
    max_dd    = ((pv - roll_max) / roll_max).min()
    calmar    = cagr / abs(max_dd) if max_dd != 0 else np.nan
    var_95    = daily_ret.quantile(0.05)

    return {
        'CAGR':         f"{cagr*100:.2f}%",
        'Volatility':   f"{vol*100:.2f}%",
        'Sharpe':       f"{sharpe:.2f}",
        'Max Drawdown': f"{max_dd*100:.2f}%",
        'Calmar':       f"{calmar:.2f}",
        'VaR 95%':      f"{var_95*100:.2f}%",
        'Total Return': f"{total_ret*100:.2f}%",
        'Final ($B)':   f"{pv.iloc[-1]/1e9:.3f}",
    }

# ── Print metrics ─────────────────────────────────────────────────────────────
bh_metrics  = {k: compute_metrics(v, k) for k, v in bh_results.items()}
ew_metrics  = {k: compute_metrics(v, k) for k, v in ew_results.items()}
sig_metrics = {k: compute_metrics(v, k) for k, v in signal_results.items()}

print("\n" + "="*90)
print("BUY-AND-HOLD BENCHMARK (EW, No Rebalancing)")
print("="*90)
print(pd.DataFrame(bh_metrics).T.to_string())

print("\n" + "="*90)
print("EQUAL-WEIGHT REBALANCED BENCHMARK")
print("="*90)
print(pd.DataFrame(ew_metrics).T.to_string())

for thresh in THRESHOLDS:
    sub = {k: v for k, v in sig_metrics.items() if f"_t{thresh}_" in k}
    print(f"\n{'='*90}")
    print(f"THREE-STAGE SIGNAL — Threshold = {thresh} | "
          f"RISK_ON: {signal_distributions[thresh]['RISK_ON']} days | "
          f"RISK_OFF: {signal_distributions[thresh]['RISK_OFF']} days")
    print("="*90)
    print(pd.DataFrame(sub).T.to_string())

# ── 15. PLOTS ─────────────────────────────────────────────────────────────────
colors_tc = {'0%': 'black', '0.5%': 'steelblue', '1%': 'darkorange', '2%': 'crimson'}

# Convenience references reused across figures
bh_pv      = bh_results['BuyAndHold_EW']['portfolio_value']
ew_ref_pv  = ew_results['EW_monthly_TC0.5%']['portfolio_value']

# ── Figure 1: Composite signal overview ──────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(16, 10))
fig.suptitle("Modified Determinant — Composite Score & Threshold Signals",
             fontsize=14, fontweight='bold')

ax = axes[0]
ax.plot(composite_score.index, composite_score.values,
        color='navy', lw=1.0, alpha=0.8, label='Composite Z-score')
for thresh, col in zip(THRESHOLDS, ['green', 'orange', 'red']):
    ax.axhline( thresh, color=col, ls='--', lw=1.0, label=f'+{thresh}')
    ax.axhline(-thresh, color=col, ls='--', lw=1.0, label=f'-{thresh}')
ax.axhline(0, color='black', lw=0.6)
ax.fill_between(composite_score.index, composite_score.values, 0,
                where=(composite_score > 0),  color='green', alpha=0.1)
ax.fill_between(composite_score.index, composite_score.values, 0,
                where=(composite_score <= 0), color='red',   alpha=0.1)
ax.set_title("Composite Determinant Score with Thresholds", fontsize=11)
ax.set_ylabel("Z-score")
ax.legend(fontsize=8, ncol=4)
ax.grid(True, alpha=0.3)

ax = axes[1]
for window, series in det_results.items():
    ax.plot(series.index, np.sign(series.values), alpha=0.6,
            label=f"{window}d window", lw=1.0)
ax.axhline(0, color='black', lw=0.8)
ax.set_title("Sign of Det(R-I) per Window", fontsize=11)
ax.set_ylabel("+1 / -1")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("composite_signal_thresholds.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Figure 2: Portfolio values per threshold vs both benchmarks ───────────────
fig, axes = plt.subplots(1, 3, figsize=(22, 6), sharey=False)
fig.suptitle(
    "Three-Stage Signal vs Benchmarks — Portfolio Value (TC=0.5%, Monthly)",
    fontsize=13, fontweight='bold')

for ax, thresh in zip(axes, THRESHOLDS):
    # Buy-and-hold
    ax.plot(bh_pv.index, bh_pv / 1e9,
            color='forestgreen', lw=1.8, ls='-.',
            label='Buy & Hold (EW)')
    # EW rebalanced
    ax.plot(ew_ref_pv.index, ew_ref_pv / 1e9,
            color='grey', lw=1.5, ls='--',
            label='EW Rebalanced')
    # Signal
    label = f"Signal3S_t{thresh}_monthly_TC0.5%"
    pv    = signal_results[label]['portfolio_value']
    ax.plot(pv.index, pv / 1e9,
            color='steelblue', lw=1.8,
            label=f'Signal (t={thresh})')

    ax.axhline(1.0, color='black', ls=':', lw=0.8)
    ax.set_title(f"Threshold = {thresh}", fontsize=11)
    ax.set_ylabel("Value ($B)")
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("signal_3stage_by_threshold.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Figure 3: Drawdowns per threshold vs both benchmarks ─────────────────────
fig, axes = plt.subplots(1, 3, figsize=(22, 5), sharey=True)
fig.suptitle(
    "Three-Stage Signal — Drawdowns vs Benchmarks (TC=0.5%, Monthly)",
    fontsize=13, fontweight='bold')

bh_dd  = (bh_pv  - bh_pv.cummax())  / bh_pv.cummax()  * 100
ew_dd  = (ew_ref_pv - ew_ref_pv.cummax()) / ew_ref_pv.cummax() * 100

for ax, thresh in zip(axes, THRESHOLDS):
    label  = f"Signal3S_t{thresh}_monthly_TC0.5%"
    pv     = signal_results[label]['portfolio_value']
    dd     = (pv - pv.cummax()) / pv.cummax() * 100

    ax.fill_between(dd.index,    dd.values,    0, alpha=0.25, color='steelblue')
    ax.fill_between(ew_dd.index, ew_dd.values, 0, alpha=0.15, color='grey')
    ax.fill_between(bh_dd.index, bh_dd.values, 0, alpha=0.15, color='forestgreen')

    ax.plot(dd.index,    dd.values,    color='steelblue',   lw=1.2,
            label=f'Signal (t={thresh})')
    ax.plot(ew_dd.index, ew_dd.values, color='grey',        lw=1.0, ls='--',
            label='EW Rebalanced')
    ax.plot(bh_dd.index, bh_dd.values, color='forestgreen', lw=1.0, ls='-.',
            label='Buy & Hold')

    ax.set_title(f"Threshold = {thresh}", fontsize=11)
    ax.set_ylabel("Drawdown (%)")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("signal_3stage_drawdowns.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Figure 4: TC sensitivity per threshold ────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(22, 6))
fig.suptitle("Three-Stage Signal — TC Sensitivity vs Benchmarks (Monthly)",
             fontsize=13, fontweight='bold')

for ax, thresh in zip(axes, THRESHOLDS):
    # Signal across TC levels
    for tc_label, col in colors_tc.items():
        label = f"Signal3S_t{thresh}_monthly_TC{tc_label}"
        pv    = signal_results[label]['portfolio_value']
        ax.plot(pv.index, pv / 1e9, color=col, lw=1.3,
                label=f'Signal TC={tc_label}')
    # Benchmarks
    ax.plot(ew_ref_pv.index, ew_ref_pv / 1e9,
            color='grey', lw=1.8, ls='--', label='EW Rebalanced')
    ax.plot(bh_pv.index, bh_pv / 1e9,
            color='forestgreen', lw=1.8, ls='-.', label='Buy & Hold')

    ax.set_title(f"Threshold = {thresh}", fontsize=11)
    ax.set_ylabel("Value ($B)")
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("signal_3stage_tc_sensitivity.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Figure 5: All three strategies on one panel (TC=0.5%, monthly) ────────────
fig, ax = plt.subplots(figsize=(16, 7))
fig.suptitle("Strategy Overview — Buy & Hold vs EW Rebalanced vs Signal (TC=0.5%)",
             fontsize=13, fontweight='bold')

ax.plot(bh_pv.index, bh_pv / 1e9,
        color='forestgreen', lw=2.0, ls='-.', label='Buy & Hold (EW)')
ax.plot(ew_ref_pv.index, ew_ref_pv / 1e9,
        color='grey', lw=2.0, ls='--', label='EW Rebalanced (monthly)')

thresh_colors = {0.5: 'steelblue', 1.0: 'darkorange', 1.5: 'crimson'}
for thresh, col in thresh_colors.items():
    label = f"Signal3S_t{thresh}_monthly_TC0.5%"
    pv    = signal_results[label]['portfolio_value']
    ax.plot(pv.index, pv / 1e9, color=col, lw=1.8,
            label=f'Signal t={thresh}')

ax.axhline(1.0, color='black', ls=':', lw=0.8)
ax.set_ylabel("Portfolio Value ($B)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("strategy_overview.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Figure Q1: Quarterly — Portfolio Value (all thresholds, TC=0.5%) ──────────
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.suptitle("Three-Stage Signal — Quarterly Portfolio Value (TC=0.5%)",
             fontsize=13, fontweight='bold')

for ax, thresh in zip(axes, THRESHOLDS):
    label = f"Signal3S_t{thresh}_quarterly_TC0.5%"
    pv    = signal_results[label]['portfolio_value']
    ew_pv = ew_results['EW_quarterly_TC0.5%']['portfolio_value']

    ax.plot(pv.index,    pv / 1e9,    color='steelblue', lw=1.8,
            label=f'Signal t={thresh}')
    ax.plot(ew_pv.index, ew_pv / 1e9, color='grey', lw=1.5, ls='--',
            label='EW Rebalanced')
    ax.plot(bh_pv.index, bh_pv / 1e9, color='forestgreen', lw=1.5, ls='-.',
            label='Buy & Hold')
    ax.axhline(1.0, color='black', ls=':', lw=0.8)
    ax.set_title(f"Threshold = {thresh}", fontsize=11)
    ax.set_ylabel("Value ($B)")
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("signal_quarterly_value.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Figure Q2: Quarterly — Drawdowns ─────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.suptitle("Three-Stage Signal — Quarterly Drawdowns (TC=0.5%)",
             fontsize=13, fontweight='bold')

for ax, thresh in zip(axes, THRESHOLDS):
    label  = f"Signal3S_t{thresh}_quarterly_TC0.5%"
    pv     = signal_results[label]['portfolio_value']
    dd     = (pv - pv.cummax()) / pv.cummax() * 100
    ew_pv  = ew_results['EW_quarterly_TC0.5%']['portfolio_value']
    ew_dd  = (ew_pv - ew_pv.cummax()) / ew_pv.cummax() * 100
    bh_dd  = (bh_pv - bh_pv.cummax()) / bh_pv.cummax() * 100

    ax.fill_between(dd.index,    dd.values,    0, alpha=0.3, color='steelblue')
    ax.fill_between(ew_dd.index, ew_dd.values, 0, alpha=0.2, color='grey')
    ax.fill_between(bh_dd.index, bh_dd.values, 0, alpha=0.15, color='forestgreen')
    ax.plot(dd.index,    dd.values,    color='steelblue',   lw=1.0,
            label=f'Signal t={thresh}')
    ax.plot(ew_dd.index, ew_dd.values, color='grey',        lw=1.0, ls='--',
            label='EW Rebalanced')
    ax.plot(bh_dd.index, bh_dd.values, color='forestgreen', lw=1.0, ls='-.',
            label='Buy & Hold')
    ax.set_title(f"Threshold = {thresh}", fontsize=11)
    ax.set_ylabel("Drawdown (%)")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("signal_quarterly_drawdowns.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Figure Q3: Quarterly — TC Sensitivity ────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.suptitle("Three-Stage Signal — Quarterly TC Sensitivity",
             fontsize=13, fontweight='bold')

for ax, thresh in zip(axes, THRESHOLDS):
    for tc_label, col in colors_tc.items():
        label = f"Signal3S_t{thresh}_quarterly_TC{tc_label}"
        pv    = signal_results[label]['portfolio_value']
        ax.plot(pv.index, pv / 1e9, color=col, lw=1.3,
                label=f'TC={tc_label}')
    ew_pv = ew_results['EW_quarterly_TC0.5%']['portfolio_value']
    ax.plot(ew_pv.index, ew_pv / 1e9, '--', color='grey',
            lw=1.5, label='EW Benchmark')
    ax.plot(bh_pv.index, bh_pv / 1e9, '-.', color='forestgreen',
            lw=1.5, label='Buy & Hold')
    ax.set_title(f"Threshold = {thresh}", fontsize=11)
    ax.set_ylabel("Value ($B)")
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("signal_quarterly_tc_sensitivity.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Figure Q4: Quarterly — Strategy Overview ─────────────────────────────────
fig, ax = plt.subplots(figsize=(16, 7))
fig.suptitle("Quarterly Strategy Overview — Buy & Hold vs EW vs Signal (TC=0.5%)",
             fontsize=13, fontweight='bold')

ax.plot(bh_pv.index, bh_pv / 1e9,
        color='forestgreen', lw=2.0, ls='-.', label='Buy & Hold (EW)')
ew_q_pv = ew_results['EW_quarterly_TC0.5%']['portfolio_value']
ax.plot(ew_q_pv.index, ew_q_pv / 1e9,
        color='grey', lw=2.0, ls='--', label='EW Rebalanced (quarterly)')

for thresh, col in thresh_colors.items():
    label = f"Signal3S_t{thresh}_quarterly_TC0.5%"
    pv    = signal_results[label]['portfolio_value']
    ax.plot(pv.index, pv / 1e9, color=col, lw=1.8,
            label=f'Signal t={thresh}')

ax.axhline(1.0, color='black', ls=':', lw=0.8)
ax.set_ylabel("Portfolio Value ($B)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("strategy_quarterly_overview.png", dpi=150, bbox_inches='tight')
plt.show()


# ── 16. SAVE ──────────────────────────────────────────────────────────────────
for label, df in {**bh_results, **ew_results, **signal_results}.items():
    df.to_csv(f"{label}_portfolio.csv")

all_metrics = pd.DataFrame({**bh_metrics, **ew_metrics, **sig_metrics}).T
all_metrics.to_csv("all_portfolio_metrics.csv")

composite_score.to_csv("composite_signal.csv")

for thresh in THRESHOLDS:
    sig_df_t = build_signal_df(composite_score, equity_assets,
                               macro_df.index, threshold=thresh)
    sig_df_t.to_csv(f"signal_df_threshold_{thresh}.csv")

print("✓ All files saved.")


In [ ]:
# @title
# ============================================================
# SIGNAL PORTFOLIO — THREE-STAGE THRESHOLD VERSION
# Signals generated from Modified Determinant Model
# + Buy-and-Hold Benchmark
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import requests
from io import StringIO

# ── 1. LOAD ALL DATA FROM GITHUB ─────────────────────────────────────────────
GITHUB_BASE = "https://raw.githubusercontent.com/kboroz/MSFE_Capstone_Project/main/01_Data/Streamlined/"

INDEX_FILE_MAP = {
    'ASX50':       'streamlined_asx_50.csv',
    'EUROSTOXX50': 'streamlined_euro_stoxx_50.csv',
    'FTSE100':     'streamlined_ftse_100.csv',
    'Ibovespa':    'streamlined_ibovespa.csv',
    'JSE40':       'streamlined_jse_top_40.csv',
    'NIKKEI225':   'streamlined_nikkei_225.csv',
    'SP500':       'streamlined_s&p_500.csv',
}

MACRO_FILE = 'streamlined_macro.csv'

def load_csv(url):
    r = requests.get(url)
    r.raise_for_status()
    df = pd.read_csv(StringIO(r.text), index_col=0, parse_dates=True)
    df.columns = df.columns.str.replace('[^a-zA-Z0-9]', '_', regex=True)
    return df.sort_index()

# Load macro
macro_df = load_csv(GITHUB_BASE + MACRO_FILE)
macro_df = macro_df[~macro_df.index.duplicated(keep='first')].sort_index()
print(f"macro_df shape : {macro_df.shape}")
print(f"Columns        : {macro_df.columns.tolist()}")

# Load individual index files
index_data = {}
for name, fname in INDEX_FILE_MAP.items():
    try:
        df = load_csv(GITHUB_BASE + fname)
        index_data[name] = df
        print(f"✓ {name:15s} | shape={df.shape} | cols={df.columns.tolist()[:4]}")
    except Exception as e:
        print(f"✗ {name}: {e}")

# ── 2. EXTRACT PRICE SERIES ───────────────────────────────────────────────────
PRICE_COLS = ['Close', 'close', 'Price', 'price', 'Adj_Close', 'Last']

def get_price_series(df, name):
    col = next((c for c in PRICE_COLS if c in df.columns), df.columns[0])
    s   = pd.to_numeric(df[col], errors='coerce').dropna()
    print(f"  {name}: using column '{col}'")
    return s

index_prices = {}
for name, df in index_data.items():
    index_prices[name] = get_price_series(df, name)

# ── 3. MODIFIED DETERMINANT SIGNAL MODEL ─────────────────────────────────────
DET_WINDOWS    = [3, 7, 28]
WINDOW_WEIGHTS = {3: 0.33, 7: 0.34, 28: 0.33}

def compute_modified_determinant(price_series_dict, windows=DET_WINDOWS):
    prices = pd.DataFrame(price_series_dict).sort_index()
    rets   = prices.pct_change().dropna()
    det_results = {}

    for window in windows:
        det_series = []
        for i in range(window, len(rets)):
            window_rets = rets.iloc[i - window: i]
            valid_cols  = window_rets.columns[window_rets.std() > 1e-10]
            w           = window_rets[valid_cols]

            if len(valid_cols) < 2:
                det_series.append(np.nan)
                continue
            try:
                R   = w.corr().values
                I   = np.eye(len(R))
                M   = R - I
                det = np.linalg.det(M)
            except Exception:
                det = np.nan
            det_series.append(det)

        idx = rets.index[window:]
        det_results[window] = pd.Series(det_series, index=idx, name=f'det_{window}d')
        print(f"  Window {window:2d}d | computed {len(det_series)} observations")

    return det_results

print("\n── Computing Modified Determinant ──")
det_results = compute_modified_determinant(index_prices)

# ── 4. COMPOSITE SCORE ────────────────────────────────────────────────────────
def compute_composite_score(det_results, window_weights=WINDOW_WEIGHTS):
    all_series = []
    for window, series in det_results.items():
        mu    = series.rolling(252, min_periods=60).mean()
        sigma = series.rolling(252, min_periods=60).std()
        z     = (series - mu) / sigma.replace(0, np.nan)
        z.name = f'z_{window}d'
        all_series.append(z)

    z_df      = pd.concat(all_series, axis=1).dropna()
    composite = sum(
        z_df[f'z_{w}d'] * window_weights[w]
        for w in window_weights if f'z_{w}d' in z_df.columns
    )
    composite = composite / sum(window_weights.values())
    return composite, z_df

composite_score, z_df = compute_composite_score(det_results)

print(f"\nComposite score shape  : {composite_score.shape}")
print(f"Date range             : {composite_score.index[0].date()} → "
      f"{composite_score.index[-1].date()}")
print(f"Score range            : {composite_score.min():.3f} → "
      f"{composite_score.max():.3f}")

# ── 5. THREE-STAGE SIGNAL CLASSIFICATION ─────────────────────────────────────
THRESHOLDS = [0.5, 1.0, 1.5]

def classify_signal(score, threshold):
    if abs(score) > threshold:
        return +1   # RISK_ON  → defensive
    else:
        return -1   # RISK_OFF → long equity

def build_signal_df(composite_score, equity_assets, macro_dates, threshold=0.5):
    binary    = composite_score.apply(lambda x: classify_signal(x, threshold))
    signal_df = pd.DataFrame(index=macro_dates)
    for asset in equity_assets:
        signal_df[asset] = binary.reindex(macro_dates).ffill().bfill()
    return signal_df

# ── 6. ASSET UNIVERSE ─────────────────────────────────────────────────────────
EQUITY_ASSETS = list(index_prices.keys())
SAFE_BONDS    = ['US_10Y', 'UK_10Y', 'AU_10Y', 'ZA_10Y']
FX            = ['AUDUSD', 'BRLUSD', 'EURUSD', 'GBPUSD', 'JPYUSD', 'ZARUSD']
COMMODITIES   = ['Oil', 'Gold', 'Bitcoin']

EXCLUDE_ASSETS = ['Bitcoin']

ALL_ASSETS = EQUITY_ASSETS + FX + COMMODITIES + SAFE_BONDS
ALL_ASSETS = [a for a in ALL_ASSETS
              if a in macro_df.columns and a not in EXCLUDE_ASSETS]

equity_assets     = [a for a in EQUITY_ASSETS if a in ALL_ASSETS]
non_equity_assets = [a for a in ALL_ASSETS if a not in equity_assets]

N           = len(ALL_ASSETS)
base_weight = 1.0 / N

print(f"\nTotal assets              : {N}")
print(f"Equity (signal-driven)    : {equity_assets}")
print(f"Non-equity (always long)  : {non_equity_assets}")
print(f"base_weight = 1/{N} = {base_weight:.4f}")

# ── 7. COUNTRY SAFE-HAVEN MAP ─────────────────────────────────────────────────
EQUITY_SAFE_HAVEN_MAP = {
    'ASX50':       ['AU_10Y', 'AUDUSD'],
    'EUROSTOXX50': ['EURUSD'],
    'FTSE100':     ['UK_10Y', 'GBPUSD'],
    'Ibovespa':    ['BRLUSD'],
    'JSE40':       ['ZA_10Y', 'ZARUSD'],
    'NIKKEI225':   ['JPYUSD'],
    'SP500':       ['US_10Y'],
}

UNIVERSE_SET = set(ALL_ASSETS)
for eq, havens in EQUITY_SAFE_HAVEN_MAP.items():
    EQUITY_SAFE_HAVEN_MAP[eq] = [h for h in havens if h in UNIVERSE_SET]

print("\nSafe-haven routing (filtered to universe):")
for eq, havens in EQUITY_SAFE_HAVEN_MAP.items():
    print(f"  {eq:15s} → {havens}")

# ── 8. BUILD RETURNS ──────────────────────────────────────────────────────────
returns_raw = macro_df[ALL_ASSETS].pct_change(fill_method=None)

for col in SAFE_BONDS:
    if col in returns_raw.columns:
        returns_raw[col] = -returns_raw[col]

if 'Oil' in macro_df.columns:
    bad_oil = macro_df['Oil'][macro_df['Oil'] <= 0].index
    for d in bad_oil:
        loc = returns_raw.index.get_loc(d)
        returns_raw.iloc[
            max(0, loc - 1): loc + 2,
            returns_raw.columns.get_loc('Oil')
        ] = 0.0

returns = returns_raw.clip(lower=-0.50, upper=1.00).fillna(0).iloc[1:]

assert not returns.isnull().any().any(), "NaNs in returns"
assert not np.isinf(returns.values).any(),  "Inf in returns"
print(f"\nReturns shape : {returns.shape}")
print("✓ Returns clean")

# ── 9. HELPER: BUILD REBALANCING DATE SET ────────────────────────────────────
def build_reb_set(returns, rebal_freq):
    actual_dates = returns.index
    if rebal_freq == 'daily':
        return set(actual_dates)
    elif rebal_freq == 'weekly':
        raw = returns.resample('W-MON').first().index
    elif rebal_freq == 'monthly':
        raw = returns.resample('MS').first().index
    elif rebal_freq == 'quarterly':
        raw = returns.resample('QS').first().index
    else:
        raise ValueError(f"Unknown rebal_freq: {rebal_freq}")

    reb_set = set()
    for d in raw:
        future = actual_dates[actual_dates >= d]
        if len(future) > 0:
            reb_set.add(future[0])
    return reb_set

# ── 10. BUY-AND-HOLD BENCHMARK ───────────────────────────────────────────────
def run_buy_and_hold(returns, all_assets, initial_capital=1e9):
    """
    Equally-weighted across all assets, set weights on day 1 and never rebalance.
    Weights drift with market moves — true passive holding.
    """
    N           = len(all_assets)
    base_weight = 1.0 / N

    pv       = float(initial_capital)
    holdings = {a: pv * base_weight for a in all_assets}

    records = []
    first   = True

    for date in returns.index:
        if first:
            first = False
            records.append({'date': date, 'portfolio_value': pv})
            continue

        new_holdings = {}
        for a in all_assets:
            new_holdings[a] = holdings[a] * (1 + returns.loc[date, a])

        holdings = new_holdings
        pv       = sum(holdings.values())
        records.append({'date': date, 'portfolio_value': pv})

    df = pd.DataFrame(records).set_index('date')
    print(f"  Buy-and-Hold | Start: ${initial_capital/1e9:.2f}B → "
          f"End: ${df['portfolio_value'].iloc[-1]/1e9:.3f}B")
    return df

# ── 11. THREE-STAGE WEIGHT BUILDER ───────────────────────────────────────────
def build_weights_3stage(sig_row, equity_assets, non_equity_assets,
                         base_weight, date, safe_haven_map):
    raw              = {a: base_weight for a in non_equity_assets}
    safe_haven_extra = {}

    for a in equity_assets:
        if sig_row is not None and a in sig_row.index and not pd.isna(sig_row[a]):
            sig = sig_row[a]
        else:
            sig = -1

        if sig > 0:
            raw[a]  = -base_weight
            havens  = safe_haven_map.get(a, [])
            havens  = [h for h in havens if h in raw]

            if havens:
                per_haven = base_weight / len(havens)
                for h in havens:
                    safe_haven_extra[h] = safe_haven_extra.get(h, 0) + per_haven
            else:
                per_ne = base_weight / len(non_equity_assets)
                for ne in non_equity_assets:
                    safe_haven_extra[ne] = safe_haven_extra.get(ne, 0) + per_ne
        else:
            raw[a] = base_weight

    for asset, extra in safe_haven_extra.items():
        raw[asset] = raw.get(asset, 0) + extra

    total = sum(raw.values())
    if total <= 0:
        for a in raw:
            raw[a] = 1.0 / len(raw)
    else:
        for a in raw:
            raw[a] /= total

    return raw

# ── 12. EQUAL-WEIGHT REBALANCED BENCHMARK ────────────────────────────────────
def run_equal_weight(returns, all_assets, tc_rate=0.005,
                     rebal_freq='monthly', initial_capital=1e9):
    N           = len(all_assets)
    base_weight = 1.0 / N
    ew          = {a: base_weight for a in all_assets}

    reb_set      = build_reb_set(returns, rebal_freq)
    actual_dates = returns.index

    pv              = float(initial_capital)
    current_weights = None
    records         = []

    for date in actual_dates:
        if date in reb_set:
            if current_weights is not None:
                turnover = sum(abs(ew.get(a, 0) - current_weights.get(a, 0))
                               for a in set(ew) | set(current_weights)) / 2.0
                pv *= (1 - turnover * tc_rate)
            else:
                pv *= (1 - tc_rate)
            current_weights = ew.copy()

        if current_weights is not None:
            dr  = sum(current_weights.get(a, 0) * returns.loc[date, a]
                      for a in all_assets)
            pv *= (1 + dr)

        records.append({'date': date, 'portfolio_value': pv})

    return pd.DataFrame(records).set_index('date')

# ── 13. THREE-STAGE SIGNAL PORTFOLIO RUNNER ──────────────────────────────────
def run_signal_portfolio_3stage(returns, signal_df, equity_assets,
                                non_equity_assets, all_assets,
                                freq='monthly', tc_rate=0.005,
                                safe_haven_map=None,
                                initial_capital=1e9):
    if safe_haven_map is None:
        safe_haven_map = {}

    N           = len(all_assets)
    base_weight = 1.0 / N

    reb_set      = build_reb_set(returns, freq)
    actual_dates = returns.index

    sig = signal_df.copy()
    sig.index = pd.to_datetime(sig.index)

    pv              = float(initial_capital)
    current_weights = None
    records         = []

    for date in actual_dates:
        if date in reb_set:
            past    = sig[sig.index <= date]
            sig_row = past.iloc[-1] if len(past) > 0 else None

            raw = build_weights_3stage(
                sig_row, equity_assets, non_equity_assets,
                base_weight, date, safe_haven_map
            )

            if current_weights is not None:
                all_keys = set(raw) | set(current_weights)
                turnover = sum(
                    abs(raw.get(a, 0) - current_weights.get(a, 0))
                    for a in all_keys
                ) / 2.0
                pv *= (1 - turnover * tc_rate)
            else:
                pv *= (1 - tc_rate)

            current_weights = raw.copy()

        if current_weights is not None:
            dr  = sum(current_weights.get(a, 0) * returns.loc[date, a]
                      for a in all_assets)
            pv *= (1 + dr)

        records.append({'date': date, 'portfolio_value': pv})

    return pd.DataFrame(records).set_index('date')

# ── 14. RUN ALL PORTFOLIOS ────────────────────────────────────────────────────
TC_LEVELS = {'0%': 0.00, '0.5%': 0.005, '1%': 0.01, '2%': 0.02}
FREQS     = ['daily', 'weekly', 'monthly', 'quarterly']

# ── Buy-and-Hold ──────────────────────────────────────────────────────────────
print("\n── Running Buy-and-Hold ──")
bh_result  = run_buy_and_hold(returns, ALL_ASSETS, initial_capital=1e9)
bh_results = {'BuyAndHold_EW': bh_result}

# ── Equal-weight rebalanced benchmarks ───────────────────────────────────────
print("\n── Running Equal-Weight Rebalanced ──")
ew_results = {}
for freq in FREQS:
    for tc_label, tc_rate in TC_LEVELS.items():
        label = f"EW_{freq}_TC{tc_label}"
        ew_results[label] = run_equal_weight(
            returns, ALL_ASSETS, tc_rate=tc_rate,
            rebal_freq=freq, initial_capital=1e9
        )
        print(f"  ✓ {label}")

# ── Three-stage signal portfolios ─────────────────────────────────────────────
print("\n── Running Signal Portfolios ──")
signal_results       = {}
signal_distributions = {}

for thresh in THRESHOLDS:
    sig_df_t   = build_signal_df(composite_score, equity_assets,
                                 macro_df.index, threshold=thresh)
    n_risk_on  = (sig_df_t[equity_assets[0]] == +1).sum()
    n_risk_off = (sig_df_t[equity_assets[0]] == -1).sum()
    total      = len(sig_df_t)

    print(f"\n  Threshold={thresh} | RISK_ON(+1): {n_risk_on} days "
          f"({n_risk_on/total*100:.1f}%) | "
          f"RISK_OFF(-1): {n_risk_off} days ({n_risk_off/total*100:.1f}%)")

    signal_distributions[thresh] = {'RISK_ON': n_risk_on, 'RISK_OFF': n_risk_off}

    for freq in FREQS:
        for tc_label, tc_rate in TC_LEVELS.items():
            label = f"Signal3S_t{thresh}_{freq}_TC{tc_label}"
            signal_results[label] = run_signal_portfolio_3stage(
                returns, sig_df_t, equity_assets, non_equity_assets,
                ALL_ASSETS, freq=freq, tc_rate=tc_rate,
                safe_haven_map=EQUITY_SAFE_HAVEN_MAP,
                initial_capital=1e9
            )
            print(f"    ✓ {label}")

print("\n✓ All portfolios computed")

# ── 15. PERFORMANCE METRICS ───────────────────────────────────────────────────
def compute_metrics(pv_df, label=''):
    pv        = pv_df['portfolio_value']
    daily_ret = pv.pct_change().dropna()
    n_years   = len(daily_ret) / 252
    total_ret = (pv.iloc[-1] / pv.iloc[0]) - 1
    cagr      = (1 + total_ret) ** (1 / n_years) - 1 if n_years > 0 else np.nan
    vol       = daily_ret.std() * np.sqrt(252)
    sharpe    = cagr / vol if vol > 0 else np.nan
    roll_max  = pv.cummax()
    max_dd    = ((pv - roll_max) / roll_max).min()
    calmar    = cagr / abs(max_dd) if max_dd != 0 else np.nan
    var_95    = daily_ret.quantile(0.05)

    return {
        'CAGR':         f"{cagr*100:.2f}%",
        'Volatility':   f"{vol*100:.2f}%",
        'Sharpe':       f"{sharpe:.2f}",
        'Max Drawdown': f"{max_dd*100:.2f}%",
        'Calmar':       f"{calmar:.2f}",
        'VaR 95%':      f"{var_95*100:.2f}%",
        'Total Return': f"{total_ret*100:.2f}%",
        'Final ($B)':   f"{pv.iloc[-1]/1e9:.3f}",
    }

# ── Print metrics ─────────────────────────────────────────────────────────────
bh_metrics  = {k: compute_metrics(v, k) for k, v in bh_results.items()}
ew_metrics  = {k: compute_metrics(v, k) for k, v in ew_results.items()}
sig_metrics = {k: compute_metrics(v, k) for k, v in signal_results.items()}

print("\n" + "="*90)
print("BUY-AND-HOLD BENCHMARK (EW, No Rebalancing)")
print("="*90)
print(pd.DataFrame(bh_metrics).T.to_string())

print("\n" + "="*90)
print("EQUAL-WEIGHT REBALANCED BENCHMARK")
print("="*90)
print(pd.DataFrame(ew_metrics).T.to_string())

for thresh in THRESHOLDS:
    sub = {k: v for k, v in sig_metrics.items() if f"_t{thresh}_" in k}
    print(f"\n{'='*90}")
    print(f"THREE-STAGE SIGNAL — Threshold = {thresh} | "
          f"RISK_ON: {signal_distributions[thresh]['RISK_ON']} days | "
          f"RISK_OFF: {signal_distributions[thresh]['RISK_OFF']} days")
    print("="*90)
    print(pd.DataFrame(sub).T.to_string())

# ── 16. PLOTS ─────────────────────────────────────────────────────────────────
colors_tc    = {'0%': 'black', '0.5%': 'steelblue', '1%': 'darkorange', '2%': 'crimson'}
freq_colors  = {'daily': 'crimson', 'weekly': 'darkorange',
                'monthly': 'steelblue', 'quarterly': 'purple'}
thresh_colors = {0.5: 'steelblue', 1.0: 'darkorange', 1.5: 'crimson'}

# Convenience references
bh_pv     = bh_results['BuyAndHold_EW']['portfolio_value']
ew_ref_pv = ew_results['EW_monthly_TC0.5%']['portfolio_value']

# ── Figure 1: Composite signal overview ──────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(16, 10))
fig.suptitle("Modified Determinant — Composite Score & Threshold Signals",
             fontsize=14, fontweight='bold')

ax = axes[0]
ax.plot(composite_score.index, composite_score.values,
        color='navy', lw=1.0, alpha=0.8, label='Composite Z-score')
for thresh, col in zip(THRESHOLDS, ['green', 'orange', 'red']):
    ax.axhline( thresh, color=col, ls='--', lw=1.0, label=f'+{thresh}')
    ax.axhline(-thresh, color=col, ls='--', lw=1.0, label=f'-{thresh}')
ax.axhline(0, color='black', lw=0.6)
ax.fill_between(composite_score.index, composite_score.values, 0,
                where=(composite_score > 0),  color='green', alpha=0.1)
ax.fill_between(composite_score.index, composite_score.values, 0,
                where=(composite_score <= 0), color='red',   alpha=0.1)
ax.set_title("Composite Determinant Score with Thresholds", fontsize=11)
ax.set_ylabel("Z-score")
ax.legend(fontsize=8, ncol=4)
ax.grid(True, alpha=0.3)

ax = axes[1]
for window, series in det_results.items():
    ax.plot(series.index, np.sign(series.values), alpha=0.6,
            label=f"{window}d window", lw=1.0)
ax.axhline(0, color='black', lw=0.8)
ax.set_title("Sign of Det(R-I) per Window", fontsize=11)
ax.set_ylabel("+1 / -1")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("composite_signal_thresholds.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Figure 2: Monthly — Portfolio value per threshold vs benchmarks ───────────
fig, axes = plt.subplots(1, 3, figsize=(22, 6), sharey=False)
fig.suptitle(
    "Three-Stage Signal vs Benchmarks — Portfolio Value (TC=0.5%, Monthly)",
    fontsize=13, fontweight='bold')

for ax, thresh in zip(axes, THRESHOLDS):
    ax.plot(bh_pv.index, bh_pv / 1e9,
            color='forestgreen', lw=1.8, ls='-.', label='Buy & Hold (EW)')
    ax.plot(ew_ref_pv.index, ew_ref_pv / 1e9,
            color='grey', lw=1.5, ls='--', label='EW Rebalanced')
    label = f"Signal3S_t{thresh}_monthly_TC0.5%"
    pv    = signal_results[label]['portfolio_value']
    ax.plot(pv.index, pv / 1e9,
            color='steelblue', lw=1.8, label=f'Signal (t={thresh})')
    ax.axhline(1.0, color='black', ls=':', lw=0.8)
    ax.set_title(f"Threshold = {thresh}", fontsize=11)
    ax.set_ylabel("Value ($B)")
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("signal_3stage_portfolio_value.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Figure 3: Monthly — Drawdowns ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(22, 5), sharey=True)
fig.suptitle("Three-Stage Signal — Drawdowns (TC=0.5%, Monthly)",
             fontsize=13, fontweight='bold')

for ax, thresh in zip(axes, THRESHOLDS):
    label  = f"Signal3S_t{thresh}_monthly_TC0.5%"
    pv     = signal_results[label]['portfolio_value']
    dd     = (pv - pv.cummax()) / pv.cummax() * 100
    ew_dd  = (ew_ref_pv - ew_ref_pv.cummax()) / ew_ref_pv.cummax() * 100
    bh_dd  = (bh_pv - bh_pv.cummax()) / bh_pv.cummax() * 100

    ax.fill_between(dd.index,    dd.values,    0, alpha=0.25, color='steelblue')
    ax.fill_between(ew_dd.index, ew_dd.values, 0, alpha=0.15, color='grey')
    ax.fill_between(bh_dd.index, bh_dd.values, 0, alpha=0.10, color='forestgreen')
    ax.plot(dd.index,    dd.values,    color='steelblue',   lw=1.2,
            label=f'Signal (t={thresh})')
    ax.plot(ew_dd.index, ew_dd.values, color='grey',        lw=1.0, ls='--',
            label='EW Rebalanced')
    ax.plot(bh_dd.index, bh_dd.values, color='forestgreen', lw=1.0, ls='-.',
            label='Buy & Hold')
    ax.set_title(f"Threshold = {thresh}", fontsize=11)
    ax.set_ylabel("Drawdown (%)")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("signal_3stage_drawdowns.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Figure 4: Monthly — TC sensitivity ────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(22, 6))
fig.suptitle("Three-Stage Signal — TC Sensitivity vs Benchmarks (Monthly)",
             fontsize=13, fontweight='bold')

for ax, thresh in zip(axes, THRESHOLDS):
    for tc_label, col in colors_tc.items():
        label = f"Signal3S_t{thresh}_monthly_TC{tc_label}"
        pv    = signal_results[label]['portfolio_value']
        ax.plot(pv.index, pv / 1e9, color=col, lw=1.3,
                label=f'Signal TC={tc_label}')
    ax.plot(ew_ref_pv.index, ew_ref_pv / 1e9,
            color='grey', lw=1.8, ls='--', label='EW Rebalanced')
    ax.plot(bh_pv.index, bh_pv / 1e9,
            color='forestgreen', lw=1.8, ls='-.', label='Buy & Hold')
    ax.set_title(f"Threshold = {thresh}", fontsize=11)
    ax.set_ylabel("Value ($B)")
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("signal_3stage_tc_sensitivity.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Figure 5: Monthly — Strategy overview ─────────────────────────────────────
fig, ax = plt.subplots(figsize=(16, 7))
fig.suptitle("Strategy Overview — Buy & Hold vs EW Rebalanced vs Signal (TC=0.5%, Monthly)",
             fontsize=13, fontweight='bold')

ax.plot(bh_pv.index, bh_pv / 1e9,
        color='forestgreen', lw=2.0, ls='-.', label='Buy & Hold (EW)')
ax.plot(ew_ref_pv.index, ew_ref_pv / 1e9,
        color='grey', lw=2.0, ls='--', label='EW Rebalanced (monthly)')

for thresh, col in thresh_colors.items():
    label = f"Signal3S_t{thresh}_monthly_TC0.5%"
    pv    = signal_results[label]['portfolio_value']
    ax.plot(pv.index, pv / 1e9, color=col, lw=1.8,
            label=f'Signal t={thresh}')

ax.axhline(1.0, color='black', ls=':', lw=0.8)
ax.set_ylabel("Portfolio Value ($B)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("strategy_monthly_overview.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Figure Q1: Quarterly — Portfolio Value ────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(22, 6))
fig.suptitle("Three-Stage Signal — Quarterly Portfolio Value (TC=0.5%)",
             fontsize=13, fontweight='bold')

ew_q_pv = ew_results['EW_quarterly_TC0.5%']['portfolio_value']

for ax, thresh in zip(axes, THRESHOLDS):
    label = f"Signal3S_t{thresh}_quarterly_TC0.5%"
    pv    = signal_results[label]['portfolio_value']
    ax.plot(pv.index,    pv / 1e9,    color='steelblue', lw=1.8,
            label=f'Signal t={thresh}')
    ax.plot(ew_q_pv.index, ew_q_pv / 1e9, color='grey', lw=1.5, ls='--',
            label='EW Rebalanced')
    ax.plot(bh_pv.index, bh_pv / 1e9, color='forestgreen', lw=1.5, ls='-.',
            label='Buy & Hold')
    ax.axhline(1.0, color='black', ls=':', lw=0.8)
    ax.set_title(f"Threshold = {thresh}", fontsize=11)
    ax.set_ylabel("Value ($B)")
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("signal_quarterly_value.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Figure Q2: Quarterly — Drawdowns ─────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(22, 5), sharey=True)
fig.suptitle("Three-Stage Signal — Quarterly Drawdowns (TC=0.5%)",
             fontsize=13, fontweight='bold')

for ax, thresh in zip(axes, THRESHOLDS):
    label  = f"Signal3S_t{thresh}_quarterly_TC0.5%"
    pv     = signal_results[label]['portfolio_value']
    dd     = (pv - pv.cummax()) / pv.cummax() * 100
    ew_dd  = (ew_q_pv - ew_q_pv.cummax()) / ew_q_pv.cummax() * 100
    bh_dd  = (bh_pv - bh_pv.cummax()) / bh_pv.cummax() * 100

    ax.fill_between(dd.index,    dd.values,    0, alpha=0.25, color='steelblue')
    ax.fill_between(ew_dd.index, ew_dd.values, 0, alpha=0.15, color='grey')
    ax.fill_between(bh_dd.index, bh_dd.values, 0, alpha=0.10, color='forestgreen')
    ax.plot(dd.index,    dd.values,    color='steelblue',   lw=1.2,
            label=f'Signal (t={thresh})')
    ax.plot(ew_dd.index, ew_dd.values, color='grey',        lw=1.0, ls='--',
            label='EW Rebalanced')
    ax.plot(bh_dd.index, bh_dd.values, color='forestgreen', lw=1.0, ls='-.',
            label='Buy & Hold')
    ax.set_title(f"Threshold = {thresh}", fontsize=11)
    ax.set_ylabel("Drawdown (%)")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("signal_quarterly_drawdowns.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Figure Q3: Quarterly — TC Sensitivity ────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(22, 6))
fig.suptitle("Three-Stage Signal — Quarterly TC Sensitivity",
             fontsize=13, fontweight='bold')

for ax, thresh in zip(axes, THRESHOLDS):
    for tc_label, col in colors_tc.items():
        label = f"Signal3S_t{thresh}_quarterly_TC{tc_label}"
        pv    = signal_results[label]['portfolio_value']
        ax.plot(pv.index, pv / 1e9, color=col, lw=1.3,
                label=f'TC={tc_label}')
    ax.plot(ew_q_pv.index, ew_q_pv / 1e9, '--', color='grey',
            lw=1.5, label='EW Benchmark')
    ax.plot(bh_pv.index, bh_pv / 1e9, '-.', color='forestgreen',
            lw=1.5, label='Buy & Hold')
    ax.set_title(f"Threshold = {thresh}", fontsize=11)
    ax.set_ylabel("Value ($B)")
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("signal_quarterly_tc_sensitivity.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Figure Q4: Quarterly — Strategy Overview ──────────────────────────────────
fig, ax = plt.subplots(figsize=(16, 7))
fig.suptitle("Quarterly Strategy Overview — Buy & Hold vs EW vs Signal (TC=0.5%)",
             fontsize=13, fontweight='bold')

ax.plot(bh_pv.index, bh_pv / 1e9,
        color='forestgreen', lw=2.0, ls='-.', label='Buy & Hold (EW)')
ax.plot(ew_q_pv.index, ew_q_pv / 1e9,
        color='grey', lw=2.0, ls='--', label='EW Rebalanced (quarterly)')

for thresh, col in thresh_colors.items():
    label = f"Signal3S_t{thresh}_quarterly_TC0.5%"
    pv    = signal_results[label]['portfolio_value']
    ax.plot(pv.index, pv / 1e9, color=col, lw=1.8,
            label=f'Signal t={thresh}')

ax.axhline(1.0, color='black', ls=':', lw=0.8)
ax.set_ylabel("Portfolio Value ($B)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("strategy_quarterly_overview.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Figure F1: Frequency Comparison — Portfolio Value ────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(22, 6))
fig.suptitle("Signal Portfolio — Rebalancing Frequency Comparison (TC=0.5%)",
             fontsize=13, fontweight='bold')

for ax, thresh in zip(axes, THRESHOLDS):
    for freq, col in freq_colors.items():
        label = f"Signal3S_t{thresh}_{freq}_TC0.5%"
        if label in signal_results:
            pv = signal_results[label]['portfolio_value']
            ax.plot(pv.index, pv / 1e9, color=col, lw=1.4,
                    label=freq.capitalize())
    ax.plot(bh_pv.index, bh_pv / 1e9,
            color='forestgreen', lw=1.8, ls='-.', label='Buy & Hold')
    ax.plot(ew_ref_pv.index, ew_ref_pv / 1e9,
            color='grey', lw=1.5, ls='--', label='EW Monthly')
    ax.axhline(1.0, color='black', ls=':', lw=0.8)
    ax.set_title(f"Threshold = {thresh}", fontsize=11)
    ax.set_ylabel("Value ($B)")
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("signal_frequency_comparison_value.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Figure F2: Frequency Comparison — Drawdowns ───────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(22, 5), sharey=True)
fig.suptitle("Signal Portfolio — Drawdowns by Frequency (TC=0.5%)",
             fontsize=13, fontweight='bold')

for ax, thresh in zip(axes, THRESHOLDS):
    for freq, col in freq_colors.items():
        label = f"Signal3S_t{thresh}_{freq}_TC0.5%"
        if label in signal_results:
            pv = signal_results[label]['portfolio_value']
            dd = (pv - pv.cummax()) / pv.cummax() * 100
            ax.plot(dd.index, dd.values, color=col, lw=1.2,
                    label=freq.capitalize())
    bh_dd = (bh_pv - bh_pv.cummax()) / bh_pv.cummax() * 100
    ax.plot(bh_dd.index, bh_dd.values,
            color='forestgreen', lw=1.5, ls='-.', label='Buy & Hold')
    ax.fill_between(bh_dd.index, bh_dd.values, 0, alpha=0.08, color='forestgreen')
    ax.set_title(f"Threshold = {thresh}", fontsize=11)
    ax.set_ylabel("Drawdown (%)")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("signal_frequency_comparison_drawdown.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Figure F3: EW Benchmark — Frequency Comparison ───────────────────────────
fig, ax = plt.subplots(figsize=(16, 7))
fig.suptitle("Equal-Weight Rebalanced — Frequency Comparison (TC=0.5%)",
             fontsize=13, fontweight='bold')

for freq, col in freq_colors.items():
    label = f"EW_{freq}_TC0.5%"
    if label in ew_results:
        pv = ew_results[label]['portfolio_value']
        ax.plot(pv.index, pv / 1e9, color=col, lw=1.6,
                label=f'EW {freq.capitalize()}')

ax.plot(bh_pv.index, bh_pv / 1e9,
        color='forestgreen', lw=2.0, ls='-.', label='Buy & Hold')
ax.axhline(1.0, color='black', ls=':', lw=0.8)
ax.set_ylabel("Portfolio Value ($B)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("ew_frequency_comparison.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Figure F4: TC Impact — Daily vs Weekly (Signal t=1.0) ────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("TC Impact on High-Frequency Rebalancing — Signal t=1.0",
             fontsize=13, fontweight='bold')

for ax, freq in zip(axes, ['daily', 'weekly']):
    for tc_label, col in colors_tc.items():
        label = f"Signal3S_t1.0_{freq}_TC{tc_label}"
        if label in signal_results:
            pv = signal_results[label]['portfolio_value']
            ax.plot(pv.index, pv / 1e9, color=col, lw=1.4,
                    label=f'TC={tc_label}')
    ax.plot(bh_pv.index, bh_pv / 1e9,
            color='forestgreen', lw=1.8, ls='-.', label='Buy & Hold')
    ax.axhline(1.0, color='black', ls=':', lw=0.8)
    ax.set_title(f"{freq.capitalize()} Rebalancing", fontsize=11)
    ax.set_ylabel("Value ($B)")
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("tc_impact_high_frequency.png", dpi=150, bbox_inches='tight')
plt.show()

# ── 17. SAVE ──────────────────────────────────────────────────────────────────
for label, df in {**bh_results, **ew_results, **signal_results}.items():
    df.to_csv(f"{label}_portfolio.csv")

all_metrics = pd.DataFrame({**bh_metrics, **ew_metrics, **sig_metrics}).T
all_metrics.to_csv("all_portfolio_metrics.csv")

composite_score.to_csv("composite_signal.csv")

for thresh in THRESHOLDS:
    sig_df_t = build_signal_df(composite_score, equity_assets,
                               macro_df.index, threshold=thresh)
    sig_df_t.to_csv(f"signal_df_threshold_{thresh}.csv")

print("✓ All files saved.")


In [ ]:
# @title
# ============================================================
# SIGNAL PORTFOLIO — THREE-STAGE THRESHOLD VERSION
# Signals generated from Modified Determinant Model
# + Buy-and-Hold Benchmark
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import requests
from io import StringIO

# ── 1. LOAD ALL DATA FROM GITHUB ─────────────────────────────────────────────
GITHUB_BASE = "https://raw.githubusercontent.com/kboroz/MSFE_Capstone_Project/main/01_Data/Streamlined/"

INDEX_FILE_MAP = {
    'ASX50':       'streamlined_asx_50.csv',
    'EUROSTOXX50': 'streamlined_euro_stoxx_50.csv',
    'FTSE100':     'streamlined_ftse_100.csv',
    'Ibovespa':    'streamlined_ibovespa.csv',
    'JSE40':       'streamlined_jse_top_40.csv',
    'NIKKEI225':   'streamlined_nikkei_225.csv',
    'SP500':       'streamlined_s&p_500.csv',
}

MACRO_FILE = 'streamlined_macro.csv'

def load_csv(url):
    r = requests.get(url)
    r.raise_for_status()
    df = pd.read_csv(StringIO(r.text), index_col=0, parse_dates=True)
    df.columns = df.columns.str.replace('[^a-zA-Z0-9]', '_', regex=True)
    return df.sort_index()

# Load macro
macro_df = load_csv(GITHUB_BASE + MACRO_FILE)
macro_df = macro_df[~macro_df.index.duplicated(keep='first')].sort_index()
print(f"macro_df shape : {macro_df.shape}")
print(f"Columns        : {macro_df.columns.tolist()}")

# Load individual index files
index_data = {}
for name, fname in INDEX_FILE_MAP.items():
    try:
        df = load_csv(GITHUB_BASE + fname)
        index_data[name] = df
        print(f"✓ {name:15s} | shape={df.shape} | cols={df.columns.tolist()[:4]}")
    except Exception as e:
        print(f"✗ {name}: {e}")

# ── 2. EXTRACT PRICE SERIES ───────────────────────────────────────────────────
PRICE_COLS = ['Close', 'close', 'Price', 'price', 'Adj_Close', 'Last']

def get_price_series(df, name):
    col = next((c for c in PRICE_COLS if c in df.columns), df.columns[0])
    s   = pd.to_numeric(df[col], errors='coerce').dropna()
    print(f"  {name}: using column '{col}'")
    return s

index_prices = {}
for name, df in index_data.items():
    index_prices[name] = get_price_series(df, name)

# ── 3. MODIFIED DETERMINANT SIGNAL MODEL ─────────────────────────────────────
DET_WINDOWS    = [3, 7, 28]
WINDOW_WEIGHTS = {3: 0.0, 7: 0.0, 28: 1.0}

def compute_modified_determinant(price_series_dict, windows=DET_WINDOWS):
    prices = pd.DataFrame(price_series_dict).sort_index()
    rets   = prices.pct_change().dropna()
    det_results = {}

    for window in windows:
        det_series = []
        for i in range(window, len(rets)):
            window_rets = rets.iloc[i - window: i]
            valid_cols  = window_rets.columns[window_rets.std() > 1e-10]
            w           = window_rets[valid_cols]

            if len(valid_cols) < 2:
                det_series.append(np.nan)
                continue
            try:
                R   = w.corr().values
                I   = np.eye(len(R))
                M   = R - I
                det = np.linalg.det(M)
            except Exception:
                det = np.nan
            det_series.append(det)

        idx = rets.index[window:]
        det_results[window] = pd.Series(det_series, index=idx, name=f'det_{window}d')
        print(f"  Window {window:2d}d | computed {len(det_series)} observations")

    return det_results

print("\n── Computing Modified Determinant ──")
det_results = compute_modified_determinant(index_prices)

# ── 4. COMPOSITE SCORE ────────────────────────────────────────────────────────
def compute_composite_score(det_results, window_weights=WINDOW_WEIGHTS):
    all_series = []
    for window, series in det_results.items():
        mu    = series.rolling(252, min_periods=60).mean()
        sigma = series.rolling(252, min_periods=60).std()
        z     = (series - mu) / sigma.replace(0, np.nan)
        z.name = f'z_{window}d'
        all_series.append(z)

    z_df      = pd.concat(all_series, axis=1).dropna()
    composite = sum(
        z_df[f'z_{w}d'] * window_weights[w]
        for w in window_weights if f'z_{w}d' in z_df.columns
    )
    composite = composite / sum(window_weights.values())
    return composite, z_df

composite_score, z_df = compute_composite_score(det_results)

print(f"\nComposite score shape  : {composite_score.shape}")
print(f"Date range             : {composite_score.index[0].date()} → "
      f"{composite_score.index[-1].date()}")
print(f"Score range            : {composite_score.min():.3f} → "
      f"{composite_score.max():.3f}")

# ── 5. THREE-STAGE SIGNAL CLASSIFICATION ─────────────────────────────────────
THRESHOLDS = [0.5, 1.0, 1.5]

def classify_signal(score, threshold):
    if abs(score) > threshold:
        return +1   # RISK_ON  → defensive
    else:
        return -1   # RISK_OFF → long equity

def build_signal_df(composite_score, equity_assets, macro_dates, threshold=0.5):
    binary    = composite_score.apply(lambda x: classify_signal(x, threshold))
    signal_df = pd.DataFrame(index=macro_dates)
    for asset in equity_assets:
        signal_df[asset] = binary.reindex(macro_dates).ffill().bfill()
    return signal_df

# ── 6. ASSET UNIVERSE ─────────────────────────────────────────────────────────
EQUITY_ASSETS = list(index_prices.keys())
SAFE_BONDS    = ['US_10Y', 'UK_10Y', 'AU_10Y', 'ZA_10Y']
FX            = ['AUDUSD', 'BRLUSD', 'EURUSD', 'GBPUSD', 'JPYUSD', 'ZARUSD']
COMMODITIES   = ['Oil', 'Gold', 'Bitcoin']

EXCLUDE_ASSETS = ['Bitcoin']

ALL_ASSETS = EQUITY_ASSETS + FX + COMMODITIES + SAFE_BONDS
ALL_ASSETS = [a for a in ALL_ASSETS
              if a in macro_df.columns and a not in EXCLUDE_ASSETS]

equity_assets     = [a for a in EQUITY_ASSETS if a in ALL_ASSETS]
non_equity_assets = [a for a in ALL_ASSETS if a not in equity_assets]

N           = len(ALL_ASSETS)
base_weight = 1.0 / N

print(f"\nTotal assets              : {N}")
print(f"Equity (signal-driven)    : {equity_assets}")
print(f"Non-equity (always long)  : {non_equity_assets}")
print(f"base_weight = 1/{N} = {base_weight:.4f}")

# ── 7. COUNTRY SAFE-HAVEN MAP ─────────────────────────────────────────────────
EQUITY_SAFE_HAVEN_MAP = {
    'ASX50':       ['AU_10Y', 'AUDUSD'],
    'EUROSTOXX50': ['EURUSD'],
    'FTSE100':     ['UK_10Y', 'GBPUSD'],
    'Ibovespa':    ['BRLUSD'],
    'JSE40':       ['ZA_10Y', 'ZARUSD'],
    'NIKKEI225':   ['JPYUSD'],
    'SP500':       ['US_10Y'],
}

UNIVERSE_SET = set(ALL_ASSETS)
for eq, havens in EQUITY_SAFE_HAVEN_MAP.items():
    EQUITY_SAFE_HAVEN_MAP[eq] = [h for h in havens if h in UNIVERSE_SET]

print("\nSafe-haven routing (filtered to universe):")
for eq, havens in EQUITY_SAFE_HAVEN_MAP.items():
    print(f"  {eq:15s} → {havens}")

# ── 8. BUILD RETURNS ──────────────────────────────────────────────────────────
returns_raw = macro_df[ALL_ASSETS].pct_change(fill_method=None)

for col in SAFE_BONDS:
    if col in returns_raw.columns:
        returns_raw[col] = -returns_raw[col]

if 'Oil' in macro_df.columns:
    bad_oil = macro_df['Oil'][macro_df['Oil'] <= 0].index
    for d in bad_oil:
        loc = returns_raw.index.get_loc(d)
        returns_raw.iloc[
            max(0, loc - 1): loc + 2,
            returns_raw.columns.get_loc('Oil')
        ] = 0.0

returns = returns_raw.clip(lower=-0.50, upper=1.00).fillna(0).iloc[1:]

assert not returns.isnull().any().any(), "NaNs in returns"
assert not np.isinf(returns.values).any(),  "Inf in returns"
print(f"\nReturns shape : {returns.shape}")
print("✓ Returns clean")

# ── 9. HELPER: BUILD REBALANCING DATE SET ────────────────────────────────────
def build_reb_set(returns, rebal_freq):
    actual_dates = returns.index
    if rebal_freq == 'daily':
        return set(actual_dates)
    elif rebal_freq == 'weekly':
        raw = returns.resample('W-MON').first().index
    elif rebal_freq == 'monthly':
        raw = returns.resample('MS').first().index
    elif rebal_freq == 'quarterly':
        raw = returns.resample('QS').first().index
    else:
        raise ValueError(f"Unknown rebal_freq: {rebal_freq}")

    reb_set = set()
    for d in raw:
        future = actual_dates[actual_dates >= d]
        if len(future) > 0:
            reb_set.add(future[0])
    return reb_set

# ── 10. BUY-AND-HOLD BENCHMARK ───────────────────────────────────────────────
def run_buy_and_hold(returns, all_assets, initial_capital=1e9):
    """
    Equally-weighted across all assets, set weights on day 1 and never rebalance.
    Weights drift with market moves — true passive holding.
    """
    N           = len(all_assets)
    base_weight = 1.0 / N

    pv       = float(initial_capital)
    holdings = {a: pv * base_weight for a in all_assets}

    records = []
    first   = True

    for date in returns.index:
        if first:
            first = False
            records.append({'date': date, 'portfolio_value': pv})
            continue

        new_holdings = {}
        for a in all_assets:
            new_holdings[a] = holdings[a] * (1 + returns.loc[date, a])

        holdings = new_holdings
        pv       = sum(holdings.values())
        records.append({'date': date, 'portfolio_value': pv})

    df = pd.DataFrame(records).set_index('date')
    print(f"  Buy-and-Hold | Start: ${initial_capital/1e9:.2f}B → "
          f"End: ${df['portfolio_value'].iloc[-1]/1e9:.3f}B")
    return df

# ── 11. THREE-STAGE WEIGHT BUILDER ───────────────────────────────────────────
def build_weights_3stage(sig_row, equity_assets, non_equity_assets,
                         base_weight, date, safe_haven_map):
    raw              = {a: base_weight for a in non_equity_assets}
    safe_haven_extra = {}

    for a in equity_assets:
        if sig_row is not None and a in sig_row.index and not pd.isna(sig_row[a]):
            sig = sig_row[a]
        else:
            sig = -1

        if sig > 0:
            raw[a]  = -base_weight
            havens  = safe_haven_map.get(a, [])
            havens  = [h for h in havens if h in raw]

            if havens:
                per_haven = base_weight / len(havens)
                for h in havens:
                    safe_haven_extra[h] = safe_haven_extra.get(h, 0) + per_haven
            else:
                per_ne = base_weight / len(non_equity_assets)
                for ne in non_equity_assets:
                    safe_haven_extra[ne] = safe_haven_extra.get(ne, 0) + per_ne
        else:
            raw[a] = base_weight

    for asset, extra in safe_haven_extra.items():
        raw[asset] = raw.get(asset, 0) + extra

    total = sum(raw.values())
    if total <= 0:
        for a in raw:
            raw[a] = 1.0 / len(raw)
    else:
        for a in raw:
            raw[a] /= total

    return raw

# ── 12. EQUAL-WEIGHT REBALANCED BENCHMARK ────────────────────────────────────
def run_equal_weight(returns, all_assets, tc_rate=0.005,
                     rebal_freq='monthly', initial_capital=1e9):
    N           = len(all_assets)
    base_weight = 1.0 / N
    ew          = {a: base_weight for a in all_assets}

    reb_set      = build_reb_set(returns, rebal_freq)
    actual_dates = returns.index

    pv              = float(initial_capital)
    current_weights = None
    records         = []

    for date in actual_dates:
        if date in reb_set:
            if current_weights is not None:
                turnover = sum(abs(ew.get(a, 0) - current_weights.get(a, 0))
                               for a in set(ew) | set(current_weights)) / 2.0
                pv *= (1 - turnover * tc_rate)
            else:
                pv *= (1 - tc_rate)
            current_weights = ew.copy()

        if current_weights is not None:
            dr  = sum(current_weights.get(a, 0) * returns.loc[date, a]
                      for a in all_assets)
            pv *= (1 + dr)

        records.append({'date': date, 'portfolio_value': pv})

    return pd.DataFrame(records).set_index('date')

# ── 13. THREE-STAGE SIGNAL PORTFOLIO RUNNER ──────────────────────────────────
def run_signal_portfolio_3stage(returns, signal_df, equity_assets,
                                non_equity_assets, all_assets,
                                freq='monthly', tc_rate=0.005,
                                safe_haven_map=None,
                                initial_capital=1e9):
    if safe_haven_map is None:
        safe_haven_map = {}

    N           = len(all_assets)
    base_weight = 1.0 / N

    reb_set      = build_reb_set(returns, freq)
    actual_dates = returns.index

    sig = signal_df.copy()
    sig.index = pd.to_datetime(sig.index)

    pv              = float(initial_capital)
    current_weights = None
    records         = []

    for date in actual_dates:
        if date in reb_set:
            past    = sig[sig.index <= date]
            sig_row = past.iloc[-1] if len(past) > 0 else None

            raw = build_weights_3stage(
                sig_row, equity_assets, non_equity_assets,
                base_weight, date, safe_haven_map
            )

            if current_weights is not None:
                all_keys = set(raw) | set(current_weights)
                turnover = sum(
                    abs(raw.get(a, 0) - current_weights.get(a, 0))
                    for a in all_keys
                ) / 2.0
                pv *= (1 - turnover * tc_rate)
            else:
                pv *= (1 - tc_rate)

            current_weights = raw.copy()

        if current_weights is not None:
            dr  = sum(current_weights.get(a, 0) * returns.loc[date, a]
                      for a in all_assets)
            pv *= (1 + dr)

        records.append({'date': date, 'portfolio_value': pv})

    return pd.DataFrame(records).set_index('date')

# ── 14. RUN ALL PORTFOLIOS ────────────────────────────────────────────────────
TC_LEVELS = {'0%': 0.00, '0.5%': 0.005, '1%': 0.01, '2%': 0.02}
FREQS     = ['daily', 'weekly', 'monthly', 'quarterly']

# ── Buy-and-Hold ──────────────────────────────────────────────────────────────
print("\n── Running Buy-and-Hold ──")
bh_result  = run_buy_and_hold(returns, ALL_ASSETS, initial_capital=1e9)
bh_results = {'BuyAndHold_EW': bh_result}

# ── Equal-weight rebalanced benchmarks ───────────────────────────────────────
print("\n── Running Equal-Weight Rebalanced ──")
ew_results = {}
for freq in FREQS:
    for tc_label, tc_rate in TC_LEVELS.items():
        label = f"EW_{freq}_TC{tc_label}"
        ew_results[label] = run_equal_weight(
            returns, ALL_ASSETS, tc_rate=tc_rate,
            rebal_freq=freq, initial_capital=1e9
        )
        print(f"  ✓ {label}")

# ── Three-stage signal portfolios ─────────────────────────────────────────────
print("\n── Running Signal Portfolios ──")
signal_results       = {}
signal_distributions = {}

for thresh in THRESHOLDS:
    sig_df_t   = build_signal_df(composite_score, equity_assets,
                                 macro_df.index, threshold=thresh)
    n_risk_on  = (sig_df_t[equity_assets[0]] == +1).sum()
    n_risk_off = (sig_df_t[equity_assets[0]] == -1).sum()
    total      = len(sig_df_t)

    print(f"\n  Threshold={thresh} | RISK_ON(+1): {n_risk_on} days "
          f"({n_risk_on/total*100:.1f}%) | "
          f"RISK_OFF(-1): {n_risk_off} days ({n_risk_off/total*100:.1f}%)")

    signal_distributions[thresh] = {'RISK_ON': n_risk_on, 'RISK_OFF': n_risk_off}

    for freq in FREQS:
        for tc_label, tc_rate in TC_LEVELS.items():
            label = f"Signal3S_t{thresh}_{freq}_TC{tc_label}"
            signal_results[label] = run_signal_portfolio_3stage(
                returns, sig_df_t, equity_assets, non_equity_assets,
                ALL_ASSETS, freq=freq, tc_rate=tc_rate,
                safe_haven_map=EQUITY_SAFE_HAVEN_MAP,
                initial_capital=1e9
            )
            print(f"    ✓ {label}")

print("\n✓ All portfolios computed")

# ── 15. PERFORMANCE METRICS ───────────────────────────────────────────────────
def compute_metrics(pv_df, label=''):
    pv        = pv_df['portfolio_value']
    daily_ret = pv.pct_change().dropna()
    n_years   = len(daily_ret) / 252
    total_ret = (pv.iloc[-1] / pv.iloc[0]) - 1
    cagr      = (1 + total_ret) ** (1 / n_years) - 1 if n_years > 0 else np.nan
    vol       = daily_ret.std() * np.sqrt(252)
    sharpe    = cagr / vol if vol > 0 else np.nan
    roll_max  = pv.cummax()
    max_dd    = ((pv - roll_max) / roll_max).min()
    calmar    = cagr / abs(max_dd) if max_dd != 0 else np.nan
    var_95    = daily_ret.quantile(0.05)

    return {
        'CAGR':         f"{cagr*100:.2f}%",
        'Volatility':   f"{vol*100:.2f}%",
        'Sharpe':       f"{sharpe:.2f}",
        'Max Drawdown': f"{max_dd*100:.2f}%",
        'Calmar':       f"{calmar:.2f}",
        'VaR 95%':      f"{var_95*100:.2f}%",
        'Total Return': f"{total_ret*100:.2f}%",
        'Final ($B)':   f"{pv.iloc[-1]/1e9:.3f}",
    }

# ── Print metrics ─────────────────────────────────────────────────────────────
bh_metrics  = {k: compute_metrics(v, k) for k, v in bh_results.items()}
ew_metrics  = {k: compute_metrics(v, k) for k, v in ew_results.items()}
sig_metrics = {k: compute_metrics(v, k) for k, v in signal_results.items()}

print("\n" + "="*90)
print("BUY-AND-HOLD BENCHMARK (EW, No Rebalancing)")
print("="*90)
print(pd.DataFrame(bh_metrics).T.to_string())

print("\n" + "="*90)
print("EQUAL-WEIGHT REBALANCED BENCHMARK")
print("="*90)
print(pd.DataFrame(ew_metrics).T.to_string())

for thresh in THRESHOLDS:
    sub = {k: v for k, v in sig_metrics.items() if f"_t{thresh}_" in k}
    print(f"\n{'='*90}")
    print(f"THREE-STAGE SIGNAL — Threshold = {thresh} | "
          f"RISK_ON: {signal_distributions[thresh]['RISK_ON']} days | "
          f"RISK_OFF: {signal_distributions[thresh]['RISK_OFF']} days")
    print("="*90)
    print(pd.DataFrame(sub).T.to_string())

# ── 16. PLOTS ─────────────────────────────────────────────────────────────────
colors_tc    = {'0%': 'black', '0.5%': 'steelblue', '1%': 'darkorange', '2%': 'crimson'}
freq_colors  = {'daily': 'crimson', 'weekly': 'darkorange',
                'monthly': 'steelblue', 'quarterly': 'purple'}
thresh_colors = {0.5: 'steelblue', 1.0: 'darkorange', 1.5: 'crimson'}

# Convenience references
bh_pv     = bh_results['BuyAndHold_EW']['portfolio_value']
ew_ref_pv = ew_results['EW_monthly_TC0.5%']['portfolio_value']

# ── Figure 1: Composite signal overview ──────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(16, 10))
fig.suptitle("Modified Determinant — Composite Score & Threshold Signals",
             fontsize=14, fontweight='bold')

ax = axes[0]
ax.plot(composite_score.index, composite_score.values,
        color='navy', lw=1.0, alpha=0.8, label='Composite Z-score')
for thresh, col in zip(THRESHOLDS, ['green', 'orange', 'red']):
    ax.axhline( thresh, color=col, ls='--', lw=1.0, label=f'+{thresh}')
    ax.axhline(-thresh, color=col, ls='--', lw=1.0, label=f'-{thresh}')
ax.axhline(0, color='black', lw=0.6)
ax.fill_between(composite_score.index, composite_score.values, 0,
                where=(composite_score > 0),  color='green', alpha=0.1)
ax.fill_between(composite_score.index, composite_score.values, 0,
                where=(composite_score <= 0), color='red',   alpha=0.1)
ax.set_title("Composite Determinant Score with Thresholds", fontsize=11)
ax.set_ylabel("Z-score")
ax.legend(fontsize=8, ncol=4)
ax.grid(True, alpha=0.3)

ax = axes[1]
for window, series in det_results.items():
    ax.plot(series.index, np.sign(series.values), alpha=0.6,
            label=f"{window}d window", lw=1.0)
ax.axhline(0, color='black', lw=0.8)
ax.set_title("Sign of Det(R-I) per Window", fontsize=11)
ax.set_ylabel("+1 / -1")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("composite_signal_thresholds.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Figure 2: Monthly — Portfolio value per threshold vs benchmarks ───────────
fig, axes = plt.subplots(1, 3, figsize=(22, 6), sharey=False)
fig.suptitle(
    "Three-Stage Signal vs Benchmarks — Portfolio Value (TC=0.5%, Monthly)",
    fontsize=13, fontweight='bold')

for ax, thresh in zip(axes, THRESHOLDS):
    ax.plot(bh_pv.index, bh_pv / 1e9,
            color='forestgreen', lw=1.8, ls='-.', label='Buy & Hold (EW)')
    ax.plot(ew_ref_pv.index, ew_ref_pv / 1e9,
            color='grey', lw=1.5, ls='--', label='EW Rebalanced')
    label = f"Signal3S_t{thresh}_monthly_TC0.5%"
    pv    = signal_results[label]['portfolio_value']
    ax.plot(pv.index, pv / 1e9,
            color='steelblue', lw=1.8, label=f'Signal (t={thresh})')
    ax.axhline(1.0, color='black', ls=':', lw=0.8)
    ax.set_title(f"Threshold = {thresh}", fontsize=11)
    ax.set_ylabel("Value ($B)")
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("signal_3stage_portfolio_value.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Figure 3: Monthly — Drawdowns ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(22, 5), sharey=True)
fig.suptitle("Three-Stage Signal — Drawdowns (TC=0.5%, Monthly)",
             fontsize=13, fontweight='bold')

for ax, thresh in zip(axes, THRESHOLDS):
    label  = f"Signal3S_t{thresh}_monthly_TC0.5%"
    pv     = signal_results[label]['portfolio_value']
    dd     = (pv - pv.cummax()) / pv.cummax() * 100
    ew_dd  = (ew_ref_pv - ew_ref_pv.cummax()) / ew_ref_pv.cummax() * 100
    bh_dd  = (bh_pv - bh_pv.cummax()) / bh_pv.cummax() * 100

    ax.fill_between(dd.index,    dd.values,    0, alpha=0.25, color='steelblue')
    ax.fill_between(ew_dd.index, ew_dd.values, 0, alpha=0.15, color='grey')
    ax.fill_between(bh_dd.index, bh_dd.values, 0, alpha=0.10, color='forestgreen')
    ax.plot(dd.index,    dd.values,    color='steelblue',   lw=1.2,
            label=f'Signal (t={thresh})')
    ax.plot(ew_dd.index, ew_dd.values, color='grey',        lw=1.0, ls='--',
            label='EW Rebalanced')
    ax.plot(bh_dd.index, bh_dd.values, color='forestgreen', lw=1.0, ls='-.',
            label='Buy & Hold')
    ax.set_title(f"Threshold = {thresh}", fontsize=11)
    ax.set_ylabel("Drawdown (%)")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("signal_3stage_drawdowns.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Figure 4: Monthly — TC sensitivity ────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(22, 6))
fig.suptitle("Three-Stage Signal — TC Sensitivity vs Benchmarks (Monthly)",
             fontsize=13, fontweight='bold')

for ax, thresh in zip(axes, THRESHOLDS):
    for tc_label, col in colors_tc.items():
        label = f"Signal3S_t{thresh}_monthly_TC{tc_label}"
        pv    = signal_results[label]['portfolio_value']
        ax.plot(pv.index, pv / 1e9, color=col, lw=1.3,
                label=f'Signal TC={tc_label}')
    ax.plot(ew_ref_pv.index, ew_ref_pv / 1e9,
            color='grey', lw=1.8, ls='--', label='EW Rebalanced')
    ax.plot(bh_pv.index, bh_pv / 1e9,
            color='forestgreen', lw=1.8, ls='-.', label='Buy & Hold')
    ax.set_title(f"Threshold = {thresh}", fontsize=11)
    ax.set_ylabel("Value ($B)")
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("signal_3stage_tc_sensitivity.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Figure 5: Monthly — Strategy overview ─────────────────────────────────────
fig, ax = plt.subplots(figsize=(16, 7))
fig.suptitle("Strategy Overview — Buy & Hold vs EW Rebalanced vs Signal (TC=0.5%, Monthly)",
             fontsize=13, fontweight='bold')

ax.plot(bh_pv.index, bh_pv / 1e9,
        color='forestgreen', lw=2.0, ls='-.', label='Buy & Hold (EW)')
ax.plot(ew_ref_pv.index, ew_ref_pv / 1e9,
        color='grey', lw=2.0, ls='--', label='EW Rebalanced (monthly)')

for thresh, col in thresh_colors.items():
    label = f"Signal3S_t{thresh}_monthly_TC0.5%"
    pv    = signal_results[label]['portfolio_value']
    ax.plot(pv.index, pv / 1e9, color=col, lw=1.8,
            label=f'Signal t={thresh}')

ax.axhline(1.0, color='black', ls=':', lw=0.8)
ax.set_ylabel("Portfolio Value ($B)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("strategy_monthly_overview.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Figure Q1: Quarterly — Portfolio Value ────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(22, 6))
fig.suptitle("Three-Stage Signal — Quarterly Portfolio Value (TC=0.5%)",
             fontsize=13, fontweight='bold')

ew_q_pv = ew_results['EW_quarterly_TC0.5%']['portfolio_value']

for ax, thresh in zip(axes, THRESHOLDS):
    label = f"Signal3S_t{thresh}_quarterly_TC0.5%"
    pv    = signal_results[label]['portfolio_value']
    ax.plot(pv.index,    pv / 1e9,    color='steelblue', lw=1.8,
            label=f'Signal t={thresh}')
    ax.plot(ew_q_pv.index, ew_q_pv / 1e9, color='grey', lw=1.5, ls='--',
            label='EW Rebalanced')
    ax.plot(bh_pv.index, bh_pv / 1e9, color='forestgreen', lw=1.5, ls='-.',
            label='Buy & Hold')
    ax.axhline(1.0, color='black', ls=':', lw=0.8)
    ax.set_title(f"Threshold = {thresh}", fontsize=11)
    ax.set_ylabel("Value ($B)")
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("signal_quarterly_value.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Figure Q2: Quarterly — Drawdowns ─────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(22, 5), sharey=True)
fig.suptitle("Three-Stage Signal — Quarterly Drawdowns (TC=0.5%)",
             fontsize=13, fontweight='bold')

for ax, thresh in zip(axes, THRESHOLDS):
    label  = f"Signal3S_t{thresh}_quarterly_TC0.5%"
    pv     = signal_results[label]['portfolio_value']
    dd     = (pv - pv.cummax()) / pv.cummax() * 100
    ew_dd  = (ew_q_pv - ew_q_pv.cummax()) / ew_q_pv.cummax() * 100
    bh_dd  = (bh_pv - bh_pv.cummax()) / bh_pv.cummax() * 100

    ax.fill_between(dd.index,    dd.values,    0, alpha=0.25, color='steelblue')
    ax.fill_between(ew_dd.index, ew_dd.values, 0, alpha=0.15, color='grey')
    ax.fill_between(bh_dd.index, bh_dd.values, 0, alpha=0.10, color='forestgreen')
    ax.plot(dd.index,    dd.values,    color='steelblue',   lw=1.2,
            label=f'Signal (t={thresh})')
    ax.plot(ew_dd.index, ew_dd.values, color='grey',        lw=1.0, ls='--',
            label='EW Rebalanced')
    ax.plot(bh_dd.index, bh_dd.values, color='forestgreen', lw=1.0, ls='-.',
            label='Buy & Hold')
    ax.set_title(f"Threshold = {thresh}", fontsize=11)
    ax.set_ylabel("Drawdown (%)")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("signal_quarterly_drawdowns.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Figure Q3: Quarterly — TC Sensitivity ────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(22, 6))
fig.suptitle("Three-Stage Signal — Quarterly TC Sensitivity",
             fontsize=13, fontweight='bold')

for ax, thresh in zip(axes, THRESHOLDS):
    for tc_label, col in colors_tc.items():
        label = f"Signal3S_t{thresh}_quarterly_TC{tc_label}"
        pv    = signal_results[label]['portfolio_value']
        ax.plot(pv.index, pv / 1e9, color=col, lw=1.3,
                label=f'TC={tc_label}')
    ax.plot(ew_q_pv.index, ew_q_pv / 1e9, '--', color='grey',
            lw=1.5, label='EW Benchmark')
    ax.plot(bh_pv.index, bh_pv / 1e9, '-.', color='forestgreen',
            lw=1.5, label='Buy & Hold')
    ax.set_title(f"Threshold = {thresh}", fontsize=11)
    ax.set_ylabel("Value ($B)")
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("signal_quarterly_tc_sensitivity.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Figure Q4: Quarterly — Strategy Overview ──────────────────────────────────
fig, ax = plt.subplots(figsize=(16, 7))
fig.suptitle("Quarterly Strategy Overview — Buy & Hold vs EW vs Signal (TC=0.5%)",
             fontsize=13, fontweight='bold')

ax.plot(bh_pv.index, bh_pv / 1e9,
        color='forestgreen', lw=2.0, ls='-.', label='Buy & Hold (EW)')
ax.plot(ew_q_pv.index, ew_q_pv / 1e9,
        color='grey', lw=2.0, ls='--', label='EW Rebalanced (quarterly)')

for thresh, col in thresh_colors.items():
    label = f"Signal3S_t{thresh}_quarterly_TC0.5%"
    pv    = signal_results[label]['portfolio_value']
    ax.plot(pv.index, pv / 1e9, color=col, lw=1.8,
            label=f'Signal t={thresh}')

ax.axhline(1.0, color='black', ls=':', lw=0.8)
ax.set_ylabel("Portfolio Value ($B)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("strategy_quarterly_overview.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Figure F1: Frequency Comparison — Portfolio Value ────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(22, 6))
fig.suptitle("Signal Portfolio — Rebalancing Frequency Comparison (TC=0.5%)",
             fontsize=13, fontweight='bold')

for ax, thresh in zip(axes, THRESHOLDS):
    for freq, col in freq_colors.items():
        label = f"Signal3S_t{thresh}_{freq}_TC0.5%"
        if label in signal_results:
            pv = signal_results[label]['portfolio_value']
            ax.plot(pv.index, pv / 1e9, color=col, lw=1.4,
                    label=freq.capitalize())
    ax.plot(bh_pv.index, bh_pv / 1e9,
            color='forestgreen', lw=1.8, ls='-.', label='Buy & Hold')
    ax.plot(ew_ref_pv.index, ew_ref_pv / 1e9,
            color='grey', lw=1.5, ls='--', label='EW Monthly')
    ax.axhline(1.0, color='black', ls=':', lw=0.8)
    ax.set_title(f"Threshold = {thresh}", fontsize=11)
    ax.set_ylabel("Value ($B)")
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("signal_frequency_comparison_value.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Figure F2: Frequency Comparison — Drawdowns ───────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(22, 5), sharey=True)
fig.suptitle("Signal Portfolio — Drawdowns by Frequency (TC=0.5%)",
             fontsize=13, fontweight='bold')

for ax, thresh in zip(axes, THRESHOLDS):
    for freq, col in freq_colors.items():
        label = f"Signal3S_t{thresh}_{freq}_TC0.5%"
        if label in signal_results:
            pv = signal_results[label]['portfolio_value']
            dd = (pv - pv.cummax()) / pv.cummax() * 100
            ax.plot(dd.index, dd.values, color=col, lw=1.2,
                    label=freq.capitalize())
    bh_dd = (bh_pv - bh_pv.cummax()) / bh_pv.cummax() * 100
    ax.plot(bh_dd.index, bh_dd.values,
            color='forestgreen', lw=1.5, ls='-.', label='Buy & Hold')
    ax.fill_between(bh_dd.index, bh_dd.values, 0, alpha=0.08, color='forestgreen')
    ax.set_title(f"Threshold = {thresh}", fontsize=11)
    ax.set_ylabel("Drawdown (%)")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("signal_frequency_comparison_drawdown.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Figure F3: EW Benchmark — Frequency Comparison ───────────────────────────
fig, ax = plt.subplots(figsize=(16, 7))
fig.suptitle("Equal-Weight Rebalanced — Frequency Comparison (TC=0.5%)",
             fontsize=13, fontweight='bold')

for freq, col in freq_colors.items():
    label = f"EW_{freq}_TC0.5%"
    if label in ew_results:
        pv = ew_results[label]['portfolio_value']
        ax.plot(pv.index, pv / 1e9, color=col, lw=1.6,
                label=f'EW {freq.capitalize()}')

ax.plot(bh_pv.index, bh_pv / 1e9,
        color='forestgreen', lw=2.0, ls='-.', label='Buy & Hold')
ax.axhline(1.0, color='black', ls=':', lw=0.8)
ax.set_ylabel("Portfolio Value ($B)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("ew_frequency_comparison.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Figure F4: TC Impact — Daily vs Weekly (Signal t=1.0) ────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("TC Impact on High-Frequency Rebalancing — Signal t=1.0",
             fontsize=13, fontweight='bold')

for ax, freq in zip(axes, ['daily', 'weekly']):
    for tc_label, col in colors_tc.items():
        label = f"Signal3S_t1.0_{freq}_TC{tc_label}"
        if label in signal_results:
            pv = signal_results[label]['portfolio_value']
            ax.plot(pv.index, pv / 1e9, color=col, lw=1.4,
                    label=f'TC={tc_label}')
    ax.plot(bh_pv.index, bh_pv / 1e9,
            color='forestgreen', lw=1.8, ls='-.', label='Buy & Hold')
    ax.axhline(1.0, color='black', ls=':', lw=0.8)
    ax.set_title(f"{freq.capitalize()} Rebalancing", fontsize=11)
    ax.set_ylabel("Value ($B)")
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("tc_impact_high_frequency.png", dpi=150, bbox_inches='tight')
plt.show()

# ── 17. SAVE ──────────────────────────────────────────────────────────────────
for label, df in {**bh_results, **ew_results, **signal_results}.items():
    df.to_csv(f"{label}_portfolio.csv")

all_metrics = pd.DataFrame({**bh_metrics, **ew_metrics, **sig_metrics}).T
all_metrics.to_csv("all_portfolio_metrics.csv")

composite_score.to_csv("composite_signal.csv")

for thresh in THRESHOLDS:
    sig_df_t = build_signal_df(composite_score, equity_assets,
                               macro_df.index, threshold=thresh)
    sig_df_t.to_csv(f"signal_df_threshold_{thresh}.csv")

print("✓ All files saved.")


In [ ]:
# @title
# ============================================================
# SIGNAL PORTFOLIO — THREE-STAGE THRESHOLD VERSION
# Signals generated from Modified Determinant Model
# + Buy-and-Hold Benchmark
# + Equal-Weight Benchmark
# + Three-Stage Signal Portfolio
# + Mean-Variance Signal Portfolio
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import requests
from io import StringIO
from scipy.optimize import minimize

# ── 1. LOAD ALL DATA FROM GITHUB ─────────────────────────────────────────────
GITHUB_BASE = "https://raw.githubusercontent.com/kboroz/MSFE_Capstone_Project/main/01_Data/Streamlined/"

INDEX_FILE_MAP = {
    'ASX50':       'streamlined_asx_50.csv',
    'EUROSTOXX50': 'streamlined_euro_stoxx_50.csv',
    'FTSE100':     'streamlined_ftse_100.csv',
    'Ibovespa':    'streamlined_ibovespa.csv',
    'JSE40':       'streamlined_jse_top_40.csv',
    'NIKKEI225':   'streamlined_nikkei_225.csv',
    'SP500':       'streamlined_s&p_500.csv',
}

MACRO_FILE = 'streamlined_macro.csv'

def load_csv(url):
    r = requests.get(url)
    r.raise_for_status()
    df = pd.read_csv(StringIO(r.text), index_col=0, parse_dates=True)
    df.columns = df.columns.str.replace('[^a-zA-Z0-9]', '_', regex=True)
    return df.sort_index()

macro_df = load_csv(GITHUB_BASE + MACRO_FILE)
macro_df = macro_df[~macro_df.index.duplicated(keep='first')].sort_index()
print(f"macro_df shape : {macro_df.shape}")
print(f"Columns        : {macro_df.columns.tolist()}")

index_data = {}
for name, fname in INDEX_FILE_MAP.items():
    try:
        df = load_csv(GITHUB_BASE + fname)
        index_data[name] = df
        print(f"✓ {name:15s} | shape={df.shape} | cols={df.columns.tolist()[:4]}")
    except Exception as e:
        print(f"✗ {name}: {e}")

# ── 2. EXTRACT PRICE SERIES ───────────────────────────────────────────────────
PRICE_COLS = ['Close', 'close', 'Price', 'price', 'Adj_Close', 'Last']

def get_price_series(df, name):
    col = next((c for c in PRICE_COLS if c in df.columns), df.columns[0])
    s = pd.to_numeric(df[col], errors='coerce').dropna()
    print(f"  {name}: using column '{col}'")
    return s

index_prices = {}
for name, df in index_data.items():
    index_prices[name] = get_price_series(df, name)

# ── 3. MODIFIED DETERMINANT SIGNAL MODEL ─────────────────────────────────────
DET_WINDOWS = [3, 7, 28]
WINDOW_WEIGHTS = {3: 0.33, 7: 0.34, 28: 0.33}

def compute_modified_determinant(price_series_dict, windows=DET_WINDOWS):
    prices = pd.DataFrame(price_series_dict).sort_index()
    rets = prices.pct_change().dropna()
    det_results = {}

    for window in windows:
        det_series = []
        for i in range(window, len(rets)):
            window_rets = rets.iloc[i - window:i]
            valid_cols = window_rets.columns[window_rets.std() > 1e-10]
            w = window_rets[valid_cols]

            if len(valid_cols) < 2:
                det_series.append(np.nan)
                continue

            try:
                R = w.corr().values
                I = np.eye(len(R))
                M = R - I
                det = np.linalg.det(M)
            except Exception:
                det = np.nan

            det_series.append(det)

        idx = rets.index[window:]
        det_results[window] = pd.Series(det_series, index=idx, name=f'det_{window}d')
        print(f"  Window {window:2d}d | computed {len(det_series)} observations")

    return det_results

print("\n── Computing Modified Determinant ──")
det_results = compute_modified_determinant(index_prices)

# ── 4. COMPOSITE SCORE ────────────────────────────────────────────────────────
def compute_composite_score(det_results, window_weights=WINDOW_WEIGHTS):
    all_series = []
    for window, series in det_results.items():
        mu = series.rolling(252, min_periods=60).mean()
        sigma = series.rolling(252, min_periods=60).std()
        z = (series - mu) / sigma.replace(0, np.nan)
        z.name = f'z_{window}d'
        all_series.append(z)

    z_df = pd.concat(all_series, axis=1).dropna()
    composite = sum(
        z_df[f'z_{w}d'] * window_weights[w]
        for w in window_weights if f'z_{w}d' in z_df.columns
    )
    composite = composite / sum(window_weights.values())
    return composite, z_df

composite_score, z_df = compute_composite_score(det_results)

print(f"\nComposite score shape  : {composite_score.shape}")
print(f"Date range             : {composite_score.index[0].date()} → {composite_score.index[-1].date()}")
print(f"Score range            : {composite_score.min():.3f} → {composite_score.max():.3f}")

# ── 5. THREE-STAGE SIGNAL CLASSIFICATION ─────────────────────────────────────
THRESHOLDS = [0.5, 1.0, 1.5]

def classify_signal(score, threshold):
    if abs(score) > threshold:
        return +1
    else:
        return -1

def build_signal_df(composite_score, equity_assets, macro_dates, threshold=0.5):
    binary = composite_score.apply(lambda x: classify_signal(x, threshold))
    signal_df = pd.DataFrame(index=macro_dates)
    for asset in equity_assets:
        signal_df[asset] = binary.reindex(macro_dates).ffill().bfill()
    return signal_df

# ── 6. ASSET UNIVERSE ─────────────────────────────────────────────────────────
EQUITY_ASSETS = list(index_prices.keys())
SAFE_BONDS = ['US_10Y', 'UK_10Y', 'AU_10Y', 'ZA_10Y']
FX = ['AUDUSD', 'BRLUSD', 'EURUSD', 'GBPUSD', 'JPYUSD', 'ZARUSD']
COMMODITIES = ['Oil', 'Gold', 'Bitcoin']
EXCLUDE_ASSETS = ['Bitcoin']

ALL_ASSETS = EQUITY_ASSETS + FX + COMMODITIES + SAFE_BONDS
ALL_ASSETS = [a for a in ALL_ASSETS if a in macro_df.columns and a not in EXCLUDE_ASSETS]

equity_assets = [a for a in EQUITY_ASSETS if a in ALL_ASSETS]
non_equity_assets = [a for a in ALL_ASSETS if a not in equity_assets]

N = len(ALL_ASSETS)
base_weight = 1.0 / N

print(f"\nTotal assets              : {N}")
print(f"Equity (signal-driven)    : {equity_assets}")
print(f"Non-equity (always long)  : {non_equity_assets}")
print(f"base_weight = 1/{N} = {base_weight:.4f}")

# ── 7. COUNTRY SAFE-HAVEN MAP ─────────────────────────────────────────────────
EQUITY_SAFE_HAVEN_MAP = {
    'ASX50':       ['AU_10Y', 'AUDUSD'],
    'EUROSTOXX50': ['EURUSD'],
    'FTSE100':     ['UK_10Y', 'GBPUSD'],
    'Ibovespa':    ['BRLUSD'],
    'JSE40':       ['ZA_10Y', 'ZARUSD'],
    'NIKKEI225':   ['JPYUSD'],
    'SP500':       ['US_10Y'],
}

UNIVERSE_SET = set(ALL_ASSETS)
for eq, havens in EQUITY_SAFE_HAVEN_MAP.items():
    EQUITY_SAFE_HAVEN_MAP[eq] = [h for h in havens if h in UNIVERSE_SET]

print("\nSafe-haven routing (filtered to universe):")
for eq, havens in EQUITY_SAFE_HAVEN_MAP.items():
    print(f"  {eq:15s} → {havens}")

# ── 8. BUILD RETURNS ──────────────────────────────────────────────────────────
returns_raw = macro_df[ALL_ASSETS].pct_change(fill_method=None)

for col in SAFE_BONDS:
    if col in returns_raw.columns:
        returns_raw[col] = -returns_raw[col]

if 'Oil' in macro_df.columns:
    bad_oil = macro_df['Oil'][macro_df['Oil'] <= 0].index
    for d in bad_oil:
        loc = returns_raw.index.get_loc(d)
        returns_raw.iloc[max(0, loc - 1): loc + 2, returns_raw.columns.get_loc('Oil')] = 0.0

returns = returns_raw.clip(lower=-0.50, upper=1.00).fillna(0).iloc[1:]

assert not returns.isnull().any().any(), "NaNs in returns"
assert not np.isinf(returns.values).any(), "Inf in returns"
print(f"\nReturns shape : {returns.shape}")
print("✓ Returns clean")

# ── 9. HELPER: BUILD REBALANCING DATE SET ────────────────────────────────────
def build_reb_set(returns, rebal_freq):
    actual_dates = returns.index
    if rebal_freq == 'daily':
        return set(actual_dates)
    elif rebal_freq == 'weekly':
        raw = returns.resample('W-MON').first().index
    elif rebal_freq == 'monthly':
        raw = returns.resample('MS').first().index
    elif rebal_freq == 'quarterly':
        raw = returns.resample('QS').first().index
    else:
        raise ValueError(f"Unknown rebal_freq: {rebal_freq}")

    reb_set = set()
    for d in raw:
        future = actual_dates[actual_dates >= d]
        if len(future) > 0:
            reb_set.add(future[0])
    return reb_set

# ── 10. BUY-AND-HOLD BENCHMARK ───────────────────────────────────────────────
def run_buy_and_hold(returns, all_assets, initial_capital=1e9):
    N = len(all_assets)
    base_weight = 1.0 / N
    pv = float(initial_capital)
    holdings = {a: pv * base_weight for a in all_assets}
    records = []
    first = True

    for date in returns.index:
        if first:
            first = False
            records.append({'date': date, 'portfolio_value': pv})
            continue
        new_holdings = {}
        for a in all_assets:
            new_holdings[a] = holdings[a] * (1 + returns.loc[date, a])
        holdings = new_holdings
        pv = sum(holdings.values())
        records.append({'date': date, 'portfolio_value': pv})

    df = pd.DataFrame(records).set_index('date')
    print(f"  Buy-and-Hold | Start: ${initial_capital/1e9:.2f}B → End: ${df['portfolio_value'].iloc[-1]/1e9:.3f}B")
    return df

# ── 11. THREE-STAGE WEIGHT BUILDER ───────────────────────────────────────────
def build_weights_3stage(sig_row, equity_assets, non_equity_assets,
                         base_weight, date, safe_haven_map):
    raw = {a: base_weight for a in non_equity_assets}
    safe_haven_extra = {}

    for a in equity_assets:
        if sig_row is not None and a in sig_row.index and not pd.isna(sig_row[a]):
            sig = sig_row[a]
        else:
            sig = -1

        if sig > 0:
            raw[a] = -base_weight
            havens = safe_haven_map.get(a, [])
            havens = [h for h in havens if h in raw]

            if havens:
                per_haven = base_weight / len(havens)
                for h in havens:
                    safe_haven_extra[h] = safe_haven_extra.get(h, 0) + per_haven
            else:
                per_ne = base_weight / len(non_equity_assets)
                for ne in non_equity_assets:
                    safe_haven_extra[ne] = safe_haven_extra.get(ne, 0) + per_ne
        else:
            raw[a] = base_weight

    for asset, extra in safe_haven_extra.items():
        raw[asset] = raw.get(asset, 0) + extra

    total = sum(raw.values())
    if total <= 0:
        for a in raw:
            raw[a] = 1.0 / len(raw)
    else:
        for a in raw:
            raw[a] /= total

    return raw

# ── 12. EQUAL-WEIGHT REBALANCED BENCHMARK ────────────────────────────────────
def run_equal_weight(returns, all_assets, tc_rate=0.005,
                     rebal_freq='monthly', initial_capital=1e9):
    N = len(all_assets)
    base_weight = 1.0 / N
    ew = {a: base_weight for a in all_assets}

    reb_set = build_reb_set(returns, rebal_freq)
    actual_dates = returns.index

    pv = float(initial_capital)
    current_weights = None
    records = []

    for date in actual_dates:
        if date in reb_set:
            if current_weights is not None:
                turnover = sum(abs(ew.get(a, 0) - current_weights.get(a, 0))
                               for a in set(ew) | set(current_weights)) / 2.0
                pv *= (1 - turnover * tc_rate)
            else:
                pv *= (1 - tc_rate)
            current_weights = ew.copy()

        if current_weights is not None:
            dr = sum(current_weights.get(a, 0) * returns.loc[date, a] for a in all_assets)
            pv *= (1 + dr)

        records.append({'date': date, 'portfolio_value': pv})

    return pd.DataFrame(records).set_index('date')

# ── 13. THREE-STAGE SIGNAL PORTFOLIO RUNNER ──────────────────────────────────
def run_signal_portfolio_3stage(returns, signal_df, equity_assets,
                                non_equity_assets, all_assets,
                                freq='monthly', tc_rate=0.005,
                                safe_haven_map=None,
                                initial_capital=1e9):
    if safe_haven_map is None:
        safe_haven_map = {}

    N = len(all_assets)
    base_weight = 1.0 / N

    reb_set = build_reb_set(returns, freq)
    actual_dates = returns.index

    sig = signal_df.copy()
    sig.index = pd.to_datetime(sig.index)

    pv = float(initial_capital)
    current_weights = None
    records = []

    for date in actual_dates:
        if date in reb_set:
            past = sig[sig.index <= date]
            sig_row = past.iloc[-1] if len(past) > 0 else None

            raw = build_weights_3stage(
                sig_row, equity_assets, non_equity_assets,
                base_weight, date, safe_haven_map
            )

            if current_weights is not None:
                all_keys = set(raw) | set(current_weights)
                turnover = sum(abs(raw.get(a, 0) - current_weights.get(a, 0)) for a in all_keys) / 2.0
                pv *= (1 - turnover * tc_rate)
            else:
                pv *= (1 - tc_rate)

            current_weights = raw.copy()

        if current_weights is not None:
            dr = sum(current_weights.get(a, 0) * returns.loc[date, a] for a in all_assets)
            pv *= (1 + dr)

        records.append({'date': date, 'portfolio_value': pv})

    return pd.DataFrame(records).set_index('date')

# ── 14. MEAN-VARIANCE SIGNAL PORTFOLIO ───────────────────────────────────────
MV_LOOKBACK = 252
MV_MIN_OBS = 60
MAX_WEIGHT = 0.30
RISK_FREE = 0.0

def mean_variance_weights(mu_vec, cov_mat, allowed_mask,
                          max_weight=MAX_WEIGHT,
                          risk_free=RISK_FREE):
    n_total = len(mu_vec)
    allowed = np.where(allowed_mask)[0]
    n_allowed = len(allowed)

    fallback = np.zeros(n_total)
    if n_allowed == 0:
        fallback[:] = 1.0 / n_total
        return fallback

    fallback[allowed] = 1.0 / n_allowed

    mu_sub = mu_vec[allowed]
    cov_sub = cov_mat[np.ix_(allowed, allowed)]
    cov_sub = cov_sub + np.eye(n_allowed) * 1e-8

    def neg_sharpe(w):
        port_ret = np.dot(w, mu_sub)
        port_var = np.dot(w, cov_sub @ w)
        port_std = np.sqrt(max(port_var, 1e-12))
        return -(port_ret - risk_free) / port_std

    constraints = [{'type': 'eq', 'fun': lambda w: np.sum(w) - 1.0}]
    bounds = [(0.0, max_weight)] * n_allowed
    w0 = np.ones(n_allowed) / n_allowed

    try:
        res = minimize(
            neg_sharpe, w0,
            method='SLSQP',
            bounds=bounds,
            constraints=constraints,
            options={'ftol': 1e-9, 'maxiter': 500}
        )
        if res.success and not np.any(np.isnan(res.x)):
            w_opt = res.x
        else:
            w_opt = w0
    except Exception:
        w_opt = w0

    w_full = np.zeros(n_total)
    w_full[allowed] = w_opt
    return w_full

def run_mv_signal_portfolio(returns, signal_df, all_assets,
                            equity_assets,
                            freq='monthly', tc_rate=0.005,
                            lookback=MV_LOOKBACK,
                            min_obs=MV_MIN_OBS,
                            max_weight=MAX_WEIGHT,
                            initial_capital=1e9):
    """
    Mean-Variance portfolio with signal constraints.

    Signal logic per asset at each rebalance:
      +1 (RISK_ON / defensive) -> weight forced to 0
      -1 (RISK_OFF / long)     -> eligible for optimiser

    Non-equity assets are always eligible (never blocked).
    """
    reb_set = build_reb_set(returns, freq)
    actual_dates = returns.index
    assets = all_assets
    n = len(assets)
    asset_idx = {a: i for i, a in enumerate(assets)}

    sig = signal_df.copy()
    sig.index = pd.to_datetime(sig.index)

    pv = float(initial_capital)
    current_weights = np.zeros(n)
    records = []
    weight_history = []
    first_reb = True

    for t, date in enumerate(actual_dates):
        if date in reb_set:
            hist = returns.iloc[max(0, t - lookback):t]

            if len(hist) < min_obs:
                new_weights = np.ones(n) / n
            else:
                mu_daily = hist.mean().values
                cov_daily = hist.cov().values

                mu_ann = mu_daily * 252
                cov_ann = cov_daily * 252

                past = sig[sig.index <= date]
                sig_row = past.iloc[-1] if len(past) > 0 else None

                allowed_mask = np.ones(n, dtype=bool)

                if sig_row is not None:
                    for eq in equity_assets:
                        if eq in asset_idx and eq in sig_row.index:
                            s = sig_row[eq]
                            if not pd.isna(s) and s > 0:
                                allowed_mask[asset_idx[eq]] = False

                new_weights = mean_variance_weights(
                    mu_ann, cov_ann, allowed_mask, max_weight=max_weight
                )

            if first_reb:
                pv *= (1 - tc_rate)
                first_reb = False
            else:
                turnover = np.sum(np.abs(new_weights - current_weights)) / 2.0
                pv *= (1 - turnover * tc_rate)

            current_weights = new_weights.copy()
            weight_history.append({'date': date, **dict(zip(assets, current_weights))})

        if not first_reb:
            dr = np.dot(current_weights, returns.loc[date].values)
            pv *= (1 + dr)

        records.append({'date': date, 'portfolio_value': pv})

    pv_df = pd.DataFrame(records).set_index('date')
    weights_df = pd.DataFrame(weight_history).set_index('date')

    print(f"  MV-Signal | freq={freq} | tc={tc_rate*100:.1f}% | Start: $1.00B → End: ${pv_df['portfolio_value'].iloc[-1]/1e9:.3f}B")
    return pv_df, weights_df

# ── 15. RUN ALL PORTFOLIOS ────────────────────────────────────────────────────
TC_LEVELS = {'0%': 0.00, '0.5%': 0.005, '1%': 0.01, '2%': 0.02}
FREQS = ['daily', 'weekly', 'monthly', 'quarterly']
MV_FREQS = ['monthly', 'quarterly']

print("\n── Running Buy-and-Hold ──")
bh_result = run_buy_and_hold(returns, ALL_ASSETS, initial_capital=1e9)
bh_results = {'BuyAndHold_EW': bh_result}

print("\n── Running Equal-Weight Rebalanced ──")
ew_results = {}
for freq in FREQS:
    for tc_label, tc_rate in TC_LEVELS.items():
        label = f"EW_{freq}_TC{tc_label}"
        ew_results[label] = run_equal_weight(
            returns, ALL_ASSETS, tc_rate=tc_rate,
            rebal_freq=freq, initial_capital=1e9
        )
        print(f"  ✓ {label}")

print("\n── Running Signal Portfolios ──")
signal_results = {}
signal_distributions = {}

for thresh in THRESHOLDS:
    sig_df_t = build_signal_df(composite_score, equity_assets, macro_df.index, threshold=thresh)
    n_risk_on = (sig_df_t[equity_assets[0]] == +1).sum()
    n_risk_off = (sig_df_t[equity_assets[0]] == -1).sum()
    total = len(sig_df_t)

    print(f"\n  Threshold={thresh} | RISK_ON(+1): {n_risk_on} days ({n_risk_on/total*100:.1f}%) | "
          f"RISK_OFF(-1): {n_risk_off} days ({n_risk_off/total*100:.1f}%)")

    signal_distributions[thresh] = {'RISK_ON': n_risk_on, 'RISK_OFF': n_risk_off}

    for freq in FREQS:
        for tc_label, tc_rate in TC_LEVELS.items():
            label = f"Signal3S_t{thresh}_{freq}_TC{tc_label}"
            signal_results[label] = run_signal_portfolio_3stage(
                returns, sig_df_t, equity_assets, non_equity_assets,
                ALL_ASSETS, freq=freq, tc_rate=tc_rate,
                safe_haven_map=EQUITY_SAFE_HAVEN_MAP,
                initial_capital=1e9
            )
            print(f"    ✓ {label}")

print("\n── Running Mean-Variance Signal Portfolios ──")
mv_results = {}
mv_weight_history = {}

for thresh in THRESHOLDS:
    sig_df_t = build_signal_df(composite_score, equity_assets, macro_df.index, threshold=thresh)
    for freq in MV_FREQS:
        for tc_label, tc_rate in TC_LEVELS.items():
            label = f"MV_t{thresh}_{freq}_TC{tc_label}"
            pv_df, w_df = run_mv_signal_portfolio(
                returns, sig_df_t, ALL_ASSETS,
                equity_assets=equity_assets,
                freq=freq, tc_rate=tc_rate,
                lookback=MV_LOOKBACK,
                min_obs=MV_MIN_OBS,
                max_weight=MAX_WEIGHT,
                initial_capital=1e9
            )
            mv_results[label] = pv_df
            mv_weight_history[label] = w_df
            print(f"  ✓ {label}")

print("\n✓ All portfolios computed")

# ── 16. PERFORMANCE METRICS ───────────────────────────────────────────────────
def compute_metrics(pv_df, label=''):
    pv = pv_df['portfolio_value']
    daily_ret = pv.pct_change().dropna()
    n_years = len(daily_ret) / 252
    total_ret = (pv.iloc[-1] / pv.iloc[0]) - 1
    cagr = (1 + total_ret) ** (1 / n_years) - 1 if n_years > 0 else np.nan
    vol = daily_ret.std() * np.sqrt(252)
    sharpe = cagr / vol if vol > 0 else np.nan
    roll_max = pv.cummax()
    max_dd = ((pv - roll_max) / roll_max).min()
    calmar = cagr / abs(max_dd) if max_dd != 0 else np.nan
    var_95 = daily_ret.quantile(0.05)

    return {
        'CAGR': f"{cagr*100:.2f}%",
        'Volatility': f"{vol*100:.2f}%",
        'Sharpe': f"{sharpe:.2f}",
        'Max Drawdown': f"{max_dd*100:.2f}%",
        'Calmar': f"{calmar:.2f}",
        'VaR 95%': f"{var_95*100:.2f}%",
        'Total Return': f"{total_ret*100:.2f}%",
        'Final ( $ B)': f"{pv.iloc[-1]/1e9:.3f}",
    }

bh_metrics = {k: compute_metrics(v, k) for k, v in bh_results.items()}
ew_metrics = {k: compute_metrics(v, k) for k, v in ew_results.items()}
sig_metrics = {k: compute_metrics(v, k) for k, v in signal_results.items()}
mv_metrics = {k: compute_metrics(v, k) for k, v in mv_results.items()}

print("\n" + "="*90)
print("BUY-AND-HOLD BENCHMARK")
print("="*90)
print(pd.DataFrame(bh_metrics).T.to_string())

print("\n" + "="*90)
print("EQUAL-WEIGHT REBALANCED BENCHMARK")
print("="*90)
print(pd.DataFrame(ew_metrics).T.to_string())

for thresh in THRESHOLDS:
    sub = {k: v for k, v in sig_metrics.items() if f"_t{thresh}_" in k}
    print(f"\n{'='*90}")
    print(f"THREE-STAGE SIGNAL — Threshold = {thresh}")
    print("="*90)
    print(pd.DataFrame(sub).T.to_string())

for thresh in THRESHOLDS:
    sub = {k: v for k, v in mv_metrics.items() if f"_t{thresh}_" in k}
    print(f"\n{'='*90}")
    print(f"MEAN-VARIANCE SIGNAL — Threshold = {thresh}")
    print("="*90)
    print(pd.DataFrame(sub).T.to_string())

# ── 17. PLOTS ─────────────────────────────────────────────────────────────────
colors_tc = {'0%': 'black', '0.5%': 'steelblue', '1%': 'darkorange', '2%': 'crimson'}
freq_colors = {'daily': 'crimson', 'weekly': 'darkorange', 'monthly': 'steelblue', 'quarterly': 'purple'}
thresh_colors = {0.5: 'steelblue', 1.0: 'darkorange', 1.5: 'crimson'}

bh_pv = bh_results['BuyAndHold_EW']['portfolio_value']
ew_ref_pv = ew_results['EW_monthly_TC0.5%']['portfolio_value']
ew_q_pv = ew_results['EW_quarterly_TC0.5%']['portfolio_value']

# Figure 1
fig, axes = plt.subplots(2, 1, figsize=(16, 10))
fig.suptitle("Modified Determinant — Composite Score & Threshold Signals",
             fontsize=14, fontweight='bold')

ax = axes[0]
ax.plot(composite_score.index, composite_score.values,
        color='navy', lw=1.0, alpha=0.8, label='Composite Z-score')
for thresh, col in zip(THRESHOLDS, ['green', 'orange', 'red']):
    ax.axhline(thresh, color=col, ls='--', lw=1.0, label=f'+{thresh}')
    ax.axhline(-thresh, color=col, ls='--', lw=1.0, label=f'-{thresh}')
ax.axhline(0, color='black', lw=0.6)
ax.set_title("Composite Determinant Score with Thresholds", fontsize=11)
ax.set_ylabel("Z-score")
ax.legend(fontsize=8, ncol=4)
ax.grid(True, alpha=0.3)

ax = axes[1]
for window, series in det_results.items():
    ax.plot(series.index, np.sign(series.values), alpha=0.6,
            label=f"{window}d window", lw=1.0)
ax.axhline(0, color='black', lw=0.8)
ax.set_title("Sign of Det(R-I) per Window", fontsize=11)
ax.set_ylabel("+1 / -1")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("composite_signal_thresholds.png", dpi=150, bbox_inches='tight')
plt.show()

# Figure 2
fig, axes = plt.subplots(1, 3, figsize=(22, 6), sharey=False)
fig.suptitle("Three-Stage Signal vs Benchmarks — Portfolio Value (TC=0.5%, Monthly)",
             fontsize=13, fontweight='bold')

for ax, thresh in zip(axes, THRESHOLDS):
    ax.plot(bh_pv.index, bh_pv / 1e9, color='forestgreen', lw=1.8, ls='-.', label='Buy & Hold (EW)')
    ax.plot(ew_ref_pv.index, ew_ref_pv / 1e9, color='grey', lw=1.5, ls='--', label='EW Rebalanced')
    label = f"Signal3S_t{thresh}_monthly_TC0.5%"
    pv = signal_results[label]['portfolio_value']
    ax.plot(pv.index, pv / 1e9, color='steelblue', lw=1.8, label=f'Signal (t={thresh})')
    ax.axhline(1.0, color='black', ls=':', lw=0.8)
    ax.set_title(f"Threshold = {thresh}", fontsize=11)
    ax.set_ylabel("Value ($B)")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("signal_3stage_portfolio_value.png", dpi=150, bbox_inches='tight')
plt.show()

# Figure 3
fig, axes = plt.subplots(1, 3, figsize=(22, 5), sharey=True)
fig.suptitle("Three-Stage Signal — Drawdowns (TC=0.5%, Monthly)",
             fontsize=13, fontweight='bold')

for ax, thresh in zip(axes, THRESHOLDS):
    label = f"Signal3S_t{thresh}_monthly_TC0.5%"
    pv = signal_results[label]['portfolio_value']
    dd = (pv - pv.cummax()) / pv.cummax() * 100
    ew_dd = (ew_ref_pv - ew_ref_pv.cummax()) / ew_ref_pv.cummax() * 100
    bh_dd = (bh_pv - bh_pv.cummax()) / bh_pv.cummax() * 100

    ax.fill_between(dd.index, dd.values, 0, alpha=0.25, color='steelblue')
    ax.fill_between(ew_dd.index, ew_dd.values, 0, alpha=0.15, color='grey')
    ax.fill_between(bh_dd.index, bh_dd.values, 0, alpha=0.10, color='forestgreen')
    ax.plot(dd.index, dd.values, color='steelblue', lw=1.2, label=f'Signal (t={thresh})')
    ax.plot(ew_dd.index, ew_dd.values, color='grey', lw=1.0, ls='--', label='EW Rebalanced')
    ax.plot(bh_dd.index, bh_dd.values, color='forestgreen', lw=1.0, ls='-.', label='Buy & Hold')
    ax.set_title(f"Threshold = {thresh}", fontsize=11)
    ax.set_ylabel("Drawdown (%)")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("signal_3stage_drawdowns.png", dpi=150, bbox_inches='tight')
plt.show()

# Figure 4
fig, axes = plt.subplots(1, 3, figsize=(22, 6))
fig.suptitle("Three-Stage Signal — TC Sensitivity vs Benchmarks (Monthly)",
             fontsize=13, fontweight='bold')

for ax, thresh in zip(axes, THRESHOLDS):
    for tc_label, col in colors_tc.items():
        label = f"Signal3S_t{thresh}_monthly_TC{tc_label}"
        pv = signal_results[label]['portfolio_value']
        ax.plot(pv.index, pv / 1e9, color=col, lw=1.3, label=f'Signal TC={tc_label}')
    ax.plot(ew_ref_pv.index, ew_ref_pv / 1e9, color='grey', lw=1.8, ls='--', label='EW Rebalanced')
    ax.plot(bh_pv.index, bh_pv / 1e9, color='forestgreen', lw=1.8, ls='-.', label='Buy & Hold')
    ax.set_title(f"Threshold = {thresh}", fontsize=11)
    ax.set_ylabel("Value ($B)")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("signal_3stage_tc_sensitivity.png", dpi=150, bbox_inches='tight')
plt.show()

# Figure 5
fig, ax = plt.subplots(figsize=(16, 7))
fig.suptitle("Strategy Overview — Buy & Hold vs EW vs Signal (TC=0.5%, Monthly)",
             fontsize=13, fontweight='bold')

ax.plot(bh_pv.index, bh_pv / 1e9, color='forestgreen', lw=2.0, ls='-.', label='Buy & Hold (EW)')
ax.plot(ew_ref_pv.index, ew_ref_pv / 1e9, color='grey', lw=2.0, ls='--', label='EW Rebalanced (monthly)')

for thresh, col in thresh_colors.items():
    label = f"Signal3S_t{thresh}_monthly_TC0.5%"
    pv = signal_results[label]['portfolio_value']
    ax.plot(pv.index, pv / 1e9, color=col, lw=1.8, label=f'Signal t={thresh}')

ax.axhline(1.0, color='black', ls=':', lw=0.8)
ax.set_ylabel("Portfolio Value ($B)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("strategy_monthly_overview.png", dpi=150, bbox_inches='tight')
plt.show()

# Figure Q1
fig, axes = plt.subplots(1, 3, figsize=(22, 6))
fig.suptitle("Three-Stage Signal — Quarterly Portfolio Value (TC=0.5%)",
             fontsize=13, fontweight='bold')

for ax, thresh in zip(axes, THRESHOLDS):
    label = f"Signal3S_t{thresh}_quarterly_TC0.5%"
    pv = signal_results[label]['portfolio_value']
    ax.plot(pv.index, pv / 1e9, color='steelblue', lw=1.8, label=f'Signal t={thresh}')
    ax.plot(ew_q_pv.index, ew_q_pv / 1e9, color='grey', lw=1.5, ls='--', label='EW Rebalanced')
    ax.plot(bh_pv.index, bh_pv / 1e9, color='forestgreen', lw=1.5, ls='-.', label='Buy & Hold')
    ax.axhline(1.0, color='black', ls=':', lw=0.8)
    ax.set_title(f"Threshold = {thresh}", fontsize=11)
    ax.set_ylabel("Value ($B)")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("signal_quarterly_value.png", dpi=150, bbox_inches='tight')
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(22, 5), sharey=True)
fig.suptitle("Three-Stage Signal — Quarterly Drawdowns (TC=0.5%)",
             fontsize=13, fontweight='bold')

for ax, thresh in zip(axes, THRESHOLDS):
    label = f"Signal3S_t{thresh}_quarterly_TC0.5%"
    pv = signal_results[label]['portfolio_value']
    dd = (pv - pv.cummax()) / pv.cummax() * 100
    ew_dd = (ew_q_pv - ew_q_pv.cummax()) / ew_q_pv.cummax() * 100
    bh_dd = (bh_pv - bh_pv.cummax()) / bh_pv.cummax() * 100

    ax.fill_between(dd.index, dd.values, 0, alpha=0.25, color='steelblue')
    ax.fill_between(ew_dd.index, ew_dd.values, 0, alpha=0.15, color='grey')
    ax.fill_between(bh_dd.index, bh_dd.values, 0, alpha=0.10, color='forestgreen')
    ax.plot(dd.index, dd.values, color='steelblue', lw=1.2, label=f'Signal (t={thresh})')
    ax.plot(ew_dd.index, ew_dd.values, color='grey', lw=1.0, ls='--', label='EW Rebalanced')
    ax.plot(bh_dd.index, bh_dd.values, color='forestgreen', lw=1.0, ls='-.', label='Buy & Hold')
    ax.set_title(f"Threshold = {thresh}", fontsize=11)
    ax.set_ylabel("Drawdown (%)")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("signal_quarterly_drawdowns.png", dpi=150, bbox_inches='tight')
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(22, 6))
fig.suptitle("Three-Stage Signal — Quarterly TC Sensitivity",
             fontsize=13, fontweight='bold')

for ax, thresh in zip(axes, THRESHOLDS):
    for tc_label, col in colors_tc.items():
        label = f"Signal3S_t{thresh}_quarterly_TC{tc_label}"
        pv = signal_results[label]['portfolio_value']
        ax.plot(pv.index, pv / 1e9, color=col, lw=1.3, label=f'TC={tc_label}')
    ax.plot(ew_q_pv.index, ew_q_pv / 1e9, '--', color='grey', lw=1.5, label='EW Benchmark')
    ax.plot(bh_pv.index, bh_pv / 1e9, '-.', color='forestgreen', lw=1.5, label='Buy & Hold')
    ax.set_title(f"Threshold = {thresh}", fontsize=11)
    ax.set_ylabel("Value ($B)")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("signal_quarterly_tc_sensitivity.png", dpi=150, bbox_inches='tight')
plt.show()

fig, ax = plt.subplots(figsize=(16, 7))
fig.suptitle("Quarterly Strategy Overview — Buy & Hold vs EW vs Signal (TC=0.5%)",
             fontsize=13, fontweight='bold')

ax.plot(bh_pv.index, bh_pv / 1e9, color='forestgreen', lw=2.0, ls='-.', label='Buy & Hold (EW)')
ax.plot(ew_q_pv.index, ew_q_pv / 1e9, color='grey', lw=2.0, ls='--', label='EW Rebalanced (quarterly)')

for thresh, col in thresh_colors.items():
    label = f"Signal3S_t{thresh}_quarterly_TC0.5%"
    pv = signal_results[label]['portfolio_value']
    ax.plot(pv.index, pv / 1e9, color=col, lw=1.8, label=f'Signal t={thresh}')

ax.axhline(1.0, color='black', ls=':', lw=0.8)
ax.set_ylabel("Portfolio Value ($B)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("strategy_quarterly_overview.png", dpi=150, bbox_inches='tight')
plt.show()

# Figure F1
fig, axes = plt.subplots(1, 3, figsize=(22, 6))
fig.suptitle("Signal Portfolio — Rebalancing Frequency Comparison (TC=0.5%)",
             fontsize=13, fontweight='bold')

for ax, thresh in zip(axes, THRESHOLDS):
    for freq, col in freq_colors.items():
        label = f"Signal3S_t{thresh}_{freq}_TC0.5%"
        if label in signal_results:
            pv = signal_results[label]['portfolio_value']
            ax.plot(pv.index, pv / 1e9, color=col, lw=1.4, label=freq.capitalize())
    ax.plot(bh_pv.index, bh_pv / 1e9, color='forestgreen', lw=1.8, ls='-.', label='Buy & Hold')
    ax.plot(ew_ref_pv.index, ew_ref_pv / 1e9, color='grey', lw=1.5, ls='--', label='EW Monthly')
    ax.axhline(1.0, color='black', ls=':', lw=0.8)
    ax.set_title(f"Threshold = {thresh}", fontsize=11)
    ax.set_ylabel("Value ($B)")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("signal_frequency_comparison_value.png", dpi=150, bbox_inches='tight')
plt.show()

# Figure F2
fig, axes = plt.subplots(1, 3, figsize=(22, 5), sharey=True)
fig.suptitle("Signal Portfolio — Drawdowns by Frequency (TC=0.5%)",
             fontsize=13, fontweight='bold')

for ax, thresh in zip(axes, THRESHOLDS):
    for freq, col in freq_colors.items():
        label = f"Signal3S_t{thresh}_{freq}_TC0.5%"
        if label in signal_results:
            pv = signal_results[label]['portfolio_value']
            dd = (pv - pv.cummax()) / pv.cummax() * 100
            ax.plot(dd.index, dd.values, color=col, lw=1.2, label=freq.capitalize())
    bh_dd = (bh_pv - bh_pv.cummax()) / bh_pv.cummax() * 100
    ax.plot(bh_dd.index, bh_dd.values, color='forestgreen', lw=1.5, ls='-.', label='Buy & Hold')
    ax.fill_between(bh_dd.index, bh_dd.values, 0, alpha=0.08, color='forestgreen')
    ax.set_title(f"Threshold = {thresh}", fontsize=11)
    ax.set_ylabel("Drawdown (%)")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("signal_frequency_comparison_drawdown.png", dpi=150, bbox_inches='tight')
plt.show()

# Figure F3
fig, ax = plt.subplots(figsize=(16, 7))
fig.suptitle("Equal-Weight Rebalanced — Frequency Comparison (TC=0.5%)",
             fontsize=13, fontweight='bold')

for freq, col in freq_colors.items():
    label = f"EW_{freq}_TC0.5%"
    if label in ew_results:
        pv = ew_results[label]['portfolio_value']
        ax.plot(pv.index, pv / 1e9, color=col, lw=1.6, label=f'EW {freq.capitalize()}')

ax.plot(bh_pv.index, bh_pv / 1e9, color='forestgreen', lw=2.0, ls='-.', label='Buy & Hold')
ax.axhline(1.0, color='black', ls=':', lw=0.8)
ax.set_ylabel("Portfolio Value ($B)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("ew_frequency_comparison.png", dpi=150, bbox_inches='tight')
plt.show()

# Figure F4
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("TC Impact on High-Frequency Rebalancing — Signal t=1.0",
             fontsize=13, fontweight='bold')

for ax, freq in zip(axes, ['daily', 'weekly']):
    for tc_label, col in colors_tc.items():
        label = f"Signal3S_t1.0_{freq}_TC{tc_label}"
        if label in signal_results:
            pv = signal_results[label]['portfolio_value']
            ax.plot(pv.index, pv / 1e9, color=col, lw=1.4, label=f'TC={tc_label}')
    ax.plot(bh_pv.index, bh_pv / 1e9, color='forestgreen', lw=1.8, ls='-.', label='Buy & Hold')
    ax.axhline(1.0, color='black', ls=':', lw=0.8)
    ax.set_title(f"{freq.capitalize()} Rebalancing", fontsize=11)
    ax.set_ylabel("Value ($B)")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("tc_impact_high_frequency.png", dpi=150, bbox_inches='tight')
plt.show()

# MV Figure 1
fig, axes = plt.subplots(1, 3, figsize=(22, 6))
fig.suptitle("Mean-Variance Signal vs Three-Stage Signal vs Benchmarks (TC=0.5%, Monthly)",
             fontsize=13, fontweight='bold')

for ax, thresh in zip(axes, THRESHOLDS):
    ax.plot(bh_pv.index, bh_pv / 1e9, color='forestgreen', lw=1.8, ls='-.', label='Buy & Hold')
    ax.plot(ew_ref_pv.index, ew_ref_pv / 1e9, color='grey', lw=1.5, ls='--', label='EW Rebalanced')

    sig_label = f"Signal3S_t{thresh}_monthly_TC0.5%"
    mv_label = f"MV_t{thresh}_monthly_TC0.5%"

    sig_pv = signal_results[sig_label]['portfolio_value']
    ax.plot(sig_pv.index, sig_pv / 1e9, color='steelblue', lw=1.6, label=f'3-Stage Signal t={thresh}')

    if mv_label in mv_results:
        mv_pv = mv_results[mv_label]['portfolio_value']
        ax.plot(mv_pv.index, mv_pv / 1e9, color='crimson', lw=1.8, ls='-', label=f'MV Signal t={thresh}')

    ax.axhline(1.0, color='black', ls=':', lw=0.8)
    ax.set_title(f"Threshold = {thresh}", fontsize=11)
    ax.set_ylabel("Value ($B)")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("mv_vs_signal_monthly.png", dpi=150, bbox_inches='tight')
plt.show()

# MV Figure 2
fig, axes = plt.subplots(1, 3, figsize=(22, 5), sharey=True)
fig.suptitle("Mean-Variance Signal — Drawdowns (TC=0.5%, Monthly)",
             fontsize=13, fontweight='bold')

for ax, thresh in zip(axes, THRESHOLDS):
    mv_label = f"MV_t{thresh}_monthly_TC0.5%"
    if mv_label not in mv_results:
        continue

    mv_pv = mv_results[mv_label]['portfolio_value']
    mv_dd = (mv_pv - mv_pv.cummax()) / mv_pv.cummax() * 100
    bh_dd = (bh_pv - bh_pv.cummax()) / bh_pv.cummax() * 100
    ew_dd = (ew_ref_pv - ew_ref_pv.cummax()) / ew_ref_pv.cummax() * 100

    sig_label = f"Signal3S_t{thresh}_monthly_TC0.5%"
    sig_pv = signal_results[sig_label]['portfolio_value']
    sig_dd = (sig_pv - sig_pv.cummax()) / sig_pv.cummax() * 100

    ax.fill_between(mv_dd.index, mv_dd.values, 0, alpha=0.25, color='crimson')
    ax.fill_between(sig_dd.index, sig_dd.values, 0, alpha=0.15, color='steelblue')
    ax.plot(mv_dd.index, mv_dd.values, color='crimson', lw=1.4, label=f'MV Signal (t={thresh})')
    ax.plot(sig_dd.index, sig_dd.values, color='steelblue', lw=1.2, ls='--', label=f'3-Stage Signal (t={thresh})')
    ax.plot(bh_dd.index, bh_dd.values, color='forestgreen', lw=1.0, ls='-.', label='Buy & Hold')
    ax.plot(ew_dd.index, ew_dd.values, color='grey', lw=1.0, ls=':', label='EW Rebalanced')
    ax.set_title(f"Threshold = {thresh}", fontsize=11)
    ax.set_ylabel("Drawdown (%)")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("mv_signal_drawdowns.png", dpi=150, bbox_inches='tight')
plt.show()

# MV Figure 3
fig, axes = plt.subplots(1, 3, figsize=(22, 6))
fig.suptitle("Mean-Variance Signal — TC Sensitivity (Monthly)",
             fontsize=13, fontweight='bold')

for ax, thresh in zip(axes, THRESHOLDS):
    for tc_label, col in colors_tc.items():
        label = f"MV_t{thresh}_monthly_TC{tc_label}"
        if label in mv_results:
            pv = mv_results[label]['portfolio_value']
            ax.plot(pv.index, pv / 1e9, color=col, lw=1.4, label=f'MV TC={tc_label}')
    ax.plot(ew_ref_pv.index, ew_ref_pv / 1e9, color='grey', lw=1.8, ls='--', label='EW Rebalanced')
    ax.plot(bh_pv.index, bh_pv / 1e9, color='forestgreen', lw=1.8, ls='-.', label='Buy & Hold')
    ax.axhline(1.0, color='black', ls=':', lw=0.8)
    ax.set_title(f"Threshold = {thresh}", fontsize=11)
    ax.set_ylabel("Value ($B)")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("mv_signal_tc_sensitivity.png", dpi=150, bbox_inches='tight')
plt.show()

# MV Figure 4
fig, axes = plt.subplots(1, 3, figsize=(22, 6))
fig.suptitle("Mean-Variance Signal — Rebalancing Frequency Comparison (TC=0.5%)",
             fontsize=13, fontweight='bold')

for ax, thresh in zip(axes, THRESHOLDS):
    for freq, col in freq_colors.items():
        label = f"MV_t{thresh}_{freq}_TC0.5%"
        if label in mv_results:
            pv = mv_results[label]['portfolio_value']
            ax.plot(pv.index, pv / 1e9, color=col, lw=1.4, label=freq.capitalize())
    ax.plot(bh_pv.index, bh_pv / 1e9, color='forestgreen', lw=1.8, ls='-.', label='Buy & Hold')
    ax.plot(ew_ref_pv.index, ew_ref_pv / 1e9, color='grey', lw=1.5, ls='--', label='EW Monthly')
    ax.axhline(1.0, color='black', ls=':', lw=0.8)
    ax.set_title(f"Threshold = {thresh}", fontsize=11)
    ax.set_ylabel("Value ($B)")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("mv_signal_frequency_comparison.png", dpi=150, bbox_inches='tight')
plt.show()

# MV Figure 5
fig, axes = plt.subplots(len(THRESHOLDS), 1, figsize=(18, 14))
fig.suptitle("Mean-Variance Signal — Weight Evolution (Monthly, TC=0.5%)",
             fontsize=13, fontweight='bold')

if len(THRESHOLDS) == 1:
    axes = [axes]

for ax, thresh in zip(axes, THRESHOLDS):
    label = f"MV_t{thresh}_monthly_TC0.5%"
    if label not in mv_weight_history:
        continue
    wdf = mv_weight_history[label]
    for col in wdf.columns:
        ax.plot(wdf.index, wdf[col], lw=1.0, alpha=0.8, label=col)
    ax.set_title(f"Threshold = {thresh}", fontsize=11)
    ax.set_ylabel("Weight")
    ax.set_ylim(-0.05, MAX_WEIGHT + 0.05)
    ax.axhline(0, color='black', lw=0.6, ls='--')
    ax.legend(fontsize=7, ncol=4, loc='upper right')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("mv_signal_weight_evolution.png", dpi=150, bbox_inches='tight')
plt.show()

# MV Figure 6
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle("Sharpe Ratio Comparison — All Strategies (Monthly, TC=0.5%)",
             fontsize=13, fontweight='bold')

for ax, thresh in zip(axes, THRESHOLDS):
    labels_bar = []
    sharpes = []

    labels_bar.append('Buy & Hold')
    sharpes.append(float(bh_metrics['BuyAndHold_EW']['Sharpe']))

    labels_bar.append('EW Monthly')
    sharpes.append(float(ew_metrics['EW_monthly_TC0.5%']['Sharpe']))

    sig_key = f"Signal3S_t{thresh}_monthly_TC0.5%"
    labels_bar.append(f'3-Stage t={thresh}')
    sharpes.append(float(sig_metrics[sig_key]['Sharpe']))

    mv_key = f"MV_t{thresh}_monthly_TC0.5%"
    if mv_key in mv_metrics:
        labels_bar.append(f'MV t={thresh}')
        sharpes.append(float(mv_metrics[mv_key]['Sharpe']))

    bar_colors = ['forestgreen', 'grey', 'steelblue', 'crimson'][:len(labels_bar)]
    bars = ax.bar(labels_bar, sharpes, color=bar_colors, alpha=0.85, edgecolor='black')

    for bar, val in zip(bars, sharpes):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.01,
                f"{val:.2f}",
                ha='center', va='bottom', fontsize=9, fontweight='bold')

    ax.axhline(0, color='black', lw=0.8)
    ax.set_title(f"Threshold = {thresh}", fontsize=11)
    ax.set_ylabel("Sharpe Ratio")
    ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig("sharpe_comparison_bar.png", dpi=150, bbox_inches='tight')
plt.show()

# MV Figure 7
fig, ax = plt.subplots(figsize=(12, 8))
fig.suptitle("Risk-Return Profile — All Strategies (Monthly, TC=0.5%)",
             fontsize=13, fontweight='bold')

def parse_pct(s):
    return float(str(s).replace('%', ''))

strategy_groups = {
    'Buy & Hold':     (bh_metrics,  ['BuyAndHold_EW'], 'forestgreen', 'o'),
    'EW Monthly':     (ew_metrics,  ['EW_monthly_TC0.5%'], 'grey', 's'),
    '3-Stage Signal': (sig_metrics, [f"Signal3S_t{t}_monthly_TC0.5%" for t in THRESHOLDS], 'steelblue', '^'),
    'MV Signal':      (mv_metrics,  [f"MV_t{t}_monthly_TC0.5%" for t in THRESHOLDS], 'crimson', 'D'),
}

for group_name, (metrics_dict, keys, col, marker) in strategy_groups.items():
    first = True
    for key in keys:
        if key not in metrics_dict:
            continue
        m = metrics_dict[key]
        cagr = parse_pct(m['CAGR'])
        vol = parse_pct(m['Volatility'])
        ax.scatter(vol, cagr, color=col, marker=marker, s=120, zorder=5,
                   label=group_name if first else None, edgecolors='black', lw=0.8)
        ax.annotate(key.split('_TC')[0],
                    (vol, cagr),
                    textcoords='offset points',
                    xytext=(6, 4),
                    fontsize=7,
                    color=col)
        first = False

ax.axhline(0, color='black', lw=0.6, ls='--')
ax.axvline(0, color='black', lw=0.6, ls='--')
ax.set_xlabel("Annualised Volatility (%)")
ax.set_ylabel("CAGR (%)")
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("risk_return_scatter.png", dpi=150, bbox_inches='tight')
plt.show()

# ── 18. SAVE ALL ──────────────────────────────────────────────────────────────
for label, df in {**bh_results, **ew_results, **signal_results, **mv_results}.items():
    df.to_csv(f"{label}_portfolio.csv")

all_metrics = pd.DataFrame({**bh_metrics, **ew_metrics, **sig_metrics, **mv_metrics}).T
all_metrics.to_csv("all_portfolio_metrics.csv")

composite_score.to_csv("composite_signal.csv")

for thresh in THRESHOLDS:
    sig_df_t = build_signal_df(composite_score, equity_assets, macro_df.index, threshold=thresh)
    sig_df_t.to_csv(f"signal_df_threshold_{thresh}.csv")

for label, wdf in mv_weight_history.items():
    wdf.to_csv(f"{label}_weights.csv")

print("✓ All files saved.")
print(f"\nFiles written:")
print(f"  Portfolio values : {len(bh_results)+len(ew_results)+len(signal_results)+len(mv_results)} CSVs")
print(f"  Weight histories : {len(mv_weight_history)} CSVs")
print(f"  Metrics summary  : all_portfolio_metrics.csv")
print(f"  Signal CSVs      : {len(THRESHOLDS)} threshold files")
print(f"  Plots saved      : 17 PNG files")

In [ ]:
# @title
# ============================================================
# SIGNAL PORTFOLIO — THREE-STAGE THRESHOLD VERSION
# Signals generated from Modified Determinant Model
# + Buy-and-Hold Benchmark
# + Equal-Weight Benchmark
# + Three-Stage Signal Portfolio
# + Mean-Variance Signal Portfolio
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import requests
from io import StringIO
from scipy.optimize import minimize

# ── 1. LOAD ALL DATA FROM GITHUB ─────────────────────────────────────────────
GITHUB_BASE = "https://raw.githubusercontent.com/kboroz/MSFE_Capstone_Project/main/01_Data/Streamlined/"

INDEX_FILE_MAP = {
    'ASX50':       'streamlined_asx_50.csv',
    'EUROSTOXX50': 'streamlined_euro_stoxx_50.csv',
    'FTSE100':     'streamlined_ftse_100.csv',
    'Ibovespa':    'streamlined_ibovespa.csv',
    'JSE40':       'streamlined_jse_top_40.csv',
    'NIKKEI225':   'streamlined_nikkei_225.csv',
    'SP500':       'streamlined_s&p_500.csv',
}

MACRO_FILE = 'streamlined_macro.csv'

def load_csv(url):
    r = requests.get(url)
    r.raise_for_status()
    df = pd.read_csv(StringIO(r.text), index_col=0, parse_dates=True)
    df.columns = df.columns.str.replace('[^a-zA-Z0-9]', '_', regex=True)
    return df.sort_index()

macro_df = load_csv(GITHUB_BASE + MACRO_FILE)
macro_df = macro_df[~macro_df.index.duplicated(keep='first')].sort_index()
print(f"macro_df shape : {macro_df.shape}")
print(f"Columns        : {macro_df.columns.tolist()}")

index_data = {}
for name, fname in INDEX_FILE_MAP.items():
    try:
        df = load_csv(GITHUB_BASE + fname)
        index_data[name] = df
        print(f"✓ {name:15s} | shape={df.shape} | cols={df.columns.tolist()[:4]}")
    except Exception as e:
        print(f"✗ {name}: {e}")

# ── 2. EXTRACT PRICE SERIES ───────────────────────────────────────────────────
PRICE_COLS = ['Close', 'close', 'Price', 'price', 'Adj_Close', 'Last']

def get_price_series(df, name):
    col = next((c for c in PRICE_COLS if c in df.columns), df.columns[0])
    s = pd.to_numeric(df[col], errors='coerce').dropna()
    print(f"  {name}: using column '{col}'")
    return s

index_prices = {}
for name, df in index_data.items():
    index_prices[name] = get_price_series(df, name)

# ── 3. MODIFIED DETERMINANT SIGNAL MODEL ─────────────────────────────────────
DET_WINDOWS = [3, 7, 28]
WINDOW_WEIGHTS = {3: 0.33, 7: 0.34, 28: 0.33}

def compute_modified_determinant(price_series_dict, windows=DET_WINDOWS):
    prices = pd.DataFrame(price_series_dict).sort_index()
    rets = prices.pct_change().dropna()
    det_results = {}

    for window in windows:
        det_series = []
        for i in range(window, len(rets)):
            window_rets = rets.iloc[i - window:i]
            valid_cols = window_rets.columns[window_rets.std() > 1e-10]
            w = window_rets[valid_cols]

            if len(valid_cols) < 2:
                det_series.append(np.nan)
                continue

            try:
                R = w.corr().values
                I = np.eye(len(R))
                M = R - I
                det = np.linalg.det(M)
            except Exception:
                det = np.nan

            det_series.append(det)

        idx = rets.index[window:]
        det_results[window] = pd.Series(det_series, index=idx, name=f'det_{window}d')
        print(f"  Window {window:2d}d | computed {len(det_series)} observations")

    return det_results

print("\n── Computing Modified Determinant ──")
det_results = compute_modified_determinant(index_prices)

# ── 4. COMPOSITE SCORE ────────────────────────────────────────────────────────
def compute_composite_score(det_results, window_weights=WINDOW_WEIGHTS):
    all_series = []
    for window, series in det_results.items():
        mu = series.rolling(252, min_periods=60).mean()
        sigma = series.rolling(252, min_periods=60).std()
        z = (series - mu) / sigma.replace(0, np.nan)
        z.name = f'z_{window}d'
        all_series.append(z)

    z_df = pd.concat(all_series, axis=1).dropna()
    composite = sum(
        z_df[f'z_{w}d'] * window_weights[w]
        for w in window_weights if f'z_{w}d' in z_df.columns
    )
    composite = composite / sum(window_weights.values())
    return composite, z_df

composite_score, z_df = compute_composite_score(det_results)

print(f"\nComposite score shape  : {composite_score.shape}")
print(f"Date range             : {composite_score.index[0].date()} → {composite_score.index[-1].date()}")
print(f"Score range            : {composite_score.min():.3f} → {composite_score.max():.3f}")

# ── 5. THREE-STAGE SIGNAL CLASSIFICATION ─────────────────────────────────────
THRESHOLDS = [0.5, 1.0, 1.5]

def classify_signal(score, threshold):
    if abs(score) > threshold:
        return +1
    else:
        return -1

def build_signal_df(composite_score, equity_assets, macro_dates, threshold=0.5):
    binary = composite_score.apply(lambda x: classify_signal(x, threshold))
    signal_df = pd.DataFrame(index=macro_dates)
    for asset in equity_assets:
        signal_df[asset] = binary.reindex(macro_dates).ffill().bfill()
    return signal_df

# ── 6. ASSET UNIVERSE ─────────────────────────────────────────────────────────
EQUITY_ASSETS = list(index_prices.keys())
SAFE_BONDS = ['US_10Y', 'UK_10Y', 'AU_10Y', 'ZA_10Y']
FX = ['AUDUSD', 'BRLUSD', 'EURUSD', 'GBPUSD', 'JPYUSD', 'ZARUSD']
COMMODITIES = ['Oil', 'Gold', 'Bitcoin']
EXCLUDE_ASSETS = ['Bitcoin']

ALL_ASSETS = EQUITY_ASSETS + FX + COMMODITIES + SAFE_BONDS
ALL_ASSETS = [a for a in ALL_ASSETS if a in macro_df.columns and a not in EXCLUDE_ASSETS]

equity_assets = [a for a in EQUITY_ASSETS if a in ALL_ASSETS]
non_equity_assets = [a for a in ALL_ASSETS if a not in equity_assets]

N = len(ALL_ASSETS)
base_weight = 1.0 / N

print(f"\nTotal assets              : {N}")
print(f"Equity (signal-driven)    : {equity_assets}")
print(f"Non-equity (always long)  : {non_equity_assets}")
print(f"base_weight = 1/{N} = {base_weight:.4f}")

# ── 7. COUNTRY SAFE-HAVEN MAP ─────────────────────────────────────────────────
EQUITY_SAFE_HAVEN_MAP = {
    'ASX50':       ['AU_10Y', 'AUDUSD'],
    'EUROSTOXX50': ['EURUSD'],
    'FTSE100':     ['UK_10Y', 'GBPUSD'],
    'Ibovespa':    ['BRLUSD'],
    'JSE40':       ['ZA_10Y', 'ZARUSD'],
    'NIKKEI225':   ['JPYUSD'],
    'SP500':       ['US_10Y'],
}

UNIVERSE_SET = set(ALL_ASSETS)
for eq, havens in EQUITY_SAFE_HAVEN_MAP.items():
    EQUITY_SAFE_HAVEN_MAP[eq] = [h for h in havens if h in UNIVERSE_SET]

print("\nSafe-haven routing (filtered to universe):")
for eq, havens in EQUITY_SAFE_HAVEN_MAP.items():
    print(f"  {eq:15s} → {havens}")

# ── 8. BUILD RETURNS ──────────────────────────────────────────────────────────
returns_raw = macro_df[ALL_ASSETS].pct_change(fill_method=None)

for col in SAFE_BONDS:
    if col in returns_raw.columns:
        returns_raw[col] = -returns_raw[col]

if 'Oil' in macro_df.columns:
    bad_oil = macro_df['Oil'][macro_df['Oil'] <= 0].index
    for d in bad_oil:
        loc = returns_raw.index.get_loc(d)
        returns_raw.iloc[max(0, loc - 1): loc + 2, returns_raw.columns.get_loc('Oil')] = 0.0

returns = returns_raw.clip(lower=-0.50, upper=1.00).fillna(0).iloc[1:]

assert not returns.isnull().any().any(), "NaNs in returns"
assert not np.isinf(returns.values).any(), "Inf in returns"
print(f"\nReturns shape : {returns.shape}")
print("✓ Returns clean")

# ── 9. HELPER: BUILD REBALANCING DATE SET ────────────────────────────────────
def build_reb_set(returns, rebal_freq):
    actual_dates = returns.index
    if rebal_freq == 'daily':
        return set(actual_dates)
    elif rebal_freq == 'weekly':
        raw = returns.resample('W-MON').first().index
    elif rebal_freq == 'monthly':
        raw = returns.resample('MS').first().index
    elif rebal_freq == 'quarterly':
        raw = returns.resample('QS').first().index
    else:
        raise ValueError(f"Unknown rebal_freq: {rebal_freq}")

    reb_set = set()
    for d in raw:
        future = actual_dates[actual_dates >= d]
        if len(future) > 0:
            reb_set.add(future[0])
    return reb_set

# ── 10. BUY-AND-HOLD BENCHMARK ───────────────────────────────────────────────
def run_buy_and_hold(returns, all_assets, initial_capital=1e9):
    N = len(all_assets)
    base_weight = 1.0 / N
    pv = float(initial_capital)
    holdings = {a: pv * base_weight for a in all_assets}
    records = []
    first = True

    for date in returns.index:
        if first:
            first = False
            records.append({'date': date, 'portfolio_value': pv})
            continue
        new_holdings = {}
        for a in all_assets:
            new_holdings[a] = holdings[a] * (1 + returns.loc[date, a])
        holdings = new_holdings
        pv = sum(holdings.values())
        records.append({'date': date, 'portfolio_value': pv})

    df = pd.DataFrame(records).set_index('date')
    print(f"  Buy-and-Hold | Start: ${initial_capital/1e9:.2f}B → End: ${df['portfolio_value'].iloc[-1]/1e9:.3f}B")
    return df

# ── 11. THREE-STAGE WEIGHT BUILDER ───────────────────────────────────────────
def build_weights_3stage(sig_row, equity_assets, non_equity_assets,
                         base_weight, date, safe_haven_map):
    raw = {a: base_weight for a in non_equity_assets}
    safe_haven_extra = {}

    for a in equity_assets:
        if sig_row is not None and a in sig_row.index and not pd.isna(sig_row[a]):
            sig = sig_row[a]
        else:
            sig = -1

        if sig > 0:
            raw[a] = -base_weight
            havens = safe_haven_map.get(a, [])
            havens = [h for h in havens if h in raw]

            if havens:
                per_haven = base_weight / len(havens)
                for h in havens:
                    safe_haven_extra[h] = safe_haven_extra.get(h, 0) + per_haven
            else:
                per_ne = base_weight / len(non_equity_assets)
                for ne in non_equity_assets:
                    safe_haven_extra[ne] = safe_haven_extra.get(ne, 0) + per_ne
        else:
            raw[a] = base_weight

    for asset, extra in safe_haven_extra.items():
        raw[asset] = raw.get(asset, 0) + extra

    total = sum(raw.values())
    if total <= 0:
        for a in raw:
            raw[a] = 1.0 / len(raw)
    else:
        for a in raw:
            raw[a] /= total

    return raw

# ── 12. EQUAL-WEIGHT REBALANCED BENCHMARK ────────────────────────────────────
def run_equal_weight(returns, all_assets, tc_rate=0.005,
                     rebal_freq='monthly', initial_capital=1e9):
    N = len(all_assets)
    base_weight = 1.0 / N
    ew = {a: base_weight for a in all_assets}

    reb_set = build_reb_set(returns, rebal_freq)
    actual_dates = returns.index

    pv = float(initial_capital)
    current_weights = None
    records = []

    for date in actual_dates:
        if date in reb_set:
            if current_weights is not None:
                turnover = sum(abs(ew.get(a, 0) - current_weights.get(a, 0))
                               for a in set(ew) | set(current_weights)) / 2.0
                pv *= (1 - turnover * tc_rate)
            else:
                pv *= (1 - tc_rate)
            current_weights = ew.copy()

        if current_weights is not None:
            dr = sum(current_weights.get(a, 0) * returns.loc[date, a] for a in all_assets)
            pv *= (1 + dr)

        records.append({'date': date, 'portfolio_value': pv})

    return pd.DataFrame(records).set_index('date')

# ── 13. THREE-STAGE SIGNAL PORTFOLIO RUNNER ──────────────────────────────────
def run_signal_portfolio_3stage(returns, signal_df, equity_assets,
                                non_equity_assets, all_assets,
                                freq='monthly', tc_rate=0.005,
                                safe_haven_map=None,
                                initial_capital=1e9):
    if safe_haven_map is None:
        safe_haven_map = {}

    N = len(all_assets)
    base_weight = 1.0 / N

    reb_set = build_reb_set(returns, freq)
    actual_dates = returns.index

    sig = signal_df.copy()
    sig.index = pd.to_datetime(sig.index)

    pv = float(initial_capital)
    current_weights = None
    records = []

    for date in actual_dates:
        if date in reb_set:
            past = sig[sig.index <= date]
            sig_row = past.iloc[-1] if len(past) > 0 else None

            raw = build_weights_3stage(
                sig_row, equity_assets, non_equity_assets,
                base_weight, date, safe_haven_map
            )

            if current_weights is not None:
                all_keys = set(raw) | set(current_weights)
                turnover = sum(abs(raw.get(a, 0) - current_weights.get(a, 0)) for a in all_keys) / 2.0
                pv *= (1 - turnover * tc_rate)
            else:
                pv *= (1 - tc_rate)

            current_weights = raw.copy()

        if current_weights is not None:
            dr = sum(current_weights.get(a, 0) * returns.loc[date, a] for a in all_assets)
            pv *= (1 + dr)

        records.append({'date': date, 'portfolio_value': pv})

    return pd.DataFrame(records).set_index('date')

# ── 14. MEAN-VARIANCE SIGNAL PORTFOLIO ───────────────────────────────────────
MV_LOOKBACK = 252
MV_MIN_OBS = 60
MAX_WEIGHT = 0.30
RISK_FREE = 0.0

def mean_variance_weights(mu_vec, cov_mat, allowed_mask,
                          max_weight=MAX_WEIGHT,
                          risk_free=RISK_FREE):
    n_total = len(mu_vec)
    allowed = np.where(allowed_mask)[0]
    n_allowed = len(allowed)

    fallback = np.zeros(n_total)
    if n_allowed == 0:
        fallback[:] = 1.0 / n_total
        return fallback

    fallback[allowed] = 1.0 / n_allowed

    mu_sub = mu_vec[allowed]
    cov_sub = cov_mat[np.ix_(allowed, allowed)]
    cov_sub = cov_sub + np.eye(n_allowed) * 1e-8

    def neg_sharpe(w):
        port_ret = np.dot(w, mu_sub)
        port_var = np.dot(w, cov_sub @ w)
        port_std = np.sqrt(max(port_var, 1e-12))
        return -(port_ret - risk_free) / port_std

    constraints = [{'type': 'eq', 'fun': lambda w: np.sum(w) - 1.0}]
    bounds = [(0.0, max_weight)] * n_allowed
    w0 = np.ones(n_allowed) / n_allowed

    try:
        res = minimize(
            neg_sharpe, w0,
            method='SLSQP',
            bounds=bounds,
            constraints=constraints,
            options={'ftol': 1e-9, 'maxiter': 500}
        )
        if res.success and not np.any(np.isnan(res.x)):
            w_opt = res.x
        else:
            w_opt = w0
    except Exception:
        w_opt = w0

    w_full = np.zeros(n_total)
    w_full[allowed] = w_opt
    return w_full

def run_mv_signal_portfolio(returns, signal_df, all_assets,
                            equity_assets,
                            freq='monthly', tc_rate=0.005,
                            lookback=MV_LOOKBACK,
                            min_obs=MV_MIN_OBS,
                            max_weight=MAX_WEIGHT,
                            initial_capital=1e9):
    """
    Mean-Variance portfolio with signal constraints.

    Signal logic per asset at each rebalance:
      +1 (RISK_ON / defensive) -> weight forced to 0
      -1 (RISK_OFF / long)     -> eligible for optimiser

    Non-equity assets are always eligible (never blocked).
    """
    reb_set = build_reb_set(returns, freq)
    actual_dates = returns.index
    assets = all_assets
    n = len(assets)
    asset_idx = {a: i for i, a in enumerate(assets)}

    sig = signal_df.copy()
    sig.index = pd.to_datetime(sig.index)

    pv = float(initial_capital)
    current_weights = np.zeros(n)
    records = []
    weight_history = []
    first_reb = True

    for t, date in enumerate(actual_dates):
        if date in reb_set:
            hist = returns.iloc[max(0, t - lookback):t]

            if len(hist) < min_obs:
                new_weights = np.ones(n) / n
            else:
                mu_daily = hist.mean().values
                cov_daily = hist.cov().values

                mu_ann = mu_daily * 252
                cov_ann = cov_daily * 252

                past = sig[sig.index <= date]
                sig_row = past.iloc[-1] if len(past) > 0 else None

                allowed_mask = np.ones(n, dtype=bool)

                if sig_row is not None:
                    for eq in equity_assets:
                        if eq in asset_idx and eq in sig_row.index:
                            s = sig_row[eq]
                            if not pd.isna(s) and s > 0:
                                allowed_mask[asset_idx[eq]] = False

                new_weights = mean_variance_weights(
                    mu_ann, cov_ann, allowed_mask, max_weight=max_weight
                )

            if first_reb:
                pv *= (1 - tc_rate)
                first_reb = False
            else:
                turnover = np.sum(np.abs(new_weights - current_weights)) / 2.0
                pv *= (1 - turnover * tc_rate)

            current_weights = new_weights.copy()
            weight_history.append({'date': date, **dict(zip(assets, current_weights))})

        if not first_reb:
            dr = np.dot(current_weights, returns.loc[date].values)
            pv *= (1 + dr)

        records.append({'date': date, 'portfolio_value': pv})

    pv_df = pd.DataFrame(records).set_index('date')
    weights_df = pd.DataFrame(weight_history).set_index('date')

    print(f"  MV-Signal | freq={freq} | tc={tc_rate*100:.1f}% | Start: $1.00B → End: ${pv_df['portfolio_value'].iloc[-1]/1e9:.3f}B")
    return pv_df, weights_df

# ── 15. RUN ALL PORTFOLIOS ────────────────────────────────────────────────────
TC_LEVELS = {'0%': 0.00, '0.5%': 0.005, '1%': 0.01, '2%': 0.02}
FREQS = ['daily', 'weekly', 'monthly', 'quarterly']
MV_FREQS = ['monthly', 'quarterly']

print("\n── Running Buy-and-Hold ──")
bh_result = run_buy_and_hold(returns, ALL_ASSETS, initial_capital=1e9)
bh_results = {'BuyAndHold_EW': bh_result}

print("\n── Running Equal-Weight Rebalanced ──")
ew_results = {}
for freq in FREQS:
    for tc_label, tc_rate in TC_LEVELS.items():
        label = f"EW_{freq}_TC{tc_label}"
        ew_results[label] = run_equal_weight(
            returns, ALL_ASSETS, tc_rate=tc_rate,
            rebal_freq=freq, initial_capital=1e9
        )
        print(f"  ✓ {label}")

print("\n── Running Signal Portfolios ──")
signal_results = {}
signal_distributions = {}

for thresh in THRESHOLDS:
    sig_df_t = build_signal_df(composite_score, equity_assets, macro_df.index, threshold=thresh)
    n_risk_on = (sig_df_t[equity_assets[0]] == +1).sum()
    n_risk_off = (sig_df_t[equity_assets[0]] == -1).sum()
    total = len(sig_df_t)

    print(f"\n  Threshold={thresh} | RISK_ON(+1): {n_risk_on} days ({n_risk_on/total*100:.1f}%) | "
          f"RISK_OFF(-1): {n_risk_off} days ({n_risk_off/total*100:.1f}%)")

    signal_distributions[thresh] = {'RISK_ON': n_risk_on, 'RISK_OFF': n_risk_off}

    for freq in FREQS:
        for tc_label, tc_rate in TC_LEVELS.items():
            label = f"Signal3S_t{thresh}_{freq}_TC{tc_label}"
            signal_results[label] = run_signal_portfolio_3stage(
                returns, sig_df_t, equity_assets, non_equity_assets,
                ALL_ASSETS, freq=freq, tc_rate=tc_rate,
                safe_haven_map=EQUITY_SAFE_HAVEN_MAP,
                initial_capital=1e9
            )
            print(f"    ✓ {label}")

print("\n── Running Mean-Variance Signal Portfolios ──")
mv_results = {}
mv_weight_history = {}

for thresh in THRESHOLDS:
    sig_df_t = build_signal_df(composite_score, equity_assets, macro_df.index, threshold=thresh)
    for freq in MV_FREQS:
        for tc_label, tc_rate in TC_LEVELS.items():
            label = f"MV_t{thresh}_{freq}_TC{tc_label}"
            pv_df, w_df = run_mv_signal_portfolio(
                returns, sig_df_t, ALL_ASSETS,
                equity_assets=equity_assets,
                freq=freq, tc_rate=tc_rate,
                lookback=MV_LOOKBACK,
                min_obs=MV_MIN_OBS,
                max_weight=MAX_WEIGHT,
                initial_capital=1e9
            )
            mv_results[label] = pv_df
            mv_weight_history[label] = w_df
            print(f"  ✓ {label}")

print("\n✓ All portfolios computed")

# ── 16. PERFORMANCE METRICS ───────────────────────────────────────────────────
def compute_metrics(pv_df, label=''):
    pv = pv_df['portfolio_value']
    daily_ret = pv.pct_change().dropna()
    n_years = len(daily_ret) / 252
    total_ret = (pv.iloc[-1] / pv.iloc[0]) - 1
    cagr = (1 + total_ret) ** (1 / n_years) - 1 if n_years > 0 else np.nan
    vol = daily_ret.std() * np.sqrt(252)
    sharpe = cagr / vol if vol > 0 else np.nan
    roll_max = pv.cummax()
    max_dd = ((pv - roll_max) / roll_max).min()
    calmar = cagr / abs(max_dd) if max_dd != 0 else np.nan
    var_95 = daily_ret.quantile(0.05)

    return {
        'CAGR': f"{cagr*100:.2f}%",
        'Volatility': f"{vol*100:.2f}%",
        'Sharpe': f"{sharpe:.2f}",
        'Max Drawdown': f"{max_dd*100:.2f}%",
        'Calmar': f"{calmar:.2f}",
        'VaR 95%': f"{var_95*100:.2f}%",
        'Total Return': f"{total_ret*100:.2f}%",
        'Final ( $ B)': f"{pv.iloc[-1]/1e9:.3f}",
    }

bh_metrics = {k: compute_metrics(v, k) for k, v in bh_results.items()}
ew_metrics = {k: compute_metrics(v, k) for k, v in ew_results.items()}
sig_metrics = {k: compute_metrics(v, k) for k, v in signal_results.items()}
mv_metrics = {k: compute_metrics(v, k) for k, v in mv_results.items()}

print("\n" + "="*90)
print("BUY-AND-HOLD BENCHMARK")
print("="*90)
print(pd.DataFrame(bh_metrics).T.to_string())

print("\n" + "="*90)
print("EQUAL-WEIGHT REBALANCED BENCHMARK")
print("="*90)
print(pd.DataFrame(ew_metrics).T.to_string())

for thresh in THRESHOLDS:
    sub = {k: v for k, v in sig_metrics.items() if f"_t{thresh}_" in k}
    print(f"\n{'='*90}")
    print(f"THREE-STAGE SIGNAL — Threshold = {thresh}")
    print("="*90)
    print(pd.DataFrame(sub).T.to_string())

for thresh in THRESHOLDS:
    sub = {k: v for k, v in mv_metrics.items() if f"_t{thresh}_" in k}
    print(f"\n{'='*90}")
    print(f"MEAN-VARIANCE SIGNAL — Threshold = {thresh}")
    print("="*90)
    print(pd.DataFrame(sub).T.to_string())

# ── 17. PLOTS ─────────────────────────────────────────────────────────────────
colors_tc = {'0%': 'black', '0.5%': 'steelblue', '1%': 'darkorange', '2%': 'crimson'}
freq_colors = {'daily': 'crimson', 'weekly': 'darkorange', 'monthly': 'steelblue', 'quarterly': 'purple'}
thresh_colors = {0.5: 'steelblue', 1.0: 'darkorange', 1.5: 'crimson'}

bh_pv = bh_results['BuyAndHold_EW']['portfolio_value']
ew_ref_pv = ew_results['EW_monthly_TC0.5%']['portfolio_value']
ew_q_pv = ew_results['EW_quarterly_TC0.5%']['portfolio_value']

# Figure 1
fig, axes = plt.subplots(2, 1, figsize=(16, 10))
fig.suptitle("Modified Determinant — Composite Score & Threshold Signals",
             fontsize=14, fontweight='bold')

ax = axes[0]
ax.plot(composite_score.index, composite_score.values,
        color='navy', lw=1.0, alpha=0.8, label='Composite Z-score')
for thresh, col in zip(THRESHOLDS, ['green', 'orange', 'red']):
    ax.axhline(thresh, color=col, ls='--', lw=1.0, label=f'+{thresh}')
    ax.axhline(-thresh, color=col, ls='--', lw=1.0, label=f'-{thresh}')
ax.axhline(0, color='black', lw=0.6)
ax.set_title("Composite Determinant Score with Thresholds", fontsize=11)
ax.set_ylabel("Z-score")
ax.legend(fontsize=8, ncol=4)
ax.grid(True, alpha=0.3)

ax = axes[1]
for window, series in det_results.items():
    ax.plot(series.index, np.sign(series.values), alpha=0.6,
            label=f"{window}d window", lw=1.0)
ax.axhline(0, color='black', lw=0.8)
ax.set_title("Sign of Det(R-I) per Window", fontsize=11)
ax.set_ylabel("+1 / -1")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("composite_signal_thresholds.png", dpi=150, bbox_inches='tight')
plt.show()

# Figure 2
fig, axes = plt.subplots(1, 3, figsize=(22, 6), sharey=False)
fig.suptitle("Three-Stage Signal vs Benchmarks — Portfolio Value (TC=0.5%, Monthly)",
             fontsize=13, fontweight='bold')

for ax, thresh in zip(axes, THRESHOLDS):
    ax.plot(bh_pv.index, bh_pv / 1e9, color='forestgreen', lw=1.8, ls='-.', label='Buy & Hold (EW)')
    ax.plot(ew_ref_pv.index, ew_ref_pv / 1e9, color='grey', lw=1.5, ls='--', label='EW Rebalanced')
    label = f"Signal3S_t{thresh}_monthly_TC0.5%"
    pv = signal_results[label]['portfolio_value']
    ax.plot(pv.index, pv / 1e9, color='steelblue', lw=1.8, label=f'Signal (t={thresh})')
    ax.axhline(1.0, color='black', ls=':', lw=0.8)
    ax.set_title(f"Threshold = {thresh}", fontsize=11)
    ax.set_ylabel("Value ($B)")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("signal_3stage_portfolio_value.png", dpi=150, bbox_inches='tight')
plt.show()

# Figure 3
fig, axes = plt.subplots(1, 3, figsize=(22, 5), sharey=True)
fig.suptitle("Three-Stage Signal — Drawdowns (TC=0.5%, Monthly)",
             fontsize=13, fontweight='bold')

for ax, thresh in zip(axes, THRESHOLDS):
    label = f"Signal3S_t{thresh}_monthly_TC0.5%"
    pv = signal_results[label]['portfolio_value']
    dd = (pv - pv.cummax()) / pv.cummax() * 100
    ew_dd = (ew_ref_pv - ew_ref_pv.cummax()) / ew_ref_pv.cummax() * 100
    bh_dd = (bh_pv - bh_pv.cummax()) / bh_pv.cummax() * 100

    ax.fill_between(dd.index, dd.values, 0, alpha=0.25, color='steelblue')
    ax.fill_between(ew_dd.index, ew_dd.values, 0, alpha=0.15, color='grey')
    ax.fill_between(bh_dd.index, bh_dd.values, 0, alpha=0.10, color='forestgreen')
    ax.plot(dd.index, dd.values, color='steelblue', lw=1.2, label=f'Signal (t={thresh})')
    ax.plot(ew_dd.index, ew_dd.values, color='grey', lw=1.0, ls='--', label='EW Rebalanced')
    ax.plot(bh_dd.index, bh_dd.values, color='forestgreen', lw=1.0, ls='-.', label='Buy & Hold')
    ax.set_title(f"Threshold = {thresh}", fontsize=11)
    ax.set_ylabel("Drawdown (%)")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("signal_3stage_drawdowns.png", dpi=150, bbox_inches='tight')
plt.show()

# Figure 4
fig, axes = plt.subplots(1, 3, figsize=(22, 6))
fig.suptitle("Three-Stage Signal — TC Sensitivity vs Benchmarks (Monthly)",
             fontsize=13, fontweight='bold')

for ax, thresh in zip(axes, THRESHOLDS):
    for tc_label, col in colors_tc.items():
        label = f"Signal3S_t{thresh}_monthly_TC{tc_label}"
        pv = signal_results[label]['portfolio_value']
        ax.plot(pv.index, pv / 1e9, color=col, lw=1.3, label=f'Signal TC={tc_label}')
    ax.plot(ew_ref_pv.index, ew_ref_pv / 1e9, color='grey', lw=1.8, ls='--', label='EW Rebalanced')
    ax.plot(bh_pv.index, bh_pv / 1e9, color='forestgreen', lw=1.8, ls='-.', label='Buy & Hold')
    ax.set_title(f"Threshold = {thresh}", fontsize=11)
    ax.set_ylabel("Value ($B)")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("signal_3stage_tc_sensitivity.png", dpi=150, bbox_inches='tight')
plt.show()

# Figure 5
fig, ax = plt.subplots(figsize=(16, 7))
fig.suptitle("Strategy Overview — Buy & Hold vs EW vs Signal (TC=0.5%, Monthly)",
             fontsize=13, fontweight='bold')

ax.plot(bh_pv.index, bh_pv / 1e9, color='forestgreen', lw=2.0, ls='-.', label='Buy & Hold (EW)')
ax.plot(ew_ref_pv.index, ew_ref_pv / 1e9, color='grey', lw=2.0, ls='--', label='EW Rebalanced (monthly)')

for thresh, col in thresh_colors.items():
    label = f"Signal3S_t{thresh}_monthly_TC0.5%"
    pv = signal_results[label]['portfolio_value']
    ax.plot(pv.index, pv / 1e9, color=col, lw=1.8, label=f'Signal t={thresh}')

ax.axhline(1.0, color='black', ls=':', lw=0.8)
ax.set_ylabel("Portfolio Value ($B)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("strategy_monthly_overview.png", dpi=150, bbox_inches='tight')
plt.show()

# Figure Q1
fig, axes = plt.subplots(1, 3, figsize=(22, 6))
fig.suptitle("Three-Stage Signal — Quarterly Portfolio Value (TC=0.5%)",
             fontsize=13, fontweight='bold')

for ax, thresh in zip(axes, THRESHOLDS):
    label = f"Signal3S_t{thresh}_quarterly_TC0.5%"
    pv = signal_results[label]['portfolio_value']
    ax.plot(pv.index, pv / 1e9, color='steelblue', lw=1.8, label=f'Signal t={thresh}')
    ax.plot(ew_q_pv.index, ew_q_pv / 1e9, color='grey', lw=1.5, ls='--', label='EW Rebalanced')
    ax.plot(bh_pv.index, bh_pv / 1e9, color='forestgreen', lw=1.5, ls='-.', label='Buy & Hold')
    ax.axhline(1.0, color='black', ls=':', lw=0.8)
    ax.set_title(f"Threshold = {thresh}", fontsize=11)
    ax.set_ylabel("Value ($B)")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("signal_quarterly_value.png", dpi=150, bbox_inches='tight')
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(22, 5), sharey=True)
fig.suptitle("Three-Stage Signal — Quarterly Drawdowns (TC=0.5%)",
             fontsize=13, fontweight='bold')

for ax, thresh in zip(axes, THRESHOLDS):
    label = f"Signal3S_t{thresh}_quarterly_TC0.5%"
    pv = signal_results[label]['portfolio_value']
    dd = (pv - pv.cummax()) / pv.cummax() * 100
    ew_dd = (ew_q_pv - ew_q_pv.cummax()) / ew_q_pv.cummax() * 100
    bh_dd = (bh_pv - bh_pv.cummax()) / bh_pv.cummax() * 100

    ax.fill_between(dd.index, dd.values, 0, alpha=0.25, color='steelblue')
    ax.fill_between(ew_dd.index, ew_dd.values, 0, alpha=0.15, color='grey')
    ax.fill_between(bh_dd.index, bh_dd.values, 0, alpha=0.10, color='forestgreen')
    ax.plot(dd.index, dd.values, color='steelblue', lw=1.2, label=f'Signal (t={thresh})')
    ax.plot(ew_dd.index, ew_dd.values, color='grey', lw=1.0, ls='--', label='EW Rebalanced')
    ax.plot(bh_dd.index, bh_dd.values, color='forestgreen', lw=1.0, ls='-.', label='Buy & Hold')
    ax.set_title(f"Threshold = {thresh}", fontsize=11)
    ax.set_ylabel("Drawdown (%)")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("signal_quarterly_drawdowns.png", dpi=150, bbox_inches='tight')
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(22, 6))
fig.suptitle("Three-Stage Signal — Quarterly TC Sensitivity",
             fontsize=13, fontweight='bold')

for ax, thresh in zip(axes, THRESHOLDS):
    for tc_label, col in colors_tc.items():
        label = f"Signal3S_t{thresh}_quarterly_TC{tc_label}"
        pv = signal_results[label]['portfolio_value']
        ax.plot(pv.index, pv / 1e9, color=col, lw=1.3, label=f'TC={tc_label}')
    ax.plot(ew_q_pv.index, ew_q_pv / 1e9, '--', color='grey', lw=1.5, label='EW Benchmark')
    ax.plot(bh_pv.index, bh_pv / 1e9, '-.', color='forestgreen', lw=1.5, label='Buy & Hold')
    ax.set_title(f"Threshold = {thresh}", fontsize=11)
    ax.set_ylabel("Value ($B)")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("signal_quarterly_tc_sensitivity.png", dpi=150, bbox_inches='tight')
plt.show()

fig, ax = plt.subplots(figsize=(16, 7))
fig.suptitle("Quarterly Strategy Overview — Buy & Hold vs EW vs Signal (TC=0.5%)",
             fontsize=13, fontweight='bold')

ax.plot(bh_pv.index, bh_pv / 1e9, color='forestgreen', lw=2.0, ls='-.', label='Buy & Hold (EW)')
ax.plot(ew_q_pv.index, ew_q_pv / 1e9, color='grey', lw=2.0, ls='--', label='EW Rebalanced (quarterly)')

for thresh, col in thresh_colors.items():
    label = f"Signal3S_t{thresh}_quarterly_TC0.5%"
    pv = signal_results[label]['portfolio_value']
    ax.plot(pv.index, pv / 1e9, color=col, lw=1.8, label=f'Signal t={thresh}')

ax.axhline(1.0, color='black', ls=':', lw=0.8)
ax.set_ylabel("Portfolio Value ($B)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("strategy_quarterly_overview.png", dpi=150, bbox_inches='tight')
plt.show()

# Figure F1
fig, axes = plt.subplots(1, 3, figsize=(22, 6))
fig.suptitle("Signal Portfolio — Rebalancing Frequency Comparison (TC=0.5%)",
             fontsize=13, fontweight='bold')

for ax, thresh in zip(axes, THRESHOLDS):
    for freq, col in freq_colors.items():
        label = f"Signal3S_t{thresh}_{freq}_TC0.5%"
        if label in signal_results:
            pv = signal_results[label]['portfolio_value']
            ax.plot(pv.index, pv / 1e9, color=col, lw=1.4, label=freq.capitalize())
    ax.plot(bh_pv.index, bh_pv / 1e9, color='forestgreen', lw=1.8, ls='-.', label='Buy & Hold')
    ax.plot(ew_ref_pv.index, ew_ref_pv / 1e9, color='grey', lw=1.5, ls='--', label='EW Monthly')
    ax.axhline(1.0, color='black', ls=':', lw=0.8)
    ax.set_title(f"Threshold = {thresh}", fontsize=11)
    ax.set_ylabel("Value ($B)")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("signal_frequency_comparison_value.png", dpi=150, bbox_inches='tight')
plt.show()

# Figure F2
fig, axes = plt.subplots(1, 3, figsize=(22, 5), sharey=True)
fig.suptitle("Signal Portfolio — Drawdowns by Frequency (TC=0.5%)",
             fontsize=13, fontweight='bold')

for ax, thresh in zip(axes, THRESHOLDS):
    for freq, col in freq_colors.items():
        label = f"Signal3S_t{thresh}_{freq}_TC0.5%"
        if label in signal_results:
            pv = signal_results[label]['portfolio_value']
            dd = (pv - pv.cummax()) / pv.cummax() * 100
            ax.plot(dd.index, dd.values, color=col, lw=1.2, label=freq.capitalize())
    bh_dd = (bh_pv - bh_pv.cummax()) / bh_pv.cummax() * 100
    ax.plot(bh_dd.index, bh_dd.values, color='forestgreen', lw=1.5, ls='-.', label='Buy & Hold')
    ax.fill_between(bh_dd.index, bh_dd.values, 0, alpha=0.08, color='forestgreen')
    ax.set_title(f"Threshold = {thresh}", fontsize=11)
    ax.set_ylabel("Drawdown (%)")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("signal_frequency_comparison_drawdown.png", dpi=150, bbox_inches='tight')
plt.show()

# Figure F3
fig, ax = plt.subplots(figsize=(16, 7))
fig.suptitle("Equal-Weight Rebalanced — Frequency Comparison (TC=0.5%)",
             fontsize=13, fontweight='bold')

for freq, col in freq_colors.items():
    label = f"EW_{freq}_TC0.5%"
    if label in ew_results:
        pv = ew_results[label]['portfolio_value']
        ax.plot(pv.index, pv / 1e9, color=col, lw=1.6, label=f'EW {freq.capitalize()}')

ax.plot(bh_pv.index, bh_pv / 1e9, color='forestgreen', lw=2.0, ls='-.', label='Buy & Hold')
ax.axhline(1.0, color='black', ls=':', lw=0.8)
ax.set_ylabel("Portfolio Value ($B)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("ew_frequency_comparison.png", dpi=150, bbox_inches='tight')
plt.show()

# Figure F4
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("TC Impact on High-Frequency Rebalancing — Signal t=1.0",
             fontsize=13, fontweight='bold')

for ax, freq in zip(axes, ['daily', 'weekly']):
    for tc_label, col in colors_tc.items():
        label = f"Signal3S_t1.0_{freq}_TC{tc_label}"
        if label in signal_results:
            pv = signal_results[label]['portfolio_value']
            ax.plot(pv.index, pv / 1e9, color=col, lw=1.4, label=f'TC={tc_label}')
    ax.plot(bh_pv.index, bh_pv / 1e9, color='forestgreen', lw=1.8, ls='-.', label='Buy & Hold')
    ax.axhline(1.0, color='black', ls=':', lw=0.8)
    ax.set_title(f"{freq.capitalize()} Rebalancing", fontsize=11)
    ax.set_ylabel("Value ($B)")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("tc_impact_high_frequency.png", dpi=150, bbox_inches='tight')
plt.show()

# MV Figure 1
fig, axes = plt.subplots(1, 3, figsize=(22, 6))
fig.suptitle("Mean-Variance Signal vs Three-Stage Signal vs Benchmarks (TC=0.5%, Monthly)",
             fontsize=13, fontweight='bold')

for ax, thresh in zip(axes, THRESHOLDS):
    ax.plot(bh_pv.index, bh_pv / 1e9, color='forestgreen', lw=1.8, ls='-.', label='Buy & Hold')
    ax.plot(ew_ref_pv.index, ew_ref_pv / 1e9, color='grey', lw=1.5, ls='--', label='EW Rebalanced')

    sig_label = f"Signal3S_t{thresh}_monthly_TC0.5%"
    mv_label = f"MV_t{thresh}_monthly_TC0.5%"

    sig_pv = signal_results[sig_label]['portfolio_value']
    ax.plot(sig_pv.index, sig_pv / 1e9, color='steelblue', lw=1.6, label=f'3-Stage Signal t={thresh}')

    if mv_label in mv_results:
        mv_pv = mv_results[mv_label]['portfolio_value']
        ax.plot(mv_pv.index, mv_pv / 1e9, color='crimson', lw=1.8, ls='-', label=f'MV Signal t={thresh}')

    ax.axhline(1.0, color='black', ls=':', lw=0.8)
    ax.set_title(f"Threshold = {thresh}", fontsize=11)
    ax.set_ylabel("Value ($B)")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("mv_vs_signal_monthly.png", dpi=150, bbox_inches='tight')
plt.show()

# MV Figure 2
fig, axes = plt.subplots(1, 3, figsize=(22, 5), sharey=True)
fig.suptitle("Mean-Variance Signal — Drawdowns (TC=0.5%, Monthly)",
             fontsize=13, fontweight='bold')

for ax, thresh in zip(axes, THRESHOLDS):
    mv_label = f"MV_t{thresh}_monthly_TC0.5%"
    if mv_label not in mv_results:
        continue

    mv_pv = mv_results[mv_label]['portfolio_value']
    mv_dd = (mv_pv - mv_pv.cummax()) / mv_pv.cummax() * 100
    bh_dd = (bh_pv - bh_pv.cummax()) / bh_pv.cummax() * 100
    ew_dd = (ew_ref_pv - ew_ref_pv.cummax()) / ew_ref_pv.cummax() * 100

    sig_label = f"Signal3S_t{thresh}_monthly_TC0.5%"
    sig_pv = signal_results[sig_label]['portfolio_value']
    sig_dd = (sig_pv - sig_pv.cummax()) / sig_pv.cummax() * 100

    ax.fill_between(mv_dd.index, mv_dd.values, 0, alpha=0.25, color='crimson')
    ax.fill_between(sig_dd.index, sig_dd.values, 0, alpha=0.15, color='steelblue')
    ax.plot(mv_dd.index, mv_dd.values, color='crimson', lw=1.4, label=f'MV Signal (t={thresh})')
    ax.plot(sig_dd.index, sig_dd.values, color='steelblue', lw=1.2, ls='--', label=f'3-Stage Signal (t={thresh})')
    ax.plot(bh_dd.index, bh_dd.values, color='forestgreen', lw=1.0, ls='-.', label='Buy & Hold')
    ax.plot(ew_dd.index, ew_dd.values, color='grey', lw=1.0, ls=':', label='EW Rebalanced')
    ax.set_title(f"Threshold = {thresh}", fontsize=11)
    ax.set_ylabel("Drawdown (%)")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("mv_signal_drawdowns.png", dpi=150, bbox_inches='tight')
plt.show()

# MV Figure 3
fig, axes = plt.subplots(1, 3, figsize=(22, 6))
fig.suptitle("Mean-Variance Signal — TC Sensitivity (Monthly)",
             fontsize=13, fontweight='bold')

for ax, thresh in zip(axes, THRESHOLDS):
    for tc_label, col in colors_tc.items():
        label = f"MV_t{thresh}_monthly_TC{tc_label}"
        if label in mv_results:
            pv = mv_results[label]['portfolio_value']
            ax.plot(pv.index, pv / 1e9, color=col, lw=1.4, label=f'MV TC={tc_label}')
    ax.plot(ew_ref_pv.index, ew_ref_pv / 1e9, color='grey', lw=1.8, ls='--', label='EW Rebalanced')
    ax.plot(bh_pv.index, bh_pv / 1e9, color='forestgreen', lw=1.8, ls='-.', label='Buy & Hold')
    ax.axhline(1.0, color='black', ls=':', lw=0.8)
    ax.set_title(f"Threshold = {thresh}", fontsize=11)
    ax.set_ylabel("Value ($B)")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("mv_signal_tc_sensitivity.png", dpi=150, bbox_inches='tight')
plt.show()

# MV Figure 4
fig, axes = plt.subplots(1, 3, figsize=(22, 6))
fig.suptitle("Mean-Variance Signal — Rebalancing Frequency Comparison (TC=0.5%)",
             fontsize=13, fontweight='bold')

for ax, thresh in zip(axes, THRESHOLDS):
    for freq, col in freq_colors.items():
        label = f"MV_t{thresh}_{freq}_TC0.5%"
        if label in mv_results:
            pv = mv_results[label]['portfolio_value']
            ax.plot(pv.index, pv / 1e9, color=col, lw=1.4, label=freq.capitalize())
    ax.plot(bh_pv.index, bh_pv / 1e9, color='forestgreen', lw=1.8, ls='-.', label='Buy & Hold')
    ax.plot(ew_ref_pv.index, ew_ref_pv / 1e9, color='grey', lw=1.5, ls='--', label='EW Monthly')
    ax.axhline(1.0, color='black', ls=':', lw=0.8)
    ax.set_title(f"Threshold = {thresh}", fontsize=11)
    ax.set_ylabel("Value ($B)")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("mv_signal_frequency_comparison.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Figure MV5: Weight Evolution (Stacked Area) ─────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(22, 5), sharey=True)
fig.suptitle("Mean-Variance Signal — Stacked Weight Evolution (TC=0.5%, Monthly)",
             fontsize=13, fontweight='bold')

for ax, thresh in zip(axes, THRESHOLDS):
    label = f"MV_t{thresh}_monthly_TC0.5%"
    if label not in mv_weight_history:
        continue

    wdf = mv_weight_history[label].copy()
    wdf = wdf.sort_index()

    # Keep only positive weights for stacked area display
    wdf_plot = wdf.clip(lower=0)

    wdf_plot.plot.area(
        ax=ax,
        stacked=True,
        alpha=0.85,
        linewidth=0
    )

    ax.set_title(f"Threshold = {thresh}", fontsize=11)
    ax.set_ylabel("Weight")
    ax.set_ylim(0, 1)
    ax.grid(True, alpha=0.3)

    if ax is axes[0]:
        ax.legend(fontsize=7, ncol=2, loc='upper left')
    else:
        ax.legend_.remove() if ax.get_legend() else None

plt.tight_layout()
plt.savefig("mv_signal_weight_evolution_stacked.png", dpi=150, bbox_inches='tight')
plt.show()

# MV Figure 6
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle("Sharpe Ratio Comparison — All Strategies (Monthly, TC=0.5%)",
             fontsize=13, fontweight='bold')

for ax, thresh in zip(axes, THRESHOLDS):
    labels_bar = []
    sharpes = []

    labels_bar.append('Buy & Hold')
    sharpes.append(float(bh_metrics['BuyAndHold_EW']['Sharpe']))

    labels_bar.append('EW Monthly')
    sharpes.append(float(ew_metrics['EW_monthly_TC0.5%']['Sharpe']))

    sig_key = f"Signal3S_t{thresh}_monthly_TC0.5%"
    labels_bar.append(f'3-Stage t={thresh}')
    sharpes.append(float(sig_metrics[sig_key]['Sharpe']))

    mv_key = f"MV_t{thresh}_monthly_TC0.5%"
    if mv_key in mv_metrics:
        labels_bar.append(f'MV t={thresh}')
        sharpes.append(float(mv_metrics[mv_key]['Sharpe']))

    bar_colors = ['forestgreen', 'grey', 'steelblue', 'crimson'][:len(labels_bar)]
    bars = ax.bar(labels_bar, sharpes, color=bar_colors, alpha=0.85, edgecolor='black')

    for bar, val in zip(bars, sharpes):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.01,
                f"{val:.2f}",
                ha='center', va='bottom', fontsize=9, fontweight='bold')

    ax.axhline(0, color='black', lw=0.8)
    ax.set_title(f"Threshold = {thresh}", fontsize=11)
    ax.set_ylabel("Sharpe Ratio")
    ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig("sharpe_comparison_bar.png", dpi=150, bbox_inches='tight')
plt.show()

# MV Figure 7
fig, ax = plt.subplots(figsize=(12, 8))
fig.suptitle("Risk-Return Profile — All Strategies (Monthly, TC=0.5%)",
             fontsize=13, fontweight='bold')

def parse_pct(s):
    return float(str(s).replace('%', ''))

strategy_groups = {
    'Buy & Hold':     (bh_metrics,  ['BuyAndHold_EW'], 'forestgreen', 'o'),
    'EW Monthly':     (ew_metrics,  ['EW_monthly_TC0.5%'], 'grey', 's'),
    '3-Stage Signal': (sig_metrics, [f"Signal3S_t{t}_monthly_TC0.5%" for t in THRESHOLDS], 'steelblue', '^'),
    'MV Signal':      (mv_metrics,  [f"MV_t{t}_monthly_TC0.5%" for t in THRESHOLDS], 'crimson', 'D'),
}

for group_name, (metrics_dict, keys, col, marker) in strategy_groups.items():
    first = True
    for key in keys:
        if key not in metrics_dict:
            continue
        m = metrics_dict[key]
        cagr = parse_pct(m['CAGR'])
        vol = parse_pct(m['Volatility'])
        ax.scatter(vol, cagr, color=col, marker=marker, s=120, zorder=5,
                   label=group_name if first else None, edgecolors='black', lw=0.8)
        ax.annotate(key.split('_TC')[0],
                    (vol, cagr),
                    textcoords='offset points',
                    xytext=(6, 4),
                    fontsize=7,
                    color=col)
        first = False

ax.axhline(0, color='black', lw=0.6, ls='--')
ax.axvline(0, color='black', lw=0.6, ls='--')
ax.set_xlabel("Annualised Volatility (%)")
ax.set_ylabel("CAGR (%)")
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("risk_return_scatter.png", dpi=150, bbox_inches='tight')
plt.show()

# ── 18. SAVE ALL ──────────────────────────────────────────────────────────────
for label, df in {**bh_results, **ew_results, **signal_results, **mv_results}.items():
    df.to_csv(f"{label}_portfolio.csv")

all_metrics = pd.DataFrame({**bh_metrics, **ew_metrics, **sig_metrics, **mv_metrics}).T
all_metrics.to_csv("all_portfolio_metrics.csv")

composite_score.to_csv("composite_signal.csv")

for thresh in THRESHOLDS:
    sig_df_t = build_signal_df(composite_score, equity_assets, macro_df.index, threshold=thresh)
    sig_df_t.to_csv(f"signal_df_threshold_{thresh}.csv")

for label, wdf in mv_weight_history.items():
    wdf.to_csv(f"{label}_weights.csv")

print("✓ All files saved.")
print(f"\nFiles written:")
print(f"  Portfolio values : {len(bh_results)+len(ew_results)+len(signal_results)+len(mv_results)} CSVs")
print(f"  Weight histories : {len(mv_weight_history)} CSVs")
print(f"  Metrics summary  : all_portfolio_metrics.csv")
print(f"  Signal CSVs      : {len(THRESHOLDS)} threshold files")
print(f"  Plots saved      : 17 PNG files")

In [ ]:
# @title
# ============================================================
# SIGNAL PORTFOLIO — ENHANCED VERSION
# Signals generated from Modified Determinant Model
# + Buy-and-Hold Benchmark
# + Equal-Weight Benchmark
# + Three-Stage Signal Portfolio (original)
# + Enhanced Signal Variants (asymmetric, continuous, smoothed, persistent, partial)
# + Mean-Variance Signal Portfolio
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import requests
from io import StringIO
from scipy.optimize import minimize

# ── 1. LOAD ALL DATA FROM GITHUB ─────────────────────────────────────────────
GITHUB_BASE = "https://raw.githubusercontent.com/kboroz/MSFE_Capstone_Project/main/01_Data/Streamlined/"

INDEX_FILE_MAP = {
    'ASX50':       'streamlined_asx_50.csv',
    'EUROSTOXX50': 'streamlined_euro_stoxx_50.csv',
    'FTSE100':     'streamlined_ftse_100.csv',
    'Ibovespa':    'streamlined_ibovespa.csv',
    'JSE40':       'streamlined_jse_top_40.csv',
    'NIKKEI225':   'streamlined_nikkei_225.csv',
    'SP500':       'streamlined_s&p_500.csv',
}

MACRO_FILE = 'streamlined_macro.csv'

def load_csv(url):
    r = requests.get(url)
    r.raise_for_status()
    df = pd.read_csv(StringIO(r.text), index_col=0, parse_dates=True)
    df.columns = df.columns.str.replace('[^a-zA-Z0-9]', '_', regex=True)
    return df.sort_index()

macro_df = load_csv(GITHUB_BASE + MACRO_FILE)
macro_df = macro_df[~macro_df.index.duplicated(keep='first')].sort_index()
print(f"macro_df shape : {macro_df.shape}")
print(f"Columns        : {macro_df.columns.tolist()}")

index_data = {}
for name, fname in INDEX_FILE_MAP.items():
    try:
        df = load_csv(GITHUB_BASE + fname)
        index_data[name] = df
        print(f"✓ {name:15s} | shape={df.shape} | cols={df.columns.tolist()[:4]}")
    except Exception as e:
        print(f"✗ {name}: {e}")

# ── 2. EXTRACT PRICE SERIES ───────────────────────────────────────────────────
PRICE_COLS = ['Close', 'close', 'Price', 'price', 'Adj_Close', 'Last']

def get_price_series(df, name):
    col = next((c for c in PRICE_COLS if c in df.columns), df.columns[0])
    s = pd.to_numeric(df[col], errors='coerce').dropna()
    print(f"  {name}: using column '{col}'")
    return s

index_prices = {}
for name, df in index_data.items():
    index_prices[name] = get_price_series(df, name)

# ── 3. MODIFIED DETERMINANT SIGNAL MODEL ─────────────────────────────────────
DET_WINDOWS = [3, 7, 28]
WINDOW_WEIGHTS = {3: 0.33, 7: 0.34, 28: 0.33}

def compute_modified_determinant(price_series_dict, windows=DET_WINDOWS):
    prices = pd.DataFrame(price_series_dict).sort_index()
    rets   = prices.pct_change().dropna()
    det_results = {}

    for window in windows:
        det_series = []
        for i in range(window, len(rets)):
            window_rets = rets.iloc[i - window:i]
            valid_cols  = window_rets.columns[window_rets.std() > 1e-10]
            w           = window_rets[valid_cols]

            if len(valid_cols) < 2:
                det_series.append(np.nan)
                continue

            try:
                R   = w.corr().values
                I   = np.eye(len(R))
                M   = R - I
                det = np.linalg.det(M)
            except Exception:
                det = np.nan

            det_series.append(det)

        idx = rets.index[window:]
        det_results[window] = pd.Series(det_series, index=idx,
                                        name=f'det_{window}d')
        print(f"  Window {window:2d}d | computed {len(det_series)} observations")

    return det_results

print("\n── Computing Modified Determinant ──")
det_results = compute_modified_determinant(index_prices)

# ── 4. COMPOSITE SCORE ────────────────────────────────────────────────────────
def compute_composite_score(det_results, window_weights=WINDOW_WEIGHTS):
    all_series = []
    for window, series in det_results.items():
        mu    = series.rolling(252, min_periods=60).mean()
        sigma = series.rolling(252, min_periods=60).std()
        z     = (series - mu) / sigma.replace(0, np.nan)
        z.name = f'z_{window}d'
        all_series.append(z)

    z_df      = pd.concat(all_series, axis=1).dropna()
    composite = sum(
        z_df[f'z_{w}d'] * window_weights[w]
        for w in window_weights if f'z_{w}d' in z_df.columns
    )
    composite = composite / sum(window_weights.values())
    return composite, z_df

composite_score, z_df = compute_composite_score(det_results)

print(f"\nComposite score shape  : {composite_score.shape}")
print(f"Date range             : {composite_score.index[0].date()} → "
      f"{composite_score.index[-1].date()}")
print(f"Score range            : {composite_score.min():.3f} → "
      f"{composite_score.max():.3f}")

# ── 5. SIGNAL BUILDERS ───────────────────────────────────────────────────────
THRESHOLDS    = [0.5, 1.0, 1.5]
SMOOTH_WINDOWS = [3, 5, 10]
MIN_DAYS_LIST  = [5, 10, 20]
MAX_REDUCES    = [0.5, 0.7, 0.9]
PARTIAL_KEEPS  = [0.25, 0.50]

# ── 5a. Original symmetric binary ────────────────────────────────────────────
def classify_signal_symmetric(score, threshold):
    return +1 if abs(score) > threshold else -1

def build_signal_df(composite_score, equity_assets, macro_dates, threshold=0.5):
    binary    = composite_score.apply(lambda x: classify_signal_symmetric(x, threshold))
    signal_df = pd.DataFrame(index=macro_dates)
    for asset in equity_assets:
        signal_df[asset] = binary.reindex(macro_dates).ffill().bfill()
    return signal_df

# ── 5b. Asymmetric (upper-tail only) ─────────────────────────────────────────
def build_signal_df_asymmetric(composite_score, equity_assets, macro_dates,
                               threshold=1.0):
    """Block equities only when correlation SPIKES (high positive z-score)."""
    binary    = composite_score.apply(lambda x: +1 if x > threshold else -1)
    signal_df = pd.DataFrame(index=macro_dates)
    for asset in equity_assets:
        signal_df[asset] = binary.reindex(macro_dates).ffill().bfill()
    return signal_df

# ── 5c. Continuous / graduated scalar ────────────────────────────────────────
def build_signal_df_continuous(composite_score, equity_assets, macro_dates,
                               threshold=1.0, max_reduce=0.8):
    """
    Returns a scalar in [1-max_reduce, 1.0] per asset per day.
    Used by build_weights_continuous() instead of the binary flag.
    """
    def score_to_scalar(s):
        if s <= 0:
            return 1.0
        elif s >= threshold:
            return 1.0 - max_reduce
        else:
            return 1.0 - max_reduce * (s / threshold)

    scalars   = composite_score.apply(score_to_scalar)
    signal_df = pd.DataFrame(index=macro_dates)
    for asset in equity_assets:
        signal_df[asset] = scalars.reindex(macro_dates).ffill().bfill()
    return signal_df

# ── 5d. Smoothed binary ───────────────────────────────────────────────────────
def build_signal_df_smoothed(composite_score, equity_assets, macro_dates,
                             threshold=1.0, smooth_window=5):
    """Smooth z-score before thresholding to reduce whipsaw."""
    smoothed  = composite_score.rolling(smooth_window, min_periods=1).mean()
    binary    = smoothed.apply(lambda x: classify_signal_symmetric(x, threshold))
    signal_df = pd.DataFrame(index=macro_dates)
    for asset in equity_assets:
        signal_df[asset] = binary.reindex(macro_dates).ffill().bfill()
    return signal_df

# ── 5e. Persistent binary ────────────────────────────────────────────────────
def build_signal_df_persistent(composite_score, equity_assets, macro_dates,
                               threshold=1.0, min_days_to_switch=10):
    """Only switch regime after signal holds for min_days_to_switch days."""
    raw       = composite_score.apply(lambda x: +1 if x > threshold else -1)
    confirmed = raw.copy()
    current   = int(raw.iloc[0])
    count     = 0

    for i in range(1, len(raw)):
        if int(raw.iloc[i]) != current:
            count += 1
            if count >= min_days_to_switch:
                current = int(raw.iloc[i])
                count   = 0
        else:
            count = 0
        confirmed.iloc[i] = current

    signal_df = pd.DataFrame(index=macro_dates)
    for asset in equity_assets:
        signal_df[asset] = confirmed.reindex(macro_dates).ffill().bfill()
    return signal_df

# ── 6. ASSET UNIVERSE ─────────────────────────────────────────────────────────
EQUITY_ASSETS = list(index_prices.keys())
SAFE_BONDS    = ['US_10Y', 'UK_10Y', 'AU_10Y', 'ZA_10Y']
FX            = ['AUDUSD', 'BRLUSD', 'EURUSD', 'GBPUSD', 'JPYUSD', 'ZARUSD']
COMMODITIES   = ['Oil', 'Gold', 'Bitcoin']
EXCLUDE_ASSETS = ['Bitcoin']

ALL_ASSETS = EQUITY_ASSETS + FX + COMMODITIES + SAFE_BONDS
ALL_ASSETS = [a for a in ALL_ASSETS
              if a in macro_df.columns and a not in EXCLUDE_ASSETS]

equity_assets     = [a for a in EQUITY_ASSETS     if a in ALL_ASSETS]
non_equity_assets = [a for a in ALL_ASSETS         if a not in equity_assets]

N           = len(ALL_ASSETS)
base_weight = 1.0 / N

print(f"\nTotal assets              : {N}")
print(f"Equity (signal-driven)    : {equity_assets}")
print(f"Non-equity (always long)  : {non_equity_assets}")
print(f"base_weight = 1/{N} = {base_weight:.4f}")

# ── 7. COUNTRY SAFE-HAVEN MAP ─────────────────────────────────────────────────
EQUITY_SAFE_HAVEN_MAP = {
    'ASX50':       ['AU_10Y', 'AUDUSD'],
    'EUROSTOXX50': ['EURUSD'],
    'FTSE100':     ['UK_10Y', 'GBPUSD'],
    'Ibovespa':    ['BRLUSD'],
    'JSE40':       ['ZA_10Y', 'ZARUSD'],
    'NIKKEI225':   ['JPYUSD'],
    'SP500':       ['US_10Y'],
}

UNIVERSE_SET = set(ALL_ASSETS)
for eq, havens in EQUITY_SAFE_HAVEN_MAP.items():
    EQUITY_SAFE_HAVEN_MAP[eq] = [h for h in havens if h in UNIVERSE_SET]

print("\nSafe-haven routing (filtered to universe):")
for eq, havens in EQUITY_SAFE_HAVEN_MAP.items():
    print(f"  {eq:15s} → {havens}")

# ── 8. BUILD RETURNS ──────────────────────────────────────────────────────────
returns_raw = macro_df[ALL_ASSETS].pct_change(fill_method=None)

for col in SAFE_BONDS:
    if col in returns_raw.columns:
        returns_raw[col] = -returns_raw[col]

if 'Oil' in macro_df.columns:
    bad_oil = macro_df['Oil'][macro_df['Oil'] <= 0].index
    for d in bad_oil:
        loc = returns_raw.index.get_loc(d)
        returns_raw.iloc[max(0, loc - 1): loc + 2,
                         returns_raw.columns.get_loc('Oil')] = 0.0

returns = returns_raw.clip(lower=-0.50, upper=1.00).fillna(0).iloc[1:]

assert not returns.isnull().any().any(), "NaNs in returns"
assert not np.isinf(returns.values).any(),  "Inf in returns"
print(f"\nReturns shape : {returns.shape}")
print("✓ Returns clean")

# ── 9. HELPER: REBALANCING DATE SET ──────────────────────────────────────────
def build_reb_set(returns, rebal_freq):
    actual_dates = returns.index
    if rebal_freq == 'daily':
        return set(actual_dates)
    elif rebal_freq == 'weekly':
        raw = returns.resample('W-MON').first().index
    elif rebal_freq == 'monthly':
        raw = returns.resample('MS').first().index
    elif rebal_freq == 'quarterly':
        raw = returns.resample('QS').first().index
    else:
        raise ValueError(f"Unknown rebal_freq: {rebal_freq}")

    reb_set = set()
    for d in raw:
        future = actual_dates[actual_dates >= d]
        if len(future) > 0:
            reb_set.add(future[0])
    return reb_set

# ── 10. BUY-AND-HOLD BENCHMARK ───────────────────────────────────────────────
def run_buy_and_hold(returns, all_assets, initial_capital=1e9):
    N           = len(all_assets)
    base_weight = 1.0 / N
    pv          = float(initial_capital)
    holdings    = {a: pv * base_weight for a in all_assets}
    records     = []
    first       = True

    for date in returns.index:
        if first:
            first = False
            records.append({'date': date, 'portfolio_value': pv})
            continue
        new_holdings = {a: holdings[a] * (1 + returns.loc[date, a])
                        for a in all_assets}
        holdings = new_holdings
        pv       = sum(holdings.values())
        records.append({'date': date, 'portfolio_value': pv})

    df = pd.DataFrame(records).set_index('date')
    print(f"  Buy-and-Hold | Start: ${initial_capital/1e9:.2f}B → "
          f"End: ${df['portfolio_value'].iloc[-1]/1e9:.3f}B")
    return df

# ── 11. WEIGHT BUILDERS ───────────────────────────────────────────────────────

# ── 11a. Original binary 3-stage weights ─────────────────────────────────────
def build_weights_3stage(sig_row, equity_assets, non_equity_assets,
                         base_weight, date, safe_haven_map):
    raw            = {a: base_weight for a in non_equity_assets}
    safe_haven_extra = {}

    for a in equity_assets:
        if sig_row is not None and a in sig_row.index and not pd.isna(sig_row[a]):
            sig = sig_row[a]
        else:
            sig = -1

        if sig > 0:
            raw[a] = -base_weight
            havens = [h for h in safe_haven_map.get(a, []) if h in raw]
            if havens:
                per_haven = base_weight / len(havens)
                for h in havens:
                    safe_haven_extra[h] = safe_haven_extra.get(h, 0) + per_haven
            else:
                per_ne = base_weight / len(non_equity_assets)
                for ne in non_equity_assets:
                    safe_haven_extra[ne] = safe_haven_extra.get(ne, 0) + per_ne
        else:
            raw[a] = base_weight

    for asset, extra in safe_haven_extra.items():
        raw[asset] = raw.get(asset, 0) + extra

    total = sum(raw.values())
    if total <= 0:
        for a in raw:
            raw[a] = 1.0 / len(raw)
    else:
        for a in raw:
            raw[a] /= total
    return raw

# ── 11b. Continuous scalar weights ───────────────────────────────────────────
def build_weights_continuous(sig_row, equity_assets, non_equity_assets,
                             base_weight, safe_haven_map):
    """
    sig_row contains a scalar in [min_weight, 1.0] per equity asset.
    Reduced equity weight is redistributed to safe havens.
    """
    raw              = {a: base_weight for a in non_equity_assets}
    safe_haven_extra = {}

    for a in equity_assets:
        scalar = float(sig_row[a]) if (sig_row is not None and
                                       a in sig_row.index and
                                       not pd.isna(sig_row[a])) else 1.0
        raw[a]   = base_weight * scalar
        released = base_weight * (1.0 - scalar)

        if released > 0:
            havens = [h for h in safe_haven_map.get(a, []) if h in raw]
            if havens:
                per_haven = released / len(havens)
                for h in havens:
                    safe_haven_extra[h] = safe_haven_extra.get(h, 0) + per_haven
            else:
                per_ne = released / len(non_equity_assets)
                for ne in non_equity_assets:
                    safe_haven_extra[ne] = safe_haven_extra.get(ne, 0) + per_ne

    for asset, extra in safe_haven_extra.items():
        raw[asset] = raw.get(asset, 0) + extra

    total = sum(raw.values())
    if total <= 0:
        for a in raw:
            raw[a] = 1.0 / len(raw)
    else:
        for a in raw:
            raw[a] /= total
    return raw

# ── 11c. Partial-hedge weights ────────────────────────────────────────────────
def build_weights_partial(sig_row, equity_assets, non_equity_assets,
                          base_weight, safe_haven_map, partial_keep=0.25):
    """
    On RISK_ON signal: keep partial_keep fraction of base_weight in equity,
    redirect the rest to safe havens.
    """
    raw              = {a: base_weight for a in non_equity_assets}
    safe_haven_extra = {}

    for a in equity_assets:
        if sig_row is not None and a in sig_row.index and not pd.isna(sig_row[a]):
            sig = sig_row[a]
        else:
            sig = -1

        if sig > 0:
            raw[a]   = base_weight * partial_keep
            released = base_weight * (1.0 - partial_keep)
            havens   = [h for h in safe_haven_map.get(a, []) if h in raw]
            if havens:
                per_haven = released / len(havens)
                for h in havens:
                    safe_haven_extra[h] = safe_haven_extra.get(h, 0) + per_haven
            else:
                per_ne = released / len(non_equity_assets)
                for ne in non_equity_assets:
                    safe_haven_extra[ne] = safe_haven_extra.get(ne, 0) + per_ne
        else:
            raw[a] = base_weight

    for asset, extra in safe_haven_extra.items():
        raw[asset] = raw.get(asset, 0) + extra

    total = sum(raw.values())
    if total <= 0:
        for a in raw:
            raw[a] = 1.0 / len(raw)
    else:
        for a in raw:
            raw[a] /= total
    return raw

# ── 12. EQUAL-WEIGHT REBALANCED BENCHMARK ────────────────────────────────────
def run_equal_weight(returns, all_assets, tc_rate=0.005,
                     rebal_freq='monthly', initial_capital=1e9):
    N           = len(all_assets)
    base_weight = 1.0 / N
    ew          = {a: base_weight for a in all_assets}
    reb_set     = build_reb_set(returns, rebal_freq)
    pv          = float(initial_capital)
    current_weights = None
    records     = []

    for date in returns.index:
        if date in reb_set:
            if current_weights is not None:
                turnover = sum(abs(ew.get(a, 0) - current_weights.get(a, 0))
                               for a in set(ew) | set(current_weights)) / 2.0
                pv *= (1 - turnover * tc_rate)
            else:
                pv *= (1 - tc_rate)
            current_weights = ew.copy()

        if current_weights is not None:
            dr  = sum(current_weights.get(a, 0) * returns.loc[date, a]
                      for a in all_assets)
            pv *= (1 + dr)

        records.append({'date': date, 'portfolio_value': pv})

    return pd.DataFrame(records).set_index('date')

# ── 13. GENERIC SIGNAL PORTFOLIO RUNNER ──────────────────────────────────────
def run_signal_portfolio(returns, signal_df, equity_assets,
                         non_equity_assets, all_assets,
                         weight_fn,          # callable: (sig_row, ...) → dict
                         freq='monthly', tc_rate=0.005,
                         initial_capital=1e9):
    """
    Generic runner.  weight_fn must accept (sig_row, equity_assets,
    non_equity_assets, base_weight, <any extra kwargs>) and return
    a dict {asset: weight}.
    """
    N           = len(all_assets)
    base_weight = 1.0 / N
    reb_set     = build_reb_set(returns, freq)

    sig = signal_df.copy()
    sig.index = pd.to_datetime(sig.index)

    pv              = float(initial_capital)
    current_weights = None
    records         = []

    for date in returns.index:
        if date in reb_set:
            past    = sig[sig.index <= date]
            sig_row = past.iloc[-1] if len(past) > 0 else None
            raw     = weight_fn(sig_row)

            if current_weights is not None:
                all_keys = set(raw) | set(current_weights)
                turnover = sum(abs(raw.get(a, 0) - current_weights.get(a, 0))
                               for a in all_keys) / 2.0
                pv *= (1 - turnover * tc_rate)
            else:
                pv *= (1 - tc_rate)

            current_weights = raw.copy()

        if current_weights is not None:
            dr  = sum(current_weights.get(a, 0) * returns.loc[date, a]
                      for a in all_assets)
            pv *= (1 + dr)

        records.append({'date': date, 'portfolio_value': pv})

    return pd.DataFrame(records).set_index('date')

# ── 14. MEAN-VARIANCE SIGNAL PORTFOLIO ───────────────────────────────────────
MV_LOOKBACK = 252
MV_MIN_OBS  = 60
MAX_WEIGHT  = 0.30
RISK_FREE   = 0.0

def mean_variance_weights(mu_vec, cov_mat, allowed_mask,
                          max_weight=MAX_WEIGHT, risk_free=RISK_FREE):
    n_total   = len(mu_vec)
    allowed   = np.where(allowed_mask)[0]
    n_allowed = len(allowed)

    fallback = np.zeros(n_total)
    if n_allowed == 0:
        fallback[:] = 1.0 / n_total
        return fallback
    fallback[allowed] = 1.0 / n_allowed

    mu_sub  = mu_vec[allowed]
    cov_sub = cov_mat[np.ix_(allowed, allowed)] + np.eye(n_allowed) * 1e-8

    def neg_sharpe(w):
        port_ret = np.dot(w, mu_sub)
        port_var = np.dot(w, cov_sub @ w)
        port_std = np.sqrt(max(port_var, 1e-12))
        return -(port_ret - risk_free) / port_std

    constraints = [{'type': 'eq', 'fun': lambda w: np.sum(w) - 1.0}]
    bounds      = [(0.0, max_weight)] * n_allowed
    w0          = np.ones(n_allowed) / n_allowed

    try:
        res = minimize(neg_sharpe, w0, method='SLSQP',
                       bounds=bounds, constraints=constraints,
                       options={'ftol': 1e-9, 'maxiter': 500})
        w_opt = res.x if (res.success and not np.any(np.isnan(res.x))) else w0
    except Exception:
        w_opt = w0

    w_full         = np.zeros(n_total)
    w_full[allowed] = w_opt
    return w_full

def run_mv_signal_portfolio(returns, signal_df, all_assets,
                            equity_assets,
                            freq='monthly', tc_rate=0.005,
                            lookback=MV_LOOKBACK, min_obs=MV_MIN_OBS,
                            max_weight=MAX_WEIGHT, initial_capital=1e9):
    reb_set      = build_reb_set(returns, freq)
    actual_dates = returns.index
    assets       = all_assets
    n            = len(assets)
    asset_idx    = {a: i for i, a in enumerate(assets)}

    sig = signal_df.copy()
    sig.index = pd.to_datetime(sig.index)

    pv              = float(initial_capital)
    current_weights = np.zeros(n)
    records         = []
    weight_history  = []
    first_reb       = True

    for t, date in enumerate(actual_dates):
        if date in reb_set:
            hist = returns.iloc[max(0, t - lookback): t]

            if len(hist) < min_obs:
                new_weights = np.ones(n) / n
            else:
                mu_ann  = hist.mean().values  * 252
                cov_ann = hist.cov().values   * 252

                past    = sig[sig.index <= date]
                sig_row = past.iloc[-1] if len(past) > 0 else None

                allowed_mask = np.ones(n, dtype=bool)
                if sig_row is not None:
                    for eq in equity_assets:
                        if eq in asset_idx and eq in sig_row.index:
                            s = sig_row[eq]
                            if not pd.isna(s) and s > 0:
                                allowed_mask[asset_idx[eq]] = False

                new_weights = mean_variance_weights(
                    mu_ann, cov_ann, allowed_mask, max_weight=max_weight)

            if first_reb:
                pv       *= (1 - tc_rate)
                first_reb = False
            else:
                turnover = np.sum(np.abs(new_weights - current_weights)) / 2.0
                pv      *= (1 - turnover * tc_rate)

            current_weights = new_weights.copy()
            weight_history.append({'date': date,
                                   **dict(zip(assets, current_weights))})

        if not first_reb:
            dr  = np.dot(current_weights, returns.loc[date].values)
            pv *= (1 + dr)

        records.append({'date': date, 'portfolio_value': pv})

    pv_df      = pd.DataFrame(records).set_index('date')
    weights_df = pd.DataFrame(weight_history).set_index('date')
    print(f"  MV | freq={freq} tc={tc_rate*100:.1f}% "
          f"End: ${pv_df['portfolio_value'].iloc[-1]/1e9:.3f}B")
    return pv_df, weights_df

# ── 15. RUN ALL PORTFOLIOS ────────────────────────────────────────────────────
TC_LEVELS = {'0%': 0.00, '0.5%': 0.005, '1%': 0.01, '2%': 0.02}
FREQS     = ['daily', 'weekly', 'monthly', 'quarterly']
MV_FREQS  = ['monthly', 'quarterly']

# ── 15a. Buy-and-Hold ─────────────────────────────────────────────────────────
print("\n── Running Buy-and-Hold ──")
bh_result  = run_buy_and_hold(returns, ALL_ASSETS, initial_capital=1e9)
bh_results = {'BuyAndHold_EW': bh_result}

# ── 15b. Equal-Weight ─────────────────────────────────────────────────────────
print("\n── Running Equal-Weight Rebalanced ──")
ew_results = {}
for freq in FREQS:
    for tc_label, tc_rate in TC_LEVELS.items():
        label = f"EW_{freq}_TC{tc_label}"
        ew_results[label] = run_equal_weight(
            returns, ALL_ASSETS, tc_rate=tc_rate,
            rebal_freq=freq, initial_capital=1e9)
        print(f"  ✓ {label}")

# ── 15c. Original 3-Stage Signal ─────────────────────────────────────────────
print("\n── Running Original 3-Stage Signal ──")
signal_results       = {}
signal_distributions = {}

for thresh in THRESHOLDS:
    sig_df_t    = build_signal_df(composite_score, equity_assets,
                                  macro_df.index, threshold=thresh)
    n_risk_on   = (sig_df_t[equity_assets[0]] == +1).sum()
    n_risk_off  = (sig_df_t[equity_assets[0]] == -1).sum()
    total       = len(sig_df_t)
    signal_distributions[thresh] = {'RISK_ON': n_risk_on, 'RISK_OFF': n_risk_off}
    print(f"  Threshold={thresh} | RISK_ON: {n_risk_on} ({n_risk_on/total*100:.1f}%) "
          f"| RISK_OFF: {n_risk_off} ({n_risk_off/total*100:.1f}%)")

    for freq in FREQS:
        for tc_label, tc_rate in TC_LEVELS.items():
            label = f"Sig3S_t{thresh}_{freq}_TC{tc_label}"

            def wfn_3s(sig_row, _eq=equity_assets, _ne=non_equity_assets,
                       _bw=base_weight, _sh=EQUITY_SAFE_HAVEN_MAP):
                return build_weights_3stage(sig_row, _eq, _ne, _bw, None, _sh)

            signal_results[label] = run_signal_portfolio(
                returns, sig_df_t, equity_assets, non_equity_assets,
                ALL_ASSETS, weight_fn=wfn_3s,
                freq=freq, tc_rate=tc_rate, initial_capital=1e9)
            print(f"    ✓ {label}")

# ── 15d. Asymmetric Signal ────────────────────────────────────────────────────
print("\n── Running Asymmetric Signal ──")
asym_results = {}

for thresh in THRESHOLDS:
    sig_df_t = build_signal_df_asymmetric(composite_score, equity_assets,
                                          macro_df.index, threshold=thresh)
    for freq in ['monthly', 'quarterly']:
        for tc_label, tc_rate in TC_LEVELS.items():
            label = f"Asym_t{thresh}_{freq}_TC{tc_label}"

            def wfn_asym(sig_row, _eq=equity_assets, _ne=non_equity_assets,
                         _bw=base_weight, _sh=EQUITY_SAFE_HAVEN_MAP):
                return build_weights_3stage(sig_row, _eq, _ne, _bw, None, _sh)

            asym_results[label] = run_signal_portfolio(
                returns, sig_df_t, equity_assets, non_equity_assets,
                ALL_ASSETS, weight_fn=wfn_asym,
                freq=freq, tc_rate=tc_rate, initial_capital=1e9)
            print(f"    ✓ {label}")

# ── 15e. Continuous / Graduated Signal ───────────────────────────────────────
print("\n── Running Continuous Signal ──")
cont_results = {}

for thresh in THRESHOLDS:
    for max_red in MAX_REDUCES:
        sig_df_t = build_signal_df_continuous(composite_score, equity_assets,
                                              macro_df.index,
                                              threshold=thresh,
                                              max_reduce=max_red)
        for freq in ['monthly', 'quarterly']:
            for tc_label, tc_rate in TC_LEVELS.items():
                label = (f"Cont_t{thresh}_mr{int(max_red*100)}"
                         f"_{freq}_TC{tc_label}")

                def wfn_cont(sig_row, _eq=equity_assets, _ne=non_equity_assets,
                             _bw=base_weight, _sh=EQUITY_SAFE_HAVEN_MAP):
                    return build_weights_continuous(sig_row, _eq, _ne, _bw, _sh)

                cont_results[label] = run_signal_portfolio(
                    returns, sig_df_t, equity_assets, non_equity_assets,
                    ALL_ASSETS, weight_fn=wfn_cont,
                    freq=freq, tc_rate=tc_rate, initial_capital=1e9)
                print(f"    ✓ {label}")

# ── 15f. Smoothed Signal ─────────────────────────────────────────────────────
print("\n── Running Smoothed Signal ──")
smooth_results = {}

for thresh in THRESHOLDS:
    for sw in SMOOTH_WINDOWS:
        sig_df_t = build_signal_df_smoothed(composite_score, equity_assets,
                                            macro_df.index,
                                            threshold=thresh,
                                            smooth_window=sw)
        for freq in ['monthly', 'quarterly']:
            for tc_label, tc_rate in TC_LEVELS.items():
                label = f"Smooth_t{thresh}_sw{sw}_{freq}_TC{tc_label}"

                def wfn_smooth(sig_row, _eq=equity_assets, _ne=non_equity_assets,
                               _bw=base_weight, _sh=EQUITY_SAFE_HAVEN_MAP):
                    return build_weights_3stage(sig_row, _eq, _ne, _bw, None, _sh)

                smooth_results[label] = run_signal_portfolio(
                    returns, sig_df_t, equity_assets, non_equity_assets,
                    ALL_ASSETS, weight_fn=wfn_smooth,
                    freq=freq, tc_rate=tc_rate, initial_capital=1e9)
                print(f"    ✓ {label}")

# ── 15g. Persistent Signal ────────────────────────────────────────────────────
print("\n── Running Persistent Signal ──")
persist_results = {}

for thresh in THRESHOLDS:
    for min_days in MIN_DAYS_LIST:
        sig_df_t = build_signal_df_persistent(composite_score, equity_assets,
                                              macro_df.index,
                                              threshold=thresh,
                                              min_days_to_switch=min_days)
        for freq in ['monthly', 'quarterly']:
            for tc_label, tc_rate in TC_LEVELS.items():
                label = f"Persist_t{thresh}_md{min_days}_{freq}_TC{tc_label}"

                def wfn_persist(sig_row, _eq=equity_assets, _ne=non_equity_assets,
                                _bw=base_weight, _sh=EQUITY_SAFE_HAVEN_MAP):
                    return build_weights_3stage(sig_row, _eq, _ne, _bw, None, _sh)

                persist_results[label] = run_signal_portfolio(
                    returns, sig_df_t, equity_assets, non_equity_assets,
                    ALL_ASSETS, weight_fn=wfn_persist,
                    freq=freq, tc_rate=tc_rate, initial_capital=1e9)
                print(f"    ✓ {label}")

# ── 15h. Partial-Hedge Signal ─────────────────────────────────────────────────
print("\n── Running Partial-Hedge Signal ──")
partial_results = {}

for thresh in THRESHOLDS:
    sig_df_t = build_signal_df(composite_score, equity_assets,
                               macro_df.index, threshold=thresh)
    for pk in PARTIAL_KEEPS:
        for freq in ['monthly', 'quarterly']:
            for tc_label, tc_rate in TC_LEVELS.items():
                label = (f"Partial_t{thresh}_pk{int(pk*100)}"
                         f"_{freq}_TC{tc_label}")

                def wfn_partial(sig_row, _eq=equity_assets, _ne=non_equity_assets,
                                _bw=base_weight, _sh=EQUITY_SAFE_HAVEN_MAP,
                                _pk=pk):
                    return build_weights_partial(sig_row, _eq, _ne, _bw, _sh, _pk)

                partial_results[label] = run_signal_portfolio(
                    returns, sig_df_t, equity_assets, non_equity_assets,
                    ALL_ASSETS, weight_fn=wfn_partial,
                    freq=freq, tc_rate=tc_rate, initial_capital=1e9)
                print(f"    ✓ {label}")

# ── 15i. Mean-Variance Signal ─────────────────────────────────────────────────
print("\n── Running Mean-Variance Signal ──")
mv_results        = {}
mv_weight_history = {}

for thresh in THRESHOLDS:
    sig_df_t = build_signal_df(composite_score, equity_assets,
                               macro_df.index, threshold=thresh)
    for freq in MV_FREQS:
        for tc_label, tc_rate in TC_LEVELS.items():
            label = f"MV_t{thresh}_{freq}_TC{tc_label}"
            pv_df, w_df = run_mv_signal_portfolio(
                returns, sig_df_t, ALL_ASSETS,
                equity_assets=equity_assets,
                freq=freq, tc_rate=tc_rate,
                lookback=MV_LOOKBACK, min_obs=MV_MIN_OBS,
                max_weight=MAX_WEIGHT, initial_capital=1e9)
            mv_results[label]        = pv_df
            mv_weight_history[label] = w_df
            print(f"  ✓ {label}")

print("\n✓ All portfolios computed")

# ── 16. PERFORMANCE METRICS ───────────────────────────────────────────────────
def compute_metrics(pv_df, label=''):
    pv        = pv_df['portfolio_value']
    daily_ret = pv.pct_change().dropna()
    n_years   = len(daily_ret) / 252
    total_ret = (pv.iloc[-1] / pv.iloc[0]) - 1
    cagr      = (1 + total_ret) ** (1 / n_years) - 1 if n_years > 0 else np.nan
    vol       = daily_ret.std() * np.sqrt(252)
    sharpe    = cagr / vol if vol > 0 else np.nan
    roll_max  = pv.cummax()
    max_dd    = ((pv - roll_max) / roll_max).min()
    calmar    = cagr / abs(max_dd) if max_dd != 0 else np.nan
    var_95    = daily_ret.quantile(0.05)
    ann_ret   = cagr
    ann_vol   = vol

    return {
        'CAGR':         f"{cagr*100:.2f}%",
        'Ann_Ret':      ann_ret,
        'Ann_Vol':      ann_vol,
        'Volatility':   f"{vol*100:.2f}%",
        'Sharpe':       f"{sharpe:.2f}",
        'Max Drawdown': f"{max_dd*100:.2f}%",
        'Calmar':       f"{calmar:.2f}",
        'VaR 95%':      f"{var_95*100:.2f}%",
        'Total Return': f"{total_ret*100:.2f}%",
        'Final ($B)':   f"{pv.iloc[-1]/1e9:.3f}",
    }

bh_metrics      = {k: compute_metrics(v, k) for k, v in bh_results.items()}
ew_metrics      = {k: compute_metrics(v, k) for k, v in ew_results.items()}
sig_metrics     = {k: compute_metrics(v, k) for k, v in signal_results.items()}
asym_metrics    = {k: compute_metrics(v, k) for k, v in asym_results.items()}
cont_metrics    = {k: compute_metrics(v, k) for k, v in cont_results.items()}
smooth_metrics  = {k: compute_metrics(v, k) for k, v in smooth_results.items()}
persist_metrics = {k: compute_metrics(v, k) for k, v in persist_results.items()}
partial_metrics = {k: compute_metrics(v, k) for k, v in partial_results.items()}
mv_metrics      = {k: compute_metrics(v, k) for k, v in mv_results.items()}

def print_metrics_table(metrics_dict, title):
    display = {k: {mk: mv for mk, mv in m.items()
                   if mk not in ('Ann_Ret', 'Ann_Vol')}
               for k, m in metrics_dict.items()}
    print(f"\n{'='*90}")
    print(title)
    print("="*90)
    print(pd.DataFrame(display).T.to_string())

print_metrics_table(bh_metrics,      "BUY-AND-HOLD")
print_metrics_table(ew_metrics,      "EQUAL-WEIGHT REBALANCED")

for thresh in THRESHOLDS:
    sub = {k: v for k, v in sig_metrics.items()     if f"_t{thresh}_" in k}
    print_metrics_table(sub, f"3-STAGE SIGNAL — threshold={thresh}")

for thresh in THRESHOLDS:
    sub = {k: v for k, v in asym_metrics.items()    if f"_t{thresh}_" in k}
    print_metrics_table(sub, f"ASYMMETRIC SIGNAL — threshold={thresh}")

for thresh in THRESHOLDS:
    sub = {k: v for k, v in cont_metrics.items()    if f"_t{thresh}_" in k}
    print_metrics_table(sub, f"CONTINUOUS SIGNAL — threshold={thresh}")

for thresh in THRESHOLDS:
    sub = {k: v for k, v in smooth_metrics.items()  if f"_t{thresh}_" in k}
    print_metrics_table(sub, f"SMOOTHED SIGNAL — threshold={thresh}")

for thresh in THRESHOLDS:
    sub = {k: v for k, v in persist_metrics.items() if f"_t{thresh}_" in k}
    print_metrics_table(sub, f"PERSISTENT SIGNAL — threshold={thresh}")

for thresh in THRESHOLDS:
    sub = {k: v for k, v in partial_metrics.items() if f"_t{thresh}_" in k}
    print_metrics_table(sub, f"PARTIAL-HEDGE SIGNAL — threshold={thresh}")

for thresh in THRESHOLDS:
    sub = {k: v for k, v in mv_metrics.items()      if f"_t{thresh}_" in k}
    print_metrics_table(sub, f"MEAN-VARIANCE SIGNAL — threshold={thresh}")

# ── 17. PLOTS ─────────────────────────────────────────────────────────────────
bh_pv     = bh_results['BuyAndHold_EW']['portfolio_value']
ew_ref_pv = ew_results['EW_monthly_TC0.5%']['portfolio_value']

def parse_pct(s):
    return float(str(s).replace('%', '')) / 100

# ── Plot A: Best of each variant vs benchmarks (monthly, TC=0.5%) ─────────────
fig, axes = plt.subplots(2, 4, figsize=(28, 12), sharey=False)
fig.suptitle("All Signal Variants vs Benchmarks — Monthly Rebalancing, TC=0.5%",
             fontsize=14, fontweight='bold')

variant_info = [
    ("3-Stage",    signal_results,  THRESHOLDS,     "Sig3S",   "steelblue"),
    ("Asymmetric", asym_results,    THRESHOLDS,     "Asym",    "darkorange"),
    ("Continuous", cont_results,    THRESHOLDS,     "Cont",    "purple"),
    ("Smoothed",   smooth_results,  THRESHOLDS,     "Smooth",  "green"),
    ("Persistent", persist_results, THRESHOLDS,     "Persist", "crimson"),
    ("Partial 25%",partial_results, THRESHOLDS,     "Partial", "brown"),
    ("MV Signal",  mv_results,      THRESHOLDS,     "MV",      "black"),
]

for ax, (name, res_dict, threshs, prefix, col) in zip(axes.flat, variant_info):
    ax.plot(bh_pv.index,     bh_pv / 1e9,     color='forestgreen',
            lw=1.5, ls='-.', label='Buy & Hold')
    ax.plot(ew_ref_pv.index, ew_ref_pv / 1e9, color='grey',
            lw=1.2, ls='--', label='EW Monthly')

    for thresh in threshs:
        candidates = [k for k in res_dict
                      if f"_t{thresh}_" in k and 'monthly' in k and 'TC0.5%' in k]
        if not candidates:
            candidates = [k for k in res_dict
                          if f"_t{thresh}_" in k and 'TC0.5%' in k]
        if candidates:
            key = candidates[0]
            pv  = res_dict[key]['portfolio_value']
            ax.plot(pv.index, pv / 1e9, lw=1.6,
                    label=f"t={thresh}")

    ax.axhline(1.0, color='black', ls=':', lw=0.7)
    ax.set_title(name, fontsize=11, fontweight='bold')
    ax.set_ylabel("Value ($B)")
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

axes.flat[-1].set_visible(False)
plt.tight_layout()
plt.savefig("all_variants_monthly.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Plot B: Risk-Return Scatter — all strategies ──────────────────────────────
fig, ax = plt.subplots(figsize=(16, 10))

all_metric_groups = [
    (bh_metrics,      'BH',       'forestgreen', 'D'),
    (ew_metrics,      'EW',       'grey',        's'),
    (sig_metrics,     '3-Stage',  'steelblue',   'o'),
    (asym_metrics,    'Asym',     'darkorange',  'o'),
    (cont_metrics,    'Cont',     'purple',      'o'),
    (smooth_metrics,  'Smooth',   'green',       'o'),
    (persist_metrics, 'Persist',  'crimson',     'o'),
    (partial_metrics, 'Partial',  'brown',       'o'),
    (mv_metrics,      'MV',       'black',       '^'),
]

for metrics_dict, grp_label, col, marker in all_metric_groups:
    first = True
    for key, m in metrics_dict.items():
        vol = m['Ann_Vol'] * 100
        ret = m['Ann_Ret'] * 100
        ax.scatter(vol, ret, color=col, marker=marker, s=80,
                   label=grp_label if first else None,
                   edgecolors='black', lw=0.5, zorder=4)
        ax.annotate(key.split('_TC')[0], (vol, ret),
                    textcoords='offset points', xytext=(4, 3),
                    fontsize=6, color=col)
        first = False

ax.axhline(0, color='black', lw=0.6, ls='--')
ax.axvline(0, color='black', lw=0.6, ls='--')
ax.set_xlabel("Annualised Volatility (%)", fontsize=12)
ax.set_ylabel("CAGR (%)", fontsize=12)
ax.set_title("Risk-Return — All Strategies", fontsize=13, fontweight='bold')
ax.legend(fontsize=10, ncol=3)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("risk_return_all.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Plot C: Sharpe Bar Chart — top 20 strategies ──────────────────────────────
all_metrics_combined = {
    **bh_metrics, **ew_metrics, **sig_metrics,
    **asym_metrics, **cont_metrics, **smooth_metrics,
    **persist_metrics, **partial_metrics, **mv_metrics
}

sharpe_series = pd.Series({
    k: float(v['Sharpe']) for k, v in all_metrics_combined.items()
}).dropna().sort_values(ascending=False)

top20 = sharpe_series.head(20)

def label_color(lbl):
    if lbl.startswith('MV'):       return 'black'
    if lbl.startswith('Sig3S'):    return 'steelblue'
    if lbl.startswith('Asym'):     return 'darkorange'
    if lbl.startswith('Cont'):     return 'purple'
    if lbl.startswith('Smooth'):   return 'green'
    if lbl.startswith('Persist'):  return 'crimson'
    if lbl.startswith('Partial'):  return 'brown'
    if lbl.startswith('EW'):       return 'grey'
    return 'forestgreen'

colors = [label_color(k) for k in top20.index]

fig, ax = plt.subplots(figsize=(16, 7))
bars = ax.barh(range(len(top20)), top20.values, color=colors, edgecolor='black', lw=0.5)
ax.set_yticks(range(len(top20)))
ax.set_yticklabels([k.replace('_TC', ' TC=') for k in top20.index], fontsize=8)
ax.set_xlabel("Sharpe Ratio")
ax.set_title("Top 20 Strategies by Sharpe Ratio", fontsize=13, fontweight='bold')
ax.axvline(0, color='black', lw=0.8)
ax.grid(True, axis='x', alpha=0.3)

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='black',      label='MV Signal'),
    Patch(facecolor='steelblue',  label='3-Stage'),
    Patch(facecolor='darkorange', label='Asymmetric'),
    Patch(facecolor='purple',     label='Continuous'),
    Patch(facecolor='green',      label='Smoothed'),
    Patch(facecolor='crimson',    label='Persistent'),
    Patch(facecolor='brown',      label='Partial Hedge'),
    Patch(facecolor='grey',       label='Equal Weight'),
    Patch(facecolor='forestgreen',label='Buy & Hold'),
]
ax.legend(handles=legend_elements, fontsize=9, loc='lower right')
plt.tight_layout()
plt.savefig("sharpe_top20.png", dpi=150, bbox_inches='tight')
plt.show()

# ── 18. SAVE ALL ──────────────────────────────────────────────────────────────
print("\n── Saving all results ──")
all_portfolios = {
    **bh_results, **ew_results, **signal_results,
    **asym_results, **cont_results, **smooth_results,
    **persist_results, **partial_results, **mv_results
}

for label, df in all_portfolios.items():
    df.to_csv(f"{label}_portfolio.csv")

all_metrics_df = pd.DataFrame({
    k: {mk: mv for mk, mv in m.items() if mk not in ('Ann_Ret', 'Ann_Vol')}
    for k, m in all_metrics_combined.items()
}).T
all_metrics_df.to_csv("all_portfolio_metrics.csv")

composite_score.to_csv("composite_signal.csv")
for thresh in THRESHOLDS:
    sig_df_t = build_signal_df(composite_score, equity_assets,
                               macro_df.index, threshold=thresh)
    sig_df_t.to_csv(f"signal_df_threshold_{thresh}.csv")

for label, wdf in mv_weight_history.items():
    wdf.to_csv(f"{label}_weights.csv")

print("✓ All files saved.")
print(f"\n  Portfolio CSVs   : {len(all_portfolios)}")
print(f"  Weight CSVs      : {len(mv_weight_history)}")
print(f"  Metrics CSV      : all_portfolio_metrics.csv")
print(f"  Signal CSVs      : {len(THRESHOLDS)} threshold files")
print(f"  Plots saved      : 3 PNG files")


In [ ]:
# @title
# ============================================================
# SIGNAL PORTFOLIO — ENHANCED VERSION
# Signals generated from Modified Determinant Model
# + Buy-and-Hold Benchmark
# + Equal-Weight Benchmark
# + Three-Stage Signal Portfolio (original)
# + Enhanced Signal Variants (asymmetric, continuous, smoothed, persistent, partial)
# + Mean-Variance Signal Portfolio
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import requests
from io import StringIO
from scipy.optimize import minimize

# ── 1. LOAD ALL DATA FROM GITHUB ─────────────────────────────────────────────
GITHUB_BASE = "https://raw.githubusercontent.com/kboroz/MSFE_Capstone_Project/main/01_Data/Streamlined/"

INDEX_FILE_MAP = {
    'ASX50':       'streamlined_asx_50.csv',
    'EUROSTOXX50': 'streamlined_euro_stoxx_50.csv',
    'FTSE100':     'streamlined_ftse_100.csv',
    'Ibovespa':    'streamlined_ibovespa.csv',
    'JSE40':       'streamlined_jse_top_40.csv',
    'NIKKEI225':   'streamlined_nikkei_225.csv',
    'SP500':       'streamlined_s&p_500.csv',
}

MACRO_FILE = 'streamlined_macro.csv'

def load_csv(url):
    r = requests.get(url)
    r.raise_for_status()
    df = pd.read_csv(StringIO(r.text), index_col=0, parse_dates=True)
    df.columns = df.columns.str.replace('[^a-zA-Z0-9]', '_', regex=True)
    return df.sort_index()

macro_df = load_csv(GITHUB_BASE + MACRO_FILE)
macro_df = macro_df[~macro_df.index.duplicated(keep='first')].sort_index()
print(f"macro_df shape : {macro_df.shape}")
print(f"Columns        : {macro_df.columns.tolist()}")

index_data = {}
for name, fname in INDEX_FILE_MAP.items():
    try:
        df = load_csv(GITHUB_BASE + fname)
        index_data[name] = df
        print(f"✓ {name:15s} | shape={df.shape} | cols={df.columns.tolist()[:4]}")
    except Exception as e:
        print(f"✗ {name}: {e}")

# ── 2. EXTRACT PRICE SERIES ───────────────────────────────────────────────────
PRICE_COLS = ['Close', 'close', 'Price', 'price', 'Adj_Close', 'Last']

def get_price_series(df, name):
    col = next((c for c in PRICE_COLS if c in df.columns), df.columns[0])
    s = pd.to_numeric(df[col], errors='coerce').dropna()
    print(f"  {name}: using column '{col}'")
    return s

index_prices = {}
for name, df in index_data.items():
    index_prices[name] = get_price_series(df, name)

# ── 3. MODIFIED DETERMINANT SIGNAL MODEL ─────────────────────────────────────
DET_WINDOWS = [3, 7, 28]
WINDOW_WEIGHTS = {3: 0.25, 7: 0.25, 28: 0.50}

def compute_modified_determinant(price_series_dict, windows=DET_WINDOWS):
    prices = pd.DataFrame(price_series_dict).sort_index()
    rets   = prices.pct_change().dropna()
    det_results = {}

    for window in windows:
        det_series = []
        for i in range(window, len(rets)):
            window_rets = rets.iloc[i - window:i]
            valid_cols  = window_rets.columns[window_rets.std() > 1e-10]
            w           = window_rets[valid_cols]

            if len(valid_cols) < 2:
                det_series.append(np.nan)
                continue

            try:
                R   = w.corr().values
                I   = np.eye(len(R))
                M   = R - I
                det = np.linalg.det(M)
            except Exception:
                det = np.nan

            det_series.append(det)

        idx = rets.index[window:]
        det_results[window] = pd.Series(det_series, index=idx,
                                        name=f'det_{window}d')
        print(f"  Window {window:2d}d | computed {len(det_series)} observations")

    return det_results

print("\n── Computing Modified Determinant ──")
det_results = compute_modified_determinant(index_prices)

# ── 4. COMPOSITE SCORE ────────────────────────────────────────────────────────
def compute_composite_score(det_results, window_weights=WINDOW_WEIGHTS):
    all_series = []
    for window, series in det_results.items():
        mu    = series.rolling(252, min_periods=60).mean()
        sigma = series.rolling(252, min_periods=60).std()
        z     = (series - mu) / sigma.replace(0, np.nan)
        z.name = f'z_{window}d'
        all_series.append(z)

    z_df      = pd.concat(all_series, axis=1).dropna()
    composite = sum(
        z_df[f'z_{w}d'] * window_weights[w]
        for w in window_weights if f'z_{w}d' in z_df.columns
    )
    composite = composite / sum(window_weights.values())
    return composite, z_df

composite_score, z_df = compute_composite_score(det_results)

print(f"\nComposite score shape  : {composite_score.shape}")
print(f"Date range             : {composite_score.index[0].date()} → "
      f"{composite_score.index[-1].date()}")
print(f"Score range            : {composite_score.min():.3f} → "
      f"{composite_score.max():.3f}")

# ── 5. SIGNAL BUILDERS ───────────────────────────────────────────────────────
THRESHOLDS    = [0.3, 0.5, 0.7]
SMOOTH_WINDOWS = [3, 5, 10]
MIN_DAYS_LIST  = [5, 10, 20]
MAX_REDUCES    = [0.5, 0.7, 0.9]
PARTIAL_KEEPS  = [0.25, 0.50]

# ── 5a. Original symmetric binary ────────────────────────────────────────────
def classify_signal_symmetric(score, threshold):
    return +1 if abs(score) > threshold else -1

def build_signal_df(composite_score, equity_assets, macro_dates, threshold=0.5):
    binary    = composite_score.apply(lambda x: classify_signal_symmetric(x, threshold))
    signal_df = pd.DataFrame(index=macro_dates)
    for asset in equity_assets:
        signal_df[asset] = binary.reindex(macro_dates).ffill().bfill()
    return signal_df

# ── 5b. Asymmetric (upper-tail only) ─────────────────────────────────────────
def build_signal_df_asymmetric(composite_score, equity_assets, macro_dates,
                               threshold=1.0):
    """Block equities only when correlation SPIKES (high positive z-score)."""
    binary    = composite_score.apply(lambda x: +1 if x > threshold else -1)
    signal_df = pd.DataFrame(index=macro_dates)
    for asset in equity_assets:
        signal_df[asset] = binary.reindex(macro_dates).ffill().bfill()
    return signal_df

# ── 5c. Continuous / graduated scalar ────────────────────────────────────────
def build_signal_df_continuous(composite_score, equity_assets, macro_dates,
                               threshold=1.0, max_reduce=0.8):
    """
    Returns a scalar in [1-max_reduce, 1.0] per asset per day.
    Used by build_weights_continuous() instead of the binary flag.
    """
    def score_to_scalar(s):
        if s <= 0:
            return 1.0
        elif s >= threshold:
            return 1.0 - max_reduce
        else:
            return 1.0 - max_reduce * (s / threshold)

    scalars   = composite_score.apply(score_to_scalar)
    signal_df = pd.DataFrame(index=macro_dates)
    for asset in equity_assets:
        signal_df[asset] = scalars.reindex(macro_dates).ffill().bfill()
    return signal_df

# ── 5d. Smoothed binary ───────────────────────────────────────────────────────
def build_signal_df_smoothed(composite_score, equity_assets, macro_dates,
                             threshold=1.0, smooth_window=5):
    """Smooth z-score before thresholding to reduce whipsaw."""
    smoothed  = composite_score.rolling(smooth_window, min_periods=1).mean()
    binary    = smoothed.apply(lambda x: classify_signal_symmetric(x, threshold))
    signal_df = pd.DataFrame(index=macro_dates)
    for asset in equity_assets:
        signal_df[asset] = binary.reindex(macro_dates).ffill().bfill()
    return signal_df

# ── 5e. Persistent binary ────────────────────────────────────────────────────
def build_signal_df_persistent(composite_score, equity_assets, macro_dates,
                               threshold=1.0, min_days_to_switch=10):
    """Only switch regime after signal holds for min_days_to_switch days."""
    raw       = composite_score.apply(lambda x: +1 if x > threshold else -1)
    confirmed = raw.copy()
    current   = int(raw.iloc[0])
    count     = 0

    for i in range(1, len(raw)):
        if int(raw.iloc[i]) != current:
            count += 1
            if count >= min_days_to_switch:
                current = int(raw.iloc[i])
                count   = 0
        else:
            count = 0
        confirmed.iloc[i] = current

    signal_df = pd.DataFrame(index=macro_dates)
    for asset in equity_assets:
        signal_df[asset] = confirmed.reindex(macro_dates).ffill().bfill()
    return signal_df

# ── 6. ASSET UNIVERSE ─────────────────────────────────────────────────────────
EQUITY_ASSETS = list(index_prices.keys())
SAFE_BONDS    = ['US_10Y', 'UK_10Y', 'AU_10Y', 'ZA_10Y']
FX            = ['AUDUSD', 'BRLUSD', 'EURUSD', 'GBPUSD', 'JPYUSD', 'ZARUSD']
COMMODITIES   = ['Oil', 'Gold', 'Bitcoin']
EXCLUDE_ASSETS = ['Bitcoin']

ALL_ASSETS = EQUITY_ASSETS + FX + COMMODITIES + SAFE_BONDS
ALL_ASSETS = [a for a in ALL_ASSETS
              if a in macro_df.columns and a not in EXCLUDE_ASSETS]

equity_assets     = [a for a in EQUITY_ASSETS     if a in ALL_ASSETS]
non_equity_assets = [a for a in ALL_ASSETS         if a not in equity_assets]

N           = len(ALL_ASSETS)
base_weight = 1.0 / N

print(f"\nTotal assets              : {N}")
print(f"Equity (signal-driven)    : {equity_assets}")
print(f"Non-equity (always long)  : {non_equity_assets}")
print(f"base_weight = 1/{N} = {base_weight:.4f}")

# ── 7. COUNTRY SAFE-HAVEN MAP ─────────────────────────────────────────────────
EQUITY_SAFE_HAVEN_MAP = {
    'ASX50':       ['AU_10Y', 'AUDUSD'],
    'EUROSTOXX50': ['EURUSD'],
    'FTSE100':     ['UK_10Y', 'GBPUSD'],
    'Ibovespa':    ['BRLUSD'],
    'JSE40':       ['ZA_10Y', 'ZARUSD'],
    'NIKKEI225':   ['JPYUSD'],
    'SP500':       ['US_10Y'],
}

UNIVERSE_SET = set(ALL_ASSETS)
for eq, havens in EQUITY_SAFE_HAVEN_MAP.items():
    EQUITY_SAFE_HAVEN_MAP[eq] = [h for h in havens if h in UNIVERSE_SET]

print("\nSafe-haven routing (filtered to universe):")
for eq, havens in EQUITY_SAFE_HAVEN_MAP.items():
    print(f"  {eq:15s} → {havens}")

# ── 8. BUILD RETURNS ──────────────────────────────────────────────────────────
returns_raw = macro_df[ALL_ASSETS].pct_change(fill_method=None)

for col in SAFE_BONDS:
    if col in returns_raw.columns:
        returns_raw[col] = -returns_raw[col]

if 'Oil' in macro_df.columns:
    bad_oil = macro_df['Oil'][macro_df['Oil'] <= 0].index
    for d in bad_oil:
        loc = returns_raw.index.get_loc(d)
        returns_raw.iloc[max(0, loc - 1): loc + 2,
                         returns_raw.columns.get_loc('Oil')] = 0.0

returns = returns_raw.clip(lower=-0.50, upper=1.00).fillna(0).iloc[1:]

assert not returns.isnull().any().any(), "NaNs in returns"
assert not np.isinf(returns.values).any(),  "Inf in returns"
print(f"\nReturns shape : {returns.shape}")
print("✓ Returns clean")

# ── 9. HELPER: REBALANCING DATE SET ──────────────────────────────────────────
def build_reb_set(returns, rebal_freq):
    actual_dates = returns.index
    if rebal_freq == 'daily':
        return set(actual_dates)
    elif rebal_freq == 'weekly':
        raw = returns.resample('W-MON').first().index
    elif rebal_freq == 'monthly':
        raw = returns.resample('MS').first().index
    elif rebal_freq == 'quarterly':
        raw = returns.resample('QS').first().index
    else:
        raise ValueError(f"Unknown rebal_freq: {rebal_freq}")

    reb_set = set()
    for d in raw:
        future = actual_dates[actual_dates >= d]
        if len(future) > 0:
            reb_set.add(future[0])
    return reb_set

# ── 10. BUY-AND-HOLD BENCHMARK ───────────────────────────────────────────────
def run_buy_and_hold(returns, all_assets, initial_capital=1e9):
    N           = len(all_assets)
    base_weight = 1.0 / N
    pv          = float(initial_capital)
    holdings    = {a: pv * base_weight for a in all_assets}
    records     = []
    first       = True

    for date in returns.index:
        if first:
            first = False
            records.append({'date': date, 'portfolio_value': pv})
            continue
        new_holdings = {a: holdings[a] * (1 + returns.loc[date, a])
                        for a in all_assets}
        holdings = new_holdings
        pv       = sum(holdings.values())
        records.append({'date': date, 'portfolio_value': pv})

    df = pd.DataFrame(records).set_index('date')
    print(f"  Buy-and-Hold | Start: ${initial_capital/1e9:.2f}B → "
          f"End: ${df['portfolio_value'].iloc[-1]/1e9:.3f}B")
    return df

# ── 11. WEIGHT BUILDERS ───────────────────────────────────────────────────────

# ── 11a. Original binary 3-stage weights ─────────────────────────────────────
def build_weights_3stage(sig_row, equity_assets, non_equity_assets,
                         base_weight, date, safe_haven_map):
    raw            = {a: base_weight for a in non_equity_assets}
    safe_haven_extra = {}

    for a in equity_assets:
        if sig_row is not None and a in sig_row.index and not pd.isna(sig_row[a]):
            sig = sig_row[a]
        else:
            sig = -1

        if sig > 0:
            raw[a] = -base_weight
            havens = [h for h in safe_haven_map.get(a, []) if h in raw]
            if havens:
                per_haven = base_weight / len(havens)
                for h in havens:
                    safe_haven_extra[h] = safe_haven_extra.get(h, 0) + per_haven
            else:
                per_ne = base_weight / len(non_equity_assets)
                for ne in non_equity_assets:
                    safe_haven_extra[ne] = safe_haven_extra.get(ne, 0) + per_ne
        else:
            raw[a] = base_weight

    for asset, extra in safe_haven_extra.items():
        raw[asset] = raw.get(asset, 0) + extra

    total = sum(raw.values())
    if total <= 0:
        for a in raw:
            raw[a] = 1.0 / len(raw)
    else:
        for a in raw:
            raw[a] /= total
    return raw

# ── 11b. Continuous scalar weights ───────────────────────────────────────────
def build_weights_continuous(sig_row, equity_assets, non_equity_assets,
                             base_weight, safe_haven_map):
    """
    sig_row contains a scalar in [min_weight, 1.0] per equity asset.
    Reduced equity weight is redistributed to safe havens.
    """
    raw              = {a: base_weight for a in non_equity_assets}
    safe_haven_extra = {}

    for a in equity_assets:
        scalar = float(sig_row[a]) if (sig_row is not None and
                                       a in sig_row.index and
                                       not pd.isna(sig_row[a])) else 1.0
        raw[a]   = base_weight * scalar
        released = base_weight * (1.0 - scalar)

        if released > 0:
            havens = [h for h in safe_haven_map.get(a, []) if h in raw]
            if havens:
                per_haven = released / len(havens)
                for h in havens:
                    safe_haven_extra[h] = safe_haven_extra.get(h, 0) + per_haven
            else:
                per_ne = released / len(non_equity_assets)
                for ne in non_equity_assets:
                    safe_haven_extra[ne] = safe_haven_extra.get(ne, 0) + per_ne

    for asset, extra in safe_haven_extra.items():
        raw[asset] = raw.get(asset, 0) + extra

    total = sum(raw.values())
    if total <= 0:
        for a in raw:
            raw[a] = 1.0 / len(raw)
    else:
        for a in raw:
            raw[a] /= total
    return raw

# ── 11c. Partial-hedge weights ────────────────────────────────────────────────
def build_weights_partial(sig_row, equity_assets, non_equity_assets,
                          base_weight, safe_haven_map, partial_keep=0.25):
    """
    On RISK_ON signal: keep partial_keep fraction of base_weight in equity,
    redirect the rest to safe havens.
    """
    raw              = {a: base_weight for a in non_equity_assets}
    safe_haven_extra = {}

    for a in equity_assets:
        if sig_row is not None and a in sig_row.index and not pd.isna(sig_row[a]):
            sig = sig_row[a]
        else:
            sig = -1

        if sig > 0:
            raw[a]   = base_weight * partial_keep
            released = base_weight * (1.0 - partial_keep)
            havens   = [h for h in safe_haven_map.get(a, []) if h in raw]
            if havens:
                per_haven = released / len(havens)
                for h in havens:
                    safe_haven_extra[h] = safe_haven_extra.get(h, 0) + per_haven
            else:
                per_ne = released / len(non_equity_assets)
                for ne in non_equity_assets:
                    safe_haven_extra[ne] = safe_haven_extra.get(ne, 0) + per_ne
        else:
            raw[a] = base_weight

    for asset, extra in safe_haven_extra.items():
        raw[asset] = raw.get(asset, 0) + extra

    total = sum(raw.values())
    if total <= 0:
        for a in raw:
            raw[a] = 1.0 / len(raw)
    else:
        for a in raw:
            raw[a] /= total
    return raw

# ── 12. EQUAL-WEIGHT REBALANCED BENCHMARK ────────────────────────────────────
def run_equal_weight(returns, all_assets, tc_rate=0.005,
                     rebal_freq='monthly', initial_capital=1e9):
    N           = len(all_assets)
    base_weight = 1.0 / N
    ew          = {a: base_weight for a in all_assets}
    reb_set     = build_reb_set(returns, rebal_freq)
    pv          = float(initial_capital)
    current_weights = None
    records     = []

    for date in returns.index:
        if date in reb_set:
            if current_weights is not None:
                turnover = sum(abs(ew.get(a, 0) - current_weights.get(a, 0))
                               for a in set(ew) | set(current_weights)) / 2.0
                pv *= (1 - turnover * tc_rate)
            else:
                pv *= (1 - tc_rate)
            current_weights = ew.copy()

        if current_weights is not None:
            dr  = sum(current_weights.get(a, 0) * returns.loc[date, a]
                      for a in all_assets)
            pv *= (1 + dr)

        records.append({'date': date, 'portfolio_value': pv})

    return pd.DataFrame(records).set_index('date')

# ── 13. GENERIC SIGNAL PORTFOLIO RUNNER ──────────────────────────────────────
def run_signal_portfolio(returns, signal_df, equity_assets,
                         non_equity_assets, all_assets,
                         weight_fn,          # callable: (sig_row, ...) → dict
                         freq='monthly', tc_rate=0.005,
                         initial_capital=1e9):
    """
    Generic runner.  weight_fn must accept (sig_row, equity_assets,
    non_equity_assets, base_weight, <any extra kwargs>) and return
    a dict {asset: weight}.
    """
    N           = len(all_assets)
    base_weight = 1.0 / N
    reb_set     = build_reb_set(returns, freq)

    sig = signal_df.copy()
    sig.index = pd.to_datetime(sig.index)

    pv              = float(initial_capital)
    current_weights = None
    records         = []

    for date in returns.index:
        if date in reb_set:
            past    = sig[sig.index <= date]
            sig_row = past.iloc[-1] if len(past) > 0 else None
            raw     = weight_fn(sig_row)

            if current_weights is not None:
                all_keys = set(raw) | set(current_weights)
                turnover = sum(abs(raw.get(a, 0) - current_weights.get(a, 0))
                               for a in all_keys) / 2.0
                pv *= (1 - turnover * tc_rate)
            else:
                pv *= (1 - tc_rate)

            current_weights = raw.copy()

        if current_weights is not None:
            dr  = sum(current_weights.get(a, 0) * returns.loc[date, a]
                      for a in all_assets)
            pv *= (1 + dr)

        records.append({'date': date, 'portfolio_value': pv})

    return pd.DataFrame(records).set_index('date')

# ── 14. MEAN-VARIANCE SIGNAL PORTFOLIO ───────────────────────────────────────
MV_LOOKBACK = 252
MV_MIN_OBS  = 60
MAX_WEIGHT  = 0.30
RISK_FREE   = 0.0

def mean_variance_weights(mu_vec, cov_mat, allowed_mask,
                          max_weight=MAX_WEIGHT, risk_free=RISK_FREE):
    n_total   = len(mu_vec)
    allowed   = np.where(allowed_mask)[0]
    n_allowed = len(allowed)

    fallback = np.zeros(n_total)
    if n_allowed == 0:
        fallback[:] = 1.0 / n_total
        return fallback
    fallback[allowed] = 1.0 / n_allowed

    mu_sub  = mu_vec[allowed]
    cov_sub = cov_mat[np.ix_(allowed, allowed)] + np.eye(n_allowed) * 1e-8

    def neg_sharpe(w):
        port_ret = np.dot(w, mu_sub)
        port_var = np.dot(w, cov_sub @ w)
        port_std = np.sqrt(max(port_var, 1e-12))
        return -(port_ret - risk_free) / port_std

    constraints = [{'type': 'eq', 'fun': lambda w: np.sum(w) - 1.0}]
    bounds      = [(0.0, max_weight)] * n_allowed
    w0          = np.ones(n_allowed) / n_allowed

    try:
        res = minimize(neg_sharpe, w0, method='SLSQP',
                       bounds=bounds, constraints=constraints,
                       options={'ftol': 1e-9, 'maxiter': 500})
        w_opt = res.x if (res.success and not np.any(np.isnan(res.x))) else w0
    except Exception:
        w_opt = w0

    w_full         = np.zeros(n_total)
    w_full[allowed] = w_opt
    return w_full

def run_mv_signal_portfolio(returns, signal_df, all_assets,
                            equity_assets,
                            freq='monthly', tc_rate=0.005,
                            lookback=MV_LOOKBACK, min_obs=MV_MIN_OBS,
                            max_weight=MAX_WEIGHT, initial_capital=1e9):
    reb_set      = build_reb_set(returns, freq)
    actual_dates = returns.index
    assets       = all_assets
    n            = len(assets)
    asset_idx    = {a: i for i, a in enumerate(assets)}

    sig = signal_df.copy()
    sig.index = pd.to_datetime(sig.index)

    pv              = float(initial_capital)
    current_weights = np.zeros(n)
    records         = []
    weight_history  = []
    first_reb       = True

    for t, date in enumerate(actual_dates):
        if date in reb_set:
            hist = returns.iloc[max(0, t - lookback): t]

            if len(hist) < min_obs:
                new_weights = np.ones(n) / n
            else:
                mu_ann  = hist.mean().values  * 252
                cov_ann = hist.cov().values   * 252

                past    = sig[sig.index <= date]
                sig_row = past.iloc[-1] if len(past) > 0 else None

                allowed_mask = np.ones(n, dtype=bool)
                if sig_row is not None:
                    for eq in equity_assets:
                        if eq in asset_idx and eq in sig_row.index:
                            s = sig_row[eq]
                            if not pd.isna(s) and s > 0:
                                allowed_mask[asset_idx[eq]] = False

                new_weights = mean_variance_weights(
                    mu_ann, cov_ann, allowed_mask, max_weight=max_weight)

            if first_reb:
                pv       *= (1 - tc_rate)
                first_reb = False
            else:
                turnover = np.sum(np.abs(new_weights - current_weights)) / 2.0
                pv      *= (1 - turnover * tc_rate)

            current_weights = new_weights.copy()
            weight_history.append({'date': date,
                                   **dict(zip(assets, current_weights))})

        if not first_reb:
            dr  = np.dot(current_weights, returns.loc[date].values)
            pv *= (1 + dr)

        records.append({'date': date, 'portfolio_value': pv})

    pv_df      = pd.DataFrame(records).set_index('date')
    weights_df = pd.DataFrame(weight_history).set_index('date')
    print(f"  MV | freq={freq} tc={tc_rate*100:.1f}% "
          f"End: ${pv_df['portfolio_value'].iloc[-1]/1e9:.3f}B")
    return pv_df, weights_df

# ── 15. RUN ALL PORTFOLIOS ────────────────────────────────────────────────────
TC_LEVELS = {'0%': 0.00, '0.5%': 0.005, '1%': 0.01, '2%': 0.02}
FREQS     = ['daily', 'weekly', 'monthly', 'quarterly']
MV_FREQS  = ['monthly', 'quarterly']

# ── 15a. Buy-and-Hold ─────────────────────────────────────────────────────────
print("\n── Running Buy-and-Hold ──")
bh_result  = run_buy_and_hold(returns, ALL_ASSETS, initial_capital=1e9)
bh_results = {'BuyAndHold_EW': bh_result}

# ── 15b. Equal-Weight ─────────────────────────────────────────────────────────
print("\n── Running Equal-Weight Rebalanced ──")
ew_results = {}
for freq in FREQS:
    for tc_label, tc_rate in TC_LEVELS.items():
        label = f"EW_{freq}_TC{tc_label}"
        ew_results[label] = run_equal_weight(
            returns, ALL_ASSETS, tc_rate=tc_rate,
            rebal_freq=freq, initial_capital=1e9)
        print(f"  ✓ {label}")

# ── 15c. Original 3-Stage Signal ─────────────────────────────────────────────
print("\n── Running Original 3-Stage Signal ──")
signal_results       = {}
signal_distributions = {}

for thresh in THRESHOLDS:
    sig_df_t    = build_signal_df(composite_score, equity_assets,
                                  macro_df.index, threshold=thresh)
    n_risk_on   = (sig_df_t[equity_assets[0]] == +1).sum()
    n_risk_off  = (sig_df_t[equity_assets[0]] == -1).sum()
    total       = len(sig_df_t)
    signal_distributions[thresh] = {'RISK_ON': n_risk_on, 'RISK_OFF': n_risk_off}
    print(f"  Threshold={thresh} | RISK_ON: {n_risk_on} ({n_risk_on/total*100:.1f}%) "
          f"| RISK_OFF: {n_risk_off} ({n_risk_off/total*100:.1f}%)")

    for freq in FREQS:
        for tc_label, tc_rate in TC_LEVELS.items():
            label = f"Sig3S_t{thresh}_{freq}_TC{tc_label}"

            def wfn_3s(sig_row, _eq=equity_assets, _ne=non_equity_assets,
                       _bw=base_weight, _sh=EQUITY_SAFE_HAVEN_MAP):
                return build_weights_3stage(sig_row, _eq, _ne, _bw, None, _sh)

            signal_results[label] = run_signal_portfolio(
                returns, sig_df_t, equity_assets, non_equity_assets,
                ALL_ASSETS, weight_fn=wfn_3s,
                freq=freq, tc_rate=tc_rate, initial_capital=1e9)
            print(f"    ✓ {label}")

# ── 15d. Asymmetric Signal ────────────────────────────────────────────────────
print("\n── Running Asymmetric Signal ──")
asym_results = {}

for thresh in THRESHOLDS:
    sig_df_t = build_signal_df_asymmetric(composite_score, equity_assets,
                                          macro_df.index, threshold=thresh)
    for freq in ['monthly', 'quarterly']:
        for tc_label, tc_rate in TC_LEVELS.items():
            label = f"Asym_t{thresh}_{freq}_TC{tc_label}"

            def wfn_asym(sig_row, _eq=equity_assets, _ne=non_equity_assets,
                         _bw=base_weight, _sh=EQUITY_SAFE_HAVEN_MAP):
                return build_weights_3stage(sig_row, _eq, _ne, _bw, None, _sh)

            asym_results[label] = run_signal_portfolio(
                returns, sig_df_t, equity_assets, non_equity_assets,
                ALL_ASSETS, weight_fn=wfn_asym,
                freq=freq, tc_rate=tc_rate, initial_capital=1e9)
            print(f"    ✓ {label}")

# ── 15e. Continuous / Graduated Signal ───────────────────────────────────────
print("\n── Running Continuous Signal ──")
cont_results = {}

for thresh in THRESHOLDS:
    for max_red in MAX_REDUCES:
        sig_df_t = build_signal_df_continuous(composite_score, equity_assets,
                                              macro_df.index,
                                              threshold=thresh,
                                              max_reduce=max_red)
        for freq in ['monthly', 'quarterly']:
            for tc_label, tc_rate in TC_LEVELS.items():
                label = (f"Cont_t{thresh}_mr{int(max_red*100)}"
                         f"_{freq}_TC{tc_label}")

                def wfn_cont(sig_row, _eq=equity_assets, _ne=non_equity_assets,
                             _bw=base_weight, _sh=EQUITY_SAFE_HAVEN_MAP):
                    return build_weights_continuous(sig_row, _eq, _ne, _bw, _sh)

                cont_results[label] = run_signal_portfolio(
                    returns, sig_df_t, equity_assets, non_equity_assets,
                    ALL_ASSETS, weight_fn=wfn_cont,
                    freq=freq, tc_rate=tc_rate, initial_capital=1e9)
                print(f"    ✓ {label}")

# ── 15f. Smoothed Signal ─────────────────────────────────────────────────────
print("\n── Running Smoothed Signal ──")
smooth_results = {}

for thresh in THRESHOLDS:
    for sw in SMOOTH_WINDOWS:
        sig_df_t = build_signal_df_smoothed(composite_score, equity_assets,
                                            macro_df.index,
                                            threshold=thresh,
                                            smooth_window=sw)
        for freq in ['monthly', 'quarterly']:
            for tc_label, tc_rate in TC_LEVELS.items():
                label = f"Smooth_t{thresh}_sw{sw}_{freq}_TC{tc_label}"

                def wfn_smooth(sig_row, _eq=equity_assets, _ne=non_equity_assets,
                               _bw=base_weight, _sh=EQUITY_SAFE_HAVEN_MAP):
                    return build_weights_3stage(sig_row, _eq, _ne, _bw, None, _sh)

                smooth_results[label] = run_signal_portfolio(
                    returns, sig_df_t, equity_assets, non_equity_assets,
                    ALL_ASSETS, weight_fn=wfn_smooth,
                    freq=freq, tc_rate=tc_rate, initial_capital=1e9)
                print(f"    ✓ {label}")

# ── 15g. Persistent Signal ────────────────────────────────────────────────────
print("\n── Running Persistent Signal ──")
persist_results = {}

for thresh in THRESHOLDS:
    for min_days in MIN_DAYS_LIST:
        sig_df_t = build_signal_df_persistent(composite_score, equity_assets,
                                              macro_df.index,
                                              threshold=thresh,
                                              min_days_to_switch=min_days)
        for freq in ['monthly', 'quarterly']:
            for tc_label, tc_rate in TC_LEVELS.items():
                label = f"Persist_t{thresh}_md{min_days}_{freq}_TC{tc_label}"

                def wfn_persist(sig_row, _eq=equity_assets, _ne=non_equity_assets,
                                _bw=base_weight, _sh=EQUITY_SAFE_HAVEN_MAP):
                    return build_weights_3stage(sig_row, _eq, _ne, _bw, None, _sh)

                persist_results[label] = run_signal_portfolio(
                    returns, sig_df_t, equity_assets, non_equity_assets,
                    ALL_ASSETS, weight_fn=wfn_persist,
                    freq=freq, tc_rate=tc_rate, initial_capital=1e9)
                print(f"    ✓ {label}")

# ── 15h. Partial-Hedge Signal ─────────────────────────────────────────────────
print("\n── Running Partial-Hedge Signal ──")
partial_results = {}

for thresh in THRESHOLDS:
    sig_df_t = build_signal_df(composite_score, equity_assets,
                               macro_df.index, threshold=thresh)
    for pk in PARTIAL_KEEPS:
        for freq in ['monthly', 'quarterly']:
            for tc_label, tc_rate in TC_LEVELS.items():
                label = (f"Partial_t{thresh}_pk{int(pk*100)}"
                         f"_{freq}_TC{tc_label}")

                def wfn_partial(sig_row, _eq=equity_assets, _ne=non_equity_assets,
                                _bw=base_weight, _sh=EQUITY_SAFE_HAVEN_MAP,
                                _pk=pk):
                    return build_weights_partial(sig_row, _eq, _ne, _bw, _sh, _pk)

                partial_results[label] = run_signal_portfolio(
                    returns, sig_df_t, equity_assets, non_equity_assets,
                    ALL_ASSETS, weight_fn=wfn_partial,
                    freq=freq, tc_rate=tc_rate, initial_capital=1e9)
                print(f"    ✓ {label}")

# ── 15i. Mean-Variance Signal ─────────────────────────────────────────────────
print("\n── Running Mean-Variance Signal ──")
mv_results        = {}
mv_weight_history = {}

for thresh in THRESHOLDS:
    sig_df_t = build_signal_df(composite_score, equity_assets,
                               macro_df.index, threshold=thresh)
    for freq in MV_FREQS:
        for tc_label, tc_rate in TC_LEVELS.items():
            label = f"MV_t{thresh}_{freq}_TC{tc_label}"
            pv_df, w_df = run_mv_signal_portfolio(
                returns, sig_df_t, ALL_ASSETS,
                equity_assets=equity_assets,
                freq=freq, tc_rate=tc_rate,
                lookback=MV_LOOKBACK, min_obs=MV_MIN_OBS,
                max_weight=MAX_WEIGHT, initial_capital=1e9)
            mv_results[label]        = pv_df
            mv_weight_history[label] = w_df
            print(f"  ✓ {label}")

print("\n✓ All portfolios computed")

# ── 16. PERFORMANCE METRICS ───────────────────────────────────────────────────
def compute_metrics(pv_df, label=''):
    pv        = pv_df['portfolio_value']
    daily_ret = pv.pct_change().dropna()
    n_years   = len(daily_ret) / 252
    total_ret = (pv.iloc[-1] / pv.iloc[0]) - 1
    cagr      = (1 + total_ret) ** (1 / n_years) - 1 if n_years > 0 else np.nan
    vol       = daily_ret.std() * np.sqrt(252)
    sharpe    = cagr / vol if vol > 0 else np.nan
    roll_max  = pv.cummax()
    max_dd    = ((pv - roll_max) / roll_max).min()
    calmar    = cagr / abs(max_dd) if max_dd != 0 else np.nan
    var_95    = daily_ret.quantile(0.05)
    ann_ret   = cagr
    ann_vol   = vol

    return {
        'CAGR':         f"{cagr*100:.2f}%",
        'Ann_Ret':      ann_ret,
        'Ann_Vol':      ann_vol,
        'Volatility':   f"{vol*100:.2f}%",
        'Sharpe':       f"{sharpe:.2f}",
        'Max Drawdown': f"{max_dd*100:.2f}%",
        'Calmar':       f"{calmar:.2f}",
        'VaR 95%':      f"{var_95*100:.2f}%",
        'Total Return': f"{total_ret*100:.2f}%",
        'Final ($B)':   f"{pv.iloc[-1]/1e9:.3f}",
    }

bh_metrics      = {k: compute_metrics(v, k) for k, v in bh_results.items()}
ew_metrics      = {k: compute_metrics(v, k) for k, v in ew_results.items()}
sig_metrics     = {k: compute_metrics(v, k) for k, v in signal_results.items()}
asym_metrics    = {k: compute_metrics(v, k) for k, v in asym_results.items()}
cont_metrics    = {k: compute_metrics(v, k) for k, v in cont_results.items()}
smooth_metrics  = {k: compute_metrics(v, k) for k, v in smooth_results.items()}
persist_metrics = {k: compute_metrics(v, k) for k, v in persist_results.items()}
partial_metrics = {k: compute_metrics(v, k) for k, v in partial_results.items()}
mv_metrics      = {k: compute_metrics(v, k) for k, v in mv_results.items()}

def print_metrics_table(metrics_dict, title):
    display = {k: {mk: mv for mk, mv in m.items()
                   if mk not in ('Ann_Ret', 'Ann_Vol')}
               for k, m in metrics_dict.items()}
    print(f"\n{'='*90}")
    print(title)
    print("="*90)
    print(pd.DataFrame(display).T.to_string())

print_metrics_table(bh_metrics,      "BUY-AND-HOLD")
print_metrics_table(ew_metrics,      "EQUAL-WEIGHT REBALANCED")

for thresh in THRESHOLDS:
    sub = {k: v for k, v in sig_metrics.items()     if f"_t{thresh}_" in k}
    print_metrics_table(sub, f"3-STAGE SIGNAL — threshold={thresh}")

for thresh in THRESHOLDS:
    sub = {k: v for k, v in asym_metrics.items()    if f"_t{thresh}_" in k}
    print_metrics_table(sub, f"ASYMMETRIC SIGNAL — threshold={thresh}")

for thresh in THRESHOLDS:
    sub = {k: v for k, v in cont_metrics.items()    if f"_t{thresh}_" in k}
    print_metrics_table(sub, f"CONTINUOUS SIGNAL — threshold={thresh}")

for thresh in THRESHOLDS:
    sub = {k: v for k, v in smooth_metrics.items()  if f"_t{thresh}_" in k}
    print_metrics_table(sub, f"SMOOTHED SIGNAL — threshold={thresh}")

for thresh in THRESHOLDS:
    sub = {k: v for k, v in persist_metrics.items() if f"_t{thresh}_" in k}
    print_metrics_table(sub, f"PERSISTENT SIGNAL — threshold={thresh}")

for thresh in THRESHOLDS:
    sub = {k: v for k, v in partial_metrics.items() if f"_t{thresh}_" in k}
    print_metrics_table(sub, f"PARTIAL-HEDGE SIGNAL — threshold={thresh}")

for thresh in THRESHOLDS:
    sub = {k: v for k, v in mv_metrics.items()      if f"_t{thresh}_" in k}
    print_metrics_table(sub, f"MEAN-VARIANCE SIGNAL — threshold={thresh}")

# ── 17. PLOTS ─────────────────────────────────────────────────────────────────
bh_pv     = bh_results['BuyAndHold_EW']['portfolio_value']
ew_ref_pv = ew_results['EW_monthly_TC0.5%']['portfolio_value']

def parse_pct(s):
    return float(str(s).replace('%', '')) / 100

# ── Plot A: Best of each variant vs benchmarks (monthly, TC=0.5%) ─────────────
fig, axes = plt.subplots(2, 4, figsize=(28, 12), sharey=False)
fig.suptitle("All Signal Variants vs Benchmarks — Monthly Rebalancing, TC=0.5%",
             fontsize=14, fontweight='bold')

variant_info = [
    ("3-Stage",    signal_results,  THRESHOLDS,     "Sig3S",   "steelblue"),
    ("Asymmetric", asym_results,    THRESHOLDS,     "Asym",    "darkorange"),
    ("Continuous", cont_results,    THRESHOLDS,     "Cont",    "purple"),
    ("Smoothed",   smooth_results,  THRESHOLDS,     "Smooth",  "green"),
    ("Persistent", persist_results, THRESHOLDS,     "Persist", "crimson"),
    ("Partial 25%",partial_results, THRESHOLDS,     "Partial", "brown"),
    ("MV Signal",  mv_results,      THRESHOLDS,     "MV",      "black"),
]

for ax, (name, res_dict, threshs, prefix, col) in zip(axes.flat, variant_info):
    ax.plot(bh_pv.index,     bh_pv / 1e9,     color='forestgreen',
            lw=1.5, ls='-.', label='Buy & Hold')
    ax.plot(ew_ref_pv.index, ew_ref_pv / 1e9, color='grey',
            lw=1.2, ls='--', label='EW Monthly')

    for thresh in threshs:
        candidates = [k for k in res_dict
                      if f"_t{thresh}_" in k and 'monthly' in k and 'TC0.5%' in k]
        if not candidates:
            candidates = [k for k in res_dict
                          if f"_t{thresh}_" in k and 'TC0.5%' in k]
        if candidates:
            key = candidates[0]
            pv  = res_dict[key]['portfolio_value']
            ax.plot(pv.index, pv / 1e9, lw=1.6,
                    label=f"t={thresh}")

    ax.axhline(1.0, color='black', ls=':', lw=0.7)
    ax.set_title(name, fontsize=11, fontweight='bold')
    ax.set_ylabel("Value ($B)")
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

axes.flat[-1].set_visible(False)
plt.tight_layout()
plt.savefig("all_variants_monthly.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Plot B: Risk-Return Scatter — all strategies ──────────────────────────────
fig, ax = plt.subplots(figsize=(16, 10))

all_metric_groups = [
    (bh_metrics,      'BH',       'forestgreen', 'D'),
    (ew_metrics,      'EW',       'grey',        's'),
    (sig_metrics,     '3-Stage',  'steelblue',   'o'),
    (asym_metrics,    'Asym',     'darkorange',  'o'),
    (cont_metrics,    'Cont',     'purple',      'o'),
    (smooth_metrics,  'Smooth',   'green',       'o'),
    (persist_metrics, 'Persist',  'crimson',     'o'),
    (partial_metrics, 'Partial',  'brown',       'o'),
    (mv_metrics,      'MV',       'black',       '^'),
]

for metrics_dict, grp_label, col, marker in all_metric_groups:
    first = True
    for key, m in metrics_dict.items():
        vol = m['Ann_Vol'] * 100
        ret = m['Ann_Ret'] * 100
        ax.scatter(vol, ret, color=col, marker=marker, s=80,
                   label=grp_label if first else None,
                   edgecolors='black', lw=0.5, zorder=4)
        ax.annotate(key.split('_TC')[0], (vol, ret),
                    textcoords='offset points', xytext=(4, 3),
                    fontsize=6, color=col)
        first = False

ax.axhline(0, color='black', lw=0.6, ls='--')
ax.axvline(0, color='black', lw=0.6, ls='--')
ax.set_xlabel("Annualised Volatility (%)", fontsize=12)
ax.set_ylabel("CAGR (%)", fontsize=12)
ax.set_title("Risk-Return — All Strategies", fontsize=13, fontweight='bold')
ax.legend(fontsize=10, ncol=3)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("risk_return_all.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Plot C: Sharpe Bar Chart — top 20 strategies ──────────────────────────────
all_metrics_combined = {
    **bh_metrics, **ew_metrics, **sig_metrics,
    **asym_metrics, **cont_metrics, **smooth_metrics,
    **persist_metrics, **partial_metrics, **mv_metrics
}

sharpe_series = pd.Series({
    k: float(v['Sharpe']) for k, v in all_metrics_combined.items()
}).dropna().sort_values(ascending=False)

top20 = sharpe_series.head(20)

def label_color(lbl):
    if lbl.startswith('MV'):       return 'black'
    if lbl.startswith('Sig3S'):    return 'steelblue'
    if lbl.startswith('Asym'):     return 'darkorange'
    if lbl.startswith('Cont'):     return 'purple'
    if lbl.startswith('Smooth'):   return 'green'
    if lbl.startswith('Persist'):  return 'crimson'
    if lbl.startswith('Partial'):  return 'brown'
    if lbl.startswith('EW'):       return 'grey'
    return 'forestgreen'

colors = [label_color(k) for k in top20.index]

fig, ax = plt.subplots(figsize=(16, 7))
bars = ax.barh(range(len(top20)), top20.values, color=colors, edgecolor='black', lw=0.5)
ax.set_yticks(range(len(top20)))
ax.set_yticklabels([k.replace('_TC', ' TC=') for k in top20.index], fontsize=8)
ax.set_xlabel("Sharpe Ratio")
ax.set_title("Top 20 Strategies by Sharpe Ratio", fontsize=13, fontweight='bold')
ax.axvline(0, color='black', lw=0.8)
ax.grid(True, axis='x', alpha=0.3)

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='black',      label='MV Signal'),
    Patch(facecolor='steelblue',  label='3-Stage'),
    Patch(facecolor='darkorange', label='Asymmetric'),
    Patch(facecolor='purple',     label='Continuous'),
    Patch(facecolor='green',      label='Smoothed'),
    Patch(facecolor='crimson',    label='Persistent'),
    Patch(facecolor='brown',      label='Partial Hedge'),
    Patch(facecolor='grey',       label='Equal Weight'),
    Patch(facecolor='forestgreen',label='Buy & Hold'),
]
ax.legend(handles=legend_elements, fontsize=9, loc='lower right')
plt.tight_layout()
plt.savefig("sharpe_top20.png", dpi=150, bbox_inches='tight')
plt.show()

# ── 18. SAVE ALL ──────────────────────────────────────────────────────────────
print("\n── Saving all results ──")
all_portfolios = {
    **bh_results, **ew_results, **signal_results,
    **asym_results, **cont_results, **smooth_results,
    **persist_results, **partial_results, **mv_results
}

for label, df in all_portfolios.items():
    df.to_csv(f"{label}_portfolio.csv")

all_metrics_df = pd.DataFrame({
    k: {mk: mv for mk, mv in m.items() if mk not in ('Ann_Ret', 'Ann_Vol')}
    for k, m in all_metrics_combined.items()
}).T
all_metrics_df.to_csv("all_portfolio_metrics.csv")

composite_score.to_csv("composite_signal.csv")
for thresh in THRESHOLDS:
    sig_df_t = build_signal_df(composite_score, equity_assets,
                               macro_df.index, threshold=thresh)
    sig_df_t.to_csv(f"signal_df_threshold_{thresh}.csv")

for label, wdf in mv_weight_history.items():
    wdf.to_csv(f"{label}_weights.csv")

print("✓ All files saved.")
print(f"\n  Portfolio CSVs   : {len(all_portfolios)}")
print(f"  Weight CSVs      : {len(mv_weight_history)}")
print(f"  Metrics CSV      : all_portfolio_metrics.csv")
print(f"  Signal CSVs      : {len(THRESHOLDS)} threshold files")
print(f"  Plots saved      : 3 PNG files")


# Truncated Time Series 2015-2021


In [ ]:
# @title
# ============================================================
# SIGNAL PORTFOLIO — ENHANCED VERSION
# Signals generated from Modified Determinant Model
# + Buy-and-Hold Benchmark
# + Equal-Weight Benchmark
# + Three-Stage Signal Portfolio (original)
# + Enhanced Signal Variants (asymmetric, continuous, smoothed, persistent, partial)
# + Mean-Variance Signal Portfolio
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import requests
from io import StringIO
from scipy.optimize import minimize

# ── 1. LOAD ALL DATA FROM GITHUB ─────────────────────────────────────────────
#GITHUB_BASE = "https://raw.githubusercontent.com/kboroz/MSFE_Capstone_Project/main/01_Data/Streamlined/"

GITHUB_BASE = "https://raw.githubusercontent.com/kboroz/MSFE_Capstone_Project/main/01_Data/2015-2021/"

INDEX_FILE_MAP = {
    'ASX50':       'first_period_asx_50.csv',
    'EUROSTOXX50': 'first_period_euro_stoxx_50.csv',
    'FTSE100':     'first_period_ftse_100.csv',
    'Ibovespa':    'first_period_ibovespa.csv',
    'JSE40':       'first_period_jse_top_40.csv',
    'NIKKEI225':   'first_period_nikkei_225.csv',
    'SP500':       'first_period_s&p_500.csv',
}






MACRO_FILE = 'first_period_macro.csv'

def load_csv(url):
    r = requests.get(url)
    r.raise_for_status()
    df = pd.read_csv(StringIO(r.text), index_col=0, parse_dates=True)
    df.columns = df.columns.str.replace('[^a-zA-Z0-9]', '_', regex=True)
    return df.sort_index()

macro_df = load_csv(GITHUB_BASE + MACRO_FILE)
macro_df = macro_df[~macro_df.index.duplicated(keep='first')].sort_index()
print(f"macro_df shape : {macro_df.shape}")
print(f"Columns        : {macro_df.columns.tolist()}")

index_data = {}
for name, fname in INDEX_FILE_MAP.items():
    try:
        df = load_csv(GITHUB_BASE + fname)
        index_data[name] = df
        print(f"✓ {name:15s} | shape={df.shape} | cols={df.columns.tolist()[:4]}")
    except Exception as e:
        print(f"✗ {name}: {e}")

# ── 2. EXTRACT PRICE SERIES ───────────────────────────────────────────────────
PRICE_COLS = ['Close', 'close', 'Price', 'price', 'Adj_Close', 'Last']

def get_price_series(df, name):
    col = next((c for c in PRICE_COLS if c in df.columns), df.columns[0])
    s = pd.to_numeric(df[col], errors='coerce').dropna()
    print(f"  {name}: using column '{col}'")
    return s

index_prices = {}
for name, df in index_data.items():
    index_prices[name] = get_price_series(df, name)

# ── 3. MODIFIED DETERMINANT SIGNAL MODEL ─────────────────────────────────────
DET_WINDOWS = [3, 7, 28]
WINDOW_WEIGHTS = {3: 0.25, 7: 0.25, 28: 0.50}

def compute_modified_determinant(price_series_dict, windows=DET_WINDOWS):
    prices = pd.DataFrame(price_series_dict).sort_index()
    rets   = prices.pct_change().dropna()
    det_results = {}

    for window in windows:
        det_series = []
        for i in range(window, len(rets)):
            window_rets = rets.iloc[i - window:i]
            valid_cols  = window_rets.columns[window_rets.std() > 1e-10]
            w           = window_rets[valid_cols]

            if len(valid_cols) < 2:
                det_series.append(np.nan)
                continue

            try:
                R   = w.corr().values
                I   = np.eye(len(R))
                M   = R - I
                det = np.linalg.det(M)
            except Exception:
                det = np.nan

            det_series.append(det)

        idx = rets.index[window:]
        det_results[window] = pd.Series(det_series, index=idx,
                                        name=f'det_{window}d')
        print(f"  Window {window:2d}d | computed {len(det_series)} observations")

    return det_results

print("\n── Computing Modified Determinant ──")
det_results = compute_modified_determinant(index_prices)

# ── 4. COMPOSITE SCORE ────────────────────────────────────────────────────────
def compute_composite_score(det_results, window_weights=WINDOW_WEIGHTS):
    all_series = []
    for window, series in det_results.items():
        mu    = series.rolling(252, min_periods=60).mean()
        sigma = series.rolling(252, min_periods=60).std()
        z     = (series - mu) / sigma.replace(0, np.nan)
        z.name = f'z_{window}d'
        all_series.append(z)

    z_df      = pd.concat(all_series, axis=1).dropna()
    composite = sum(
        z_df[f'z_{w}d'] * window_weights[w]
        for w in window_weights if f'z_{w}d' in z_df.columns
    )
    composite = composite / sum(window_weights.values())
    return composite, z_df

composite_score, z_df = compute_composite_score(det_results)

print(f"\nComposite score shape  : {composite_score.shape}")
print(f"Date range             : {composite_score.index[0].date()} → "
      f"{composite_score.index[-1].date()}")
print(f"Score range            : {composite_score.min():.3f} → "
      f"{composite_score.max():.3f}")

# ── 5. SIGNAL BUILDERS ───────────────────────────────────────────────────────
THRESHOLDS    = [0.3, 0.5, 0.7]
SMOOTH_WINDOWS = [3, 5, 10]
MIN_DAYS_LIST  = [5, 10, 20]
MAX_REDUCES    = [0.5, 0.7, 0.9]
PARTIAL_KEEPS  = [0.25, 0.50]

# ── 5a. Original symmetric binary ────────────────────────────────────────────
def classify_signal_symmetric(score, threshold):
    return +1 if abs(score) > threshold else -1

def build_signal_df(composite_score, equity_assets, macro_dates, threshold=0.5):
    binary    = composite_score.apply(lambda x: classify_signal_symmetric(x, threshold))
    signal_df = pd.DataFrame(index=macro_dates)
    for asset in equity_assets:
        signal_df[asset] = binary.reindex(macro_dates).ffill().bfill()
    return signal_df

# ── 5b. Asymmetric (upper-tail only) ─────────────────────────────────────────
def build_signal_df_asymmetric(composite_score, equity_assets, macro_dates,
                               threshold=1.0):
    """Block equities only when correlation SPIKES (high positive z-score)."""
    binary    = composite_score.apply(lambda x: +1 if x > threshold else -1)
    signal_df = pd.DataFrame(index=macro_dates)
    for asset in equity_assets:
        signal_df[asset] = binary.reindex(macro_dates).ffill().bfill()
    return signal_df

# ── 5c. Continuous / graduated scalar ────────────────────────────────────────
def build_signal_df_continuous(composite_score, equity_assets, macro_dates,
                               threshold=1.0, max_reduce=0.8):
    """
    Returns a scalar in [1-max_reduce, 1.0] per asset per day.
    Used by build_weights_continuous() instead of the binary flag.
    """
    def score_to_scalar(s):
        if s <= 0:
            return 1.0
        elif s >= threshold:
            return 1.0 - max_reduce
        else:
            return 1.0 - max_reduce * (s / threshold)

    scalars   = composite_score.apply(score_to_scalar)
    signal_df = pd.DataFrame(index=macro_dates)
    for asset in equity_assets:
        signal_df[asset] = scalars.reindex(macro_dates).ffill().bfill()
    return signal_df

# ── 5d. Smoothed binary ───────────────────────────────────────────────────────
def build_signal_df_smoothed(composite_score, equity_assets, macro_dates,
                             threshold=1.0, smooth_window=5):
    """Smooth z-score before thresholding to reduce whipsaw."""
    smoothed  = composite_score.rolling(smooth_window, min_periods=1).mean()
    binary    = smoothed.apply(lambda x: classify_signal_symmetric(x, threshold))
    signal_df = pd.DataFrame(index=macro_dates)
    for asset in equity_assets:
        signal_df[asset] = binary.reindex(macro_dates).ffill().bfill()
    return signal_df

# ── 5e. Persistent binary ────────────────────────────────────────────────────
def build_signal_df_persistent(composite_score, equity_assets, macro_dates,
                               threshold=1.0, min_days_to_switch=10):
    """Only switch regime after signal holds for min_days_to_switch days."""
    raw       = composite_score.apply(lambda x: +1 if x > threshold else -1)
    confirmed = raw.copy()
    current   = int(raw.iloc[0])
    count     = 0

    for i in range(1, len(raw)):
        if int(raw.iloc[i]) != current:
            count += 1
            if count >= min_days_to_switch:
                current = int(raw.iloc[i])
                count   = 0
        else:
            count = 0
        confirmed.iloc[i] = current

    signal_df = pd.DataFrame(index=macro_dates)
    for asset in equity_assets:
        signal_df[asset] = confirmed.reindex(macro_dates).ffill().bfill()
    return signal_df

# ── 6. ASSET UNIVERSE ─────────────────────────────────────────────────────────
EQUITY_ASSETS = list(index_prices.keys())
SAFE_BONDS    = ['US_10Y', 'UK_10Y', 'AU_10Y', 'ZA_10Y']
FX            = ['AUDUSD', 'BRLUSD', 'EURUSD', 'GBPUSD', 'JPYUSD', 'ZARUSD']
COMMODITIES   = ['Oil', 'Gold', 'Bitcoin']
EXCLUDE_ASSETS = ['Bitcoin']

ALL_ASSETS = EQUITY_ASSETS + FX + COMMODITIES + SAFE_BONDS
ALL_ASSETS = [a for a in ALL_ASSETS
              if a in macro_df.columns and a not in EXCLUDE_ASSETS]

equity_assets     = [a for a in EQUITY_ASSETS     if a in ALL_ASSETS]
non_equity_assets = [a for a in ALL_ASSETS         if a not in equity_assets]

N           = len(ALL_ASSETS)
base_weight = 1.0 / N

print(f"\nTotal assets              : {N}")
print(f"Equity (signal-driven)    : {equity_assets}")
print(f"Non-equity (always long)  : {non_equity_assets}")
print(f"base_weight = 1/{N} = {base_weight:.4f}")

# ── 7. COUNTRY SAFE-HAVEN MAP ─────────────────────────────────────────────────
EQUITY_SAFE_HAVEN_MAP = {
    'ASX50':       ['AU_10Y', 'AUDUSD'],
    'EUROSTOXX50': ['EURUSD'],
    'FTSE100':     ['UK_10Y', 'GBPUSD'],
    'Ibovespa':    ['BRLUSD'],
    'JSE40':       ['ZA_10Y', 'ZARUSD'],
    'NIKKEI225':   ['JPYUSD'],
    'SP500':       ['US_10Y'],
}

UNIVERSE_SET = set(ALL_ASSETS)
for eq, havens in EQUITY_SAFE_HAVEN_MAP.items():
    EQUITY_SAFE_HAVEN_MAP[eq] = [h for h in havens if h in UNIVERSE_SET]

print("\nSafe-haven routing (filtered to universe):")
for eq, havens in EQUITY_SAFE_HAVEN_MAP.items():
    print(f"  {eq:15s} → {havens}")

# ── 8. BUILD RETURNS ──────────────────────────────────────────────────────────
returns_raw = macro_df[ALL_ASSETS].pct_change(fill_method=None)

for col in SAFE_BONDS:
    if col in returns_raw.columns:
        returns_raw[col] = -returns_raw[col]

if 'Oil' in macro_df.columns:
    bad_oil = macro_df['Oil'][macro_df['Oil'] <= 0].index
    for d in bad_oil:
        loc = returns_raw.index.get_loc(d)
        returns_raw.iloc[max(0, loc - 1): loc + 2,
                         returns_raw.columns.get_loc('Oil')] = 0.0

returns = returns_raw.clip(lower=-0.50, upper=1.00).fillna(0).iloc[1:]

assert not returns.isnull().any().any(), "NaNs in returns"
assert not np.isinf(returns.values).any(),  "Inf in returns"
print(f"\nReturns shape : {returns.shape}")
print("✓ Returns clean")

# ── 9. HELPER: REBALANCING DATE SET ──────────────────────────────────────────
def build_reb_set(returns, rebal_freq):
    actual_dates = returns.index
    if rebal_freq == 'daily':
        return set(actual_dates)
    elif rebal_freq == 'weekly':
        raw = returns.resample('W-MON').first().index
    elif rebal_freq == 'monthly':
        raw = returns.resample('MS').first().index
    elif rebal_freq == 'quarterly':
        raw = returns.resample('QS').first().index
    else:
        raise ValueError(f"Unknown rebal_freq: {rebal_freq}")

    reb_set = set()
    for d in raw:
        future = actual_dates[actual_dates >= d]
        if len(future) > 0:
            reb_set.add(future[0])
    return reb_set

# ── 10. BUY-AND-HOLD BENCHMARK ───────────────────────────────────────────────
def run_buy_and_hold(returns, all_assets, initial_capital=1e9):
    N           = len(all_assets)
    base_weight = 1.0 / N
    pv          = float(initial_capital)
    holdings    = {a: pv * base_weight for a in all_assets}
    records     = []
    first       = True

    for date in returns.index:
        if first:
            first = False
            records.append({'date': date, 'portfolio_value': pv})
            continue
        new_holdings = {a: holdings[a] * (1 + returns.loc[date, a])
                        for a in all_assets}
        holdings = new_holdings
        pv       = sum(holdings.values())
        records.append({'date': date, 'portfolio_value': pv})

    df = pd.DataFrame(records).set_index('date')
    print(f"  Buy-and-Hold | Start: ${initial_capital/1e9:.2f}B → "
          f"End: ${df['portfolio_value'].iloc[-1]/1e9:.3f}B")
    return df

# ── 11. WEIGHT BUILDERS ───────────────────────────────────────────────────────

# ── 11a. Original binary 3-stage weights ─────────────────────────────────────
def build_weights_3stage(sig_row, equity_assets, non_equity_assets,
                         base_weight, date, safe_haven_map):
    raw            = {a: base_weight for a in non_equity_assets}
    safe_haven_extra = {}

    for a in equity_assets:
        if sig_row is not None and a in sig_row.index and not pd.isna(sig_row[a]):
            sig = sig_row[a]
        else:
            sig = -1

        if sig > 0:
            raw[a] = -base_weight
            havens = [h for h in safe_haven_map.get(a, []) if h in raw]
            if havens:
                per_haven = base_weight / len(havens)
                for h in havens:
                    safe_haven_extra[h] = safe_haven_extra.get(h, 0) + per_haven
            else:
                per_ne = base_weight / len(non_equity_assets)
                for ne in non_equity_assets:
                    safe_haven_extra[ne] = safe_haven_extra.get(ne, 0) + per_ne
        else:
            raw[a] = base_weight

    for asset, extra in safe_haven_extra.items():
        raw[asset] = raw.get(asset, 0) + extra

    total = sum(raw.values())
    if total <= 0:
        for a in raw:
            raw[a] = 1.0 / len(raw)
    else:
        for a in raw:
            raw[a] /= total
    return raw

# ── 11b. Continuous scalar weights ───────────────────────────────────────────
def build_weights_continuous(sig_row, equity_assets, non_equity_assets,
                             base_weight, safe_haven_map):
    """
    sig_row contains a scalar in [min_weight, 1.0] per equity asset.
    Reduced equity weight is redistributed to safe havens.
    """
    raw              = {a: base_weight for a in non_equity_assets}
    safe_haven_extra = {}

    for a in equity_assets:
        scalar = float(sig_row[a]) if (sig_row is not None and
                                       a in sig_row.index and
                                       not pd.isna(sig_row[a])) else 1.0
        raw[a]   = base_weight * scalar
        released = base_weight * (1.0 - scalar)

        if released > 0:
            havens = [h for h in safe_haven_map.get(a, []) if h in raw]
            if havens:
                per_haven = released / len(havens)
                for h in havens:
                    safe_haven_extra[h] = safe_haven_extra.get(h, 0) + per_haven
            else:
                per_ne = released / len(non_equity_assets)
                for ne in non_equity_assets:
                    safe_haven_extra[ne] = safe_haven_extra.get(ne, 0) + per_ne

    for asset, extra in safe_haven_extra.items():
        raw[asset] = raw.get(asset, 0) + extra

    total = sum(raw.values())
    if total <= 0:
        for a in raw:
            raw[a] = 1.0 / len(raw)
    else:
        for a in raw:
            raw[a] /= total
    return raw

# ── 11c. Partial-hedge weights ────────────────────────────────────────────────
def build_weights_partial(sig_row, equity_assets, non_equity_assets,
                          base_weight, safe_haven_map, partial_keep=0.25):
    """
    On RISK_ON signal: keep partial_keep fraction of base_weight in equity,
    redirect the rest to safe havens.
    """
    raw              = {a: base_weight for a in non_equity_assets}
    safe_haven_extra = {}

    for a in equity_assets:
        if sig_row is not None and a in sig_row.index and not pd.isna(sig_row[a]):
            sig = sig_row[a]
        else:
            sig = -1

        if sig > 0:
            raw[a]   = base_weight * partial_keep
            released = base_weight * (1.0 - partial_keep)
            havens   = [h for h in safe_haven_map.get(a, []) if h in raw]
            if havens:
                per_haven = released / len(havens)
                for h in havens:
                    safe_haven_extra[h] = safe_haven_extra.get(h, 0) + per_haven
            else:
                per_ne = released / len(non_equity_assets)
                for ne in non_equity_assets:
                    safe_haven_extra[ne] = safe_haven_extra.get(ne, 0) + per_ne
        else:
            raw[a] = base_weight

    for asset, extra in safe_haven_extra.items():
        raw[asset] = raw.get(asset, 0) + extra

    total = sum(raw.values())
    if total <= 0:
        for a in raw:
            raw[a] = 1.0 / len(raw)
    else:
        for a in raw:
            raw[a] /= total
    return raw

# ── 12. EQUAL-WEIGHT REBALANCED BENCHMARK ────────────────────────────────────
def run_equal_weight(returns, all_assets, tc_rate=0.005,
                     rebal_freq='monthly', initial_capital=1e9):
    N           = len(all_assets)
    base_weight = 1.0 / N
    ew          = {a: base_weight for a in all_assets}
    reb_set     = build_reb_set(returns, rebal_freq)
    pv          = float(initial_capital)
    current_weights = None
    records     = []

    for date in returns.index:
        if date in reb_set:
            if current_weights is not None:
                turnover = sum(abs(ew.get(a, 0) - current_weights.get(a, 0))
                               for a in set(ew) | set(current_weights)) / 2.0
                pv *= (1 - turnover * tc_rate)
            else:
                pv *= (1 - tc_rate)
            current_weights = ew.copy()

        if current_weights is not None:
            dr  = sum(current_weights.get(a, 0) * returns.loc[date, a]
                      for a in all_assets)
            pv *= (1 + dr)

        records.append({'date': date, 'portfolio_value': pv})

    return pd.DataFrame(records).set_index('date')

# ── 13. GENERIC SIGNAL PORTFOLIO RUNNER ──────────────────────────────────────
def run_signal_portfolio(returns, signal_df, equity_assets,
                         non_equity_assets, all_assets,
                         weight_fn,          # callable: (sig_row, ...) → dict
                         freq='monthly', tc_rate=0.005,
                         initial_capital=1e9):
    """
    Generic runner.  weight_fn must accept (sig_row, equity_assets,
    non_equity_assets, base_weight, <any extra kwargs>) and return
    a dict {asset: weight}.
    """
    N           = len(all_assets)
    base_weight = 1.0 / N
    reb_set     = build_reb_set(returns, freq)

    sig = signal_df.copy()
    sig.index = pd.to_datetime(sig.index)

    pv              = float(initial_capital)
    current_weights = None
    records         = []

    for date in returns.index:
        if date in reb_set:
            past    = sig[sig.index <= date]
            sig_row = past.iloc[-1] if len(past) > 0 else None
            raw     = weight_fn(sig_row)

            if current_weights is not None:
                all_keys = set(raw) | set(current_weights)
                turnover = sum(abs(raw.get(a, 0) - current_weights.get(a, 0))
                               for a in all_keys) / 2.0
                pv *= (1 - turnover * tc_rate)
            else:
                pv *= (1 - tc_rate)

            current_weights = raw.copy()

        if current_weights is not None:
            dr  = sum(current_weights.get(a, 0) * returns.loc[date, a]
                      for a in all_assets)
            pv *= (1 + dr)

        records.append({'date': date, 'portfolio_value': pv})

    return pd.DataFrame(records).set_index('date')

# ── 14. MEAN-VARIANCE SIGNAL PORTFOLIO ───────────────────────────────────────
MV_LOOKBACK = 252
MV_MIN_OBS  = 60
MAX_WEIGHT  = 0.30
RISK_FREE   = 0.0

def mean_variance_weights(mu_vec, cov_mat, allowed_mask,
                          max_weight=MAX_WEIGHT, risk_free=RISK_FREE):
    n_total   = len(mu_vec)
    allowed   = np.where(allowed_mask)[0]
    n_allowed = len(allowed)

    fallback = np.zeros(n_total)
    if n_allowed == 0:
        fallback[:] = 1.0 / n_total
        return fallback
    fallback[allowed] = 1.0 / n_allowed

    mu_sub  = mu_vec[allowed]
    cov_sub = cov_mat[np.ix_(allowed, allowed)] + np.eye(n_allowed) * 1e-8

    def neg_sharpe(w):
        port_ret = np.dot(w, mu_sub)
        port_var = np.dot(w, cov_sub @ w)
        port_std = np.sqrt(max(port_var, 1e-12))
        return -(port_ret - risk_free) / port_std

    constraints = [{'type': 'eq', 'fun': lambda w: np.sum(w) - 1.0}]
    bounds      = [(0.0, max_weight)] * n_allowed
    w0          = np.ones(n_allowed) / n_allowed

    try:
        res = minimize(neg_sharpe, w0, method='SLSQP',
                       bounds=bounds, constraints=constraints,
                       options={'ftol': 1e-9, 'maxiter': 500})
        w_opt = res.x if (res.success and not np.any(np.isnan(res.x))) else w0
    except Exception:
        w_opt = w0

    w_full         = np.zeros(n_total)
    w_full[allowed] = w_opt
    return w_full

def run_mv_signal_portfolio(returns, signal_df, all_assets,
                            equity_assets,
                            freq='monthly', tc_rate=0.005,
                            lookback=MV_LOOKBACK, min_obs=MV_MIN_OBS,
                            max_weight=MAX_WEIGHT, initial_capital=1e9):
    reb_set      = build_reb_set(returns, freq)
    actual_dates = returns.index
    assets       = all_assets
    n            = len(assets)
    asset_idx    = {a: i for i, a in enumerate(assets)}

    sig = signal_df.copy()
    sig.index = pd.to_datetime(sig.index)

    pv              = float(initial_capital)
    current_weights = np.zeros(n)
    records         = []
    weight_history  = []
    first_reb       = True

    for t, date in enumerate(actual_dates):
        if date in reb_set:
            hist = returns.iloc[max(0, t - lookback): t]

            if len(hist) < min_obs:
                new_weights = np.ones(n) / n
            else:
                mu_ann  = hist.mean().values  * 252
                cov_ann = hist.cov().values   * 252

                past    = sig[sig.index <= date]
                sig_row = past.iloc[-1] if len(past) > 0 else None

                allowed_mask = np.ones(n, dtype=bool)
                if sig_row is not None:
                    for eq in equity_assets:
                        if eq in asset_idx and eq in sig_row.index:
                            s = sig_row[eq]
                            if not pd.isna(s) and s > 0:
                                allowed_mask[asset_idx[eq]] = False

                new_weights = mean_variance_weights(
                    mu_ann, cov_ann, allowed_mask, max_weight=max_weight)

            if first_reb:
                pv       *= (1 - tc_rate)
                first_reb = False
            else:
                turnover = np.sum(np.abs(new_weights - current_weights)) / 2.0
                pv      *= (1 - turnover * tc_rate)

            current_weights = new_weights.copy()
            weight_history.append({'date': date,
                                   **dict(zip(assets, current_weights))})

        if not first_reb:
            dr  = np.dot(current_weights, returns.loc[date].values)
            pv *= (1 + dr)

        records.append({'date': date, 'portfolio_value': pv})

    pv_df      = pd.DataFrame(records).set_index('date')
    weights_df = pd.DataFrame(weight_history).set_index('date')
    print(f"  MV | freq={freq} tc={tc_rate*100:.1f}% "
          f"End: ${pv_df['portfolio_value'].iloc[-1]/1e9:.3f}B")
    return pv_df, weights_df

# ── 15. RUN ALL PORTFOLIOS ────────────────────────────────────────────────────
TC_LEVELS = {'0%': 0.00, '0.5%': 0.005, '1%': 0.01, '2%': 0.02}
FREQS     = ['daily', 'weekly', 'monthly', 'quarterly']
MV_FREQS  = ['monthly', 'quarterly']

# ── 15a. Buy-and-Hold ─────────────────────────────────────────────────────────
print("\n── Running Buy-and-Hold ──")
bh_result  = run_buy_and_hold(returns, ALL_ASSETS, initial_capital=1e9)
bh_results = {'BuyAndHold_EW': bh_result}

# ── 15b. Equal-Weight ─────────────────────────────────────────────────────────
print("\n── Running Equal-Weight Rebalanced ──")
ew_results = {}
for freq in FREQS:
    for tc_label, tc_rate in TC_LEVELS.items():
        label = f"EW_{freq}_TC{tc_label}"
        ew_results[label] = run_equal_weight(
            returns, ALL_ASSETS, tc_rate=tc_rate,
            rebal_freq=freq, initial_capital=1e9)
        print(f"  ✓ {label}")

# ── 15c. Original 3-Stage Signal ─────────────────────────────────────────────
print("\n── Running Original 3-Stage Signal ──")
signal_results       = {}
signal_distributions = {}

for thresh in THRESHOLDS:
    sig_df_t    = build_signal_df(composite_score, equity_assets,
                                  macro_df.index, threshold=thresh)
    n_risk_on   = (sig_df_t[equity_assets[0]] == +1).sum()
    n_risk_off  = (sig_df_t[equity_assets[0]] == -1).sum()
    total       = len(sig_df_t)
    signal_distributions[thresh] = {'RISK_ON': n_risk_on, 'RISK_OFF': n_risk_off}
    print(f"  Threshold={thresh} | RISK_ON: {n_risk_on} ({n_risk_on/total*100:.1f}%) "
          f"| RISK_OFF: {n_risk_off} ({n_risk_off/total*100:.1f}%)")

    for freq in FREQS:
        for tc_label, tc_rate in TC_LEVELS.items():
            label = f"Sig3S_t{thresh}_{freq}_TC{tc_label}"

            def wfn_3s(sig_row, _eq=equity_assets, _ne=non_equity_assets,
                       _bw=base_weight, _sh=EQUITY_SAFE_HAVEN_MAP):
                return build_weights_3stage(sig_row, _eq, _ne, _bw, None, _sh)

            signal_results[label] = run_signal_portfolio(
                returns, sig_df_t, equity_assets, non_equity_assets,
                ALL_ASSETS, weight_fn=wfn_3s,
                freq=freq, tc_rate=tc_rate, initial_capital=1e9)
            print(f"    ✓ {label}")

# ── 15d. Asymmetric Signal ────────────────────────────────────────────────────
print("\n── Running Asymmetric Signal ──")
asym_results = {}

for thresh in THRESHOLDS:
    sig_df_t = build_signal_df_asymmetric(composite_score, equity_assets,
                                          macro_df.index, threshold=thresh)
    for freq in ['monthly', 'quarterly']:
        for tc_label, tc_rate in TC_LEVELS.items():
            label = f"Asym_t{thresh}_{freq}_TC{tc_label}"

            def wfn_asym(sig_row, _eq=equity_assets, _ne=non_equity_assets,
                         _bw=base_weight, _sh=EQUITY_SAFE_HAVEN_MAP):
                return build_weights_3stage(sig_row, _eq, _ne, _bw, None, _sh)

            asym_results[label] = run_signal_portfolio(
                returns, sig_df_t, equity_assets, non_equity_assets,
                ALL_ASSETS, weight_fn=wfn_asym,
                freq=freq, tc_rate=tc_rate, initial_capital=1e9)
            print(f"    ✓ {label}")

# ── 15e. Continuous / Graduated Signal ───────────────────────────────────────
print("\n── Running Continuous Signal ──")
cont_results = {}

for thresh in THRESHOLDS:
    for max_red in MAX_REDUCES:
        sig_df_t = build_signal_df_continuous(composite_score, equity_assets,
                                              macro_df.index,
                                              threshold=thresh,
                                              max_reduce=max_red)
        for freq in ['monthly', 'quarterly']:
            for tc_label, tc_rate in TC_LEVELS.items():
                label = (f"Cont_t{thresh}_mr{int(max_red*100)}"
                         f"_{freq}_TC{tc_label}")

                def wfn_cont(sig_row, _eq=equity_assets, _ne=non_equity_assets,
                             _bw=base_weight, _sh=EQUITY_SAFE_HAVEN_MAP):
                    return build_weights_continuous(sig_row, _eq, _ne, _bw, _sh)

                cont_results[label] = run_signal_portfolio(
                    returns, sig_df_t, equity_assets, non_equity_assets,
                    ALL_ASSETS, weight_fn=wfn_cont,
                    freq=freq, tc_rate=tc_rate, initial_capital=1e9)
                print(f"    ✓ {label}")

# ── 15f. Smoothed Signal ─────────────────────────────────────────────────────
print("\n── Running Smoothed Signal ──")
smooth_results = {}

for thresh in THRESHOLDS:
    for sw in SMOOTH_WINDOWS:
        sig_df_t = build_signal_df_smoothed(composite_score, equity_assets,
                                            macro_df.index,
                                            threshold=thresh,
                                            smooth_window=sw)
        for freq in ['monthly', 'quarterly']:
            for tc_label, tc_rate in TC_LEVELS.items():
                label = f"Smooth_t{thresh}_sw{sw}_{freq}_TC{tc_label}"

                def wfn_smooth(sig_row, _eq=equity_assets, _ne=non_equity_assets,
                               _bw=base_weight, _sh=EQUITY_SAFE_HAVEN_MAP):
                    return build_weights_3stage(sig_row, _eq, _ne, _bw, None, _sh)

                smooth_results[label] = run_signal_portfolio(
                    returns, sig_df_t, equity_assets, non_equity_assets,
                    ALL_ASSETS, weight_fn=wfn_smooth,
                    freq=freq, tc_rate=tc_rate, initial_capital=1e9)
                print(f"    ✓ {label}")

# ── 15g. Persistent Signal ────────────────────────────────────────────────────
print("\n── Running Persistent Signal ──")
persist_results = {}

for thresh in THRESHOLDS:
    for min_days in MIN_DAYS_LIST:
        sig_df_t = build_signal_df_persistent(composite_score, equity_assets,
                                              macro_df.index,
                                              threshold=thresh,
                                              min_days_to_switch=min_days)
        for freq in ['monthly', 'quarterly']:
            for tc_label, tc_rate in TC_LEVELS.items():
                label = f"Persist_t{thresh}_md{min_days}_{freq}_TC{tc_label}"

                def wfn_persist(sig_row, _eq=equity_assets, _ne=non_equity_assets,
                                _bw=base_weight, _sh=EQUITY_SAFE_HAVEN_MAP):
                    return build_weights_3stage(sig_row, _eq, _ne, _bw, None, _sh)

                persist_results[label] = run_signal_portfolio(
                    returns, sig_df_t, equity_assets, non_equity_assets,
                    ALL_ASSETS, weight_fn=wfn_persist,
                    freq=freq, tc_rate=tc_rate, initial_capital=1e9)
                print(f"    ✓ {label}")

# ── 15h. Partial-Hedge Signal ─────────────────────────────────────────────────
print("\n── Running Partial-Hedge Signal ──")
partial_results = {}

for thresh in THRESHOLDS:
    sig_df_t = build_signal_df(composite_score, equity_assets,
                               macro_df.index, threshold=thresh)
    for pk in PARTIAL_KEEPS:
        for freq in ['monthly', 'quarterly']:
            for tc_label, tc_rate in TC_LEVELS.items():
                label = (f"Partial_t{thresh}_pk{int(pk*100)}"
                         f"_{freq}_TC{tc_label}")

                def wfn_partial(sig_row, _eq=equity_assets, _ne=non_equity_assets,
                                _bw=base_weight, _sh=EQUITY_SAFE_HAVEN_MAP,
                                _pk=pk):
                    return build_weights_partial(sig_row, _eq, _ne, _bw, _sh, _pk)

                partial_results[label] = run_signal_portfolio(
                    returns, sig_df_t, equity_assets, non_equity_assets,
                    ALL_ASSETS, weight_fn=wfn_partial,
                    freq=freq, tc_rate=tc_rate, initial_capital=1e9)
                print(f"    ✓ {label}")

# ── 15i. Mean-Variance Signal ─────────────────────────────────────────────────
print("\n── Running Mean-Variance Signal ──")
mv_results        = {}
mv_weight_history = {}

for thresh in THRESHOLDS:
    sig_df_t = build_signal_df(composite_score, equity_assets,
                               macro_df.index, threshold=thresh)
    for freq in MV_FREQS:
        for tc_label, tc_rate in TC_LEVELS.items():
            label = f"MV_t{thresh}_{freq}_TC{tc_label}"
            pv_df, w_df = run_mv_signal_portfolio(
                returns, sig_df_t, ALL_ASSETS,
                equity_assets=equity_assets,
                freq=freq, tc_rate=tc_rate,
                lookback=MV_LOOKBACK, min_obs=MV_MIN_OBS,
                max_weight=MAX_WEIGHT, initial_capital=1e9)
            mv_results[label]        = pv_df
            mv_weight_history[label] = w_df
            print(f"  ✓ {label}")

print("\n✓ All portfolios computed")

# ── 16. PERFORMANCE METRICS ───────────────────────────────────────────────────
def compute_metrics(pv_df, label=''):
    pv        = pv_df['portfolio_value']
    daily_ret = pv.pct_change().dropna()
    n_years   = len(daily_ret) / 252
    total_ret = (pv.iloc[-1] / pv.iloc[0]) - 1
    cagr      = (1 + total_ret) ** (1 / n_years) - 1 if n_years > 0 else np.nan
    vol       = daily_ret.std() * np.sqrt(252)
    sharpe    = cagr / vol if vol > 0 else np.nan
    roll_max  = pv.cummax()
    max_dd    = ((pv - roll_max) / roll_max).min()
    calmar    = cagr / abs(max_dd) if max_dd != 0 else np.nan
    var_95    = daily_ret.quantile(0.05)
    ann_ret   = cagr
    ann_vol   = vol

    return {
        'CAGR':         f"{cagr*100:.2f}%",
        'Ann_Ret':      ann_ret,
        'Ann_Vol':      ann_vol,
        'Volatility':   f"{vol*100:.2f}%",
        'Sharpe':       f"{sharpe:.2f}",
        'Max Drawdown': f"{max_dd*100:.2f}%",
        'Calmar':       f"{calmar:.2f}",
        'VaR 95%':      f"{var_95*100:.2f}%",
        'Total Return': f"{total_ret*100:.2f}%",
        'Final ($B)':   f"{pv.iloc[-1]/1e9:.3f}",
    }

bh_metrics      = {k: compute_metrics(v, k) for k, v in bh_results.items()}
ew_metrics      = {k: compute_metrics(v, k) for k, v in ew_results.items()}
sig_metrics     = {k: compute_metrics(v, k) for k, v in signal_results.items()}
asym_metrics    = {k: compute_metrics(v, k) for k, v in asym_results.items()}
cont_metrics    = {k: compute_metrics(v, k) for k, v in cont_results.items()}
smooth_metrics  = {k: compute_metrics(v, k) for k, v in smooth_results.items()}
persist_metrics = {k: compute_metrics(v, k) for k, v in persist_results.items()}
partial_metrics = {k: compute_metrics(v, k) for k, v in partial_results.items()}
mv_metrics      = {k: compute_metrics(v, k) for k, v in mv_results.items()}

def print_metrics_table(metrics_dict, title):
    display = {k: {mk: mv for mk, mv in m.items()
                   if mk not in ('Ann_Ret', 'Ann_Vol')}
               for k, m in metrics_dict.items()}
    print(f"\n{'='*90}")
    print(title)
    print("="*90)
    print(pd.DataFrame(display).T.to_string())

print_metrics_table(bh_metrics,      "BUY-AND-HOLD")
print_metrics_table(ew_metrics,      "EQUAL-WEIGHT REBALANCED")

for thresh in THRESHOLDS:
    sub = {k: v for k, v in sig_metrics.items()     if f"_t{thresh}_" in k}
    print_metrics_table(sub, f"3-STAGE SIGNAL — threshold={thresh}")

for thresh in THRESHOLDS:
    sub = {k: v for k, v in asym_metrics.items()    if f"_t{thresh}_" in k}
    print_metrics_table(sub, f"ASYMMETRIC SIGNAL — threshold={thresh}")

for thresh in THRESHOLDS:
    sub = {k: v for k, v in cont_metrics.items()    if f"_t{thresh}_" in k}
    print_metrics_table(sub, f"CONTINUOUS SIGNAL — threshold={thresh}")

for thresh in THRESHOLDS:
    sub = {k: v for k, v in smooth_metrics.items()  if f"_t{thresh}_" in k}
    print_metrics_table(sub, f"SMOOTHED SIGNAL — threshold={thresh}")

for thresh in THRESHOLDS:
    sub = {k: v for k, v in persist_metrics.items() if f"_t{thresh}_" in k}
    print_metrics_table(sub, f"PERSISTENT SIGNAL — threshold={thresh}")

for thresh in THRESHOLDS:
    sub = {k: v for k, v in partial_metrics.items() if f"_t{thresh}_" in k}
    print_metrics_table(sub, f"PARTIAL-HEDGE SIGNAL — threshold={thresh}")

for thresh in THRESHOLDS:
    sub = {k: v for k, v in mv_metrics.items()      if f"_t{thresh}_" in k}
    print_metrics_table(sub, f"MEAN-VARIANCE SIGNAL — threshold={thresh}")

# ── 17. PLOTS ─────────────────────────────────────────────────────────────────
bh_pv     = bh_results['BuyAndHold_EW']['portfolio_value']
ew_ref_pv = ew_results['EW_monthly_TC0.5%']['portfolio_value']

def parse_pct(s):
    return float(str(s).replace('%', '')) / 100

# ── Plot A: Best of each variant vs benchmarks (monthly, TC=0.5%) ─────────────
fig, axes = plt.subplots(2, 4, figsize=(28, 12), sharey=False)
fig.suptitle("All Signal Variants vs Benchmarks — Monthly Rebalancing, TC=0.5%",
             fontsize=14, fontweight='bold')

variant_info = [
    ("3-Stage",    signal_results,  THRESHOLDS,     "Sig3S",   "steelblue"),
    ("Asymmetric", asym_results,    THRESHOLDS,     "Asym",    "darkorange"),
    ("Continuous", cont_results,    THRESHOLDS,     "Cont",    "purple"),
    ("Smoothed",   smooth_results,  THRESHOLDS,     "Smooth",  "green"),
    ("Persistent", persist_results, THRESHOLDS,     "Persist", "crimson"),
    ("Partial 25%",partial_results, THRESHOLDS,     "Partial", "brown"),
    ("MV Signal",  mv_results,      THRESHOLDS,     "MV",      "black"),
]

for ax, (name, res_dict, threshs, prefix, col) in zip(axes.flat, variant_info):
    ax.plot(bh_pv.index,     bh_pv / 1e9,     color='forestgreen',
            lw=1.5, ls='-.', label='Buy & Hold')
    ax.plot(ew_ref_pv.index, ew_ref_pv / 1e9, color='grey',
            lw=1.2, ls='--', label='EW Monthly')

    for thresh in threshs:
        candidates = [k for k in res_dict
                      if f"_t{thresh}_" in k and 'monthly' in k and 'TC0.5%' in k]
        if not candidates:
            candidates = [k for k in res_dict
                          if f"_t{thresh}_" in k and 'TC0.5%' in k]
        if candidates:
            key = candidates[0]
            pv  = res_dict[key]['portfolio_value']
            ax.plot(pv.index, pv / 1e9, lw=1.6,
                    label=f"t={thresh}")

    ax.axhline(1.0, color='black', ls=':', lw=0.7)
    ax.set_title(name, fontsize=11, fontweight='bold')
    ax.set_ylabel("Value ($B)")
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

axes.flat[-1].set_visible(False)
plt.tight_layout()
plt.savefig("all_variants_monthly.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Plot B: Risk-Return Scatter — all strategies ──────────────────────────────
fig, ax = plt.subplots(figsize=(16, 10))

all_metric_groups = [
    (bh_metrics,      'BH',       'forestgreen', 'D'),
    (ew_metrics,      'EW',       'grey',        's'),
    (sig_metrics,     '3-Stage',  'steelblue',   'o'),
    (asym_metrics,    'Asym',     'darkorange',  'o'),
    (cont_metrics,    'Cont',     'purple',      'o'),
    (smooth_metrics,  'Smooth',   'green',       'o'),
    (persist_metrics, 'Persist',  'crimson',     'o'),
    (partial_metrics, 'Partial',  'brown',       'o'),
    (mv_metrics,      'MV',       'black',       '^'),
]

for metrics_dict, grp_label, col, marker in all_metric_groups:
    first = True
    for key, m in metrics_dict.items():
        vol = m['Ann_Vol'] * 100
        ret = m['Ann_Ret'] * 100
        ax.scatter(vol, ret, color=col, marker=marker, s=80,
                   label=grp_label if first else None,
                   edgecolors='black', lw=0.5, zorder=4)
        ax.annotate(key.split('_TC')[0], (vol, ret),
                    textcoords='offset points', xytext=(4, 3),
                    fontsize=6, color=col)
        first = False

ax.axhline(0, color='black', lw=0.6, ls='--')
ax.axvline(0, color='black', lw=0.6, ls='--')
ax.set_xlabel("Annualised Volatility (%)", fontsize=12)
ax.set_ylabel("CAGR (%)", fontsize=12)
ax.set_title("Risk-Return — All Strategies", fontsize=13, fontweight='bold')
ax.legend(fontsize=10, ncol=3)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("risk_return_all.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Plot C: Sharpe Bar Chart — top 20 strategies ──────────────────────────────
all_metrics_combined = {
    **bh_metrics, **ew_metrics, **sig_metrics,
    **asym_metrics, **cont_metrics, **smooth_metrics,
    **persist_metrics, **partial_metrics, **mv_metrics
}

sharpe_series = pd.Series({
    k: float(v['Sharpe']) for k, v in all_metrics_combined.items()
}).dropna().sort_values(ascending=False)

top20 = sharpe_series.head(20)

def label_color(lbl):
    if lbl.startswith('MV'):       return 'black'
    if lbl.startswith('Sig3S'):    return 'steelblue'
    if lbl.startswith('Asym'):     return 'darkorange'
    if lbl.startswith('Cont'):     return 'purple'
    if lbl.startswith('Smooth'):   return 'green'
    if lbl.startswith('Persist'):  return 'crimson'
    if lbl.startswith('Partial'):  return 'brown'
    if lbl.startswith('EW'):       return 'grey'
    return 'forestgreen'

colors = [label_color(k) for k in top20.index]

fig, ax = plt.subplots(figsize=(16, 7))
bars = ax.barh(range(len(top20)), top20.values, color=colors, edgecolor='black', lw=0.5)
ax.set_yticks(range(len(top20)))
ax.set_yticklabels([k.replace('_TC', ' TC=') for k in top20.index], fontsize=8)
ax.set_xlabel("Sharpe Ratio")
ax.set_title("Top 20 Strategies by Sharpe Ratio", fontsize=13, fontweight='bold')
ax.axvline(0, color='black', lw=0.8)
ax.grid(True, axis='x', alpha=0.3)

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='black',      label='MV Signal'),
    Patch(facecolor='steelblue',  label='3-Stage'),
    Patch(facecolor='darkorange', label='Asymmetric'),
    Patch(facecolor='purple',     label='Continuous'),
    Patch(facecolor='green',      label='Smoothed'),
    Patch(facecolor='crimson',    label='Persistent'),
    Patch(facecolor='brown',      label='Partial Hedge'),
    Patch(facecolor='grey',       label='Equal Weight'),
    Patch(facecolor='forestgreen',label='Buy & Hold'),
]
ax.legend(handles=legend_elements, fontsize=9, loc='lower right')
plt.tight_layout()
plt.savefig("sharpe_top20.png", dpi=150, bbox_inches='tight')
plt.show()

# ── 18. SAVE ALL ──────────────────────────────────────────────────────────────
print("\n── Saving all results ──")
all_portfolios = {
    **bh_results, **ew_results, **signal_results,
    **asym_results, **cont_results, **smooth_results,
    **persist_results, **partial_results, **mv_results
}

for label, df in all_portfolios.items():
    df.to_csv(f"{label}_portfolio.csv")

all_metrics_df = pd.DataFrame({
    k: {mk: mv for mk, mv in m.items() if mk not in ('Ann_Ret', 'Ann_Vol')}
    for k, m in all_metrics_combined.items()
}).T
all_metrics_df.to_csv("all_portfolio_metrics.csv")

composite_score.to_csv("composite_signal.csv")
for thresh in THRESHOLDS:
    sig_df_t = build_signal_df(composite_score, equity_assets,
                               macro_df.index, threshold=thresh)
    sig_df_t.to_csv(f"signal_df_threshold_{thresh}.csv")

for label, wdf in mv_weight_history.items():
    wdf.to_csv(f"{label}_weights.csv")

print("✓ All files saved.")
print(f"\n  Portfolio CSVs   : {len(all_portfolios)}")
print(f"  Weight CSVs      : {len(mv_weight_history)}")
print(f"  Metrics CSV      : all_portfolio_metrics.csv")
print(f"  Signal CSVs      : {len(THRESHOLDS)} threshold files")
print(f"  Plots saved      : 3 PNG files")


# Full Domain 2015-2025

In [ ]:
# @title
# ============================================================
# SIGNAL PORTFOLIO — ENHANCED VERSION
# Signals generated from Modified Determinant Model
# + Buy-and-Hold Benchmark
# + Equal-Weight Benchmark
# + Three-Stage Signal Portfolio (original)
# + Enhanced Signal Variants (asymmetric, continuous, smoothed, persistent, partial)
# + Mean-Variance Signal Portfolio
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import requests
from io import StringIO
from scipy.optimize import minimize

# ── 1. LOAD ALL DATA FROM GITHUB ─────────────────────────────────────────────
GITHUB_BASE = "https://raw.githubusercontent.com/kboroz/MSFE_Capstone_Project/main/01_Data/Streamlined/"

INDEX_FILE_MAP = {
    'ASX50':       'streamlined_asx_50.csv',
    'EUROSTOXX50': 'streamlined_euro_stoxx_50.csv',
    'FTSE100':     'streamlined_ftse_100.csv',
    'Ibovespa':    'streamlined_ibovespa.csv',
    'JSE40':       'streamlined_jse_top_40.csv',
    'NIKKEI225':   'streamlined_nikkei_225.csv',
    'SP500':       'streamlined_s&p_500.csv',
}

MACRO_FILE = 'streamlined_macro.csv'

def load_csv(url):
    r = requests.get(url)
    r.raise_for_status()
    df = pd.read_csv(StringIO(r.text), index_col=0, parse_dates=True)
    df.columns = df.columns.str.replace('[^a-zA-Z0-9]', '_', regex=True)
    return df.sort_index()

macro_df = load_csv(GITHUB_BASE + MACRO_FILE)
macro_df = macro_df[~macro_df.index.duplicated(keep='first')].sort_index()
print(f"macro_df shape : {macro_df.shape}")
print(f"Columns        : {macro_df.columns.tolist()}")

index_data = {}
for name, fname in INDEX_FILE_MAP.items():
    try:
        df = load_csv(GITHUB_BASE + fname)
        index_data[name] = df
        print(f"✓ {name:15s} | shape={df.shape} | cols={df.columns.tolist()[:4]}")
    except Exception as e:
        print(f"✗ {name}: {e}")

# ── 2. EXTRACT PRICE SERIES ───────────────────────────────────────────────────
PRICE_COLS = ['Close', 'close', 'Price', 'price', 'Adj_Close', 'Last']

def get_price_series(df, name):
    col = next((c for c in PRICE_COLS if c in df.columns), df.columns[0])
    s = pd.to_numeric(df[col], errors='coerce').dropna()
    print(f"  {name}: using column '{col}'")
    return s

index_prices = {}
for name, df in index_data.items():
    index_prices[name] = get_price_series(df, name)

# ── 3. MODIFIED DETERMINANT SIGNAL MODEL ─────────────────────────────────────
DET_WINDOWS = [3, 7, 28]
WINDOW_WEIGHTS = {3: 0.3333, 7: 0.3333, 28: 0.3333}

def compute_modified_determinant(price_series_dict, windows=DET_WINDOWS):
    prices = pd.DataFrame(price_series_dict).sort_index()
    rets   = prices.pct_change().dropna()
    det_results = {}

    for window in windows:
        det_series = []
        for i in range(window, len(rets)):
            window_rets = rets.iloc[i - window:i]
            valid_cols  = window_rets.columns[window_rets.std() > 1e-10]
            w           = window_rets[valid_cols]

            if len(valid_cols) < 2:
                det_series.append(np.nan)
                continue

            try:
                R   = w.corr().values
                I   = np.eye(len(R))
                M   = R - I
                det = np.linalg.det(M)
            except Exception:
                det = np.nan

            det_series.append(det)

        idx = rets.index[window:]
        det_results[window] = pd.Series(det_series, index=idx,
                                        name=f'det_{window}d')
        print(f"  Window {window:2d}d | computed {len(det_series)} observations")

    return det_results

print("\n── Computing Modified Determinant ──")
det_results = compute_modified_determinant(index_prices)

# ── 4. COMPOSITE SCORE ────────────────────────────────────────────────────────
def compute_composite_score(det_results, window_weights=WINDOW_WEIGHTS):
    all_series = []
    for window, series in det_results.items():
        mu    = series.rolling(252, min_periods=60).mean()
        sigma = series.rolling(252, min_periods=60).std()
        z     = (series - mu) / sigma.replace(0, np.nan)
        z.name = f'z_{window}d'
        all_series.append(z)

    z_df      = pd.concat(all_series, axis=1).dropna()
    composite = sum(
        z_df[f'z_{w}d'] * window_weights[w]
        for w in window_weights if f'z_{w}d' in z_df.columns
    )
    composite = composite / sum(window_weights.values())
    return composite, z_df

composite_score, z_df = compute_composite_score(det_results)

print(f"\nComposite score shape  : {composite_score.shape}")
print(f"Date range             : {composite_score.index[0].date()} → "
      f"{composite_score.index[-1].date()}")
print(f"Score range            : {composite_score.min():.3f} → "
      f"{composite_score.max():.3f}")

# ── 5. SIGNAL BUILDERS ───────────────────────────────────────────────────────
THRESHOLDS    = [0.3, 0.5, 0.7]
SMOOTH_WINDOWS = [3, 5, 10]
MIN_DAYS_LIST  = [5, 10, 20]
MAX_REDUCES    = [0.5, 0.7, 0.9]
PARTIAL_KEEPS  = [0.25, 0.50]

# ── 5a. Original symmetric binary ────────────────────────────────────────────
def classify_signal_symmetric(score, threshold):
    return +1 if abs(score) > threshold else -1

def build_signal_df(composite_score, equity_assets, macro_dates, threshold=0.5):
    binary    = composite_score.apply(lambda x: classify_signal_symmetric(x, threshold))
    signal_df = pd.DataFrame(index=macro_dates)
    for asset in equity_assets:
        signal_df[asset] = binary.reindex(macro_dates).ffill().bfill()
    return signal_df

# ── 5b. Asymmetric (upper-tail only) ─────────────────────────────────────────
def build_signal_df_asymmetric(composite_score, equity_assets, macro_dates,
                               threshold=1.0):
    """Block equities only when correlation SPIKES (high positive z-score)."""
    binary    = composite_score.apply(lambda x: +1 if x > threshold else -1)
    signal_df = pd.DataFrame(index=macro_dates)
    for asset in equity_assets:
        signal_df[asset] = binary.reindex(macro_dates).ffill().bfill()
    return signal_df

# ── 5c. Continuous / graduated scalar ────────────────────────────────────────
def build_signal_df_continuous(composite_score, equity_assets, macro_dates,
                               threshold=1.0, max_reduce=0.8):
    """
    Returns a scalar in [1-max_reduce, 1.0] per asset per day.
    Used by build_weights_continuous() instead of the binary flag.
    """
    def score_to_scalar(s):
        if s <= 0:
            return 1.0
        elif s >= threshold:
            return 1.0 - max_reduce
        else:
            return 1.0 - max_reduce * (s / threshold)

    scalars   = composite_score.apply(score_to_scalar)
    signal_df = pd.DataFrame(index=macro_dates)
    for asset in equity_assets:
        signal_df[asset] = scalars.reindex(macro_dates).ffill().bfill()
    return signal_df

# ── 5d. Smoothed binary ───────────────────────────────────────────────────────
def build_signal_df_smoothed(composite_score, equity_assets, macro_dates,
                             threshold=1.0, smooth_window=5):
    """Smooth z-score before thresholding to reduce whipsaw."""
    smoothed  = composite_score.rolling(smooth_window, min_periods=1).mean()
    binary    = smoothed.apply(lambda x: classify_signal_symmetric(x, threshold))
    signal_df = pd.DataFrame(index=macro_dates)
    for asset in equity_assets:
        signal_df[asset] = binary.reindex(macro_dates).ffill().bfill()
    return signal_df

# ── 5e. Persistent binary ────────────────────────────────────────────────────
def build_signal_df_persistent(composite_score, equity_assets, macro_dates,
                               threshold=1.0, min_days_to_switch=10):
    """Only switch regime after signal holds for min_days_to_switch days."""
    raw       = composite_score.apply(lambda x: +1 if x > threshold else -1)
    confirmed = raw.copy()
    current   = int(raw.iloc[0])
    count     = 0

    for i in range(1, len(raw)):
        if int(raw.iloc[i]) != current:
            count += 1
            if count >= min_days_to_switch:
                current = int(raw.iloc[i])
                count   = 0
        else:
            count = 0
        confirmed.iloc[i] = current

    signal_df = pd.DataFrame(index=macro_dates)
    for asset in equity_assets:
        signal_df[asset] = confirmed.reindex(macro_dates).ffill().bfill()
    return signal_df

# ── 6. ASSET UNIVERSE ─────────────────────────────────────────────────────────
EQUITY_ASSETS = list(index_prices.keys())
SAFE_BONDS    = ['US_10Y', 'UK_10Y', 'AU_10Y', 'ZA_10Y']
FX            = ['AUDUSD', 'BRLUSD', 'EURUSD', 'GBPUSD', 'JPYUSD', 'ZARUSD']
COMMODITIES   = ['Oil', 'Gold', 'Bitcoin']
EXCLUDE_ASSETS = ['Bitcoin']

ALL_ASSETS = EQUITY_ASSETS + FX + COMMODITIES + SAFE_BONDS
ALL_ASSETS = [a for a in ALL_ASSETS
              if a in macro_df.columns and a not in EXCLUDE_ASSETS]

equity_assets     = [a for a in EQUITY_ASSETS     if a in ALL_ASSETS]
non_equity_assets = [a for a in ALL_ASSETS         if a not in equity_assets]

N           = len(ALL_ASSETS)
base_weight = 1.0 / N

print(f"\nTotal assets              : {N}")
print(f"Equity (signal-driven)    : {equity_assets}")
print(f"Non-equity (always long)  : {non_equity_assets}")
print(f"base_weight = 1/{N} = {base_weight:.4f}")

# ── 7. COUNTRY SAFE-HAVEN MAP ─────────────────────────────────────────────────
EQUITY_SAFE_HAVEN_MAP = {
    'ASX50':       ['AU_10Y', 'AUDUSD'],
    'EUROSTOXX50': ['EURUSD'],
    'FTSE100':     ['UK_10Y', 'GBPUSD'],
    'Ibovespa':    ['BRLUSD'],
    'JSE40':       ['ZA_10Y', 'ZARUSD'],
    'NIKKEI225':   ['JPYUSD'],
    'SP500':       ['US_10Y'],
}

UNIVERSE_SET = set(ALL_ASSETS)
for eq, havens in EQUITY_SAFE_HAVEN_MAP.items():
    EQUITY_SAFE_HAVEN_MAP[eq] = [h for h in havens if h in UNIVERSE_SET]

print("\nSafe-haven routing (filtered to universe):")
for eq, havens in EQUITY_SAFE_HAVEN_MAP.items():
    print(f"  {eq:15s} → {havens}")

# ── 8. BUILD RETURNS ──────────────────────────────────────────────────────────
returns_raw = macro_df[ALL_ASSETS].pct_change(fill_method=None)

for col in SAFE_BONDS:
    if col in returns_raw.columns:
        returns_raw[col] = -returns_raw[col]

if 'Oil' in macro_df.columns:
    bad_oil = macro_df['Oil'][macro_df['Oil'] <= 0].index
    for d in bad_oil:
        loc = returns_raw.index.get_loc(d)
        returns_raw.iloc[max(0, loc - 1): loc + 2,
                         returns_raw.columns.get_loc('Oil')] = 0.0

returns = returns_raw.clip(lower=-0.50, upper=1.00).fillna(0).iloc[1:]

assert not returns.isnull().any().any(), "NaNs in returns"
assert not np.isinf(returns.values).any(),  "Inf in returns"
print(f"\nReturns shape : {returns.shape}")
print("✓ Returns clean")

# ── 9. HELPER: REBALANCING DATE SET ──────────────────────────────────────────
def build_reb_set(returns, rebal_freq):
    actual_dates = returns.index
    if rebal_freq == 'daily':
        return set(actual_dates)
    elif rebal_freq == 'weekly':
        raw = returns.resample('W-MON').first().index
    elif rebal_freq == 'monthly':
        raw = returns.resample('MS').first().index
    elif rebal_freq == 'quarterly':
        raw = returns.resample('QS').first().index
    else:
        raise ValueError(f"Unknown rebal_freq: {rebal_freq}")

    reb_set = set()
    for d in raw:
        future = actual_dates[actual_dates >= d]
        if len(future) > 0:
            reb_set.add(future[0])
    return reb_set

# ── 10. BUY-AND-HOLD BENCHMARK ───────────────────────────────────────────────
def run_buy_and_hold(returns, all_assets, initial_capital=1e9):
    N           = len(all_assets)
    base_weight = 1.0 / N
    pv          = float(initial_capital)
    holdings    = {a: pv * base_weight for a in all_assets}
    records     = []
    first       = True

    for date in returns.index:
        if first:
            first = False
            records.append({'date': date, 'portfolio_value': pv})
            continue
        new_holdings = {a: holdings[a] * (1 + returns.loc[date, a])
                        for a in all_assets}
        holdings = new_holdings
        pv       = sum(holdings.values())
        records.append({'date': date, 'portfolio_value': pv})

    df = pd.DataFrame(records).set_index('date')
    print(f"  Buy-and-Hold | Start: ${initial_capital/1e9:.2f}B → "
          f"End: ${df['portfolio_value'].iloc[-1]/1e9:.3f}B")
    return df

# ── 11. WEIGHT BUILDERS ───────────────────────────────────────────────────────

# ── 11a. Original binary 3-stage weights ─────────────────────────────────────
def build_weights_3stage(sig_row, equity_assets, non_equity_assets,
                         base_weight, date, safe_haven_map):
    raw            = {a: base_weight for a in non_equity_assets}
    safe_haven_extra = {}

    for a in equity_assets:
        if sig_row is not None and a in sig_row.index and not pd.isna(sig_row[a]):
            sig = sig_row[a]
        else:
            sig = -1

        if sig > 0:
            raw[a] = -base_weight
            havens = [h for h in safe_haven_map.get(a, []) if h in raw]
            if havens:
                per_haven = base_weight / len(havens)
                for h in havens:
                    safe_haven_extra[h] = safe_haven_extra.get(h, 0) + per_haven
            else:
                per_ne = base_weight / len(non_equity_assets)
                for ne in non_equity_assets:
                    safe_haven_extra[ne] = safe_haven_extra.get(ne, 0) + per_ne
        else:
            raw[a] = base_weight

    for asset, extra in safe_haven_extra.items():
        raw[asset] = raw.get(asset, 0) + extra

    total = sum(raw.values())
    if total <= 0:
        for a in raw:
            raw[a] = 1.0 / len(raw)
    else:
        for a in raw:
            raw[a] /= total
    return raw

# ── 11b. Continuous scalar weights ───────────────────────────────────────────
def build_weights_continuous(sig_row, equity_assets, non_equity_assets,
                             base_weight, safe_haven_map):
    """
    sig_row contains a scalar in [min_weight, 1.0] per equity asset.
    Reduced equity weight is redistributed to safe havens.
    """
    raw              = {a: base_weight for a in non_equity_assets}
    safe_haven_extra = {}

    for a in equity_assets:
        scalar = float(sig_row[a]) if (sig_row is not None and
                                       a in sig_row.index and
                                       not pd.isna(sig_row[a])) else 1.0
        raw[a]   = base_weight * scalar
        released = base_weight * (1.0 - scalar)

        if released > 0:
            havens = [h for h in safe_haven_map.get(a, []) if h in raw]
            if havens:
                per_haven = released / len(havens)
                for h in havens:
                    safe_haven_extra[h] = safe_haven_extra.get(h, 0) + per_haven
            else:
                per_ne = released / len(non_equity_assets)
                for ne in non_equity_assets:
                    safe_haven_extra[ne] = safe_haven_extra.get(ne, 0) + per_ne

    for asset, extra in safe_haven_extra.items():
        raw[asset] = raw.get(asset, 0) + extra

    total = sum(raw.values())
    if total <= 0:
        for a in raw:
            raw[a] = 1.0 / len(raw)
    else:
        for a in raw:
            raw[a] /= total
    return raw

# ── 11c. Partial-hedge weights ────────────────────────────────────────────────
def build_weights_partial(sig_row, equity_assets, non_equity_assets,
                          base_weight, safe_haven_map, partial_keep=0.25):
    """
    On RISK_ON signal: keep partial_keep fraction of base_weight in equity,
    redirect the rest to safe havens.
    """
    raw              = {a: base_weight for a in non_equity_assets}
    safe_haven_extra = {}

    for a in equity_assets:
        if sig_row is not None and a in sig_row.index and not pd.isna(sig_row[a]):
            sig = sig_row[a]
        else:
            sig = -1

        if sig > 0:
            raw[a]   = base_weight * partial_keep
            released = base_weight * (1.0 - partial_keep)
            havens   = [h for h in safe_haven_map.get(a, []) if h in raw]
            if havens:
                per_haven = released / len(havens)
                for h in havens:
                    safe_haven_extra[h] = safe_haven_extra.get(h, 0) + per_haven
            else:
                per_ne = released / len(non_equity_assets)
                for ne in non_equity_assets:
                    safe_haven_extra[ne] = safe_haven_extra.get(ne, 0) + per_ne
        else:
            raw[a] = base_weight

    for asset, extra in safe_haven_extra.items():
        raw[asset] = raw.get(asset, 0) + extra

    total = sum(raw.values())
    if total <= 0:
        for a in raw:
            raw[a] = 1.0 / len(raw)
    else:
        for a in raw:
            raw[a] /= total
    return raw

# ── 12. EQUAL-WEIGHT REBALANCED BENCHMARK ────────────────────────────────────
def run_equal_weight(returns, all_assets, tc_rate=0.005,
                     rebal_freq='monthly', initial_capital=1e9):
    N           = len(all_assets)
    base_weight = 1.0 / N
    ew          = {a: base_weight for a in all_assets}
    reb_set     = build_reb_set(returns, rebal_freq)
    pv          = float(initial_capital)
    current_weights = None
    records     = []

    for date in returns.index:
        if date in reb_set:
            if current_weights is not None:
                turnover = sum(abs(ew.get(a, 0) - current_weights.get(a, 0))
                               for a in set(ew) | set(current_weights)) / 2.0
                pv *= (1 - turnover * tc_rate)
            else:
                pv *= (1 - tc_rate)
            current_weights = ew.copy()

        if current_weights is not None:
            dr  = sum(current_weights.get(a, 0) * returns.loc[date, a]
                      for a in all_assets)
            pv *= (1 + dr)

        records.append({'date': date, 'portfolio_value': pv})

    return pd.DataFrame(records).set_index('date')

# ── 13. GENERIC SIGNAL PORTFOLIO RUNNER ──────────────────────────────────────
def run_signal_portfolio(returns, signal_df, equity_assets,
                         non_equity_assets, all_assets,
                         weight_fn,          # callable: (sig_row, ...) → dict
                         freq='monthly', tc_rate=0.005,
                         initial_capital=1e9):
    """
    Generic runner.  weight_fn must accept (sig_row, equity_assets,
    non_equity_assets, base_weight, <any extra kwargs>) and return
    a dict {asset: weight}.
    """
    N           = len(all_assets)
    base_weight = 1.0 / N
    reb_set     = build_reb_set(returns, freq)

    sig = signal_df.copy()
    sig.index = pd.to_datetime(sig.index)

    pv              = float(initial_capital)
    current_weights = None
    records         = []

    for date in returns.index:
        if date in reb_set:
            past    = sig[sig.index <= date]
            sig_row = past.iloc[-1] if len(past) > 0 else None
            raw     = weight_fn(sig_row)

            if current_weights is not None:
                all_keys = set(raw) | set(current_weights)
                turnover = sum(abs(raw.get(a, 0) - current_weights.get(a, 0))
                               for a in all_keys) / 2.0
                pv *= (1 - turnover * tc_rate)
            else:
                pv *= (1 - tc_rate)

            current_weights = raw.copy()

        if current_weights is not None:
            dr  = sum(current_weights.get(a, 0) * returns.loc[date, a]
                      for a in all_assets)
            pv *= (1 + dr)

        records.append({'date': date, 'portfolio_value': pv})

    return pd.DataFrame(records).set_index('date')

# ── 14. MEAN-VARIANCE SIGNAL PORTFOLIO ───────────────────────────────────────
MV_LOOKBACK = 252
MV_MIN_OBS  = 60
MAX_WEIGHT  = 0.30
RISK_FREE   = 0.0

def mean_variance_weights(mu_vec, cov_mat, allowed_mask,
                          max_weight=MAX_WEIGHT, risk_free=RISK_FREE):
    n_total   = len(mu_vec)
    allowed   = np.where(allowed_mask)[0]
    n_allowed = len(allowed)

    fallback = np.zeros(n_total)
    if n_allowed == 0:
        fallback[:] = 1.0 / n_total
        return fallback
    fallback[allowed] = 1.0 / n_allowed

    mu_sub  = mu_vec[allowed]
    cov_sub = cov_mat[np.ix_(allowed, allowed)] + np.eye(n_allowed) * 1e-8

    def neg_sharpe(w):
        port_ret = np.dot(w, mu_sub)
        port_var = np.dot(w, cov_sub @ w)
        port_std = np.sqrt(max(port_var, 1e-12))
        return -(port_ret - risk_free) / port_std

    constraints = [{'type': 'eq', 'fun': lambda w: np.sum(w) - 1.0}]
    bounds      = [(0.0, max_weight)] * n_allowed
    w0          = np.ones(n_allowed) / n_allowed

    try:
        res = minimize(neg_sharpe, w0, method='SLSQP',
                       bounds=bounds, constraints=constraints,
                       options={'ftol': 1e-9, 'maxiter': 500})
        w_opt = res.x if (res.success and not np.any(np.isnan(res.x))) else w0
    except Exception:
        w_opt = w0

    w_full         = np.zeros(n_total)
    w_full[allowed] = w_opt
    return w_full

def run_mv_signal_portfolio(returns, signal_df, all_assets,
                            equity_assets,
                            freq='monthly', tc_rate=0.005,
                            lookback=MV_LOOKBACK, min_obs=MV_MIN_OBS,
                            max_weight=MAX_WEIGHT, initial_capital=1e9):
    reb_set      = build_reb_set(returns, freq)
    actual_dates = returns.index
    assets       = all_assets
    n            = len(assets)
    asset_idx    = {a: i for i, a in enumerate(assets)}

    sig = signal_df.copy()
    sig.index = pd.to_datetime(sig.index)

    pv              = float(initial_capital)
    current_weights = np.zeros(n)
    records         = []
    weight_history  = []
    first_reb       = True

    for t, date in enumerate(actual_dates):
        if date in reb_set:
            hist = returns.iloc[max(0, t - lookback): t]

            if len(hist) < min_obs:
                new_weights = np.ones(n) / n
            else:
                mu_ann  = hist.mean().values  * 252
                cov_ann = hist.cov().values   * 252

                past    = sig[sig.index <= date]
                sig_row = past.iloc[-1] if len(past) > 0 else None

                allowed_mask = np.ones(n, dtype=bool)
                if sig_row is not None:
                    for eq in equity_assets:
                        if eq in asset_idx and eq in sig_row.index:
                            s = sig_row[eq]
                            if not pd.isna(s) and s > 0:
                                allowed_mask[asset_idx[eq]] = False

                new_weights = mean_variance_weights(
                    mu_ann, cov_ann, allowed_mask, max_weight=max_weight)

            if first_reb:
                pv       *= (1 - tc_rate)
                first_reb = False
            else:
                turnover = np.sum(np.abs(new_weights - current_weights)) / 2.0
                pv      *= (1 - turnover * tc_rate)

            current_weights = new_weights.copy()
            weight_history.append({'date': date,
                                   **dict(zip(assets, current_weights))})

        if not first_reb:
            dr  = np.dot(current_weights, returns.loc[date].values)
            pv *= (1 + dr)

        records.append({'date': date, 'portfolio_value': pv})

    pv_df      = pd.DataFrame(records).set_index('date')
    weights_df = pd.DataFrame(weight_history).set_index('date')
    print(f"  MV | freq={freq} tc={tc_rate*100:.1f}% "
          f"End: ${pv_df['portfolio_value'].iloc[-1]/1e9:.3f}B")
    return pv_df, weights_df

# ── 15. RUN ALL PORTFOLIOS ────────────────────────────────────────────────────
TC_LEVELS = {'0%': 0.00, '0.5%': 0.005, '1%': 0.01, '2%': 0.02}
FREQS     = ['daily', 'weekly', 'monthly', 'quarterly']
MV_FREQS  = ['monthly', 'quarterly']

# ── 15a. Buy-and-Hold ─────────────────────────────────────────────────────────
print("\n── Running Buy-and-Hold ──")
bh_result  = run_buy_and_hold(returns, ALL_ASSETS, initial_capital=1e9)
bh_results = {'BuyAndHold_EW': bh_result}

# ── 15b. Equal-Weight ─────────────────────────────────────────────────────────
print("\n── Running Equal-Weight Rebalanced ──")
ew_results = {}
for freq in FREQS:
    for tc_label, tc_rate in TC_LEVELS.items():
        label = f"EW_{freq}_TC{tc_label}"
        ew_results[label] = run_equal_weight(
            returns, ALL_ASSETS, tc_rate=tc_rate,
            rebal_freq=freq, initial_capital=1e9)
        print(f"  ✓ {label}")

# ── 15c. Original 3-Stage Signal ─────────────────────────────────────────────
print("\n── Running Original 3-Stage Signal ──")
signal_results       = {}
signal_distributions = {}

for thresh in THRESHOLDS:
    sig_df_t    = build_signal_df(composite_score, equity_assets,
                                  macro_df.index, threshold=thresh)
    n_risk_on   = (sig_df_t[equity_assets[0]] == +1).sum()
    n_risk_off  = (sig_df_t[equity_assets[0]] == -1).sum()
    total       = len(sig_df_t)
    signal_distributions[thresh] = {'RISK_ON': n_risk_on, 'RISK_OFF': n_risk_off}
    print(f"  Threshold={thresh} | RISK_ON: {n_risk_on} ({n_risk_on/total*100:.1f}%) "
          f"| RISK_OFF: {n_risk_off} ({n_risk_off/total*100:.1f}%)")

    for freq in FREQS:
        for tc_label, tc_rate in TC_LEVELS.items():
            label = f"Sig3S_t{thresh}_{freq}_TC{tc_label}"

            def wfn_3s(sig_row, _eq=equity_assets, _ne=non_equity_assets,
                       _bw=base_weight, _sh=EQUITY_SAFE_HAVEN_MAP):
                return build_weights_3stage(sig_row, _eq, _ne, _bw, None, _sh)

            signal_results[label] = run_signal_portfolio(
                returns, sig_df_t, equity_assets, non_equity_assets,
                ALL_ASSETS, weight_fn=wfn_3s,
                freq=freq, tc_rate=tc_rate, initial_capital=1e9)
            print(f"    ✓ {label}")

# ── 15d. Asymmetric Signal ────────────────────────────────────────────────────
print("\n── Running Asymmetric Signal ──")
asym_results = {}

for thresh in THRESHOLDS:
    sig_df_t = build_signal_df_asymmetric(composite_score, equity_assets,
                                          macro_df.index, threshold=thresh)
    for freq in ['monthly', 'quarterly']:
        for tc_label, tc_rate in TC_LEVELS.items():
            label = f"Asym_t{thresh}_{freq}_TC{tc_label}"

            def wfn_asym(sig_row, _eq=equity_assets, _ne=non_equity_assets,
                         _bw=base_weight, _sh=EQUITY_SAFE_HAVEN_MAP):
                return build_weights_3stage(sig_row, _eq, _ne, _bw, None, _sh)

            asym_results[label] = run_signal_portfolio(
                returns, sig_df_t, equity_assets, non_equity_assets,
                ALL_ASSETS, weight_fn=wfn_asym,
                freq=freq, tc_rate=tc_rate, initial_capital=1e9)
            print(f"    ✓ {label}")

# ── 15e. Continuous / Graduated Signal ───────────────────────────────────────
print("\n── Running Continuous Signal ──")
cont_results = {}

for thresh in THRESHOLDS:
    for max_red in MAX_REDUCES:
        sig_df_t = build_signal_df_continuous(composite_score, equity_assets,
                                              macro_df.index,
                                              threshold=thresh,
                                              max_reduce=max_red)
        for freq in ['monthly', 'quarterly']:
            for tc_label, tc_rate in TC_LEVELS.items():
                label = (f"Cont_t{thresh}_mr{int(max_red*100)}"
                         f"_{freq}_TC{tc_label}")

                def wfn_cont(sig_row, _eq=equity_assets, _ne=non_equity_assets,
                             _bw=base_weight, _sh=EQUITY_SAFE_HAVEN_MAP):
                    return build_weights_continuous(sig_row, _eq, _ne, _bw, _sh)

                cont_results[label] = run_signal_portfolio(
                    returns, sig_df_t, equity_assets, non_equity_assets,
                    ALL_ASSETS, weight_fn=wfn_cont,
                    freq=freq, tc_rate=tc_rate, initial_capital=1e9)
                print(f"    ✓ {label}")

# ── 15f. Smoothed Signal ─────────────────────────────────────────────────────
print("\n── Running Smoothed Signal ──")
smooth_results = {}

for thresh in THRESHOLDS:
    for sw in SMOOTH_WINDOWS:
        sig_df_t = build_signal_df_smoothed(composite_score, equity_assets,
                                            macro_df.index,
                                            threshold=thresh,
                                            smooth_window=sw)
        for freq in ['monthly', 'quarterly']:
            for tc_label, tc_rate in TC_LEVELS.items():
                label = f"Smooth_t{thresh}_sw{sw}_{freq}_TC{tc_label}"

                def wfn_smooth(sig_row, _eq=equity_assets, _ne=non_equity_assets,
                               _bw=base_weight, _sh=EQUITY_SAFE_HAVEN_MAP):
                    return build_weights_3stage(sig_row, _eq, _ne, _bw, None, _sh)

                smooth_results[label] = run_signal_portfolio(
                    returns, sig_df_t, equity_assets, non_equity_assets,
                    ALL_ASSETS, weight_fn=wfn_smooth,
                    freq=freq, tc_rate=tc_rate, initial_capital=1e9)
                print(f"    ✓ {label}")

# ── 15g. Persistent Signal ────────────────────────────────────────────────────
print("\n── Running Persistent Signal ──")
persist_results = {}

for thresh in THRESHOLDS:
    for min_days in MIN_DAYS_LIST:
        sig_df_t = build_signal_df_persistent(composite_score, equity_assets,
                                              macro_df.index,
                                              threshold=thresh,
                                              min_days_to_switch=min_days)
        for freq in ['monthly', 'quarterly']:
            for tc_label, tc_rate in TC_LEVELS.items():
                label = f"Persist_t{thresh}_md{min_days}_{freq}_TC{tc_label}"

                def wfn_persist(sig_row, _eq=equity_assets, _ne=non_equity_assets,
                                _bw=base_weight, _sh=EQUITY_SAFE_HAVEN_MAP):
                    return build_weights_3stage(sig_row, _eq, _ne, _bw, None, _sh)

                persist_results[label] = run_signal_portfolio(
                    returns, sig_df_t, equity_assets, non_equity_assets,
                    ALL_ASSETS, weight_fn=wfn_persist,
                    freq=freq, tc_rate=tc_rate, initial_capital=1e9)
                print(f"    ✓ {label}")

# ── 15h. Partial-Hedge Signal ─────────────────────────────────────────────────
print("\n── Running Partial-Hedge Signal ──")
partial_results = {}

for thresh in THRESHOLDS:
    sig_df_t = build_signal_df(composite_score, equity_assets,
                               macro_df.index, threshold=thresh)
    for pk in PARTIAL_KEEPS:
        for freq in ['monthly', 'quarterly']:
            for tc_label, tc_rate in TC_LEVELS.items():
                label = (f"Partial_t{thresh}_pk{int(pk*100)}"
                         f"_{freq}_TC{tc_label}")

                def wfn_partial(sig_row, _eq=equity_assets, _ne=non_equity_assets,
                                _bw=base_weight, _sh=EQUITY_SAFE_HAVEN_MAP,
                                _pk=pk):
                    return build_weights_partial(sig_row, _eq, _ne, _bw, _sh, _pk)

                partial_results[label] = run_signal_portfolio(
                    returns, sig_df_t, equity_assets, non_equity_assets,
                    ALL_ASSETS, weight_fn=wfn_partial,
                    freq=freq, tc_rate=tc_rate, initial_capital=1e9)
                print(f"    ✓ {label}")

# ── 15i. Mean-Variance Signal ─────────────────────────────────────────────────
print("\n── Running Mean-Variance Signal ──")
mv_results        = {}
mv_weight_history = {}

for thresh in THRESHOLDS:
    sig_df_t = build_signal_df(composite_score, equity_assets,
                               macro_df.index, threshold=thresh)
    for freq in MV_FREQS:
        for tc_label, tc_rate in TC_LEVELS.items():
            label = f"MV_t{thresh}_{freq}_TC{tc_label}"
            pv_df, w_df = run_mv_signal_portfolio(
                returns, sig_df_t, ALL_ASSETS,
                equity_assets=equity_assets,
                freq=freq, tc_rate=tc_rate,
                lookback=MV_LOOKBACK, min_obs=MV_MIN_OBS,
                max_weight=MAX_WEIGHT, initial_capital=1e9)
            mv_results[label]        = pv_df
            mv_weight_history[label] = w_df
            print(f"  ✓ {label}")

print("\n✓ All portfolios computed")

# ── 16. PERFORMANCE METRICS ───────────────────────────────────────────────────
def compute_metrics(pv_df, label=''):
    pv        = pv_df['portfolio_value']
    daily_ret = pv.pct_change().dropna()
    n_years   = len(daily_ret) / 252
    total_ret = (pv.iloc[-1] / pv.iloc[0]) - 1
    cagr      = (1 + total_ret) ** (1 / n_years) - 1 if n_years > 0 else np.nan
    vol       = daily_ret.std() * np.sqrt(252)
    sharpe    = cagr / vol if vol > 0 else np.nan
    roll_max  = pv.cummax()
    max_dd    = ((pv - roll_max) / roll_max).min()
    calmar    = cagr / abs(max_dd) if max_dd != 0 else np.nan
    var_95    = daily_ret.quantile(0.05)
    ann_ret   = cagr
    ann_vol   = vol

    return {
        'CAGR':         f"{cagr*100:.2f}%",
        'Ann_Ret':      ann_ret,
        'Ann_Vol':      ann_vol,
        'Volatility':   f"{vol*100:.2f}%",
        'Sharpe':       f"{sharpe:.2f}",
        'Max Drawdown': f"{max_dd*100:.2f}%",
        'Calmar':       f"{calmar:.2f}",
        'VaR 95%':      f"{var_95*100:.2f}%",
        'Total Return': f"{total_ret*100:.2f}%",
        'Final ($B)':   f"{pv.iloc[-1]/1e9:.3f}",
    }

bh_metrics      = {k: compute_metrics(v, k) for k, v in bh_results.items()}
ew_metrics      = {k: compute_metrics(v, k) for k, v in ew_results.items()}
sig_metrics     = {k: compute_metrics(v, k) for k, v in signal_results.items()}
asym_metrics    = {k: compute_metrics(v, k) for k, v in asym_results.items()}
cont_metrics    = {k: compute_metrics(v, k) for k, v in cont_results.items()}
smooth_metrics  = {k: compute_metrics(v, k) for k, v in smooth_results.items()}
persist_metrics = {k: compute_metrics(v, k) for k, v in persist_results.items()}
partial_metrics = {k: compute_metrics(v, k) for k, v in partial_results.items()}
mv_metrics      = {k: compute_metrics(v, k) for k, v in mv_results.items()}

def print_metrics_table(metrics_dict, title):
    display = {k: {mk: mv for mk, mv in m.items()
                   if mk not in ('Ann_Ret', 'Ann_Vol')}
               for k, m in metrics_dict.items()}
    print(f"\n{'='*90}")
    print(title)
    print("="*90)
    print(pd.DataFrame(display).T.to_string())

print_metrics_table(bh_metrics,      "BUY-AND-HOLD")
print_metrics_table(ew_metrics,      "EQUAL-WEIGHT REBALANCED")

for thresh in THRESHOLDS:
    sub = {k: v for k, v in sig_metrics.items()     if f"_t{thresh}_" in k}
    print_metrics_table(sub, f"3-STAGE SIGNAL — threshold={thresh}")

for thresh in THRESHOLDS:
    sub = {k: v for k, v in asym_metrics.items()    if f"_t{thresh}_" in k}
    print_metrics_table(sub, f"ASYMMETRIC SIGNAL — threshold={thresh}")

for thresh in THRESHOLDS:
    sub = {k: v for k, v in cont_metrics.items()    if f"_t{thresh}_" in k}
    print_metrics_table(sub, f"CONTINUOUS SIGNAL — threshold={thresh}")

for thresh in THRESHOLDS:
    sub = {k: v for k, v in smooth_metrics.items()  if f"_t{thresh}_" in k}
    print_metrics_table(sub, f"SMOOTHED SIGNAL — threshold={thresh}")

for thresh in THRESHOLDS:
    sub = {k: v for k, v in persist_metrics.items() if f"_t{thresh}_" in k}
    print_metrics_table(sub, f"PERSISTENT SIGNAL — threshold={thresh}")

for thresh in THRESHOLDS:
    sub = {k: v for k, v in partial_metrics.items() if f"_t{thresh}_" in k}
    print_metrics_table(sub, f"PARTIAL-HEDGE SIGNAL — threshold={thresh}")

for thresh in THRESHOLDS:
    sub = {k: v for k, v in mv_metrics.items()      if f"_t{thresh}_" in k}
    print_metrics_table(sub, f"MEAN-VARIANCE SIGNAL — threshold={thresh}")

# ── 17. PLOTS ─────────────────────────────────────────────────────────────────
bh_pv     = bh_results['BuyAndHold_EW']['portfolio_value']
ew_ref_pv = ew_results['EW_monthly_TC0.5%']['portfolio_value']

def parse_pct(s):
    return float(str(s).replace('%', '')) / 100

# ── Plot A: Best of each variant vs benchmarks (monthly, TC=0.5%) ─────────────
fig, axes = plt.subplots(2, 4, figsize=(28, 12), sharey=False)
fig.suptitle("All Signal Variants vs Benchmarks — Monthly Rebalancing, TC=0.5%",
             fontsize=14, fontweight='bold')

variant_info = [
    ("3-Stage",    signal_results,  THRESHOLDS,     "Sig3S",   "steelblue"),
    ("Asymmetric", asym_results,    THRESHOLDS,     "Asym",    "darkorange"),
    ("Continuous", cont_results,    THRESHOLDS,     "Cont",    "purple"),
    ("Smoothed",   smooth_results,  THRESHOLDS,     "Smooth",  "green"),
    ("Persistent", persist_results, THRESHOLDS,     "Persist", "crimson"),
    ("Partial 25%",partial_results, THRESHOLDS,     "Partial", "brown"),
    ("MV Signal",  mv_results,      THRESHOLDS,     "MV",      "black"),
]

for ax, (name, res_dict, threshs, prefix, col) in zip(axes.flat, variant_info):
    ax.plot(bh_pv.index,     bh_pv / 1e9,     color='forestgreen',
            lw=1.5, ls='-.', label='Buy & Hold')
    ax.plot(ew_ref_pv.index, ew_ref_pv / 1e9, color='grey',
            lw=1.2, ls='--', label='EW Monthly')

    for thresh in threshs:
        candidates = [k for k in res_dict
                      if f"_t{thresh}_" in k and 'monthly' in k and 'TC0.5%' in k]
        if not candidates:
            candidates = [k for k in res_dict
                          if f"_t{thresh}_" in k and 'TC0.5%' in k]
        if candidates:
            key = candidates[0]
            pv  = res_dict[key]['portfolio_value']
            ax.plot(pv.index, pv / 1e9, lw=1.6,
                    label=f"t={thresh}")

    ax.axhline(1.0, color='black', ls=':', lw=0.7)
    ax.set_title(name, fontsize=11, fontweight='bold')
    ax.set_ylabel("Value ($B)")
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

axes.flat[-1].set_visible(False)
plt.tight_layout()
plt.savefig("all_variants_monthly.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Plot B: Risk-Return Scatter — all strategies ──────────────────────────────
fig, ax = plt.subplots(figsize=(16, 10))

all_metric_groups = [
    (bh_metrics,      'BH',       'forestgreen', 'D'),
    (ew_metrics,      'EW',       'grey',        's'),
    (sig_metrics,     '3-Stage',  'steelblue',   'o'),
    (asym_metrics,    'Asym',     'darkorange',  'o'),
    (cont_metrics,    'Cont',     'purple',      'o'),
    (smooth_metrics,  'Smooth',   'green',       'o'),
    (persist_metrics, 'Persist',  'crimson',     'o'),
    (partial_metrics, 'Partial',  'brown',       'o'),
    (mv_metrics,      'MV',       'black',       '^'),
]

for metrics_dict, grp_label, col, marker in all_metric_groups:
    first = True
    for key, m in metrics_dict.items():
        vol = m['Ann_Vol'] * 100
        ret = m['Ann_Ret'] * 100
        ax.scatter(vol, ret, color=col, marker=marker, s=80,
                   label=grp_label if first else None,
                   edgecolors='black', lw=0.5, zorder=4)
        ax.annotate(key.split('_TC')[0], (vol, ret),
                    textcoords='offset points', xytext=(4, 3),
                    fontsize=6, color=col)
        first = False

ax.axhline(0, color='black', lw=0.6, ls='--')
ax.axvline(0, color='black', lw=0.6, ls='--')
ax.set_xlabel("Annualised Volatility (%)", fontsize=12)
ax.set_ylabel("CAGR (%)", fontsize=12)
ax.set_title("Risk-Return — All Strategies", fontsize=13, fontweight='bold')
ax.legend(fontsize=10, ncol=3)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("risk_return_all.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Plot C: Sharpe Bar Chart — top 20 strategies ──────────────────────────────
all_metrics_combined = {
    **bh_metrics, **ew_metrics, **sig_metrics,
    **asym_metrics, **cont_metrics, **smooth_metrics,
    **persist_metrics, **partial_metrics, **mv_metrics
}

sharpe_series = pd.Series({
    k: float(v['Sharpe']) for k, v in all_metrics_combined.items()
}).dropna().sort_values(ascending=False)

top20 = sharpe_series.head(20)

def label_color(lbl):
    if lbl.startswith('MV'):       return 'black'
    if lbl.startswith('Sig3S'):    return 'steelblue'
    if lbl.startswith('Asym'):     return 'darkorange'
    if lbl.startswith('Cont'):     return 'purple'
    if lbl.startswith('Smooth'):   return 'green'
    if lbl.startswith('Persist'):  return 'crimson'
    if lbl.startswith('Partial'):  return 'brown'
    if lbl.startswith('EW'):       return 'grey'
    return 'forestgreen'

colors = [label_color(k) for k in top20.index]

fig, ax = plt.subplots(figsize=(16, 7))
bars = ax.barh(range(len(top20)), top20.values, color=colors, edgecolor='black', lw=0.5)
ax.set_yticks(range(len(top20)))
ax.set_yticklabels([k.replace('_TC', ' TC=') for k in top20.index], fontsize=8)
ax.set_xlabel("Sharpe Ratio")
ax.set_title("Top 20 Strategies by Sharpe Ratio", fontsize=13, fontweight='bold')
ax.axvline(0, color='black', lw=0.8)
ax.grid(True, axis='x', alpha=0.3)

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='black',      label='MV Signal'),
    Patch(facecolor='steelblue',  label='3-Stage'),
    Patch(facecolor='darkorange', label='Asymmetric'),
    Patch(facecolor='purple',     label='Continuous'),
    Patch(facecolor='green',      label='Smoothed'),
    Patch(facecolor='crimson',    label='Persistent'),
    Patch(facecolor='brown',      label='Partial Hedge'),
    Patch(facecolor='grey',       label='Equal Weight'),
    Patch(facecolor='forestgreen',label='Buy & Hold'),
]
ax.legend(handles=legend_elements, fontsize=9, loc='lower right')
plt.tight_layout()
plt.savefig("sharpe_top20.png", dpi=150, bbox_inches='tight')
plt.show()

# ── 18. SAVE ALL ──────────────────────────────────────────────────────────────
print("\n── Saving all results ──")
all_portfolios = {
    **bh_results, **ew_results, **signal_results,
    **asym_results, **cont_results, **smooth_results,
    **persist_results, **partial_results, **mv_results
}

for label, df in all_portfolios.items():
    df.to_csv(f"{label}_portfolio.csv")

all_metrics_df = pd.DataFrame({
    k: {mk: mv for mk, mv in m.items() if mk not in ('Ann_Ret', 'Ann_Vol')}
    for k, m in all_metrics_combined.items()
}).T
all_metrics_df.to_csv("all_portfolio_metrics.csv")

composite_score.to_csv("composite_signal.csv")
for thresh in THRESHOLDS:
    sig_df_t = build_signal_df(composite_score, equity_assets,
                               macro_df.index, threshold=thresh)
    sig_df_t.to_csv(f"signal_df_threshold_{thresh}.csv")

for label, wdf in mv_weight_history.items():
    wdf.to_csv(f"{label}_weights.csv")

print("✓ All files saved.")
print(f"\n  Portfolio CSVs   : {len(all_portfolios)}")
print(f"  Weight CSVs      : {len(mv_weight_history)}")
print(f"  Metrics CSV      : all_portfolio_metrics.csv")
print(f"  Signal CSVs      : {len(THRESHOLDS)} threshold files")
print(f"  Plots saved      : 3 PNG files")


In [ ]:
# @title
# ============================================================
# SIGNAL PORTFOLIO — ENHANCED VERSION
# Signals generated from Modified Determinant Model
# + Buy-and-Hold Benchmark
# + Equal-Weight Benchmark
# + Three-Stage Signal Portfolio (original)
# + Enhanced Signal Variants (asymmetric, continuous, smoothed, persistent, partial)
# + Mean-Variance Signal Portfolio
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import requests
from io import StringIO
from scipy.optimize import minimize

# ── 1. LOAD ALL DATA FROM GITHUB ─────────────────────────────────────────────
GITHUB_BASE = "https://raw.githubusercontent.com/kboroz/MSFE_Capstone_Project/main/01_Data/Streamlined/"

INDEX_FILE_MAP = {
    'ASX50':       'streamlined_asx_50.csv',
    'EUROSTOXX50': 'streamlined_euro_stoxx_50.csv',
    'FTSE100':     'streamlined_ftse_100.csv',
    'Ibovespa':    'streamlined_ibovespa.csv',
    'JSE40':       'streamlined_jse_top_40.csv',
    'NIKKEI225':   'streamlined_nikkei_225.csv',
    'SP500':       'streamlined_s&p_500.csv',
}

MACRO_FILE = 'streamlined_macro.csv'

def load_csv(url):
    r = requests.get(url)
    r.raise_for_status()
    df = pd.read_csv(StringIO(r.text), index_col=0, parse_dates=True)
    df.columns = df.columns.str.replace('[^a-zA-Z0-9]', '_', regex=True)
    return df.sort_index()

macro_df = load_csv(GITHUB_BASE + MACRO_FILE)
macro_df = macro_df[~macro_df.index.duplicated(keep='first')].sort_index()
print(f"macro_df shape : {macro_df.shape}")
print(f"Columns        : {macro_df.columns.tolist()}")

index_data = {}
for name, fname in INDEX_FILE_MAP.items():
    try:
        df = load_csv(GITHUB_BASE + fname)
        index_data[name] = df
        print(f"✓ {name:15s} | shape={df.shape} | cols={df.columns.tolist()[:4]}")
    except Exception as e:
        print(f"✗ {name}: {e}")

# ── 2. EXTRACT PRICE SERIES ───────────────────────────────────────────────────
PRICE_COLS = ['Close', 'close', 'Price', 'price', 'Adj_Close', 'Last']

def get_price_series(df, name):
    col = next((c for c in PRICE_COLS if c in df.columns), df.columns[0])
    s = pd.to_numeric(df[col], errors='coerce').dropna()
    print(f"  {name}: using column '{col}'")
    return s

index_prices = {}
for name, df in index_data.items():
    index_prices[name] = get_price_series(df, name)

# ── 3. MODIFIED DETERMINANT SIGNAL MODEL ─────────────────────────────────────
DET_WINDOWS = [3, 7, 28]
WINDOW_WEIGHTS = {3: 0.25, 7: 0.25, 28: 0.50}

def compute_modified_determinant(price_series_dict, windows=DET_WINDOWS):
    prices = pd.DataFrame(price_series_dict).sort_index()
    rets   = prices.pct_change().dropna()
    det_results = {}

    for window in windows:
        det_series = []
        for i in range(window, len(rets)):
            window_rets = rets.iloc[i - window:i]
            valid_cols  = window_rets.columns[window_rets.std() > 1e-10]
            w           = window_rets[valid_cols]

            if len(valid_cols) < 2:
                det_series.append(np.nan)
                continue

            try:
                R   = w.corr().values
                I   = np.eye(len(R))
                M   = R - I
                det = np.linalg.det(M)
            except Exception:
                det = np.nan

            det_series.append(det)

        idx = rets.index[window:]
        det_results[window] = pd.Series(det_series, index=idx,
                                        name=f'det_{window}d')
        print(f"  Window {window:2d}d | computed {len(det_series)} observations")

    return det_results

print("\n── Computing Modified Determinant ──")
det_results = compute_modified_determinant(index_prices)

# ── 4. COMPOSITE SCORE ────────────────────────────────────────────────────────
def compute_composite_score(det_results, window_weights=WINDOW_WEIGHTS):
    all_series = []
    for window, series in det_results.items():
        mu    = series.rolling(252, min_periods=60).mean()
        sigma = series.rolling(252, min_periods=60).std()
        z     = (series - mu) / sigma.replace(0, np.nan)
        z.name = f'z_{window}d'
        all_series.append(z)

    z_df      = pd.concat(all_series, axis=1).dropna()
    composite = sum(
        z_df[f'z_{w}d'] * window_weights[w]
        for w in window_weights if f'z_{w}d' in z_df.columns
    )
    composite = composite / sum(window_weights.values())
    return composite, z_df

composite_score, z_df = compute_composite_score(det_results)

print(f"\nComposite score shape  : {composite_score.shape}")
print(f"Date range             : {composite_score.index[0].date()} → "
      f"{composite_score.index[-1].date()}")
print(f"Score range            : {composite_score.min():.3f} → "
      f"{composite_score.max():.3f}")

# ── 5. SIGNAL BUILDERS ───────────────────────────────────────────────────────
THRESHOLDS    = [0.3, 0.5, 0.7]
SMOOTH_WINDOWS = [3, 5, 10]
MIN_DAYS_LIST  = [5, 10, 20]
MAX_REDUCES    = [0.5, 0.7, 0.9]
PARTIAL_KEEPS  = [0.25, 0.50]

# ── 5a. Original symmetric binary ────────────────────────────────────────────
def classify_signal_symmetric(score, threshold):
    return +1 if abs(score) > threshold else -1

def build_signal_df(composite_score, equity_assets, macro_dates, threshold=0.5):
    binary    = composite_score.apply(lambda x: classify_signal_symmetric(x, threshold))
    signal_df = pd.DataFrame(index=macro_dates)
    for asset in equity_assets:
        signal_df[asset] = binary.reindex(macro_dates).ffill().bfill()
    return signal_df

# ── 5b. Asymmetric (upper-tail only) ─────────────────────────────────────────
def build_signal_df_asymmetric(composite_score, equity_assets, macro_dates,
                               threshold=1.0):
    """Block equities only when correlation SPIKES (high positive z-score)."""
    binary    = composite_score.apply(lambda x: +1 if x > threshold else -1)
    signal_df = pd.DataFrame(index=macro_dates)
    for asset in equity_assets:
        signal_df[asset] = binary.reindex(macro_dates).ffill().bfill()
    return signal_df

# ── 5c. Continuous / graduated scalar ────────────────────────────────────────
def build_signal_df_continuous(composite_score, equity_assets, macro_dates,
                               threshold=1.0, max_reduce=0.8):
    """
    Returns a scalar in [1-max_reduce, 1.0] per asset per day.
    Used by build_weights_continuous() instead of the binary flag.
    """
    def score_to_scalar(s):
        if s <= 0:
            return 1.0
        elif s >= threshold:
            return 1.0 - max_reduce
        else:
            return 1.0 - max_reduce * (s / threshold)

    scalars   = composite_score.apply(score_to_scalar)
    signal_df = pd.DataFrame(index=macro_dates)
    for asset in equity_assets:
        signal_df[asset] = scalars.reindex(macro_dates).ffill().bfill()
    return signal_df

# ── 5d. Smoothed binary ───────────────────────────────────────────────────────
def build_signal_df_smoothed(composite_score, equity_assets, macro_dates,
                             threshold=1.0, smooth_window=5):
    """Smooth z-score before thresholding to reduce whipsaw."""
    smoothed  = composite_score.rolling(smooth_window, min_periods=1).mean()
    binary    = smoothed.apply(lambda x: classify_signal_symmetric(x, threshold))
    signal_df = pd.DataFrame(index=macro_dates)
    for asset in equity_assets:
        signal_df[asset] = binary.reindex(macro_dates).ffill().bfill()
    return signal_df

# ── 5e. Persistent binary ────────────────────────────────────────────────────
def build_signal_df_persistent(composite_score, equity_assets, macro_dates,
                               threshold=1.0, min_days_to_switch=10):
    """Only switch regime after signal holds for min_days_to_switch days."""
    raw       = composite_score.apply(lambda x: +1 if x > threshold else -1)
    confirmed = raw.copy()
    current   = int(raw.iloc[0])
    count     = 0

    for i in range(1, len(raw)):
        if int(raw.iloc[i]) != current:
            count += 1
            if count >= min_days_to_switch:
                current = int(raw.iloc[i])
                count   = 0
        else:
            count = 0
        confirmed.iloc[i] = current

    signal_df = pd.DataFrame(index=macro_dates)
    for asset in equity_assets:
        signal_df[asset] = confirmed.reindex(macro_dates).ffill().bfill()
    return signal_df

# ── 6. ASSET UNIVERSE ─────────────────────────────────────────────────────────
EQUITY_ASSETS = list(index_prices.keys())
SAFE_BONDS    = ['US_10Y', 'UK_10Y', 'AU_10Y', 'ZA_10Y']
FX            = ['AUDUSD', 'BRLUSD', 'EURUSD', 'GBPUSD', 'JPYUSD', 'ZARUSD']
COMMODITIES   = ['Oil', 'Gold', 'Bitcoin']
EXCLUDE_ASSETS = ['Bitcoin']

ALL_ASSETS = EQUITY_ASSETS + FX + COMMODITIES + SAFE_BONDS
ALL_ASSETS = [a for a in ALL_ASSETS
              if a in macro_df.columns and a not in EXCLUDE_ASSETS]

equity_assets     = [a for a in EQUITY_ASSETS     if a in ALL_ASSETS]
non_equity_assets = [a for a in ALL_ASSETS         if a not in equity_assets]

N           = len(ALL_ASSETS)
base_weight = 1.0 / N

print(f"\nTotal assets              : {N}")
print(f"Equity (signal-driven)    : {equity_assets}")
print(f"Non-equity (always long)  : {non_equity_assets}")
print(f"base_weight = 1/{N} = {base_weight:.4f}")

# ── 7. COUNTRY SAFE-HAVEN MAP ─────────────────────────────────────────────────
EQUITY_SAFE_HAVEN_MAP = {
    'ASX50':       ['AU_10Y', 'AUDUSD'],
    'EUROSTOXX50': ['EURUSD'],
    'FTSE100':     ['UK_10Y', 'GBPUSD'],
    'Ibovespa':    ['BRLUSD'],
    'JSE40':       ['ZA_10Y', 'ZARUSD'],
    'NIKKEI225':   ['JPYUSD'],
    'SP500':       ['US_10Y'],
}

UNIVERSE_SET = set(ALL_ASSETS)
for eq, havens in EQUITY_SAFE_HAVEN_MAP.items():
    EQUITY_SAFE_HAVEN_MAP[eq] = [h for h in havens if h in UNIVERSE_SET]

print("\nSafe-haven routing (filtered to universe):")
for eq, havens in EQUITY_SAFE_HAVEN_MAP.items():
    print(f"  {eq:15s} → {havens}")

# ── 8. BUILD RETURNS ──────────────────────────────────────────────────────────
returns_raw = macro_df[ALL_ASSETS].pct_change(fill_method=None)

for col in SAFE_BONDS:
    if col in returns_raw.columns:
        returns_raw[col] = -returns_raw[col]

if 'Oil' in macro_df.columns:
    bad_oil = macro_df['Oil'][macro_df['Oil'] <= 0].index
    for d in bad_oil:
        loc = returns_raw.index.get_loc(d)
        returns_raw.iloc[max(0, loc - 1): loc + 2,
                         returns_raw.columns.get_loc('Oil')] = 0.0

returns = returns_raw.clip(lower=-0.50, upper=1.00).fillna(0).iloc[1:]

assert not returns.isnull().any().any(), "NaNs in returns"
assert not np.isinf(returns.values).any(),  "Inf in returns"
print(f"\nReturns shape : {returns.shape}")
print("✓ Returns clean")

# ── 9. HELPER: REBALANCING DATE SET ──────────────────────────────────────────
def build_reb_set(returns, rebal_freq):
    actual_dates = returns.index
    if rebal_freq == 'daily':
        return set(actual_dates)
    elif rebal_freq == 'weekly':
        raw = returns.resample('W-MON').first().index
    elif rebal_freq == 'monthly':
        raw = returns.resample('MS').first().index
    elif rebal_freq == 'quarterly':
        raw = returns.resample('QS').first().index
    else:
        raise ValueError(f"Unknown rebal_freq: {rebal_freq}")

    reb_set = set()
    for d in raw:
        future = actual_dates[actual_dates >= d]
        if len(future) > 0:
            reb_set.add(future[0])
    return reb_set

# ── 10. BUY-AND-HOLD BENCHMARK ───────────────────────────────────────────────
def run_buy_and_hold(returns, all_assets, initial_capital=1e9):
    N           = len(all_assets)
    base_weight = 1.0 / N
    pv          = float(initial_capital)
    holdings    = {a: pv * base_weight for a in all_assets}
    records     = []
    first       = True

    for date in returns.index:
        if first:
            first = False
            records.append({'date': date, 'portfolio_value': pv})
            continue
        new_holdings = {a: holdings[a] * (1 + returns.loc[date, a])
                        for a in all_assets}
        holdings = new_holdings
        pv       = sum(holdings.values())
        records.append({'date': date, 'portfolio_value': pv})

    df = pd.DataFrame(records).set_index('date')
    print(f"  Buy-and-Hold | Start: ${initial_capital/1e9:.2f}B → "
          f"End: ${df['portfolio_value'].iloc[-1]/1e9:.3f}B")
    return df

# ── 11. WEIGHT BUILDERS ───────────────────────────────────────────────────────

# ── 11a. Original binary 3-stage weights ─────────────────────────────────────
def build_weights_3stage(sig_row, equity_assets, non_equity_assets,
                         base_weight, date, safe_haven_map):
    raw            = {a: base_weight for a in non_equity_assets}
    safe_haven_extra = {}

    for a in equity_assets:
        if sig_row is not None and a in sig_row.index and not pd.isna(sig_row[a]):
            sig = sig_row[a]
        else:
            sig = -1

        if sig > 0:
            raw[a] = -base_weight
            havens = [h for h in safe_haven_map.get(a, []) if h in raw]
            if havens:
                per_haven = base_weight / len(havens)
                for h in havens:
                    safe_haven_extra[h] = safe_haven_extra.get(h, 0) + per_haven
            else:
                per_ne = base_weight / len(non_equity_assets)
                for ne in non_equity_assets:
                    safe_haven_extra[ne] = safe_haven_extra.get(ne, 0) + per_ne
        else:
            raw[a] = base_weight

    for asset, extra in safe_haven_extra.items():
        raw[asset] = raw.get(asset, 0) + extra

    total = sum(raw.values())
    if total <= 0:
        for a in raw:
            raw[a] = 1.0 / len(raw)
    else:
        for a in raw:
            raw[a] /= total
    return raw

# ── 11b. Continuous scalar weights ───────────────────────────────────────────
def build_weights_continuous(sig_row, equity_assets, non_equity_assets,
                             base_weight, safe_haven_map):
    """
    sig_row contains a scalar in [min_weight, 1.0] per equity asset.
    Reduced equity weight is redistributed to safe havens.
    """
    raw              = {a: base_weight for a in non_equity_assets}
    safe_haven_extra = {}

    for a in equity_assets:
        scalar = float(sig_row[a]) if (sig_row is not None and
                                       a in sig_row.index and
                                       not pd.isna(sig_row[a])) else 1.0
        raw[a]   = base_weight * scalar
        released = base_weight * (1.0 - scalar)

        if released > 0:
            havens = [h for h in safe_haven_map.get(a, []) if h in raw]
            if havens:
                per_haven = released / len(havens)
                for h in havens:
                    safe_haven_extra[h] = safe_haven_extra.get(h, 0) + per_haven
            else:
                per_ne = released / len(non_equity_assets)
                for ne in non_equity_assets:
                    safe_haven_extra[ne] = safe_haven_extra.get(ne, 0) + per_ne

    for asset, extra in safe_haven_extra.items():
        raw[asset] = raw.get(asset, 0) + extra

    total = sum(raw.values())
    if total <= 0:
        for a in raw:
            raw[a] = 1.0 / len(raw)
    else:
        for a in raw:
            raw[a] /= total
    return raw

# ── 11c. Partial-hedge weights ────────────────────────────────────────────────
def build_weights_partial(sig_row, equity_assets, non_equity_assets,
                          base_weight, safe_haven_map, partial_keep=0.25):
    """
    On RISK_ON signal: keep partial_keep fraction of base_weight in equity,
    redirect the rest to safe havens.
    """
    raw              = {a: base_weight for a in non_equity_assets}
    safe_haven_extra = {}

    for a in equity_assets:
        if sig_row is not None and a in sig_row.index and not pd.isna(sig_row[a]):
            sig = sig_row[a]
        else:
            sig = -1

        if sig > 0:
            raw[a]   = base_weight * partial_keep
            released = base_weight * (1.0 - partial_keep)
            havens   = [h for h in safe_haven_map.get(a, []) if h in raw]
            if havens:
                per_haven = released / len(havens)
                for h in havens:
                    safe_haven_extra[h] = safe_haven_extra.get(h, 0) + per_haven
            else:
                per_ne = released / len(non_equity_assets)
                for ne in non_equity_assets:
                    safe_haven_extra[ne] = safe_haven_extra.get(ne, 0) + per_ne
        else:
            raw[a] = base_weight

    for asset, extra in safe_haven_extra.items():
        raw[asset] = raw.get(asset, 0) + extra

    total = sum(raw.values())
    if total <= 0:
        for a in raw:
            raw[a] = 1.0 / len(raw)
    else:
        for a in raw:
            raw[a] /= total
    return raw

# ── 12. EQUAL-WEIGHT REBALANCED BENCHMARK ────────────────────────────────────
def run_equal_weight(returns, all_assets, tc_rate=0.005,
                     rebal_freq='monthly', initial_capital=1e9):
    N           = len(all_assets)
    base_weight = 1.0 / N
    ew          = {a: base_weight for a in all_assets}
    reb_set     = build_reb_set(returns, rebal_freq)
    pv          = float(initial_capital)
    current_weights = None
    records     = []

    for date in returns.index:
        if date in reb_set:
            if current_weights is not None:
                turnover = sum(abs(ew.get(a, 0) - current_weights.get(a, 0))
                               for a in set(ew) | set(current_weights)) / 2.0
                pv *= (1 - turnover * tc_rate)
            else:
                pv *= (1 - tc_rate)
            current_weights = ew.copy()

        if current_weights is not None:
            dr  = sum(current_weights.get(a, 0) * returns.loc[date, a]
                      for a in all_assets)
            pv *= (1 + dr)

        records.append({'date': date, 'portfolio_value': pv})

    return pd.DataFrame(records).set_index('date')

# ── 13. GENERIC SIGNAL PORTFOLIO RUNNER ──────────────────────────────────────
def run_signal_portfolio(returns, signal_df, equity_assets,
                         non_equity_assets, all_assets,
                         weight_fn,          # callable: (sig_row, ...) → dict
                         freq='monthly', tc_rate=0.005,
                         initial_capital=1e9):
    """
    Generic runner.  weight_fn must accept (sig_row, equity_assets,
    non_equity_assets, base_weight, <any extra kwargs>) and return
    a dict {asset: weight}.
    """
    N           = len(all_assets)
    base_weight = 1.0 / N
    reb_set     = build_reb_set(returns, freq)

    sig = signal_df.copy()
    sig.index = pd.to_datetime(sig.index)

    pv              = float(initial_capital)
    current_weights = None
    records         = []

    for date in returns.index:
        if date in reb_set:
            past    = sig[sig.index <= date]
            sig_row = past.iloc[-1] if len(past) > 0 else None
            raw     = weight_fn(sig_row)

            if current_weights is not None:
                all_keys = set(raw) | set(current_weights)
                turnover = sum(abs(raw.get(a, 0) - current_weights.get(a, 0))
                               for a in all_keys) / 2.0
                pv *= (1 - turnover * tc_rate)
            else:
                pv *= (1 - tc_rate)

            current_weights = raw.copy()

        if current_weights is not None:
            dr  = sum(current_weights.get(a, 0) * returns.loc[date, a]
                      for a in all_assets)
            pv *= (1 + dr)

        records.append({'date': date, 'portfolio_value': pv})

    return pd.DataFrame(records).set_index('date')

# ── 14. MEAN-VARIANCE SIGNAL PORTFOLIO ───────────────────────────────────────
MV_LOOKBACK = 252
MV_MIN_OBS  = 60
MAX_WEIGHT  = 0.30
RISK_FREE   = 0.0

def mean_variance_weights(mu_vec, cov_mat, allowed_mask,
                          max_weight=MAX_WEIGHT, risk_free=RISK_FREE):
    n_total   = len(mu_vec)
    allowed   = np.where(allowed_mask)[0]
    n_allowed = len(allowed)

    fallback = np.zeros(n_total)
    if n_allowed == 0:
        fallback[:] = 1.0 / n_total
        return fallback
    fallback[allowed] = 1.0 / n_allowed

    mu_sub  = mu_vec[allowed]
    cov_sub = cov_mat[np.ix_(allowed, allowed)] + np.eye(n_allowed) * 1e-8

    def neg_sharpe(w):
        port_ret = np.dot(w, mu_sub)
        port_var = np.dot(w, cov_sub @ w)
        port_std = np.sqrt(max(port_var, 1e-12))
        return -(port_ret - risk_free) / port_std

    constraints = [{'type': 'eq', 'fun': lambda w: np.sum(w) - 1.0}]
    bounds      = [(0.0, max_weight)] * n_allowed
    w0          = np.ones(n_allowed) / n_allowed

    try:
        res = minimize(neg_sharpe, w0, method='SLSQP',
                       bounds=bounds, constraints=constraints,
                       options={'ftol': 1e-9, 'maxiter': 500})
        w_opt = res.x if (res.success and not np.any(np.isnan(res.x))) else w0
    except Exception:
        w_opt = w0

    w_full         = np.zeros(n_total)
    w_full[allowed] = w_opt
    return w_full

def run_mv_signal_portfolio(returns, signal_df, all_assets,
                            equity_assets,
                            freq='monthly', tc_rate=0.005,
                            lookback=MV_LOOKBACK, min_obs=MV_MIN_OBS,
                            max_weight=MAX_WEIGHT, initial_capital=1e9):
    reb_set      = build_reb_set(returns, freq)
    actual_dates = returns.index
    assets       = all_assets
    n            = len(assets)
    asset_idx    = {a: i for i, a in enumerate(assets)}

    sig = signal_df.copy()
    sig.index = pd.to_datetime(sig.index)

    pv              = float(initial_capital)
    current_weights = np.zeros(n)
    records         = []
    weight_history  = []
    first_reb       = True

    for t, date in enumerate(actual_dates):
        if date in reb_set:
            hist = returns.iloc[max(0, t - lookback): t]

            if len(hist) < min_obs:
                new_weights = np.ones(n) / n
            else:
                mu_ann  = hist.mean().values  * 252
                cov_ann = hist.cov().values   * 252

                past    = sig[sig.index <= date]
                sig_row = past.iloc[-1] if len(past) > 0 else None

                allowed_mask = np.ones(n, dtype=bool)
                if sig_row is not None:
                    for eq in equity_assets:
                        if eq in asset_idx and eq in sig_row.index:
                            s = sig_row[eq]
                            if not pd.isna(s) and s > 0:
                                allowed_mask[asset_idx[eq]] = False

                new_weights = mean_variance_weights(
                    mu_ann, cov_ann, allowed_mask, max_weight=max_weight)

            if first_reb:
                pv       *= (1 - tc_rate)
                first_reb = False
            else:
                turnover = np.sum(np.abs(new_weights - current_weights)) / 2.0
                pv      *= (1 - turnover * tc_rate)

            current_weights = new_weights.copy()
            weight_history.append({'date': date,
                                   **dict(zip(assets, current_weights))})

        if not first_reb:
            dr  = np.dot(current_weights, returns.loc[date].values)
            pv *= (1 + dr)

        records.append({'date': date, 'portfolio_value': pv})

    pv_df      = pd.DataFrame(records).set_index('date')
    weights_df = pd.DataFrame(weight_history).set_index('date')
    print(f"  MV | freq={freq} tc={tc_rate*100:.1f}% "
          f"End: ${pv_df['portfolio_value'].iloc[-1]/1e9:.3f}B")
    return pv_df, weights_df

# ── 15. RUN ALL PORTFOLIOS ────────────────────────────────────────────────────
TC_LEVELS = {'0%': 0.00, '0.5%': 0.005, '1%': 0.01, '2%': 0.02}
FREQS     = ['daily', 'weekly', 'monthly', 'quarterly']
MV_FREQS  = ['monthly', 'quarterly']

# ── 15a. Buy-and-Hold ─────────────────────────────────────────────────────────
print("\n── Running Buy-and-Hold ──")
bh_result  = run_buy_and_hold(returns, ALL_ASSETS, initial_capital=1e9)
bh_results = {'BuyAndHold_EW': bh_result}

# ── 15b. Equal-Weight ─────────────────────────────────────────────────────────
print("\n── Running Equal-Weight Rebalanced ──")
ew_results = {}
for freq in FREQS:
    for tc_label, tc_rate in TC_LEVELS.items():
        label = f"EW_{freq}_TC{tc_label}"
        ew_results[label] = run_equal_weight(
            returns, ALL_ASSETS, tc_rate=tc_rate,
            rebal_freq=freq, initial_capital=1e9)
        print(f"  ✓ {label}")

# ── 15c. Original 3-Stage Signal ─────────────────────────────────────────────
print("\n── Running Original 3-Stage Signal ──")
signal_results       = {}
signal_distributions = {}

for thresh in THRESHOLDS:
    sig_df_t    = build_signal_df(composite_score, equity_assets,
                                  macro_df.index, threshold=thresh)
    n_risk_on   = (sig_df_t[equity_assets[0]] == +1).sum()
    n_risk_off  = (sig_df_t[equity_assets[0]] == -1).sum()
    total       = len(sig_df_t)
    signal_distributions[thresh] = {'RISK_ON': n_risk_on, 'RISK_OFF': n_risk_off}
    print(f"  Threshold={thresh} | RISK_ON: {n_risk_on} ({n_risk_on/total*100:.1f}%) "
          f"| RISK_OFF: {n_risk_off} ({n_risk_off/total*100:.1f}%)")

    for freq in FREQS:
        for tc_label, tc_rate in TC_LEVELS.items():
            label = f"Sig3S_t{thresh}_{freq}_TC{tc_label}"

            def wfn_3s(sig_row, _eq=equity_assets, _ne=non_equity_assets,
                       _bw=base_weight, _sh=EQUITY_SAFE_HAVEN_MAP):
                return build_weights_3stage(sig_row, _eq, _ne, _bw, None, _sh)

            signal_results[label] = run_signal_portfolio(
                returns, sig_df_t, equity_assets, non_equity_assets,
                ALL_ASSETS, weight_fn=wfn_3s,
                freq=freq, tc_rate=tc_rate, initial_capital=1e9)
            print(f"    ✓ {label}")

# ── 15d. Asymmetric Signal ────────────────────────────────────────────────────
print("\n── Running Asymmetric Signal ──")
asym_results = {}

for thresh in THRESHOLDS:
    sig_df_t = build_signal_df_asymmetric(composite_score, equity_assets,
                                          macro_df.index, threshold=thresh)
    for freq in ['monthly', 'quarterly']:
        for tc_label, tc_rate in TC_LEVELS.items():
            label = f"Asym_t{thresh}_{freq}_TC{tc_label}"

            def wfn_asym(sig_row, _eq=equity_assets, _ne=non_equity_assets,
                         _bw=base_weight, _sh=EQUITY_SAFE_HAVEN_MAP):
                return build_weights_3stage(sig_row, _eq, _ne, _bw, None, _sh)

            asym_results[label] = run_signal_portfolio(
                returns, sig_df_t, equity_assets, non_equity_assets,
                ALL_ASSETS, weight_fn=wfn_asym,
                freq=freq, tc_rate=tc_rate, initial_capital=1e9)
            print(f"    ✓ {label}")

# ── 15e. Continuous / Graduated Signal ───────────────────────────────────────
print("\n── Running Continuous Signal ──")
cont_results = {}

for thresh in THRESHOLDS:
    for max_red in MAX_REDUCES:
        sig_df_t = build_signal_df_continuous(composite_score, equity_assets,
                                              macro_df.index,
                                              threshold=thresh,
                                              max_reduce=max_red)
        for freq in ['monthly', 'quarterly']:
            for tc_label, tc_rate in TC_LEVELS.items():
                label = (f"Cont_t{thresh}_mr{int(max_red*100)}"
                         f"_{freq}_TC{tc_label}")

                def wfn_cont(sig_row, _eq=equity_assets, _ne=non_equity_assets,
                             _bw=base_weight, _sh=EQUITY_SAFE_HAVEN_MAP):
                    return build_weights_continuous(sig_row, _eq, _ne, _bw, _sh)

                cont_results[label] = run_signal_portfolio(
                    returns, sig_df_t, equity_assets, non_equity_assets,
                    ALL_ASSETS, weight_fn=wfn_cont,
                    freq=freq, tc_rate=tc_rate, initial_capital=1e9)
                print(f"    ✓ {label}")

# ── 15f. Smoothed Signal ─────────────────────────────────────────────────────
print("\n── Running Smoothed Signal ──")
smooth_results = {}

for thresh in THRESHOLDS:
    for sw in SMOOTH_WINDOWS:
        sig_df_t = build_signal_df_smoothed(composite_score, equity_assets,
                                            macro_df.index,
                                            threshold=thresh,
                                            smooth_window=sw)
        for freq in ['monthly', 'quarterly']:
            for tc_label, tc_rate in TC_LEVELS.items():
                label = f"Smooth_t{thresh}_sw{sw}_{freq}_TC{tc_label}"

                def wfn_smooth(sig_row, _eq=equity_assets, _ne=non_equity_assets,
                               _bw=base_weight, _sh=EQUITY_SAFE_HAVEN_MAP):
                    return build_weights_3stage(sig_row, _eq, _ne, _bw, None, _sh)

                smooth_results[label] = run_signal_portfolio(
                    returns, sig_df_t, equity_assets, non_equity_assets,
                    ALL_ASSETS, weight_fn=wfn_smooth,
                    freq=freq, tc_rate=tc_rate, initial_capital=1e9)
                print(f"    ✓ {label}")

# ── 15g. Persistent Signal ────────────────────────────────────────────────────
print("\n── Running Persistent Signal ──")
persist_results = {}

for thresh in THRESHOLDS:
    for min_days in MIN_DAYS_LIST:
        sig_df_t = build_signal_df_persistent(composite_score, equity_assets,
                                              macro_df.index,
                                              threshold=thresh,
                                              min_days_to_switch=min_days)
        for freq in ['monthly', 'quarterly']:
            for tc_label, tc_rate in TC_LEVELS.items():
                label = f"Persist_t{thresh}_md{min_days}_{freq}_TC{tc_label}"

                def wfn_persist(sig_row, _eq=equity_assets, _ne=non_equity_assets,
                                _bw=base_weight, _sh=EQUITY_SAFE_HAVEN_MAP):
                    return build_weights_3stage(sig_row, _eq, _ne, _bw, None, _sh)

                persist_results[label] = run_signal_portfolio(
                    returns, sig_df_t, equity_assets, non_equity_assets,
                    ALL_ASSETS, weight_fn=wfn_persist,
                    freq=freq, tc_rate=tc_rate, initial_capital=1e9)
                print(f"    ✓ {label}")

# ── 15h. Partial-Hedge Signal ─────────────────────────────────────────────────
print("\n── Running Partial-Hedge Signal ──")
partial_results = {}

for thresh in THRESHOLDS:
    sig_df_t = build_signal_df(composite_score, equity_assets,
                               macro_df.index, threshold=thresh)
    for pk in PARTIAL_KEEPS:
        for freq in ['monthly', 'quarterly']:
            for tc_label, tc_rate in TC_LEVELS.items():
                label = (f"Partial_t{thresh}_pk{int(pk*100)}"
                         f"_{freq}_TC{tc_label}")

                def wfn_partial(sig_row, _eq=equity_assets, _ne=non_equity_assets,
                                _bw=base_weight, _sh=EQUITY_SAFE_HAVEN_MAP,
                                _pk=pk):
                    return build_weights_partial(sig_row, _eq, _ne, _bw, _sh, _pk)

                partial_results[label] = run_signal_portfolio(
                    returns, sig_df_t, equity_assets, non_equity_assets,
                    ALL_ASSETS, weight_fn=wfn_partial,
                    freq=freq, tc_rate=tc_rate, initial_capital=1e9)
                print(f"    ✓ {label}")

# ── 15i. Mean-Variance Signal ─────────────────────────────────────────────────
print("\n── Running Mean-Variance Signal ──")
mv_results        = {}
mv_weight_history = {}

for thresh in THRESHOLDS:
    sig_df_t = build_signal_df(composite_score, equity_assets,
                               macro_df.index, threshold=thresh)
    for freq in MV_FREQS:
        for tc_label, tc_rate in TC_LEVELS.items():
            label = f"MV_t{thresh}_{freq}_TC{tc_label}"
            pv_df, w_df = run_mv_signal_portfolio(
                returns, sig_df_t, ALL_ASSETS,
                equity_assets=equity_assets,
                freq=freq, tc_rate=tc_rate,
                lookback=MV_LOOKBACK, min_obs=MV_MIN_OBS,
                max_weight=MAX_WEIGHT, initial_capital=1e9)
            mv_results[label]        = pv_df
            mv_weight_history[label] = w_df
            print(f"  ✓ {label}")

print("\n✓ All portfolios computed")

# ── 16. PERFORMANCE METRICS ───────────────────────────────────────────────────
def compute_metrics(pv_df, label=''):
    pv        = pv_df['portfolio_value']
    daily_ret = pv.pct_change().dropna()
    n_years   = len(daily_ret) / 252
    total_ret = (pv.iloc[-1] / pv.iloc[0]) - 1
    cagr      = (1 + total_ret) ** (1 / n_years) - 1 if n_years > 0 else np.nan
    vol       = daily_ret.std() * np.sqrt(252)
    sharpe    = cagr / vol if vol > 0 else np.nan
    roll_max  = pv.cummax()
    max_dd    = ((pv - roll_max) / roll_max).min()
    calmar    = cagr / abs(max_dd) if max_dd != 0 else np.nan
    var_95    = daily_ret.quantile(0.05)
    ann_ret   = cagr
    ann_vol   = vol

    return {
        'CAGR':         f"{cagr*100:.2f}%",
        'Ann_Ret':      ann_ret,
        'Ann_Vol':      ann_vol,
        'Volatility':   f"{vol*100:.2f}%",
        'Sharpe':       f"{sharpe:.2f}",
        'Max Drawdown': f"{max_dd*100:.2f}%",
        'Calmar':       f"{calmar:.2f}",
        'VaR 95%':      f"{var_95*100:.2f}%",
        'Total Return': f"{total_ret*100:.2f}%",
        'Final ($B)':   f"{pv.iloc[-1]/1e9:.3f}",
    }

bh_metrics      = {k: compute_metrics(v, k) for k, v in bh_results.items()}
ew_metrics      = {k: compute_metrics(v, k) for k, v in ew_results.items()}
sig_metrics     = {k: compute_metrics(v, k) for k, v in signal_results.items()}
asym_metrics    = {k: compute_metrics(v, k) for k, v in asym_results.items()}
cont_metrics    = {k: compute_metrics(v, k) for k, v in cont_results.items()}
smooth_metrics  = {k: compute_metrics(v, k) for k, v in smooth_results.items()}
persist_metrics = {k: compute_metrics(v, k) for k, v in persist_results.items()}
partial_metrics = {k: compute_metrics(v, k) for k, v in partial_results.items()}
mv_metrics      = {k: compute_metrics(v, k) for k, v in mv_results.items()}

def print_metrics_table(metrics_dict, title):
    display = {k: {mk: mv for mk, mv in m.items()
                   if mk not in ('Ann_Ret', 'Ann_Vol')}
               for k, m in metrics_dict.items()}
    print(f"\n{'='*90}")
    print(title)
    print("="*90)
    print(pd.DataFrame(display).T.to_string())

print_metrics_table(bh_metrics,      "BUY-AND-HOLD")
print_metrics_table(ew_metrics,      "EQUAL-WEIGHT REBALANCED")

for thresh in THRESHOLDS:
    sub = {k: v for k, v in sig_metrics.items()     if f"_t{thresh}_" in k}
    print_metrics_table(sub, f"3-STAGE SIGNAL — threshold={thresh}")

for thresh in THRESHOLDS:
    sub = {k: v for k, v in asym_metrics.items()    if f"_t{thresh}_" in k}
    print_metrics_table(sub, f"ASYMMETRIC SIGNAL — threshold={thresh}")

for thresh in THRESHOLDS:
    sub = {k: v for k, v in cont_metrics.items()    if f"_t{thresh}_" in k}
    print_metrics_table(sub, f"CONTINUOUS SIGNAL — threshold={thresh}")

for thresh in THRESHOLDS:
    sub = {k: v for k, v in smooth_metrics.items()  if f"_t{thresh}_" in k}
    print_metrics_table(sub, f"SMOOTHED SIGNAL — threshold={thresh}")

for thresh in THRESHOLDS:
    sub = {k: v for k, v in persist_metrics.items() if f"_t{thresh}_" in k}
    print_metrics_table(sub, f"PERSISTENT SIGNAL — threshold={thresh}")

for thresh in THRESHOLDS:
    sub = {k: v for k, v in partial_metrics.items() if f"_t{thresh}_" in k}
    print_metrics_table(sub, f"PARTIAL-HEDGE SIGNAL — threshold={thresh}")

for thresh in THRESHOLDS:
    sub = {k: v for k, v in mv_metrics.items()      if f"_t{thresh}_" in k}
    print_metrics_table(sub, f"MEAN-VARIANCE SIGNAL — threshold={thresh}")

# ── 17. PLOTS ─────────────────────────────────────────────────────────────────
bh_pv     = bh_results['BuyAndHold_EW']['portfolio_value']
ew_ref_pv = ew_results['EW_monthly_TC0.5%']['portfolio_value']

def parse_pct(s):
    return float(str(s).replace('%', '')) / 100

# ── Plot A: Best of each variant vs benchmarks (monthly, TC=0.5%) ─────────────
fig, axes = plt.subplots(2, 4, figsize=(28, 12), sharey=False)
fig.suptitle("All Signal Variants vs Benchmarks — Monthly Rebalancing, TC=0.5%",
             fontsize=14, fontweight='bold')

variant_info = [
    ("3-Stage",    signal_results,  THRESHOLDS,     "Sig3S",   "steelblue"),
    ("Asymmetric", asym_results,    THRESHOLDS,     "Asym",    "darkorange"),
    ("Continuous", cont_results,    THRESHOLDS,     "Cont",    "purple"),
    ("Smoothed",   smooth_results,  THRESHOLDS,     "Smooth",  "green"),
    ("Persistent", persist_results, THRESHOLDS,     "Persist", "crimson"),
    ("Partial 25%",partial_results, THRESHOLDS,     "Partial", "brown"),
    ("MV Signal",  mv_results,      THRESHOLDS,     "MV",      "black"),
]

for ax, (name, res_dict, threshs, prefix, col) in zip(axes.flat, variant_info):
    ax.plot(bh_pv.index,     bh_pv / 1e9,     color='forestgreen',
            lw=1.5, ls='-.', label='Buy & Hold')
    ax.plot(ew_ref_pv.index, ew_ref_pv / 1e9, color='grey',
            lw=1.2, ls='--', label='EW Monthly')

    for thresh in threshs:
        candidates = [k for k in res_dict
                      if f"_t{thresh}_" in k and 'monthly' in k and 'TC0.5%' in k]
        if not candidates:
            candidates = [k for k in res_dict
                          if f"_t{thresh}_" in k and 'TC0.5%' in k]
        if candidates:
            key = candidates[0]
            pv  = res_dict[key]['portfolio_value']
            ax.plot(pv.index, pv / 1e9, lw=1.6,
                    label=f"t={thresh}")

    ax.axhline(1.0, color='black', ls=':', lw=0.7)
    ax.set_title(name, fontsize=11, fontweight='bold')
    ax.set_ylabel("Value ($B)")
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

axes.flat[-1].set_visible(False)
plt.tight_layout()
plt.savefig("all_variants_monthly.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Plot B: Risk-Return Scatter — all strategies ──────────────────────────────
fig, ax = plt.subplots(figsize=(16, 10))

all_metric_groups = [
    (bh_metrics,      'BH',       'forestgreen', 'D'),
    (ew_metrics,      'EW',       'grey',        's'),
    (sig_metrics,     '3-Stage',  'steelblue',   'o'),
    (asym_metrics,    'Asym',     'darkorange',  'o'),
    (cont_metrics,    'Cont',     'purple',      'o'),
    (smooth_metrics,  'Smooth',   'green',       'o'),
    (persist_metrics, 'Persist',  'crimson',     'o'),
    (partial_metrics, 'Partial',  'brown',       'o'),
    (mv_metrics,      'MV',       'black',       '^'),
]

for metrics_dict, grp_label, col, marker in all_metric_groups:
    first = True
    for key, m in metrics_dict.items():
        vol = m['Ann_Vol'] * 100
        ret = m['Ann_Ret'] * 100
        ax.scatter(vol, ret, color=col, marker=marker, s=80,
                   label=grp_label if first else None,
                   edgecolors='black', lw=0.5, zorder=4)
        ax.annotate(key.split('_TC')[0], (vol, ret),
                    textcoords='offset points', xytext=(4, 3),
                    fontsize=6, color=col)
        first = False

ax.axhline(0, color='black', lw=0.6, ls='--')
ax.axvline(0, color='black', lw=0.6, ls='--')
ax.set_xlabel("Annualised Volatility (%)", fontsize=12)
ax.set_ylabel("CAGR (%)", fontsize=12)
ax.set_title("Risk-Return — All Strategies", fontsize=13, fontweight='bold')
ax.legend(fontsize=10, ncol=3)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("risk_return_all.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Plot C: Sharpe Bar Chart — top 20 strategies ──────────────────────────────
all_metrics_combined = {
    **bh_metrics, **ew_metrics, **sig_metrics,
    **asym_metrics, **cont_metrics, **smooth_metrics,
    **persist_metrics, **partial_metrics, **mv_metrics
}

sharpe_series = pd.Series({
    k: float(v['Sharpe']) for k, v in all_metrics_combined.items()
}).dropna().sort_values(ascending=False)

top20 = sharpe_series.head(20)

def label_color(lbl):
    if lbl.startswith('MV'):       return 'black'
    if lbl.startswith('Sig3S'):    return 'steelblue'
    if lbl.startswith('Asym'):     return 'darkorange'
    if lbl.startswith('Cont'):     return 'purple'
    if lbl.startswith('Smooth'):   return 'green'
    if lbl.startswith('Persist'):  return 'crimson'
    if lbl.startswith('Partial'):  return 'brown'
    if lbl.startswith('EW'):       return 'grey'
    return 'forestgreen'

colors = [label_color(k) for k in top20.index]

fig, ax = plt.subplots(figsize=(16, 7))
bars = ax.barh(range(len(top20)), top20.values, color=colors, edgecolor='black', lw=0.5)
ax.set_yticks(range(len(top20)))
ax.set_yticklabels([k.replace('_TC', ' TC=') for k in top20.index], fontsize=8)
ax.set_xlabel("Sharpe Ratio")
ax.set_title("Top 20 Strategies by Sharpe Ratio", fontsize=13, fontweight='bold')
ax.axvline(0, color='black', lw=0.8)
ax.grid(True, axis='x', alpha=0.3)

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='black',      label='MV Signal'),
    Patch(facecolor='steelblue',  label='3-Stage'),
    Patch(facecolor='darkorange', label='Asymmetric'),
    Patch(facecolor='purple',     label='Continuous'),
    Patch(facecolor='green',      label='Smoothed'),
    Patch(facecolor='crimson',    label='Persistent'),
    Patch(facecolor='brown',      label='Partial Hedge'),
    Patch(facecolor='grey',       label='Equal Weight'),
    Patch(facecolor='forestgreen',label='Buy & Hold'),
]
ax.legend(handles=legend_elements, fontsize=9, loc='lower right')
plt.tight_layout()
plt.savefig("sharpe_top20.png", dpi=150, bbox_inches='tight')
plt.show()

# ── 18. SAVE ALL ──────────────────────────────────────────────────────────────
print("\n── Saving all results ──")
all_portfolios = {
    **bh_results, **ew_results, **signal_results,
    **asym_results, **cont_results, **smooth_results,
    **persist_results, **partial_results, **mv_results
}

for label, df in all_portfolios.items():
    df.to_csv(f"{label}_portfolio.csv")

all_metrics_df = pd.DataFrame({
    k: {mk: mv for mk, mv in m.items() if mk not in ('Ann_Ret', 'Ann_Vol')}
    for k, m in all_metrics_combined.items()
}).T
all_metrics_df.to_csv("all_portfolio_metrics.csv")

composite_score.to_csv("composite_signal.csv")
for thresh in THRESHOLDS:
    sig_df_t = build_signal_df(composite_score, equity_assets,
                               macro_df.index, threshold=thresh)
    sig_df_t.to_csv(f"signal_df_threshold_{thresh}.csv")

for label, wdf in mv_weight_history.items():
    wdf.to_csv(f"{label}_weights.csv")

print("✓ All files saved.")
print(f"\n  Portfolio CSVs   : {len(all_portfolios)}")
print(f"  Weight CSVs      : {len(mv_weight_history)}")
print(f"  Metrics CSV      : all_portfolio_metrics.csv")
print(f"  Signal CSVs      : {len(THRESHOLDS)} threshold files")
print(f"  Plots saved      : 3 PNG files")


In [ ]:
# @title
# ============================================================
# SIGNAL PORTFOLIO — ENHANCED VERSION
# Signals generated from Modified Determinant Model
# + Buy-and-Hold Benchmark
# + Equal-Weight Benchmark
# + Three-Stage Signal Portfolio (original)
# + Enhanced Signal Variants (asymmetric, continuous, smoothed, persistent, partial)
# + Mean-Variance Signal Portfolio
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import requests
from io import StringIO
from scipy.optimize import minimize

# ── 1. LOAD ALL DATA FROM GITHUB ─────────────────────────────────────────────
GITHUB_BASE = "https://raw.githubusercontent.com/kboroz/MSFE_Capstone_Project/main/01_Data/Streamlined/"

INDEX_FILE_MAP = {
    'ASX50':       'streamlined_asx_50.csv',
    'EUROSTOXX50': 'streamlined_euro_stoxx_50.csv',
    'FTSE100':     'streamlined_ftse_100.csv',
    'Ibovespa':    'streamlined_ibovespa.csv',
    'JSE40':       'streamlined_jse_top_40.csv',
    'NIKKEI225':   'streamlined_nikkei_225.csv',
    'SP500':       'streamlined_s&p_500.csv',
}

MACRO_FILE = 'streamlined_macro.csv'

def load_csv(url):
    r = requests.get(url)
    r.raise_for_status()
    df = pd.read_csv(StringIO(r.text), index_col=0, parse_dates=True)
    df.columns = df.columns.str.replace('[^a-zA-Z0-9]', '_', regex=True)
    return df.sort_index()

macro_df = load_csv(GITHUB_BASE + MACRO_FILE)
macro_df = macro_df[~macro_df.index.duplicated(keep='first')].sort_index()
print(f"macro_df shape : {macro_df.shape}")
print(f"Columns        : {macro_df.columns.tolist()}")

index_data = {}
for name, fname in INDEX_FILE_MAP.items():
    try:
        df = load_csv(GITHUB_BASE + fname)
        index_data[name] = df
        print(f"✓ {name:15s} | shape={df.shape} | cols={df.columns.tolist()[:4]}")
    except Exception as e:
        print(f"✗ {name}: {e}")

# ── 2. EXTRACT PRICE SERIES ───────────────────────────────────────────────────
PRICE_COLS = ['Close', 'close', 'Price', 'price', 'Adj_Close', 'Last']

def get_price_series(df, name):
    col = next((c for c in PRICE_COLS if c in df.columns), df.columns[0])
    s = pd.to_numeric(df[col], errors='coerce').dropna()
    print(f"  {name}: using column '{col}'")
    return s

index_prices = {}
for name, df in index_data.items():
    index_prices[name] = get_price_series(df, name)

# ── 3. MODIFIED DETERMINANT SIGNAL MODEL ─────────────────────────────────────
DET_WINDOWS = [3, 7, 28]
WINDOW_WEIGHTS = {3: 0.25, 7: 0.25, 28: 0.50}

def compute_modified_determinant(price_series_dict, windows=DET_WINDOWS):
    prices = pd.DataFrame(price_series_dict).sort_index()
    rets   = prices.pct_change().dropna()
    det_results = {}

    for window in windows:
        det_series = []
        for i in range(window, len(rets)):
            window_rets = rets.iloc[i - window:i]
            valid_cols  = window_rets.columns[window_rets.std() > 1e-10]
            w           = window_rets[valid_cols]

            if len(valid_cols) < 2:
                det_series.append(np.nan)
                continue

            try:
                R   = w.corr().values
                I   = np.eye(len(R))
                M   = R - I
                det = np.linalg.det(M)
            except Exception:
                det = np.nan

            det_series.append(det)

        idx = rets.index[window:]
        det_results[window] = pd.Series(det_series, index=idx,
                                        name=f'det_{window}d')
        print(f"  Window {window:2d}d | computed {len(det_series)} observations")

    return det_results

print("\n── Computing Modified Determinant ──")
det_results = compute_modified_determinant(index_prices)

# ── 4. COMPOSITE SCORE ────────────────────────────────────────────────────────
def compute_composite_score(det_results, window_weights=WINDOW_WEIGHTS):
    all_series = []
    for window, series in det_results.items():
        mu    = series.rolling(252, min_periods=60).mean()
        sigma = series.rolling(252, min_periods=60).std()
        z     = (series - mu) / sigma.replace(0, np.nan)
        z.name = f'z_{window}d'
        all_series.append(z)

    z_df      = pd.concat(all_series, axis=1).dropna()
    composite = sum(
        z_df[f'z_{w}d'] * window_weights[w]
        for w in window_weights if f'z_{w}d' in z_df.columns
    )
    composite = composite / sum(window_weights.values())
    return composite, z_df

composite_score, z_df = compute_composite_score(det_results)

print(f"\nComposite score shape  : {composite_score.shape}")
print(f"Date range             : {composite_score.index[0].date()} → "
      f"{composite_score.index[-1].date()}")
print(f"Score range            : {composite_score.min():.3f} → "
      f"{composite_score.max():.3f}")

# ── 5. SIGNAL BUILDERS ───────────────────────────────────────────────────────
THRESHOLDS    = [0.3, 0.6, 0.9]
SMOOTH_WINDOWS = [3, 5, 10]
MIN_DAYS_LIST  = [5, 10, 20]
MAX_REDUCES    = [0.5, 0.7, 0.9]
PARTIAL_KEEPS  = [0.25, 0.50]

# ── 5a. Original symmetric binary ────────────────────────────────────────────
def classify_signal_symmetric(score, threshold):
    return +1 if abs(score) > threshold else -1

def build_signal_df(composite_score, equity_assets, macro_dates, threshold=0.5):
    binary    = composite_score.apply(lambda x: classify_signal_symmetric(x, threshold))
    signal_df = pd.DataFrame(index=macro_dates)
    for asset in equity_assets:
        signal_df[asset] = binary.reindex(macro_dates).ffill().bfill()
    return signal_df

# ── 5b. Asymmetric (upper-tail only) ─────────────────────────────────────────
def build_signal_df_asymmetric(composite_score, equity_assets, macro_dates,
                               threshold=1.0):
    """Block equities only when correlation SPIKES (high positive z-score)."""
    binary    = composite_score.apply(lambda x: +1 if x > threshold else -1)
    signal_df = pd.DataFrame(index=macro_dates)
    for asset in equity_assets:
        signal_df[asset] = binary.reindex(macro_dates).ffill().bfill()
    return signal_df

# ── 5c. Continuous / graduated scalar ────────────────────────────────────────
def build_signal_df_continuous(composite_score, equity_assets, macro_dates,
                               threshold=1.0, max_reduce=0.8):
    """
    Returns a scalar in [1-max_reduce, 1.0] per asset per day.
    Used by build_weights_continuous() instead of the binary flag.
    """
    def score_to_scalar(s):
        if s <= 0:
            return 1.0
        elif s >= threshold:
            return 1.0 - max_reduce
        else:
            return 1.0 - max_reduce * (s / threshold)

    scalars   = composite_score.apply(score_to_scalar)
    signal_df = pd.DataFrame(index=macro_dates)
    for asset in equity_assets:
        signal_df[asset] = scalars.reindex(macro_dates).ffill().bfill()
    return signal_df

# ── 5d. Smoothed binary ───────────────────────────────────────────────────────
def build_signal_df_smoothed(composite_score, equity_assets, macro_dates,
                             threshold=1.0, smooth_window=5):
    """Smooth z-score before thresholding to reduce whipsaw."""
    smoothed  = composite_score.rolling(smooth_window, min_periods=1).mean()
    binary    = smoothed.apply(lambda x: classify_signal_symmetric(x, threshold))
    signal_df = pd.DataFrame(index=macro_dates)
    for asset in equity_assets:
        signal_df[asset] = binary.reindex(macro_dates).ffill().bfill()
    return signal_df

# ── 5e. Persistent binary ────────────────────────────────────────────────────
def build_signal_df_persistent(composite_score, equity_assets, macro_dates,
                               threshold=1.0, min_days_to_switch=10):
    """Only switch regime after signal holds for min_days_to_switch days."""
    raw       = composite_score.apply(lambda x: +1 if x > threshold else -1)
    confirmed = raw.copy()
    current   = int(raw.iloc[0])
    count     = 0

    for i in range(1, len(raw)):
        if int(raw.iloc[i]) != current:
            count += 1
            if count >= min_days_to_switch:
                current = int(raw.iloc[i])
                count   = 0
        else:
            count = 0
        confirmed.iloc[i] = current

    signal_df = pd.DataFrame(index=macro_dates)
    for asset in equity_assets:
        signal_df[asset] = confirmed.reindex(macro_dates).ffill().bfill()
    return signal_df

# ── 6. ASSET UNIVERSE ─────────────────────────────────────────────────────────
EQUITY_ASSETS = list(index_prices.keys())
SAFE_BONDS    = ['US_10Y', 'UK_10Y', 'AU_10Y', 'ZA_10Y']
FX            = ['AUDUSD', 'BRLUSD', 'EURUSD', 'GBPUSD', 'JPYUSD', 'ZARUSD']
COMMODITIES   = ['Oil', 'Gold', 'Bitcoin']
EXCLUDE_ASSETS = ['Bitcoin']

ALL_ASSETS = EQUITY_ASSETS + FX + COMMODITIES + SAFE_BONDS
ALL_ASSETS = [a for a in ALL_ASSETS
              if a in macro_df.columns and a not in EXCLUDE_ASSETS]

equity_assets     = [a for a in EQUITY_ASSETS     if a in ALL_ASSETS]
non_equity_assets = [a for a in ALL_ASSETS         if a not in equity_assets]

N           = len(ALL_ASSETS)
base_weight = 1.0 / N

print(f"\nTotal assets              : {N}")
print(f"Equity (signal-driven)    : {equity_assets}")
print(f"Non-equity (always long)  : {non_equity_assets}")
print(f"base_weight = 1/{N} = {base_weight:.4f}")

# ── 7. COUNTRY SAFE-HAVEN MAP ─────────────────────────────────────────────────
EQUITY_SAFE_HAVEN_MAP = {
    'ASX50':       ['AU_10Y', 'AUDUSD'],
    'EUROSTOXX50': ['EURUSD'],
    'FTSE100':     ['UK_10Y', 'GBPUSD'],
    'Ibovespa':    ['BRLUSD'],
    'JSE40':       ['ZA_10Y', 'ZARUSD'],
    'NIKKEI225':   ['JPYUSD'],
    'SP500':       ['US_10Y'],
}

UNIVERSE_SET = set(ALL_ASSETS)
for eq, havens in EQUITY_SAFE_HAVEN_MAP.items():
    EQUITY_SAFE_HAVEN_MAP[eq] = [h for h in havens if h in UNIVERSE_SET]

print("\nSafe-haven routing (filtered to universe):")
for eq, havens in EQUITY_SAFE_HAVEN_MAP.items():
    print(f"  {eq:15s} → {havens}")

# ── 8. BUILD RETURNS ──────────────────────────────────────────────────────────
returns_raw = macro_df[ALL_ASSETS].pct_change(fill_method=None)

for col in SAFE_BONDS:
    if col in returns_raw.columns:
        returns_raw[col] = -returns_raw[col]

if 'Oil' in macro_df.columns:
    bad_oil = macro_df['Oil'][macro_df['Oil'] <= 0].index
    for d in bad_oil:
        loc = returns_raw.index.get_loc(d)
        returns_raw.iloc[max(0, loc - 1): loc + 2,
                         returns_raw.columns.get_loc('Oil')] = 0.0

returns = returns_raw.clip(lower=-0.50, upper=1.00).fillna(0).iloc[1:]

assert not returns.isnull().any().any(), "NaNs in returns"
assert not np.isinf(returns.values).any(),  "Inf in returns"
print(f"\nReturns shape : {returns.shape}")
print("✓ Returns clean")

# ── 9. HELPER: REBALANCING DATE SET ──────────────────────────────────────────
def build_reb_set(returns, rebal_freq):
    actual_dates = returns.index
    if rebal_freq == 'daily':
        return set(actual_dates)
    elif rebal_freq == 'weekly':
        raw = returns.resample('W-MON').first().index
    elif rebal_freq == 'monthly':
        raw = returns.resample('MS').first().index
    elif rebal_freq == 'quarterly':
        raw = returns.resample('QS').first().index
    else:
        raise ValueError(f"Unknown rebal_freq: {rebal_freq}")

    reb_set = set()
    for d in raw:
        future = actual_dates[actual_dates >= d]
        if len(future) > 0:
            reb_set.add(future[0])
    return reb_set

# ── 10. BUY-AND-HOLD BENCHMARK ───────────────────────────────────────────────
def run_buy_and_hold(returns, all_assets, initial_capital=1e9):
    N           = len(all_assets)
    base_weight = 1.0 / N
    pv          = float(initial_capital)
    holdings    = {a: pv * base_weight for a in all_assets}
    records     = []
    first       = True

    for date in returns.index:
        if first:
            first = False
            records.append({'date': date, 'portfolio_value': pv})
            continue
        new_holdings = {a: holdings[a] * (1 + returns.loc[date, a])
                        for a in all_assets}
        holdings = new_holdings
        pv       = sum(holdings.values())
        records.append({'date': date, 'portfolio_value': pv})

    df = pd.DataFrame(records).set_index('date')
    print(f"  Buy-and-Hold | Start: ${initial_capital/1e9:.2f}B → "
          f"End: ${df['portfolio_value'].iloc[-1]/1e9:.3f}B")
    return df

# ── 11. WEIGHT BUILDERS ───────────────────────────────────────────────────────

# ── 11a. Original binary 3-stage weights ─────────────────────────────────────
def build_weights_3stage(sig_row, equity_assets, non_equity_assets,
                         base_weight, date, safe_haven_map):
    raw            = {a: base_weight for a in non_equity_assets}
    safe_haven_extra = {}

    for a in equity_assets:
        if sig_row is not None and a in sig_row.index and not pd.isna(sig_row[a]):
            sig = sig_row[a]
        else:
            sig = -1

        if sig > 0:
            raw[a] = -base_weight
            havens = [h for h in safe_haven_map.get(a, []) if h in raw]
            if havens:
                per_haven = base_weight / len(havens)
                for h in havens:
                    safe_haven_extra[h] = safe_haven_extra.get(h, 0) + per_haven
            else:
                per_ne = base_weight / len(non_equity_assets)
                for ne in non_equity_assets:
                    safe_haven_extra[ne] = safe_haven_extra.get(ne, 0) + per_ne
        else:
            raw[a] = base_weight

    for asset, extra in safe_haven_extra.items():
        raw[asset] = raw.get(asset, 0) + extra

    total = sum(raw.values())
    if total <= 0:
        for a in raw:
            raw[a] = 1.0 / len(raw)
    else:
        for a in raw:
            raw[a] /= total
    return raw

# ── 11b. Continuous scalar weights ───────────────────────────────────────────
def build_weights_continuous(sig_row, equity_assets, non_equity_assets,
                             base_weight, safe_haven_map):
    """
    sig_row contains a scalar in [min_weight, 1.0] per equity asset.
    Reduced equity weight is redistributed to safe havens.
    """
    raw              = {a: base_weight for a in non_equity_assets}
    safe_haven_extra = {}

    for a in equity_assets:
        scalar = float(sig_row[a]) if (sig_row is not None and
                                       a in sig_row.index and
                                       not pd.isna(sig_row[a])) else 1.0
        raw[a]   = base_weight * scalar
        released = base_weight * (1.0 - scalar)

        if released > 0:
            havens = [h for h in safe_haven_map.get(a, []) if h in raw]
            if havens:
                per_haven = released / len(havens)
                for h in havens:
                    safe_haven_extra[h] = safe_haven_extra.get(h, 0) + per_haven
            else:
                per_ne = released / len(non_equity_assets)
                for ne in non_equity_assets:
                    safe_haven_extra[ne] = safe_haven_extra.get(ne, 0) + per_ne

    for asset, extra in safe_haven_extra.items():
        raw[asset] = raw.get(asset, 0) + extra

    total = sum(raw.values())
    if total <= 0:
        for a in raw:
            raw[a] = 1.0 / len(raw)
    else:
        for a in raw:
            raw[a] /= total
    return raw

# ── 11c. Partial-hedge weights ────────────────────────────────────────────────
def build_weights_partial(sig_row, equity_assets, non_equity_assets,
                          base_weight, safe_haven_map, partial_keep=0.25):
    """
    On RISK_ON signal: keep partial_keep fraction of base_weight in equity,
    redirect the rest to safe havens.
    """
    raw              = {a: base_weight for a in non_equity_assets}
    safe_haven_extra = {}

    for a in equity_assets:
        if sig_row is not None and a in sig_row.index and not pd.isna(sig_row[a]):
            sig = sig_row[a]
        else:
            sig = -1

        if sig > 0:
            raw[a]   = base_weight * partial_keep
            released = base_weight * (1.0 - partial_keep)
            havens   = [h for h in safe_haven_map.get(a, []) if h in raw]
            if havens:
                per_haven = released / len(havens)
                for h in havens:
                    safe_haven_extra[h] = safe_haven_extra.get(h, 0) + per_haven
            else:
                per_ne = released / len(non_equity_assets)
                for ne in non_equity_assets:
                    safe_haven_extra[ne] = safe_haven_extra.get(ne, 0) + per_ne
        else:
            raw[a] = base_weight

    for asset, extra in safe_haven_extra.items():
        raw[asset] = raw.get(asset, 0) + extra

    total = sum(raw.values())
    if total <= 0:
        for a in raw:
            raw[a] = 1.0 / len(raw)
    else:
        for a in raw:
            raw[a] /= total
    return raw

# ── 12. EQUAL-WEIGHT REBALANCED BENCHMARK ────────────────────────────────────
def run_equal_weight(returns, all_assets, tc_rate=0.005,
                     rebal_freq='monthly', initial_capital=1e9):
    N           = len(all_assets)
    base_weight = 1.0 / N
    ew          = {a: base_weight for a in all_assets}
    reb_set     = build_reb_set(returns, rebal_freq)
    pv          = float(initial_capital)
    current_weights = None
    records     = []

    for date in returns.index:
        if date in reb_set:
            if current_weights is not None:
                turnover = sum(abs(ew.get(a, 0) - current_weights.get(a, 0))
                               for a in set(ew) | set(current_weights)) / 2.0
                pv *= (1 - turnover * tc_rate)
            else:
                pv *= (1 - tc_rate)
            current_weights = ew.copy()

        if current_weights is not None:
            dr  = sum(current_weights.get(a, 0) * returns.loc[date, a]
                      for a in all_assets)
            pv *= (1 + dr)

        records.append({'date': date, 'portfolio_value': pv})

    return pd.DataFrame(records).set_index('date')

# ── 13. GENERIC SIGNAL PORTFOLIO RUNNER ──────────────────────────────────────
def run_signal_portfolio(returns, signal_df, equity_assets,
                         non_equity_assets, all_assets,
                         weight_fn,          # callable: (sig_row, ...) → dict
                         freq='monthly', tc_rate=0.005,
                         initial_capital=1e9):
    """
    Generic runner.  weight_fn must accept (sig_row, equity_assets,
    non_equity_assets, base_weight, <any extra kwargs>) and return
    a dict {asset: weight}.
    """
    N           = len(all_assets)
    base_weight = 1.0 / N
    reb_set     = build_reb_set(returns, freq)

    sig = signal_df.copy()
    sig.index = pd.to_datetime(sig.index)

    pv              = float(initial_capital)
    current_weights = None
    records         = []

    for date in returns.index:
        if date in reb_set:
            past    = sig[sig.index <= date]
            sig_row = past.iloc[-1] if len(past) > 0 else None
            raw     = weight_fn(sig_row)

            if current_weights is not None:
                all_keys = set(raw) | set(current_weights)
                turnover = sum(abs(raw.get(a, 0) - current_weights.get(a, 0))
                               for a in all_keys) / 2.0
                pv *= (1 - turnover * tc_rate)
            else:
                pv *= (1 - tc_rate)

            current_weights = raw.copy()

        if current_weights is not None:
            dr  = sum(current_weights.get(a, 0) * returns.loc[date, a]
                      for a in all_assets)
            pv *= (1 + dr)

        records.append({'date': date, 'portfolio_value': pv})

    return pd.DataFrame(records).set_index('date')

# ── 14. MEAN-VARIANCE SIGNAL PORTFOLIO ───────────────────────────────────────
MV_LOOKBACK = 252
MV_MIN_OBS  = 60
MAX_WEIGHT  = 0.30
RISK_FREE   = 0.0

def mean_variance_weights(mu_vec, cov_mat, allowed_mask,
                          max_weight=MAX_WEIGHT, risk_free=RISK_FREE):
    n_total   = len(mu_vec)
    allowed   = np.where(allowed_mask)[0]
    n_allowed = len(allowed)

    fallback = np.zeros(n_total)
    if n_allowed == 0:
        fallback[:] = 1.0 / n_total
        return fallback
    fallback[allowed] = 1.0 / n_allowed

    mu_sub  = mu_vec[allowed]
    cov_sub = cov_mat[np.ix_(allowed, allowed)] + np.eye(n_allowed) * 1e-8

    def neg_sharpe(w):
        port_ret = np.dot(w, mu_sub)
        port_var = np.dot(w, cov_sub @ w)
        port_std = np.sqrt(max(port_var, 1e-12))
        return -(port_ret - risk_free) / port_std

    constraints = [{'type': 'eq', 'fun': lambda w: np.sum(w) - 1.0}]
    bounds      = [(0.0, max_weight)] * n_allowed
    w0          = np.ones(n_allowed) / n_allowed

    try:
        res = minimize(neg_sharpe, w0, method='SLSQP',
                       bounds=bounds, constraints=constraints,
                       options={'ftol': 1e-9, 'maxiter': 500})
        w_opt = res.x if (res.success and not np.any(np.isnan(res.x))) else w0
    except Exception:
        w_opt = w0

    w_full         = np.zeros(n_total)
    w_full[allowed] = w_opt
    return w_full

def run_mv_signal_portfolio(returns, signal_df, all_assets,
                            equity_assets,
                            freq='monthly', tc_rate=0.005,
                            lookback=MV_LOOKBACK, min_obs=MV_MIN_OBS,
                            max_weight=MAX_WEIGHT, initial_capital=1e9):
    reb_set      = build_reb_set(returns, freq)
    actual_dates = returns.index
    assets       = all_assets
    n            = len(assets)
    asset_idx    = {a: i for i, a in enumerate(assets)}

    sig = signal_df.copy()
    sig.index = pd.to_datetime(sig.index)

    pv              = float(initial_capital)
    current_weights = np.zeros(n)
    records         = []
    weight_history  = []
    first_reb       = True

    for t, date in enumerate(actual_dates):
        if date in reb_set:
            hist = returns.iloc[max(0, t - lookback): t]

            if len(hist) < min_obs:
                new_weights = np.ones(n) / n
            else:
                mu_ann  = hist.mean().values  * 252
                cov_ann = hist.cov().values   * 252

                past    = sig[sig.index <= date]
                sig_row = past.iloc[-1] if len(past) > 0 else None

                allowed_mask = np.ones(n, dtype=bool)
                if sig_row is not None:
                    for eq in equity_assets:
                        if eq in asset_idx and eq in sig_row.index:
                            s = sig_row[eq]
                            if not pd.isna(s) and s > 0:
                                allowed_mask[asset_idx[eq]] = False

                new_weights = mean_variance_weights(
                    mu_ann, cov_ann, allowed_mask, max_weight=max_weight)

            if first_reb:
                pv       *= (1 - tc_rate)
                first_reb = False
            else:
                turnover = np.sum(np.abs(new_weights - current_weights)) / 2.0
                pv      *= (1 - turnover * tc_rate)

            current_weights = new_weights.copy()
            weight_history.append({'date': date,
                                   **dict(zip(assets, current_weights))})

        if not first_reb:
            dr  = np.dot(current_weights, returns.loc[date].values)
            pv *= (1 + dr)

        records.append({'date': date, 'portfolio_value': pv})

    pv_df      = pd.DataFrame(records).set_index('date')
    weights_df = pd.DataFrame(weight_history).set_index('date')
    print(f"  MV | freq={freq} tc={tc_rate*100:.1f}% "
          f"End: ${pv_df['portfolio_value'].iloc[-1]/1e9:.3f}B")
    return pv_df, weights_df

# ── 15. RUN ALL PORTFOLIOS ────────────────────────────────────────────────────
TC_LEVELS = {'0%': 0.00, '0.5%': 0.005, '1%': 0.01, '2%': 0.02}
FREQS     = ['daily', 'weekly', 'monthly', 'quarterly']
MV_FREQS  = ['monthly', 'quarterly']

# ── 15a. Buy-and-Hold ─────────────────────────────────────────────────────────
print("\n── Running Buy-and-Hold ──")
bh_result  = run_buy_and_hold(returns, ALL_ASSETS, initial_capital=1e9)
bh_results = {'BuyAndHold_EW': bh_result}

# ── 15b. Equal-Weight ─────────────────────────────────────────────────────────
print("\n── Running Equal-Weight Rebalanced ──")
ew_results = {}
for freq in FREQS:
    for tc_label, tc_rate in TC_LEVELS.items():
        label = f"EW_{freq}_TC{tc_label}"
        ew_results[label] = run_equal_weight(
            returns, ALL_ASSETS, tc_rate=tc_rate,
            rebal_freq=freq, initial_capital=1e9)
        print(f"  ✓ {label}")

# ── 15c. Original 3-Stage Signal ─────────────────────────────────────────────
print("\n── Running Original 3-Stage Signal ──")
signal_results       = {}
signal_distributions = {}

for thresh in THRESHOLDS:
    sig_df_t    = build_signal_df(composite_score, equity_assets,
                                  macro_df.index, threshold=thresh)
    n_risk_on   = (sig_df_t[equity_assets[0]] == +1).sum()
    n_risk_off  = (sig_df_t[equity_assets[0]] == -1).sum()
    total       = len(sig_df_t)
    signal_distributions[thresh] = {'RISK_ON': n_risk_on, 'RISK_OFF': n_risk_off}
    print(f"  Threshold={thresh} | RISK_ON: {n_risk_on} ({n_risk_on/total*100:.1f}%) "
          f"| RISK_OFF: {n_risk_off} ({n_risk_off/total*100:.1f}%)")

    for freq in FREQS:
        for tc_label, tc_rate in TC_LEVELS.items():
            label = f"Sig3S_t{thresh}_{freq}_TC{tc_label}"

            def wfn_3s(sig_row, _eq=equity_assets, _ne=non_equity_assets,
                       _bw=base_weight, _sh=EQUITY_SAFE_HAVEN_MAP):
                return build_weights_3stage(sig_row, _eq, _ne, _bw, None, _sh)

            signal_results[label] = run_signal_portfolio(
                returns, sig_df_t, equity_assets, non_equity_assets,
                ALL_ASSETS, weight_fn=wfn_3s,
                freq=freq, tc_rate=tc_rate, initial_capital=1e9)
            print(f"    ✓ {label}")

# ── 15d. Asymmetric Signal ────────────────────────────────────────────────────
print("\n── Running Asymmetric Signal ──")
asym_results = {}

for thresh in THRESHOLDS:
    sig_df_t = build_signal_df_asymmetric(composite_score, equity_assets,
                                          macro_df.index, threshold=thresh)
    for freq in ['monthly', 'quarterly']:
        for tc_label, tc_rate in TC_LEVELS.items():
            label = f"Asym_t{thresh}_{freq}_TC{tc_label}"

            def wfn_asym(sig_row, _eq=equity_assets, _ne=non_equity_assets,
                         _bw=base_weight, _sh=EQUITY_SAFE_HAVEN_MAP):
                return build_weights_3stage(sig_row, _eq, _ne, _bw, None, _sh)

            asym_results[label] = run_signal_portfolio(
                returns, sig_df_t, equity_assets, non_equity_assets,
                ALL_ASSETS, weight_fn=wfn_asym,
                freq=freq, tc_rate=tc_rate, initial_capital=1e9)
            print(f"    ✓ {label}")

# ── 15e. Continuous / Graduated Signal ───────────────────────────────────────
print("\n── Running Continuous Signal ──")
cont_results = {}

for thresh in THRESHOLDS:
    for max_red in MAX_REDUCES:
        sig_df_t = build_signal_df_continuous(composite_score, equity_assets,
                                              macro_df.index,
                                              threshold=thresh,
                                              max_reduce=max_red)
        for freq in ['monthly', 'quarterly']:
            for tc_label, tc_rate in TC_LEVELS.items():
                label = (f"Cont_t{thresh}_mr{int(max_red*100)}"
                         f"_{freq}_TC{tc_label}")

                def wfn_cont(sig_row, _eq=equity_assets, _ne=non_equity_assets,
                             _bw=base_weight, _sh=EQUITY_SAFE_HAVEN_MAP):
                    return build_weights_continuous(sig_row, _eq, _ne, _bw, _sh)

                cont_results[label] = run_signal_portfolio(
                    returns, sig_df_t, equity_assets, non_equity_assets,
                    ALL_ASSETS, weight_fn=wfn_cont,
                    freq=freq, tc_rate=tc_rate, initial_capital=1e9)
                print(f"    ✓ {label}")

# ── 15f. Smoothed Signal ─────────────────────────────────────────────────────
print("\n── Running Smoothed Signal ──")
smooth_results = {}

for thresh in THRESHOLDS:
    for sw in SMOOTH_WINDOWS:
        sig_df_t = build_signal_df_smoothed(composite_score, equity_assets,
                                            macro_df.index,
                                            threshold=thresh,
                                            smooth_window=sw)
        for freq in ['monthly', 'quarterly']:
            for tc_label, tc_rate in TC_LEVELS.items():
                label = f"Smooth_t{thresh}_sw{sw}_{freq}_TC{tc_label}"

                def wfn_smooth(sig_row, _eq=equity_assets, _ne=non_equity_assets,
                               _bw=base_weight, _sh=EQUITY_SAFE_HAVEN_MAP):
                    return build_weights_3stage(sig_row, _eq, _ne, _bw, None, _sh)

                smooth_results[label] = run_signal_portfolio(
                    returns, sig_df_t, equity_assets, non_equity_assets,
                    ALL_ASSETS, weight_fn=wfn_smooth,
                    freq=freq, tc_rate=tc_rate, initial_capital=1e9)
                print(f"    ✓ {label}")

# ── 15g. Persistent Signal ────────────────────────────────────────────────────
print("\n── Running Persistent Signal ──")
persist_results = {}

for thresh in THRESHOLDS:
    for min_days in MIN_DAYS_LIST:
        sig_df_t = build_signal_df_persistent(composite_score, equity_assets,
                                              macro_df.index,
                                              threshold=thresh,
                                              min_days_to_switch=min_days)
        for freq in ['monthly', 'quarterly']:
            for tc_label, tc_rate in TC_LEVELS.items():
                label = f"Persist_t{thresh}_md{min_days}_{freq}_TC{tc_label}"

                def wfn_persist(sig_row, _eq=equity_assets, _ne=non_equity_assets,
                                _bw=base_weight, _sh=EQUITY_SAFE_HAVEN_MAP):
                    return build_weights_3stage(sig_row, _eq, _ne, _bw, None, _sh)

                persist_results[label] = run_signal_portfolio(
                    returns, sig_df_t, equity_assets, non_equity_assets,
                    ALL_ASSETS, weight_fn=wfn_persist,
                    freq=freq, tc_rate=tc_rate, initial_capital=1e9)
                print(f"    ✓ {label}")

# ── 15h. Partial-Hedge Signal ─────────────────────────────────────────────────
print("\n── Running Partial-Hedge Signal ──")
partial_results = {}

for thresh in THRESHOLDS:
    sig_df_t = build_signal_df(composite_score, equity_assets,
                               macro_df.index, threshold=thresh)
    for pk in PARTIAL_KEEPS:
        for freq in ['monthly', 'quarterly']:
            for tc_label, tc_rate in TC_LEVELS.items():
                label = (f"Partial_t{thresh}_pk{int(pk*100)}"
                         f"_{freq}_TC{tc_label}")

                def wfn_partial(sig_row, _eq=equity_assets, _ne=non_equity_assets,
                                _bw=base_weight, _sh=EQUITY_SAFE_HAVEN_MAP,
                                _pk=pk):
                    return build_weights_partial(sig_row, _eq, _ne, _bw, _sh, _pk)

                partial_results[label] = run_signal_portfolio(
                    returns, sig_df_t, equity_assets, non_equity_assets,
                    ALL_ASSETS, weight_fn=wfn_partial,
                    freq=freq, tc_rate=tc_rate, initial_capital=1e9)
                print(f"    ✓ {label}")

# ── 15i. Mean-Variance Signal ─────────────────────────────────────────────────
print("\n── Running Mean-Variance Signal ──")
mv_results        = {}
mv_weight_history = {}

for thresh in THRESHOLDS:
    sig_df_t = build_signal_df(composite_score, equity_assets,
                               macro_df.index, threshold=thresh)
    for freq in MV_FREQS:
        for tc_label, tc_rate in TC_LEVELS.items():
            label = f"MV_t{thresh}_{freq}_TC{tc_label}"
            pv_df, w_df = run_mv_signal_portfolio(
                returns, sig_df_t, ALL_ASSETS,
                equity_assets=equity_assets,
                freq=freq, tc_rate=tc_rate,
                lookback=MV_LOOKBACK, min_obs=MV_MIN_OBS,
                max_weight=MAX_WEIGHT, initial_capital=1e9)
            mv_results[label]        = pv_df
            mv_weight_history[label] = w_df
            print(f"  ✓ {label}")

print("\n✓ All portfolios computed")

# ── 16. PERFORMANCE METRICS ───────────────────────────────────────────────────
def compute_metrics(pv_df, label=''):
    pv        = pv_df['portfolio_value']
    daily_ret = pv.pct_change().dropna()
    n_years   = len(daily_ret) / 252
    total_ret = (pv.iloc[-1] / pv.iloc[0]) - 1
    cagr      = (1 + total_ret) ** (1 / n_years) - 1 if n_years > 0 else np.nan
    vol       = daily_ret.std() * np.sqrt(252)
    sharpe    = cagr / vol if vol > 0 else np.nan
    roll_max  = pv.cummax()
    max_dd    = ((pv - roll_max) / roll_max).min()
    calmar    = cagr / abs(max_dd) if max_dd != 0 else np.nan
    var_95    = daily_ret.quantile(0.05)
    ann_ret   = cagr
    ann_vol   = vol

    return {
        'CAGR':         f"{cagr*100:.2f}%",
        'Ann_Ret':      ann_ret,
        'Ann_Vol':      ann_vol,
        'Volatility':   f"{vol*100:.2f}%",
        'Sharpe':       f"{sharpe:.2f}",
        'Max Drawdown': f"{max_dd*100:.2f}%",
        'Calmar':       f"{calmar:.2f}",
        'VaR 95%':      f"{var_95*100:.2f}%",
        'Total Return': f"{total_ret*100:.2f}%",
        'Final ($B)':   f"{pv.iloc[-1]/1e9:.3f}",
    }

bh_metrics      = {k: compute_metrics(v, k) for k, v in bh_results.items()}
ew_metrics      = {k: compute_metrics(v, k) for k, v in ew_results.items()}
sig_metrics     = {k: compute_metrics(v, k) for k, v in signal_results.items()}
asym_metrics    = {k: compute_metrics(v, k) for k, v in asym_results.items()}
cont_metrics    = {k: compute_metrics(v, k) for k, v in cont_results.items()}
smooth_metrics  = {k: compute_metrics(v, k) for k, v in smooth_results.items()}
persist_metrics = {k: compute_metrics(v, k) for k, v in persist_results.items()}
partial_metrics = {k: compute_metrics(v, k) for k, v in partial_results.items()}
mv_metrics      = {k: compute_metrics(v, k) for k, v in mv_results.items()}

def print_metrics_table(metrics_dict, title):
    display = {k: {mk: mv for mk, mv in m.items()
                   if mk not in ('Ann_Ret', 'Ann_Vol')}
               for k, m in metrics_dict.items()}
    print(f"\n{'='*90}")
    print(title)
    print("="*90)
    print(pd.DataFrame(display).T.to_string())

print_metrics_table(bh_metrics,      "BUY-AND-HOLD")
print_metrics_table(ew_metrics,      "EQUAL-WEIGHT REBALANCED")

for thresh in THRESHOLDS:
    sub = {k: v for k, v in sig_metrics.items()     if f"_t{thresh}_" in k}
    print_metrics_table(sub, f"3-STAGE SIGNAL — threshold={thresh}")

for thresh in THRESHOLDS:
    sub = {k: v for k, v in asym_metrics.items()    if f"_t{thresh}_" in k}
    print_metrics_table(sub, f"ASYMMETRIC SIGNAL — threshold={thresh}")

for thresh in THRESHOLDS:
    sub = {k: v for k, v in cont_metrics.items()    if f"_t{thresh}_" in k}
    print_metrics_table(sub, f"CONTINUOUS SIGNAL — threshold={thresh}")

for thresh in THRESHOLDS:
    sub = {k: v for k, v in smooth_metrics.items()  if f"_t{thresh}_" in k}
    print_metrics_table(sub, f"SMOOTHED SIGNAL — threshold={thresh}")

for thresh in THRESHOLDS:
    sub = {k: v for k, v in persist_metrics.items() if f"_t{thresh}_" in k}
    print_metrics_table(sub, f"PERSISTENT SIGNAL — threshold={thresh}")

for thresh in THRESHOLDS:
    sub = {k: v for k, v in partial_metrics.items() if f"_t{thresh}_" in k}
    print_metrics_table(sub, f"PARTIAL-HEDGE SIGNAL — threshold={thresh}")

for thresh in THRESHOLDS:
    sub = {k: v for k, v in mv_metrics.items()      if f"_t{thresh}_" in k}
    print_metrics_table(sub, f"MEAN-VARIANCE SIGNAL — threshold={thresh}")

# ── 17. PLOTS ─────────────────────────────────────────────────────────────────
bh_pv     = bh_results['BuyAndHold_EW']['portfolio_value']
ew_ref_pv = ew_results['EW_monthly_TC0.5%']['portfolio_value']

def parse_pct(s):
    return float(str(s).replace('%', '')) / 100

# ── Plot A: Best of each variant vs benchmarks (monthly, TC=0.5%) ─────────────
fig, axes = plt.subplots(2, 4, figsize=(28, 12), sharey=False)
fig.suptitle("All Signal Variants vs Benchmarks — Monthly Rebalancing, TC=0.5%",
             fontsize=14, fontweight='bold')

variant_info = [
    ("3-Stage",    signal_results,  THRESHOLDS,     "Sig3S",   "steelblue"),
    ("Asymmetric", asym_results,    THRESHOLDS,     "Asym",    "darkorange"),
    ("Continuous", cont_results,    THRESHOLDS,     "Cont",    "purple"),
    ("Smoothed",   smooth_results,  THRESHOLDS,     "Smooth",  "green"),
    ("Persistent", persist_results, THRESHOLDS,     "Persist", "crimson"),
    ("Partial 25%",partial_results, THRESHOLDS,     "Partial", "brown"),
    ("MV Signal",  mv_results,      THRESHOLDS,     "MV",      "black"),
]

for ax, (name, res_dict, threshs, prefix, col) in zip(axes.flat, variant_info):
    ax.plot(bh_pv.index,     bh_pv / 1e9,     color='forestgreen',
            lw=1.5, ls='-.', label='Buy & Hold')
    ax.plot(ew_ref_pv.index, ew_ref_pv / 1e9, color='grey',
            lw=1.2, ls='--', label='EW Monthly')

    for thresh in threshs:
        candidates = [k for k in res_dict
                      if f"_t{thresh}_" in k and 'monthly' in k and 'TC0.5%' in k]
        if not candidates:
            candidates = [k for k in res_dict
                          if f"_t{thresh}_" in k and 'TC0.5%' in k]
        if candidates:
            key = candidates[0]
            pv  = res_dict[key]['portfolio_value']
            ax.plot(pv.index, pv / 1e9, lw=1.6,
                    label=f"t={thresh}")

    ax.axhline(1.0, color='black', ls=':', lw=0.7)
    ax.set_title(name, fontsize=11, fontweight='bold')
    ax.set_ylabel("Value ($B)")
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

axes.flat[-1].set_visible(False)
plt.tight_layout()
plt.savefig("all_variants_monthly.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Plot B: Risk-Return Scatter — all strategies ──────────────────────────────
fig, ax = plt.subplots(figsize=(16, 10))

all_metric_groups = [
    (bh_metrics,      'BH',       'forestgreen', 'D'),
    (ew_metrics,      'EW',       'grey',        's'),
    (sig_metrics,     '3-Stage',  'steelblue',   'o'),
    (asym_metrics,    'Asym',     'darkorange',  'o'),
    (cont_metrics,    'Cont',     'purple',      'o'),
    (smooth_metrics,  'Smooth',   'green',       'o'),
    (persist_metrics, 'Persist',  'crimson',     'o'),
    (partial_metrics, 'Partial',  'brown',       'o'),
    (mv_metrics,      'MV',       'black',       '^'),
]

for metrics_dict, grp_label, col, marker in all_metric_groups:
    first = True
    for key, m in metrics_dict.items():
        vol = m['Ann_Vol'] * 100
        ret = m['Ann_Ret'] * 100
        ax.scatter(vol, ret, color=col, marker=marker, s=80,
                   label=grp_label if first else None,
                   edgecolors='black', lw=0.5, zorder=4)
        ax.annotate(key.split('_TC')[0], (vol, ret),
                    textcoords='offset points', xytext=(4, 3),
                    fontsize=6, color=col)
        first = False

ax.axhline(0, color='black', lw=0.6, ls='--')
ax.axvline(0, color='black', lw=0.6, ls='--')
ax.set_xlabel("Annualised Volatility (%)", fontsize=12)
ax.set_ylabel("CAGR (%)", fontsize=12)
ax.set_title("Risk-Return — All Strategies", fontsize=13, fontweight='bold')
ax.legend(fontsize=10, ncol=3)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("risk_return_all.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Plot C: Sharpe Bar Chart — top 20 strategies ──────────────────────────────
all_metrics_combined = {
    **bh_metrics, **ew_metrics, **sig_metrics,
    **asym_metrics, **cont_metrics, **smooth_metrics,
    **persist_metrics, **partial_metrics, **mv_metrics
}

sharpe_series = pd.Series({
    k: float(v['Sharpe']) for k, v in all_metrics_combined.items()
}).dropna().sort_values(ascending=False)

top20 = sharpe_series.head(20)

def label_color(lbl):
    if lbl.startswith('MV'):       return 'black'
    if lbl.startswith('Sig3S'):    return 'steelblue'
    if lbl.startswith('Asym'):     return 'darkorange'
    if lbl.startswith('Cont'):     return 'purple'
    if lbl.startswith('Smooth'):   return 'green'
    if lbl.startswith('Persist'):  return 'crimson'
    if lbl.startswith('Partial'):  return 'brown'
    if lbl.startswith('EW'):       return 'grey'
    return 'forestgreen'

colors = [label_color(k) for k in top20.index]

fig, ax = plt.subplots(figsize=(16, 7))
bars = ax.barh(range(len(top20)), top20.values, color=colors, edgecolor='black', lw=0.5)
ax.set_yticks(range(len(top20)))
ax.set_yticklabels([k.replace('_TC', ' TC=') for k in top20.index], fontsize=8)
ax.set_xlabel("Sharpe Ratio")
ax.set_title("Top 20 Strategies by Sharpe Ratio", fontsize=13, fontweight='bold')
ax.axvline(0, color='black', lw=0.8)
ax.grid(True, axis='x', alpha=0.3)

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='black',      label='MV Signal'),
    Patch(facecolor='steelblue',  label='3-Stage'),
    Patch(facecolor='darkorange', label='Asymmetric'),
    Patch(facecolor='purple',     label='Continuous'),
    Patch(facecolor='green',      label='Smoothed'),
    Patch(facecolor='crimson',    label='Persistent'),
    Patch(facecolor='brown',      label='Partial Hedge'),
    Patch(facecolor='grey',       label='Equal Weight'),
    Patch(facecolor='forestgreen',label='Buy & Hold'),
]
ax.legend(handles=legend_elements, fontsize=9, loc='lower right')
plt.tight_layout()
plt.savefig("sharpe_top20.png", dpi=150, bbox_inches='tight')
plt.show()

# ── 18. SAVE ALL ──────────────────────────────────────────────────────────────
print("\n── Saving all results ──")
all_portfolios = {
    **bh_results, **ew_results, **signal_results,
    **asym_results, **cont_results, **smooth_results,
    **persist_results, **partial_results, **mv_results
}

for label, df in all_portfolios.items():
    df.to_csv(f"{label}_portfolio.csv")

all_metrics_df = pd.DataFrame({
    k: {mk: mv for mk, mv in m.items() if mk not in ('Ann_Ret', 'Ann_Vol')}
    for k, m in all_metrics_combined.items()
}).T
all_metrics_df.to_csv("all_portfolio_metrics.csv")

composite_score.to_csv("composite_signal.csv")
for thresh in THRESHOLDS:
    sig_df_t = build_signal_df(composite_score, equity_assets,
                               macro_df.index, threshold=thresh)
    sig_df_t.to_csv(f"signal_df_threshold_{thresh}.csv")

for label, wdf in mv_weight_history.items():
    wdf.to_csv(f"{label}_weights.csv")

print("✓ All files saved.")
print(f"\n  Portfolio CSVs   : {len(all_portfolios)}")
print(f"  Weight CSVs      : {len(mv_weight_history)}")
print(f"  Metrics CSV      : all_portfolio_metrics.csv")
print(f"  Signal CSVs      : {len(THRESHOLDS)} threshold files")
print(f"  Plots saved      : 3 PNG files")


In [ ]:
# @title
# ============================================================
# SIGNAL PORTFOLIO — ENHANCED VERSION
# Signals generated from Modified Determinant Model
# + Buy-and-Hold Benchmark
# + Equal-Weight Benchmark
# + Three-Stage Signal Portfolio (original)
# + Enhanced Signal Variants (asymmetric, continuous, smoothed, persistent, partial)
# + Mean-Variance Signal Portfolio
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import requests
from io import StringIO
from scipy.optimize import minimize

# ── 1. LOAD ALL DATA FROM GITHUB ─────────────────────────────────────────────
GITHUB_BASE = "https://raw.githubusercontent.com/kboroz/MSFE_Capstone_Project/main/01_Data/Streamlined/"

INDEX_FILE_MAP = {
    'ASX50':       'streamlined_asx_50.csv',
    'EUROSTOXX50': 'streamlined_euro_stoxx_50.csv',
    'FTSE100':     'streamlined_ftse_100.csv',
    'Ibovespa':    'streamlined_ibovespa.csv',
    'JSE40':       'streamlined_jse_top_40.csv',
    'NIKKEI225':   'streamlined_nikkei_225.csv',
    'SP500':       'streamlined_s&p_500.csv',
}

MACRO_FILE = 'streamlined_macro.csv'

def load_csv(url):
    r = requests.get(url)
    r.raise_for_status()
    df = pd.read_csv(StringIO(r.text), index_col=0, parse_dates=True)
    df.columns = df.columns.str.replace('[^a-zA-Z0-9]', '_', regex=True)
    return df.sort_index()

macro_df = load_csv(GITHUB_BASE + MACRO_FILE)
macro_df = macro_df[~macro_df.index.duplicated(keep='first')].sort_index()
print(f"macro_df shape : {macro_df.shape}")
print(f"Columns        : {macro_df.columns.tolist()}")

index_data = {}
for name, fname in INDEX_FILE_MAP.items():
    try:
        df = load_csv(GITHUB_BASE + fname)
        index_data[name] = df
        print(f"✓ {name:15s} | shape={df.shape} | cols={df.columns.tolist()[:4]}")
    except Exception as e:
        print(f"✗ {name}: {e}")

# ── 2. EXTRACT PRICE SERIES ───────────────────────────────────────────────────
PRICE_COLS = ['Close', 'close', 'Price', 'price', 'Adj_Close', 'Last']

def get_price_series(df, name):
    col = next((c for c in PRICE_COLS if c in df.columns), df.columns[0])
    s = pd.to_numeric(df[col], errors='coerce').dropna()
    print(f"  {name}: using column '{col}'")
    return s

index_prices = {}
for name, df in index_data.items():
    index_prices[name] = get_price_series(df, name)

# ── 3. MODIFIED DETERMINANT SIGNAL MODEL ─────────────────────────────────────
DET_WINDOWS = [3, 7, 28]
WINDOW_WEIGHTS = {3: 0.25, 7: 0.25, 28: 0.50}

def compute_modified_determinant(price_series_dict, windows=DET_WINDOWS):
    prices = pd.DataFrame(price_series_dict).sort_index()
    rets   = prices.pct_change().dropna()
    det_results = {}

    for window in windows:
        det_series = []
        for i in range(window, len(rets)):
            window_rets = rets.iloc[i - window:i]
            valid_cols  = window_rets.columns[window_rets.std() > 1e-10]
            w           = window_rets[valid_cols]

            if len(valid_cols) < 2:
                det_series.append(np.nan)
                continue

            try:
                R   = w.corr().values
                I   = np.eye(len(R))
                M   = R - I
                det = np.linalg.det(M)
            except Exception:
                det = np.nan

            det_series.append(det)

        idx = rets.index[window:]
        det_results[window] = pd.Series(det_series, index=idx,
                                        name=f'det_{window}d')
        print(f"  Window {window:2d}d | computed {len(det_series)} observations")

    return det_results

print("\n── Computing Modified Determinant ──")
det_results = compute_modified_determinant(index_prices)

# ── 4. COMPOSITE SCORE ────────────────────────────────────────────────────────
def compute_composite_score(det_results, window_weights=WINDOW_WEIGHTS):
    all_series = []
    for window, series in det_results.items():
        mu    = series.rolling(252, min_periods=60).mean()
        sigma = series.rolling(252, min_periods=60).std()
        z     = (series - mu) / sigma.replace(0, np.nan)
        z.name = f'z_{window}d'
        all_series.append(z)

    z_df      = pd.concat(all_series, axis=1).dropna()
    composite = sum(
        z_df[f'z_{w}d'] * window_weights[w]
        for w in window_weights if f'z_{w}d' in z_df.columns
    )
    composite = composite / sum(window_weights.values())
    return composite, z_df

composite_score, z_df = compute_composite_score(det_results)

print(f"\nComposite score shape  : {composite_score.shape}")
print(f"Date range             : {composite_score.index[0].date()} → "
      f"{composite_score.index[-1].date()}")
print(f"Score range            : {composite_score.min():.3f} → "
      f"{composite_score.max():.3f}")

# ── 5. SIGNAL BUILDERS ───────────────────────────────────────────────────────
THRESHOLDS    = [0.4, 0.8, 1.2]
SMOOTH_WINDOWS = [3, 5, 10]
MIN_DAYS_LIST  = [5, 10, 20]
MAX_REDUCES    = [0.5, 0.7, 0.9]
PARTIAL_KEEPS  = [0.25, 0.50]

# ── 5a. Original symmetric binary ────────────────────────────────────────────
def classify_signal_symmetric(score, threshold):
    return +1 if abs(score) > threshold else -1

def build_signal_df(composite_score, equity_assets, macro_dates, threshold=0.5):
    binary    = composite_score.apply(lambda x: classify_signal_symmetric(x, threshold))
    signal_df = pd.DataFrame(index=macro_dates)
    for asset in equity_assets:
        signal_df[asset] = binary.reindex(macro_dates).ffill().bfill()
    return signal_df

# ── 5b. Asymmetric (upper-tail only) ─────────────────────────────────────────
def build_signal_df_asymmetric(composite_score, equity_assets, macro_dates,
                               threshold=1.0):
    """Block equities only when correlation SPIKES (high positive z-score)."""
    binary    = composite_score.apply(lambda x: +1 if x > threshold else -1)
    signal_df = pd.DataFrame(index=macro_dates)
    for asset in equity_assets:
        signal_df[asset] = binary.reindex(macro_dates).ffill().bfill()
    return signal_df

# ── 5c. Continuous / graduated scalar ────────────────────────────────────────
def build_signal_df_continuous(composite_score, equity_assets, macro_dates,
                               threshold=1.0, max_reduce=0.8):
    """
    Returns a scalar in [1-max_reduce, 1.0] per asset per day.
    Used by build_weights_continuous() instead of the binary flag.
    """
    def score_to_scalar(s):
        if s <= 0:
            return 1.0
        elif s >= threshold:
            return 1.0 - max_reduce
        else:
            return 1.0 - max_reduce * (s / threshold)

    scalars   = composite_score.apply(score_to_scalar)
    signal_df = pd.DataFrame(index=macro_dates)
    for asset in equity_assets:
        signal_df[asset] = scalars.reindex(macro_dates).ffill().bfill()
    return signal_df

# ── 5d. Smoothed binary ───────────────────────────────────────────────────────
def build_signal_df_smoothed(composite_score, equity_assets, macro_dates,
                             threshold=1.0, smooth_window=5):
    """Smooth z-score before thresholding to reduce whipsaw."""
    smoothed  = composite_score.rolling(smooth_window, min_periods=1).mean()
    binary    = smoothed.apply(lambda x: classify_signal_symmetric(x, threshold))
    signal_df = pd.DataFrame(index=macro_dates)
    for asset in equity_assets:
        signal_df[asset] = binary.reindex(macro_dates).ffill().bfill()
    return signal_df

# ── 5e. Persistent binary ────────────────────────────────────────────────────
def build_signal_df_persistent(composite_score, equity_assets, macro_dates,
                               threshold=1.0, min_days_to_switch=10):
    """Only switch regime after signal holds for min_days_to_switch days."""
    raw       = composite_score.apply(lambda x: +1 if x > threshold else -1)
    confirmed = raw.copy()
    current   = int(raw.iloc[0])
    count     = 0

    for i in range(1, len(raw)):
        if int(raw.iloc[i]) != current:
            count += 1
            if count >= min_days_to_switch:
                current = int(raw.iloc[i])
                count   = 0
        else:
            count = 0
        confirmed.iloc[i] = current

    signal_df = pd.DataFrame(index=macro_dates)
    for asset in equity_assets:
        signal_df[asset] = confirmed.reindex(macro_dates).ffill().bfill()
    return signal_df

# ── 6. ASSET UNIVERSE ─────────────────────────────────────────────────────────
EQUITY_ASSETS = list(index_prices.keys())
SAFE_BONDS    = ['US_10Y', 'UK_10Y', 'AU_10Y', 'ZA_10Y']
FX            = ['AUDUSD', 'BRLUSD', 'EURUSD', 'GBPUSD', 'JPYUSD', 'ZARUSD']
COMMODITIES   = ['Oil', 'Gold', 'Bitcoin']
EXCLUDE_ASSETS = ['Bitcoin']

ALL_ASSETS = EQUITY_ASSETS + FX + COMMODITIES + SAFE_BONDS
ALL_ASSETS = [a for a in ALL_ASSETS
              if a in macro_df.columns and a not in EXCLUDE_ASSETS]

equity_assets     = [a for a in EQUITY_ASSETS     if a in ALL_ASSETS]
non_equity_assets = [a for a in ALL_ASSETS         if a not in equity_assets]

N           = len(ALL_ASSETS)
base_weight = 1.0 / N

print(f"\nTotal assets              : {N}")
print(f"Equity (signal-driven)    : {equity_assets}")
print(f"Non-equity (always long)  : {non_equity_assets}")
print(f"base_weight = 1/{N} = {base_weight:.4f}")

# ── 7. COUNTRY SAFE-HAVEN MAP ─────────────────────────────────────────────────
EQUITY_SAFE_HAVEN_MAP = {
    'ASX50':       ['AU_10Y', 'AUDUSD'],
    'EUROSTOXX50': ['EURUSD'],
    'FTSE100':     ['UK_10Y', 'GBPUSD'],
    'Ibovespa':    ['BRLUSD'],
    'JSE40':       ['ZA_10Y', 'ZARUSD'],
    'NIKKEI225':   ['JPYUSD'],
    'SP500':       ['US_10Y'],
}

UNIVERSE_SET = set(ALL_ASSETS)
for eq, havens in EQUITY_SAFE_HAVEN_MAP.items():
    EQUITY_SAFE_HAVEN_MAP[eq] = [h for h in havens if h in UNIVERSE_SET]

print("\nSafe-haven routing (filtered to universe):")
for eq, havens in EQUITY_SAFE_HAVEN_MAP.items():
    print(f"  {eq:15s} → {havens}")

# ── 8. BUILD RETURNS ──────────────────────────────────────────────────────────
returns_raw = macro_df[ALL_ASSETS].pct_change(fill_method=None)

for col in SAFE_BONDS:
    if col in returns_raw.columns:
        returns_raw[col] = -returns_raw[col]

if 'Oil' in macro_df.columns:
    bad_oil = macro_df['Oil'][macro_df['Oil'] <= 0].index
    for d in bad_oil:
        loc = returns_raw.index.get_loc(d)
        returns_raw.iloc[max(0, loc - 1): loc + 2,
                         returns_raw.columns.get_loc('Oil')] = 0.0

returns = returns_raw.clip(lower=-0.50, upper=1.00).fillna(0).iloc[1:]

assert not returns.isnull().any().any(), "NaNs in returns"
assert not np.isinf(returns.values).any(),  "Inf in returns"
print(f"\nReturns shape : {returns.shape}")
print("✓ Returns clean")

# ── 9. HELPER: REBALANCING DATE SET ──────────────────────────────────────────
def build_reb_set(returns, rebal_freq):
    actual_dates = returns.index
    if rebal_freq == 'daily':
        return set(actual_dates)
    elif rebal_freq == 'weekly':
        raw = returns.resample('W-MON').first().index
    elif rebal_freq == 'monthly':
        raw = returns.resample('MS').first().index
    elif rebal_freq == 'quarterly':
        raw = returns.resample('QS').first().index
    else:
        raise ValueError(f"Unknown rebal_freq: {rebal_freq}")

    reb_set = set()
    for d in raw:
        future = actual_dates[actual_dates >= d]
        if len(future) > 0:
            reb_set.add(future[0])
    return reb_set

# ── 10. BUY-AND-HOLD BENCHMARK ───────────────────────────────────────────────
def run_buy_and_hold(returns, all_assets, initial_capital=1e9):
    N           = len(all_assets)
    base_weight = 1.0 / N
    pv          = float(initial_capital)
    holdings    = {a: pv * base_weight for a in all_assets}
    records     = []
    first       = True

    for date in returns.index:
        if first:
            first = False
            records.append({'date': date, 'portfolio_value': pv})
            continue
        new_holdings = {a: holdings[a] * (1 + returns.loc[date, a])
                        for a in all_assets}
        holdings = new_holdings
        pv       = sum(holdings.values())
        records.append({'date': date, 'portfolio_value': pv})

    df = pd.DataFrame(records).set_index('date')
    print(f"  Buy-and-Hold | Start: ${initial_capital/1e9:.2f}B → "
          f"End: ${df['portfolio_value'].iloc[-1]/1e9:.3f}B")
    return df

# ── 11. WEIGHT BUILDERS ───────────────────────────────────────────────────────

# ── 11a. Original binary 3-stage weights ─────────────────────────────────────
def build_weights_3stage(sig_row, equity_assets, non_equity_assets,
                         base_weight, date, safe_haven_map):
    raw            = {a: base_weight for a in non_equity_assets}
    safe_haven_extra = {}

    for a in equity_assets:
        if sig_row is not None and a in sig_row.index and not pd.isna(sig_row[a]):
            sig = sig_row[a]
        else:
            sig = -1

        if sig > 0:
            raw[a] = -base_weight
            havens = [h for h in safe_haven_map.get(a, []) if h in raw]
            if havens:
                per_haven = base_weight / len(havens)
                for h in havens:
                    safe_haven_extra[h] = safe_haven_extra.get(h, 0) + per_haven
            else:
                per_ne = base_weight / len(non_equity_assets)
                for ne in non_equity_assets:
                    safe_haven_extra[ne] = safe_haven_extra.get(ne, 0) + per_ne
        else:
            raw[a] = base_weight

    for asset, extra in safe_haven_extra.items():
        raw[asset] = raw.get(asset, 0) + extra

    total = sum(raw.values())
    if total <= 0:
        for a in raw:
            raw[a] = 1.0 / len(raw)
    else:
        for a in raw:
            raw[a] /= total
    return raw

# ── 11b. Continuous scalar weights ───────────────────────────────────────────
def build_weights_continuous(sig_row, equity_assets, non_equity_assets,
                             base_weight, safe_haven_map):
    """
    sig_row contains a scalar in [min_weight, 1.0] per equity asset.
    Reduced equity weight is redistributed to safe havens.
    """
    raw              = {a: base_weight for a in non_equity_assets}
    safe_haven_extra = {}

    for a in equity_assets:
        scalar = float(sig_row[a]) if (sig_row is not None and
                                       a in sig_row.index and
                                       not pd.isna(sig_row[a])) else 1.0
        raw[a]   = base_weight * scalar
        released = base_weight * (1.0 - scalar)

        if released > 0:
            havens = [h for h in safe_haven_map.get(a, []) if h in raw]
            if havens:
                per_haven = released / len(havens)
                for h in havens:
                    safe_haven_extra[h] = safe_haven_extra.get(h, 0) + per_haven
            else:
                per_ne = released / len(non_equity_assets)
                for ne in non_equity_assets:
                    safe_haven_extra[ne] = safe_haven_extra.get(ne, 0) + per_ne

    for asset, extra in safe_haven_extra.items():
        raw[asset] = raw.get(asset, 0) + extra

    total = sum(raw.values())
    if total <= 0:
        for a in raw:
            raw[a] = 1.0 / len(raw)
    else:
        for a in raw:
            raw[a] /= total
    return raw

# ── 11c. Partial-hedge weights ────────────────────────────────────────────────
def build_weights_partial(sig_row, equity_assets, non_equity_assets,
                          base_weight, safe_haven_map, partial_keep=0.25):
    """
    On RISK_ON signal: keep partial_keep fraction of base_weight in equity,
    redirect the rest to safe havens.
    """
    raw              = {a: base_weight for a in non_equity_assets}
    safe_haven_extra = {}

    for a in equity_assets:
        if sig_row is not None and a in sig_row.index and not pd.isna(sig_row[a]):
            sig = sig_row[a]
        else:
            sig = -1

        if sig > 0:
            raw[a]   = base_weight * partial_keep
            released = base_weight * (1.0 - partial_keep)
            havens   = [h for h in safe_haven_map.get(a, []) if h in raw]
            if havens:
                per_haven = released / len(havens)
                for h in havens:
                    safe_haven_extra[h] = safe_haven_extra.get(h, 0) + per_haven
            else:
                per_ne = released / len(non_equity_assets)
                for ne in non_equity_assets:
                    safe_haven_extra[ne] = safe_haven_extra.get(ne, 0) + per_ne
        else:
            raw[a] = base_weight

    for asset, extra in safe_haven_extra.items():
        raw[asset] = raw.get(asset, 0) + extra

    total = sum(raw.values())
    if total <= 0:
        for a in raw:
            raw[a] = 1.0 / len(raw)
    else:
        for a in raw:
            raw[a] /= total
    return raw

# ── 12. EQUAL-WEIGHT REBALANCED BENCHMARK ────────────────────────────────────
def run_equal_weight(returns, all_assets, tc_rate=0.005,
                     rebal_freq='monthly', initial_capital=1e9):
    N           = len(all_assets)
    base_weight = 1.0 / N
    ew          = {a: base_weight for a in all_assets}
    reb_set     = build_reb_set(returns, rebal_freq)
    pv          = float(initial_capital)
    current_weights = None
    records     = []

    for date in returns.index:
        if date in reb_set:
            if current_weights is not None:
                turnover = sum(abs(ew.get(a, 0) - current_weights.get(a, 0))
                               for a in set(ew) | set(current_weights)) / 2.0
                pv *= (1 - turnover * tc_rate)
            else:
                pv *= (1 - tc_rate)
            current_weights = ew.copy()

        if current_weights is not None:
            dr  = sum(current_weights.get(a, 0) * returns.loc[date, a]
                      for a in all_assets)
            pv *= (1 + dr)

        records.append({'date': date, 'portfolio_value': pv})

    return pd.DataFrame(records).set_index('date')

# ── 13. GENERIC SIGNAL PORTFOLIO RUNNER ──────────────────────────────────────
def run_signal_portfolio(returns, signal_df, equity_assets,
                         non_equity_assets, all_assets,
                         weight_fn,          # callable: (sig_row, ...) → dict
                         freq='monthly', tc_rate=0.005,
                         initial_capital=1e9):
    """
    Generic runner.  weight_fn must accept (sig_row, equity_assets,
    non_equity_assets, base_weight, <any extra kwargs>) and return
    a dict {asset: weight}.
    """
    N           = len(all_assets)
    base_weight = 1.0 / N
    reb_set     = build_reb_set(returns, freq)

    sig = signal_df.copy()
    sig.index = pd.to_datetime(sig.index)

    pv              = float(initial_capital)
    current_weights = None
    records         = []

    for date in returns.index:
        if date in reb_set:
            past    = sig[sig.index <= date]
            sig_row = past.iloc[-1] if len(past) > 0 else None
            raw     = weight_fn(sig_row)

            if current_weights is not None:
                all_keys = set(raw) | set(current_weights)
                turnover = sum(abs(raw.get(a, 0) - current_weights.get(a, 0))
                               for a in all_keys) / 2.0
                pv *= (1 - turnover * tc_rate)
            else:
                pv *= (1 - tc_rate)

            current_weights = raw.copy()

        if current_weights is not None:
            dr  = sum(current_weights.get(a, 0) * returns.loc[date, a]
                      for a in all_assets)
            pv *= (1 + dr)

        records.append({'date': date, 'portfolio_value': pv})

    return pd.DataFrame(records).set_index('date')

# ── 14. MEAN-VARIANCE SIGNAL PORTFOLIO ───────────────────────────────────────
MV_LOOKBACK = 252
MV_MIN_OBS  = 60
MAX_WEIGHT  = 0.30
RISK_FREE   = 0.0

def mean_variance_weights(mu_vec, cov_mat, allowed_mask,
                          max_weight=MAX_WEIGHT, risk_free=RISK_FREE):
    n_total   = len(mu_vec)
    allowed   = np.where(allowed_mask)[0]
    n_allowed = len(allowed)

    fallback = np.zeros(n_total)
    if n_allowed == 0:
        fallback[:] = 1.0 / n_total
        return fallback
    fallback[allowed] = 1.0 / n_allowed

    mu_sub  = mu_vec[allowed]
    cov_sub = cov_mat[np.ix_(allowed, allowed)] + np.eye(n_allowed) * 1e-8

    def neg_sharpe(w):
        port_ret = np.dot(w, mu_sub)
        port_var = np.dot(w, cov_sub @ w)
        port_std = np.sqrt(max(port_var, 1e-12))
        return -(port_ret - risk_free) / port_std

    constraints = [{'type': 'eq', 'fun': lambda w: np.sum(w) - 1.0}]
    bounds      = [(0.0, max_weight)] * n_allowed
    w0          = np.ones(n_allowed) / n_allowed

    try:
        res = minimize(neg_sharpe, w0, method='SLSQP',
                       bounds=bounds, constraints=constraints,
                       options={'ftol': 1e-9, 'maxiter': 500})
        w_opt = res.x if (res.success and not np.any(np.isnan(res.x))) else w0
    except Exception:
        w_opt = w0

    w_full         = np.zeros(n_total)
    w_full[allowed] = w_opt
    return w_full

def run_mv_signal_portfolio(returns, signal_df, all_assets,
                            equity_assets,
                            freq='monthly', tc_rate=0.005,
                            lookback=MV_LOOKBACK, min_obs=MV_MIN_OBS,
                            max_weight=MAX_WEIGHT, initial_capital=1e9):
    reb_set      = build_reb_set(returns, freq)
    actual_dates = returns.index
    assets       = all_assets
    n            = len(assets)
    asset_idx    = {a: i for i, a in enumerate(assets)}

    sig = signal_df.copy()
    sig.index = pd.to_datetime(sig.index)

    pv              = float(initial_capital)
    current_weights = np.zeros(n)
    records         = []
    weight_history  = []
    first_reb       = True

    for t, date in enumerate(actual_dates):
        if date in reb_set:
            hist = returns.iloc[max(0, t - lookback): t]

            if len(hist) < min_obs:
                new_weights = np.ones(n) / n
            else:
                mu_ann  = hist.mean().values  * 252
                cov_ann = hist.cov().values   * 252

                past    = sig[sig.index <= date]
                sig_row = past.iloc[-1] if len(past) > 0 else None

                allowed_mask = np.ones(n, dtype=bool)
                if sig_row is not None:
                    for eq in equity_assets:
                        if eq in asset_idx and eq in sig_row.index:
                            s = sig_row[eq]
                            if not pd.isna(s) and s > 0:
                                allowed_mask[asset_idx[eq]] = False

                new_weights = mean_variance_weights(
                    mu_ann, cov_ann, allowed_mask, max_weight=max_weight)

            if first_reb:
                pv       *= (1 - tc_rate)
                first_reb = False
            else:
                turnover = np.sum(np.abs(new_weights - current_weights)) / 2.0
                pv      *= (1 - turnover * tc_rate)

            current_weights = new_weights.copy()
            weight_history.append({'date': date,
                                   **dict(zip(assets, current_weights))})

        if not first_reb:
            dr  = np.dot(current_weights, returns.loc[date].values)
            pv *= (1 + dr)

        records.append({'date': date, 'portfolio_value': pv})

    pv_df      = pd.DataFrame(records).set_index('date')
    weights_df = pd.DataFrame(weight_history).set_index('date')
    print(f"  MV | freq={freq} tc={tc_rate*100:.1f}% "
          f"End: ${pv_df['portfolio_value'].iloc[-1]/1e9:.3f}B")
    return pv_df, weights_df

# ── 15. RUN ALL PORTFOLIOS ────────────────────────────────────────────────────
TC_LEVELS = {'0%': 0.00, '0.5%': 0.005, '1%': 0.01, '2%': 0.02}
FREQS     = ['daily', 'weekly', 'monthly', 'quarterly']
MV_FREQS  = ['monthly', 'quarterly']

# ── 15a. Buy-and-Hold ─────────────────────────────────────────────────────────
print("\n── Running Buy-and-Hold ──")
bh_result  = run_buy_and_hold(returns, ALL_ASSETS, initial_capital=1e9)
bh_results = {'BuyAndHold_EW': bh_result}

# ── 15b. Equal-Weight ─────────────────────────────────────────────────────────
print("\n── Running Equal-Weight Rebalanced ──")
ew_results = {}
for freq in FREQS:
    for tc_label, tc_rate in TC_LEVELS.items():
        label = f"EW_{freq}_TC{tc_label}"
        ew_results[label] = run_equal_weight(
            returns, ALL_ASSETS, tc_rate=tc_rate,
            rebal_freq=freq, initial_capital=1e9)
        print(f"  ✓ {label}")

# ── 15c. Original 3-Stage Signal ─────────────────────────────────────────────
print("\n── Running Original 3-Stage Signal ──")
signal_results       = {}
signal_distributions = {}

for thresh in THRESHOLDS:
    sig_df_t    = build_signal_df(composite_score, equity_assets,
                                  macro_df.index, threshold=thresh)
    n_risk_on   = (sig_df_t[equity_assets[0]] == +1).sum()
    n_risk_off  = (sig_df_t[equity_assets[0]] == -1).sum()
    total       = len(sig_df_t)
    signal_distributions[thresh] = {'RISK_ON': n_risk_on, 'RISK_OFF': n_risk_off}
    print(f"  Threshold={thresh} | RISK_ON: {n_risk_on} ({n_risk_on/total*100:.1f}%) "
          f"| RISK_OFF: {n_risk_off} ({n_risk_off/total*100:.1f}%)")

    for freq in FREQS:
        for tc_label, tc_rate in TC_LEVELS.items():
            label = f"Sig3S_t{thresh}_{freq}_TC{tc_label}"

            def wfn_3s(sig_row, _eq=equity_assets, _ne=non_equity_assets,
                       _bw=base_weight, _sh=EQUITY_SAFE_HAVEN_MAP):
                return build_weights_3stage(sig_row, _eq, _ne, _bw, None, _sh)

            signal_results[label] = run_signal_portfolio(
                returns, sig_df_t, equity_assets, non_equity_assets,
                ALL_ASSETS, weight_fn=wfn_3s,
                freq=freq, tc_rate=tc_rate, initial_capital=1e9)
            print(f"    ✓ {label}")

# ── 15d. Asymmetric Signal ────────────────────────────────────────────────────
print("\n── Running Asymmetric Signal ──")
asym_results = {}

for thresh in THRESHOLDS:
    sig_df_t = build_signal_df_asymmetric(composite_score, equity_assets,
                                          macro_df.index, threshold=thresh)
    for freq in ['monthly', 'quarterly']:
        for tc_label, tc_rate in TC_LEVELS.items():
            label = f"Asym_t{thresh}_{freq}_TC{tc_label}"

            def wfn_asym(sig_row, _eq=equity_assets, _ne=non_equity_assets,
                         _bw=base_weight, _sh=EQUITY_SAFE_HAVEN_MAP):
                return build_weights_3stage(sig_row, _eq, _ne, _bw, None, _sh)

            asym_results[label] = run_signal_portfolio(
                returns, sig_df_t, equity_assets, non_equity_assets,
                ALL_ASSETS, weight_fn=wfn_asym,
                freq=freq, tc_rate=tc_rate, initial_capital=1e9)
            print(f"    ✓ {label}")

# ── 15e. Continuous / Graduated Signal ───────────────────────────────────────
print("\n── Running Continuous Signal ──")
cont_results = {}

for thresh in THRESHOLDS:
    for max_red in MAX_REDUCES:
        sig_df_t = build_signal_df_continuous(composite_score, equity_assets,
                                              macro_df.index,
                                              threshold=thresh,
                                              max_reduce=max_red)
        for freq in ['monthly', 'quarterly']:
            for tc_label, tc_rate in TC_LEVELS.items():
                label = (f"Cont_t{thresh}_mr{int(max_red*100)}"
                         f"_{freq}_TC{tc_label}")

                def wfn_cont(sig_row, _eq=equity_assets, _ne=non_equity_assets,
                             _bw=base_weight, _sh=EQUITY_SAFE_HAVEN_MAP):
                    return build_weights_continuous(sig_row, _eq, _ne, _bw, _sh)

                cont_results[label] = run_signal_portfolio(
                    returns, sig_df_t, equity_assets, non_equity_assets,
                    ALL_ASSETS, weight_fn=wfn_cont,
                    freq=freq, tc_rate=tc_rate, initial_capital=1e9)
                print(f"    ✓ {label}")

# ── 15f. Smoothed Signal ─────────────────────────────────────────────────────
print("\n── Running Smoothed Signal ──")
smooth_results = {}

for thresh in THRESHOLDS:
    for sw in SMOOTH_WINDOWS:
        sig_df_t = build_signal_df_smoothed(composite_score, equity_assets,
                                            macro_df.index,
                                            threshold=thresh,
                                            smooth_window=sw)
        for freq in ['monthly', 'quarterly']:
            for tc_label, tc_rate in TC_LEVELS.items():
                label = f"Smooth_t{thresh}_sw{sw}_{freq}_TC{tc_label}"

                def wfn_smooth(sig_row, _eq=equity_assets, _ne=non_equity_assets,
                               _bw=base_weight, _sh=EQUITY_SAFE_HAVEN_MAP):
                    return build_weights_3stage(sig_row, _eq, _ne, _bw, None, _sh)

                smooth_results[label] = run_signal_portfolio(
                    returns, sig_df_t, equity_assets, non_equity_assets,
                    ALL_ASSETS, weight_fn=wfn_smooth,
                    freq=freq, tc_rate=tc_rate, initial_capital=1e9)
                print(f"    ✓ {label}")

# ── 15g. Persistent Signal ────────────────────────────────────────────────────
print("\n── Running Persistent Signal ──")
persist_results = {}

for thresh in THRESHOLDS:
    for min_days in MIN_DAYS_LIST:
        sig_df_t = build_signal_df_persistent(composite_score, equity_assets,
                                              macro_df.index,
                                              threshold=thresh,
                                              min_days_to_switch=min_days)
        for freq in ['monthly', 'quarterly']:
            for tc_label, tc_rate in TC_LEVELS.items():
                label = f"Persist_t{thresh}_md{min_days}_{freq}_TC{tc_label}"

                def wfn_persist(sig_row, _eq=equity_assets, _ne=non_equity_assets,
                                _bw=base_weight, _sh=EQUITY_SAFE_HAVEN_MAP):
                    return build_weights_3stage(sig_row, _eq, _ne, _bw, None, _sh)

                persist_results[label] = run_signal_portfolio(
                    returns, sig_df_t, equity_assets, non_equity_assets,
                    ALL_ASSETS, weight_fn=wfn_persist,
                    freq=freq, tc_rate=tc_rate, initial_capital=1e9)
                print(f"    ✓ {label}")

# ── 15h. Partial-Hedge Signal ─────────────────────────────────────────────────
print("\n── Running Partial-Hedge Signal ──")
partial_results = {}

for thresh in THRESHOLDS:
    sig_df_t = build_signal_df(composite_score, equity_assets,
                               macro_df.index, threshold=thresh)
    for pk in PARTIAL_KEEPS:
        for freq in ['monthly', 'quarterly']:
            for tc_label, tc_rate in TC_LEVELS.items():
                label = (f"Partial_t{thresh}_pk{int(pk*100)}"
                         f"_{freq}_TC{tc_label}")

                def wfn_partial(sig_row, _eq=equity_assets, _ne=non_equity_assets,
                                _bw=base_weight, _sh=EQUITY_SAFE_HAVEN_MAP,
                                _pk=pk):
                    return build_weights_partial(sig_row, _eq, _ne, _bw, _sh, _pk)

                partial_results[label] = run_signal_portfolio(
                    returns, sig_df_t, equity_assets, non_equity_assets,
                    ALL_ASSETS, weight_fn=wfn_partial,
                    freq=freq, tc_rate=tc_rate, initial_capital=1e9)
                print(f"    ✓ {label}")

# ── 15i. Mean-Variance Signal ─────────────────────────────────────────────────
print("\n── Running Mean-Variance Signal ──")
mv_results        = {}
mv_weight_history = {}

for thresh in THRESHOLDS:
    sig_df_t = build_signal_df(composite_score, equity_assets,
                               macro_df.index, threshold=thresh)
    for freq in MV_FREQS:
        for tc_label, tc_rate in TC_LEVELS.items():
            label = f"MV_t{thresh}_{freq}_TC{tc_label}"
            pv_df, w_df = run_mv_signal_portfolio(
                returns, sig_df_t, ALL_ASSETS,
                equity_assets=equity_assets,
                freq=freq, tc_rate=tc_rate,
                lookback=MV_LOOKBACK, min_obs=MV_MIN_OBS,
                max_weight=MAX_WEIGHT, initial_capital=1e9)
            mv_results[label]        = pv_df
            mv_weight_history[label] = w_df
            print(f"  ✓ {label}")

print("\n✓ All portfolios computed")

# ── 16. PERFORMANCE METRICS ───────────────────────────────────────────────────
def compute_metrics(pv_df, label=''):
    pv        = pv_df['portfolio_value']
    daily_ret = pv.pct_change().dropna()
    n_years   = len(daily_ret) / 252
    total_ret = (pv.iloc[-1] / pv.iloc[0]) - 1
    cagr      = (1 + total_ret) ** (1 / n_years) - 1 if n_years > 0 else np.nan
    vol       = daily_ret.std() * np.sqrt(252)
    sharpe    = cagr / vol if vol > 0 else np.nan
    roll_max  = pv.cummax()
    max_dd    = ((pv - roll_max) / roll_max).min()
    calmar    = cagr / abs(max_dd) if max_dd != 0 else np.nan
    var_95    = daily_ret.quantile(0.05)
    ann_ret   = cagr
    ann_vol   = vol

    return {
        'CAGR':         f"{cagr*100:.2f}%",
        'Ann_Ret':      ann_ret,
        'Ann_Vol':      ann_vol,
        'Volatility':   f"{vol*100:.2f}%",
        'Sharpe':       f"{sharpe:.2f}",
        'Max Drawdown': f"{max_dd*100:.2f}%",
        'Calmar':       f"{calmar:.2f}",
        'VaR 95%':      f"{var_95*100:.2f}%",
        'Total Return': f"{total_ret*100:.2f}%",
        'Final ($B)':   f"{pv.iloc[-1]/1e9:.3f}",
    }

bh_metrics      = {k: compute_metrics(v, k) for k, v in bh_results.items()}
ew_metrics      = {k: compute_metrics(v, k) for k, v in ew_results.items()}
sig_metrics     = {k: compute_metrics(v, k) for k, v in signal_results.items()}
asym_metrics    = {k: compute_metrics(v, k) for k, v in asym_results.items()}
cont_metrics    = {k: compute_metrics(v, k) for k, v in cont_results.items()}
smooth_metrics  = {k: compute_metrics(v, k) for k, v in smooth_results.items()}
persist_metrics = {k: compute_metrics(v, k) for k, v in persist_results.items()}
partial_metrics = {k: compute_metrics(v, k) for k, v in partial_results.items()}
mv_metrics      = {k: compute_metrics(v, k) for k, v in mv_results.items()}

def print_metrics_table(metrics_dict, title):
    display = {k: {mk: mv for mk, mv in m.items()
                   if mk not in ('Ann_Ret', 'Ann_Vol')}
               for k, m in metrics_dict.items()}
    print(f"\n{'='*90}")
    print(title)
    print("="*90)
    print(pd.DataFrame(display).T.to_string())

print_metrics_table(bh_metrics,      "BUY-AND-HOLD")
print_metrics_table(ew_metrics,      "EQUAL-WEIGHT REBALANCED")

for thresh in THRESHOLDS:
    sub = {k: v for k, v in sig_metrics.items()     if f"_t{thresh}_" in k}
    print_metrics_table(sub, f"3-STAGE SIGNAL — threshold={thresh}")

for thresh in THRESHOLDS:
    sub = {k: v for k, v in asym_metrics.items()    if f"_t{thresh}_" in k}
    print_metrics_table(sub, f"ASYMMETRIC SIGNAL — threshold={thresh}")

for thresh in THRESHOLDS:
    sub = {k: v for k, v in cont_metrics.items()    if f"_t{thresh}_" in k}
    print_metrics_table(sub, f"CONTINUOUS SIGNAL — threshold={thresh}")

for thresh in THRESHOLDS:
    sub = {k: v for k, v in smooth_metrics.items()  if f"_t{thresh}_" in k}
    print_metrics_table(sub, f"SMOOTHED SIGNAL — threshold={thresh}")

for thresh in THRESHOLDS:
    sub = {k: v for k, v in persist_metrics.items() if f"_t{thresh}_" in k}
    print_metrics_table(sub, f"PERSISTENT SIGNAL — threshold={thresh}")

for thresh in THRESHOLDS:
    sub = {k: v for k, v in partial_metrics.items() if f"_t{thresh}_" in k}
    print_metrics_table(sub, f"PARTIAL-HEDGE SIGNAL — threshold={thresh}")

for thresh in THRESHOLDS:
    sub = {k: v for k, v in mv_metrics.items()      if f"_t{thresh}_" in k}
    print_metrics_table(sub, f"MEAN-VARIANCE SIGNAL — threshold={thresh}")

# ── 17. PLOTS ─────────────────────────────────────────────────────────────────
bh_pv     = bh_results['BuyAndHold_EW']['portfolio_value']
ew_ref_pv = ew_results['EW_monthly_TC0.5%']['portfolio_value']

def parse_pct(s):
    return float(str(s).replace('%', '')) / 100

# ── Plot A: Best of each variant vs benchmarks (monthly, TC=0.5%) ─────────────
fig, axes = plt.subplots(2, 4, figsize=(28, 12), sharey=False)
fig.suptitle("All Signal Variants vs Benchmarks — Monthly Rebalancing, TC=0.5%",
             fontsize=14, fontweight='bold')

variant_info = [
    ("3-Stage",    signal_results,  THRESHOLDS,     "Sig3S",   "steelblue"),
    ("Asymmetric", asym_results,    THRESHOLDS,     "Asym",    "darkorange"),
    ("Continuous", cont_results,    THRESHOLDS,     "Cont",    "purple"),
    ("Smoothed",   smooth_results,  THRESHOLDS,     "Smooth",  "green"),
    ("Persistent", persist_results, THRESHOLDS,     "Persist", "crimson"),
    ("Partial 25%",partial_results, THRESHOLDS,     "Partial", "brown"),
    ("MV Signal",  mv_results,      THRESHOLDS,     "MV",      "black"),
]

for ax, (name, res_dict, threshs, prefix, col) in zip(axes.flat, variant_info):
    ax.plot(bh_pv.index,     bh_pv / 1e9,     color='forestgreen',
            lw=1.5, ls='-.', label='Buy & Hold')
    ax.plot(ew_ref_pv.index, ew_ref_pv / 1e9, color='grey',
            lw=1.2, ls='--', label='EW Monthly')

    for thresh in threshs:
        candidates = [k for k in res_dict
                      if f"_t{thresh}_" in k and 'monthly' in k and 'TC0.5%' in k]
        if not candidates:
            candidates = [k for k in res_dict
                          if f"_t{thresh}_" in k and 'TC0.5%' in k]
        if candidates:
            key = candidates[0]
            pv  = res_dict[key]['portfolio_value']
            ax.plot(pv.index, pv / 1e9, lw=1.6,
                    label=f"t={thresh}")

    ax.axhline(1.0, color='black', ls=':', lw=0.7)
    ax.set_title(name, fontsize=11, fontweight='bold')
    ax.set_ylabel("Value ($B)")
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

axes.flat[-1].set_visible(False)
plt.tight_layout()
plt.savefig("all_variants_monthly.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Plot B: Risk-Return Scatter — all strategies ──────────────────────────────
fig, ax = plt.subplots(figsize=(16, 10))

all_metric_groups = [
    (bh_metrics,      'BH',       'forestgreen', 'D'),
    (ew_metrics,      'EW',       'grey',        's'),
    (sig_metrics,     '3-Stage',  'steelblue',   'o'),
    (asym_metrics,    'Asym',     'darkorange',  'o'),
    (cont_metrics,    'Cont',     'purple',      'o'),
    (smooth_metrics,  'Smooth',   'green',       'o'),
    (persist_metrics, 'Persist',  'crimson',     'o'),
    (partial_metrics, 'Partial',  'brown',       'o'),
    (mv_metrics,      'MV',       'black',       '^'),
]

for metrics_dict, grp_label, col, marker in all_metric_groups:
    first = True
    for key, m in metrics_dict.items():
        vol = m['Ann_Vol'] * 100
        ret = m['Ann_Ret'] * 100
        ax.scatter(vol, ret, color=col, marker=marker, s=80,
                   label=grp_label if first else None,
                   edgecolors='black', lw=0.5, zorder=4)
        ax.annotate(key.split('_TC')[0], (vol, ret),
                    textcoords='offset points', xytext=(4, 3),
                    fontsize=6, color=col)
        first = False

ax.axhline(0, color='black', lw=0.6, ls='--')
ax.axvline(0, color='black', lw=0.6, ls='--')
ax.set_xlabel("Annualised Volatility (%)", fontsize=12)
ax.set_ylabel("CAGR (%)", fontsize=12)
ax.set_title("Risk-Return — All Strategies", fontsize=13, fontweight='bold')
ax.legend(fontsize=10, ncol=3)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("risk_return_all.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Plot C: Sharpe Bar Chart — top 20 strategies ──────────────────────────────
all_metrics_combined = {
    **bh_metrics, **ew_metrics, **sig_metrics,
    **asym_metrics, **cont_metrics, **smooth_metrics,
    **persist_metrics, **partial_metrics, **mv_metrics
}

sharpe_series = pd.Series({
    k: float(v['Sharpe']) for k, v in all_metrics_combined.items()
}).dropna().sort_values(ascending=False)

top20 = sharpe_series.head(20)

def label_color(lbl):
    if lbl.startswith('MV'):       return 'black'
    if lbl.startswith('Sig3S'):    return 'steelblue'
    if lbl.startswith('Asym'):     return 'darkorange'
    if lbl.startswith('Cont'):     return 'purple'
    if lbl.startswith('Smooth'):   return 'green'
    if lbl.startswith('Persist'):  return 'crimson'
    if lbl.startswith('Partial'):  return 'brown'
    if lbl.startswith('EW'):       return 'grey'
    return 'forestgreen'

colors = [label_color(k) for k in top20.index]

fig, ax = plt.subplots(figsize=(16, 7))
bars = ax.barh(range(len(top20)), top20.values, color=colors, edgecolor='black', lw=0.5)
ax.set_yticks(range(len(top20)))
ax.set_yticklabels([k.replace('_TC', ' TC=') for k in top20.index], fontsize=8)
ax.set_xlabel("Sharpe Ratio")
ax.set_title("Top 20 Strategies by Sharpe Ratio", fontsize=13, fontweight='bold')
ax.axvline(0, color='black', lw=0.8)
ax.grid(True, axis='x', alpha=0.3)

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='black',      label='MV Signal'),
    Patch(facecolor='steelblue',  label='3-Stage'),
    Patch(facecolor='darkorange', label='Asymmetric'),
    Patch(facecolor='purple',     label='Continuous'),
    Patch(facecolor='green',      label='Smoothed'),
    Patch(facecolor='crimson',    label='Persistent'),
    Patch(facecolor='brown',      label='Partial Hedge'),
    Patch(facecolor='grey',       label='Equal Weight'),
    Patch(facecolor='forestgreen',label='Buy & Hold'),
]
ax.legend(handles=legend_elements, fontsize=9, loc='lower right')
plt.tight_layout()
plt.savefig("sharpe_top20.png", dpi=150, bbox_inches='tight')
plt.show()

# ── 18. SAVE ALL ──────────────────────────────────────────────────────────────
print("\n── Saving all results ──")
all_portfolios = {
    **bh_results, **ew_results, **signal_results,
    **asym_results, **cont_results, **smooth_results,
    **persist_results, **partial_results, **mv_results
}

for label, df in all_portfolios.items():
    df.to_csv(f"{label}_portfolio.csv")

all_metrics_df = pd.DataFrame({
    k: {mk: mv for mk, mv in m.items() if mk not in ('Ann_Ret', 'Ann_Vol')}
    for k, m in all_metrics_combined.items()
}).T
all_metrics_df.to_csv("all_portfolio_metrics.csv")

composite_score.to_csv("composite_signal.csv")
for thresh in THRESHOLDS:
    sig_df_t = build_signal_df(composite_score, equity_assets,
                               macro_df.index, threshold=thresh)
    sig_df_t.to_csv(f"signal_df_threshold_{thresh}.csv")

for label, wdf in mv_weight_history.items():
    wdf.to_csv(f"{label}_weights.csv")

print("✓ All files saved.")
print(f"\n  Portfolio CSVs   : {len(all_portfolios)}")
print(f"  Weight CSVs      : {len(mv_weight_history)}")
print(f"  Metrics CSV      : all_portfolio_metrics.csv")
print(f"  Signal CSVs      : {len(THRESHOLDS)} threshold files")
print(f"  Plots saved      : 3 PNG files")


In [ ]:
# @title
# ============================================================
# SIGNAL PORTFOLIO — ENHANCED VERSION
# Signals generated from Modified Determinant Model
# + Buy-and-Hold Benchmark
# + Equal-Weight Benchmark
# + Three-Stage Signal Portfolio (original)
# + Enhanced Signal Variants (asymmetric, continuous, smoothed, persistent, partial)
# + Mean-Variance Signal Portfolio
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import requests
from io import StringIO
from scipy.optimize import minimize

# ── 1. LOAD ALL DATA FROM GITHUB ─────────────────────────────────────────────
GITHUB_BASE = "https://raw.githubusercontent.com/kboroz/MSFE_Capstone_Project/main/01_Data/Streamlined/"

INDEX_FILE_MAP = {
    'ASX50':       'streamlined_asx_50.csv',
    'EUROSTOXX50': 'streamlined_euro_stoxx_50.csv',
    'FTSE100':     'streamlined_ftse_100.csv',
    'Ibovespa':    'streamlined_ibovespa.csv',
    'JSE40':       'streamlined_jse_top_40.csv',
    'NIKKEI225':   'streamlined_nikkei_225.csv',
    'SP500':       'streamlined_s&p_500.csv',
}

MACRO_FILE = 'streamlined_macro.csv'

def load_csv(url):
    r = requests.get(url)
    r.raise_for_status()
    df = pd.read_csv(StringIO(r.text), index_col=0, parse_dates=True)
    df.columns = df.columns.str.replace('[^a-zA-Z0-9]', '_', regex=True)
    return df.sort_index()

macro_df = load_csv(GITHUB_BASE + MACRO_FILE)
macro_df = macro_df[~macro_df.index.duplicated(keep='first')].sort_index()
print(f"macro_df shape : {macro_df.shape}")
print(f"Columns        : {macro_df.columns.tolist()}")

index_data = {}
for name, fname in INDEX_FILE_MAP.items():
    try:
        df = load_csv(GITHUB_BASE + fname)
        index_data[name] = df
        print(f"✓ {name:15s} | shape={df.shape} | cols={df.columns.tolist()[:4]}")
    except Exception as e:
        print(f"✗ {name}: {e}")

# ── 2. EXTRACT PRICE SERIES ───────────────────────────────────────────────────
PRICE_COLS = ['Close', 'close', 'Price', 'price', 'Adj_Close', 'Last']

def get_price_series(df, name):
    col = next((c for c in PRICE_COLS if c in df.columns), df.columns[0])
    s = pd.to_numeric(df[col], errors='coerce').dropna()
    print(f"  {name}: using column '{col}'")
    return s

index_prices = {}
for name, df in index_data.items():
    index_prices[name] = get_price_series(df, name)

# ── 3. MODIFIED DETERMINANT SIGNAL MODEL ─────────────────────────────────────
DET_WINDOWS = [3, 7, 28]
WINDOW_WEIGHTS = {3: 0.2, 7: 0.3, 28: 0.5}

def compute_modified_determinant(price_series_dict, windows=DET_WINDOWS):
    prices = pd.DataFrame(price_series_dict).sort_index()
    rets   = prices.pct_change().dropna()
    det_results = {}

    for window in windows:
        det_series = []
        for i in range(window, len(rets)):
            window_rets = rets.iloc[i - window:i]
            valid_cols  = window_rets.columns[window_rets.std() > 1e-10]
            w           = window_rets[valid_cols]

            if len(valid_cols) < 2:
                det_series.append(np.nan)
                continue

            try:
                R   = w.corr().values
                I   = np.eye(len(R))
                M   = R - I
                det = np.linalg.det(M)
            except Exception:
                det = np.nan

            det_series.append(det)

        idx = rets.index[window:]
        det_results[window] = pd.Series(det_series, index=idx,
                                        name=f'det_{window}d')
        print(f"  Window {window:2d}d | computed {len(det_series)} observations")

    return det_results

print("\n── Computing Modified Determinant ──")
det_results = compute_modified_determinant(index_prices)

# ── 4. COMPOSITE SCORE ────────────────────────────────────────────────────────
def compute_composite_score(det_results, window_weights=WINDOW_WEIGHTS):
    all_series = []
    for window, series in det_results.items():
        mu    = series.rolling(252, min_periods=60).mean()
        sigma = series.rolling(252, min_periods=60).std()
        z     = (series - mu) / sigma.replace(0, np.nan)
        z.name = f'z_{window}d'
        all_series.append(z)

    z_df      = pd.concat(all_series, axis=1).dropna()
    composite = sum(
        z_df[f'z_{w}d'] * window_weights[w]
        for w in window_weights if f'z_{w}d' in z_df.columns
    )
    composite = composite / sum(window_weights.values())
    return composite, z_df

composite_score, z_df = compute_composite_score(det_results)

print(f"\nComposite score shape  : {composite_score.shape}")
print(f"Date range             : {composite_score.index[0].date()} → "
      f"{composite_score.index[-1].date()}")
print(f"Score range            : {composite_score.min():.3f} → "
      f"{composite_score.max():.3f}")

# ── 5. SIGNAL BUILDERS ───────────────────────────────────────────────────────
THRESHOLDS    = [0.3, 0.5, 0.7]
SMOOTH_WINDOWS = [3, 5, 10]
MIN_DAYS_LIST  = [5, 10, 20]
MAX_REDUCES    = [0.5, 0.7, 0.9]
PARTIAL_KEEPS  = [0.25, 0.50]

# ── 5a. Original symmetric binary ────────────────────────────────────────────
def classify_signal_symmetric(score, threshold):
    return +1 if abs(score) > threshold else -1

def build_signal_df(composite_score, equity_assets, macro_dates, threshold=0.5):
    binary    = composite_score.apply(lambda x: classify_signal_symmetric(x, threshold))
    signal_df = pd.DataFrame(index=macro_dates)
    for asset in equity_assets:
        signal_df[asset] = binary.reindex(macro_dates).ffill().bfill()
    return signal_df

# ── 5b. Asymmetric (upper-tail only) ─────────────────────────────────────────
def build_signal_df_asymmetric(composite_score, equity_assets, macro_dates,
                               threshold=1.0):
    """Block equities only when correlation SPIKES (high positive z-score)."""
    binary    = composite_score.apply(lambda x: +1 if x > threshold else -1)
    signal_df = pd.DataFrame(index=macro_dates)
    for asset in equity_assets:
        signal_df[asset] = binary.reindex(macro_dates).ffill().bfill()
    return signal_df

# ── 5c. Continuous / graduated scalar ────────────────────────────────────────
def build_signal_df_continuous(composite_score, equity_assets, macro_dates,
                               threshold=1.0, max_reduce=0.8):
    """
    Returns a scalar in [1-max_reduce, 1.0] per asset per day.
    Used by build_weights_continuous() instead of the binary flag.
    """
    def score_to_scalar(s):
        if s <= 0:
            return 1.0
        elif s >= threshold:
            return 1.0 - max_reduce
        else:
            return 1.0 - max_reduce * (s / threshold)

    scalars   = composite_score.apply(score_to_scalar)
    signal_df = pd.DataFrame(index=macro_dates)
    for asset in equity_assets:
        signal_df[asset] = scalars.reindex(macro_dates).ffill().bfill()
    return signal_df

# ── 5d. Smoothed binary ───────────────────────────────────────────────────────
def build_signal_df_smoothed(composite_score, equity_assets, macro_dates,
                             threshold=1.0, smooth_window=5):
    """Smooth z-score before thresholding to reduce whipsaw."""
    smoothed  = composite_score.rolling(smooth_window, min_periods=1).mean()
    binary    = smoothed.apply(lambda x: classify_signal_symmetric(x, threshold))
    signal_df = pd.DataFrame(index=macro_dates)
    for asset in equity_assets:
        signal_df[asset] = binary.reindex(macro_dates).ffill().bfill()
    return signal_df

# ── 5e. Persistent binary ────────────────────────────────────────────────────
def build_signal_df_persistent(composite_score, equity_assets, macro_dates,
                               threshold=1.0, min_days_to_switch=10):
    """Only switch regime after signal holds for min_days_to_switch days."""
    raw       = composite_score.apply(lambda x: +1 if x > threshold else -1)
    confirmed = raw.copy()
    current   = int(raw.iloc[0])
    count     = 0

    for i in range(1, len(raw)):
        if int(raw.iloc[i]) != current:
            count += 1
            if count >= min_days_to_switch:
                current = int(raw.iloc[i])
                count   = 0
        else:
            count = 0
        confirmed.iloc[i] = current

    signal_df = pd.DataFrame(index=macro_dates)
    for asset in equity_assets:
        signal_df[asset] = confirmed.reindex(macro_dates).ffill().bfill()
    return signal_df

# ── 6. ASSET UNIVERSE ─────────────────────────────────────────────────────────
EQUITY_ASSETS = list(index_prices.keys())
SAFE_BONDS    = ['US_10Y', 'UK_10Y', 'AU_10Y', 'ZA_10Y']
FX            = ['AUDUSD', 'BRLUSD', 'EURUSD', 'GBPUSD', 'JPYUSD', 'ZARUSD']
COMMODITIES   = ['Oil', 'Gold', 'Bitcoin']
EXCLUDE_ASSETS = ['Bitcoin']

ALL_ASSETS = EQUITY_ASSETS + FX + COMMODITIES + SAFE_BONDS
ALL_ASSETS = [a for a in ALL_ASSETS
              if a in macro_df.columns and a not in EXCLUDE_ASSETS]

equity_assets     = [a for a in EQUITY_ASSETS     if a in ALL_ASSETS]
non_equity_assets = [a for a in ALL_ASSETS         if a not in equity_assets]

N           = len(ALL_ASSETS)
base_weight = 1.0 / N

print(f"\nTotal assets              : {N}")
print(f"Equity (signal-driven)    : {equity_assets}")
print(f"Non-equity (always long)  : {non_equity_assets}")
print(f"base_weight = 1/{N} = {base_weight:.4f}")

# ── 7. COUNTRY SAFE-HAVEN MAP ─────────────────────────────────────────────────
EQUITY_SAFE_HAVEN_MAP = {
    'ASX50':       ['AU_10Y', 'AUDUSD'],
    'EUROSTOXX50': ['EURUSD'],
    'FTSE100':     ['UK_10Y', 'GBPUSD'],
    'Ibovespa':    ['BRLUSD'],
    'JSE40':       ['ZA_10Y', 'ZARUSD'],
    'NIKKEI225':   ['JPYUSD'],
    'SP500':       ['US_10Y'],
}

UNIVERSE_SET = set(ALL_ASSETS)
for eq, havens in EQUITY_SAFE_HAVEN_MAP.items():
    EQUITY_SAFE_HAVEN_MAP[eq] = [h for h in havens if h in UNIVERSE_SET]

print("\nSafe-haven routing (filtered to universe):")
for eq, havens in EQUITY_SAFE_HAVEN_MAP.items():
    print(f"  {eq:15s} → {havens}")

# ── 8. BUILD RETURNS ──────────────────────────────────────────────────────────
returns_raw = macro_df[ALL_ASSETS].pct_change(fill_method=None)

for col in SAFE_BONDS:
    if col in returns_raw.columns:
        returns_raw[col] = -returns_raw[col]

if 'Oil' in macro_df.columns:
    bad_oil = macro_df['Oil'][macro_df['Oil'] <= 0].index
    for d in bad_oil:
        loc = returns_raw.index.get_loc(d)
        returns_raw.iloc[max(0, loc - 1): loc + 2,
                         returns_raw.columns.get_loc('Oil')] = 0.0

returns = returns_raw.clip(lower=-0.50, upper=1.00).fillna(0).iloc[1:]

assert not returns.isnull().any().any(), "NaNs in returns"
assert not np.isinf(returns.values).any(),  "Inf in returns"
print(f"\nReturns shape : {returns.shape}")
print("✓ Returns clean")

# ── 9. HELPER: REBALANCING DATE SET ──────────────────────────────────────────
def build_reb_set(returns, rebal_freq):
    actual_dates = returns.index
    if rebal_freq == 'daily':
        return set(actual_dates)
    elif rebal_freq == 'weekly':
        raw = returns.resample('W-MON').first().index
    elif rebal_freq == 'monthly':
        raw = returns.resample('MS').first().index
    elif rebal_freq == 'quarterly':
        raw = returns.resample('QS').first().index
    else:
        raise ValueError(f"Unknown rebal_freq: {rebal_freq}")

    reb_set = set()
    for d in raw:
        future = actual_dates[actual_dates >= d]
        if len(future) > 0:
            reb_set.add(future[0])
    return reb_set

# ── 10. BUY-AND-HOLD BENCHMARK ───────────────────────────────────────────────
def run_buy_and_hold(returns, all_assets, initial_capital=1e9):
    N           = len(all_assets)
    base_weight = 1.0 / N
    pv          = float(initial_capital)
    holdings    = {a: pv * base_weight for a in all_assets}
    records     = []
    first       = True

    for date in returns.index:
        if first:
            first = False
            records.append({'date': date, 'portfolio_value': pv})
            continue
        new_holdings = {a: holdings[a] * (1 + returns.loc[date, a])
                        for a in all_assets}
        holdings = new_holdings
        pv       = sum(holdings.values())
        records.append({'date': date, 'portfolio_value': pv})

    df = pd.DataFrame(records).set_index('date')
    print(f"  Buy-and-Hold | Start: ${initial_capital/1e9:.2f}B → "
          f"End: ${df['portfolio_value'].iloc[-1]/1e9:.3f}B")
    return df

# ── 11. WEIGHT BUILDERS ───────────────────────────────────────────────────────

# ── 11a. Original binary 3-stage weights ─────────────────────────────────────
def build_weights_3stage(sig_row, equity_assets, non_equity_assets,
                         base_weight, date, safe_haven_map):
    raw            = {a: base_weight for a in non_equity_assets}
    safe_haven_extra = {}

    for a in equity_assets:
        if sig_row is not None and a in sig_row.index and not pd.isna(sig_row[a]):
            sig = sig_row[a]
        else:
            sig = -1

        if sig > 0:
            raw[a] = -base_weight
            havens = [h for h in safe_haven_map.get(a, []) if h in raw]
            if havens:
                per_haven = base_weight / len(havens)
                for h in havens:
                    safe_haven_extra[h] = safe_haven_extra.get(h, 0) + per_haven
            else:
                per_ne = base_weight / len(non_equity_assets)
                for ne in non_equity_assets:
                    safe_haven_extra[ne] = safe_haven_extra.get(ne, 0) + per_ne
        else:
            raw[a] = base_weight

    for asset, extra in safe_haven_extra.items():
        raw[asset] = raw.get(asset, 0) + extra

    total = sum(raw.values())
    if total <= 0:
        for a in raw:
            raw[a] = 1.0 / len(raw)
    else:
        for a in raw:
            raw[a] /= total
    return raw

# ── 11b. Continuous scalar weights ───────────────────────────────────────────
def build_weights_continuous(sig_row, equity_assets, non_equity_assets,
                             base_weight, safe_haven_map):
    """
    sig_row contains a scalar in [min_weight, 1.0] per equity asset.
    Reduced equity weight is redistributed to safe havens.
    """
    raw              = {a: base_weight for a in non_equity_assets}
    safe_haven_extra = {}

    for a in equity_assets:
        scalar = float(sig_row[a]) if (sig_row is not None and
                                       a in sig_row.index and
                                       not pd.isna(sig_row[a])) else 1.0
        raw[a]   = base_weight * scalar
        released = base_weight * (1.0 - scalar)

        if released > 0:
            havens = [h for h in safe_haven_map.get(a, []) if h in raw]
            if havens:
                per_haven = released / len(havens)
                for h in havens:
                    safe_haven_extra[h] = safe_haven_extra.get(h, 0) + per_haven
            else:
                per_ne = released / len(non_equity_assets)
                for ne in non_equity_assets:
                    safe_haven_extra[ne] = safe_haven_extra.get(ne, 0) + per_ne

    for asset, extra in safe_haven_extra.items():
        raw[asset] = raw.get(asset, 0) + extra

    total = sum(raw.values())
    if total <= 0:
        for a in raw:
            raw[a] = 1.0 / len(raw)
    else:
        for a in raw:
            raw[a] /= total
    return raw

# ── 11c. Partial-hedge weights ────────────────────────────────────────────────
def build_weights_partial(sig_row, equity_assets, non_equity_assets,
                          base_weight, safe_haven_map, partial_keep=0.25):
    """
    On RISK_ON signal: keep partial_keep fraction of base_weight in equity,
    redirect the rest to safe havens.
    """
    raw              = {a: base_weight for a in non_equity_assets}
    safe_haven_extra = {}

    for a in equity_assets:
        if sig_row is not None and a in sig_row.index and not pd.isna(sig_row[a]):
            sig = sig_row[a]
        else:
            sig = -1

        if sig > 0:
            raw[a]   = base_weight * partial_keep
            released = base_weight * (1.0 - partial_keep)
            havens   = [h for h in safe_haven_map.get(a, []) if h in raw]
            if havens:
                per_haven = released / len(havens)
                for h in havens:
                    safe_haven_extra[h] = safe_haven_extra.get(h, 0) + per_haven
            else:
                per_ne = released / len(non_equity_assets)
                for ne in non_equity_assets:
                    safe_haven_extra[ne] = safe_haven_extra.get(ne, 0) + per_ne
        else:
            raw[a] = base_weight

    for asset, extra in safe_haven_extra.items():
        raw[asset] = raw.get(asset, 0) + extra

    total = sum(raw.values())
    if total <= 0:
        for a in raw:
            raw[a] = 1.0 / len(raw)
    else:
        for a in raw:
            raw[a] /= total
    return raw

# ── 12. EQUAL-WEIGHT REBALANCED BENCHMARK ────────────────────────────────────
def run_equal_weight(returns, all_assets, tc_rate=0.005,
                     rebal_freq='monthly', initial_capital=1e9):
    N           = len(all_assets)
    base_weight = 1.0 / N
    ew          = {a: base_weight for a in all_assets}
    reb_set     = build_reb_set(returns, rebal_freq)
    pv          = float(initial_capital)
    current_weights = None
    records     = []

    for date in returns.index:
        if date in reb_set:
            if current_weights is not None:
                turnover = sum(abs(ew.get(a, 0) - current_weights.get(a, 0))
                               for a in set(ew) | set(current_weights)) / 2.0
                pv *= (1 - turnover * tc_rate)
            else:
                pv *= (1 - tc_rate)
            current_weights = ew.copy()

        if current_weights is not None:
            dr  = sum(current_weights.get(a, 0) * returns.loc[date, a]
                      for a in all_assets)
            pv *= (1 + dr)

        records.append({'date': date, 'portfolio_value': pv})

    return pd.DataFrame(records).set_index('date')

# ── 13. GENERIC SIGNAL PORTFOLIO RUNNER ──────────────────────────────────────
def run_signal_portfolio(returns, signal_df, equity_assets,
                         non_equity_assets, all_assets,
                         weight_fn,          # callable: (sig_row, ...) → dict
                         freq='monthly', tc_rate=0.005,
                         initial_capital=1e9):
    """
    Generic runner.  weight_fn must accept (sig_row, equity_assets,
    non_equity_assets, base_weight, <any extra kwargs>) and return
    a dict {asset: weight}.
    """
    N           = len(all_assets)
    base_weight = 1.0 / N
    reb_set     = build_reb_set(returns, freq)

    sig = signal_df.copy()
    sig.index = pd.to_datetime(sig.index)

    pv              = float(initial_capital)
    current_weights = None
    records         = []

    for date in returns.index:
        if date in reb_set:
            past    = sig[sig.index <= date]
            sig_row = past.iloc[-1] if len(past) > 0 else None
            raw     = weight_fn(sig_row)

            if current_weights is not None:
                all_keys = set(raw) | set(current_weights)
                turnover = sum(abs(raw.get(a, 0) - current_weights.get(a, 0))
                               for a in all_keys) / 2.0
                pv *= (1 - turnover * tc_rate)
            else:
                pv *= (1 - tc_rate)

            current_weights = raw.copy()

        if current_weights is not None:
            dr  = sum(current_weights.get(a, 0) * returns.loc[date, a]
                      for a in all_assets)
            pv *= (1 + dr)

        records.append({'date': date, 'portfolio_value': pv})

    return pd.DataFrame(records).set_index('date')

# ── 14. MEAN-VARIANCE SIGNAL PORTFOLIO ───────────────────────────────────────
MV_LOOKBACK = 252
MV_MIN_OBS  = 60
MAX_WEIGHT  = 0.30
RISK_FREE   = 0.0

def mean_variance_weights(mu_vec, cov_mat, allowed_mask,
                          max_weight=MAX_WEIGHT, risk_free=RISK_FREE):
    n_total   = len(mu_vec)
    allowed   = np.where(allowed_mask)[0]
    n_allowed = len(allowed)

    fallback = np.zeros(n_total)
    if n_allowed == 0:
        fallback[:] = 1.0 / n_total
        return fallback
    fallback[allowed] = 1.0 / n_allowed

    mu_sub  = mu_vec[allowed]
    cov_sub = cov_mat[np.ix_(allowed, allowed)] + np.eye(n_allowed) * 1e-8

    def neg_sharpe(w):
        port_ret = np.dot(w, mu_sub)
        port_var = np.dot(w, cov_sub @ w)
        port_std = np.sqrt(max(port_var, 1e-12))
        return -(port_ret - risk_free) / port_std

    constraints = [{'type': 'eq', 'fun': lambda w: np.sum(w) - 1.0}]
    bounds      = [(0.0, max_weight)] * n_allowed
    w0          = np.ones(n_allowed) / n_allowed

    try:
        res = minimize(neg_sharpe, w0, method='SLSQP',
                       bounds=bounds, constraints=constraints,
                       options={'ftol': 1e-9, 'maxiter': 500})
        w_opt = res.x if (res.success and not np.any(np.isnan(res.x))) else w0
    except Exception:
        w_opt = w0

    w_full         = np.zeros(n_total)
    w_full[allowed] = w_opt
    return w_full

def run_mv_signal_portfolio(returns, signal_df, all_assets,
                            equity_assets,
                            freq='monthly', tc_rate=0.005,
                            lookback=MV_LOOKBACK, min_obs=MV_MIN_OBS,
                            max_weight=MAX_WEIGHT, initial_capital=1e9):
    reb_set      = build_reb_set(returns, freq)
    actual_dates = returns.index
    assets       = all_assets
    n            = len(assets)
    asset_idx    = {a: i for i, a in enumerate(assets)}

    sig = signal_df.copy()
    sig.index = pd.to_datetime(sig.index)

    pv              = float(initial_capital)
    current_weights = np.zeros(n)
    records         = []
    weight_history  = []
    first_reb       = True

    for t, date in enumerate(actual_dates):
        if date in reb_set:
            hist = returns.iloc[max(0, t - lookback): t]

            if len(hist) < min_obs:
                new_weights = np.ones(n) / n
            else:
                mu_ann  = hist.mean().values  * 252
                cov_ann = hist.cov().values   * 252

                past    = sig[sig.index <= date]
                sig_row = past.iloc[-1] if len(past) > 0 else None

                allowed_mask = np.ones(n, dtype=bool)
                if sig_row is not None:
                    for eq in equity_assets:
                        if eq in asset_idx and eq in sig_row.index:
                            s = sig_row[eq]
                            if not pd.isna(s) and s > 0:
                                allowed_mask[asset_idx[eq]] = False

                new_weights = mean_variance_weights(
                    mu_ann, cov_ann, allowed_mask, max_weight=max_weight)

            if first_reb:
                pv       *= (1 - tc_rate)
                first_reb = False
            else:
                turnover = np.sum(np.abs(new_weights - current_weights)) / 2.0
                pv      *= (1 - turnover * tc_rate)

            current_weights = new_weights.copy()
            weight_history.append({'date': date,
                                   **dict(zip(assets, current_weights))})

        if not first_reb:
            dr  = np.dot(current_weights, returns.loc[date].values)
            pv *= (1 + dr)

        records.append({'date': date, 'portfolio_value': pv})

    pv_df      = pd.DataFrame(records).set_index('date')
    weights_df = pd.DataFrame(weight_history).set_index('date')
    print(f"  MV | freq={freq} tc={tc_rate*100:.1f}% "
          f"End: ${pv_df['portfolio_value'].iloc[-1]/1e9:.3f}B")
    return pv_df, weights_df

# ── 15. RUN ALL PORTFOLIOS ────────────────────────────────────────────────────
TC_LEVELS = {'0%': 0.00, '0.5%': 0.005, '1%': 0.01, '2%': 0.02}
FREQS     = ['daily', 'weekly', 'monthly', 'quarterly']
MV_FREQS  = ['monthly', 'quarterly']

# ── 15a. Buy-and-Hold ─────────────────────────────────────────────────────────
print("\n── Running Buy-and-Hold ──")
bh_result  = run_buy_and_hold(returns, ALL_ASSETS, initial_capital=1e9)
bh_results = {'BuyAndHold_EW': bh_result}

# ── 15b. Equal-Weight ─────────────────────────────────────────────────────────
print("\n── Running Equal-Weight Rebalanced ──")
ew_results = {}
for freq in FREQS:
    for tc_label, tc_rate in TC_LEVELS.items():
        label = f"EW_{freq}_TC{tc_label}"
        ew_results[label] = run_equal_weight(
            returns, ALL_ASSETS, tc_rate=tc_rate,
            rebal_freq=freq, initial_capital=1e9)
        print(f"  ✓ {label}")

# ── 15c. Original 3-Stage Signal ─────────────────────────────────────────────
print("\n── Running Original 3-Stage Signal ──")
signal_results       = {}
signal_distributions = {}

for thresh in THRESHOLDS:
    sig_df_t    = build_signal_df(composite_score, equity_assets,
                                  macro_df.index, threshold=thresh)
    n_risk_on   = (sig_df_t[equity_assets[0]] == +1).sum()
    n_risk_off  = (sig_df_t[equity_assets[0]] == -1).sum()
    total       = len(sig_df_t)
    signal_distributions[thresh] = {'RISK_ON': n_risk_on, 'RISK_OFF': n_risk_off}
    print(f"  Threshold={thresh} | RISK_ON: {n_risk_on} ({n_risk_on/total*100:.1f}%) "
          f"| RISK_OFF: {n_risk_off} ({n_risk_off/total*100:.1f}%)")

    for freq in FREQS:
        for tc_label, tc_rate in TC_LEVELS.items():
            label = f"Sig3S_t{thresh}_{freq}_TC{tc_label}"

            def wfn_3s(sig_row, _eq=equity_assets, _ne=non_equity_assets,
                       _bw=base_weight, _sh=EQUITY_SAFE_HAVEN_MAP):
                return build_weights_3stage(sig_row, _eq, _ne, _bw, None, _sh)

            signal_results[label] = run_signal_portfolio(
                returns, sig_df_t, equity_assets, non_equity_assets,
                ALL_ASSETS, weight_fn=wfn_3s,
                freq=freq, tc_rate=tc_rate, initial_capital=1e9)
            print(f"    ✓ {label}")

# ── 15d. Asymmetric Signal ────────────────────────────────────────────────────
print("\n── Running Asymmetric Signal ──")
asym_results = {}

for thresh in THRESHOLDS:
    sig_df_t = build_signal_df_asymmetric(composite_score, equity_assets,
                                          macro_df.index, threshold=thresh)
    for freq in ['monthly', 'quarterly']:
        for tc_label, tc_rate in TC_LEVELS.items():
            label = f"Asym_t{thresh}_{freq}_TC{tc_label}"

            def wfn_asym(sig_row, _eq=equity_assets, _ne=non_equity_assets,
                         _bw=base_weight, _sh=EQUITY_SAFE_HAVEN_MAP):
                return build_weights_3stage(sig_row, _eq, _ne, _bw, None, _sh)

            asym_results[label] = run_signal_portfolio(
                returns, sig_df_t, equity_assets, non_equity_assets,
                ALL_ASSETS, weight_fn=wfn_asym,
                freq=freq, tc_rate=tc_rate, initial_capital=1e9)
            print(f"    ✓ {label}")

# ── 15e. Continuous / Graduated Signal ───────────────────────────────────────
print("\n── Running Continuous Signal ──")
cont_results = {}

for thresh in THRESHOLDS:
    for max_red in MAX_REDUCES:
        sig_df_t = build_signal_df_continuous(composite_score, equity_assets,
                                              macro_df.index,
                                              threshold=thresh,
                                              max_reduce=max_red)
        for freq in ['monthly', 'quarterly']:
            for tc_label, tc_rate in TC_LEVELS.items():
                label = (f"Cont_t{thresh}_mr{int(max_red*100)}"
                         f"_{freq}_TC{tc_label}")

                def wfn_cont(sig_row, _eq=equity_assets, _ne=non_equity_assets,
                             _bw=base_weight, _sh=EQUITY_SAFE_HAVEN_MAP):
                    return build_weights_continuous(sig_row, _eq, _ne, _bw, _sh)

                cont_results[label] = run_signal_portfolio(
                    returns, sig_df_t, equity_assets, non_equity_assets,
                    ALL_ASSETS, weight_fn=wfn_cont,
                    freq=freq, tc_rate=tc_rate, initial_capital=1e9)
                print(f"    ✓ {label}")

# ── 15f. Smoothed Signal ─────────────────────────────────────────────────────
print("\n── Running Smoothed Signal ──")
smooth_results = {}

for thresh in THRESHOLDS:
    for sw in SMOOTH_WINDOWS:
        sig_df_t = build_signal_df_smoothed(composite_score, equity_assets,
                                            macro_df.index,
                                            threshold=thresh,
                                            smooth_window=sw)
        for freq in ['monthly', 'quarterly']:
            for tc_label, tc_rate in TC_LEVELS.items():
                label = f"Smooth_t{thresh}_sw{sw}_{freq}_TC{tc_label}"

                def wfn_smooth(sig_row, _eq=equity_assets, _ne=non_equity_assets,
                               _bw=base_weight, _sh=EQUITY_SAFE_HAVEN_MAP):
                    return build_weights_3stage(sig_row, _eq, _ne, _bw, None, _sh)

                smooth_results[label] = run_signal_portfolio(
                    returns, sig_df_t, equity_assets, non_equity_assets,
                    ALL_ASSETS, weight_fn=wfn_smooth,
                    freq=freq, tc_rate=tc_rate, initial_capital=1e9)
                print(f"    ✓ {label}")

# ── 15g. Persistent Signal ────────────────────────────────────────────────────
print("\n── Running Persistent Signal ──")
persist_results = {}

for thresh in THRESHOLDS:
    for min_days in MIN_DAYS_LIST:
        sig_df_t = build_signal_df_persistent(composite_score, equity_assets,
                                              macro_df.index,
                                              threshold=thresh,
                                              min_days_to_switch=min_days)
        for freq in ['monthly', 'quarterly']:
            for tc_label, tc_rate in TC_LEVELS.items():
                label = f"Persist_t{thresh}_md{min_days}_{freq}_TC{tc_label}"

                def wfn_persist(sig_row, _eq=equity_assets, _ne=non_equity_assets,
                                _bw=base_weight, _sh=EQUITY_SAFE_HAVEN_MAP):
                    return build_weights_3stage(sig_row, _eq, _ne, _bw, None, _sh)

                persist_results[label] = run_signal_portfolio(
                    returns, sig_df_t, equity_assets, non_equity_assets,
                    ALL_ASSETS, weight_fn=wfn_persist,
                    freq=freq, tc_rate=tc_rate, initial_capital=1e9)
                print(f"    ✓ {label}")

# ── 15h. Partial-Hedge Signal ─────────────────────────────────────────────────
print("\n── Running Partial-Hedge Signal ──")
partial_results = {}

for thresh in THRESHOLDS:
    sig_df_t = build_signal_df(composite_score, equity_assets,
                               macro_df.index, threshold=thresh)
    for pk in PARTIAL_KEEPS:
        for freq in ['monthly', 'quarterly']:
            for tc_label, tc_rate in TC_LEVELS.items():
                label = (f"Partial_t{thresh}_pk{int(pk*100)}"
                         f"_{freq}_TC{tc_label}")

                def wfn_partial(sig_row, _eq=equity_assets, _ne=non_equity_assets,
                                _bw=base_weight, _sh=EQUITY_SAFE_HAVEN_MAP,
                                _pk=pk):
                    return build_weights_partial(sig_row, _eq, _ne, _bw, _sh, _pk)

                partial_results[label] = run_signal_portfolio(
                    returns, sig_df_t, equity_assets, non_equity_assets,
                    ALL_ASSETS, weight_fn=wfn_partial,
                    freq=freq, tc_rate=tc_rate, initial_capital=1e9)
                print(f"    ✓ {label}")

# ── 15i. Mean-Variance Signal ─────────────────────────────────────────────────
print("\n── Running Mean-Variance Signal ──")
mv_results        = {}
mv_weight_history = {}

for thresh in THRESHOLDS:
    sig_df_t = build_signal_df(composite_score, equity_assets,
                               macro_df.index, threshold=thresh)
    for freq in MV_FREQS:
        for tc_label, tc_rate in TC_LEVELS.items():
            label = f"MV_t{thresh}_{freq}_TC{tc_label}"
            pv_df, w_df = run_mv_signal_portfolio(
                returns, sig_df_t, ALL_ASSETS,
                equity_assets=equity_assets,
                freq=freq, tc_rate=tc_rate,
                lookback=MV_LOOKBACK, min_obs=MV_MIN_OBS,
                max_weight=MAX_WEIGHT, initial_capital=1e9)
            mv_results[label]        = pv_df
            mv_weight_history[label] = w_df
            print(f"  ✓ {label}")

print("\n✓ All portfolios computed")

# ── 16. PERFORMANCE METRICS ───────────────────────────────────────────────────
def compute_metrics(pv_df, label=''):
    pv        = pv_df['portfolio_value']
    daily_ret = pv.pct_change().dropna()
    n_years   = len(daily_ret) / 252
    total_ret = (pv.iloc[-1] / pv.iloc[0]) - 1
    cagr      = (1 + total_ret) ** (1 / n_years) - 1 if n_years > 0 else np.nan
    vol       = daily_ret.std() * np.sqrt(252)
    sharpe    = cagr / vol if vol > 0 else np.nan
    roll_max  = pv.cummax()
    max_dd    = ((pv - roll_max) / roll_max).min()
    calmar    = cagr / abs(max_dd) if max_dd != 0 else np.nan
    var_95    = daily_ret.quantile(0.05)
    ann_ret   = cagr
    ann_vol   = vol

    return {
        'CAGR':         f"{cagr*100:.2f}%",
        'Ann_Ret':      ann_ret,
        'Ann_Vol':      ann_vol,
        'Volatility':   f"{vol*100:.2f}%",
        'Sharpe':       f"{sharpe:.2f}",
        'Max Drawdown': f"{max_dd*100:.2f}%",
        'Calmar':       f"{calmar:.2f}",
        'VaR 95%':      f"{var_95*100:.2f}%",
        'Total Return': f"{total_ret*100:.2f}%",
        'Final ($B)':   f"{pv.iloc[-1]/1e9:.3f}",
    }

bh_metrics      = {k: compute_metrics(v, k) for k, v in bh_results.items()}
ew_metrics      = {k: compute_metrics(v, k) for k, v in ew_results.items()}
sig_metrics     = {k: compute_metrics(v, k) for k, v in signal_results.items()}
asym_metrics    = {k: compute_metrics(v, k) for k, v in asym_results.items()}
cont_metrics    = {k: compute_metrics(v, k) for k, v in cont_results.items()}
smooth_metrics  = {k: compute_metrics(v, k) for k, v in smooth_results.items()}
persist_metrics = {k: compute_metrics(v, k) for k, v in persist_results.items()}
partial_metrics = {k: compute_metrics(v, k) for k, v in partial_results.items()}
mv_metrics      = {k: compute_metrics(v, k) for k, v in mv_results.items()}

def print_metrics_table(metrics_dict, title):
    display = {k: {mk: mv for mk, mv in m.items()
                   if mk not in ('Ann_Ret', 'Ann_Vol')}
               for k, m in metrics_dict.items()}
    print(f"\n{'='*90}")
    print(title)
    print("="*90)
    print(pd.DataFrame(display).T.to_string())

print_metrics_table(bh_metrics,      "BUY-AND-HOLD")
print_metrics_table(ew_metrics,      "EQUAL-WEIGHT REBALANCED")

for thresh in THRESHOLDS:
    sub = {k: v for k, v in sig_metrics.items()     if f"_t{thresh}_" in k}
    print_metrics_table(sub, f"3-STAGE SIGNAL — threshold={thresh}")

for thresh in THRESHOLDS:
    sub = {k: v for k, v in asym_metrics.items()    if f"_t{thresh}_" in k}
    print_metrics_table(sub, f"ASYMMETRIC SIGNAL — threshold={thresh}")

for thresh in THRESHOLDS:
    sub = {k: v for k, v in cont_metrics.items()    if f"_t{thresh}_" in k}
    print_metrics_table(sub, f"CONTINUOUS SIGNAL — threshold={thresh}")

for thresh in THRESHOLDS:
    sub = {k: v for k, v in smooth_metrics.items()  if f"_t{thresh}_" in k}
    print_metrics_table(sub, f"SMOOTHED SIGNAL — threshold={thresh}")

for thresh in THRESHOLDS:
    sub = {k: v for k, v in persist_metrics.items() if f"_t{thresh}_" in k}
    print_metrics_table(sub, f"PERSISTENT SIGNAL — threshold={thresh}")

for thresh in THRESHOLDS:
    sub = {k: v for k, v in partial_metrics.items() if f"_t{thresh}_" in k}
    print_metrics_table(sub, f"PARTIAL-HEDGE SIGNAL — threshold={thresh}")

for thresh in THRESHOLDS:
    sub = {k: v for k, v in mv_metrics.items()      if f"_t{thresh}_" in k}
    print_metrics_table(sub, f"MEAN-VARIANCE SIGNAL — threshold={thresh}")

# ── 17. PLOTS ─────────────────────────────────────────────────────────────────
bh_pv     = bh_results['BuyAndHold_EW']['portfolio_value']
ew_ref_pv = ew_results['EW_monthly_TC0.5%']['portfolio_value']

def parse_pct(s):
    return float(str(s).replace('%', '')) / 100

# ── Plot A: Best of each variant vs benchmarks (monthly, TC=0.5%) ─────────────
fig, axes = plt.subplots(2, 4, figsize=(28, 12), sharey=False)
fig.suptitle("All Signal Variants vs Benchmarks — Monthly Rebalancing, TC=0.5%",
             fontsize=14, fontweight='bold')

variant_info = [
    ("3-Stage",    signal_results,  THRESHOLDS,     "Sig3S",   "steelblue"),
    ("Asymmetric", asym_results,    THRESHOLDS,     "Asym",    "darkorange"),
    ("Continuous", cont_results,    THRESHOLDS,     "Cont",    "purple"),
    ("Smoothed",   smooth_results,  THRESHOLDS,     "Smooth",  "green"),
    ("Persistent", persist_results, THRESHOLDS,     "Persist", "crimson"),
    ("Partial 25%",partial_results, THRESHOLDS,     "Partial", "brown"),
    ("MV Signal",  mv_results,      THRESHOLDS,     "MV",      "black"),
]

for ax, (name, res_dict, threshs, prefix, col) in zip(axes.flat, variant_info):
    ax.plot(bh_pv.index,     bh_pv / 1e9,     color='forestgreen',
            lw=1.5, ls='-.', label='Buy & Hold')
    ax.plot(ew_ref_pv.index, ew_ref_pv / 1e9, color='grey',
            lw=1.2, ls='--', label='EW Monthly')

    for thresh in threshs:
        candidates = [k for k in res_dict
                      if f"_t{thresh}_" in k and 'monthly' in k and 'TC0.5%' in k]
        if not candidates:
            candidates = [k for k in res_dict
                          if f"_t{thresh}_" in k and 'TC0.5%' in k]
        if candidates:
            key = candidates[0]
            pv  = res_dict[key]['portfolio_value']
            ax.plot(pv.index, pv / 1e9, lw=1.6,
                    label=f"t={thresh}")

    ax.axhline(1.0, color='black', ls=':', lw=0.7)
    ax.set_title(name, fontsize=11, fontweight='bold')
    ax.set_ylabel("Value ($B)")
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

axes.flat[-1].set_visible(False)
plt.tight_layout()
plt.savefig("all_variants_monthly.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Plot B: Risk-Return Scatter — all strategies ──────────────────────────────
fig, ax = plt.subplots(figsize=(16, 10))

all_metric_groups = [
    (bh_metrics,      'BH',       'forestgreen', 'D'),
    (ew_metrics,      'EW',       'grey',        's'),
    (sig_metrics,     '3-Stage',  'steelblue',   'o'),
    (asym_metrics,    'Asym',     'darkorange',  'o'),
    (cont_metrics,    'Cont',     'purple',      'o'),
    (smooth_metrics,  'Smooth',   'green',       'o'),
    (persist_metrics, 'Persist',  'crimson',     'o'),
    (partial_metrics, 'Partial',  'brown',       'o'),
    (mv_metrics,      'MV',       'black',       '^'),
]

for metrics_dict, grp_label, col, marker in all_metric_groups:
    first = True
    for key, m in metrics_dict.items():
        vol = m['Ann_Vol'] * 100
        ret = m['Ann_Ret'] * 100
        ax.scatter(vol, ret, color=col, marker=marker, s=80,
                   label=grp_label if first else None,
                   edgecolors='black', lw=0.5, zorder=4)
        ax.annotate(key.split('_TC')[0], (vol, ret),
                    textcoords='offset points', xytext=(4, 3),
                    fontsize=6, color=col)
        first = False

ax.axhline(0, color='black', lw=0.6, ls='--')
ax.axvline(0, color='black', lw=0.6, ls='--')
ax.set_xlabel("Annualised Volatility (%)", fontsize=12)
ax.set_ylabel("CAGR (%)", fontsize=12)
ax.set_title("Risk-Return — All Strategies", fontsize=13, fontweight='bold')
ax.legend(fontsize=10, ncol=3)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("risk_return_all.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Plot C: Sharpe Bar Chart — top 20 strategies ──────────────────────────────
all_metrics_combined = {
    **bh_metrics, **ew_metrics, **sig_metrics,
    **asym_metrics, **cont_metrics, **smooth_metrics,
    **persist_metrics, **partial_metrics, **mv_metrics
}

sharpe_series = pd.Series({
    k: float(v['Sharpe']) for k, v in all_metrics_combined.items()
}).dropna().sort_values(ascending=False)

top20 = sharpe_series.head(20)

def label_color(lbl):
    if lbl.startswith('MV'):       return 'black'
    if lbl.startswith('Sig3S'):    return 'steelblue'
    if lbl.startswith('Asym'):     return 'darkorange'
    if lbl.startswith('Cont'):     return 'purple'
    if lbl.startswith('Smooth'):   return 'green'
    if lbl.startswith('Persist'):  return 'crimson'
    if lbl.startswith('Partial'):  return 'brown'
    if lbl.startswith('EW'):       return 'grey'
    return 'forestgreen'

colors = [label_color(k) for k in top20.index]

fig, ax = plt.subplots(figsize=(16, 7))
bars = ax.barh(range(len(top20)), top20.values, color=colors, edgecolor='black', lw=0.5)
ax.set_yticks(range(len(top20)))
ax.set_yticklabels([k.replace('_TC', ' TC=') for k in top20.index], fontsize=8)
ax.set_xlabel("Sharpe Ratio")
ax.set_title("Top 20 Strategies by Sharpe Ratio", fontsize=13, fontweight='bold')
ax.axvline(0, color='black', lw=0.8)
ax.grid(True, axis='x', alpha=0.3)

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='black',      label='MV Signal'),
    Patch(facecolor='steelblue',  label='3-Stage'),
    Patch(facecolor='darkorange', label='Asymmetric'),
    Patch(facecolor='purple',     label='Continuous'),
    Patch(facecolor='green',      label='Smoothed'),
    Patch(facecolor='crimson',    label='Persistent'),
    Patch(facecolor='brown',      label='Partial Hedge'),
    Patch(facecolor='grey',       label='Equal Weight'),
    Patch(facecolor='forestgreen',label='Buy & Hold'),
]
ax.legend(handles=legend_elements, fontsize=9, loc='lower right')
plt.tight_layout()
plt.savefig("sharpe_top20.png", dpi=150, bbox_inches='tight')
plt.show()

# ── 18. SAVE ALL ──────────────────────────────────────────────────────────────
print("\n── Saving all results ──")
all_portfolios = {
    **bh_results, **ew_results, **signal_results,
    **asym_results, **cont_results, **smooth_results,
    **persist_results, **partial_results, **mv_results
}

for label, df in all_portfolios.items():
    df.to_csv(f"{label}_portfolio.csv")

all_metrics_df = pd.DataFrame({
    k: {mk: mv for mk, mv in m.items() if mk not in ('Ann_Ret', 'Ann_Vol')}
    for k, m in all_metrics_combined.items()
}).T
all_metrics_df.to_csv("all_portfolio_metrics.csv")

composite_score.to_csv("composite_signal.csv")
for thresh in THRESHOLDS:
    sig_df_t = build_signal_df(composite_score, equity_assets,
                               macro_df.index, threshold=thresh)
    sig_df_t.to_csv(f"signal_df_threshold_{thresh}.csv")

for label, wdf in mv_weight_history.items():
    wdf.to_csv(f"{label}_weights.csv")

print("✓ All files saved.")
print(f"\n  Portfolio CSVs   : {len(all_portfolios)}")
print(f"  Weight CSVs      : {len(mv_weight_history)}")
print(f"  Metrics CSV      : all_portfolio_metrics.csv")
print(f"  Signal CSVs      : {len(THRESHOLDS)} threshold files")
print(f"  Plots saved      : 3 PNG files")


In [ ]:
# @title
# ============================================================
# SIGNAL PORTFOLIO — ENHANCED VERSION
# Signals generated from Modified Determinant Model
# + Buy-and-Hold Benchmark
# + Equal-Weight Benchmark
# + Three-Stage Signal Portfolio (original)
# + Enhanced Signal Variants (asymmetric, continuous, smoothed, persistent, partial)
# + Mean-Variance Signal Portfolio
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import requests
from io import StringIO
from scipy.optimize import minimize

# ── 1. LOAD ALL DATA FROM GITHUB ─────────────────────────────────────────────
GITHUB_BASE = "https://raw.githubusercontent.com/kboroz/MSFE_Capstone_Project/main/01_Data/Streamlined/"

INDEX_FILE_MAP = {
    'ASX50':       'streamlined_asx_50.csv',
    'EUROSTOXX50': 'streamlined_euro_stoxx_50.csv',
    'FTSE100':     'streamlined_ftse_100.csv',
    'Ibovespa':    'streamlined_ibovespa.csv',
    'JSE40':       'streamlined_jse_top_40.csv',
    'NIKKEI225':   'streamlined_nikkei_225.csv',
    'SP500':       'streamlined_s&p_500.csv',
}

MACRO_FILE = 'streamlined_macro.csv'

def load_csv(url):
    r = requests.get(url)
    r.raise_for_status()
    df = pd.read_csv(StringIO(r.text), index_col=0, parse_dates=True)
    df.columns = df.columns.str.replace('[^a-zA-Z0-9]', '_', regex=True)
    return df.sort_index()

macro_df = load_csv(GITHUB_BASE + MACRO_FILE)
macro_df = macro_df[~macro_df.index.duplicated(keep='first')].sort_index()
print(f"macro_df shape : {macro_df.shape}")
print(f"Columns        : {macro_df.columns.tolist()}")

index_data = {}
for name, fname in INDEX_FILE_MAP.items():
    try:
        df = load_csv(GITHUB_BASE + fname)
        index_data[name] = df
        print(f"✓ {name:15s} | shape={df.shape} | cols={df.columns.tolist()[:4]}")
    except Exception as e:
        print(f"✗ {name}: {e}")

# ── 2. EXTRACT PRICE SERIES ───────────────────────────────────────────────────
PRICE_COLS = ['Close', 'close', 'Price', 'price', 'Adj_Close', 'Last']

def get_price_series(df, name):
    col = next((c for c in PRICE_COLS if c in df.columns), df.columns[0])
    s = pd.to_numeric(df[col], errors='coerce').dropna()
    print(f"  {name}: using column '{col}'")
    return s

index_prices = {}
for name, df in index_data.items():
    index_prices[name] = get_price_series(df, name)

# ── 3. MODIFIED DETERMINANT SIGNAL MODEL ─────────────────────────────────────
DET_WINDOWS = [3, 7, 28]
WINDOW_WEIGHTS = {3: 0.2, 7: 0.3, 28: 0.5}

def compute_modified_determinant(price_series_dict, windows=DET_WINDOWS):
    prices = pd.DataFrame(price_series_dict).sort_index()
    rets   = prices.pct_change().dropna()
    det_results = {}

    for window in windows:
        det_series = []
        for i in range(window, len(rets)):
            window_rets = rets.iloc[i - window:i]
            valid_cols  = window_rets.columns[window_rets.std() > 1e-10]
            w           = window_rets[valid_cols]

            if len(valid_cols) < 2:
                det_series.append(np.nan)
                continue

            try:
                R   = w.corr().values
                I   = np.eye(len(R))
                M   = R - I
                det = np.linalg.det(M)
            except Exception:
                det = np.nan

            det_series.append(det)

        idx = rets.index[window:]
        det_results[window] = pd.Series(det_series, index=idx,
                                        name=f'det_{window}d')
        print(f"  Window {window:2d}d | computed {len(det_series)} observations")

    return det_results

print("\n── Computing Modified Determinant ──")
det_results = compute_modified_determinant(index_prices)

# ── 4. COMPOSITE SCORE ────────────────────────────────────────────────────────
def compute_composite_score(det_results, window_weights=WINDOW_WEIGHTS):
    all_series = []
    for window, series in det_results.items():
        mu    = series.rolling(252, min_periods=60).mean()
        sigma = series.rolling(252, min_periods=60).std()
        z     = (series - mu) / sigma.replace(0, np.nan)
        z.name = f'z_{window}d'
        all_series.append(z)

    z_df      = pd.concat(all_series, axis=1).dropna()
    composite = sum(
        z_df[f'z_{w}d'] * window_weights[w]
        for w in window_weights if f'z_{w}d' in z_df.columns
    )
    composite = composite / sum(window_weights.values())
    return composite, z_df

composite_score, z_df = compute_composite_score(det_results)

print(f"\nComposite score shape  : {composite_score.shape}")
print(f"Date range             : {composite_score.index[0].date()} → "
      f"{composite_score.index[-1].date()}")
print(f"Score range            : {composite_score.min():.3f} → "
      f"{composite_score.max():.3f}")

# ── 5. SIGNAL BUILDERS ───────────────────────────────────────────────────────
THRESHOLDS    = [0.3, 0.6, 0.9]
SMOOTH_WINDOWS = [3, 5, 10]
MIN_DAYS_LIST  = [5, 10, 20]
MAX_REDUCES    = [0.5, 0.7, 0.9]
PARTIAL_KEEPS  = [0.25, 0.50]

# ── 5a. Original symmetric binary ────────────────────────────────────────────
def classify_signal_symmetric(score, threshold):
    return +1 if abs(score) > threshold else -1

def build_signal_df(composite_score, equity_assets, macro_dates, threshold=0.5):
    binary    = composite_score.apply(lambda x: classify_signal_symmetric(x, threshold))
    signal_df = pd.DataFrame(index=macro_dates)
    for asset in equity_assets:
        signal_df[asset] = binary.reindex(macro_dates).ffill().bfill()
    return signal_df

# ── 5b. Asymmetric (upper-tail only) ─────────────────────────────────────────
def build_signal_df_asymmetric(composite_score, equity_assets, macro_dates,
                               threshold=1.0):
    """Block equities only when correlation SPIKES (high positive z-score)."""
    binary    = composite_score.apply(lambda x: +1 if x > threshold else -1)
    signal_df = pd.DataFrame(index=macro_dates)
    for asset in equity_assets:
        signal_df[asset] = binary.reindex(macro_dates).ffill().bfill()
    return signal_df

# ── 5c. Continuous / graduated scalar ────────────────────────────────────────
def build_signal_df_continuous(composite_score, equity_assets, macro_dates,
                               threshold=1.0, max_reduce=0.8):
    """
    Returns a scalar in [1-max_reduce, 1.0] per asset per day.
    Used by build_weights_continuous() instead of the binary flag.
    """
    def score_to_scalar(s):
        if s <= 0:
            return 1.0
        elif s >= threshold:
            return 1.0 - max_reduce
        else:
            return 1.0 - max_reduce * (s / threshold)

    scalars   = composite_score.apply(score_to_scalar)
    signal_df = pd.DataFrame(index=macro_dates)
    for asset in equity_assets:
        signal_df[asset] = scalars.reindex(macro_dates).ffill().bfill()
    return signal_df

# ── 5d. Smoothed binary ───────────────────────────────────────────────────────
def build_signal_df_smoothed(composite_score, equity_assets, macro_dates,
                             threshold=1.0, smooth_window=5):
    """Smooth z-score before thresholding to reduce whipsaw."""
    smoothed  = composite_score.rolling(smooth_window, min_periods=1).mean()
    binary    = smoothed.apply(lambda x: classify_signal_symmetric(x, threshold))
    signal_df = pd.DataFrame(index=macro_dates)
    for asset in equity_assets:
        signal_df[asset] = binary.reindex(macro_dates).ffill().bfill()
    return signal_df

# ── 5e. Persistent binary ────────────────────────────────────────────────────
def build_signal_df_persistent(composite_score, equity_assets, macro_dates,
                               threshold=1.0, min_days_to_switch=10):
    """Only switch regime after signal holds for min_days_to_switch days."""
    raw       = composite_score.apply(lambda x: +1 if x > threshold else -1)
    confirmed = raw.copy()
    current   = int(raw.iloc[0])
    count     = 0

    for i in range(1, len(raw)):
        if int(raw.iloc[i]) != current:
            count += 1
            if count >= min_days_to_switch:
                current = int(raw.iloc[i])
                count   = 0
        else:
            count = 0
        confirmed.iloc[i] = current

    signal_df = pd.DataFrame(index=macro_dates)
    for asset in equity_assets:
        signal_df[asset] = confirmed.reindex(macro_dates).ffill().bfill()
    return signal_df

# ── 6. ASSET UNIVERSE ─────────────────────────────────────────────────────────
EQUITY_ASSETS = list(index_prices.keys())
SAFE_BONDS    = ['US_10Y', 'UK_10Y', 'AU_10Y', 'ZA_10Y']
FX            = ['AUDUSD', 'BRLUSD', 'EURUSD', 'GBPUSD', 'JPYUSD', 'ZARUSD']
COMMODITIES   = ['Oil', 'Gold', 'Bitcoin']
EXCLUDE_ASSETS = ['Bitcoin']

ALL_ASSETS = EQUITY_ASSETS + FX + COMMODITIES + SAFE_BONDS
ALL_ASSETS = [a for a in ALL_ASSETS
              if a in macro_df.columns and a not in EXCLUDE_ASSETS]

equity_assets     = [a for a in EQUITY_ASSETS     if a in ALL_ASSETS]
non_equity_assets = [a for a in ALL_ASSETS         if a not in equity_assets]

N           = len(ALL_ASSETS)
base_weight = 1.0 / N

print(f"\nTotal assets              : {N}")
print(f"Equity (signal-driven)    : {equity_assets}")
print(f"Non-equity (always long)  : {non_equity_assets}")
print(f"base_weight = 1/{N} = {base_weight:.4f}")

# ── 7. COUNTRY SAFE-HAVEN MAP ─────────────────────────────────────────────────
EQUITY_SAFE_HAVEN_MAP = {
    'ASX50':       ['AU_10Y', 'AUDUSD'],
    'EUROSTOXX50': ['EURUSD'],
    'FTSE100':     ['UK_10Y', 'GBPUSD'],
    'Ibovespa':    ['BRLUSD'],
    'JSE40':       ['ZA_10Y', 'ZARUSD'],
    'NIKKEI225':   ['JPYUSD'],
    'SP500':       ['US_10Y'],
}

UNIVERSE_SET = set(ALL_ASSETS)
for eq, havens in EQUITY_SAFE_HAVEN_MAP.items():
    EQUITY_SAFE_HAVEN_MAP[eq] = [h for h in havens if h in UNIVERSE_SET]

print("\nSafe-haven routing (filtered to universe):")
for eq, havens in EQUITY_SAFE_HAVEN_MAP.items():
    print(f"  {eq:15s} → {havens}")

# ── 8. BUILD RETURNS ──────────────────────────────────────────────────────────
returns_raw = macro_df[ALL_ASSETS].pct_change(fill_method=None)

for col in SAFE_BONDS:
    if col in returns_raw.columns:
        returns_raw[col] = -returns_raw[col]

if 'Oil' in macro_df.columns:
    bad_oil = macro_df['Oil'][macro_df['Oil'] <= 0].index
    for d in bad_oil:
        loc = returns_raw.index.get_loc(d)
        returns_raw.iloc[max(0, loc - 1): loc + 2,
                         returns_raw.columns.get_loc('Oil')] = 0.0

returns = returns_raw.clip(lower=-0.50, upper=1.00).fillna(0).iloc[1:]

assert not returns.isnull().any().any(), "NaNs in returns"
assert not np.isinf(returns.values).any(),  "Inf in returns"
print(f"\nReturns shape : {returns.shape}")
print("✓ Returns clean")

# ── 9. HELPER: REBALANCING DATE SET ──────────────────────────────────────────
def build_reb_set(returns, rebal_freq):
    actual_dates = returns.index
    if rebal_freq == 'daily':
        return set(actual_dates)
    elif rebal_freq == 'weekly':
        raw = returns.resample('W-MON').first().index
    elif rebal_freq == 'monthly':
        raw = returns.resample('MS').first().index
    elif rebal_freq == 'quarterly':
        raw = returns.resample('QS').first().index
    else:
        raise ValueError(f"Unknown rebal_freq: {rebal_freq}")

    reb_set = set()
    for d in raw:
        future = actual_dates[actual_dates >= d]
        if len(future) > 0:
            reb_set.add(future[0])
    return reb_set

# ── 10. BUY-AND-HOLD BENCHMARK ───────────────────────────────────────────────
def run_buy_and_hold(returns, all_assets, initial_capital=1e9):
    N           = len(all_assets)
    base_weight = 1.0 / N
    pv          = float(initial_capital)
    holdings    = {a: pv * base_weight for a in all_assets}
    records     = []
    first       = True

    for date in returns.index:
        if first:
            first = False
            records.append({'date': date, 'portfolio_value': pv})
            continue
        new_holdings = {a: holdings[a] * (1 + returns.loc[date, a])
                        for a in all_assets}
        holdings = new_holdings
        pv       = sum(holdings.values())
        records.append({'date': date, 'portfolio_value': pv})

    df = pd.DataFrame(records).set_index('date')
    print(f"  Buy-and-Hold | Start: ${initial_capital/1e9:.2f}B → "
          f"End: ${df['portfolio_value'].iloc[-1]/1e9:.3f}B")
    return df

# ── 11. WEIGHT BUILDERS ───────────────────────────────────────────────────────

# ── 11a. Original binary 3-stage weights ─────────────────────────────────────
def build_weights_3stage(sig_row, equity_assets, non_equity_assets,
                         base_weight, date, safe_haven_map):
    raw            = {a: base_weight for a in non_equity_assets}
    safe_haven_extra = {}

    for a in equity_assets:
        if sig_row is not None and a in sig_row.index and not pd.isna(sig_row[a]):
            sig = sig_row[a]
        else:
            sig = -1

        if sig > 0:
            raw[a] = -base_weight
            havens = [h for h in safe_haven_map.get(a, []) if h in raw]
            if havens:
                per_haven = base_weight / len(havens)
                for h in havens:
                    safe_haven_extra[h] = safe_haven_extra.get(h, 0) + per_haven
            else:
                per_ne = base_weight / len(non_equity_assets)
                for ne in non_equity_assets:
                    safe_haven_extra[ne] = safe_haven_extra.get(ne, 0) + per_ne
        else:
            raw[a] = base_weight

    for asset, extra in safe_haven_extra.items():
        raw[asset] = raw.get(asset, 0) + extra

    total = sum(raw.values())
    if total <= 0:
        for a in raw:
            raw[a] = 1.0 / len(raw)
    else:
        for a in raw:
            raw[a] /= total
    return raw

# ── 11b. Continuous scalar weights ───────────────────────────────────────────
def build_weights_continuous(sig_row, equity_assets, non_equity_assets,
                             base_weight, safe_haven_map):
    """
    sig_row contains a scalar in [min_weight, 1.0] per equity asset.
    Reduced equity weight is redistributed to safe havens.
    """
    raw              = {a: base_weight for a in non_equity_assets}
    safe_haven_extra = {}

    for a in equity_assets:
        scalar = float(sig_row[a]) if (sig_row is not None and
                                       a in sig_row.index and
                                       not pd.isna(sig_row[a])) else 1.0
        raw[a]   = base_weight * scalar
        released = base_weight * (1.0 - scalar)

        if released > 0:
            havens = [h for h in safe_haven_map.get(a, []) if h in raw]
            if havens:
                per_haven = released / len(havens)
                for h in havens:
                    safe_haven_extra[h] = safe_haven_extra.get(h, 0) + per_haven
            else:
                per_ne = released / len(non_equity_assets)
                for ne in non_equity_assets:
                    safe_haven_extra[ne] = safe_haven_extra.get(ne, 0) + per_ne

    for asset, extra in safe_haven_extra.items():
        raw[asset] = raw.get(asset, 0) + extra

    total = sum(raw.values())
    if total <= 0:
        for a in raw:
            raw[a] = 1.0 / len(raw)
    else:
        for a in raw:
            raw[a] /= total
    return raw

# ── 11c. Partial-hedge weights ────────────────────────────────────────────────
def build_weights_partial(sig_row, equity_assets, non_equity_assets,
                          base_weight, safe_haven_map, partial_keep=0.25):
    """
    On RISK_ON signal: keep partial_keep fraction of base_weight in equity,
    redirect the rest to safe havens.
    """
    raw              = {a: base_weight for a in non_equity_assets}
    safe_haven_extra = {}

    for a in equity_assets:
        if sig_row is not None and a in sig_row.index and not pd.isna(sig_row[a]):
            sig = sig_row[a]
        else:
            sig = -1

        if sig > 0:
            raw[a]   = base_weight * partial_keep
            released = base_weight * (1.0 - partial_keep)
            havens   = [h for h in safe_haven_map.get(a, []) if h in raw]
            if havens:
                per_haven = released / len(havens)
                for h in havens:
                    safe_haven_extra[h] = safe_haven_extra.get(h, 0) + per_haven
            else:
                per_ne = released / len(non_equity_assets)
                for ne in non_equity_assets:
                    safe_haven_extra[ne] = safe_haven_extra.get(ne, 0) + per_ne
        else:
            raw[a] = base_weight

    for asset, extra in safe_haven_extra.items():
        raw[asset] = raw.get(asset, 0) + extra

    total = sum(raw.values())
    if total <= 0:
        for a in raw:
            raw[a] = 1.0 / len(raw)
    else:
        for a in raw:
            raw[a] /= total
    return raw

# ── 12. EQUAL-WEIGHT REBALANCED BENCHMARK ────────────────────────────────────
def run_equal_weight(returns, all_assets, tc_rate=0.005,
                     rebal_freq='monthly', initial_capital=1e9):
    N           = len(all_assets)
    base_weight = 1.0 / N
    ew          = {a: base_weight for a in all_assets}
    reb_set     = build_reb_set(returns, rebal_freq)
    pv          = float(initial_capital)
    current_weights = None
    records     = []

    for date in returns.index:
        if date in reb_set:
            if current_weights is not None:
                turnover = sum(abs(ew.get(a, 0) - current_weights.get(a, 0))
                               for a in set(ew) | set(current_weights)) / 2.0
                pv *= (1 - turnover * tc_rate)
            else:
                pv *= (1 - tc_rate)
            current_weights = ew.copy()

        if current_weights is not None:
            dr  = sum(current_weights.get(a, 0) * returns.loc[date, a]
                      for a in all_assets)
            pv *= (1 + dr)

        records.append({'date': date, 'portfolio_value': pv})

    return pd.DataFrame(records).set_index('date')

# ── 13. GENERIC SIGNAL PORTFOLIO RUNNER ──────────────────────────────────────
def run_signal_portfolio(returns, signal_df, equity_assets,
                         non_equity_assets, all_assets,
                         weight_fn,          # callable: (sig_row, ...) → dict
                         freq='monthly', tc_rate=0.005,
                         initial_capital=1e9):
    """
    Generic runner.  weight_fn must accept (sig_row, equity_assets,
    non_equity_assets, base_weight, <any extra kwargs>) and return
    a dict {asset: weight}.
    """
    N           = len(all_assets)
    base_weight = 1.0 / N
    reb_set     = build_reb_set(returns, freq)

    sig = signal_df.copy()
    sig.index = pd.to_datetime(sig.index)

    pv              = float(initial_capital)
    current_weights = None
    records         = []

    for date in returns.index:
        if date in reb_set:
            past    = sig[sig.index <= date]
            sig_row = past.iloc[-1] if len(past) > 0 else None
            raw     = weight_fn(sig_row)

            if current_weights is not None:
                all_keys = set(raw) | set(current_weights)
                turnover = sum(abs(raw.get(a, 0) - current_weights.get(a, 0))
                               for a in all_keys) / 2.0
                pv *= (1 - turnover * tc_rate)
            else:
                pv *= (1 - tc_rate)

            current_weights = raw.copy()

        if current_weights is not None:
            dr  = sum(current_weights.get(a, 0) * returns.loc[date, a]
                      for a in all_assets)
            pv *= (1 + dr)

        records.append({'date': date, 'portfolio_value': pv})

    return pd.DataFrame(records).set_index('date')

# ── 14. MEAN-VARIANCE SIGNAL PORTFOLIO ───────────────────────────────────────
MV_LOOKBACK = 252
MV_MIN_OBS  = 60
MAX_WEIGHT  = 0.30
RISK_FREE   = 0.0

def mean_variance_weights(mu_vec, cov_mat, allowed_mask,
                          max_weight=MAX_WEIGHT, risk_free=RISK_FREE):
    n_total   = len(mu_vec)
    allowed   = np.where(allowed_mask)[0]
    n_allowed = len(allowed)

    fallback = np.zeros(n_total)
    if n_allowed == 0:
        fallback[:] = 1.0 / n_total
        return fallback
    fallback[allowed] = 1.0 / n_allowed

    mu_sub  = mu_vec[allowed]
    cov_sub = cov_mat[np.ix_(allowed, allowed)] + np.eye(n_allowed) * 1e-8

    def neg_sharpe(w):
        port_ret = np.dot(w, mu_sub)
        port_var = np.dot(w, cov_sub @ w)
        port_std = np.sqrt(max(port_var, 1e-12))
        return -(port_ret - risk_free) / port_std

    constraints = [{'type': 'eq', 'fun': lambda w: np.sum(w) - 1.0}]
    bounds      = [(0.0, max_weight)] * n_allowed
    w0          = np.ones(n_allowed) / n_allowed

    try:
        res = minimize(neg_sharpe, w0, method='SLSQP',
                       bounds=bounds, constraints=constraints,
                       options={'ftol': 1e-9, 'maxiter': 500})
        w_opt = res.x if (res.success and not np.any(np.isnan(res.x))) else w0
    except Exception:
        w_opt = w0

    w_full         = np.zeros(n_total)
    w_full[allowed] = w_opt
    return w_full

def run_mv_signal_portfolio(returns, signal_df, all_assets,
                            equity_assets,
                            freq='monthly', tc_rate=0.005,
                            lookback=MV_LOOKBACK, min_obs=MV_MIN_OBS,
                            max_weight=MAX_WEIGHT, initial_capital=1e9):
    reb_set      = build_reb_set(returns, freq)
    actual_dates = returns.index
    assets       = all_assets
    n            = len(assets)
    asset_idx    = {a: i for i, a in enumerate(assets)}

    sig = signal_df.copy()
    sig.index = pd.to_datetime(sig.index)

    pv              = float(initial_capital)
    current_weights = np.zeros(n)
    records         = []
    weight_history  = []
    first_reb       = True

    for t, date in enumerate(actual_dates):
        if date in reb_set:
            hist = returns.iloc[max(0, t - lookback): t]

            if len(hist) < min_obs:
                new_weights = np.ones(n) / n
            else:
                mu_ann  = hist.mean().values  * 252
                cov_ann = hist.cov().values   * 252

                past    = sig[sig.index <= date]
                sig_row = past.iloc[-1] if len(past) > 0 else None

                allowed_mask = np.ones(n, dtype=bool)
                if sig_row is not None:
                    for eq in equity_assets:
                        if eq in asset_idx and eq in sig_row.index:
                            s = sig_row[eq]
                            if not pd.isna(s) and s > 0:
                                allowed_mask[asset_idx[eq]] = False

                new_weights = mean_variance_weights(
                    mu_ann, cov_ann, allowed_mask, max_weight=max_weight)

            if first_reb:
                pv       *= (1 - tc_rate)
                first_reb = False
            else:
                turnover = np.sum(np.abs(new_weights - current_weights)) / 2.0
                pv      *= (1 - turnover * tc_rate)

            current_weights = new_weights.copy()
            weight_history.append({'date': date,
                                   **dict(zip(assets, current_weights))})

        if not first_reb:
            dr  = np.dot(current_weights, returns.loc[date].values)
            pv *= (1 + dr)

        records.append({'date': date, 'portfolio_value': pv})

    pv_df      = pd.DataFrame(records).set_index('date')
    weights_df = pd.DataFrame(weight_history).set_index('date')
    print(f"  MV | freq={freq} tc={tc_rate*100:.1f}% "
          f"End: ${pv_df['portfolio_value'].iloc[-1]/1e9:.3f}B")
    return pv_df, weights_df

# ── 15. RUN ALL PORTFOLIOS ────────────────────────────────────────────────────
TC_LEVELS = {'0%': 0.00, '0.5%': 0.005, '1%': 0.01, '2%': 0.02}
FREQS     = ['daily', 'weekly', 'monthly', 'quarterly']
MV_FREQS  = ['monthly', 'quarterly']

# ── 15a. Buy-and-Hold ─────────────────────────────────────────────────────────
print("\n── Running Buy-and-Hold ──")
bh_result  = run_buy_and_hold(returns, ALL_ASSETS, initial_capital=1e9)
bh_results = {'BuyAndHold_EW': bh_result}

# ── 15b. Equal-Weight ─────────────────────────────────────────────────────────
print("\n── Running Equal-Weight Rebalanced ──")
ew_results = {}
for freq in FREQS:
    for tc_label, tc_rate in TC_LEVELS.items():
        label = f"EW_{freq}_TC{tc_label}"
        ew_results[label] = run_equal_weight(
            returns, ALL_ASSETS, tc_rate=tc_rate,
            rebal_freq=freq, initial_capital=1e9)
        print(f"  ✓ {label}")

# ── 15c. Original 3-Stage Signal ─────────────────────────────────────────────
print("\n── Running Original 3-Stage Signal ──")
signal_results       = {}
signal_distributions = {}

for thresh in THRESHOLDS:
    sig_df_t    = build_signal_df(composite_score, equity_assets,
                                  macro_df.index, threshold=thresh)
    n_risk_on   = (sig_df_t[equity_assets[0]] == +1).sum()
    n_risk_off  = (sig_df_t[equity_assets[0]] == -1).sum()
    total       = len(sig_df_t)
    signal_distributions[thresh] = {'RISK_ON': n_risk_on, 'RISK_OFF': n_risk_off}
    print(f"  Threshold={thresh} | RISK_ON: {n_risk_on} ({n_risk_on/total*100:.1f}%) "
          f"| RISK_OFF: {n_risk_off} ({n_risk_off/total*100:.1f}%)")

    for freq in FREQS:
        for tc_label, tc_rate in TC_LEVELS.items():
            label = f"Sig3S_t{thresh}_{freq}_TC{tc_label}"

            def wfn_3s(sig_row, _eq=equity_assets, _ne=non_equity_assets,
                       _bw=base_weight, _sh=EQUITY_SAFE_HAVEN_MAP):
                return build_weights_3stage(sig_row, _eq, _ne, _bw, None, _sh)

            signal_results[label] = run_signal_portfolio(
                returns, sig_df_t, equity_assets, non_equity_assets,
                ALL_ASSETS, weight_fn=wfn_3s,
                freq=freq, tc_rate=tc_rate, initial_capital=1e9)
            print(f"    ✓ {label}")

# ── 15d. Asymmetric Signal ────────────────────────────────────────────────────
print("\n── Running Asymmetric Signal ──")
asym_results = {}

for thresh in THRESHOLDS:
    sig_df_t = build_signal_df_asymmetric(composite_score, equity_assets,
                                          macro_df.index, threshold=thresh)
    for freq in ['monthly', 'quarterly']:
        for tc_label, tc_rate in TC_LEVELS.items():
            label = f"Asym_t{thresh}_{freq}_TC{tc_label}"

            def wfn_asym(sig_row, _eq=equity_assets, _ne=non_equity_assets,
                         _bw=base_weight, _sh=EQUITY_SAFE_HAVEN_MAP):
                return build_weights_3stage(sig_row, _eq, _ne, _bw, None, _sh)

            asym_results[label] = run_signal_portfolio(
                returns, sig_df_t, equity_assets, non_equity_assets,
                ALL_ASSETS, weight_fn=wfn_asym,
                freq=freq, tc_rate=tc_rate, initial_capital=1e9)
            print(f"    ✓ {label}")

# ── 15e. Continuous / Graduated Signal ───────────────────────────────────────
print("\n── Running Continuous Signal ──")
cont_results = {}

for thresh in THRESHOLDS:
    for max_red in MAX_REDUCES:
        sig_df_t = build_signal_df_continuous(composite_score, equity_assets,
                                              macro_df.index,
                                              threshold=thresh,
                                              max_reduce=max_red)
        for freq in ['monthly', 'quarterly']:
            for tc_label, tc_rate in TC_LEVELS.items():
                label = (f"Cont_t{thresh}_mr{int(max_red*100)}"
                         f"_{freq}_TC{tc_label}")

                def wfn_cont(sig_row, _eq=equity_assets, _ne=non_equity_assets,
                             _bw=base_weight, _sh=EQUITY_SAFE_HAVEN_MAP):
                    return build_weights_continuous(sig_row, _eq, _ne, _bw, _sh)

                cont_results[label] = run_signal_portfolio(
                    returns, sig_df_t, equity_assets, non_equity_assets,
                    ALL_ASSETS, weight_fn=wfn_cont,
                    freq=freq, tc_rate=tc_rate, initial_capital=1e9)
                print(f"    ✓ {label}")

# ── 15f. Smoothed Signal ─────────────────────────────────────────────────────
print("\n── Running Smoothed Signal ──")
smooth_results = {}

for thresh in THRESHOLDS:
    for sw in SMOOTH_WINDOWS:
        sig_df_t = build_signal_df_smoothed(composite_score, equity_assets,
                                            macro_df.index,
                                            threshold=thresh,
                                            smooth_window=sw)
        for freq in ['monthly', 'quarterly']:
            for tc_label, tc_rate in TC_LEVELS.items():
                label = f"Smooth_t{thresh}_sw{sw}_{freq}_TC{tc_label}"

                def wfn_smooth(sig_row, _eq=equity_assets, _ne=non_equity_assets,
                               _bw=base_weight, _sh=EQUITY_SAFE_HAVEN_MAP):
                    return build_weights_3stage(sig_row, _eq, _ne, _bw, None, _sh)

                smooth_results[label] = run_signal_portfolio(
                    returns, sig_df_t, equity_assets, non_equity_assets,
                    ALL_ASSETS, weight_fn=wfn_smooth,
                    freq=freq, tc_rate=tc_rate, initial_capital=1e9)
                print(f"    ✓ {label}")

# ── 15g. Persistent Signal ────────────────────────────────────────────────────
print("\n── Running Persistent Signal ──")
persist_results = {}

for thresh in THRESHOLDS:
    for min_days in MIN_DAYS_LIST:
        sig_df_t = build_signal_df_persistent(composite_score, equity_assets,
                                              macro_df.index,
                                              threshold=thresh,
                                              min_days_to_switch=min_days)
        for freq in ['monthly', 'quarterly']:
            for tc_label, tc_rate in TC_LEVELS.items():
                label = f"Persist_t{thresh}_md{min_days}_{freq}_TC{tc_label}"

                def wfn_persist(sig_row, _eq=equity_assets, _ne=non_equity_assets,
                                _bw=base_weight, _sh=EQUITY_SAFE_HAVEN_MAP):
                    return build_weights_3stage(sig_row, _eq, _ne, _bw, None, _sh)

                persist_results[label] = run_signal_portfolio(
                    returns, sig_df_t, equity_assets, non_equity_assets,
                    ALL_ASSETS, weight_fn=wfn_persist,
                    freq=freq, tc_rate=tc_rate, initial_capital=1e9)
                print(f"    ✓ {label}")

# ── 15h. Partial-Hedge Signal ─────────────────────────────────────────────────
print("\n── Running Partial-Hedge Signal ──")
partial_results = {}

for thresh in THRESHOLDS:
    sig_df_t = build_signal_df(composite_score, equity_assets,
                               macro_df.index, threshold=thresh)
    for pk in PARTIAL_KEEPS:
        for freq in ['monthly', 'quarterly']:
            for tc_label, tc_rate in TC_LEVELS.items():
                label = (f"Partial_t{thresh}_pk{int(pk*100)}"
                         f"_{freq}_TC{tc_label}")

                def wfn_partial(sig_row, _eq=equity_assets, _ne=non_equity_assets,
                                _bw=base_weight, _sh=EQUITY_SAFE_HAVEN_MAP,
                                _pk=pk):
                    return build_weights_partial(sig_row, _eq, _ne, _bw, _sh, _pk)

                partial_results[label] = run_signal_portfolio(
                    returns, sig_df_t, equity_assets, non_equity_assets,
                    ALL_ASSETS, weight_fn=wfn_partial,
                    freq=freq, tc_rate=tc_rate, initial_capital=1e9)
                print(f"    ✓ {label}")

# ── 15i. Mean-Variance Signal ─────────────────────────────────────────────────
print("\n── Running Mean-Variance Signal ──")
mv_results        = {}
mv_weight_history = {}

for thresh in THRESHOLDS:
    sig_df_t = build_signal_df(composite_score, equity_assets,
                               macro_df.index, threshold=thresh)
    for freq in MV_FREQS:
        for tc_label, tc_rate in TC_LEVELS.items():
            label = f"MV_t{thresh}_{freq}_TC{tc_label}"
            pv_df, w_df = run_mv_signal_portfolio(
                returns, sig_df_t, ALL_ASSETS,
                equity_assets=equity_assets,
                freq=freq, tc_rate=tc_rate,
                lookback=MV_LOOKBACK, min_obs=MV_MIN_OBS,
                max_weight=MAX_WEIGHT, initial_capital=1e9)
            mv_results[label]        = pv_df
            mv_weight_history[label] = w_df
            print(f"  ✓ {label}")

print("\n✓ All portfolios computed")

# ── 16. PERFORMANCE METRICS ───────────────────────────────────────────────────
def compute_metrics(pv_df, label=''):
    pv        = pv_df['portfolio_value']
    daily_ret = pv.pct_change().dropna()
    n_years   = len(daily_ret) / 252
    total_ret = (pv.iloc[-1] / pv.iloc[0]) - 1
    cagr      = (1 + total_ret) ** (1 / n_years) - 1 if n_years > 0 else np.nan
    vol       = daily_ret.std() * np.sqrt(252)
    sharpe    = cagr / vol if vol > 0 else np.nan
    roll_max  = pv.cummax()
    max_dd    = ((pv - roll_max) / roll_max).min()
    calmar    = cagr / abs(max_dd) if max_dd != 0 else np.nan
    var_95    = daily_ret.quantile(0.05)
    ann_ret   = cagr
    ann_vol   = vol

    return {
        'CAGR':         f"{cagr*100:.2f}%",
        'Ann_Ret':      ann_ret,
        'Ann_Vol':      ann_vol,
        'Volatility':   f"{vol*100:.2f}%",
        'Sharpe':       f"{sharpe:.2f}",
        'Max Drawdown': f"{max_dd*100:.2f}%",
        'Calmar':       f"{calmar:.2f}",
        'VaR 95%':      f"{var_95*100:.2f}%",
        'Total Return': f"{total_ret*100:.2f}%",
        'Final ($B)':   f"{pv.iloc[-1]/1e9:.3f}",
    }

bh_metrics      = {k: compute_metrics(v, k) for k, v in bh_results.items()}
ew_metrics      = {k: compute_metrics(v, k) for k, v in ew_results.items()}
sig_metrics     = {k: compute_metrics(v, k) for k, v in signal_results.items()}
asym_metrics    = {k: compute_metrics(v, k) for k, v in asym_results.items()}
cont_metrics    = {k: compute_metrics(v, k) for k, v in cont_results.items()}
smooth_metrics  = {k: compute_metrics(v, k) for k, v in smooth_results.items()}
persist_metrics = {k: compute_metrics(v, k) for k, v in persist_results.items()}
partial_metrics = {k: compute_metrics(v, k) for k, v in partial_results.items()}
mv_metrics      = {k: compute_metrics(v, k) for k, v in mv_results.items()}

def print_metrics_table(metrics_dict, title):
    display = {k: {mk: mv for mk, mv in m.items()
                   if mk not in ('Ann_Ret', 'Ann_Vol')}
               for k, m in metrics_dict.items()}
    print(f"\n{'='*90}")
    print(title)
    print("="*90)
    print(pd.DataFrame(display).T.to_string())

print_metrics_table(bh_metrics,      "BUY-AND-HOLD")
print_metrics_table(ew_metrics,      "EQUAL-WEIGHT REBALANCED")

for thresh in THRESHOLDS:
    sub = {k: v for k, v in sig_metrics.items()     if f"_t{thresh}_" in k}
    print_metrics_table(sub, f"3-STAGE SIGNAL — threshold={thresh}")

for thresh in THRESHOLDS:
    sub = {k: v for k, v in asym_metrics.items()    if f"_t{thresh}_" in k}
    print_metrics_table(sub, f"ASYMMETRIC SIGNAL — threshold={thresh}")

for thresh in THRESHOLDS:
    sub = {k: v for k, v in cont_metrics.items()    if f"_t{thresh}_" in k}
    print_metrics_table(sub, f"CONTINUOUS SIGNAL — threshold={thresh}")

for thresh in THRESHOLDS:
    sub = {k: v for k, v in smooth_metrics.items()  if f"_t{thresh}_" in k}
    print_metrics_table(sub, f"SMOOTHED SIGNAL — threshold={thresh}")

for thresh in THRESHOLDS:
    sub = {k: v for k, v in persist_metrics.items() if f"_t{thresh}_" in k}
    print_metrics_table(sub, f"PERSISTENT SIGNAL — threshold={thresh}")

for thresh in THRESHOLDS:
    sub = {k: v for k, v in partial_metrics.items() if f"_t{thresh}_" in k}
    print_metrics_table(sub, f"PARTIAL-HEDGE SIGNAL — threshold={thresh}")

for thresh in THRESHOLDS:
    sub = {k: v for k, v in mv_metrics.items()      if f"_t{thresh}_" in k}
    print_metrics_table(sub, f"MEAN-VARIANCE SIGNAL — threshold={thresh}")

# ── 17. PLOTS ─────────────────────────────────────────────────────────────────
bh_pv     = bh_results['BuyAndHold_EW']['portfolio_value']
ew_ref_pv = ew_results['EW_monthly_TC0.5%']['portfolio_value']

def parse_pct(s):
    return float(str(s).replace('%', '')) / 100

# ── Plot A: Best of each variant vs benchmarks (monthly, TC=0.5%) ─────────────
fig, axes = plt.subplots(2, 4, figsize=(28, 12), sharey=False)
fig.suptitle("All Signal Variants vs Benchmarks — Monthly Rebalancing, TC=0.5%",
             fontsize=14, fontweight='bold')

variant_info = [
    ("3-Stage",    signal_results,  THRESHOLDS,     "Sig3S",   "steelblue"),
    ("Asymmetric", asym_results,    THRESHOLDS,     "Asym",    "darkorange"),
    ("Continuous", cont_results,    THRESHOLDS,     "Cont",    "purple"),
    ("Smoothed",   smooth_results,  THRESHOLDS,     "Smooth",  "green"),
    ("Persistent", persist_results, THRESHOLDS,     "Persist", "crimson"),
    ("Partial 25%",partial_results, THRESHOLDS,     "Partial", "brown"),
    ("MV Signal",  mv_results,      THRESHOLDS,     "MV",      "black"),
]

for ax, (name, res_dict, threshs, prefix, col) in zip(axes.flat, variant_info):
    ax.plot(bh_pv.index,     bh_pv / 1e9,     color='forestgreen',
            lw=1.5, ls='-.', label='Buy & Hold')
    ax.plot(ew_ref_pv.index, ew_ref_pv / 1e9, color='grey',
            lw=1.2, ls='--', label='EW Monthly')

    for thresh in threshs:
        candidates = [k for k in res_dict
                      if f"_t{thresh}_" in k and 'monthly' in k and 'TC0.5%' in k]
        if not candidates:
            candidates = [k for k in res_dict
                          if f"_t{thresh}_" in k and 'TC0.5%' in k]
        if candidates:
            key = candidates[0]
            pv  = res_dict[key]['portfolio_value']
            ax.plot(pv.index, pv / 1e9, lw=1.6,
                    label=f"t={thresh}")

    ax.axhline(1.0, color='black', ls=':', lw=0.7)
    ax.set_title(name, fontsize=11, fontweight='bold')
    ax.set_ylabel("Value ($B)")
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

axes.flat[-1].set_visible(False)
plt.tight_layout()
plt.savefig("all_variants_monthly.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Plot B: Risk-Return Scatter — all strategies ──────────────────────────────
fig, ax = plt.subplots(figsize=(16, 10))

all_metric_groups = [
    (bh_metrics,      'BH',       'forestgreen', 'D'),
    (ew_metrics,      'EW',       'grey',        's'),
    (sig_metrics,     '3-Stage',  'steelblue',   'o'),
    (asym_metrics,    'Asym',     'darkorange',  'o'),
    (cont_metrics,    'Cont',     'purple',      'o'),
    (smooth_metrics,  'Smooth',   'green',       'o'),
    (persist_metrics, 'Persist',  'crimson',     'o'),
    (partial_metrics, 'Partial',  'brown',       'o'),
    (mv_metrics,      'MV',       'black',       '^'),
]

for metrics_dict, grp_label, col, marker in all_metric_groups:
    first = True
    for key, m in metrics_dict.items():
        vol = m['Ann_Vol'] * 100
        ret = m['Ann_Ret'] * 100
        ax.scatter(vol, ret, color=col, marker=marker, s=80,
                   label=grp_label if first else None,
                   edgecolors='black', lw=0.5, zorder=4)
        ax.annotate(key.split('_TC')[0], (vol, ret),
                    textcoords='offset points', xytext=(4, 3),
                    fontsize=6, color=col)
        first = False

ax.axhline(0, color='black', lw=0.6, ls='--')
ax.axvline(0, color='black', lw=0.6, ls='--')
ax.set_xlabel("Annualised Volatility (%)", fontsize=12)
ax.set_ylabel("CAGR (%)", fontsize=12)
ax.set_title("Risk-Return — All Strategies", fontsize=13, fontweight='bold')
ax.legend(fontsize=10, ncol=3)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("risk_return_all.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Plot C: Sharpe Bar Chart — top 20 strategies ──────────────────────────────
all_metrics_combined = {
    **bh_metrics, **ew_metrics, **sig_metrics,
    **asym_metrics, **cont_metrics, **smooth_metrics,
    **persist_metrics, **partial_metrics, **mv_metrics
}

sharpe_series = pd.Series({
    k: float(v['Sharpe']) for k, v in all_metrics_combined.items()
}).dropna().sort_values(ascending=False)

top20 = sharpe_series.head(20)

def label_color(lbl):
    if lbl.startswith('MV'):       return 'black'
    if lbl.startswith('Sig3S'):    return 'steelblue'
    if lbl.startswith('Asym'):     return 'darkorange'
    if lbl.startswith('Cont'):     return 'purple'
    if lbl.startswith('Smooth'):   return 'green'
    if lbl.startswith('Persist'):  return 'crimson'
    if lbl.startswith('Partial'):  return 'brown'
    if lbl.startswith('EW'):       return 'grey'
    return 'forestgreen'

colors = [label_color(k) for k in top20.index]

fig, ax = plt.subplots(figsize=(16, 7))
bars = ax.barh(range(len(top20)), top20.values, color=colors, edgecolor='black', lw=0.5)
ax.set_yticks(range(len(top20)))
ax.set_yticklabels([k.replace('_TC', ' TC=') for k in top20.index], fontsize=8)
ax.set_xlabel("Sharpe Ratio")
ax.set_title("Top 20 Strategies by Sharpe Ratio", fontsize=13, fontweight='bold')
ax.axvline(0, color='black', lw=0.8)
ax.grid(True, axis='x', alpha=0.3)

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='black',      label='MV Signal'),
    Patch(facecolor='steelblue',  label='3-Stage'),
    Patch(facecolor='darkorange', label='Asymmetric'),
    Patch(facecolor='purple',     label='Continuous'),
    Patch(facecolor='green',      label='Smoothed'),
    Patch(facecolor='crimson',    label='Persistent'),
    Patch(facecolor='brown',      label='Partial Hedge'),
    Patch(facecolor='grey',       label='Equal Weight'),
    Patch(facecolor='forestgreen',label='Buy & Hold'),
]
ax.legend(handles=legend_elements, fontsize=9, loc='lower right')
plt.tight_layout()
plt.savefig("sharpe_top20.png", dpi=150, bbox_inches='tight')
plt.show()

# ── 18. SAVE ALL ──────────────────────────────────────────────────────────────
print("\n── Saving all results ──")
all_portfolios = {
    **bh_results, **ew_results, **signal_results,
    **asym_results, **cont_results, **smooth_results,
    **persist_results, **partial_results, **mv_results
}

for label, df in all_portfolios.items():
    df.to_csv(f"{label}_portfolio.csv")

all_metrics_df = pd.DataFrame({
    k: {mk: mv for mk, mv in m.items() if mk not in ('Ann_Ret', 'Ann_Vol')}
    for k, m in all_metrics_combined.items()
}).T
all_metrics_df.to_csv("all_portfolio_metrics.csv")

composite_score.to_csv("composite_signal.csv")
for thresh in THRESHOLDS:
    sig_df_t = build_signal_df(composite_score, equity_assets,
                               macro_df.index, threshold=thresh)
    sig_df_t.to_csv(f"signal_df_threshold_{thresh}.csv")

for label, wdf in mv_weight_history.items():
    wdf.to_csv(f"{label}_weights.csv")

print("✓ All files saved.")
print(f"\n  Portfolio CSVs   : {len(all_portfolios)}")
print(f"  Weight CSVs      : {len(mv_weight_history)}")
print(f"  Metrics CSV      : all_portfolio_metrics.csv")
print(f"  Signal CSVs      : {len(THRESHOLDS)} threshold files")
print(f"  Plots saved      : 3 PNG files")


In [ ]:
# @title
# ============================================================
# SIGNAL PORTFOLIO — ENHANCED VERSION
# Signals generated from Modified Determinant Model
# + Buy-and-Hold Benchmark
# + Equal-Weight Benchmark
# + Three-Stage Signal Portfolio (original)
# + Enhanced Signal Variants (asymmetric, continuous, smoothed, persistent, partial)
# + Mean-Variance Signal Portfolio
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import requests
from io import StringIO
from scipy.optimize import minimize

# ── 1. LOAD ALL DATA FROM GITHUB ─────────────────────────────────────────────
GITHUB_BASE = "https://raw.githubusercontent.com/kboroz/MSFE_Capstone_Project/main/01_Data/Streamlined/"

INDEX_FILE_MAP = {
    'ASX50':       'streamlined_asx_50.csv',
    'EUROSTOXX50': 'streamlined_euro_stoxx_50.csv',
    'FTSE100':     'streamlined_ftse_100.csv',
    'Ibovespa':    'streamlined_ibovespa.csv',
    'JSE40':       'streamlined_jse_top_40.csv',
    'NIKKEI225':   'streamlined_nikkei_225.csv',
    'SP500':       'streamlined_s&p_500.csv',
}

MACRO_FILE = 'streamlined_macro.csv'

def load_csv(url):
    r = requests.get(url)
    r.raise_for_status()
    df = pd.read_csv(StringIO(r.text), index_col=0, parse_dates=True)
    df.columns = df.columns.str.replace('[^a-zA-Z0-9]', '_', regex=True)
    return df.sort_index()

macro_df = load_csv(GITHUB_BASE + MACRO_FILE)
macro_df = macro_df[~macro_df.index.duplicated(keep='first')].sort_index()
print(f"macro_df shape : {macro_df.shape}")
print(f"Columns        : {macro_df.columns.tolist()}")

index_data = {}
for name, fname in INDEX_FILE_MAP.items():
    try:
        df = load_csv(GITHUB_BASE + fname)
        index_data[name] = df
        print(f"✓ {name:15s} | shape={df.shape} | cols={df.columns.tolist()[:4]}")
    except Exception as e:
        print(f"✗ {name}: {e}")

# ── 2. EXTRACT PRICE SERIES ───────────────────────────────────────────────────
PRICE_COLS = ['Close', 'close', 'Price', 'price', 'Adj_Close', 'Last']

def get_price_series(df, name):
    col = next((c for c in PRICE_COLS if c in df.columns), df.columns[0])
    s = pd.to_numeric(df[col], errors='coerce').dropna()
    print(f"  {name}: using column '{col}'")
    return s

index_prices = {}
for name, df in index_data.items():
    index_prices[name] = get_price_series(df, name)

# ── 3. MODIFIED DETERMINANT SIGNAL MODEL ─────────────────────────────────────
DET_WINDOWS = [3, 7, 28]
WINDOW_WEIGHTS = {3: 0.2, 7: 0.3, 28: 0.5}

def compute_modified_determinant(price_series_dict, windows=DET_WINDOWS):
    prices = pd.DataFrame(price_series_dict).sort_index()
    rets   = prices.pct_change().dropna()
    det_results = {}

    for window in windows:
        det_series = []
        for i in range(window, len(rets)):
            window_rets = rets.iloc[i - window:i]
            valid_cols  = window_rets.columns[window_rets.std() > 1e-10]
            w           = window_rets[valid_cols]

            if len(valid_cols) < 2:
                det_series.append(np.nan)
                continue

            try:
                R   = w.corr().values
                I   = np.eye(len(R))
                M   = R - I
                det = np.linalg.det(M)
            except Exception:
                det = np.nan

            det_series.append(det)

        idx = rets.index[window:]
        det_results[window] = pd.Series(det_series, index=idx,
                                        name=f'det_{window}d')
        print(f"  Window {window:2d}d | computed {len(det_series)} observations")

    return det_results

print("\n── Computing Modified Determinant ──")
det_results = compute_modified_determinant(index_prices)

# ── 4. COMPOSITE SCORE ────────────────────────────────────────────────────────
def compute_composite_score(det_results, window_weights=WINDOW_WEIGHTS):
    all_series = []
    for window, series in det_results.items():
        mu    = series.rolling(252, min_periods=60).mean()
        sigma = series.rolling(252, min_periods=60).std()
        z     = (series - mu) / sigma.replace(0, np.nan)
        z.name = f'z_{window}d'
        all_series.append(z)

    z_df      = pd.concat(all_series, axis=1).dropna()
    composite = sum(
        z_df[f'z_{w}d'] * window_weights[w]
        for w in window_weights if f'z_{w}d' in z_df.columns
    )
    composite = composite / sum(window_weights.values())
    return composite, z_df

composite_score, z_df = compute_composite_score(det_results)

print(f"\nComposite score shape  : {composite_score.shape}")
print(f"Date range             : {composite_score.index[0].date()} → "
      f"{composite_score.index[-1].date()}")
print(f"Score range            : {composite_score.min():.3f} → "
      f"{composite_score.max():.3f}")

# ── 5. SIGNAL BUILDERS ───────────────────────────────────────────────────────
THRESHOLDS    = [0.3, 0.6, 0.9]
SMOOTH_WINDOWS = [3, 5, 10]
MIN_DAYS_LIST  = [5, 10, 20]
MAX_REDUCES    = [0.5, 0.7, 0.9]
PARTIAL_KEEPS  = [0.25, 0.50]

# ── 5a. Original symmetric binary ────────────────────────────────────────────
def classify_signal_symmetric(score, threshold):
    return +1 if abs(score) > threshold else -1

def build_signal_df(composite_score, equity_assets, macro_dates, threshold=0.5):
    binary    = composite_score.apply(lambda x: classify_signal_symmetric(x, threshold))
    signal_df = pd.DataFrame(index=macro_dates)
    for asset in equity_assets:
        signal_df[asset] = binary.reindex(macro_dates).ffill().bfill()
    return signal_df

# ── 5b. Asymmetric (upper-tail only) ─────────────────────────────────────────
def build_signal_df_asymmetric(composite_score, equity_assets, macro_dates,
                               threshold=1.0):
    """Block equities only when correlation SPIKES (high positive z-score)."""
    binary    = composite_score.apply(lambda x: +1 if x > threshold else -1)
    signal_df = pd.DataFrame(index=macro_dates)
    for asset in equity_assets:
        signal_df[asset] = binary.reindex(macro_dates).ffill().bfill()
    return signal_df

# ── 5c. Continuous / graduated scalar ────────────────────────────────────────
def build_signal_df_continuous(composite_score, equity_assets, macro_dates,
                               threshold=1.0, max_reduce=0.8):
    """
    Returns a scalar in [1-max_reduce, 1.0] per asset per day.
    Used by build_weights_continuous() instead of the binary flag.
    """
    def score_to_scalar(s):
        if s <= 0:
            return 1.0
        elif s >= threshold:
            return 1.0 - max_reduce
        else:
            return 1.0 - max_reduce * (s / threshold)

    scalars   = composite_score.apply(score_to_scalar)
    signal_df = pd.DataFrame(index=macro_dates)
    for asset in equity_assets:
        signal_df[asset] = scalars.reindex(macro_dates).ffill().bfill()
    return signal_df

# ── 5d. Smoothed binary ───────────────────────────────────────────────────────
def build_signal_df_smoothed(composite_score, equity_assets, macro_dates,
                             threshold=1.0, smooth_window=5):
    """Smooth z-score before thresholding to reduce whipsaw."""
    smoothed  = composite_score.rolling(smooth_window, min_periods=1).mean()
    binary    = smoothed.apply(lambda x: classify_signal_symmetric(x, threshold))
    signal_df = pd.DataFrame(index=macro_dates)
    for asset in equity_assets:
        signal_df[asset] = binary.reindex(macro_dates).ffill().bfill()
    return signal_df

# ── 5e. Persistent binary ────────────────────────────────────────────────────
def build_signal_df_persistent(composite_score, equity_assets, macro_dates,
                               threshold=1.0, min_days_to_switch=10):
    """Only switch regime after signal holds for min_days_to_switch days."""
    raw       = composite_score.apply(lambda x: +1 if x > threshold else -1)
    confirmed = raw.copy()
    current   = int(raw.iloc[0])
    count     = 0

    for i in range(1, len(raw)):
        if int(raw.iloc[i]) != current:
            count += 1
            if count >= min_days_to_switch:
                current = int(raw.iloc[i])
                count   = 0
        else:
            count = 0
        confirmed.iloc[i] = current

    signal_df = pd.DataFrame(index=macro_dates)
    for asset in equity_assets:
        signal_df[asset] = confirmed.reindex(macro_dates).ffill().bfill()
    return signal_df

# ── 6. ASSET UNIVERSE ─────────────────────────────────────────────────────────
EQUITY_ASSETS = list(index_prices.keys())
SAFE_BONDS    = ['US_10Y', 'UK_10Y', 'AU_10Y', 'ZA_10Y']
FX            = ['AUDUSD', 'BRLUSD', 'EURUSD', 'GBPUSD', 'JPYUSD', 'ZARUSD']
COMMODITIES   = ['Oil', 'Gold', 'Bitcoin']
EXCLUDE_ASSETS = ['Bitcoin']

ALL_ASSETS = EQUITY_ASSETS + FX + COMMODITIES + SAFE_BONDS
ALL_ASSETS = [a for a in ALL_ASSETS
              if a in macro_df.columns and a not in EXCLUDE_ASSETS]

equity_assets     = [a for a in EQUITY_ASSETS     if a in ALL_ASSETS]
non_equity_assets = [a for a in ALL_ASSETS         if a not in equity_assets]

N           = len(ALL_ASSETS)
base_weight = 1.0 / N

print(f"\nTotal assets              : {N}")
print(f"Equity (signal-driven)    : {equity_assets}")
print(f"Non-equity (always long)  : {non_equity_assets}")
print(f"base_weight = 1/{N} = {base_weight:.4f}")

# ── 7. COUNTRY SAFE-HAVEN MAP ─────────────────────────────────────────────────
EQUITY_SAFE_HAVEN_MAP = {
    'ASX50':       ['AU_10Y', 'AUDUSD'],
    'EUROSTOXX50': ['EURUSD'],
    'FTSE100':     ['UK_10Y', 'GBPUSD'],
    'Ibovespa':    ['BRLUSD'],
    'JSE40':       ['ZA_10Y', 'ZARUSD'],
    'NIKKEI225':   ['JPYUSD'],
    'SP500':       ['US_10Y'],
}

UNIVERSE_SET = set(ALL_ASSETS)
for eq, havens in EQUITY_SAFE_HAVEN_MAP.items():
    EQUITY_SAFE_HAVEN_MAP[eq] = [h for h in havens if h in UNIVERSE_SET]

print("\nSafe-haven routing (filtered to universe):")
for eq, havens in EQUITY_SAFE_HAVEN_MAP.items():
    print(f"  {eq:15s} → {havens}")

# ── 8. BUILD RETURNS ──────────────────────────────────────────────────────────
returns_raw = macro_df[ALL_ASSETS].pct_change(fill_method=None)

for col in SAFE_BONDS:
    if col in returns_raw.columns:
        returns_raw[col] = -returns_raw[col]

if 'Oil' in macro_df.columns:
    bad_oil = macro_df['Oil'][macro_df['Oil'] <= 0].index
    for d in bad_oil:
        loc = returns_raw.index.get_loc(d)
        returns_raw.iloc[max(0, loc - 1): loc + 2,
                         returns_raw.columns.get_loc('Oil')] = 0.0

returns = returns_raw.clip(lower=-0.50, upper=1.00).fillna(0).iloc[1:]

assert not returns.isnull().any().any(), "NaNs in returns"
assert not np.isinf(returns.values).any(),  "Inf in returns"
print(f"\nReturns shape : {returns.shape}")
print("✓ Returns clean")

# ── 9. HELPER: REBALANCING DATE SET ──────────────────────────────────────────
def build_reb_set(returns, rebal_freq):
    actual_dates = returns.index
    if rebal_freq == 'daily':
        return set(actual_dates)
    elif rebal_freq == 'weekly':
        raw = returns.resample('W-MON').first().index
    elif rebal_freq == 'monthly':
        raw = returns.resample('MS').first().index
    elif rebal_freq == 'quarterly':
        raw = returns.resample('QS').first().index
    else:
        raise ValueError(f"Unknown rebal_freq: {rebal_freq}")

    reb_set = set()
    for d in raw:
        future = actual_dates[actual_dates >= d]
        if len(future) > 0:
            reb_set.add(future[0])
    return reb_set

# ── 10. BUY-AND-HOLD BENCHMARK ───────────────────────────────────────────────
def run_buy_and_hold(returns, all_assets, initial_capital=1e9):
    N           = len(all_assets)
    base_weight = 1.0 / N
    pv          = float(initial_capital)
    holdings    = {a: pv * base_weight for a in all_assets}
    records     = []
    first       = True

    for date in returns.index:
        if first:
            first = False
            records.append({'date': date, 'portfolio_value': pv})
            continue
        new_holdings = {a: holdings[a] * (1 + returns.loc[date, a])
                        for a in all_assets}
        holdings = new_holdings
        pv       = sum(holdings.values())
        records.append({'date': date, 'portfolio_value': pv})

    df = pd.DataFrame(records).set_index('date')
    print(f"  Buy-and-Hold | Start: ${initial_capital/1e9:.2f}B → "
          f"End: ${df['portfolio_value'].iloc[-1]/1e9:.3f}B")
    return df

# ── 11. WEIGHT BUILDERS ───────────────────────────────────────────────────────

# ── 11a. Original binary 3-stage weights ─────────────────────────────────────
def build_weights_3stage(sig_row, equity_assets, non_equity_assets,
                         base_weight, date, safe_haven_map):
    raw            = {a: base_weight for a in non_equity_assets}
    safe_haven_extra = {}

    for a in equity_assets:
        if sig_row is not None and a in sig_row.index and not pd.isna(sig_row[a]):
            sig = sig_row[a]
        else:
            sig = -1

        if sig > 0:
            raw[a] = -base_weight
            havens = [h for h in safe_haven_map.get(a, []) if h in raw]
            if havens:
                per_haven = base_weight / len(havens)
                for h in havens:
                    safe_haven_extra[h] = safe_haven_extra.get(h, 0) + per_haven
            else:
                per_ne = base_weight / len(non_equity_assets)
                for ne in non_equity_assets:
                    safe_haven_extra[ne] = safe_haven_extra.get(ne, 0) + per_ne
        else:
            raw[a] = base_weight

    for asset, extra in safe_haven_extra.items():
        raw[asset] = raw.get(asset, 0) + extra

    total = sum(raw.values())
    if total <= 0:
        for a in raw:
            raw[a] = 1.0 / len(raw)
    else:
        for a in raw:
            raw[a] /= total
    return raw

# ── 11b. Continuous scalar weights ───────────────────────────────────────────
def build_weights_continuous(sig_row, equity_assets, non_equity_assets,
                             base_weight, safe_haven_map):
    """
    sig_row contains a scalar in [min_weight, 1.0] per equity asset.
    Reduced equity weight is redistributed to safe havens.
    """
    raw              = {a: base_weight for a in non_equity_assets}
    safe_haven_extra = {}

    for a in equity_assets:
        scalar = float(sig_row[a]) if (sig_row is not None and
                                       a in sig_row.index and
                                       not pd.isna(sig_row[a])) else 1.0
        raw[a]   = base_weight * scalar
        released = base_weight * (1.0 - scalar)

        if released > 0:
            havens = [h for h in safe_haven_map.get(a, []) if h in raw]
            if havens:
                per_haven = released / len(havens)
                for h in havens:
                    safe_haven_extra[h] = safe_haven_extra.get(h, 0) + per_haven
            else:
                per_ne = released / len(non_equity_assets)
                for ne in non_equity_assets:
                    safe_haven_extra[ne] = safe_haven_extra.get(ne, 0) + per_ne

    for asset, extra in safe_haven_extra.items():
        raw[asset] = raw.get(asset, 0) + extra

    total = sum(raw.values())
    if total <= 0:
        for a in raw:
            raw[a] = 1.0 / len(raw)
    else:
        for a in raw:
            raw[a] /= total
    return raw

# ── 11c. Partial-hedge weights ────────────────────────────────────────────────
def build_weights_partial(sig_row, equity_assets, non_equity_assets,
                          base_weight, safe_haven_map, partial_keep=0.25):
    """
    On RISK_ON signal: keep partial_keep fraction of base_weight in equity,
    redirect the rest to safe havens.
    """
    raw              = {a: base_weight for a in non_equity_assets}
    safe_haven_extra = {}

    for a in equity_assets:
        if sig_row is not None and a in sig_row.index and not pd.isna(sig_row[a]):
            sig = sig_row[a]
        else:
            sig = -1

        if sig > 0:
            raw[a]   = base_weight * partial_keep
            released = base_weight * (1.0 - partial_keep)
            havens   = [h for h in safe_haven_map.get(a, []) if h in raw]
            if havens:
                per_haven = released / len(havens)
                for h in havens:
                    safe_haven_extra[h] = safe_haven_extra.get(h, 0) + per_haven
            else:
                per_ne = released / len(non_equity_assets)
                for ne in non_equity_assets:
                    safe_haven_extra[ne] = safe_haven_extra.get(ne, 0) + per_ne
        else:
            raw[a] = base_weight

    for asset, extra in safe_haven_extra.items():
        raw[asset] = raw.get(asset, 0) + extra

    total = sum(raw.values())
    if total <= 0:
        for a in raw:
            raw[a] = 1.0 / len(raw)
    else:
        for a in raw:
            raw[a] /= total
    return raw

# ── 12. EQUAL-WEIGHT REBALANCED BENCHMARK ────────────────────────────────────
def run_equal_weight(returns, all_assets, tc_rate=0.005,
                     rebal_freq='monthly', initial_capital=1e9):
    N           = len(all_assets)
    base_weight = 1.0 / N
    ew          = {a: base_weight for a in all_assets}
    reb_set     = build_reb_set(returns, rebal_freq)
    pv          = float(initial_capital)
    current_weights = None
    records     = []

    for date in returns.index:
        if date in reb_set:
            if current_weights is not None:
                turnover = sum(abs(ew.get(a, 0) - current_weights.get(a, 0))
                               for a in set(ew) | set(current_weights)) / 2.0
                pv *= (1 - turnover * tc_rate)
            else:
                pv *= (1 - tc_rate)
            current_weights = ew.copy()

        if current_weights is not None:
            dr  = sum(current_weights.get(a, 0) * returns.loc[date, a]
                      for a in all_assets)
            pv *= (1 + dr)

        records.append({'date': date, 'portfolio_value': pv})

    return pd.DataFrame(records).set_index('date')

# ── 13. GENERIC SIGNAL PORTFOLIO RUNNER ──────────────────────────────────────
def run_signal_portfolio(returns, signal_df, equity_assets,
                         non_equity_assets, all_assets,
                         weight_fn,          # callable: (sig_row, ...) → dict
                         freq='monthly', tc_rate=0.005,
                         initial_capital=1e9):
    """
    Generic runner.  weight_fn must accept (sig_row, equity_assets,
    non_equity_assets, base_weight, <any extra kwargs>) and return
    a dict {asset: weight}.
    """
    N           = len(all_assets)
    base_weight = 1.0 / N
    reb_set     = build_reb_set(returns, freq)

    sig = signal_df.copy()
    sig.index = pd.to_datetime(sig.index)

    pv              = float(initial_capital)
    current_weights = None
    records         = []

    for date in returns.index:
        if date in reb_set:
            past    = sig[sig.index <= date]
            sig_row = past.iloc[-1] if len(past) > 0 else None
            raw     = weight_fn(sig_row)

            if current_weights is not None:
                all_keys = set(raw) | set(current_weights)
                turnover = sum(abs(raw.get(a, 0) - current_weights.get(a, 0))
                               for a in all_keys) / 2.0
                pv *= (1 - turnover * tc_rate)
            else:
                pv *= (1 - tc_rate)

            current_weights = raw.copy()

        if current_weights is not None:
            dr  = sum(current_weights.get(a, 0) * returns.loc[date, a]
                      for a in all_assets)
            pv *= (1 + dr)

        records.append({'date': date, 'portfolio_value': pv})

    return pd.DataFrame(records).set_index('date')

# ── 14. MEAN-VARIANCE SIGNAL PORTFOLIO ───────────────────────────────────────
MV_LOOKBACK = 252
MV_MIN_OBS  = 60
MAX_WEIGHT  = 0.30
RISK_FREE   = 0.0

def mean_variance_weights(mu_vec, cov_mat, allowed_mask,
                          max_weight=MAX_WEIGHT, risk_free=RISK_FREE):
    n_total   = len(mu_vec)
    allowed   = np.where(allowed_mask)[0]
    n_allowed = len(allowed)

    fallback = np.zeros(n_total)
    if n_allowed == 0:
        fallback[:] = 1.0 / n_total
        return fallback
    fallback[allowed] = 1.0 / n_allowed

    mu_sub  = mu_vec[allowed]
    cov_sub = cov_mat[np.ix_(allowed, allowed)] + np.eye(n_allowed) * 1e-8

    def neg_sharpe(w):
        port_ret = np.dot(w, mu_sub)
        port_var = np.dot(w, cov_sub @ w)
        port_std = np.sqrt(max(port_var, 1e-12))
        return -(port_ret - risk_free) / port_std

    constraints = [{'type': 'eq', 'fun': lambda w: np.sum(w) - 1.0}]
    bounds      = [(0.0, max_weight)] * n_allowed
    w0          = np.ones(n_allowed) / n_allowed

    try:
        res = minimize(neg_sharpe, w0, method='SLSQP',
                       bounds=bounds, constraints=constraints,
                       options={'ftol': 1e-9, 'maxiter': 500})
        w_opt = res.x if (res.success and not np.any(np.isnan(res.x))) else w0
    except Exception:
        w_opt = w0

    w_full         = np.zeros(n_total)
    w_full[allowed] = w_opt
    return w_full

def run_mv_signal_portfolio(returns, signal_df, all_assets,
                            equity_assets,
                            freq='monthly', tc_rate=0.005,
                            lookback=MV_LOOKBACK, min_obs=MV_MIN_OBS,
                            max_weight=MAX_WEIGHT, initial_capital=1e9):
    reb_set      = build_reb_set(returns, freq)
    actual_dates = returns.index
    assets       = all_assets
    n            = len(assets)
    asset_idx    = {a: i for i, a in enumerate(assets)}

    sig = signal_df.copy()
    sig.index = pd.to_datetime(sig.index)

    pv              = float(initial_capital)
    current_weights = np.zeros(n)
    records         = []
    weight_history  = []
    first_reb       = True

    for t, date in enumerate(actual_dates):
        if date in reb_set:
            hist = returns.iloc[max(0, t - lookback): t]

            if len(hist) < min_obs:
                new_weights = np.ones(n) / n
            else:
                mu_ann  = hist.mean().values  * 252
                cov_ann = hist.cov().values   * 252

                past    = sig[sig.index <= date]
                sig_row = past.iloc[-1] if len(past) > 0 else None

                allowed_mask = np.ones(n, dtype=bool)
                if sig_row is not None:
                    for eq in equity_assets:
                        if eq in asset_idx and eq in sig_row.index:
                            s = sig_row[eq]
                            if not pd.isna(s) and s > 0:
                                allowed_mask[asset_idx[eq]] = False

                new_weights = mean_variance_weights(
                    mu_ann, cov_ann, allowed_mask, max_weight=max_weight)

            if first_reb:
                pv       *= (1 - tc_rate)
                first_reb = False
            else:
                turnover = np.sum(np.abs(new_weights - current_weights)) / 2.0
                pv      *= (1 - turnover * tc_rate)

            current_weights = new_weights.copy()
            weight_history.append({'date': date,
                                   **dict(zip(assets, current_weights))})

        if not first_reb:
            dr  = np.dot(current_weights, returns.loc[date].values)
            pv *= (1 + dr)

        records.append({'date': date, 'portfolio_value': pv})

    pv_df      = pd.DataFrame(records).set_index('date')
    weights_df = pd.DataFrame(weight_history).set_index('date')
    print(f"  MV | freq={freq} tc={tc_rate*100:.1f}% "
          f"End: ${pv_df['portfolio_value'].iloc[-1]/1e9:.3f}B")
    return pv_df, weights_df

# ── 15. RUN ALL PORTFOLIOS ────────────────────────────────────────────────────
TC_LEVELS = {'0%': 0.00, '0.5%': 0.005, '1%': 0.01, '2%': 0.02}
FREQS     = ['daily', 'weekly', 'monthly', 'quarterly']
MV_FREQS  = ['monthly', 'quarterly']

# ── 15a. Buy-and-Hold ─────────────────────────────────────────────────────────
print("\n── Running Buy-and-Hold ──")
bh_result  = run_buy_and_hold(returns, ALL_ASSETS, initial_capital=1e9)
bh_results = {'BuyAndHold_EW': bh_result}

# ── 15b. Equal-Weight ─────────────────────────────────────────────────────────
print("\n── Running Equal-Weight Rebalanced ──")
ew_results = {}
for freq in FREQS:
    for tc_label, tc_rate in TC_LEVELS.items():
        label = f"EW_{freq}_TC{tc_label}"
        ew_results[label] = run_equal_weight(
            returns, ALL_ASSETS, tc_rate=tc_rate,
            rebal_freq=freq, initial_capital=1e9)
        print(f"  ✓ {label}")

# ── 15c. Original 3-Stage Signal ─────────────────────────────────────────────
print("\n── Running Original 3-Stage Signal ──")
signal_results       = {}
signal_distributions = {}

for thresh in THRESHOLDS:
    sig_df_t    = build_signal_df(composite_score, equity_assets,
                                  macro_df.index, threshold=thresh)
    n_risk_on   = (sig_df_t[equity_assets[0]] == +1).sum()
    n_risk_off  = (sig_df_t[equity_assets[0]] == -1).sum()
    total       = len(sig_df_t)
    signal_distributions[thresh] = {'RISK_ON': n_risk_on, 'RISK_OFF': n_risk_off}
    print(f"  Threshold={thresh} | RISK_ON: {n_risk_on} ({n_risk_on/total*100:.1f}%) "
          f"| RISK_OFF: {n_risk_off} ({n_risk_off/total*100:.1f}%)")

    for freq in FREQS:
        for tc_label, tc_rate in TC_LEVELS.items():
            label = f"Sig3S_t{thresh}_{freq}_TC{tc_label}"

            def wfn_3s(sig_row, _eq=equity_assets, _ne=non_equity_assets,
                       _bw=base_weight, _sh=EQUITY_SAFE_HAVEN_MAP):
                return build_weights_3stage(sig_row, _eq, _ne, _bw, None, _sh)

            signal_results[label] = run_signal_portfolio(
                returns, sig_df_t, equity_assets, non_equity_assets,
                ALL_ASSETS, weight_fn=wfn_3s,
                freq=freq, tc_rate=tc_rate, initial_capital=1e9)
            print(f"    ✓ {label}")

# ── 15d. Asymmetric Signal ────────────────────────────────────────────────────
print("\n── Running Asymmetric Signal ──")
asym_results = {}

for thresh in THRESHOLDS:
    sig_df_t = build_signal_df_asymmetric(composite_score, equity_assets,
                                          macro_df.index, threshold=thresh)
    for freq in ['monthly', 'quarterly']:
        for tc_label, tc_rate in TC_LEVELS.items():
            label = f"Asym_t{thresh}_{freq}_TC{tc_label}"

            def wfn_asym(sig_row, _eq=equity_assets, _ne=non_equity_assets,
                         _bw=base_weight, _sh=EQUITY_SAFE_HAVEN_MAP):
                return build_weights_3stage(sig_row, _eq, _ne, _bw, None, _sh)

            asym_results[label] = run_signal_portfolio(
                returns, sig_df_t, equity_assets, non_equity_assets,
                ALL_ASSETS, weight_fn=wfn_asym,
                freq=freq, tc_rate=tc_rate, initial_capital=1e9)
            print(f"    ✓ {label}")

# ── 15e. Continuous / Graduated Signal ───────────────────────────────────────
print("\n── Running Continuous Signal ──")
cont_results = {}

for thresh in THRESHOLDS:
    for max_red in MAX_REDUCES:
        sig_df_t = build_signal_df_continuous(composite_score, equity_assets,
                                              macro_df.index,
                                              threshold=thresh,
                                              max_reduce=max_red)
        for freq in ['monthly', 'quarterly']:
            for tc_label, tc_rate in TC_LEVELS.items():
                label = (f"Cont_t{thresh}_mr{int(max_red*100)}"
                         f"_{freq}_TC{tc_label}")

                def wfn_cont(sig_row, _eq=equity_assets, _ne=non_equity_assets,
                             _bw=base_weight, _sh=EQUITY_SAFE_HAVEN_MAP):
                    return build_weights_continuous(sig_row, _eq, _ne, _bw, _sh)

                cont_results[label] = run_signal_portfolio(
                    returns, sig_df_t, equity_assets, non_equity_assets,
                    ALL_ASSETS, weight_fn=wfn_cont,
                    freq=freq, tc_rate=tc_rate, initial_capital=1e9)
                print(f"    ✓ {label}")

# ── 15f. Smoothed Signal ─────────────────────────────────────────────────────
print("\n── Running Smoothed Signal ──")
smooth_results = {}

for thresh in THRESHOLDS:
    for sw in SMOOTH_WINDOWS:
        sig_df_t = build_signal_df_smoothed(composite_score, equity_assets,
                                            macro_df.index,
                                            threshold=thresh,
                                            smooth_window=sw)
        for freq in ['monthly', 'quarterly']:
            for tc_label, tc_rate in TC_LEVELS.items():
                label = f"Smooth_t{thresh}_sw{sw}_{freq}_TC{tc_label}"

                def wfn_smooth(sig_row, _eq=equity_assets, _ne=non_equity_assets,
                               _bw=base_weight, _sh=EQUITY_SAFE_HAVEN_MAP):
                    return build_weights_3stage(sig_row, _eq, _ne, _bw, None, _sh)

                smooth_results[label] = run_signal_portfolio(
                    returns, sig_df_t, equity_assets, non_equity_assets,
                    ALL_ASSETS, weight_fn=wfn_smooth,
                    freq=freq, tc_rate=tc_rate, initial_capital=1e9)
                print(f"    ✓ {label}")

# ── 15g. Persistent Signal ────────────────────────────────────────────────────
print("\n── Running Persistent Signal ──")
persist_results = {}

for thresh in THRESHOLDS:
    for min_days in MIN_DAYS_LIST:
        sig_df_t = build_signal_df_persistent(composite_score, equity_assets,
                                              macro_df.index,
                                              threshold=thresh,
                                              min_days_to_switch=min_days)
        for freq in ['monthly', 'quarterly']:
            for tc_label, tc_rate in TC_LEVELS.items():
                label = f"Persist_t{thresh}_md{min_days}_{freq}_TC{tc_label}"

                def wfn_persist(sig_row, _eq=equity_assets, _ne=non_equity_assets,
                                _bw=base_weight, _sh=EQUITY_SAFE_HAVEN_MAP):
                    return build_weights_3stage(sig_row, _eq, _ne, _bw, None, _sh)

                persist_results[label] = run_signal_portfolio(
                    returns, sig_df_t, equity_assets, non_equity_assets,
                    ALL_ASSETS, weight_fn=wfn_persist,
                    freq=freq, tc_rate=tc_rate, initial_capital=1e9)
                print(f"    ✓ {label}")

# ── 15h. Partial-Hedge Signal ─────────────────────────────────────────────────
print("\n── Running Partial-Hedge Signal ──")
partial_results = {}

for thresh in THRESHOLDS:
    sig_df_t = build_signal_df(composite_score, equity_assets,
                               macro_df.index, threshold=thresh)
    for pk in PARTIAL_KEEPS:
        for freq in ['monthly', 'quarterly']:
            for tc_label, tc_rate in TC_LEVELS.items():
                label = (f"Partial_t{thresh}_pk{int(pk*100)}"
                         f"_{freq}_TC{tc_label}")

                def wfn_partial(sig_row, _eq=equity_assets, _ne=non_equity_assets,
                                _bw=base_weight, _sh=EQUITY_SAFE_HAVEN_MAP,
                                _pk=pk):
                    return build_weights_partial(sig_row, _eq, _ne, _bw, _sh, _pk)

                partial_results[label] = run_signal_portfolio(
                    returns, sig_df_t, equity_assets, non_equity_assets,
                    ALL_ASSETS, weight_fn=wfn_partial,
                    freq=freq, tc_rate=tc_rate, initial_capital=1e9)
                print(f"    ✓ {label}")

# ── 15i. Mean-Variance Signal ─────────────────────────────────────────────────
print("\n── Running Mean-Variance Signal ──")
mv_results        = {}
mv_weight_history = {}

for thresh in THRESHOLDS:
    sig_df_t = build_signal_df(composite_score, equity_assets,
                               macro_df.index, threshold=thresh)
    for freq in MV_FREQS:
        for tc_label, tc_rate in TC_LEVELS.items():
            label = f"MV_t{thresh}_{freq}_TC{tc_label}"
            pv_df, w_df = run_mv_signal_portfolio(
                returns, sig_df_t, ALL_ASSETS,
                equity_assets=equity_assets,
                freq=freq, tc_rate=tc_rate,
                lookback=MV_LOOKBACK, min_obs=MV_MIN_OBS,
                max_weight=MAX_WEIGHT, initial_capital=1e9)
            mv_results[label]        = pv_df
            mv_weight_history[label] = w_df
            print(f"  ✓ {label}")

print("\n✓ All portfolios computed")

# ── 16. PERFORMANCE METRICS ───────────────────────────────────────────────────
def compute_metrics(pv_df, label=''):
    pv        = pv_df['portfolio_value']
    daily_ret = pv.pct_change().dropna()
    n_years   = len(daily_ret) / 252
    total_ret = (pv.iloc[-1] / pv.iloc[0]) - 1
    cagr      = (1 + total_ret) ** (1 / n_years) - 1 if n_years > 0 else np.nan
    vol       = daily_ret.std() * np.sqrt(252)
    sharpe    = cagr / vol if vol > 0 else np.nan
    roll_max  = pv.cummax()
    max_dd    = ((pv - roll_max) / roll_max).min()
    calmar    = cagr / abs(max_dd) if max_dd != 0 else np.nan
    var_95    = daily_ret.quantile(0.05)
    ann_ret   = cagr
    ann_vol   = vol

    return {
        'CAGR':         f"{cagr*100:.2f}%",
        'Ann_Ret':      ann_ret,
        'Ann_Vol':      ann_vol,
        'Volatility':   f"{vol*100:.2f}%",
        'Sharpe':       f"{sharpe:.2f}",
        'Max Drawdown': f"{max_dd*100:.2f}%",
        'Calmar':       f"{calmar:.2f}",
        'VaR 95%':      f"{var_95*100:.2f}%",
        'Total Return': f"{total_ret*100:.2f}%",
        'Final ($B)':   f"{pv.iloc[-1]/1e9:.3f}",
    }

bh_metrics      = {k: compute_metrics(v, k) for k, v in bh_results.items()}
ew_metrics      = {k: compute_metrics(v, k) for k, v in ew_results.items()}
sig_metrics     = {k: compute_metrics(v, k) for k, v in signal_results.items()}
asym_metrics    = {k: compute_metrics(v, k) for k, v in asym_results.items()}
cont_metrics    = {k: compute_metrics(v, k) for k, v in cont_results.items()}
smooth_metrics  = {k: compute_metrics(v, k) for k, v in smooth_results.items()}
persist_metrics = {k: compute_metrics(v, k) for k, v in persist_results.items()}
partial_metrics = {k: compute_metrics(v, k) for k, v in partial_results.items()}
mv_metrics      = {k: compute_metrics(v, k) for k, v in mv_results.items()}

def print_metrics_table(metrics_dict, title):
    display = {k: {mk: mv for mk, mv in m.items()
                   if mk not in ('Ann_Ret', 'Ann_Vol')}
               for k, m in metrics_dict.items()}
    print(f"\n{'='*90}")
    print(title)
    print("="*90)
    print(pd.DataFrame(display).T.to_string())

print_metrics_table(bh_metrics,      "BUY-AND-HOLD")
print_metrics_table(ew_metrics,      "EQUAL-WEIGHT REBALANCED")

for thresh in THRESHOLDS:
    sub = {k: v for k, v in sig_metrics.items()     if f"_t{thresh}_" in k}
    print_metrics_table(sub, f"3-STAGE SIGNAL — threshold={thresh}")

for thresh in THRESHOLDS:
    sub = {k: v for k, v in asym_metrics.items()    if f"_t{thresh}_" in k}
    print_metrics_table(sub, f"ASYMMETRIC SIGNAL — threshold={thresh}")

for thresh in THRESHOLDS:
    sub = {k: v for k, v in cont_metrics.items()    if f"_t{thresh}_" in k}
    print_metrics_table(sub, f"CONTINUOUS SIGNAL — threshold={thresh}")

for thresh in THRESHOLDS:
    sub = {k: v for k, v in smooth_metrics.items()  if f"_t{thresh}_" in k}
    print_metrics_table(sub, f"SMOOTHED SIGNAL — threshold={thresh}")

for thresh in THRESHOLDS:
    sub = {k: v for k, v in persist_metrics.items() if f"_t{thresh}_" in k}
    print_metrics_table(sub, f"PERSISTENT SIGNAL — threshold={thresh}")

for thresh in THRESHOLDS:
    sub = {k: v for k, v in partial_metrics.items() if f"_t{thresh}_" in k}
    print_metrics_table(sub, f"PARTIAL-HEDGE SIGNAL — threshold={thresh}")

for thresh in THRESHOLDS:
    sub = {k: v for k, v in mv_metrics.items()      if f"_t{thresh}_" in k}
    print_metrics_table(sub, f"MEAN-VARIANCE SIGNAL — threshold={thresh}")

# ── 17. PLOTS ─────────────────────────────────────────────────────────────────
bh_pv     = bh_results['BuyAndHold_EW']['portfolio_value']
ew_ref_pv = ew_results['EW_monthly_TC0.5%']['portfolio_value']

def parse_pct(s):
    return float(str(s).replace('%', '')) / 100

# ── Plot A: Best of each variant vs benchmarks (monthly, TC=0.5%) ─────────────
fig, axes = plt.subplots(2, 4, figsize=(28, 12), sharey=False)
fig.suptitle("All Signal Variants vs Benchmarks — Monthly Rebalancing, TC=0.5%",
             fontsize=14, fontweight='bold')

variant_info = [
    ("3-Stage",    signal_results,  THRESHOLDS,     "Sig3S",   "steelblue"),
    ("Asymmetric", asym_results,    THRESHOLDS,     "Asym",    "darkorange"),
    ("Continuous", cont_results,    THRESHOLDS,     "Cont",    "purple"),
    ("Smoothed",   smooth_results,  THRESHOLDS,     "Smooth",  "green"),
    ("Persistent", persist_results, THRESHOLDS,     "Persist", "crimson"),
    ("Partial 25%",partial_results, THRESHOLDS,     "Partial", "brown"),
    ("MV Signal",  mv_results,      THRESHOLDS,     "MV",      "black"),
]

for ax, (name, res_dict, threshs, prefix, col) in zip(axes.flat, variant_info):
    ax.plot(bh_pv.index,     bh_pv / 1e9,     color='forestgreen',
            lw=1.5, ls='-.', label='Buy & Hold')
    ax.plot(ew_ref_pv.index, ew_ref_pv / 1e9, color='grey',
            lw=1.2, ls='--', label='EW Monthly')

    for thresh in threshs:
        candidates = [k for k in res_dict
                      if f"_t{thresh}_" in k and 'monthly' in k and 'TC0.5%' in k]
        if not candidates:
            candidates = [k for k in res_dict
                          if f"_t{thresh}_" in k and 'TC0.5%' in k]
        if candidates:
            key = candidates[0]
            pv  = res_dict[key]['portfolio_value']
            ax.plot(pv.index, pv / 1e9, lw=1.6,
                    label=f"t={thresh}")

    ax.axhline(1.0, color='black', ls=':', lw=0.7)
    ax.set_title(name, fontsize=11, fontweight='bold')
    ax.set_ylabel("Value ($B)")
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

axes.flat[-1].set_visible(False)
plt.tight_layout()
plt.savefig("all_variants_monthly.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Plot B: Risk-Return Scatter — all strategies ──────────────────────────────
fig, ax = plt.subplots(figsize=(16, 10))

all_metric_groups = [
    (bh_metrics,      'BH',       'forestgreen', 'D'),
    (ew_metrics,      'EW',       'grey',        's'),
    (sig_metrics,     '3-Stage',  'steelblue',   'o'),
    (asym_metrics,    'Asym',     'darkorange',  'o'),
    (cont_metrics,    'Cont',     'purple',      'o'),
    (smooth_metrics,  'Smooth',   'green',       'o'),
    (persist_metrics, 'Persist',  'crimson',     'o'),
    (partial_metrics, 'Partial',  'brown',       'o'),
    (mv_metrics,      'MV',       'black',       '^'),
]

for metrics_dict, grp_label, col, marker in all_metric_groups:
    first = True
    for key, m in metrics_dict.items():
        vol = m['Ann_Vol'] * 100
        ret = m['Ann_Ret'] * 100
        ax.scatter(vol, ret, color=col, marker=marker, s=80,
                   label=grp_label if first else None,
                   edgecolors='black', lw=0.5, zorder=4)
        ax.annotate(key.split('_TC')[0], (vol, ret),
                    textcoords='offset points', xytext=(4, 3),
                    fontsize=6, color=col)
        first = False

ax.axhline(0, color='black', lw=0.6, ls='--')
ax.axvline(0, color='black', lw=0.6, ls='--')
ax.set_xlabel("Annualised Volatility (%)", fontsize=12)
ax.set_ylabel("CAGR (%)", fontsize=12)
ax.set_title("Risk-Return — All Strategies", fontsize=13, fontweight='bold')
ax.legend(fontsize=10, ncol=3)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("risk_return_all.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Plot C: Sharpe Bar Chart — top 20 strategies ──────────────────────────────
all_metrics_combined = {
    **bh_metrics, **ew_metrics, **sig_metrics,
    **asym_metrics, **cont_metrics, **smooth_metrics,
    **persist_metrics, **partial_metrics, **mv_metrics
}

sharpe_series = pd.Series({
    k: float(v['Sharpe']) for k, v in all_metrics_combined.items()
}).dropna().sort_values(ascending=False)

top20 = sharpe_series.head(20)

def label_color(lbl):
    if lbl.startswith('MV'):       return 'black'
    if lbl.startswith('Sig3S'):    return 'steelblue'
    if lbl.startswith('Asym'):     return 'darkorange'
    if lbl.startswith('Cont'):     return 'purple'
    if lbl.startswith('Smooth'):   return 'green'
    if lbl.startswith('Persist'):  return 'crimson'
    if lbl.startswith('Partial'):  return 'brown'
    if lbl.startswith('EW'):       return 'grey'
    return 'forestgreen'

colors = [label_color(k) for k in top20.index]

fig, ax = plt.subplots(figsize=(16, 7))
bars = ax.barh(range(len(top20)), top20.values, color=colors, edgecolor='black', lw=0.5)
ax.set_yticks(range(len(top20)))
ax.set_yticklabels([k.replace('_TC', ' TC=') for k in top20.index], fontsize=8)
ax.set_xlabel("Sharpe Ratio")
ax.set_title("Top 20 Strategies by Sharpe Ratio", fontsize=13, fontweight='bold')
ax.axvline(0, color='black', lw=0.8)
ax.grid(True, axis='x', alpha=0.3)

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='black',      label='MV Signal'),
    Patch(facecolor='steelblue',  label='3-Stage'),
    Patch(facecolor='darkorange', label='Asymmetric'),
    Patch(facecolor='purple',     label='Continuous'),
    Patch(facecolor='green',      label='Smoothed'),
    Patch(facecolor='crimson',    label='Persistent'),
    Patch(facecolor='brown',      label='Partial Hedge'),
    Patch(facecolor='grey',       label='Equal Weight'),
    Patch(facecolor='forestgreen',label='Buy & Hold'),
]
ax.legend(handles=legend_elements, fontsize=9, loc='lower right')
plt.tight_layout()
plt.savefig("sharpe_top20.png", dpi=150, bbox_inches='tight')
plt.show()

# ── 18. SAVE ALL ──────────────────────────────────────────────────────────────
print("\n── Saving all results ──")
all_portfolios = {
    **bh_results, **ew_results, **signal_results,
    **asym_results, **cont_results, **smooth_results,
    **persist_results, **partial_results, **mv_results
}

for label, df in all_portfolios.items():
    df.to_csv(f"{label}_portfolio.csv")

all_metrics_df = pd.DataFrame({
    k: {mk: mv for mk, mv in m.items() if mk not in ('Ann_Ret', 'Ann_Vol')}
    for k, m in all_metrics_combined.items()
}).T
all_metrics_df.to_csv("all_portfolio_metrics.csv")

composite_score.to_csv("composite_signal.csv")
for thresh in THRESHOLDS:
    sig_df_t = build_signal_df(composite_score, equity_assets,
                               macro_df.index, threshold=thresh)
    sig_df_t.to_csv(f"signal_df_threshold_{thresh}.csv")

for label, wdf in mv_weight_history.items():
    wdf.to_csv(f"{label}_weights.csv")

print("✓ All files saved.")
print(f"\n  Portfolio CSVs   : {len(all_portfolios)}")
print(f"  Weight CSVs      : {len(mv_weight_history)}")
print(f"  Metrics CSV      : all_portfolio_metrics.csv")
print(f"  Signal CSVs      : {len(THRESHOLDS)} threshold files")
print(f"  Plots saved      : 3 PNG files")


In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# SECTION 0: PARAMETER GRID DEFINITION
# ════════════════════════════════════════════════════════════════════════════════

import itertools
from datetime import datetime

print("\n" + "="*110)
print("PARAMETER STUDY: SIGNAL PORTFOLIO OPTIMIZATION")
print("="*110)

# ── Parameter Grid ──
PARAM_GRID = {
    'threshold': [0.3, 0.6, 0.9],
    'signal_type': ['symmetric', 'asymmetric'],
    'smooth_window': [3, 5, 10],
    'min_days_persist': [5, 10, 20],
    'freq': ['monthly', 'quarterly'],
    'tc_rate': [0.001, 0.005],  # 0.1% and 0.5%
}

# Generate all combinations
param_combinations = list(itertools.product(
    PARAM_GRID['threshold'],
    PARAM_GRID['signal_type'],
    PARAM_GRID['smooth_window'],
    PARAM_GRID['min_days_persist'],
    PARAM_GRID['freq'],
    PARAM_GRID['tc_rate']
))

print(f"Total parameter combinations: {len(param_combinations)}")
print(f"Estimated runtime: {len(param_combinations) * 0.5:.0f} seconds (~{len(param_combinations) * 0.5 / 60:.1f} minutes)\n")

# ════════════════════════════════════════════════════════════════════════════════
# SECTION 1: SIGNAL BUILDERS (CLEANED UP)
# ════════════════════════════════════════════════════════════════════════════════

def classify_signal_symmetric(score, threshold):
    """Binary: +1 if |score| > threshold, else -1"""
    return +1 if abs(score) > threshold else -1

def classify_signal_asymmetric(score, threshold):
    """Asymmetric: +1 if score > threshold, else -1"""
    return +1 if score > threshold else -1

def build_signal_df_symmetric(composite_score, equity_assets, macro_dates, threshold=0.5):
    """Standard symmetric binary signal"""
    binary = composite_score.apply(lambda x: classify_signal_symmetric(x, threshold))
    signal_df = pd.DataFrame(index=macro_dates)
    for asset in equity_assets:
        signal_df[asset] = binary.reindex(macro_dates).ffill().bfill()
    return signal_df

def build_signal_df_asymmetric(composite_score, equity_assets, macro_dates, threshold=1.0):
    """Asymmetric upper-tail only"""
    binary = composite_score.apply(lambda x: classify_signal_asymmetric(x, threshold))
    signal_df = pd.DataFrame(index=macro_dates)
    for asset in equity_assets:
        signal_df[asset] = binary.reindex(macro_dates).ffill().bfill()
    return signal_df

def build_signal_df_smoothed(composite_score, equity_assets, macro_dates,
                             threshold=1.0, smooth_window=5, signal_type='symmetric'):
    """Smoothed signal before thresholding"""
    smoothed = composite_score.rolling(smooth_window, min_periods=1).mean()

    if signal_type == 'symmetric':
        binary = smoothed.apply(lambda x: classify_signal_symmetric(x, threshold))
    else:
        binary = smoothed.apply(lambda x: classify_signal_asymmetric(x, threshold))

    signal_df = pd.DataFrame(index=macro_dates)
    for asset in equity_assets:
        signal_df[asset] = binary.reindex(macro_dates).ffill().bfill()
    return signal_df

def build_signal_df_persistent(composite_score, equity_assets, macro_dates,
                               threshold=1.0, min_days=10, signal_type='symmetric'):
    """Only switch regime after signal holds for min_days consecutive days"""
    if signal_type == 'symmetric':
        raw = composite_score.apply(lambda x: classify_signal_symmetric(x, threshold))
    else:
        raw = composite_score.apply(lambda x: classify_signal_asymmetric(x, threshold))

    confirmed = raw.copy()
    current = int(raw.iloc[0])
    count = 0

    for i in range(1, len(raw)):
        if int(raw.iloc[i]) != current:
            count += 1
            if count >= min_days:
                current = int(raw.iloc[i])
                count = 0
        else:
            count = 0
        confirmed.iloc[i] = current

    signal_df = pd.DataFrame(index=macro_dates)
    for asset in equity_assets:
        signal_df[asset] = confirmed.reindex(macro_dates).ffill().bfill()
    return signal_df

# ════════════════════════════════════════════════════════════════════════════════
# SECTION 2: PARAMETER STUDY EXECUTION
# ════════════════════════════════════════════════════════════════════════════════

study_results = []
failed_combos = []

print("\n" + "="*110)
print("RUNNING PARAMETER STUDY")
print("="*110 + "\n")

start_time = datetime.now()

for combo_idx, (thresh, sig_type, smooth_win, min_days, freq, tc_rate) in enumerate(param_combinations, 1):

    param_label = (f"t{thresh}_sig{sig_type[0]}_sm{smooth_win}_"
                   f"pd{min_days}_{freq}_tc{tc_rate*100:.1f}pct")

    try:
        # Build signal with current parameters
        if smooth_win > 1:
            sig_df = build_signal_df_smoothed(
                composite_score, equity_assets, macro_df.index,
                threshold=thresh, smooth_window=smooth_win,
                signal_type=sig_type
            )
        else:
            # Use persistent signal with current parameters
            sig_df = build_signal_df_persistent(
                composite_score, equity_assets, macro_df.index,
                threshold=thresh, min_days=min_days,
                signal_type=sig_type
            )

        # Run portfolio backtest
        def wfn(sig_row, date, _eq=equity_assets, _ne=non_equity_assets,
               _bw=base_weight, _sh=EQUITY_SAFE_HAVEN_MAP):
            return build_weights_3stage(sig_row, _eq, _ne, _bw, date, _sh)

        pnl_df = run_signal_portfolio(
            returns, sig_df, equity_assets, non_equity_assets,
            ALL_ASSETS, weight_fn=wfn,
            freq=freq, tc_rate=tc_rate, initial_capital=1e9
        )

        # Calculate metrics
        pv = pnl_df['portfolio_value']
        total_ret = (pv.iloc[-1] / pv.iloc[0]) - 1
        days = (pv.index[-1] - pv.index[0]).days
        cagr = (pv.iloc[-1] / pv.iloc[0]) ** (365.25 / days) - 1

        daily_returns = pv.pct_change().dropna()
        ann_vol = daily_returns.std() * np.sqrt(252)
        sharpe = cagr / ann_vol if ann_vol > 0 else 0

        # Maximum drawdown
        cum_ret = (pv / pv.iloc[0]) - 1
        running_max = cum_ret.expanding().max()
        drawdown_series = (cum_ret - running_max) / (1 + running_max)
        max_dd = drawdown_series.min()

        calmar = cagr / abs(max_dd) if max_dd != 0 else 0

        # Store results
        study_results.append({
            'param_label': param_label,
            'threshold': thresh,
            'signal_type': sig_type,
            'smooth_window': smooth_win,
            'min_days_persist': min_days,
            'frequency': freq,
            'tc_rate': tc_rate,
            'CAGR': cagr,
            'Volatility': ann_vol,
            'Sharpe': sharpe,
            'Max_Drawdown': max_dd,
            'Calmar': calmar,
            'Total_Return': total_ret,
            'Final_Value': pv.iloc[-1],
            'Status': 'Success'
        })

        # Progress indicator
        if combo_idx % 10 == 0 or combo_idx == 1:
            elapsed = (datetime.now() - start_time).total_seconds()
            rate = combo_idx / (elapsed / 60)
            remaining_mins = (len(param_combinations) - combo_idx) / rate
            print(f"[{combo_idx:3d}/{len(param_combinations)}] {param_label:50s} | "
                  f"Sharpe: {sharpe:6.3f} | DD: {max_dd*100:6.2f}% | "
                  f"ETA: {remaining_mins:.0f} min")

    except Exception as e:
        failed_combos.append({
            'param_label': param_label,
            'error': str(e)
        })
        study_results.append({
            'param_label': param_label,
            'threshold': thresh,
            'signal_type': sig_type,
            'smooth_window': smooth_win,
            'min_days_persist': min_days,
            'frequency': freq,
            'tc_rate': tc_rate,
            'Status': f'Failed: {str(e)[:50]}'
        })

elapsed_time = (datetime.now() - start_time).total_seconds()

print("\n" + "="*110)
print(f"PARAMETER STUDY COMPLETE")
print("="*110)
print(f"Total time: {elapsed_time/60:.1f} minutes")
print(f"Successful: {len([r for r in study_results if r['Status'] == 'Success'])}")
print(f"Failed: {len(failed_combos)}\n")

# ════════════════════════════════════════════════════════════════════════════════
# SECTION 3: ANALYSIS OF RESULTS
# ════════════════════════════════════════════════════════════════════════════════

# Convert to DataFrame
study_df = pd.DataFrame(study_results)
study_df_success = study_df[study_df['Status'] == 'Success'].copy()

if len(study_df_success) > 0:

    print("\n" + "="*110)
    print("TOP 10 STRATEGIES BY SHARPE RATIO")
    print("="*110)

    top_sharpe = study_df_success.nlargest(10, 'Sharpe')[
        ['param_label', 'threshold', 'signal_type', 'smooth_window',
         'min_days_persist', 'frequency', 'tc_rate', 'CAGR', 'Volatility',
         'Sharpe', 'Max_Drawdown', 'Calmar']
    ]

    print(top_sharpe.to_string(index=False))

    print("\n" + "="*110)
    print("TOP 10 STRATEGIES BY MAX DRAWDOWN (Best Protection)")
    print("="*110)

    top_dd = study_df_success.nsmallest(10, 'Max_Drawdown')[
        ['param_label', 'threshold', 'signal_type', 'smooth_window',
         'min_days_persist', 'frequency', 'tc_rate', 'CAGR', 'Volatility',
         'Sharpe', 'Max_Drawdown', 'Calmar']
    ]

    print(top_dd.to_string(index=False))

    print("\n" + "="*110)
    print("TOP 10 STRATEGIES BY CALMAR RATIO (Risk-Adjusted)")
    print("="*110)

    top_calmar = study_df_success.nlargest(10, 'Calmar')[
        ['param_label', 'threshold', 'signal_type', 'smooth_window',
         'min_days_persist', 'frequency', 'tc_rate', 'CAGR', 'Volatility',
         'Sharpe', 'Max_Drawdown', 'Calmar']
    ]

    print(top_calmar.to_string(index=False))

    # ── Parameter Sensitivity Analysis ──
    print("\n" + "="*110)
    print("PARAMETER SENSITIVITY ANALYSIS")
    print("="*110)

    print("\n── BY THRESHOLD ──")
    thresh_analysis = study_df_success.groupby('threshold').agg({
        'Sharpe': ['mean', 'std', 'max'],
        'Max_Drawdown': ['mean', 'min'],
        'CAGR': 'mean'
    }).round(4)
    print(thresh_analysis)

    print("\n── BY SIGNAL TYPE ──")
    sigtype_analysis = study_df_success.groupby('signal_type').agg({
        'Sharpe': ['mean', 'std', 'max'],
        'Max_Drawdown': ['mean', 'min'],
        'CAGR': 'mean'
    }).round(4)
    print(sigtype_analysis)

    print("\n── BY SMOOTH WINDOW ──")
    smooth_analysis = study_df_success.groupby('smooth_window').agg({
        'Sharpe': ['mean', 'std', 'max'],
        'Max_Drawdown': ['mean', 'min'],
        'CAGR': 'mean'
    }).round(4)
    print(smooth_analysis)

    print("\n── BY MIN DAYS PERSIST ──")
    mindays_analysis = study_df_success.groupby('min_days_persist').agg({
        'Sharpe': ['mean', 'std', 'max'],
        'Max_Drawdown': ['mean', 'min'],
        'CAGR': 'mean'
    }).round(4)
    print(mindays_analysis)

    print("\n── BY FREQUENCY ──")
    freq_analysis = study_df_success.groupby('frequency').agg({
        'Sharpe': ['mean', 'std', 'max'],
        'Max_Drawdown': ['mean', 'min'],
        'CAGR': 'mean'
    }).round(4)
    print(freq_analysis)

    print("\n── BY TRANSACTION COST ──")
    tc_analysis = study_df_success.groupby('tc_rate').agg({
        'Sharpe': ['mean', 'std', 'max'],
        'Max_Drawdown': ['mean', 'min'],
        'CAGR': 'mean'
    }).round(4)
    print(tc_analysis)

    # ════════════════════════════════════════════════════════════════════════════════
    # SECTION 4: VISUALIZATION
    # ════════════════════════════════════════════════════════════════════════════════

    print("\n" + "="*110)
    print("GENERATING PARAMETER STUDY VISUALIZATIONS")
    print("="*110 + "\n")

    # ── Plot 1: Sharpe vs Drawdown by Signal Type ──
    fig, ax = plt.subplots(figsize=(12, 8))

    for sig_type in study_df_success['signal_type'].unique():
        subset = study_df_success[study_df_success['signal_type'] == sig_type]
        ax.scatter(subset['Max_Drawdown'] * 100, subset['Sharpe'],
                  s=100, alpha=0.6, label=sig_type)

    ax.set_xlabel("Max Drawdown (%)", fontsize=12)
    ax.set_ylabel("Sharpe Ratio", fontsize=12)
    ax.set_title("Parameter Study: Sharpe vs Drawdown by Signal Type", fontsize=13, fontweight='bold')
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig("param_study_01_sharpe_vs_dd.png", dpi=150, bbox_inches='tight')
    print("✓ Saved: param_study_01_sharpe_vs_dd.png")
    plt.show()

    # ── Plot 2: Sharpe by Threshold ──
    fig, ax = plt.subplots(figsize=(12, 6))

    for sig_type in study_df_success['signal_type'].unique():
        subset = study_df_success[study_df_success['signal_type'] == sig_type]
        by_thresh = subset.groupby('threshold')['Sharpe'].agg(['mean', 'std'])
        ax.errorbar(by_thresh.index, by_thresh['mean'], yerr=by_thresh['std'],
                   marker='o', label=sig_type, capsize=5, markersize=8)

    ax.set_xlabel("Threshold", fontsize=12)
    ax.set_ylabel("Sharpe Ratio", fontsize=12)
    ax.set_title("Parameter Study: Sharpe Ratio by Threshold", fontsize=13, fontweight='bold')
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig("param_study_02_threshold_sensitivity.png", dpi=150, bbox_inches='tight')
    print("✓ Saved: param_study_02_threshold_sensitivity.png")
    plt.show()

    # ── Plot 3: Smooth Window Impact ──
    fig, ax = plt.subplots(figsize=(12, 6))

    by_smooth = study_df_success.groupby('smooth_window')[['Sharpe', 'Max_Drawdown']].mean()
    ax_twin = ax.twinx()

    ax.plot(by_smooth.index, by_smooth['Sharpe'], marker='o', color='green',
           linewidth=2, markersize=8, label='Sharpe (left)')
    ax_twin.plot(by_smooth.index, by_smooth['Max_Drawdown'] * 100, marker='s',
                color='red', linewidth=2, markersize=8, label='Max DD (right)')

    ax.set_xlabel("Smooth Window (days)", fontsize=12)
    ax.set_ylabel("Sharpe Ratio", fontsize=12, color='green')
    ax_twin.set_ylabel("Max Drawdown (%)", fontsize=12, color='red')
    ax.set_title("Parameter Study: Smoothing Window Impact", fontsize=13, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.legend(loc='upper left', fontsize=11)
    ax_twin.legend(loc='upper right', fontsize=11)
    plt.tight_layout()
    plt.savefig("param_study_03_smooth_sensitivity.png", dpi=150, bbox_inches='tight')
    print("✓ Saved: param_study_03_smooth_sensitivity.png")
    plt.show()

    # ── Plot 4: Frequency & TC Impact ──
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))

    # Frequency
    by_freq = study_df_success.groupby('frequency')['Sharpe'].agg(['mean', 'std', 'count'])
    axes[0].bar(by_freq.index, by_freq['mean'], yerr=by_freq['std'],
               capsize=5, alpha=0.7, color=['#1f77b4', '#ff7f0e'])
    axes[0].set_ylabel("Sharpe Ratio", fontsize=11)
    axes[0].set_title("Sharpe by Rebalance Frequency", fontsize=12, fontweight='bold')
    axes[0].grid(True, alpha=0.3, axis='y')

    # Transaction Cost
    by_tc = study_df_success.groupby('tc_rate')['Sharpe'].agg(['mean', 'std'])
    tc_labels = [f"{tc*100:.1f}%" for tc in by_tc.index]
    axes[1].bar(tc_labels, by_tc['mean'], yerr=by_tc['std'],
               capsize=5, alpha=0.7, color=['#2ca02c', '#d62728'])
    axes[1].set_ylabel("Sharpe Ratio", fontsize=11)
    axes[1].set_title("Sharpe by Transaction Cost", fontsize=12, fontweight='bold')
    axes[1].grid(True, alpha=0.3, axis='y')

    plt.tight_layout()
    plt.savefig("param_study_04_freq_tc_impact.png", dpi=150, bbox_inches='tight')
    print("✓ Saved: param_study_04_freq_tc_impact.png")
    plt.show()

    # ════════════════════════════════════════════════════════════════════════════════
    # SECTION 5: SAVE RESULTS
    # ════════════════════════════════════════════════════════════════════════════════

    study_df.to_csv("parameter_study_full_results.csv", index=False)
    study_df_success.to_csv("parameter_study_successful_results.csv", index=False)

    print("\n" + "="*110)
    print("RESULTS SAVED")
    print("="*110)
    print("✓ Saved: parameter_study_full_results.csv")
    print("✓ Saved: parameter_study_successful_results.csv")

    # ── Summary Statistics ──
    print("\n" + "="*110)


In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# SECTION 6: PLOT TOP 5 PARAMETER COMBINATIONS
# ════════════════════════════════════════════════════════════════════════════════

print("\n" + "="*110)
print("PLOTTING TOP 5 PARAMETER COMBINATIONS")
print("="*110 + "\n")

# Get top 5 by Sharpe Ratio
top_5_sharpe = study_df_success.nlargest(5, 'Sharpe')

print("Top 5 Combinations by Sharpe Ratio:")
print("-" * 110)
for idx, (i, row) in enumerate(top_5_sharpe.iterrows(), 1):
    print(f"{idx}. {row['param_label']}")
    print(f"   Sharpe: {row['Sharpe']:.3f} | CAGR: {row['CAGR']*100:.2f}% | "
          f"Vol: {row['Volatility']*100:.2f}% | Max DD: {row['Max_Drawdown']*100:.2f}% | "
          f"Calmar: {row['Calmar']:.3f}")
print()

# Reconstruct and run the top 5 portfolios to get their time series
top_5_results = {}

for idx, (i, row) in enumerate(top_5_sharpe.iterrows(), 1):

    thresh = row['threshold']
    sig_type = row['signal_type']
    smooth_win = row['smooth_window']
    min_days = row['min_days_persist']
    freq = row['frequency']
    tc_rate = row['tc_rate']

    param_label = row['param_label']

    print(f"[{idx}/5] Reconstructing: {param_label}...", end=" ", flush=True)

    try:
        # Build signal
        if smooth_win > 1:
            sig_df = build_signal_df_smoothed(
                composite_score, equity_assets, macro_df.index,
                threshold=thresh, smooth_window=smooth_win,
                signal_type=sig_type
            )
        else:
            sig_df = build_signal_df_persistent(
                composite_score, equity_assets, macro_df.index,
                threshold=thresh, min_days=min_days,
                signal_type=sig_type
            )

        # Run portfolio
        def wfn(sig_row, date, _eq=equity_assets, _ne=non_equity_assets,
               _bw=base_weight, _sh=EQUITY_SAFE_HAVEN_MAP):
            return build_weights_3stage(sig_row, _eq, _ne, _bw, date, _sh)

        pnl_df = run_signal_portfolio(
            returns, sig_df, equity_assets, non_equity_assets,
            ALL_ASSETS, weight_fn=wfn,
            freq=freq, tc_rate=tc_rate, initial_capital=1e9
        )

        top_5_results[param_label] = pnl_df
        print("✓")

    except Exception as e:
        print(f"✗ Error: {str(e)[:50]}")

print()

# ── Plot 1: Portfolio Values Over Time ──
fig, ax = plt.subplots(figsize=(16, 9))

colors_top5 = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']

for (param_label, pnl_df), color in zip(top_5_results.items(), colors_top5):
    pv = pnl_df['portfolio_value'] / 1e9  # Convert to billions
    ax.plot(pv.index, pv.values, linewidth=2.5, label=param_label, color=color)

ax.axhline(1.0, color='black', linestyle=':', linewidth=1, alpha=0.5, label='Initial Capital')
ax.set_xlabel("Date", fontsize=12, fontweight='bold')
ax.set_ylabel("Portfolio Value ($B)", fontsize=12, fontweight='bold')
ax.set_title("Top 5 Parameter Combinations: Portfolio Value Over Time",
            fontsize=14, fontweight='bold')
ax.legend(fontsize=10, loc='upper left', framealpha=0.95)
ax.grid(True, alpha=0.3)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))

plt.tight_layout()
plt.savefig("top5_01_portfolio_values.png", dpi=150, bbox_inches='tight')
print("✓ Saved: top5_01_portfolio_values.png")
plt.show()

# ── Plot 2: Cumulative Returns ──
fig, ax = plt.subplots(figsize=(16, 9))

for (param_label, pnl_df), color in zip(top_5_results.items(), colors_top5):
    pv = pnl_df['portfolio_value']
    cum_ret = (pv / pv.iloc[0] - 1) * 100
    ax.plot(cum_ret.index, cum_ret.values, linewidth=2.5, label=param_label, color=color)

ax.axhline(0, color='black', linestyle='-', linewidth=1, alpha=0.5)
ax.set_xlabel("Date", fontsize=12, fontweight='bold')
ax.set_ylabel("Cumulative Return (%)", fontsize=12, fontweight='bold')
ax.set_title("Top 5 Parameter Combinations: Cumulative Returns",
            fontsize=14, fontweight='bold')
ax.legend(fontsize=10, loc='upper left', framealpha=0.95)
ax.grid(True, alpha=0.3)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.0f}%"))

plt.tight_layout()
plt.savefig("top5_02_cumulative_returns.png", dpi=150, bbox_inches='tight')
print("✓ Saved: top5_02_cumulative_returns.png")
plt.show()

# ── Plot 3: Drawdown Series ──
fig, axes = plt.subplots(5, 1, figsize=(16, 14))

for ax, (param_label, pnl_df), color in zip(axes, top_5_results.items(), colors_top5):
    pv = pnl_df['portfolio_value']
    cum_ret = (pv / pv.iloc[0]) - 1
    running_max = cum_ret.expanding().max()
    drawdown = (cum_ret - running_max) / (1 + running_max) * 100

    ax.fill_between(drawdown.index, drawdown.values, 0, alpha=0.5, color=color)
    ax.plot(drawdown.index, drawdown.values, linewidth=1.5, color=color)

    max_dd = drawdown.min()
    ax.set_ylabel("Drawdown (%)", fontsize=10, fontweight='bold')
    ax.set_title(f"{param_label} | Max DD: {max_dd:.2f}%",
                fontweight='bold', fontsize=11)
    ax.grid(True, alpha=0.3)
    ax.axhline(0, color='black', linewidth=0.5)

axes[-1].set_xlabel("Date", fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig("top5_03_drawdowns.png", dpi=150, bbox_inches='tight')
print("✓ Saved: top5_03_drawdowns.png")
plt.show()

# ── Plot 4: Rolling Volatility (252-day) ──
fig, ax = plt.subplots(figsize=(16, 9))

for (param_label, pnl_df), color in zip(top_5_results.items(), colors_top5):
    pv = pnl_df['portfolio_value']
    daily_ret = pv.pct_change().dropna()
    rolling_vol = daily_ret.rolling(252).std() * np.sqrt(252) * 100

    ax.plot(rolling_vol.index, rolling_vol.values, linewidth=2,
           label=param_label, color=color, alpha=0.8)

ax.set_xlabel("Date", fontsize=12, fontweight='bold')
ax.set_ylabel("Rolling 252-Day Volatility (%)", fontsize=12, fontweight='bold')
ax.set_title("Top 5 Parameter Combinations: Rolling Volatility",
            fontsize=14, fontweight='bold')
ax.legend(fontsize=10, loc='best', framealpha=0.95)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("top5_04_rolling_volatility.png", dpi=150, bbox_inches='tight')
print("✓ Saved: top5_04_rolling_volatility.png")
plt.show()

# ── Plot 5: Risk-Return Scatter (Efficient Frontier) ──
fig, ax = plt.subplots(figsize=(12, 9))

for (param_label, pnl_df), color in zip(top_5_results.items(), colors_top5):
    pv = pnl_df['portfolio_value']

    # Calculate metrics
    total_ret = (pv.iloc[-1] / pv.iloc[0]) - 1
    days = (pv.index[-1] - pv.index[0]).days
    cagr = (pv.iloc[-1] / pv.iloc[0]) ** (365.25 / days) - 1

    daily_ret = pv.pct_change().dropna()
    ann_vol = daily_ret.std() * np.sqrt(252)

    sharpe = cagr / ann_vol if ann_vol > 0 else 0

    ax.scatter(ann_vol * 100, cagr * 100, s=400, alpha=0.7,
              color=color, edgecolors='black', linewidth=2)
    ax.annotate(param_label.split('_')[0:3],
               (ann_vol * 100, cagr * 100),
               fontsize=9, ha='center', va='bottom', fontweight='bold')

ax.set_xlabel("Volatility (%)", fontsize=12, fontweight='bold')
ax.set_ylabel("CAGR (%)", fontsize=12, fontweight='bold')
ax.set_title("Top 5 Parameter Combinations: Risk-Return Profile",
            fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("top5_05_risk_return.png", dpi=150, bbox_inches='tight')
print("✓ Saved: top5_05_risk_return.png")
plt.show()

# ── Plot 6: Daily P&L Distribution ──
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for ax, (param_label, pnl_df), color in zip(axes.flat[:5], top_5_results.items(), colors_top5):
    pv = pnl_df['portfolio_value']
    daily_ret = pv.pct_change().dropna() * 100

    ax.hist(daily_ret.values, bins=50, alpha=0.7, color=color, edgecolor='black')
    ax.axvline(daily_ret.mean(), color='red', linestyle='--', linewidth=2, label=f"Mean: {daily_ret.mean():.2f}%")
    ax.axvline(daily_ret.std(), color='green', linestyle='--', linewidth=2, label=f"Std: {daily_ret.std():.2f}%")

    ax.set_xlabel("Daily Return (%)", fontsize=10)
    ax.set_ylabel("Frequency", fontsize=10)
    ax.set_title(f"{param_label[:40]}", fontsize=10, fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3, axis='y')

# Hide the 6th subplot
axes.flat[5].axis('off')

plt.tight_layout()
plt.savefig("top5_06_daily_returns_dist.png", dpi=150, bbox_inches='tight')
print("✓ Saved: top5_06_daily_returns_dist.png")
plt.show()

# ── Plot 7: Correlation Heatmap of Returns ──
fig, ax = plt.subplots(figsize=(10, 8))

# Get returns for all top 5
returns_dict = {}
for param_label, pnl_df in top_5_results.items():
    pv = pnl_df['portfolio_value']
    returns_dict[param_label[:30]] = pv.pct_change().dropna() * 100

returns_combined = pd.concat(returns_dict, axis=1)
corr_matrix = returns_combined.corr()

im = ax.imshow(corr_matrix, cmap='RdYlGn', aspect='auto', vmin=-1, vmax=1)
ax.set_xticks(range(len(corr_matrix)))
ax.set_yticks(range(len(corr_matrix)))
ax.set_xticklabels([label[:20] for label in corr_matrix.columns], rotation=45, ha='right', fontsize=9)
ax.set_yticklabels([label[:20] for label in corr_matrix.index], fontsize=9)

# Add correlation values
for i in range(len(corr_matrix)):
    for j in range(len(corr_matrix)):
        text = ax.text(j, i, f'{corr_matrix.iloc[i, j]:.2f}',
                      ha="center", va="center", color="black", fontsize=9, fontweight='bold')

ax.set_title("Correlation Matrix: Top 5 Parameter Combinations",
            fontsize=13, fontweight='bold', pad=20)
plt.colorbar(im, ax=ax, label='Correlation')

plt.tight_layout()
plt.savefig("top5_07_correlation_heatmap.png", dpi=150, bbox_inches='tight')
print("✓ Saved: top5_07_correlation_heatmap.png")
plt.show()

# ── Summary Table ──
print("\n" + "="*110)
print("TOP 5 PARAMETER COMBINATIONS - DETAILED METRICS")
print("="*110 + "\n")

summary_data = []
for param_label, pnl_df in top_5_results.items():
    pv = pnl_df['portfolio_value']

    total_ret = (pv.iloc[-1] / pv.iloc[0]) - 1
    days = (pv.index[-1] - pv.index[0]).days
    cagr = (pv.iloc[-1] / pv.iloc[0]) ** (365.25 / days) - 1

    daily_ret = pv.pct_change().dropna()
    ann_vol = daily_ret.std() * np.sqrt(252)

    sharpe = cagr / ann_vol if ann_vol > 0 else 0

    cum_ret = (pv / pv.iloc[0]) - 1
    running_max = cum_ret.expanding().max()
    drawdown_series = (cum_ret - running_max) / (1 + running_max)
    max_dd = drawdown_series.min()

    calmar = cagr / abs(max_dd) if max_dd != 0 else 0

    # Count trades (rebalances)
    trades = (pnl_df['daily_pnl'].abs() > 0).sum()

    summary_data.append({
        'Strategy': param_label[:45],
        'CAGR (%)': f"{cagr*100:.2f}",
        'Vol (%)': f"{ann_vol*100:.2f}",
        'Sharpe': f"{sharpe:.3f}",
        'Max DD (%)': f"{max_dd*100:.2f}",
        'Calmar': f"{calmar:.3f}",
        'Final Value ($B)': f"{pv.iloc[-1]/1e9:.3f}",
    })

summary_df = pd.DataFrame(summary_data)
print(summary_df.to_string(index=False))

# Save summary
summary_df.to_csv("top5_summary.csv", index=False)
print("\n✓ Saved: top5_summary.csv")

print("\n" + "="*110)
print("TOP 5 VISUALIZATIONS COMPLETE")
print("="*110)


In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# SECTION 0: COMPREHENSIVE PARAMETER GRID DEFINITION
# ════════════════════════════════════════════════════════════════════════════════

import itertools
from datetime import datetime

print("\n" + "="*110)
print("COMPREHENSIVE PARAMETER STUDY: SIGNAL PORTFOLIO OPTIMIZATION")
print("="*110)

# ── DETERMINANT COMPUTATION PARAMETERS ──
DET_WINDOW_CONFIGS = [
    {3: 0.2, 7: 0.3, 28: 0.5},      # Original balanced
    {3: 0.4, 7: 0.3, 28: 0.3},      # Short-term weighted
    {7: 0.2, 28: 0.8},               # Long-term focused
    {3: 0.33, 7: 0.33, 28: 0.34},   # Equal weight
]

DET_TYPES = ['standard', 'downside', 'tail']  # Which determinant to use

# ── SIGNAL PARAMETERS ──
PARAM_GRID = {
    'threshold': [0.3, 0.6, 0.9, 1.2],
    'signal_type': ['symmetric', 'asymmetric'],
    'smooth_window': [1, 3, 5, 10],  # 1 = no smoothing
    'min_days_persist': [1, 5, 10, 20],  # 1 = no persistence
    'freq': ['monthly', 'quarterly'],
    'tc_rate': [0.001, 0.005],  # 0.1% and 0.5%
}

# Generate all combinations
det_window_combos = DET_WINDOW_CONFIGS
det_type_combos = DET_TYPES
signal_combos = list(itertools.product(
    PARAM_GRID['threshold'],
    PARAM_GRID['signal_type'],
    PARAM_GRID['smooth_window'],
    PARAM_GRID['min_days_persist'],
    PARAM_GRID['freq'],
    PARAM_GRID['tc_rate']
))

total_combos = len(det_window_combos) * len(det_type_combos) * len(signal_combos)

print(f"Determinant window configs: {len(det_window_combos)}")
print(f"Determinant types: {len(det_type_combos)}")
print(f"Signal parameter combos: {len(signal_combos)}")
print(f"Total parameter combinations: {total_combos}")
print(f"Estimated runtime: {total_combos * 0.3:.0f} seconds (~{total_combos * 0.3 / 60:.1f} minutes)\n")

# ════════════════════════════════════════════════════════════════════════════════
# SECTION 1: RECOMPUTABLE DETERMINANT FUNCTIONS
# ════════════════════════════════════════════════════════════════════════════════

def compute_determinant_standard(price_series_dict, windows=[3, 7, 28]):
    """Compute standard correlation-based determinant"""
    prices = pd.DataFrame(price_series_dict).sort_index()
    rets = prices.pct_change().dropna()

    det_results = {}

    for window in windows:
        det_series = []

        for i in range(window, len(rets)):
            window_rets = rets.iloc[i - window:i]
            valid_cols = window_rets.columns[window_rets.std() > 1e-10]

            if len(valid_cols) < 2:
                det_series.append(np.nan)
                continue

            try:
                w = window_rets[valid_cols]
                R = w.corr().values
                I = np.eye(len(R))
                M = R - I
                det = np.linalg.det(M)
            except:
                det = np.nan

            det_series.append(det)

        det_results[window] = pd.Series(
            det_series,
            index=rets.index[window:],
            name=f'det_{window}d'
        )

    return det_results

def compute_determinant_downside(price_series_dict, windows=[3, 7, 28]):
    """Compute downside-only determinant (negative returns only)"""
    prices = pd.DataFrame(price_series_dict).sort_index()
    rets = prices.pct_change().dropna()

    det_results = {}

    for window in windows:
        det_series = []

        for i in range(window, len(rets)):
            window_rets = rets.iloc[i - window:i]
            down_rets = window_rets[window_rets.mean(axis=1) < 0]

            if len(down_rets) < 2:
                det_series.append(np.nan)
                continue

            valid_cols = down_rets.columns[down_rets.std() > 1e-10]

            if len(valid_cols) < 2:
                det_series.append(np.nan)
                continue

            try:
                w = down_rets[valid_cols]
                R = w.corr().values
                I = np.eye(len(R))
                M = R - I
                det = np.linalg.det(M)
            except:
                det = np.nan

            det_series.append(det)

        det_results[window] = pd.Series(
            det_series,
            index=rets.index[window:],
            name=f'det_down_{window}d'
        )

    return det_results

def compute_determinant_tail(price_series_dict, windows=[3, 7, 28], tail_percentile=10):
    """Compute tail-only determinant (extreme stress days)"""
    prices = pd.DataFrame(price_series_dict).sort_index()
    rets = prices.pct_change().dropna()

    agg_ret = rets.mean(axis=1)
    tail_threshold = agg_ret.quantile(tail_percentile / 100)
    tail_days = agg_ret < tail_threshold

    det_results = {}

    for window in windows:
        det_series = []

        for i in range(window, len(rets)):
            window_rets = rets.iloc[i - window:i]
            tail_window = tail_days.iloc[i - window:i]

            if tail_window.sum() < 1:
                det_series.append(np.nan)
                continue

            tail_rets = window_rets[tail_window]
            valid_cols = tail_rets.columns[tail_rets.std() > 1e-10]

            if len(valid_cols) < 2:
                det_series.append(np.nan)
                continue

            try:
                w = tail_rets[valid_cols]
                R = w.corr().values
                I = np.eye(len(R))
                M = R - I
                det = np.linalg.det(M)
            except:
                det = np.nan

            det_series.append(det)

        det_results[window] = pd.Series(
            det_series,
            index=rets.index[window:],
            name=f'det_tail_{window}d'
        )

    return det_results

def compute_composite_score_from_dets(det_results, window_weights):
    """Compute composite z-score from determinant results"""
    all_series = []
    for window, series in det_results.items():
        mu = series.rolling(252, min_periods=60).mean()
        sigma = series.rolling(252, min_periods=60).std()
        z = (series - mu) / sigma.replace(0, np.nan)
        z.name = f'z_{window}d'
        all_series.append(z)

    z_df = pd.concat(all_series, axis=1).dropna()

    # Blend using provided weights
    composite = pd.Series(0, index=z_df.index)
    for window, weight in window_weights.items():
        col_name = f'z_{window}d'
        if col_name in z_df.columns:
            composite += z_df[col_name] * weight

    composite = composite / sum(window_weights.values())
    return composite

# ════════════════════════════════════════════════════════════════════════════════
# SECTION 2: SIGNAL BUILDERS (UNCHANGED)
# ════════════════════════════════════════════════════════════════════════════════

def classify_signal_symmetric(score, threshold):
    """Binary: +1 if |score| > threshold, else -1"""
    return +1 if abs(score) > threshold else -1

def classify_signal_asymmetric(score, threshold):
    """Asymmetric: +1 if score > threshold, else -1"""
    return +1 if score > threshold else -1

def build_signal_df_smoothed(composite_score, equity_assets, macro_dates,
                             threshold=1.0, smooth_window=5, signal_type='symmetric'):
    """Smoothed signal before thresholding"""
    if smooth_window <= 1:
        smoothed = composite_score
    else:
        smoothed = composite_score.rolling(smooth_window, min_periods=1).mean()

    if signal_type == 'symmetric':
        binary = smoothed.apply(lambda x: classify_signal_symmetric(x, threshold))
    else:
        binary = smoothed.apply(lambda x: classify_signal_asymmetric(x, threshold))

    signal_df = pd.DataFrame(index=macro_dates)
    for asset in equity_assets:
        signal_df[asset] = binary.reindex(macro_dates).ffill().bfill()
    return signal_df

def build_signal_df_persistent(composite_score, equity_assets, macro_dates,
                               threshold=1.0, min_days=10, signal_type='symmetric'):
    """Only switch regime after signal holds for min_days consecutive days"""
    if signal_type == 'symmetric':
        raw = composite_score.apply(lambda x: classify_signal_symmetric(x, threshold))
    else:
        raw = composite_score.apply(lambda x: classify_signal_asymmetric(x, threshold))

    if min_days <= 1:
        confirmed = raw.copy()
    else:
        confirmed = raw.copy()
        current = int(raw.iloc[0])
        count = 0

        for i in range(1, len(raw)):
            if int(raw.iloc[i]) != current:
                count += 1
                if count >= min_days:
                    current = int(raw.iloc[i])
                    count = 0
            else:
                count = 0
            confirmed.iloc[i] = current

    signal_df = pd.DataFrame(index=macro_dates)
    for asset in equity_assets:
        signal_df[asset] = confirmed.reindex(macro_dates).ffill().bfill()
    return signal_df

def build_signal(composite_score, equity_assets, macro_dates,
                threshold=1.0, smooth_window=1, min_days=1, signal_type='symmetric'):
    """Build signal with both smoothing and persistence"""

    # Apply smoothing first
    if smooth_window > 1:
        smoothed = composite_score.rolling(smooth_window, min_periods=1).mean()
    else:
        smoothed = composite_score

    # Then apply persistence
    if signal_type == 'symmetric':
        raw = smoothed.apply(lambda x: classify_signal_symmetric(x, threshold))
    else:
        raw = smoothed.apply(lambda x: classify_signal_asymmetric(x, threshold))

    if min_days <= 1:
        confirmed = raw.copy()
    else:
        confirmed = raw.copy()
        current = int(raw.iloc[0])
        count = 0

        for i in range(1, len(raw)):
            if int(raw.iloc[i]) != current:
                count += 1
                if count >= min_days:
                    current = int(raw.iloc[i])
                    count = 0
            else:
                count = 0
            confirmed.iloc[i] = current

    signal_df = pd.DataFrame(index=macro_dates)
    for asset in equity_assets:
        signal_df[asset] = confirmed.reindex(macro_dates).ffill().bfill()
    return signal_df

# ════════════════════════════════════════════════════════════════════════════════
# SECTION 3: PARAMETER STUDY EXECUTION
# ════════════════════════════════════════════════════════════════════════════════

study_results = []
failed_combos = []

print("\n" + "="*110)
print("RUNNING COMPREHENSIVE PARAMETER STUDY")
print("="*110 + "\n")

start_time = datetime.now()
combo_counter = 0

for det_window_config in det_window_combos:

    for det_type in det_type_combos:

        # Compute determinants once per (window_config, det_type) combo
        print(f"\n[Computing determinants] Windows: {list(det_window_config.keys())}, "
              f"Type: {det_type}...", flush=True)

        try:
            if det_type == 'standard':
                det_results = compute_determinant_standard(index_prices,
                                                          windows=list(det_window_config.keys()))
            elif det_type == 'downside':
                det_results = compute_determinant_downside(index_prices,
                                                          windows=list(det_window_config.keys()))
            else:  # tail
                det_results = compute_determinant_tail(index_prices,
                                                      windows=list(det_window_config.keys()))

            composite = compute_composite_score_from_dets(det_results, det_window_config)
            print(f"  ✓ Composite score computed: {composite.shape}")

        except Exception as e:
            print(f"  ✗ Failed to compute determinants: {str(e)[:60]}")
            continue

        # Now test all signal parameter combinations
        for (thresh, sig_type, smooth_win, min_days, freq, tc_rate) in signal_combos:

            combo_counter += 1

            param_label = (f"det{det_type[0]}_w{list(det_window_config.keys())}_"
                          f"t{thresh}_sig{sig_type[0]}_sm{smooth_win}_"
                          f"pd{min_days}_{freq}_tc{tc_rate*100:.1f}pct")

            try:
                # Build signal
                sig_df = build_signal(
                    composite, equity_assets, macro_df.index,
                    threshold=thresh, smooth_window=smooth_win,
                    min_days=min_days, signal_type=sig_type
                )

                # Run portfolio backtest
                def wfn(sig_row, date, _eq=equity_assets, _ne=non_equity_assets,
                       _bw=base_weight, _sh=EQUITY_SAFE_HAVEN_MAP):
                    return build_weights_3stage(sig_row, _eq, _ne, _bw, date, _sh)

                pnl_df = run_signal_portfolio(
                    returns, sig_df, equity_assets, non_equity_assets,
                    ALL_ASSETS, weight_fn=wfn,
                    freq=freq, tc_rate=tc_rate, initial_capital=1e9
                )

                # Calculate metrics
                pv = pnl_df['portfolio_value']
                total_ret = (pv.iloc[-1] / pv.iloc[0]) - 1
                days = (pv.index[-1] - pv.index[0]).days
                cagr = (pv.iloc[-1] / pv.iloc[0]) ** (365.25 / days) - 1

                daily_returns = pv.pct_change().dropna()
                ann_vol = daily_returns.std() * np.sqrt(252)
                sharpe = cagr / ann_vol if ann_vol > 0 else 0

                # Maximum drawdown
                cum_ret = (pv / pv.iloc[0]) - 1
                running_max = cum_ret.expanding().max()
                drawdown_series = (cum_ret - running_max) / (1 + running_max)
                max_dd = drawdown_series.min()

                calmar = cagr / abs(max_dd) if max_dd != 0 else 0

                # Store results
                study_results.append({
                    'param_label': param_label,
                    'det_type': det_type,
                    'det_windows': str(list(det_window_config.keys())),
                    'window_weights': str(det_window_config),
                    'threshold': thresh,
                    'signal_type': sig_type,
                    'smooth_window': smooth_win,
                    'min_days_persist': min_days,
                    'frequency': freq,
                    'tc_rate': tc_rate,
                    'CAGR': cagr,
                    'Volatility': ann_vol,
                    'Sharpe': sharpe,
                    'Max_Drawdown': max_dd,
                    'Calmar': calmar,
                    'Total_Return': total_ret,
                    'Final_Value': pv.iloc[-1],
                    'Status': 'Success'
                })

                # Progress indicator
                if combo_counter % 50 == 0:
                    elapsed = (datetime.now() - start_time).total_seconds()
                    rate = combo_counter / (elapsed / 60)
                    remaining_mins = (total_combos - combo_counter) / rate
                    print(f"[{combo_counter:4d}/{total_combos}] {param_label[:60]:60s} | "
                          f"Sharpe: {sharpe:6.3f} | ETA: {remaining_mins:5.0f} min")

            except Exception as e:
                failed_combos.append({
                    'param_label': param_label,
                    'error': str(e)[:100]
                })
                study_results.append({
                    'param_label': param_label,
                    'det_type': det_type,
                    'det_windows': str(list(det_window_config.keys())),
                    'Status': f'Failed: {str(e)[:50]}'
                })

elapsed_time = (datetime.now() - start_time).total_seconds()

print("\n" + "="*110)
print(f"COMPREHENSIVE PARAMETER STUDY COMPLETE")
print("="*110)
print(f"Total time: {elapsed_time/60:.1f} minutes ({elapsed_time/3600:.1f} hours)")
print(f"Successful: {len([r for r in study_results if r['Status'] == 'Success'])}")
print(f"Failed: {len(failed_combos)}\n")

# ════════════════════════════════════════════════════════════════════════════════
# SECTION 4: ANALYSIS OF RESULTS
# ════════════════════════════════════════════════════════════════════════════════

# Convert to DataFrame
study_df = pd.DataFrame(study_results)
study_df_success = study_df[study_df['Status'] == 'Success'].copy()

if len(study_df_success) > 0:

    print("\n" + "="*110)
    print("TOP 15 STRATEGIES BY SHARPE RATIO")
    print("="*110)

    top_sharpe = study_df_success.nlargest(15, 'Sharpe')[
        ['param_label', 'det_type', 'det_windows', 'threshold', 'signal_type',
         'smooth_window', 'min_days_persist', 'frequency', 'tc_rate',
         'CAGR', 'Volatility', 'Sharpe', 'Max_Drawdown', 'Calmar']
    ]

    for idx, (i, row) in enumerate(top_sharpe.iterrows(), 1):
        print(f"\n{idx}. {row['param_label']}")
        print(f"   Det Type: {row['det_type']:12s} | Windows: {row['det_windows']:20s}")
        print(f"   CAGR: {row['CAGR']*100:6.2f}% | Vol: {row['Volatility']*100:6.2f}% | "
              f"Sharpe: {row['Sharpe']:6.3f} | DD: {row['Max_Drawdown']*100:6.2f}% | "
              f"Calmar: {row['Calmar']:6.3f}")

    print("\n" + "="*110)
    print("TOP 10 STRATEGIES BY MAX DRAWDOWN (Best Protection)")
    print("="*110)

    top_dd = study_df_success.nsmallest(10, 'Max_Drawdown')[
        ['param_label', 'det_type', 'det_windows', 'threshold', 'signal_type',
         'smooth_window', 'min_days_persist', 'frequency', 'tc_rate',
         'CAGR', 'Volatility', 'Sharpe', 'Max_Drawdown', 'Calmar']
    ]

    for idx, (i, row) in enumerate(top_dd.iterrows(), 1):
        print(f"\n{idx}. {row['param_label']}")
        print(f"   CAGR: {row['CAGR']*100:6.2f}% | Vol: {row['Volatility']*100:6.2f}% | "
              f"Sharpe: {row['Sharpe']:6.3f} | DD: {row['Max_Drawdown']*100:6.2f}% | "
              f"Calmar: {row['Calmar']:6.3f}")

    print("\n" + "="*110)
    print("TOP 10 STRATEGIES BY CALMAR RATIO")
    print("="*110)

    top_calmar = study_df_success.nlargest(10, 'Calmar')[
        ['param_label', 'det_type', 'det_windows', 'threshold', 'signal_type',
         'smooth_window', 'min_days_persist', 'frequency', 'tc_rate',
         'CAGR', 'Volatility', 'Sharpe', 'Max_Drawdown', 'Calmar']
    ]

    for idx, (i, row) in enumerate(top_calmar.iterrows(), 1):
        print(f"\n{idx}. {row['param_label']}")
        print(f"   CAGR: {row['CAGR']*100:6.2f}% | Vol: {row['Volatility']*100:6.2f}% | "
              f"Sharpe: {row['Sharpe']:6.3f} | DD: {row['Max_Drawdown']*100:6.2f}% | "
              f"Calmar: {row['Calmar']:6.3f}")

    # ── Parameter Sensitivity Analysis ──
    print("\n" + "="*110)
    print("PARAMETER SENSITIVITY ANALYSIS")
    print("="*110)

    print("\n── BY DETERMINANT TYPE ──")
    det_analysis = study_df_success.groupby('det_type').agg({
        'Sharpe': ['mean', 'std', 'max', 'count'],
        'Max_Drawdown': ['mean', 'min'],
        'CAGR': 'mean',
        'Calmar': 'mean'
    }).round(4)
    print(det_analysis)

    print("\n── BY THRESHOLD ──")
    thresh_analysis = study_df_success.groupby('threshold').agg({
        'Sharpe': ['mean', 'std', 'max'],
        'Max_Drawdown': ['mean', 'min'],
        'CAGR': 'mean'
    }).round(4)
    print(thresh_analysis)

    print("\n── BY SIGNAL TYPE ──")
    sigtype_analysis = study_df_success.groupby('signal_type').agg({
        'Sharpe': ['mean', 'std', 'max'],
        'Max_Drawdown': ['mean', 'min'],
        'CAGR': 'mean'
    }).round(4)
    print(sigtype_analysis)

    print("\n── BY SMOOTH WINDOW ──")
    smooth_analysis = study_df_success.groupby('smooth_window').agg({
        'Sharpe': ['mean', 'std', 'max'],
        'Max_Drawdown': ['mean', 'min'],
        'CAGR': 'mean'
    }).round(4)
    print(smooth_analysis)

    print("\n── BY MIN DAYS PERSIST ──")
    mindays_analysis = study_df_success.groupby('min_days_persist').agg({
        'Sharpe': ['mean', 'std', 'max'],
        'Max_Drawdown': ['mean', 'min'],
        'CAGR': 'mean'
    }).round(4)
    print(mindays_analysis)

    print("\n── BY FREQUENCY ──")
    freq_analysis = study_df_success.groupby('frequency').agg({
        'Sharpe': ['mean', 'std', 'max'],
        'Max_Drawdown': ['mean', 'min'],
        'CAGR': 'mean'
    }).round(4)
    print(freq_analysis)

    print("\n── BY TRANSACTION COST ──")
    tc_analysis = study_df_success.groupby('tc_rate').agg({
        'Sharpe': ['mean', 'std', 'max'],
        'Max_Drawdown': ['mean', 'min'],
        'CAGR': 'mean'
    }).round(4)
    print(tc_analysis)

    # ════════════════════════════════════════════════════════════════════════════════
    # SECTION 5: VISUALIZATIONS
    # ════════════════════════════════════════════════════════════════════════════════

    print("\n" + "="*110)
    print("GENERATING COMPREHENSIVE VISUALIZATIONS")
    print("="*110 + "\n")

    # ── Plot 1: Determinant Type Comparison ──
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))

    metrics = ['Sharpe', 'Max_Drawdown', 'CAGR', 'Calmar']
    for ax, metric in zip(axes.flat, metrics):
        det_perf = study_df_success.groupby('det_type')[metric].apply(list)

        ax.boxplot([det_perf[det] for det in ['standard', 'downside', 'tail']],
                   labels=['Standard', 'Downside', 'Tail'],
                   patch_artist=True)

        if metric == 'Max_Drawdown':
            ax.set_ylabel("Max Drawdown", fontsize=11)
        elif metric == 'Sharpe':
            ax.set_ylabel("Sharpe Ratio", fontsize=11)
        elif metric == 'CAGR':
            ax.set_ylabel("CAGR", fontsize=11)
        else:
            ax.set_ylabel("Calmar Ratio", fontsize=11)

        ax.set_title(f"{metric} by Determinant Type", fontsize=12, fontweight='bold')
        ax.grid(True, alpha=0.3, axis='y')

    plt.tight_layout()
    plt.savefig("param_study_01_det_type_comparison.png", dpi=150, bbox_inches='tight')
    print("✓ Saved: param_study_01_det_type_comparison.png")
    plt.show()

    # ── Plot 2: Sharpe vs Drawdown by Determinant Type ──
    fig, ax = plt.subplots(figsize=(13, 8))

    colors_det = {'standard': '#1f77b4', 'downside': '#ff7f0e', 'tail': '#2ca02c'}

    for det_type in study_df_success['det_type'].unique():
        subset = study_df_success[study_df_success['det_type'] == det_type]
        ax.scatter(subset['Max_Drawdown'] * 100, subset['Sharpe'],
                  s=80, alpha=0.6, label=det_type, color=colors_det.get(det_type, '#000000'))

    ax.set_xlabel("Max Drawdown (%)", fontsize=12, fontweight='bold')
    ax.set_ylabel("Sharpe Ratio", fontsize=12, fontweight='bold')
    ax.set_title("Risk-Return Profile by Determinant Type", fontsize=13, fontweight='bold')
    ax.legend(fontsize=11, loc='best')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig("param_study_02_sharpe_vs_dd_by_det.png", dpi=150, bbox_inches='tight')
    print("✓ Saved: param_study_02_sharpe_vs_dd_by_det.png")
    plt.show()

    # ── Plot 3: Signal Parameter Sensitivity ──
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))

    params_to_plot = [
        ('threshold', 'Threshold', axes[0, 0]),
        ('signal_type', 'Signal Type', axes[0, 1]),
        ('smooth_window', 'Smooth Window (days)', axes[0, 2]),
        ('min_days_persist', 'Min Days Persist', axes[1, 0]),
        ('frequency', 'Rebalance Frequency', axes[1, 1]),
        ('tc_rate', 'Transaction Cost (%)', axes[1, 2]),
    ]

    for param, label, ax in params_to_plot:
        by_param = study_df_success.groupby(param)['Sharpe'].agg(['mean', 'std', 'count'])

        if param == 'tc_rate':
            x_labels = [f"{x*100:.1f}%" for x in by_param.index]
        elif param == 'signal_type':
            x_labels = ['Sym', 'Asym']
        else:
            x_labels = [str(x) for x in by_param.index]

        ax.bar(range(len(by_param)), by_param['mean'],
               yerr=by_param['std'], capsize=5, alpha=0.7, color='steelblue')
        ax.set_xticks(range(len(by_param)))
        ax.set_xticklabels(x_labels, fontsize=10)
        ax.set_ylabel("Sharpe Ratio", fontsize=10)
        ax.set_title(f"Sharpe by {label}", fontsize=11, fontweight='bold')
        ax.grid(True, alpha=0.3, axis='y')

    plt.tight_layout()
    plt.savefig("param_study_03_signal_sensitivity.png", dpi=150, bbox_inches='tight')
    print("✓ Saved: param_study_03_signal_sensitivity.png")
    plt.show()

    # ── Plot 4: 2D Heatmap - Threshold vs Smooth Window ──
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    for idx, det_type in enumerate(['standard', 'downside', 'tail']):
        subset = study_df_success[study_df_success['det_type'] == det_type]

        pivot = subset.pivot_table(
            values='Sharpe',
            index='smooth_window',
            columns='threshold',
            aggfunc='mean'
        )

        im = axes[idx].imshow(pivot.values, cmap='RdYlGn', aspect='auto')
        axes[idx].set_xticks(range(len(pivot.columns)))
        axes[idx].set_yticks(range(len(pivot.index)))
        axes[idx].set_xticklabels(pivot.columns, fontsize=10)
        axes[idx].set_yticklabels(pivot.index, fontsize=10)
        axes[idx].set_xlabel("Threshold", fontsize=11, fontweight='bold')
        axes[idx].set_ylabel("Smooth Window", fontsize=11, fontweight='bold')
        axes[idx].set_title(f"Sharpe Ratio: {det_type.capitalize()}", fontsize=12, fontweight='bold')

        plt.colorbar(im, ax=axes[idx], label='Sharpe')

    plt.tight_layout()
    plt.savefig("param_study_04_threshold_smooth_heatmap.png", dpi=150, bbox_inches='tight')
    print("✓ Saved: param_study_04_threshold_smooth_heatmap.png")
    plt.show()

    # ── Plot 5: 3D Surface - Threshold vs Min Days vs Sharpe ──
    from mpl_toolkits.mplot3d import Axes3D

    fig = plt.figure(figsize=(16, 5))

    for idx, det_type in enumerate(['standard', 'downside', 'tail']):
        subset = study_df_success[study_df_success['det_type'] == det_type]

        ax = fig.add_subplot(1, 3, idx+1, projection='3d')

        # Create pivot table
        pivot = subset.pivot_table(
            values='Sharpe',
            index='min_days_persist',
            columns='threshold',
            aggfunc='mean'
        )

        X = pivot.columns.values
        Y = pivot.index.values
        X, Y = np.meshgrid(X, Y)
        Z = pivot.values

        surf = ax.plot_surface(X, Y, Z, cmap='viridis', alpha=0.8)
        ax.set_xlabel("Threshold", fontsize=10)
        ax.set_ylabel("Min Days Persist", fontsize=10)
        ax.set_zlabel("Sharpe Ratio", fontsize=10)
        ax.set_title(f"{det_type.capitalize()}", fontsize=11, fontweight='bold')

        fig.colorbar(surf, ax=ax, shrink=0.5)

    plt.tight_layout()
    plt.savefig("param_study_05_3d_threshold_mindays.png", dpi=150, bbox_inches='tight')
    print("✓ Saved: param_study_05_3d_threshold_mindays.png")
    plt.show()

    # ════════════════════════════════════════════════════════════════════════════════
    # SECTION 6: SAVE RESULTS
    # ════════════════════════════════════════════════════════════════════════════════

    study_df.to_csv("parameter_study_comprehensive_results.csv", index=False)
    study_df_success.to_csv("parameter_study_comprehensive_successful.csv", index=False)

    print("\n" + "="*110)
    print("RESULTS SAVED")
    print("="*110)
    print("✓ Saved: parameter_study_comprehensive_results.csv")
    print("✓ Saved: parameter_study_comprehensive_successful.csv")

    # ── Summary ──
    print("\n" + "="*110)
    print("SUMMARY STATISTICS")
    print("="*110)
    print(f"\nTotal successful combinations: {len(study_df_success)}")
    print(f"Sharpe Ratio range:          {study_df_success['Sharpe'].min():.3f} → {study_df_success['Sharpe'].max():.3f}")
    print(f"Max Drawdown range:          {study_df_success['Max_Drawdown'].min()*100:.2f}% → {study_df_success['Max_Drawdown'].max()*100:.2f}%")
    print(f"CAGR range:                  {study_df_success['CAGR'].min()*100:.2f}% → {study_df_success['CAGR'].max()*100:.2f}%")
    print(f"Calmar ratio range:          {study_df_success['Calmar'].min():.3f} → {study_df_success['Calmar'].max():.3f}")

else:
    print("⚠ No successful combinations found!")


In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# SECTION 7: RECONSTRUCT & PLOT TOP 5 COMBINATIONS (FIXED)
# ════════════════════════════════════════════════════════════════════════════════

print("\n" + "="*110)
print("RECONSTRUCTING AND PLOTTING TOP 5 PARAMETER COMBINATIONS")
print("="*110 + "\n")

# Get top 5 by Sharpe Ratio
top_5_sharpe = study_df_success.nlargest(5, 'Sharpe').copy()

print("Top 5 Combinations by Sharpe Ratio:")
print("-" * 110)
for idx, (i, row) in enumerate(top_5_sharpe.iterrows(), 1):
    print(f"{idx}. Sharpe: {row['Sharpe']:.3f} | CAGR: {row['CAGR']*100:.2f}% | "
          f"Vol: {row['Volatility']*100:.2f}% | Max DD: {row['Max_Drawdown']*100:.2f}%")
    print(f"   {row['param_label']}\n")

# Store portfolio values and metadata
top_5_portfolios = {}
top_5_metadata = {}

for idx, (i, row) in enumerate(top_5_sharpe.iterrows(), 1):

    param_label = row['param_label']

    # ── EXTRACT PARAMETERS FROM ROW (NOT FROM LABEL) ──
    det_type = row['det_type']
    # Parse window weights string like "{'7': 0.8, '28': 0.2}" back to dict
    window_weights_str = row['window_weights']
    window_weights = eval(window_weights_str)  # Convert string to dict
    det_windows = list(map(int, window_weights.keys()))  # Extract windows as integers

    thresh = row['threshold']
    sig_type = row['signal_type']
    smooth_win = int(row['smooth_window'])
    min_days = int(row['min_days_persist'])
    freq = row['frequency']
    tc_rate = row['tc_rate']

    print(f"[{idx}/5] Reconstructing: {param_label[:80]}...", end=" ", flush=True)

    try:
        # ── STEP 1: Recompute Determinants ──
        if det_type == 'standard':
            det_results = compute_determinant_standard(index_prices, windows=det_windows)
        elif det_type == 'downside':
            det_results = compute_determinant_downside(index_prices, windows=det_windows)
        else:  # tail
            det_results = compute_determinant_tail(index_prices, windows=det_windows)

        # ── STEP 2: Compute Composite Score ──
        composite = compute_composite_score_from_dets(det_results, window_weights)

        # ── STEP 3: Build Signal ──
        if smooth_win > 1:
            sig_df = build_signal_df_smoothed(
                composite, equity_assets, macro_df.index,
                threshold=thresh, smooth_window=smooth_win,
                signal_type=sig_type
            )
        else:
            sig_df = build_signal_df_persistent(
                composite, equity_assets, macro_df.index,
                threshold=thresh, min_days=min_days,
                signal_type=sig_type
            )

        # ── STEP 4: Run Portfolio ──
        def wfn(sig_row, date, _eq=equity_assets, _ne=non_equity_assets,
               _bw=base_weight, _sh=EQUITY_SAFE_HAVEN_MAP):
            return build_weights_3stage(sig_row, _eq, _ne, _bw, date, _sh)

        pnl_df = run_signal_portfolio(
            returns, sig_df, equity_assets, non_equity_assets,
            ALL_ASSETS, weight_fn=wfn,
            freq=freq, tc_rate=tc_rate, initial_capital=1e9
        )

        # Store results
        top_5_portfolios[param_label] = pnl_df['portfolio_value']
        top_5_metadata[param_label] = {
            'sharpe': row['Sharpe'],
            'cagr': row['CAGR'],
            'vol': row['Volatility'],
            'max_dd': row['Max_Drawdown'],
            'calmar': row['Calmar'],
            'det_type': det_type,
            'det_windows': det_windows,
            'threshold': thresh,
            'signal_type': sig_type,
            'smooth_window': smooth_win,
            'min_days_persist': min_days,
            'frequency': freq,
            'tc_rate': tc_rate
        }

        print("✓")

    except Exception as e:
        print(f"✗ Error: {str(e)[:80]}")
        import traceback
        traceback.print_exc()
        continue

print()

if len(top_5_portfolios) > 0:

    # ════════════════════════════════════════════════════════════════════════════════
    # PLOT 1: Portfolio Values Over Time (Absolute)
    # ════════════════════════════════════════════════════════════════════════════════

    fig, ax = plt.subplots(figsize=(16, 9))

    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']

    for (label, pv), color in zip(top_5_portfolios.items(), colors):
        meta = top_5_metadata[label]
        legend_label = (f"#{list(top_5_portfolios.keys()).index(label)+1} | "
                       f"Sharpe: {meta['sharpe']:.3f} | CAGR: {meta['cagr']*100:.1f}% | "
                       f"DD: {meta['max_dd']*100:.1f}%")
        ax.plot(pv.index, pv.values / 1e9, linewidth=2.5, label=legend_label, color=color)

    ax.set_xlabel("Date", fontsize=12, fontweight='bold')
    ax.set_ylabel("Portfolio Value ($B)", fontsize=12, fontweight='bold')
    ax.set_title("Top 5 Signal Strategies: Portfolio Value Evolution", fontsize=14, fontweight='bold', pad=20)
    ax.legend(fontsize=10, loc='best', framealpha=0.95)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig("top5_01_portfolio_values_absolute.png", dpi=150, bbox_inches='tight')
    print("✓ Saved: top5_01_portfolio_values_absolute.png")
    plt.show()

    # ════════════════════════════════════════════════════════════════════════════════
    # PLOT 2: Cumulative Returns (Indexed to 100)
    # ════════════════════════════════════════════════════════════════════════════════

    fig, ax = plt.subplots(figsize=(16, 9))

    for (label, pv), color in zip(top_5_portfolios.items(), colors):
        meta = top_5_metadata[label]
        cum_ret = (pv / pv.iloc[0] - 1) * 100
        legend_label = (f"#{list(top_5_portfolios.keys()).index(label)+1} | "
                       f"Sharpe: {meta['sharpe']:.3f} | Return: {cum_ret.iloc[-1]:.1f}%")
        ax.plot(cum_ret.index, cum_ret.values, linewidth=2.5, label=legend_label, color=color)

    ax.set_xlabel("Date", fontsize=12, fontweight='bold')
    ax.set_ylabel("Cumulative Return (%)", fontsize=12, fontweight='bold')
    ax.set_title("Top 5 Signal Strategies: Cumulative Returns", fontsize=14, fontweight='bold', pad=20)
    ax.legend(fontsize=10, loc='best', framealpha=0.95)
    ax.grid(True, alpha=0.3)
    ax.axhline(y=0, color='black', linestyle='--', alpha=0.3, linewidth=1)
    plt.tight_layout()
    plt.savefig("top5_02_cumulative_returns.png", dpi=150, bbox_inches='tight')
    print("✓ Saved: top5_02_cumulative_returns.png")
    plt.show()

    # ════════════════════════════════════════════════════════════════════════════════
    # PLOT 3: Drawdown Series (Individual Subplots)
    # ════════════════════════════════════════════════════════════════════════════════

    fig, axes = plt.subplots(5, 1, figsize=(16, 14))

    for ax, (label, pv), color in zip(axes, top_5_portfolios.items(), colors):
        meta = top_5_metadata[label]
        cum_ret = pv / pv.iloc[0] - 1
        running_max = cum_ret.expanding().max()
        drawdown = (cum_ret - running_max) / (1 + running_max) * 100

        ax.fill_between(drawdown.index, drawdown.values, 0, alpha=0.6, color=color)
        ax.plot(drawdown.index, drawdown.values, color=color, linewidth=1.5)

        idx_num = list(top_5_portfolios.keys()).index(label) + 1
        title = (f"#{idx_num} | Sharpe: {meta['sharpe']:.3f} | "
                f"Max DD: {meta['max_dd']*100:.1f}% | CAGR: {meta['cagr']*100:.1f}% | "
                f"Det: {meta['det_type']} | Windows: {meta['det_windows']} | Thresh: {meta['threshold']}")
        ax.set_title(title, fontsize=11, fontweight='bold')
        ax.set_ylabel("Drawdown (%)", fontsize=10)
        ax.grid(True, alpha=0.3)
        ax.axhline(y=0, color='black', linestyle='-', alpha=0.5, linewidth=0.5)

    axes[-1].set_xlabel("Date", fontsize=12, fontweight='bold')
    fig.suptitle("Top 5 Strategies: Individual Drawdown Series", fontsize=14, fontweight='bold', y=0.995)
    plt.tight_layout()
    plt.savefig("top5_03_drawdown_series.png", dpi=150, bbox_inches='tight')
    print("✓ Saved: top5_03_drawdown_series.png")
    plt.show()

    # ════════════════════════════════════════════════════════════════════════════════
    # PLOT 4: Rolling 252-Day Volatility
    # ════════════════════════════════════════════════════════════════════════════════

    fig, ax = plt.subplots(figsize=(16, 9))

    for (label, pv), color in zip(top_5_portfolios.items(), colors):
        meta = top_5_metadata[label]
        daily_ret = pv.pct_change()
        rolling_vol = daily_ret.rolling(252).std() * np.sqrt(252) * 100

        idx_num = list(top_5_portfolios.keys()).index(label) + 1
        legend_label = (f"#{idx_num} | Ann. Vol: {rolling_vol.iloc[-1]:.1f}% | "
                       f"Avg Vol: {rolling_vol.mean():.1f}%")
        ax.plot(rolling_vol.index, rolling_vol.values, linewidth=2, label=legend_label, color=color)

    ax.set_xlabel("Date", fontsize=12, fontweight='bold')
    ax.set_ylabel("Rolling 252-Day Volatility (%)", fontsize=12, fontweight='bold')
    ax.set_title("Top 5 Strategies: Rolling Annualized Volatility", fontsize=14, fontweight='bold', pad=20)
    ax.legend(fontsize=10, loc='best', framealpha=0.95)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig("top5_04_rolling_volatility.png", dpi=150, bbox_inches='tight')
    print("✓ Saved: top5_04_rolling_volatility.png")
    plt.show()

    # ════════════════════════════════════════════════════════════════════════════════
    # PLOT 5: Monthly Returns Heatmap (Each Strategy)
    # ════════════════════════════════════════════════════════════════════════════════

    fig, axes = plt.subplots(1, 5, figsize=(20, 5))

    for ax, (label, pv), color in zip(axes, top_5_portfolios.items(), colors):
        meta = top_5_metadata[label]

        # Calculate monthly returns
        monthly_pv = pv.resample('M').last()
        monthly_ret = monthly_pv.pct_change().dropna() * 100

        # Create year-month structure
        dates = pd.to_datetime(monthly_ret.index)
        years = sorted(dates.year.unique())
        months = range(1, 13)

        # Create heatmap data
        heatmap_data = np.full((len(years), 12), np.nan)
        for i, year in enumerate(years):
            for j, month in enumerate(months):
                mask = (dates.year == year) & (dates.month == month)
                if mask.any():
                    heatmap_data[i, j] = monthly_ret[mask].values[0]

        im = ax.imshow(heatmap_data, cmap='RdYlGn', aspect='auto', vmin=-10, vmax=10)
        ax.set_xticks(range(12))
        ax.set_xticklabels(['J','F','M','A','M','J','J','A','S','O','N','D'], fontsize=9)
        ax.set_yticks(range(len(years)))
        ax.set_yticklabels(years, fontsize=9)

        idx_num = list(top_5_portfolios.keys()).index(label) + 1
        ax.set_title(f"#{idx_num} | Sharpe: {meta['sharpe']:.3f}", fontsize=11, fontweight='bold')
        ax.set_ylabel("Year", fontsize=10)

        # Add text annotations
        for i in range(len(years)):
            for j in range(12):
                if not np.isnan(heatmap_data[i, j]):
                    text = ax.text(j, i, f"{heatmap_data[i, j]:.0f}",
                                 ha="center", va="center", color="black", fontsize=7)

    fig.suptitle("Top 5 Strategies: Monthly Returns Heatmap (%)", fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig("top5_05_monthly_returns_heatmap.png", dpi=150, bbox_inches='tight')
    print("✓ Saved: top5_05_monthly_returns_heatmap.png")
    plt.show()

    # ════════════════════════════════════════════════════════════════════════════════
    # PLOT 6: Risk-Return Profile Comparison
    # ════════════════════════════════════════════════════════════════════════════════

    fig, ax = plt.subplots(figsize=(13, 9))

    for (label, pv), color in zip(top_5_portfolios.items(), colors):
        meta = top_5_metadata[label]
        idx_num = list(top_5_portfolios.keys()).index(label) + 1

        ax.scatter(meta['vol']*100, meta['cagr']*100, s=500, alpha=0.7,
                  color=color, edgecolors='black', linewidth=2)
        ax.annotate(f"#{idx_num}\nSharpe:{meta['sharpe']:.2f}",
                   xy=(meta['vol']*100, meta['cagr']*100),
                   ha='center', va='center', fontsize=10, fontweight='bold')

    ax.set_xlabel("Volatility (%)", fontsize=12, fontweight='bold')
    ax.set_ylabel("CAGR (%)", fontsize=12, fontweight='bold')
    ax.set_title("Top 5 Strategies: Risk-Return Profile", fontsize=14, fontweight='bold', pad=20)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig("top5_06_risk_return_profile.png", dpi=150, bbox_inches='tight')
    print("✓ Saved: top5_06_risk_return_profile.png")
    plt.show()

    # ════════════════════════════════════════════════════════════════════════════════
    # PLOT 7: Detailed Metrics Comparison Table
    # ════════════════════════════════════════════════════════════════════════════════

    print("\n" + "="*110)
    print("TOP 5 STRATEGIES - DETAILED COMPARISON")
    print("="*110 + "\n")

    comparison_data = []

    for idx, (label, pv) in enumerate(top_5_portfolios.items(), 1):
        meta = top_5_metadata[label]

        total_ret = (pv.iloc[-1] / pv.iloc[0] - 1)
        final_value = pv.iloc[-1] / 1e9

        comparison_data.append({
            '#': idx,
            'CAGR (%)': f"{meta['cagr']*100:.2f}",
            'Vol (%)': f"{meta['vol']*100:.2f}",
            'Sharpe': f"{meta['sharpe']:.3f}",
            'Max DD (%)': f"{meta['max_dd']*100:.2f}",
            'Calmar': f"{meta['calmar']:.3f}",
            'Total Return (%)': f"{total_ret*100:.2f}",
            'Final Value ($B)': f"{final_value:.2f}",
            'Det Type': meta['det_type'].capitalize(),
            'Windows': str(meta['det_windows']),
            'Threshold': f"{meta['threshold']:.1f}",
            'Signal': meta['signal_type'][:3].capitalize(),
            'Smooth': meta['smooth_window'],
            'Persist': meta['min_days_persist'],
            'Freq': meta['frequency'][:3].capitalize(),
            'TC (%)': f"{meta['tc_rate']*100:.1f}"
        })

    comparison_df = pd.DataFrame(comparison_data)
    print(comparison_df.to_string(index=False))

    # Save comparison
    comparison_df.to_csv("top5_comparison_metrics.csv", index=False)
    print("\n✓ Saved: top5_comparison_metrics.csv")

    print("\n" + "="*110)
    print("TOP 5 PORTFOLIO VISUALIZATIONS COMPLETE")
    print("="*110)

else:
    print("⚠ No portfolios could be reconstructed!")


In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# SECTION 0: COMPREHENSIVE PARAMETER GRID DEFINITION WITH FINE THRESHOLD SCAN
# ════════════════════════════════════════════════════════════════════════════════

import itertools
from datetime import datetime
import numpy as np

print("\n" + "="*110)
print("COMPREHENSIVE PARAMETER STUDY: SIGNAL PORTFOLIO OPTIMIZATION")
print("WITH FINE-GRAINED THRESHOLD SCANNING (0.1 STEP SIZE)")
print("="*110)

# ── DETERMINANT COMPUTATION PARAMETERS ──
DET_WINDOW_CONFIGS = [
    {3: 0.2, 7: 0.3, 28: 0.5},      # Original balanced
    {3: 0.4, 7: 0.3, 28: 0.3},      # Short-term weighted
    {7: 0.2, 28: 0.8},               # Long-term focused
    {3: 0.33, 7: 0.33, 28: 0.34},   # Equal weight
]

DET_TYPES = ['standard', 'downside', 'tail']  # Which determinant to use

# ── GENERATE FINE-GRAINED THRESHOLD RANGE ──
# Using numpy to create thresholds from 0.1 to 2.0 with 0.1 step
THRESHOLD_FINE = list(np.round(np.arange(0.1, 2.1, 0.1), 1))

print(f"Fine-grained thresholds: {THRESHOLD_FINE}")
print(f"Number of threshold values: {len(THRESHOLD_FINE)}\n")

# ── SIGNAL PARAMETERS ──
PARAM_GRID = {
    'threshold': THRESHOLD_FINE,  # Fine-grained scan
    'signal_type': ['symmetric', 'asymmetric'],
    'smooth_window': [1, 3, 5, 10],  # 1 = no smoothing
    'min_days_persist': [1, 5, 10, 20],  # 1 = no persistence
    'freq': ['monthly', 'quarterly'],
    'tc_rate': [0.001, 0.005],  # 0.1% and 0.5%
}

# Generate all combinations
det_window_combos = DET_WINDOW_CONFIGS
det_type_combos = DET_TYPES
signal_combos = list(itertools.product(
    PARAM_GRID['threshold'],
    PARAM_GRID['signal_type'],
    PARAM_GRID['smooth_window'],
    PARAM_GRID['min_days_persist'],
    PARAM_GRID['freq'],
    PARAM_GRID['tc_rate']
))

total_combos = len(det_window_combos) * len(det_type_combos) * len(signal_combos)

print(f"Determinant window configs: {len(det_window_combos)}")
print(f"Determinant types: {len(det_type_combos)}")
print(f"Signal parameter combos: {len(signal_combos)}")
print(f"  - Thresholds: {len(PARAM_GRID['threshold'])}")
print(f"  - Signal types: {len(PARAM_GRID['signal_type'])}")
print(f"  - Smooth windows: {len(PARAM_GRID['smooth_window'])}")
print(f"  - Min days persist: {len(PARAM_GRID['min_days_persist'])}")
print(f"  - Frequencies: {len(PARAM_GRID['freq'])}")
print(f"  - Transaction costs: {len(PARAM_GRID['tc_rate'])}")
print(f"\nTotal parameter combinations: {total_combos}")
print(f"Estimated runtime: {total_combos * 0.3:.0f} seconds (~{total_combos * 0.3 / 60:.1f} minutes)\n")

# ════════════════════════════════════════════════════════════════════════════════
# SECTION 1: RECOMPUTABLE DETERMINANT FUNCTIONS (UNCHANGED)
# ════════════════════════════════════════════════════════════════════════════════

def compute_determinant_standard(price_series_dict, windows=[3, 7, 28]):
    """Compute standard correlation-based determinant"""
    prices = pd.DataFrame(price_series_dict).sort_index()
    rets = prices.pct_change().dropna()

    det_results = {}

    for window in windows:
        det_series = []

        for i in range(window, len(rets)):
            window_rets = rets.iloc[i - window:i]
            valid_cols = window_rets.columns[window_rets.std() > 1e-10]

            if len(valid_cols) < 2:
                det_series.append(np.nan)
                continue

            try:
                w = window_rets[valid_cols]
                R = w.corr().values
                I = np.eye(len(R))
                M = R - I
                det = np.linalg.det(M)
            except:
                det = np.nan

            det_series.append(det)

        det_results[window] = pd.Series(
            det_series,
            index=rets.index[window:],
            name=f'det_{window}d'
        )

    return det_results

def compute_determinant_downside(price_series_dict, windows=[3, 7, 28]):
    """Compute downside-only determinant (negative returns only)"""
    prices = pd.DataFrame(price_series_dict).sort_index()
    rets = prices.pct_change().dropna()

    det_results = {}

    for window in windows:
        det_series = []

        for i in range(window, len(rets)):
            window_rets = rets.iloc[i - window:i]
            down_rets = window_rets[window_rets.mean(axis=1) < 0]

            if len(down_rets) < 2:
                det_series.append(np.nan)
                continue

            valid_cols = down_rets.columns[down_rets.std() > 1e-10]

            if len(valid_cols) < 2:
                det_series.append(np.nan)
                continue

            try:
                w = down_rets[valid_cols]
                R = w.corr().values
                I = np.eye(len(R))
                M = R - I
                det = np.linalg.det(M)
            except:
                det = np.nan

            det_series.append(det)

        det_results[window] = pd.Series(
            det_series,
            index=rets.index[window:],
            name=f'det_down_{window}d'
        )

    return det_results

def compute_determinant_tail(price_series_dict, windows=[3, 7, 28], tail_percentile=10):
    """Compute tail-only determinant (extreme stress days)"""
    prices = pd.DataFrame(price_series_dict).sort_index()
    rets = prices.pct_change().dropna()

    agg_ret = rets.mean(axis=1)
    tail_threshold = agg_ret.quantile(tail_percentile / 100)
    tail_days = agg_ret < tail_threshold

    det_results = {}

    for window in windows:
        det_series = []

        for i in range(window, len(rets)):
            window_rets = rets.iloc[i - window:i]
            tail_window = tail_days.iloc[i - window:i]

            if tail_window.sum() < 1:
                det_series.append(np.nan)
                continue

            tail_rets = window_rets[tail_window]
            valid_cols = tail_rets.columns[tail_rets.std() > 1e-10]

            if len(valid_cols) < 2:
                det_series.append(np.nan)
                continue

            try:
                w = tail_rets[valid_cols]
                R = w.corr().values
                I = np.eye(len(R))
                M = R - I
                det = np.linalg.det(M)
            except:
                det = np.nan

            det_series.append(det)

        det_results[window] = pd.Series(
            det_series,
            index=rets.index[window:],
            name=f'det_tail_{window}d'
        )

    return det_results

def compute_composite_score_from_dets(det_results, window_weights):
    """Compute composite z-score from determinant results"""
    all_series = []
    for window, series in det_results.items():
        mu = series.rolling(252, min_periods=60).mean()
        sigma = series.rolling(252, min_periods=60).std()
        z = (series - mu) / sigma.replace(0, np.nan)
        z.name = f'z_{window}d'
        all_series.append(z)

    z_df = pd.concat(all_series, axis=1).dropna()

    # Blend using provided weights
    composite = pd.Series(0, index=z_df.index)
    for window, weight in window_weights.items():
        col_name = f'z_{window}d'
        if col_name in z_df.columns:
            composite += z_df[col_name] * weight

    composite = composite / sum(window_weights.values())
    return composite

# ════════════════════════════════════════════════════════════════════════════════
# SECTION 2: SIGNAL BUILDERS (UNCHANGED)
# ════════════════════════════════════════════════════════════════════════════════

def classify_signal_symmetric(score, threshold):
    """Binary: +1 if |score| > threshold, else -1"""
    return +1 if abs(score) > threshold else -1

def classify_signal_asymmetric(score, threshold):
    """Asymmetric: +1 if score > threshold, else -1"""
    return +1 if score > threshold else -1

def build_signal(composite_score, equity_assets, macro_dates,
                threshold=1.0, smooth_window=1, min_days=1, signal_type='symmetric'):
    """Build signal with both smoothing and persistence"""

    # Apply smoothing first
    if smooth_window > 1:
        smoothed = composite_score.rolling(smooth_window, min_periods=1).mean()
    else:
        smoothed = composite_score

    # Then apply persistence
    if signal_type == 'symmetric':
        raw = smoothed.apply(lambda x: classify_signal_symmetric(x, threshold))
    else:
        raw = smoothed.apply(lambda x: classify_signal_asymmetric(x, threshold))

    if min_days <= 1:
        confirmed = raw.copy()
    else:
        confirmed = raw.copy()
        current = int(raw.iloc[0])
        count = 0

        for i in range(1, len(raw)):
            if int(raw.iloc[i]) != current:
                count += 1
                if count >= min_days:
                    current = int(raw.iloc[i])
                    count = 0
            else:
                count = 0
            confirmed.iloc[i] = current

    signal_df = pd.DataFrame(index=macro_dates)
    for asset in equity_assets:
        signal_df[asset] = confirmed.reindex(macro_dates).ffill().bfill()
    return signal_df

# ════════════════════════════════════════════════════════════════════════════════
# SECTION 3: PARAMETER STUDY EXECUTION WITH PROGRESS TRACKING
# ════════════════════════════════════════════════════════════════════════════════

study_results = []
failed_combos = []

print("\n" + "="*110)
print("RUNNING COMPREHENSIVE PARAMETER STUDY WITH FINE THRESHOLD SCAN")
print("="*110 + "\n")

start_time = datetime.now()
combo_counter = 0

for det_idx, det_window_config in enumerate(det_window_combos, 1):

    for det_type_idx, det_type in enumerate(det_type_combos, 1):

        # Compute determinants once per (window_config, det_type) combo
        print(f"\n[{det_idx}/{len(det_window_combos)} | {det_type_idx}/{len(det_type_combos)}] "
              f"Windows: {list(det_window_config.keys())}, Type: {det_type}...", flush=True)

        try:
            if det_type == 'standard':
                det_results = compute_determinant_standard(index_prices,
                                                          windows=list(det_window_config.keys()))
            elif det_type == 'downside':
                det_results = compute_determinant_downside(index_prices,
                                                          windows=list(det_window_config.keys()))
            else:  # tail
                det_results = compute_determinant_tail(index_prices,
                                                      windows=list(det_window_config.keys()))

            composite = compute_composite_score_from_dets(det_results, det_window_config)
            print(f"  ✓ Composite score computed: {composite.shape}")

        except Exception as e:
            print(f"  ✗ Failed to compute determinants: {str(e)[:60]}")
            continue

        # Now test all signal parameter combinations
        for (thresh, sig_type, smooth_win, min_days, freq, tc_rate) in signal_combos:

            combo_counter += 1

            param_label = (f"det{det_type[0]}_w{list(det_window_config.keys())}_"
                          f"t{thresh:.1f}_sig{sig_type[0]}_sm{smooth_win}_"
                          f"pd{min_days}_{freq}_tc{tc_rate*100:.1f}pct")

            try:
                # Build signal
                sig_df = build_signal(
                    composite, equity_assets, macro_df.index,
                    threshold=thresh, smooth_window=smooth_win,
                    min_days=min_days, signal_type=sig_type
                )

                # Run portfolio backtest
                def wfn(sig_row, date, _eq=equity_assets, _ne=non_equity_assets,
                       _bw=base_weight, _sh=EQUITY_SAFE_HAVEN_MAP):
                    return build_weights_3stage(sig_row, _eq, _ne, _bw, date, _sh)

                pnl_df = run_signal_portfolio(
                    returns, sig_df, equity_assets, non_equity_assets,
                    ALL_ASSETS, weight_fn=wfn,
                    freq=freq, tc_rate=tc_rate, initial_capital=1e9
                )

                # Calculate metrics
                pv = pnl_df['portfolio_value']
                total_ret = (pv.iloc[-1] / pv.iloc[0]) - 1
                days = (pv.index[-1] - pv.index[0]).days
                cagr = (pv.iloc[-1] / pv.iloc[0]) ** (365.25 / days) - 1

                daily_returns = pv.pct_change().dropna()
                ann_vol = daily_returns.std() * np.sqrt(252)
                sharpe = cagr / ann_vol if ann_vol > 0 else 0

                # Maximum drawdown
                cum_ret = (pv / pv.iloc[0]) - 1
                running_max = cum_ret.expanding().max()
                drawdown_series = (cum_ret - running_max) / (1 + running_max)
                max_dd = drawdown_series.min()

                calmar = cagr / abs(max_dd) if max_dd != 0 else 0

                # Store results
                study_results.append({
                    'param_label': param_label,
                    'det_type': det_type,
                    'det_windows': str(list(det_window_config.keys())),
                    'window_weights': str(det_window_config),
                    'threshold': thresh,
                    'signal_type': sig_type,
                    'smooth_window': smooth_win,
                    'min_days_persist': min_days,
                    'frequency': freq,
                    'tc_rate': tc_rate,
                    'CAGR': cagr,
                    'Volatility': ann_vol,
                    'Sharpe': sharpe,
                    'Max_Drawdown': max_dd,
                    'Calmar': calmar,
                    'Total_Return': total_ret,
                    'Final_Value': pv.iloc[-1],
                    'Status': 'Success'
                })

                # Progress indicator (every 100 combos)
                if combo_counter % 100 == 0:
                    elapsed = (datetime.now() - start_time).total_seconds()
                    rate = combo_counter / (elapsed / 60)
                    remaining_mins = (total_combos - combo_counter) / rate
                    pct_complete = 100 * combo_counter / total_combos
                    print(f"  [{combo_counter:5d}/{total_combos}] {pct_complete:5.1f}% | "
                          f"Sharpe: {sharpe:6.3f} | ETA: {remaining_mins:5.0f} min | Rate: {rate:5.1f} combos/min")

            except Exception as e:
                error_msg = str(e)[:100]
                failed_combos.append({
                    'param_label': param_label,
                    'error': error_msg
                })
                study_results.append({
                    'param_label': param_label,
                    'det_type': det_type,
                    'det_windows': str(list(det_window_config.keys())),
                    'Status': f'Failed: {error_msg[:50]}'
                })

elapsed_time = (datetime.now() - start_time).total_seconds()

print("\n" + "="*110)
print(f"COMPREHENSIVE PARAMETER STUDY COMPLETE")
print("="*110)
print(f"Total time: {elapsed_time/60:.1f} minutes ({elapsed_time/3600:.1f} hours)")
print(f"Successful: {len([r for r in study_results if r.get('Status') == 'Success'])}")
print(f"Failed: {len(failed_combos)}\n")

# ════════════════════════════════════════════════════════════════════════════════
# SECTION 4: THRESHOLD SENSITIVITY ANALYSIS
# ════════════════════════════════════════════════════════════════════════════════

study_df = pd.DataFrame(study_results)
study_df_success = study_df[study_df['Status'] == 'Success'].copy()

if len(study_df_success) > 0:

    print("\n" + "="*110)
    print("THRESHOLD SENSITIVITY ANALYSIS")
    print("="*110)

    threshold_analysis = study_df_success.groupby('threshold').agg({
        'Sharpe': ['mean', 'std', 'max', 'count'],
        'Max_Drawdown': ['mean', 'min'],
        'CAGR': ['mean', 'max'],
        'Calmar': ['mean', 'max']
    }).round(4)

    print("\nAggregated metrics by threshold:")
    print(threshold_analysis)

    # Find optimal threshold
    optimal_threshold = study_df_success.groupby('threshold')['Sharpe'].mean().idxmax()
    print(f"\n✓ Optimal threshold (by avg Sharpe): {optimal_threshold:.1f}")

    # Find threshold with best max Sharpe
    best_sharpe_threshold = study_df_success.groupby('threshold')['Sharpe'].max().idxmax()
    print(f"✓ Threshold with best Sharpe achieved: {best_sharpe_threshold:.1f}")

    # ── Plot 6: Threshold Sensitivity Curves ──
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))

    thresh_by_det = {}
    for det_type in study_df_success['det_type'].unique():
        thresh_by_det[det_type] = study_df_success[study_df_success['det_type'] == det_type].groupby('threshold').agg({
            'Sharpe': ['mean', 'std'],
            'Max_Drawdown': ['mean'],
            'CAGR': ['mean'],
            'Calmar': ['mean']
        })

    # Plot 1: Sharpe by Threshold for each determinant type
    ax = axes[0, 0]
    colors_det = {'standard': '#1f77b4', 'downside': '#ff7f0e', 'tail': '#2ca02c'}
    for det_type in sorted(thresh_by_det.keys()):
        data = thresh_by_det[det_type]['Sharpe']
        ax.plot(data.index, data['mean'], marker='o', label=det_type,
               color=colors_det.get(det_type, '#000000'), linewidth=2.5, markersize=6)
        ax.fill_between(data.index,
                        data['mean'] - data['std'],
                        data['mean'] + data['std'],
                        alpha=0.2, color=colors_det.get(det_type, '#000000'))
    ax.set_xlabel("Threshold", fontsize=11, fontweight='bold')
    ax.set_ylabel("Sharpe Ratio", fontsize=11, fontweight='bold')
    ax.set_title("Sharpe Ratio vs Threshold (by Determinant Type)", fontsize=12, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)

    # Plot 2: Max Drawdown by Threshold for each determinant type
    ax = axes[0, 1]
    for det_type in sorted(thresh_by_det.keys()):
        data = thresh_by_det[det_type]['Max_Drawdown']
        ax.plot(data.index, data['mean'] * 100, marker='s', label=det_type,
               color=colors_det.get(det_type, '#000000'), linewidth=2.5, markersize=6)
    ax.set_xlabel("Threshold", fontsize=11, fontweight='bold')
    ax.set_ylabel("Max Drawdown (%)", fontsize=11, fontweight='bold')
    ax.set_title("Max Drawdown vs Threshold (by Determinant Type)", fontsize=12, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)

    # Plot 3: CAGR by Threshold for each determinant type
    ax = axes[1, 0]
    for det_type in sorted(thresh_by_det.keys()):
        data = thresh_by_det[det_type]['CAGR']
        ax.plot(data.index, data['mean'] * 100, marker='^', label=det_type,
               color=colors_det.get(det_type, '#000000'), linewidth=2.5, markersize=6)
    ax.set_xlabel("Threshold", fontsize=11, fontweight='bold')
    ax.set_ylabel("CAGR (%)", fontsize=11, fontweight='bold')
    ax.set_title("CAGR vs Threshold (by Determinant Type)", fontsize=12, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)

    # Plot 4: Calmar by Threshold for each determinant type
    ax = axes[1, 1]
    for det_type in sorted(thresh_by_det.keys()):
        data = thresh_by_det[det_type]['Calmar']
        ax.plot(data.index, data['mean'], marker='d', label=det_type,
               color=colors_det.get(det_type, '#000000'), linewidth=2.5, markersize=6)
    ax.set_xlabel("Threshold", fontsize=11, fontweight='bold')
    ax.set_ylabel("Calmar Ratio", fontsize=11, fontweight='bold')
    ax.set_title("Calmar Ratio vs Threshold (by Determinant Type)", fontsize=12, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig("param_study_06_threshold_sensitivity_curves.png", dpi=150, bbox_inches='tight')
    print("\n✓ Saved: param_study_06_threshold_sensitivity_curves.png")
    plt.show()

    # ════════════════════════════════════════════════════════════════════════════════
    # SECTION 5: TOP STRATEGIES RANKING
    # ════════════════════════════════════════════════════════════════════════════════

    print("\n" + "="*110)
    print("TOP 20 STRATEGIES BY SHARPE RATIO")
    print("="*110)

    top_sharpe = study_df_success.nlargest(20, 'Sharpe')[
        ['param_label', 'threshold', 'det_type', 'signal_type',
         'smooth_window', 'min_days_persist', 'frequency', 'CAGR',
         'Volatility', 'Sharpe', 'Max_Drawdown', 'Calmar']
    ]

    for idx, (i, row) in enumerate(top_sharpe.iterrows(), 1):
        print(f"\n{idx:2d}. Threshold: {row['threshold']:4.1f} | Det: {row['det_type']:8s} | "
              f"Signal: {row['signal_type']:10s} | Smooth: {row['smooth_window']:2.0f} | "
              f"Persist: {row['min_days_persist']:2.0f}")
        print(f"    CAGR: {row['CAGR']*100:6.2f}% | Vol: {row['Volatility']*100:6.2f}% | "
              f"Sharpe: {row['Sharpe']:6.3f} | DD: {row['Max_Drawdown']*100:6.2f}% | "
              f"Calmar: {row['Calmar']:6.3f}")

    # ════════════════════════════════════════════════════════════════════════════════
    # SECTION 6: SAVE COMPREHENSIVE RESULTS
    # ════════════════════════════════════════════════════════════════════════════════

    study_df.to_csv("parameter_study_fine_threshold_all_results.csv", index=False)
    study_df_success.to_csv("parameter_study_fine_threshold_successful.csv", index=False)

    # Save threshold sensitivity summary
    threshold_analysis.to_csv("threshold_sensitivity_summary.csv")

    print("\n" + "="*110)
    print("RESULTS SAVED")
    print("="*110)
    print("✓ Saved: parameter_study_fine_threshold_all_results.csv")
    print("✓ Saved: parameter_study_fine_threshold_successful.csv")
    print("✓ Saved: threshold_sensitivity_summary.csv")

    # ── Summary ──
    print("\n" + "="*110)
    print("SUMMARY STATISTICS")
    print("="*110)
    print(f"\nTotal successful combinations: {len(study_df_success)}")
    print(f"Sharpe Ratio range:          {study_df_success['Sharpe'].min():.3f} → {study_df_success['Sharpe'].max():.3f}")
    print(f"Max Drawdown range:          {study_df_success['Max_Drawdown'].min()*100:.2f}% → {study_df_success['Max_Drawdown'].max()*100:.2f}%")
    print(f"CAGR range:                  {study_df_success['CAGR'].min()*100:.2f}% → {study_df_success['CAGR'].max()*100:.2f}%")
    print(f"Calmar ratio range:          {study_df_success['Calmar'].min():.3f} → {study_df_success['Calmar'].max():.3f}")
    print(f"\nOptimal Threshold (avg Sharpe): {optimal_threshold:.1f}")
    print(f"Best Sharpe Threshold: {best_sharpe_threshold:.1f}")

else:
    print("⚠ No successful combinations found!")


In [ ]:
# @title
# ============================================================
# SIGNAL PORTFOLIO — THREE-STAGE THRESHOLD VERSION
# Powers of Det(R-I): Original vs Order 2
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import requests
from io import StringIO

# ── 1. LOAD ALL DATA FROM GITHUB ─────────────────────────────────────────────
GITHUB_BASE = "https://raw.githubusercontent.com/kboroz/MSFE_Capstone_Project/main/01_Data/Streamlined/"

INDEX_FILE_MAP = {
    'ASX50':       'streamlined_asx_50.csv',
    'EUROSTOXX50': 'streamlined_euro_stoxx_50.csv',
    'FTSE100':     'streamlined_ftse_100.csv',
    'Ibovespa':    'streamlined_ibovespa.csv',
    'JSE40':       'streamlined_jse_top_40.csv',
    'NIKKEI225':   'streamlined_nikkei_225.csv',
    'SP500':       'streamlined_s&p_500.csv',
}

MACRO_FILE = 'streamlined_macro.csv'

def load_csv(url):
    r = requests.get(url)
    r.raise_for_status()
    df = pd.read_csv(StringIO(r.text), index_col=0, parse_dates=True)
    df.columns = df.columns.str.replace('[^a-zA-Z0-9]', '_', regex=True)
    return df.sort_index()

# Load macro
macro_df = load_csv(GITHUB_BASE + MACRO_FILE)
macro_df = macro_df[~macro_df.index.duplicated(keep='first')].sort_index()
print(f"macro_df shape : {macro_df.shape}")
print(f"Columns        : {macro_df.columns.tolist()}")

# Load individual index files
index_data = {}
for name, fname in INDEX_FILE_MAP.items():
    try:
        df = load_csv(GITHUB_BASE + fname)
        index_data[name] = df
        print(f"✓ {name:15s} | shape={df.shape} | cols={df.columns.tolist()[:4]}")
    except Exception as e:
        print(f"✗ {name}: {e}")

# ── 2. EXTRACT PRICE SERIES ───────────────────────────────────────────────────
PRICE_COLS = ['Close', 'close', 'Price', 'price', 'Adj_Close', 'Last']

def get_price_series(df, name):
    col = next((c for c in PRICE_COLS if c in df.columns), df.columns[0])
    s   = pd.to_numeric(df[col], errors='coerce').dropna()
    print(f"  {name}: using column '{col}'")
    return s

index_prices = {}
for name, df in index_data.items():
    index_prices[name] = get_price_series(df, name)

# ── 3. MODIFIED DETERMINANT SIGNAL MODEL (WITH POWERS) ────────────────────────
DET_WINDOWS    = [3, 7, 28]
WINDOW_WEIGHTS = {3: 0.20, 7: 0.35, 28: 0.45}
DET_ORDERS     = [1, 2]  # NEW: test both det(R-I)^1 and det(R-I)^2

def compute_determinant_with_powers(price_series_dict, windows=DET_WINDOWS, orders=DET_ORDERS):
    """
    Compute det(R-I) raised to different powers.
    Returns dict: {order: {window: Series}}
    """
    prices = pd.DataFrame(price_series_dict).sort_index()
    rets   = prices.pct_change().dropna()

    results_by_order = {}

    for order in orders:
        det_results = {}

        for window in windows:
            det_series = []
            for i in range(window, len(rets)):
                window_rets = rets.iloc[i - window: i]
                valid_cols  = window_rets.columns[window_rets.std() > 1e-10]
                w           = window_rets[valid_cols]

                if len(valid_cols) < 2:
                    det_series.append(np.nan)
                    continue
                try:
                    R   = w.corr().values
                    I   = np.eye(len(R))
                    M   = R - I
                    det = np.linalg.det(M)

                    # Apply power
                    det_powered = det ** order

                except Exception:
                    det_powered = np.nan
                det_series.append(det_powered)

            idx = rets.index[window:]
            det_results[window] = pd.Series(det_series, index=idx,
                                            name=f'det_order{order}_{window}d')
            print(f"  Order {order} | Window {window:2d}d | computed {len(det_series)} observations")

        results_by_order[order] = det_results

    return results_by_order

print("\n── Computing Determinant with Powers ──")
det_results_by_order = compute_determinant_with_powers(index_prices,
                                                        windows=DET_WINDOWS,
                                                        orders=DET_ORDERS)

# ── 4. COMPOSITE SCORE (FOR EACH ORDER) ────────────────────────────────────────
def compute_composite_score(det_results, order, window_weights=WINDOW_WEIGHTS):
    """
    Returns raw weighted-average of normalised determinant values.
    Normalised per window via rolling z-score so windows are comparable.
    """
    all_series = []
    for window, series in det_results.items():
        # z-score normalise so scale is comparable across windows
        mu    = series.rolling(252, min_periods=60).mean()
        sigma = series.rolling(252, min_periods=60).std()
        z     = (series - mu) / sigma.replace(0, np.nan)
        z.name = f'z_order{order}_{window}d'
        all_series.append(z)

    z_df      = pd.concat(all_series, axis=1).dropna()
    composite = sum(
        z_df[f'z_order{order}_{w}d'] * window_weights[w]
        for w in window_weights if f'z_order{order}_{w}d' in z_df.columns
    )
    composite = composite / sum(window_weights.values())
    composite.name = f'composite_order{order}'
    return composite, z_df

# Compute composite scores for each order
composites_by_order = {}
z_dfs_by_order = {}

for order in DET_ORDERS:
    composite, z_df = compute_composite_score(det_results_by_order[order], order)
    composites_by_order[order] = composite
    z_dfs_by_order[order] = z_df

    print(f"\nOrder {order} composite score shape  : {composite.shape}")
    print(f"Date range                         : {composite.index[0].date()} → "
          f"{composite.index[-1].date()}")
    print(f"Score range                        : {composite.min():.3f} → "
          f"{composite.max():.3f}")

# ── 5. THREE-STAGE SIGNAL CLASSIFICATION ─────────────────────────────────────
THRESHOLDS = [0.3, 0.5, 0.7]   # tune here

def classify_signal(score, threshold):
    """
    Returns +1 (RISK_ON / defensive) or -1 (RISK_OFF / long equity)
    """
    if abs(score) > threshold:
        return +1   # high |det| → abnormal → defensive
    else:
        return -1   # low  |det| → calm     → long equity

def build_signal_df(composite_score, equity_assets, macro_dates, threshold=0.5):
    """
    Apply three-stage classification and reindex to portfolio dates.
    """
    binary = composite_score.apply(lambda x: classify_signal(x, threshold))
    signal_df = pd.DataFrame(index=macro_dates)
    for asset in equity_assets:
        signal_df[asset] = binary.reindex(macro_dates).ffill().bfill()
    return signal_df

# ── 6. ASSET UNIVERSE ─────────────────────────────────────────────────────────
EQUITY_ASSETS = list(index_prices.keys())
SAFE_BONDS    = ['US_10Y', 'UK_10Y', 'AU_10Y', 'ZA_10Y']
FX            = ['AUDUSD', 'BRLUSD', 'EURUSD', 'GBPUSD', 'JPYUSD', 'ZARUSD']
COMMODITIES   = ['Oil', 'Gold', 'Bitcoin']

ALL_ASSETS        = EQUITY_ASSETS + FX + COMMODITIES + SAFE_BONDS
ALL_ASSETS        = [a for a in ALL_ASSETS if a in macro_df.columns]
equity_assets     = [a for a in EQUITY_ASSETS if a in ALL_ASSETS]
non_equity_assets = [a for a in ALL_ASSETS if a not in equity_assets]

N           = len(ALL_ASSETS)
base_weight = 1.0 / N

print(f"\nTotal assets              : {N}")
print(f"Equity (signal-driven)    : {equity_assets}")
print(f"Non-equity (always long)  : {non_equity_assets}")
print(f"base_weight = 1/{N} = {base_weight:.4f}")

# ── 7. COUNTRY SAFE-HAVEN MAP ─────────────────────────────────────────────────
EQUITY_SAFE_HAVEN_MAP = {
    'ASX50':       ['AU_10Y', 'AUDUSD'],
    'EUROSTOXX50': ['EURUSD'],
    'FTSE100':     ['UK_10Y', 'GBPUSD'],
    'Ibovespa':    ['BRLUSD'],
    'JSE40':       ['ZA_10Y', 'ZARUSD'],
    'NIKKEI225':   ['JPYUSD'],
    'SP500':       ['US_10Y'],
}

UNIVERSE_SET = set(ALL_ASSETS)
for eq, havens in EQUITY_SAFE_HAVEN_MAP.items():
    EQUITY_SAFE_HAVEN_MAP[eq] = [h for h in havens if h in UNIVERSE_SET]

print("\nSafe-haven routing (filtered to universe):")
for eq, havens in EQUITY_SAFE_HAVEN_MAP.items():
    print(f"  {eq:15s} → {havens}")

# ── 8. BUILD RETURNS ──────────────────────────────────────────────────────────
returns_raw = macro_df[ALL_ASSETS].pct_change(fill_method=None)

for col in SAFE_BONDS:
    if col in returns_raw.columns:
        returns_raw[col] = -returns_raw[col]   # invert yield → price

if 'Oil' in macro_df.columns:
    bad_oil = macro_df['Oil'][macro_df['Oil'] <= 0].index
    for d in bad_oil:
        loc = returns_raw.index.get_loc(d)
        returns_raw.iloc[
            max(0, loc - 1): loc + 2,
            returns_raw.columns.get_loc('Oil')
        ] = 0.0

returns = returns_raw.clip(lower=-0.50, upper=1.00).fillna(0).iloc[1:]

assert not returns.isnull().any().any(), "NaNs in returns"
assert not np.isinf(returns.values).any(),  "Inf in returns"
print(f"\nReturns shape : {returns.shape}")
print("✓ Returns clean")

# ── 9. THREE-STAGE WEIGHT BUILDER ────────────────────────────────────────────
def build_weights_3stage(sig_row, equity_assets, non_equity_assets,
                         base_weight, date, safe_haven_map):
    """
    RISK_OFF (-1) : long equity at base_weight
    RISK_ON  (+1) : short equity, redistribute proceeds to country safe havens
    """
    raw = {a: base_weight for a in non_equity_assets}
    safe_haven_extra = {}

    for a in equity_assets:
        if sig_row is not None and a in sig_row.index and not pd.isna(sig_row[a]):
            sig = sig_row[a]
        else:
            sig = -1   # default: long equity

        if sig > 0:
            # RISK_ON: short equity, redirect base_weight to safe havens
            raw[a]   = -base_weight
            havens   = safe_haven_map.get(a, [])
            havens   = [h for h in havens if h in raw]

            if havens:
                per_haven = base_weight / len(havens)
                for h in havens:
                    safe_haven_extra[h] = safe_haven_extra.get(h, 0) + per_haven
            else:
                per_ne = base_weight / len(non_equity_assets)
                for ne in non_equity_assets:
                    safe_haven_extra[ne] = safe_haven_extra.get(ne, 0) + per_ne
        else:
            # RISK_OFF: long equity
            raw[a] = base_weight

    for asset, extra in safe_haven_extra.items():
        raw[asset] = raw.get(asset, 0) + extra

    total = sum(raw.values())
    if total <= 0:
        for a in raw:
            raw[a] = 1.0 / len(raw)
    else:
        for a in raw:
            raw[a] /= total

    return raw

# ── 10. EQUAL-WEIGHT BENCHMARK ────────────────────────────────────────────────
def run_equal_weight(returns, all_assets, tc_rate=0.005,
                     rebal_freq='monthly', initial_capital=1e9):
    N           = len(all_assets)
    base_weight = 1.0 / N
    ew          = {a: base_weight for a in all_assets}

    freq_code     = 'MS' if rebal_freq == 'monthly' else 'QS'
    reb_dates_raw = returns.resample(freq_code).first().index
    actual_dates  = returns.index

    reb_set = set()
    for d in reb_dates_raw:
        future = actual_dates[actual_dates >= d]
        if len(future) > 0:
            reb_set.add(future[0])

    pv              = float(initial_capital)
    current_weights = None
    records         = []

    for date in actual_dates:
        if date in reb_set:
            if current_weights is not None:
                turnover = sum(abs(ew.get(a, 0) - current_weights.get(a, 0))
                               for a in set(ew) | set(current_weights)) / 2.0
                pv *= (1 - turnover * tc_rate)
            else:
                pv *= (1 - tc_rate)
            current_weights = ew.copy()

        if current_weights is not None:
            dr  = sum(current_weights.get(a, 0) * returns.loc[date, a]
                      for a in all_assets)
            pv *= (1 + dr)

        records.append({'date': date, 'portfolio_value': pv})

    return pd.DataFrame(records).set_index('date')

# ── 11. THREE-STAGE SIGNAL PORTFOLIO RUNNER ───────────────────────────────────
def run_signal_portfolio_3stage(returns, signal_df, equity_assets,
                                non_equity_assets, all_assets,
                                freq='monthly', tc_rate=0.005,
                                safe_haven_map=None,
                                initial_capital=1e9):

    if safe_haven_map is None:
        safe_haven_map = {}

    N           = len(all_assets)
    base_weight = 1.0 / N

    freq_code     = 'MS' if freq == 'monthly' else 'QS'
    reb_dates_raw = returns.resample(freq_code).first().index
    actual_dates  = returns.index

    reb_set = set()
    for d in reb_dates_raw:
        future = actual_dates[actual_dates >= d]
        if len(future) > 0:
            reb_set.add(future[0])

    sig = signal_df.copy()
    sig.index = pd.to_datetime(sig.index)

    pv              = float(initial_capital)
    current_weights = None
    records         = []

    for date in actual_dates:
        if date in reb_set:
            past    = sig[sig.index <= date]
            sig_row = past.iloc[-1] if len(past) > 0 else None

            raw = build_weights_3stage(
                sig_row, equity_assets, non_equity_assets,
                base_weight, date, safe_haven_map
            )

            if current_weights is not None:
                all_keys = set(raw) | set(current_weights)
                turnover = sum(
                    abs(raw.get(a, 0) - current_weights.get(a, 0))
                    for a in all_keys
                ) / 2.0
                pv *= (1 - turnover * tc_rate)
            else:
                pv *= (1 - tc_rate)

            current_weights = raw.copy()

        if current_weights is not None:
            dr  = sum(current_weights.get(a, 0) * returns.loc[date, a]
                      for a in all_assets)
            pv *= (1 + dr)

        records.append({'date': date, 'portfolio_value': pv})

    return pd.DataFrame(records).set_index('date')

# ── 12. RUN ALL PORTFOLIOS ────────────────────────────────────────────────────
TC_LEVELS = {'0%': 0.00, '0.5%': 0.005, '1%': 0.01, '2%': 0.02}
FREQS     = ['monthly', 'quarterly']

# Equal-weight benchmarks
ew_results = {}
for freq in FREQS:
    for tc_label, tc_rate in TC_LEVELS.items():
        label = f"EW_{freq}_TC{tc_label}"
        ew_results[label] = run_equal_weight(
            returns, ALL_ASSETS, tc_rate=tc_rate,
            rebal_freq=freq, initial_capital=1e9
        )
        print(f"✓ {label}")

# Three-stage signal portfolios across all orders, thresholds
signal_results = {}
signal_distributions = {}

for order in DET_ORDERS:
    composite_score = composites_by_order[order]

    print(f"\n{'='*90}")
    print(f"RUNNING PORTFOLIOS FOR ORDER {order}")
    print(f"{'='*90}")

    for thresh in THRESHOLDS:
        sig_df_t = build_signal_df(composite_score, equity_assets,
                                   macro_df.index, threshold=thresh)

        # Print signal distribution for this threshold
        n_risk_on  = (sig_df_t[equity_assets[0]] == +1).sum()
        n_risk_off = (sig_df_t[equity_assets[0]] == -1).sum()
        total      = len(sig_df_t)
        print(f"\nOrder {order} | Threshold={thresh} | RISK_ON(+1): {n_risk_on} days "
              f"({n_risk_on/total*100:.1f}%) | "
              f"RISK_OFF(-1): {n_risk_off} days ({n_risk_off/total*100:.1f}%)")

        signal_distributions[(order, thresh)] = {'RISK_ON': n_risk_on, 'RISK_OFF': n_risk_off}

        for freq in FREQS:
            for tc_label, tc_rate in TC_LEVELS.items():
                label = f"Signal3S_o{order}_t{thresh}_{freq}_TC{tc_label}"
                signal_results[label] = run_signal_portfolio_3stage(
                    returns, sig_df_t, equity_assets, non_equity_assets,
                    ALL_ASSETS, freq=freq, tc_rate=tc_rate,
                    safe_haven_map=EQUITY_SAFE_HAVEN_MAP,
                    initial_capital=1e9
                )
                print(f"  ✓ {label}")

print("\n✓ All portfolios computed")

# ── 13. PERFORMANCE METRICS ───────────────────────────────────────────────────
def compute_metrics(pv_df, label=''):
    pv        = pv_df['portfolio_value']
    daily_ret = pv.pct_change().dropna()
    n_years   = len(daily_ret) / 252
    total_ret = (pv.iloc[-1] / pv.iloc[0]) - 1
    cagr      = (1 + total_ret) ** (1 / n_years) - 1 if n_years > 0 else np.nan
    vol       = daily_ret.std() * np.sqrt(252)
    sharpe    = cagr / vol if vol > 0 else np.nan
    roll_max  = pv.cummax()
    max_dd    = ((pv - roll_max) / roll_max).min()
    calmar    = cagr / abs(max_dd) if max_dd != 0 else np.nan
    var_95    = daily_ret.quantile(0.05)

    return {
        'CAGR':         f"{cagr*100:.2f}%",
        'Volatility':   f"{vol*100:.2f}%",
        'Sharpe':       f"{sharpe:.2f}",
        'Max Drawdown': f"{max_dd*100:.2f}%",
        'Calmar':       f"{calmar:.2f}",
        'VaR 95%':      f"{var_95*100:.2f}%",
        'Total Return': f"{total_ret*100:.2f}%",
        'Final ($B)':   f"{pv.iloc[-1]/1e9:.3f}",
    }

# Print results grouped by order and threshold
ew_metrics = {k: compute_metrics(v, k) for k, v in ew_results.items()}

print("\n" + "="*90)
print("EQUAL-WEIGHT BENCHMARK")
print("="*90)
print(pd.DataFrame(ew_metrics).T.to_string())

sig_metrics = {k: compute_metrics(v, k) for k, v in signal_results.items()}

for order in DET_ORDERS:
    print(f"\n{'='*110}")
    print(f"THREE-STAGE SIGNAL — ORDER {order} [det(R-I)^{order}]")
    print(f"{'='*110}")

    for thresh in THRESHOLDS:
        dist = signal_distributions[(order, thresh)]
        sub = {k: v for k, v in sig_metrics.items() if f"_o{order}_t{thresh}_" in k}
        print(f"\n{'─'*110}")
        print(f"Threshold = {thresh} | RISK_ON: {dist['RISK_ON']} days | RISK_OFF: {dist['RISK_OFF']} days")
        print(f"{'─'*110}")
        print(pd.DataFrame(sub).T.to_string())

# ── 14. PLOTS ─────────────────────────────────────────────────────────────────
colors_order = {1: 'navy', 2: 'darkred'}
colors_tc   = {'0%': 'black', '0.5%': 'steelblue', '1%': 'darkorange', '2%': 'crimson'}
colors_freq = {'monthly': 'steelblue', 'quarterly': 'darkorange'}

# ── Figure 1: Composite signals by order ──────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(16, 10))
fig.suptitle("Composite Score: Order 1 vs Order 2 [det(R-I)^k]",
             fontsize=14, fontweight='bold')

for idx, order in enumerate(DET_ORDERS):
    ax = axes[idx]
    composite = composites_by_order[order]

    ax.plot(composite.index, composite.values,
            color=colors_order[order], lw=1.0, alpha=0.8,
            label=f'Order {order}')

    for thresh, col in zip(THRESHOLDS, ['green', 'orange', 'red']):
        ax.axhline( thresh, color=col, ls='--', lw=1.0, alpha=0.5)
        ax.axhline(-thresh, color=col, ls='--', lw=1.0, alpha=0.5)

    ax.axhline(0, color='black', lw=0.6)
    ax.fill_between(composite.index, composite.values, 0,
                    where=(composite > 0), color='green', alpha=0.1)
    ax.fill_between(composite.index, composite.values, 0,
                    where=(composite <= 0), color='red', alpha=0.1)

    ax.set_title(f"Order {order}: det(R-I)^{order} (Thresholds: {THRESHOLDS})",
                 fontsize=11)
    ax.set_ylabel("Z-score")
    ax.legend(fontsize=9, ncol=4, loc='upper left')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("composite_signal_orders.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Figure 2: Portfolio values — Order 1 vs 2 (TC=0.5%, monthly, threshold=0.5) ──
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("Portfolio Value: Order 1 vs Order 2 (TC=0.5%, Monthly, Threshold=0.5)",
             fontsize=13, fontweight='bold')

ew_pv = ew_results['EW_monthly_TC0.5%']['portfolio_value']

for ax, order in zip(axes, DET_ORDERS):
    ax.plot(ew_pv.index, ew_pv / 1e9, '--', color='grey',
            lw=1.5, label='EW Benchmark')

    label = f"Signal3S_o{order}_t0.5_monthly_TC0.5%"
    pv    = signal_results[label]['portfolio_value']
    ax.plot(pv.index, pv / 1e9, color=colors_order[order], lw=1.8,
            label=f'Signal (Order {order})')

    ax.axhline(1.0, color='black', ls=':', lw=0.8)
    ax.set_title(f"Order {order}: det(R-I)^{order}", fontsize=11)
    ax.set_ylabel("Value ($B)")
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("signal_orders_comparison.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Figure 3: Drawdowns — Order 1 vs 2 ──────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle("Drawdowns: Order 1 vs Order 2 (TC=0.5%, Monthly, Threshold=0.5)",
             fontsize=13, fontweight='bold')

ew_pv  = ew_results['EW_monthly_TC0.5%']['portfolio_value']
ew_dd  = (ew_pv - ew_pv.cummax()) / ew_pv.cummax() * 100

for ax, order in zip(axes, DET_ORDERS):
    label  = f"Signal3S_o{order}_t0.5_monthly_TC0.5%"
    pv     = signal_results[label]['portfolio_value']
    dd     = (pv - pv.cummax()) / pv.cummax() * 100

    ax.fill_between(dd.index,  dd.values,  0, alpha=0.3, color=colors_order[order])
    ax.fill_between(ew_dd.index, ew_dd.values, 0, alpha=0.2, color='grey')
    ax.plot(dd.index,    dd.values,    color=colors_order[order], lw=1.0,
            label=f'Signal (Order {order})')
    ax.plot(ew_dd.index, ew_dd.values, color='grey',      lw=1.0, ls='--',
            label='EW Benchmark')
    ax.set_title(f"Order {order}: det(R-I)^{order}", fontsize=11)
    ax.set_ylabel("Drawdown (%)")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("signal_orders_drawdowns.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Figure 4: Order comparison — TC sensitivity ────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("TC Sensitivity Comparison: Order 1 vs Order 2 (Monthly, Threshold=0.5)",
             fontsize=13, fontweight='bold')

for ax, order in zip(axes, DET_ORDERS):
    for tc_label, col in colors_tc.items():
        label = f"Signal3S_o{order}_t0.5_monthly_TC{tc_label}"
        pv    = signal_results[label]['portfolio_value']
        ax.plot(pv.index, pv / 1e9, color=col, lw=1.3,
                label=f'TC={tc_label}')

    ew_pv = ew_results['EW_monthly_TC0.5%']['portfolio_value']
    ax.plot(ew_pv.index, ew_pv / 1e9, '--', color='grey',
            lw=1.5, label='EW Benchmark')

    ax.set_title(f"Order {order}: det(R-I)^{order}", fontsize=11)
    ax.set_ylabel("Value ($B)")
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("signal_orders_tc_sensitivity.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Figure 5: Orders × Thresholds heatmap (Sharpe ratio) ─────────────────────
sharpe_matrix = []
for order in DET_ORDERS:
    row = []
    for thresh in THRESHOLDS:
        label = f"Signal3S_o{order}_t{thresh}_monthly_TC0.5%"
        metrics = sig_metrics[label]
        sharpe_str = metrics['Sharpe']
        sharpe_val = float(sharpe_str)
        row.append(sharpe_val)
    sharpe_matrix.append(row)

sharpe_df = pd.DataFrame(sharpe_matrix,
                         index=[f"Order {o}" for o in DET_ORDERS],
                         columns=[f"t={t}" for t in THRESHOLDS])

fig, ax = plt.subplots(figsize=(8, 5))
im = ax.imshow(sharpe_df.values, cmap='RdYlGn', aspect='auto')

ax.set_xticks(np.arange(len(sharpe_df.columns)))
ax.set_yticks(np.arange(len(sharpe_df.index)))
ax.set_xticklabels(sharpe_df.columns)
ax.set_yticklabels(sharpe_df.index)

for i in range(len(sharpe_df.index)):
    for j in range(len(sharpe_df.columns)):
        text = ax.text(j, i, f"{sharpe_df.iloc[i, j]:.2f}",
                      ha="center", va="center", color="black", fontweight='bold')

ax.set_title("Sharpe Ratio: Orders × Thresholds (Monthly, TC=0.5%)",
             fontsize=12, fontweight='bold')
cbar = plt.colorbar(im, ax=ax)
cbar.set_label('Sharpe Ratio', rotation=270, labelpad=20)

plt.tight_layout()
plt.savefig("signal_orders_sharpe_heatmap.png", dpi=150, bbox_inches='tight')
plt.show()

# ── 15. SAVE ──────────────────────────────────────────────────────────────────
for label, df in {**ew_results, **signal_results}.items():
    df.to_csv(f"{label}_portfolio.csv")

all_metrics = pd.DataFrame({**ew_metrics, **sig_metrics}).T
all_metrics.to_csv("all_portfolio_metrics.csv")

for order in DET_ORDERS:
    composites_by_order[order].to_csv(f"composite_signal_order{order}.csv")

    for thresh in THRESHOLDS:
        sig_df_t = build_signal_df(composites_by_order[order], equity_assets,
                                   macro_df.index, threshold=thresh)
        sig_df_t.to_csv(f"signal_df_order{order}_threshold_{thresh}.csv")

print("\n✓ All files saved.")


In [ ]:
# @title
# ============================================================
# SIGNAL PORTFOLIO — THREE-STAGE THRESHOLD VERSION
# Signals generated from Modified Determinant Model
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import requests
from io import StringIO

# ── 1. LOAD ALL DATA FROM GITHUB ─────────────────────────────────────────────
GITHUB_BASE = "https://raw.githubusercontent.com/kboroz/MSFE_Capstone_Project/main/01_Data/Streamlined/"

INDEX_FILE_MAP = {
    'ASX50':       'streamlined_asx_50.csv',
    'EUROSTOXX50': 'streamlined_euro_stoxx_50.csv',
    'FTSE100':     'streamlined_ftse_100.csv',
    'Ibovespa':    'streamlined_ibovespa.csv',
    'JSE40':       'streamlined_jse_top_40.csv',
    'NIKKEI225':   'streamlined_nikkei_225.csv',
    'SP500':       'streamlined_s&p_500.csv',
}

MACRO_FILE = 'streamlined_macro.csv'

def load_csv(url):
    r = requests.get(url)
    r.raise_for_status()
    df = pd.read_csv(StringIO(r.text), index_col=0, parse_dates=True)
    df.columns = df.columns.str.replace('[^a-zA-Z0-9]', '_', regex=True)
    return df.sort_index()

# Load macro
macro_df = load_csv(GITHUB_BASE + MACRO_FILE)
macro_df = macro_df[~macro_df.index.duplicated(keep='first')].sort_index()
print(f"macro_df shape : {macro_df.shape}")
print(f"Columns        : {macro_df.columns.tolist()}")

# Load individual index files
index_data = {}
for name, fname in INDEX_FILE_MAP.items():
    try:
        df = load_csv(GITHUB_BASE + fname)
        index_data[name] = df
        print(f"✓ {name:15s} | shape={df.shape} | cols={df.columns.tolist()[:4]}")
    except Exception as e:
        print(f"✗ {name}: {e}")

# ── 2. EXTRACT PRICE SERIES ───────────────────────────────────────────────────
PRICE_COLS = ['Close', 'close', 'Price', 'price', 'Adj_Close', 'Last']

def get_price_series(df, name):
    col = next((c for c in PRICE_COLS if c in df.columns), df.columns[0])
    s   = pd.to_numeric(df[col], errors='coerce').dropna()
    print(f"  {name}: using column '{col}'")
    return s

index_prices = {}
for name, df in index_data.items():
    index_prices[name] = get_price_series(df, name)

# ── 3. MODIFIED DETERMINANT SIGNAL MODEL ─────────────────────────────────────
DET_WINDOWS    = [3, 7, 28]
WINDOW_WEIGHTS = {3: 0.20, 7: 0.35, 28: 0.45}

def compute_modified_determinant(price_series_dict, windows=DET_WINDOWS):
    prices = pd.DataFrame(price_series_dict).sort_index()
    rets   = prices.pct_change().dropna()
    det_results = {}

    for window in windows:
        det_series = []
        for i in range(window, len(rets)):
            window_rets = rets.iloc[i - window: i]
            valid_cols  = window_rets.columns[window_rets.std() > 1e-10]
            w           = window_rets[valid_cols]

            if len(valid_cols) < 2:
                det_series.append(np.nan)
                continue
            try:
                R   = w.corr().values
                I   = np.eye(len(R))
                M   = R - I
                det = np.linalg.det(M)
            except Exception:
                det = np.nan
            det_series.append(det)

        idx = rets.index[window:]
        det_results[window] = pd.Series(det_series, index=idx, name=f'det_{window}d')
        print(f"  Window {window:2d}d | computed {len(det_series)} observations")

    return det_results

print("\n── Computing Modified Determinant ──")
det_results = compute_modified_determinant(index_prices)

# ── 4. COMPOSITE SCORE ────────────────────────────────────────────────────────
def compute_composite_score(det_results, window_weights=WINDOW_WEIGHTS):
    """
    Returns raw weighted-average of normalised determinant values (not signs).
    Normalised per window via rolling z-score so windows are comparable.
    """
    all_series = []
    for window, series in det_results.items():
        # z-score normalise so scale is comparable across windows
        mu    = series.rolling(252, min_periods=60).mean()
        sigma = series.rolling(252, min_periods=60).std()
        z     = (series - mu) / sigma.replace(0, np.nan)
        z.name = f'z_{window}d'
        all_series.append(z)

    z_df      = pd.concat(all_series, axis=1).dropna()
    composite = sum(
        z_df[f'z_{w}d'] * window_weights[w]
        for w in window_weights if f'z_{w}d' in z_df.columns
    )
    composite = composite / sum(window_weights.values())
    return composite, z_df

composite_score, z_df = compute_composite_score(det_results)

print(f"\nComposite score shape  : {composite_score.shape}")
print(f"Date range             : {composite_score.index[0].date()} → "
      f"{composite_score.index[-1].date()}")
print(f"Score range            : {composite_score.min():.3f} → "
      f"{composite_score.max():.3f}")

# ── 5. THREE-STAGE SIGNAL CLASSIFICATION ─────────────────────────────────────
# |score| > threshold  → RISK_ON  (+1): high det magnitude = abnormal regime
#                         → short equity, long country bonds + FX
# |score| <= threshold → RISK_OFF (-1): low det magnitude  = calm regime
#                         → long equity

THRESHOLDS = [0.3, 0.5, 0.7]   # tune here

def classify_signal(score, threshold):
    """
    Returns +1 (RISK_ON / defensive) or -1 (RISK_OFF / long equity)
    """
    if abs(score) > threshold:
        return +1   # high |det| → abnormal → defensive
    else:
        return -1   # low  |det| → calm     → long equity

def build_signal_df(composite_score, equity_assets, macro_dates, threshold=0.5):
    """
    Apply three-stage classification and reindex to portfolio dates.
    """
    binary = composite_score.apply(lambda x: classify_signal(x, threshold))
    signal_df = pd.DataFrame(index=macro_dates)
    for asset in equity_assets:
        signal_df[asset] = binary.reindex(macro_dates).ffill().bfill()
    return signal_df

# ── 6. ASSET UNIVERSE ─────────────────────────────────────────────────────────
EQUITY_ASSETS = list(index_prices.keys())
SAFE_BONDS    = ['US_10Y', 'UK_10Y', 'AU_10Y', 'ZA_10Y']
FX            = ['AUDUSD', 'BRLUSD', 'EURUSD', 'GBPUSD', 'JPYUSD', 'ZARUSD']
COMMODITIES   = ['Oil', 'Gold', 'Bitcoin']

ALL_ASSETS        = EQUITY_ASSETS + FX + COMMODITIES + SAFE_BONDS
ALL_ASSETS        = [a for a in ALL_ASSETS if a in macro_df.columns]
equity_assets     = [a for a in EQUITY_ASSETS if a in ALL_ASSETS]
non_equity_assets = [a for a in ALL_ASSETS if a not in equity_assets]

N           = len(ALL_ASSETS)
base_weight = 1.0 / N

print(f"\nTotal assets              : {N}")
print(f"Equity (signal-driven)    : {equity_assets}")
print(f"Non-equity (always long)  : {non_equity_assets}")
print(f"base_weight = 1/{N} = {base_weight:.4f}")

# ── 7. COUNTRY SAFE-HAVEN MAP ─────────────────────────────────────────────────
# When an equity index is shorted (RISK_ON), proceeds are redistributed
# to that country's bond + FX where available in the universe

EQUITY_SAFE_HAVEN_MAP = {
    'ASX50':       ['AU_10Y', 'AUDUSD'],
    'EUROSTOXX50': ['EURUSD'],            # no EU bond in universe
    'FTSE100':     ['UK_10Y', 'GBPUSD'],
    'Ibovespa':    ['BRLUSD'],            # no Brazil bond in universe
    'JSE40':       ['ZA_10Y', 'ZARUSD'],
    'NIKKEI225':   ['JPYUSD'],            # no JP bond in universe
    'SP500':       ['US_10Y'],            # USD is base currency
}

# Filter to assets actually in universe
UNIVERSE_SET = set(ALL_ASSETS)
for eq, havens in EQUITY_SAFE_HAVEN_MAP.items():
    EQUITY_SAFE_HAVEN_MAP[eq] = [h for h in havens if h in UNIVERSE_SET]

print("\nSafe-haven routing (filtered to universe):")
for eq, havens in EQUITY_SAFE_HAVEN_MAP.items():
    print(f"  {eq:15s} → {havens}")

# ── 8. BUILD RETURNS ──────────────────────────────────────────────────────────
returns_raw = macro_df[ALL_ASSETS].pct_change(fill_method=None)

for col in SAFE_BONDS:
    if col in returns_raw.columns:
        returns_raw[col] = -returns_raw[col]   # invert yield → price

if 'Oil' in macro_df.columns:
    bad_oil = macro_df['Oil'][macro_df['Oil'] <= 0].index
    for d in bad_oil:
        loc = returns_raw.index.get_loc(d)
        returns_raw.iloc[
            max(0, loc - 1): loc + 2,
            returns_raw.columns.get_loc('Oil')
        ] = 0.0

returns = returns_raw.clip(lower=-0.50, upper=1.00).fillna(0).iloc[1:]

assert not returns.isnull().any().any(), "NaNs in returns"
assert not np.isinf(returns.values).any(),  "Inf in returns"
print(f"\nReturns shape : {returns.shape}")
print("✓ Returns clean")

# ── 9. THREE-STAGE WEIGHT BUILDER ────────────────────────────────────────────
def build_weights_3stage(sig_row, equity_assets, non_equity_assets,
                         base_weight, date, safe_haven_map):
    """
    RISK_OFF (-1) : long equity at base_weight
    RISK_ON  (+1) : short equity, redistribute proceeds to country safe havens
                    or equally across all non-equity if no safe haven available
    Non-equity assets always start at base_weight; safe-haven assets receive
    additional weight from shorted equities.
    """
    raw = {a: base_weight for a in non_equity_assets}

    safe_haven_extra = {}   # accumulate extra weight per safe-haven asset

    for a in equity_assets:
        if sig_row is not None and a in sig_row.index and not pd.isna(sig_row[a]):
            sig = sig_row[a]
        else:
            sig = -1   # default: long equity

        if sig > 0:
            # RISK_ON: short equity, redirect base_weight to safe havens
            raw[a]   = -base_weight
            havens   = safe_haven_map.get(a, [])
            havens   = [h for h in havens if h in raw]   # only if in universe

            if havens:
                per_haven = base_weight / len(havens)
                for h in havens:
                    safe_haven_extra[h] = safe_haven_extra.get(h, 0) + per_haven
            else:
                # No safe haven available → spread across all non-equity
                per_ne = base_weight / len(non_equity_assets)
                for ne in non_equity_assets:
                    safe_haven_extra[ne] = safe_haven_extra.get(ne, 0) + per_ne
        else:
            # RISK_OFF: long equity
            raw[a] = base_weight

    # Apply safe-haven extras
    for asset, extra in safe_haven_extra.items():
        raw[asset] = raw.get(asset, 0) + extra

    # Normalise to sum = 1 (handles edge cases)
    total = sum(raw.values())
    if total <= 0:
        for a in raw:
            raw[a] = 1.0 / len(raw)
    else:
        for a in raw:
            raw[a] /= total

    return raw

# ── 10. EQUAL-WEIGHT BENCHMARK ────────────────────────────────────────────────
def run_equal_weight(returns, all_assets, tc_rate=0.005,
                     rebal_freq='monthly', initial_capital=1e9):
    N           = len(all_assets)
    base_weight = 1.0 / N
    ew          = {a: base_weight for a in all_assets}

    freq_code     = 'MS' if rebal_freq == 'monthly' else 'QS'
    reb_dates_raw = returns.resample(freq_code).first().index
    actual_dates  = returns.index

    reb_set = set()
    for d in reb_dates_raw:
        future = actual_dates[actual_dates >= d]
        if len(future) > 0:
            reb_set.add(future[0])

    pv              = float(initial_capital)
    current_weights = None
    records         = []

    for date in actual_dates:
        if date in reb_set:
            if current_weights is not None:
                turnover = sum(abs(ew.get(a, 0) - current_weights.get(a, 0))
                               for a in set(ew) | set(current_weights)) / 2.0
                pv *= (1 - turnover * tc_rate)
            else:
                pv *= (1 - tc_rate)
            current_weights = ew.copy()

        if current_weights is not None:
            dr  = sum(current_weights.get(a, 0) * returns.loc[date, a]
                      for a in all_assets)
            pv *= (1 + dr)

        records.append({'date': date, 'portfolio_value': pv})

    return pd.DataFrame(records).set_index('date')

# ── 11. THREE-STAGE SIGNAL PORTFOLIO RUNNER ───────────────────────────────────
def run_signal_portfolio_3stage(returns, signal_df, equity_assets,
                                non_equity_assets, all_assets,
                                freq='monthly', tc_rate=0.005,
                                safe_haven_map=None,
                                initial_capital=1e9):

    if safe_haven_map is None:
        safe_haven_map = {}

    N           = len(all_assets)
    base_weight = 1.0 / N

    freq_code     = 'MS' if freq == 'monthly' else 'QS'
    reb_dates_raw = returns.resample(freq_code).first().index
    actual_dates  = returns.index

    reb_set = set()
    for d in reb_dates_raw:
        future = actual_dates[actual_dates >= d]
        if len(future) > 0:
            reb_set.add(future[0])

    sig = signal_df.copy()
    sig.index = pd.to_datetime(sig.index)

    pv              = float(initial_capital)
    current_weights = None
    records         = []

    for date in actual_dates:
        if date in reb_set:
            past    = sig[sig.index <= date]
            sig_row = past.iloc[-1] if len(past) > 0 else None

            raw = build_weights_3stage(
                sig_row, equity_assets, non_equity_assets,
                base_weight, date, safe_haven_map
            )

            if current_weights is not None:
                all_keys = set(raw) | set(current_weights)
                turnover = sum(
                    abs(raw.get(a, 0) - current_weights.get(a, 0))
                    for a in all_keys
                ) / 2.0
                pv *= (1 - turnover * tc_rate)
            else:
                pv *= (1 - tc_rate)

            current_weights = raw.copy()

        if current_weights is not None:
            dr  = sum(current_weights.get(a, 0) * returns.loc[date, a]
                      for a in all_assets)
            pv *= (1 + dr)

        records.append({'date': date, 'portfolio_value': pv})

    return pd.DataFrame(records).set_index('date')

# ── 12. RUN ALL PORTFOLIOS ────────────────────────────────────────────────────
TC_LEVELS = {'0%': 0.00, '0.5%': 0.005, '1%': 0.01, '2%': 0.02}
FREQS     = ['monthly', 'quarterly']

# Equal-weight benchmarks
ew_results = {}
for freq in FREQS:
    for tc_label, tc_rate in TC_LEVELS.items():
        label = f"EW_{freq}_TC{tc_label}"
        ew_results[label] = run_equal_weight(
            returns, ALL_ASSETS, tc_rate=tc_rate,
            rebal_freq=freq, initial_capital=1e9
        )
        print(f"✓ {label}")

# Three-stage signal portfolios across all thresholds
signal_results = {}
signal_distributions = {}

for thresh in THRESHOLDS:
    sig_df_t = build_signal_df(composite_score, equity_assets,
                               macro_df.index, threshold=thresh)

    # Print signal distribution for this threshold
    n_risk_on  = (sig_df_t[equity_assets[0]] == +1).sum()
    n_risk_off = (sig_df_t[equity_assets[0]] == -1).sum()
    total      = len(sig_df_t)
    print(f"\nThreshold={thresh} | RISK_ON(+1): {n_risk_on} days "
          f"({n_risk_on/total*100:.1f}%) | "
          f"RISK_OFF(-1): {n_risk_off} days ({n_risk_off/total*100:.1f}%)")

    signal_distributions[thresh] = {'RISK_ON': n_risk_on, 'RISK_OFF': n_risk_off}

    for freq in FREQS:
        for tc_label, tc_rate in TC_LEVELS.items():
            label = f"Signal3S_t{thresh}_{freq}_TC{tc_label}"
            signal_results[label] = run_signal_portfolio_3stage(
                returns, sig_df_t, equity_assets, non_equity_assets,
                ALL_ASSETS, freq=freq, tc_rate=tc_rate,
                safe_haven_map=EQUITY_SAFE_HAVEN_MAP,
                initial_capital=1e9
            )
            print(f"  ✓ {label}")

print("\n✓ All portfolios computed")

# ── 13. PERFORMANCE METRICS ───────────────────────────────────────────────────
def compute_metrics(pv_df, label=''):
    pv        = pv_df['portfolio_value']
    daily_ret = pv.pct_change().dropna()
    n_years   = len(daily_ret) / 252
    total_ret = (pv.iloc[-1] / pv.iloc[0]) - 1
    cagr      = (1 + total_ret) ** (1 / n_years) - 1 if n_years > 0 else np.nan
    vol       = daily_ret.std() * np.sqrt(252)
    sharpe    = cagr / vol if vol > 0 else np.nan
    roll_max  = pv.cummax()
    max_dd    = ((pv - roll_max) / roll_max).min()
    calmar    = cagr / abs(max_dd) if max_dd != 0 else np.nan
    var_95    = daily_ret.quantile(0.05)

    return {
        'CAGR':         f"{cagr*100:.2f}%",
        'Volatility':   f"{vol*100:.2f}%",
        'Sharpe':       f"{sharpe:.2f}",
        'Max Drawdown': f"{max_dd*100:.2f}%",
        'Calmar':       f"{calmar:.2f}",
        'VaR 95%':      f"{var_95*100:.2f}%",
        'Total Return': f"{total_ret*100:.2f}%",
        'Final ($B)':   f"{pv.iloc[-1]/1e9:.3f}",
    }

# Print results grouped by threshold
ew_metrics = {k: compute_metrics(v, k) for k, v in ew_results.items()}

print("\n" + "="*90)
print("EQUAL-WEIGHT BENCHMARK")
print("="*90)
print(pd.DataFrame(ew_metrics).T.to_string())

sig_metrics = {k: compute_metrics(v, k) for k, v in signal_results.items()}

for thresh in THRESHOLDS:
    sub = {k: v for k, v in sig_metrics.items() if f"_t{thresh}_" in k}
    print(f"\n{'='*90}")
    print(f"THREE-STAGE SIGNAL — Threshold = {thresh} | "
          f"RISK_ON: {signal_distributions[thresh]['RISK_ON']} days | "
          f"RISK_OFF: {signal_distributions[thresh]['RISK_OFF']} days")
    print("="*90)
    print(pd.DataFrame(sub).T.to_string())

# ── 14. PLOTS ─────────────────────────────────────────────────────────────────
# One figure per threshold + one overview figure

colors_tc   = {'0%': 'black', '0.5%': 'steelblue', '1%': 'darkorange', '2%': 'crimson'}
colors_freq = {'monthly': 'steelblue', 'quarterly': 'darkorange'}

# ── Figure 1: Composite signal + safe-haven routing overview ──────────────────
fig, axes = plt.subplots(2, 1, figsize=(16, 10))
fig.suptitle("Modified Determinant — Composite Score & Threshold Signals",
             fontsize=14, fontweight='bold')

ax = axes[0]
ax.plot(composite_score.index, composite_score.values,
        color='navy', lw=1.0, alpha=0.8, label='Composite Z-score')
for thresh, col in zip(THRESHOLDS, ['green', 'orange', 'red']):
    ax.axhline( thresh, color=col, ls='--', lw=1.0, label=f'+{thresh}')
    ax.axhline(-thresh, color=col, ls='--', lw=1.0, label=f'-{thresh}')
ax.axhline(0, color='black', lw=0.6)
ax.fill_between(composite_score.index, composite_score.values, 0,
                where=(composite_score > 0), color='green', alpha=0.1)
ax.fill_between(composite_score.index, composite_score.values, 0,
                where=(composite_score <= 0), color='red', alpha=0.1)
ax.set_title("Composite Determinant Score with Thresholds", fontsize=11)
ax.set_ylabel("Z-score")
ax.legend(fontsize=8, ncol=4)
ax.grid(True, alpha=0.3)

ax = axes[1]
for window, series in det_results.items():
    ax.plot(series.index, np.sign(series.values), alpha=0.6,
            label=f"{window}d window", lw=1.0)
ax.axhline(0, color='black', lw=0.8)
ax.set_title("Sign of Det(R-I) per Window", fontsize=11)
ax.set_ylabel("+1 / -1")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("composite_signal_thresholds.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Figure 2: Portfolio values per threshold (TC=0.5%, monthly) ───────────────
fig, axes = plt.subplots(1, 3, figsize=(20, 6), sharey=False)
fig.suptitle("Three-Stage Signal Portfolio — Value by Threshold (TC=0.5%, Monthly)",
             fontsize=13, fontweight='bold')

for ax, thresh in zip(axes, THRESHOLDS):
    # EW reference
    ew_pv = ew_results['EW_monthly_TC0.5%']['portfolio_value']
    ax.plot(ew_pv.index, ew_pv / 1e9, '--', color='grey',
            lw=1.5, label='EW Benchmark')

    label = f"Signal3S_t{thresh}_monthly_TC0.5%"
    pv    = signal_results[label]['portfolio_value']
    ax.plot(pv.index, pv / 1e9, color='steelblue', lw=1.8,
            label=f'Signal (t={thresh})')

    ax.axhline(1.0, color='black', ls=':', lw=0.8)
    ax.set_title(f"Threshold = {thresh}", fontsize=11)
    ax.set_ylabel("Value ($B)")
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("signal_3stage_by_threshold.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Figure 3: Drawdowns per threshold ────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(20, 5), sharey=True)
fig.suptitle("Three-Stage Signal Portfolio — Drawdowns by Threshold (TC=0.5%, Monthly)",
             fontsize=13, fontweight='bold')

for ax, thresh in zip(axes, THRESHOLDS):
    label  = f"Signal3S_t{thresh}_monthly_TC0.5%"
    pv     = signal_results[label]['portfolio_value']
    dd     = (pv - pv.cummax()) / pv.cummax() * 100
    ew_pv  = ew_results['EW_monthly_TC0.5%']['portfolio_value']
    ew_dd  = (ew_pv - ew_pv.cummax()) / ew_pv.cummax() * 100

    ax.fill_between(dd.index,  dd.values,  0, alpha=0.3, color='steelblue')
    ax.fill_between(ew_dd.index, ew_dd.values, 0, alpha=0.2, color='grey')
    ax.plot(dd.index,    dd.values,    color='steelblue', lw=1.0,
            label=f'Signal (t={thresh})')
    ax.plot(ew_dd.index, ew_dd.values, color='grey',      lw=1.0, ls='--',
            label='EW Benchmark')
    ax.set_title(f"Threshold = {thresh}", fontsize=11)
    ax.set_ylabel("Drawdown (%)")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("signal_3stage_drawdowns.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Figure 4: TC sensitivity per threshold ────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.suptitle("Three-Stage Signal — TC Sensitivity (Monthly)",
             fontsize=13, fontweight='bold')

for ax, thresh in zip(axes, THRESHOLDS):
    for tc_label, col in colors_tc.items():
        label = f"Signal3S_t{thresh}_monthly_TC{tc_label}"
        pv    = signal_results[label]['portfolio_value']
        ax.plot(pv.index, pv / 1e9, color=col, lw=1.3,
                label=f'TC={tc_label}')
    ew_pv = ew_results['EW_monthly_TC0.5%']['portfolio_value']
    ax.plot(ew_pv.index, ew_pv / 1e9, '--', color='grey',
            lw=1.5, label='EW Benchmark')
    ax.set_title(f"Threshold = {thresh}", fontsize=11)
    ax.set_ylabel("Value ($B)")
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("signal_3stage_tc_sensitivity.png", dpi=150, bbox_inches='tight')
plt.show()

# ── 15. SAVE ──────────────────────────────────────────────────────────────────
for label, df in {**ew_results, **signal_results}.items():
    df.to_csv(f"{label}_portfolio.csv")

all_metrics = pd.DataFrame({**ew_metrics, **sig_metrics}).T
all_metrics.to_csv("all_portfolio_metrics.csv")

composite_score.to_csv("composite_signal.csv")

for thresh in THRESHOLDS:
    sig_df_t = build_signal_df(composite_score, equity_assets,
                               macro_df.index, threshold=thresh)
    sig_df_t.to_csv(f"signal_df_threshold_{thresh}.csv")

print("✓ All files saved.")


In [ ]:
# @title
# ============================================================
# SIGNAL PORTFOLIO — THREE-STAGE THRESHOLD VERSION
# Signals generated from Modified Determinant Model
# WITH INVERTED SIGNAL VARIANT
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import requests
from io import StringIO

# ── 1. LOAD ALL DATA FROM GITHUB ─────────────────────────────────────────────
GITHUB_BASE = "https://raw.githubusercontent.com/kboroz/MSFE_Capstone_Project/main/01_Data/Streamlined/"

INDEX_FILE_MAP = {
    'ASX50':       'streamlined_asx_50.csv',
    'EUROSTOXX50': 'streamlined_euro_stoxx_50.csv',
    'FTSE100':     'streamlined_ftse_100.csv',
    'Ibovespa':    'streamlined_ibovespa.csv',
    'JSE40':       'streamlined_jse_top_40.csv',
    'NIKKEI225':   'streamlined_nikkei_225.csv',
    'SP500':       'streamlined_s&p_500.csv',
}

MACRO_FILE = 'streamlined_macro.csv'

def load_csv(url):
    r = requests.get(url)
    r.raise_for_status()
    df = pd.read_csv(StringIO(r.text), index_col=0, parse_dates=True)
    df.columns = df.columns.str.replace('[^a-zA-Z0-9]', '_', regex=True)
    return df.sort_index()

# Load macro
macro_df = load_csv(GITHUB_BASE + MACRO_FILE)
macro_df = macro_df[~macro_df.index.duplicated(keep='first')].sort_index()
print(f"macro_df shape : {macro_df.shape}")
print(f"Columns        : {macro_df.columns.tolist()}")

# Load individual index files
index_data = {}
for name, fname in INDEX_FILE_MAP.items():
    try:
        df = load_csv(GITHUB_BASE + fname)
        index_data[name] = df
        print(f"✓ {name:15s} | shape={df.shape} | cols={df.columns.tolist()[:4]}")
    except Exception as e:
        print(f"✗ {name}: {e}")

# ── 2. EXTRACT PRICE SERIES ───────────────────────────────────────────────────
PRICE_COLS = ['Close', 'close', 'Price', 'price', 'Adj_Close', 'Last']

def get_price_series(df, name):
    col = next((c for c in PRICE_COLS if c in df.columns), df.columns[0])
    s   = pd.to_numeric(df[col], errors='coerce').dropna()
    print(f"  {name}: using column '{col}'")
    return s

index_prices = {}
for name, df in index_data.items():
    index_prices[name] = get_price_series(df, name)

# ── 3. MODIFIED DETERMINANT SIGNAL MODEL ─────────────────────────────────────
DET_WINDOWS    = [3, 7, 28]
WINDOW_WEIGHTS = {3: 0.20, 7: 0.35, 28: 0.45}

def compute_modified_determinant(price_series_dict, windows=DET_WINDOWS):
    prices = pd.DataFrame(price_series_dict).sort_index()
    rets   = prices.pct_change().dropna()
    det_results = {}

    for window in windows:
        det_series = []
        for i in range(window, len(rets)):
            window_rets = rets.iloc[i - window: i]
            valid_cols  = window_rets.columns[window_rets.std() > 1e-10]
            w           = window_rets[valid_cols]

            if len(valid_cols) < 2:
                det_series.append(np.nan)
                continue
            try:
                R   = w.corr().values
                I   = np.eye(len(R))
                M   = R - I
                det = np.linalg.det(M)
            except Exception:
                det = np.nan
            det_series.append(det)

        idx = rets.index[window:]
        det_results[window] = pd.Series(det_series, index=idx, name=f'det_{window}d')
        print(f"  Window {window:2d}d | computed {len(det_series)} observations")

    return det_results

print("\n── Computing Modified Determinant ──")
det_results = compute_modified_determinant(index_prices)

# ── 4. COMPOSITE SCORE ────────────────────────────────────────────────────────
def compute_composite_score(det_results, window_weights=WINDOW_WEIGHTS):
    """
    Returns raw weighted-average of normalised determinant values (not signs).
    Normalised per window via rolling z-score so windows are comparable.
    """
    all_series = []
    for window, series in det_results.items():
        # z-score normalise so scale is comparable across windows
        mu    = series.rolling(252, min_periods=60).mean()
        sigma = series.rolling(252, min_periods=60).std()
        z     = (series - mu) / sigma.replace(0, np.nan)
        z.name = f'z_{window}d'
        all_series.append(z)

    z_df      = pd.concat(all_series, axis=1).dropna()
    composite = sum(
        z_df[f'z_{w}d'] * window_weights[w]
        for w in window_weights if f'z_{w}d' in z_df.columns
    )
    composite = composite / sum(window_weights.values())
    return composite, z_df

composite_score, z_df = compute_composite_score(det_results)

print(f"\nComposite score shape  : {composite_score.shape}")
print(f"Date range             : {composite_score.index[0].date()} → "
      f"{composite_score.index[-1].date()}")
print(f"Score range            : {composite_score.min():.3f} → "
      f"{composite_score.max():.3f}")

# ── 5. THREE-STAGE SIGNAL CLASSIFICATION ─────────────────────────────────────
# |score| > threshold  → RISK_ON  (+1): high det magnitude = abnormal regime
#                         → short equity, long country bonds + FX
# |score| <= threshold → RISK_OFF (-1): low det magnitude  = calm regime
#                         → long equity

THRESHOLDS = [0.3, 0.5, 0.7]   # tune here

def classify_signal(score, threshold, inverted=False):
    """
    Returns +1 (RISK_ON / defensive) or -1 (RISK_OFF / long equity)
    If inverted=True, flips the decision.
    """
    base_signal = +1 if abs(score) > threshold else -1

    if inverted:
        return -base_signal  # flip the signal
    else:
        return base_signal

def build_signal_df(composite_score, equity_assets, macro_dates, threshold=0.5, inverted=False):
    """
    Apply three-stage classification and reindex to portfolio dates.
    If inverted=True, takes opposite positions.
    """
    binary = composite_score.apply(lambda x: classify_signal(x, threshold, inverted=inverted))
    signal_df = pd.DataFrame(index=macro_dates)
    for asset in equity_assets:
        signal_df[asset] = binary.reindex(macro_dates).ffill().bfill()
    return signal_df

# ── 6. ASSET UNIVERSE ─────────────────────────────────────────────────────────
EQUITY_ASSETS = list(index_prices.keys())
SAFE_BONDS    = ['US_10Y', 'UK_10Y', 'AU_10Y', 'ZA_10Y']
FX            = ['AUDUSD', 'BRLUSD', 'EURUSD', 'GBPUSD', 'JPYUSD', 'ZARUSD']
COMMODITIES   = ['Oil', 'Gold', 'Bitcoin']

ALL_ASSETS        = EQUITY_ASSETS + FX + COMMODITIES + SAFE_BONDS
ALL_ASSETS        = [a for a in ALL_ASSETS if a in macro_df.columns]
equity_assets     = [a for a in EQUITY_ASSETS if a in ALL_ASSETS]
non_equity_assets = [a for a in ALL_ASSETS if a not in equity_assets]

N           = len(ALL_ASSETS)
base_weight = 1.0 / N

print(f"\nTotal assets              : {N}")
print(f"Equity (signal-driven)    : {equity_assets}")
print(f"Non-equity (always long)  : {non_equity_assets}")
print(f"base_weight = 1/{N} = {base_weight:.4f}")

# ── 7. COUNTRY SAFE-HAVEN MAP ─────────────────────────────────────────────────
EQUITY_SAFE_HAVEN_MAP = {
    'ASX50':       ['AU_10Y', 'AUDUSD'],
    'EUROSTOXX50': ['EURUSD'],
    'FTSE100':     ['UK_10Y', 'GBPUSD'],
    'Ibovespa':    ['BRLUSD'],
    'JSE40':       ['ZA_10Y', 'ZARUSD'],
    'NIKKEI225':   ['JPYUSD'],
    'SP500':       ['US_10Y'],
}

UNIVERSE_SET = set(ALL_ASSETS)
for eq, havens in EQUITY_SAFE_HAVEN_MAP.items():
    EQUITY_SAFE_HAVEN_MAP[eq] = [h for h in havens if h in UNIVERSE_SET]

print("\nSafe-haven routing (filtered to universe):")
for eq, havens in EQUITY_SAFE_HAVEN_MAP.items():
    print(f"  {eq:15s} → {havens}")

# ── 8. BUILD RETURNS ──────────────────────────────────────────────────────────
returns_raw = macro_df[ALL_ASSETS].pct_change(fill_method=None)

for col in SAFE_BONDS:
    if col in returns_raw.columns:
        returns_raw[col] = -returns_raw[col]   # invert yield → price

if 'Oil' in macro_df.columns:
    bad_oil = macro_df['Oil'][macro_df['Oil'] <= 0].index
    for d in bad_oil:
        loc = returns_raw.index.get_loc(d)
        returns_raw.iloc[
            max(0, loc - 1): loc + 2,
            returns_raw.columns.get_loc('Oil')
        ] = 0.0

returns = returns_raw.clip(lower=-0.50, upper=1.00).fillna(0).iloc[1:]

assert not returns.isnull().any().any(), "NaNs in returns"
assert not np.isinf(returns.values).any(),  "Inf in returns"
print(f"\nReturns shape : {returns.shape}")
print("✓ Returns clean")

# ── 9. THREE-STAGE WEIGHT BUILDER ────────────────────────────────────────────
def build_weights_3stage(sig_row, equity_assets, non_equity_assets,
                         base_weight, date, safe_haven_map):
    """
    RISK_OFF (-1) : long equity at base_weight
    RISK_ON  (+1) : short equity, redistribute proceeds to country safe havens
                    or equally across all non-equity if no safe haven available
    Non-equity assets always start at base_weight; safe-haven assets receive
    additional weight from shorted equities.
    """
    raw = {a: base_weight for a in non_equity_assets}

    safe_haven_extra = {}   # accumulate extra weight per safe-haven asset

    for a in equity_assets:
        if sig_row is not None and a in sig_row.index and not pd.isna(sig_row[a]):
            sig = sig_row[a]
        else:
            sig = -1   # default: long equity

        if sig > 0:
            # RISK_ON: short equity, redirect base_weight to safe havens
            raw[a]   = -base_weight
            havens   = safe_haven_map.get(a, [])
            havens   = [h for h in havens if h in raw]   # only if in universe

            if havens:
                per_haven = base_weight / len(havens)
                for h in havens:
                    safe_haven_extra[h] = safe_haven_extra.get(h, 0) + per_haven
            else:
                # No safe haven available → spread across all non-equity
                per_ne = base_weight / len(non_equity_assets)
                for ne in non_equity_assets:
                    safe_haven_extra[ne] = safe_haven_extra.get(ne, 0) + per_ne
        else:
            # RISK_OFF: long equity
            raw[a] = base_weight

    # Apply safe-haven extras
    for asset, extra in safe_haven_extra.items():
        raw[asset] = raw.get(asset, 0) + extra

    # Normalise to sum = 1 (handles edge cases)
    total = sum(raw.values())
    if total <= 0:
        for a in raw:
            raw[a] = 1.0 / len(raw)
    else:
        for a in raw:
            raw[a] /= total

    return raw

# ── 10. EQUAL-WEIGHT BENCHMARK ────────────────────────────────────────────────
def run_equal_weight(returns, all_assets, tc_rate=0.005,
                     rebal_freq='monthly', initial_capital=1e9):
    N           = len(all_assets)
    base_weight = 1.0 / N
    ew          = {a: base_weight for a in all_assets}

    freq_code     = 'MS' if rebal_freq == 'monthly' else 'QS'
    reb_dates_raw = returns.resample(freq_code).first().index
    actual_dates  = returns.index

    reb_set = set()
    for d in reb_dates_raw:
        future = actual_dates[actual_dates >= d]
        if len(future) > 0:
            reb_set.add(future[0])

    pv              = float(initial_capital)
    current_weights = None
    records         = []

    for date in actual_dates:
        if date in reb_set:
            if current_weights is not None:
                turnover = sum(abs(ew.get(a, 0) - current_weights.get(a, 0))
                               for a in set(ew) | set(current_weights)) / 2.0
                pv *= (1 - turnover * tc_rate)
            else:
                pv *= (1 - tc_rate)
            current_weights = ew.copy()

        if current_weights is not None:
            dr  = sum(current_weights.get(a, 0) * returns.loc[date, a]
                      for a in all_assets)
            pv *= (1 + dr)

        records.append({'date': date, 'portfolio_value': pv})

    return pd.DataFrame(records).set_index('date')

# ── 11. THREE-STAGE SIGNAL PORTFOLIO RUNNER ───────────────────────────────────
def run_signal_portfolio_3stage(returns, signal_df, equity_assets,
                                non_equity_assets, all_assets,
                                freq='monthly', tc_rate=0.005,
                                safe_haven_map=None,
                                initial_capital=1e9):

    if safe_haven_map is None:
        safe_haven_map = {}

    N           = len(all_assets)
    base_weight = 1.0 / N

    freq_code     = 'MS' if freq == 'monthly' else 'QS'
    reb_dates_raw = returns.resample(freq_code).first().index
    actual_dates  = returns.index

    reb_set = set()
    for d in reb_dates_raw:
        future = actual_dates[actual_dates >= d]
        if len(future) > 0:
            reb_set.add(future[0])

    sig = signal_df.copy()
    sig.index = pd.to_datetime(sig.index)

    pv              = float(initial_capital)
    current_weights = None
    records         = []

    for date in actual_dates:
        if date in reb_set:
            past    = sig[sig.index <= date]
            sig_row = past.iloc[-1] if len(past) > 0 else None

            raw = build_weights_3stage(
                sig_row, equity_assets, non_equity_assets,
                base_weight, date, safe_haven_map
            )

            if current_weights is not None:
                all_keys = set(raw) | set(current_weights)
                turnover = sum(
                    abs(raw.get(a, 0) - current_weights.get(a, 0))
                    for a in all_keys
                ) / 2.0
                pv *= (1 - turnover * tc_rate)
            else:
                pv *= (1 - tc_rate)

            current_weights = raw.copy()

        if current_weights is not None:
            dr  = sum(current_weights.get(a, 0) * returns.loc[date, a]
                      for a in all_assets)
            pv *= (1 + dr)

        records.append({'date': date, 'portfolio_value': pv})

    return pd.DataFrame(records).set_index('date')

# ── 12. RUN ALL PORTFOLIOS ────────────────────────────────────────────────────
TC_LEVELS = {'0%': 0.00, '0.5%': 0.005, '1%': 0.01, '2%': 0.02}
FREQS     = ['monthly', 'quarterly']

# Equal-weight benchmarks
ew_results = {}
for freq in FREQS:
    for tc_label, tc_rate in TC_LEVELS.items():
        label = f"EW_{freq}_TC{tc_label}"
        ew_results[label] = run_equal_weight(
            returns, ALL_ASSETS, tc_rate=tc_rate,
            rebal_freq=freq, initial_capital=1e9
        )
        print(f"✓ {label}")

# Three-stage signal portfolios across all thresholds (both normal and inverted)
signal_results = {}
signal_distributions = {}

SIGNAL_VARIANTS = {'normal': False, 'inverted': True}

for variant_name, inverted_flag in SIGNAL_VARIANTS.items():
    print(f"\n{'='*90}")
    print(f"RUNNING {variant_name.upper()} SIGNAL PORTFOLIOS")
    print(f"{'='*90}")

    for thresh in THRESHOLDS:
        sig_df_t = build_signal_df(composite_score, equity_assets,
                                   macro_df.index, threshold=thresh,
                                   inverted=inverted_flag)

        # Print signal distribution for this threshold
        n_risk_on  = (sig_df_t[equity_assets[0]] == +1).sum()
        n_risk_off = (sig_df_t[equity_assets[0]] == -1).sum()
        total      = len(sig_df_t)
        print(f"\n{variant_name.upper()} | Threshold={thresh} | RISK_ON(+1): {n_risk_on} days "
              f"({n_risk_on/total*100:.1f}%) | "
              f"RISK_OFF(-1): {n_risk_off} days ({n_risk_off/total*100:.1f}%)")

        signal_distributions[(variant_name, thresh)] = {'RISK_ON': n_risk_on, 'RISK_OFF': n_risk_off}

        for freq in FREQS:
            for tc_label, tc_rate in TC_LEVELS.items():
                label = f"Signal3S_{variant_name}_t{thresh}_{freq}_TC{tc_label}"
                signal_results[label] = run_signal_portfolio_3stage(
                    returns, sig_df_t, equity_assets, non_equity_assets,
                    ALL_ASSETS, freq=freq, tc_rate=tc_rate,
                    safe_haven_map=EQUITY_SAFE_HAVEN_MAP,
                    initial_capital=1e9
                )
                print(f"  ✓ {label}")

print("\n✓ All portfolios computed")

# ── 13. PERFORMANCE METRICS ───────────────────────────────────────────────────
def compute_metrics(pv_df, label=''):
    pv        = pv_df['portfolio_value']
    daily_ret = pv.pct_change().dropna()
    n_years   = len(daily_ret) / 252
    total_ret = (pv.iloc[-1] / pv.iloc[0]) - 1
    cagr      = (1 + total_ret) ** (1 / n_years) - 1 if n_years > 0 else np.nan
    vol       = daily_ret.std() * np.sqrt(252)
    sharpe    = cagr / vol if vol > 0 else np.nan
    roll_max  = pv.cummax()
    max_dd    = ((pv - roll_max) / roll_max).min()
    calmar    = cagr / abs(max_dd) if max_dd != 0 else np.nan
    var_95    = daily_ret.quantile(0.05)

    return {
        'CAGR':         f"{cagr*100:.2f}%",
        'Volatility':   f"{vol*100:.2f}%",
        'Sharpe':       f"{sharpe:.2f}",
        'Max Drawdown': f"{max_dd*100:.2f}%",
        'Calmar':       f"{calmar:.2f}",
        'VaR 95%':      f"{var_95*100:.2f}%",
        'Total Return': f"{total_ret*100:.2f}%",
        'Final ($B)':   f"{pv.iloc[-1]/1e9:.3f}",
    }

# Print results grouped by variant and threshold
ew_metrics = {k: compute_metrics(v, k) for k, v in ew_results.items()}

print("\n" + "="*90)
print("EQUAL-WEIGHT BENCHMARK")
print("="*90)
print(pd.DataFrame(ew_metrics).T.to_string())

sig_metrics = {k: compute_metrics(v, k) for k, v in signal_results.items()}

for variant_name in SIGNAL_VARIANTS.keys():
    print(f"\n{'='*110}")
    print(f"THREE-STAGE SIGNAL — {variant_name.upper()} VARIANT")
    print(f"{'='*110}")

    for thresh in THRESHOLDS:
        sub = {k: v for k, v in sig_metrics.items() if f"_{variant_name}_t{thresh}_" in k}
        dist_key = (variant_name, thresh)
        dist = signal_distributions[dist_key]

        print(f"\n{'─'*110}")
        print(f"Threshold = {thresh} | RISK_ON: {dist['RISK_ON']} days | RISK_OFF: {dist['RISK_OFF']} days")
        print(f"{'─'*110}")
        print(pd.DataFrame(sub).T.to_string())

# ── 14. PLOTS ─────────────────────────────────────────────────────────────────
colors_tc   = {'0%': 'black', '0.5%': 'steelblue', '1%': 'darkorange', '2%': 'crimson'}
colors_variant = {'normal': 'steelblue', 'inverted': 'darkred'}

# ── Figure 1: Composite signal + thresholds ────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(16, 10))
fig.suptitle("Modified Determinant — Composite Score & Threshold Signals",
             fontsize=14, fontweight='bold')

ax = axes[0]
ax.plot(composite_score.index, composite_score.values,
        color='navy', lw=1.0, alpha=0.8, label='Composite Z-score')
for thresh, col in zip(THRESHOLDS, ['green', 'orange', 'red']):
    ax.axhline( thresh, color=col, ls='--', lw=1.0, label=f'+{thresh}')
    ax.axhline(-thresh, color=col, ls='--', lw=1.0, label=f'-{thresh}')
ax.axhline(0, color='black', lw=0.6)
ax.fill_between(composite_score.index, composite_score.values, 0,
                where=(composite_score > 0), color='green', alpha=0.1)
ax.fill_between(composite_score.index, composite_score.values, 0,
                where=(composite_score <= 0), color='red', alpha=0.1)
ax.set_title("Composite Determinant Score with Thresholds", fontsize=11)
ax.set_ylabel("Z-score")
ax.legend(fontsize=8, ncol=4)
ax.grid(True, alpha=0.3)

ax = axes[1]
for window, series in det_results.items():
    ax.plot(series.index, np.sign(series.values), alpha=0.6,
            label=f"{window}d window", lw=1.0)
ax.axhline(0, color='black', lw=0.8)
ax.set_title("Sign of Det(R-I) per Window", fontsize=11)
ax.set_ylabel("+1 / -1")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("composite_signal_thresholds.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Figure 2: Normal vs Inverted (TC=0.5%, monthly, threshold=0.5) ───────────────
fig, axes = plt.subplots(1, 2, figsize=(18, 6))
fig.suptitle("Normal vs Inverted Signal — Portfolio Value (TC=0.5%, Monthly, Threshold=0.5)",
             fontsize=13, fontweight='bold')

ew_pv = ew_results['EW_monthly_TC0.5%']['portfolio_value']

for ax, variant_name in zip(axes, SIGNAL_VARIANTS.keys()):
    ax.plot(ew_pv.index, ew_pv / 1e9, '--', color='grey',
            lw=1.5, label='EW Benchmark')

    label = f"Signal3S_{variant_name}_t0.5_monthly_TC0.5%"
    pv    = signal_results[label]['portfolio_value']
    ax.plot(pv.index, pv / 1e9, color=colors_variant[variant_name], lw=1.8,
            label=f'Signal ({variant_name.upper()})')

    ax.axhline(1.0, color='black', ls=':', lw=0.8)
    ax.set_title(f"{variant_name.upper()} Signal", fontsize=11)
    ax.set_ylabel("Value ($B)")
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("signal_normal_vs_inverted.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Figure 3: Drawdowns — Normal vs Inverted ──────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(18, 5))
fig.suptitle("Drawdowns: Normal vs Inverted (TC=0.5%, Monthly, Threshold=0.5)",
             fontsize=13, fontweight='bold')

ew_pv  = ew_results['EW_monthly_TC0.5%']['portfolio_value']
ew_dd  = (ew_pv - ew_pv.cummax()) / ew_pv.cummax() * 100

for ax, variant_name in zip(axes, SIGNAL_VARIANTS.keys()):
    label  = f"Signal3S_{variant_name}_t0.5_monthly_TC0.5%"
    pv     = signal_results[label]['portfolio_value']
    dd     = (pv - pv.cummax()) / pv.cummax() * 100

    ax.fill_between(dd.index,  dd.values,  0, alpha=0.3,
                    color=colors_variant[variant_name])
    ax.fill_between(ew_dd.index, ew_dd.values, 0, alpha=0.2, color='grey')
    ax.plot(dd.index,    dd.values,    color=colors_variant[variant_name], lw=1.0,
            label=f'Signal ({variant_name.upper()})')
    ax.plot(ew_dd.index, ew_dd.values, color='grey',      lw=1.0, ls='--',
            label='EW Benchmark')
    ax.set_title(f"{variant_name.upper()} Signal", fontsize=11)
    ax.set_ylabel("Drawdown (%)")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("signal_drawdowns_normal_vs_inverted.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Figure 4: All thresholds × variants heatmap (Sharpe) ─────────────────────────
sharpe_matrix = {}
for variant_name in SIGNAL_VARIANTS.keys():
    row = []
    for thresh in THRESHOLDS:
        label = f"Signal3S_{variant_name}_t{thresh}_monthly_TC0.5%"
        metrics = sig_metrics[label]
        sharpe_str = metrics['Sharpe']
        sharpe_val = float(sharpe_str)  # Convert string to float
        row.append(sharpe_val)
    sharpe_matrix[variant_name] = row

# Create DataFrame with float values (not strings)
sharpe_df = pd.DataFrame(
    sharpe_matrix,
    index=[v.upper() for v in SIGNAL_VARIANTS.keys()],
    columns=[f"t={t}" for t in THRESHOLDS]
).astype(float)  # ← ENSURE FLOAT TYPE

print("\nSharpe Ratio Matrix:")
print(sharpe_df)

fig, ax = plt.subplots(figsize=(10, 5))
im = ax.imshow(sharpe_df.values, cmap='RdYlGn', aspect='auto')

ax.set_xticks(np.arange(len(sharpe_df.columns)))
ax.set_yticks(np.arange(len(sharpe_df.index)))
ax.set_xticklabels(sharpe_df.columns)
ax.set_yticklabels(sharpe_df.index)

for i in range(len(sharpe_df.index)):
    for j in range(len(sharpe_df.columns)):
        val = float(sharpe_df.iloc[i, j])  # Explicit conversion
        text = ax.text(j, i, f"{val:.2f}",
                      ha="center", va="center", color="black", fontweight='bold')

ax.set_title("Sharpe Ratio: Normal vs Inverted × Thresholds (Monthly, TC=0.5%)",
             fontsize=12, fontweight='bold')
cbar = plt.colorbar(im, ax=ax)
cbar.set_label('Sharpe Ratio', rotation=270, labelpad=20)

plt.tight_layout()
plt.savefig("signal_sharpe_normal_vs_inverted.png", dpi=150, bbox_inches='tight')
plt.show()


# ── Figure 5: TC sensitivity (threshold=0.5) ────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(18, 6))
fig.suptitle("TC Sensitivity: Normal vs Inverted (Monthly, Threshold=0.5)",
             fontsize=13, fontweight='bold')

for ax, variant_name in zip(axes, SIGNAL_VARIANTS.keys()):
    for tc_label, col in colors_tc.items():
        label = f"Signal3S_{variant_name}_t0.5_monthly_TC{tc_label}"
        pv    = signal_results[label]['portfolio_value']
        ax.plot(pv.index, pv / 1e9, color=col, lw=1.3,
                label=f'TC={tc_label}')

    ew_pv = ew_results['EW_monthly_TC0.5%']['portfolio_value']
    ax.plot(ew_pv.index, ew_pv / 1e9, '--', color='grey',
            lw=1.5, label='EW Benchmark')

    ax.set_title(f"{variant_name.upper()} Signal", fontsize=11)
    ax.set_ylabel("Value ($B)")
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("signal_tc_sensitivity_normal_vs_inverted.png", dpi=150, bbox_inches='tight')
plt.show()

# ── 15. SAVE ──────────────────────────────────────────────────────────────────
for label, df in {**ew_results, **signal_results}.items():
    df.to_csv(f"{label}_portfolio.csv")

all_metrics = pd.DataFrame({**ew_metrics, **sig_metrics}).T
all_metrics.to_csv("all_portfolio_metrics.csv")

composite_score.to_csv("composite_signal.csv")

for variant_name in SIGNAL_VARIANTS.keys():
    for thresh in THRESHOLDS:
        sig_df_t = build_signal_df(composite_score, equity_assets,
                                   macro_df.index, threshold=thresh,
                                   inverted=(variant_name == 'inverted'))
        sig_df_t.to_csv(f"signal_df_{variant_name}_threshold_{thresh}.csv")

print("\n✓ All files saved.")


In [ ]:
# @title
# ============================================================
# SIGNAL PORTFOLIO — THREE-STAGE THRESHOLD VERSION
# WITH COMBINATORIAL PURGED CROSS-VALIDATION (CPCV)
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import requests
from io import StringIO
from itertools import combinations
from datetime import timedelta

# ── 1. LOAD ALL DATA FROM GITHUB ─────────────────────────────────────────────
GITHUB_BASE = "https://raw.githubusercontent.com/kboroz/MSFE_Capstone_Project/main/01_Data/Streamlined/"

INDEX_FILE_MAP = {
    'ASX50':       'streamlined_asx_50.csv',
    'EUROSTOXX50': 'streamlined_euro_stoxx_50.csv',
    'FTSE100':     'streamlined_ftse_100.csv',
    'Ibovespa':    'streamlined_ibovespa.csv',
    'JSE40':       'streamlined_jse_top_40.csv',
    'NIKKEI225':   'streamlined_nikkei_225.csv',
    'SP500':       'streamlined_s&p_500.csv',
}

MACRO_FILE = 'streamlined_macro.csv'

def load_csv(url):
    r = requests.get(url)
    r.raise_for_status()
    df = pd.read_csv(StringIO(r.text), index_col=0, parse_dates=True)
    df.columns = df.columns.str.replace('[^a-zA-Z0-9]', '_', regex=True)
    return df.sort_index()

# Load macro
macro_df = load_csv(GITHUB_BASE + MACRO_FILE)
macro_df = macro_df[~macro_df.index.duplicated(keep='first')].sort_index()
print(f"macro_df shape : {macro_df.shape}")

# Load individual index files
index_data = {}
for name, fname in INDEX_FILE_MAP.items():
    try:
        df = load_csv(GITHUB_BASE + fname)
        index_data[name] = df
        print(f"✓ {name:15s} | shape={df.shape}")
    except Exception as e:
        print(f"✗ {name}: {e}")

# ── 2. EXTRACT PRICE SERIES ───────────────────────────────────────────────────
PRICE_COLS = ['Close', 'close', 'Price', 'price', 'Adj_Close', 'Last']

def get_price_series(df, name):
    col = next((c for c in PRICE_COLS if c in df.columns), df.columns[0])
    s   = pd.to_numeric(df[col], errors='coerce').dropna()
    return s

index_prices = {}
for name, df in index_data.items():
    index_prices[name] = get_price_series(df, name)

# ── 3. MODIFIED DETERMINANT SIGNAL MODEL ─────────────────────────────────────
DET_WINDOWS    = [3, 7, 28]
WINDOW_WEIGHTS = {3: 0.20, 7: 0.35, 28: 0.45}

def compute_modified_determinant(price_series_dict, windows=DET_WINDOWS):
    prices = pd.DataFrame(price_series_dict).sort_index()
    rets   = prices.pct_change().dropna()
    det_results = {}

    for window in windows:
        det_series = []
        for i in range(window, len(rets)):
            window_rets = rets.iloc[i - window: i]
            valid_cols  = window_rets.columns[window_rets.std() > 1e-10]
            w           = window_rets[valid_cols]

            if len(valid_cols) < 2:
                det_series.append(np.nan)
                continue
            try:
                R   = w.corr().values
                I   = np.eye(len(R))
                M   = R - I
                det = np.linalg.det(M)
            except Exception:
                det = np.nan
            det_series.append(det)

        idx = rets.index[window:]
        det_results[window] = pd.Series(det_series, index=idx, name=f'det_{window}d')

    return det_results

print("\n── Computing Modified Determinant ──")
det_results = compute_modified_determinant(index_prices)

# ── 4. COMPOSITE SCORE ────────────────────────────────────────────────────────
def compute_composite_score(det_results, window_weights=WINDOW_WEIGHTS):
    all_series = []
    for window, series in det_results.items():
        mu    = series.rolling(252, min_periods=60).mean()
        sigma = series.rolling(252, min_periods=60).std()
        z     = (series - mu) / sigma.replace(0, np.nan)
        z.name = f'z_{window}d'
        all_series.append(z)

    z_df      = pd.concat(all_series, axis=1).dropna()
    composite = sum(
        z_df[f'z_{w}d'] * window_weights[w]
        for w in window_weights if f'z_{w}d' in z_df.columns
    )
    composite = composite / sum(window_weights.values())
    return composite, z_df

composite_score, z_df = compute_composite_score(det_results)

print(f"Composite score shape  : {composite_score.shape}")
print(f"Date range             : {composite_score.index[0].date()} → "
      f"{composite_score.index[-1].date()}")

# ── 5. SIGNAL CLASSIFICATION ──────────────────────────────────────────────────
THRESHOLDS = [0.3, 0.5, 0.7]

def classify_signal(score, threshold):
    if abs(score) > threshold:
        return +1   # RISK_ON
    else:
        return -1   # RISK_OFF

def build_signal_df(composite_score, equity_assets, macro_dates, threshold=0.5):
    binary = composite_score.apply(lambda x: classify_signal(x, threshold))
    signal_df = pd.DataFrame(index=macro_dates)
    for asset in equity_assets:
        signal_df[asset] = binary.reindex(macro_dates).ffill().bfill()
    return signal_df

# ── 6. ASSET UNIVERSE ─────────────────────────────────────────────────────────
EQUITY_ASSETS = list(index_prices.keys())
SAFE_BONDS    = ['US_10Y', 'UK_10Y', 'AU_10Y', 'ZA_10Y']
FX            = ['AUDUSD', 'BRLUSD', 'EURUSD', 'GBPUSD', 'JPYUSD', 'ZARUSD']
COMMODITIES   = ['Oil', 'Gold', 'Bitcoin']

ALL_ASSETS        = EQUITY_ASSETS + FX + COMMODITIES + SAFE_BONDS
ALL_ASSETS        = [a for a in ALL_ASSETS if a in macro_df.columns]
equity_assets     = [a for a in EQUITY_ASSETS if a in ALL_ASSETS]
non_equity_assets = [a for a in ALL_ASSETS if a not in equity_assets]

N           = len(ALL_ASSETS)
base_weight = 1.0 / N

print(f"\nTotal assets: {N} | Equity: {len(equity_assets)} | Non-equity: {len(non_equity_assets)}")

# ── 7. SAFE-HAVEN MAP ─────────────────────────────────────────────────────────
EQUITY_SAFE_HAVEN_MAP = {
    'ASX50':       ['AU_10Y', 'AUDUSD'],
    'EUROSTOXX50': ['EURUSD'],
    'FTSE100':     ['UK_10Y', 'GBPUSD'],
    'Ibovespa':    ['BRLUSD'],
    'JSE40':       ['ZA_10Y', 'ZARUSD'],
    'NIKKEI225':   ['JPYUSD'],
    'SP500':       ['US_10Y'],
}

UNIVERSE_SET = set(ALL_ASSETS)
for eq, havens in EQUITY_SAFE_HAVEN_MAP.items():
    EQUITY_SAFE_HAVEN_MAP[eq] = [h for h in havens if h in UNIVERSE_SET]

# ── 8. BUILD RETURNS ──────────────────────────────────────────────────────────
returns_raw = macro_df[ALL_ASSETS].pct_change(fill_method=None)

for col in SAFE_BONDS:
    if col in returns_raw.columns:
        returns_raw[col] = -returns_raw[col]

if 'Oil' in macro_df.columns:
    bad_oil = macro_df['Oil'][macro_df['Oil'] <= 0].index
    for d in bad_oil:
        try:
            loc = returns_raw.index.get_loc(d)
            returns_raw.iloc[
                max(0, loc - 1): loc + 2,
                returns_raw.columns.get_loc('Oil')
            ] = 0.0
        except:
            pass

returns = returns_raw.clip(lower=-0.50, upper=1.00).fillna(0).iloc[1:]

print(f"Returns shape : {returns.shape}")

# ── 9. WEIGHT BUILDER ─────────────────────────────────────────────────────────
def build_weights_3stage(sig_row, equity_assets, non_equity_assets,
                         base_weight, date, safe_haven_map):
    raw = {a: base_weight for a in non_equity_assets}
    safe_haven_extra = {}

    for a in equity_assets:
        if sig_row is not None and a in sig_row.index and not pd.isna(sig_row[a]):
            sig = sig_row[a]
        else:
            sig = -1

        if sig > 0:
            raw[a]   = -base_weight
            havens   = safe_haven_map.get(a, [])
            havens   = [h for h in havens if h in raw]

            if havens:
                per_haven = base_weight / len(havens)
                for h in havens:
                    safe_haven_extra[h] = safe_haven_extra.get(h, 0) + per_haven
            else:
                per_ne = base_weight / len(non_equity_assets)
                for ne in non_equity_assets:
                    safe_haven_extra[ne] = safe_haven_extra.get(ne, 0) + per_ne
        else:
            raw[a] = base_weight

    for asset, extra in safe_haven_extra.items():
        raw[asset] = raw.get(asset, 0) + extra

    total = sum(raw.values())
    if total <= 0:
        for a in raw:
            raw[a] = 1.0 / len(raw)
    else:
        for a in raw:
            raw[a] /= total

    return raw

# ── 10. PORTFOLIO RUNNERS ─────────────────────────────────────────────────────
def run_equal_weight(returns, all_assets, tc_rate=0.005,
                     rebal_freq='monthly', initial_capital=1e9):
    N           = len(all_assets)
    base_weight = 1.0 / N
    ew          = {a: base_weight for a in all_assets}

    freq_code     = 'MS' if rebal_freq == 'monthly' else 'QS'
    reb_dates_raw = returns.resample(freq_code).first().index
    actual_dates  = returns.index

    reb_set = set()
    for d in reb_dates_raw:
        future = actual_dates[actual_dates >= d]
        if len(future) > 0:
            reb_set.add(future[0])

    pv              = float(initial_capital)
    current_weights = None
    records         = []

    for date in actual_dates:
        if date in reb_set:
            if current_weights is not None:
                turnover = sum(abs(ew.get(a, 0) - current_weights.get(a, 0))
                               for a in set(ew) | set(current_weights)) / 2.0
                pv *= (1 - turnover * tc_rate)
            else:
                pv *= (1 - tc_rate)
            current_weights = ew.copy()

        if current_weights is not None:
            dr  = sum(current_weights.get(a, 0) * returns.loc[date, a]
                      for a in all_assets)
            pv *= (1 + dr)

        records.append({'date': date, 'portfolio_value': pv})

    return pd.DataFrame(records).set_index('date')

def run_signal_portfolio_3stage(returns, signal_df, equity_assets,
                                non_equity_assets, all_assets,
                                freq='monthly', tc_rate=0.005,
                                safe_haven_map=None,
                                initial_capital=1e9):

    if safe_haven_map is None:
        safe_haven_map = {}

    N           = len(all_assets)
    base_weight = 1.0 / N

    freq_code     = 'MS' if freq == 'monthly' else 'QS'
    reb_dates_raw = returns.resample(freq_code).first().index
    actual_dates  = returns.index

    reb_set = set()
    for d in reb_dates_raw:
        future = actual_dates[actual_dates >= d]
        if len(future) > 0:
            reb_set.add(future[0])

    sig = signal_df.copy()
    sig.index = pd.to_datetime(sig.index)

    pv              = float(initial_capital)
    current_weights = None
    records         = []

    for date in actual_dates:
        if date in reb_set:
            past    = sig[sig.index <= date]
            sig_row = past.iloc[-1] if len(past) > 0 else None

            raw = build_weights_3stage(
                sig_row, equity_assets, non_equity_assets,
                base_weight, date, safe_haven_map
            )

            if current_weights is not None:
                all_keys = set(raw) | set(current_weights)
                turnover = sum(
                    abs(raw.get(a, 0) - current_weights.get(a, 0))
                    for a in all_keys
                ) / 2.0
                pv *= (1 - turnover * tc_rate)
            else:
                pv *= (1 - tc_rate)

            current_weights = raw.copy()

        if current_weights is not None:
            dr  = sum(current_weights.get(a, 0) * returns.loc[date, a]
                      for a in all_assets)
            pv *= (1 + dr)

        records.append({'date': date, 'portfolio_value': pv})

    return pd.DataFrame(records).set_index('date')

# ── 11. COMBINATORIAL PURGED CROSS-VALIDATION (CPCV) ──────────────────────────
class CombinatorialPurgedCV:
    """
    Implements Combinatorial Purged Cross-Validation (Lopez de Prado).

    Splits data into k contiguous folds, then generates all C(k, m) combinations
    of m test folds, with embargo applied to prevent look-ahead bias.
    """

    def __init__(self, n_splits=5, n_test_folds=2, embargo_pct=0.01):
        """
        Args:
            n_splits: Total number of folds (k)
            n_test_folds: Number of test folds per combination (m)
            embargo_pct: Embargo width as % of fold length
        """
        self.n_splits = n_splits
        self.n_test_folds = n_test_folds
        self.embargo_pct = embargo_pct

    def split(self, X, y=None, groups=None):
        """
        Generate (train_idx, test_idx) pairs for all C(k, m) combinations.
        Yields tuples of (train_indices, test_indices, test_fold_ids).
        """
        n_samples = len(X)
        fold_size = n_samples // self.n_splits
        embargo_size = max(1, int(n_samples * self.embargo_pct))

        # Create fold boundaries
        fold_starts = [i * fold_size for i in range(self.n_splits)]
        fold_ends = fold_starts[1:] + [n_samples]

        # Generate all combinations of test fold indices
        test_fold_combos = list(combinations(range(self.n_splits), self.n_test_folds))

        print(f"\n{'='*80}")
        print(f"COMBINATORIAL PURGED CROSS-VALIDATION")
        print(f"{'='*80}")
        print(f"Total folds (k)          : {self.n_splits}")
        print(f"Test folds per combo (m) : {self.n_test_folds}")
        print(f"Total combinations       : C({self.n_splits},{self.n_test_folds}) = {len(test_fold_combos)}")
        print(f"Fold size                : ~{fold_size} observations")
        print(f"Embargo size             : {embargo_size} observations ({self.embargo_pct*100:.1f}%)")
        print(f"{'='*80}\n")

        for combo_idx, test_fold_ids in enumerate(test_fold_combos, 1):
            # Collect test indices from selected folds
            test_idx = []
            for fold_id in test_fold_ids:
                test_idx.extend(range(fold_starts[fold_id], fold_ends[fold_id]))
            test_idx = np.array(sorted(test_idx))

            # Apply embargo: exclude data before and after test folds
            embargo_left = max(0, min(fold_starts[fold_id] for fold_id in test_fold_ids) - embargo_size)
            embargo_right = min(n_samples, max(fold_ends[fold_id] for fold_id in test_fold_ids) + embargo_size)

            # Train indices = everything except test + embargo
            embargo_mask = np.ones(n_samples, dtype=bool)
            embargo_mask[embargo_left:embargo_right] = False
            train_idx = np.where(embargo_mask)[0]

            print(f"Combination {combo_idx:2d}: Test folds {test_fold_ids} | "
                  f"Train: {len(train_idx)} | Test: {len(test_idx)} | "
                  f"Embargo: [{embargo_left:5d}, {embargo_right:5d}]")

            yield train_idx, test_idx, test_fold_ids

# ── 12. CPCV-BASED BACKTESTING ────────────────────────────────────────────────
def backtest_with_cpcv(returns, composite_score, signal_func, equity_assets,
                       non_equity_assets, all_assets, safe_haven_map,
                       threshold=0.5, freq='monthly', tc_rate=0.005,
                       n_splits=5, n_test_folds=2, initial_capital=1e9):
    """
    Run signal portfolio using CPCV splits.
    Returns dict of results per combination.
    """

    cv = CombinatorialPurgedCV(n_splits=n_splits, n_test_folds=n_test_folds, embargo_pct=0.01)

    cpcv_results = {}

    # Align composite_score with returns
    common_dates = returns.index.intersection(composite_score.index)
    returns_aligned = returns.loc[common_dates]
    score_aligned = composite_score.loc[common_dates]

    for train_idx, test_idx, test_fold_ids in cv.split(returns_aligned):
        train_dates = returns_aligned.index[train_idx]
        test_dates = returns_aligned.index[test_idx]

        # Build signal from ENTIRE dataset (not just training)
        # This ensures signal is not "trained" in traditional sense
        sig_df = signal_func(composite_score, equity_assets,
                            returns_aligned.index, threshold=threshold)

        # Run portfolio on test period only
        test_returns = returns_aligned.loc[test_dates]
        test_signal_df = sig_df.loc[test_dates]

        pv = run_signal_portfolio_3stage(
            test_returns, test_signal_df, equity_assets, non_equity_assets,
            all_assets, freq=freq, tc_rate=tc_rate,
            safe_haven_map=safe_haven_map, initial_capital=initial_capital
        )

        combo_label = f"Folds_{'-'.join(map(str, test_fold_ids))}"
        cpcv_results[combo_label] = {
            'portfolio_value': pv,
            'train_dates': train_dates,
            'test_dates': test_dates,
            'train_idx': train_idx,
            'test_idx': test_idx
        }

    return cpcv_results

# ── 13. RUN CPCV BACKTESTS ────────────────────────────────────────────────────
print("\n" + "="*80)
print("RUNNING CPCV BACKTESTS")
print("="*80)

TC_LEVELS = {'0.5%': 0.005, '1%': 0.01}
FREQS = ['monthly']
N_SPLITS = 5
N_TEST_FOLDS = 2

cpcv_all_results = {}

for thresh in THRESHOLDS:
    for freq in FREQS:
        for tc_label, tc_rate in TC_LEVELS.items():
            label = f"Signal3S_CPCV_t{thresh}_{freq}_TC{tc_label}"

            cpcv_results = backtest_with_cpcv(
                returns, composite_score, build_signal_df,
                equity_assets, non_equity_assets, ALL_ASSETS,
                EQUITY_SAFE_HAVEN_MAP,
                threshold=thresh, freq=freq, tc_rate=tc_rate,
                n_splits=N_SPLITS, n_test_folds=N_TEST_FOLDS,
                initial_capital=1e9
            )

            cpcv_all_results[label] = cpcv_results
            print(f"✓ {label}")

# ── 14. AGGREGATE CPCV RESULTS ────────────────────────────────────────────────
def compute_metrics(pv_df):
    pv        = pv_df['portfolio_value']
    daily_ret = pv.pct_change().dropna()
    n_years   = len(daily_ret) / 252
    total_ret = (pv.iloc[-1] / pv.iloc[0]) - 1 if len(pv) > 0 else np.nan
    cagr      = (1 + total_ret) ** (1 / n_years) - 1 if n_years > 0 else np.nan
    vol       = daily_ret.std() * np.sqrt(252)
    sharpe    = cagr / vol if vol > 0 else np.nan
    roll_max  = pv.cummax()
    max_dd    = ((pv - roll_max) / roll_max).min()
    calmar    = cagr / abs(max_dd) if max_dd != 0 else np.nan

    return {
        'CAGR': cagr,
        'Volatility': vol,
        'Sharpe': sharpe,
        'Max Drawdown': max_dd,
        'Calmar': calmar,
        'Total Return': total_ret,
        'Final Value': pv.iloc[-1] if len(pv) > 0 else np.nan
    }

# Aggregate metrics per strategy (average across combinations)
cpcv_metrics_summary = {}

for label, combos in cpcv_all_results.items():
    metrics_list = []
    for combo_label, result in combos.items():
        pv_df = result['portfolio_value']
        metrics = compute_metrics(pv_df)
        metrics['Combo'] = combo_label
        metrics_list.append(metrics)

    metrics_df = pd.DataFrame(metrics_list)
    cpcv_metrics_summary[label] = {
        'all_combos': metrics_df,
        'mean': metrics_df[['CAGR', 'Volatility', 'Sharpe', 'Max Drawdown', 'Calmar', 'Total Return']].mean(),
        'std': metrics_df[['CAGR', 'Volatility', 'Sharpe', 'Max Drawdown', 'Calmar', 'Total Return']].std()
    }

# Print aggregated results
print("\n" + "="*90)
print("CPCV RESULTS — AGGREGATED ACROSS ALL COMBINATIONS")
print("="*90)

for label, summary in cpcv_metrics_summary.items():
    print(f"\n{label}")
    print("-" * 90)
    print("MEAN METRICS:")
    print(f"  CAGR:        {summary['mean']['CAGR']*100:6.2f}%")
    print(f"  Volatility:  {summary['mean']['Volatility']*100:6.2f}%")
    print(f"  Sharpe:      {summary['mean']['Sharpe']:6.2f}")
    print(f"  Max DD:      {summary['mean']['Max Drawdown']*100:6.2f}%")
    print(f"  Calmar:      {summary['mean']['Calmar']:6.2f}")
    print(f"  Total Ret:   {summary['mean']['Total Return']*100:6.2f}%")
    print("\nSTDDEV ACROSS COMBINATIONS:")
    print(f"  Sharpe (σ):  {summary['std']['Sharpe']:6.2f}")
    print(f"  CAGR (σ):    {summary['std']['CAGR']*100:6.2f}%")
    print("\nPER-COMBINATION METRICS:")
    print(summary['all_combos'].to_string(index=False))

# ── 15. VISUALIZATIONS ────────────────────────────────────────────────────────

# Figure 1: CPCV Split Timeline
fig, ax = plt.subplots(figsize=(16, 6))

n_samples = len(returns)
fold_size = n_samples // N_SPLITS
colors_fold = plt.cm.tab10(np.linspace(0, 1, N_SPLITS))

for fold_id in range(N_SPLITS):
    fold_start = fold_id * fold_size
    fold_end = (fold_id + 1) * fold_size if fold_id < N_SPLITS - 1 else n_samples
    ax.barh(0, fold_end - fold_start, left=fold_start, height=0.5,
            color=colors_fold[fold_id], edgecolor='black', linewidth=1.5,
            label=f'Fold {fold_id}')

ax.set_xlim(0, n_samples)
ax.set_ylim(-1, 1)
ax.set_xlabel("Sample Index", fontsize=12)
ax.set_title(f"CPCV Fold Structure (k={N_SPLITS} folds, m={N_TEST_FOLDS} test folds per combo)",
             fontsize=13, fontweight='bold')
ax.legend(loc='upper center', ncol=N_SPLITS, fontsize=10)
ax.set_yticks([])
ax.grid(True, alpha=0.2, axis='x')

plt.tight_layout()
plt.savefig("cpcv_fold_structure.png", dpi=150, bbox_inches='tight')
plt.show()

# Figure 2: Sharpe Ratio Distribution Across Combinations
fig, axes = plt.subplots(1, len(THRESHOLDS), figsize=(18, 5))
if len(THRESHOLDS) == 1:
    axes = [axes]

for ax, thresh in zip(axes, THRESHOLDS):
    labels_for_thresh = [l for l in cpcv_metrics_summary.keys() if f"_t{thresh}_" in l]

    for label in labels_for_thresh:
        combo_metrics = cpcv_metrics_summary[label]['all_combos']
        sharpes = combo_metrics['Sharpe'].values
        ax.scatter([thresh] * len(sharpes), sharpes, s=100, alpha=0.6,
                  label=label.split('_TC')[-1])

    ax.axhline(0, color='black', ls='--', lw=0.8, alpha=0.5)
    ax.set_xlabel("Threshold", fontsize=11)
    ax.set_ylabel("Sharpe Ratio", fontsize=11)
    ax.set_title(f"Threshold = {thresh}", fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=9)

fig.suptitle("CPCV: Sharpe Ratio Distribution Across Combinations",
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig("cpcv_sharpe_distribution.png", dpi=150, bbox_inches='tight')
plt.show()

# Figure 3: Portfolio Values Across Combinations (One Threshold Example)
thresh_example = THRESHOLDS[1]  # middle threshold
label_example = f"Signal3S_CPCV_t{thresh_example}_monthly_TC0.5%"

fig, ax = plt.subplots(figsize=(16, 7))

for combo_label, result in cpcv_all_results[label_example].items():
    pv = result['portfolio_value']['portfolio_value']
    ax.plot(pv.index, pv / 1e9, lw=1.5, alpha=0.7, label=combo_label)

ax.set_xlabel("Date", fontsize=12)
ax.set_ylabel("Portfolio Value ($B)", fontsize=12)
ax.set_title(f"CPCV Portfolio Paths — {label_example}",
             fontsize=13, fontweight='bold')
ax.legend(fontsize=9, ncol=2, loc='best')
ax.grid(True, alpha=0.3)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))

plt.tight_layout()
plt.savefig(f"cpcv_portfolio_paths_t{thresh_example}.png", dpi=150, bbox_inches='tight')
plt.show()

# Figure 4: Heatmap of Sharpe Ratios by Threshold × Combination
sharpe_matrix_cpcv = {}
for label, summary in cpcv_metrics_summary.items():
    thresh = float(label.split('_t')[1].split('_')[0])
    if thresh not in sharpe_matrix_cpcv:
        sharpe_matrix_cpcv[thresh] = []
    sharpe_matrix_cpcv[thresh].append(summary['mean']['Sharpe'])

fig, ax = plt.subplots(figsize=(10, 5))

# Extract thresholds and mean Sharpes
thresholds_sorted = sorted(sharpe_matrix_cpcv.keys())
sharpes_mean = [sharpe_matrix_cpcv[t][0] for t in thresholds_sorted]

bars = ax.bar(range(len(thresholds_sorted)), sharpes_mean, color='steelblue', edgecolor='black', linewidth=1.5)

# Add value labels on bars
for bar, sharpe in zip(bars, sharpes_mean):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
           f'{sharpe:.2f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_xlabel("Threshold", fontsize=12)
ax.set_ylabel("Mean Sharpe Ratio (across combinations)", fontsize=12)
ax.set_title("CPCV: Mean Sharpe Ratio by Threshold", fontsize=13, fontweight='bold')
ax.set_xticks(range(len(thresholds_sorted)))
ax.set_xticklabels([f"{t:.1f}" for t in thresholds_sorted])
ax.grid(True, alpha=0.3, axis='y')
ax.axhline(0, color='black', lw=0.8)

plt.tight_layout()
plt.savefig("cpcv_sharpe_by_threshold.png", dpi=150, bbox_inches='tight')
plt.show()

# ── 16. SAVE ALL RESULTS ──────────────────────────────────────────────────────
print("\n" + "="*80)
print("SAVING RESULTS")
print("="*80)

# Save CPCV metrics summary
for label, summary in cpcv_metrics_summary.items():
    summary['all_combos'].to_csv(f"{label}_all_combinations.csv", index=False)
    summary['mean'].to_csv(f"{label}_mean_metrics.csv")
    summary['std'].to_csv(f"{label}_std_metrics.csv")
    print(f"✓ {label}_*.csv")

# Save individual portfolio values
for label, combos in cpcv_all_results.items():
    for combo_label, result in combos.items():
        pv_df = result['portfolio_value']
        filename = f"{label}_{combo_label}_portfolio.csv"
        pv_df.to_csv(filename)

print(f"\n✓ All CPCV results saved ({len(cpcv_all_results)} strategies)")


In [ ]:
# @title
# ============================================================
# SIGNAL PORTFOLIO — THREE-STAGE THRESHOLD VERSION
# Signals generated from Modified Determinant Model
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import requests
from io import StringIO

# ── 1. LOAD ALL DATA FROM GITHUB ─────────────────────────────────────────────
GITHUB_BASE = "https://raw.githubusercontent.com/kboroz/MSFE_Capstone_Project/main/01_Data/Streamlined/"

INDEX_FILE_MAP = {
    'ASX50':       'streamlined_asx_50.csv',
    'EUROSTOXX50': 'streamlined_euro_stoxx_50.csv',
    'FTSE100':     'streamlined_ftse_100.csv',
    'Ibovespa':    'streamlined_ibovespa.csv',
    'JSE40':       'streamlined_jse_top_40.csv',
    'NIKKEI225':   'streamlined_nikkei_225.csv',
    'SP500':       'streamlined_s&p_500.csv',
}

MACRO_FILE = 'streamlined_macro.csv'

def load_csv(url):
    r = requests.get(url)
    r.raise_for_status()
    df = pd.read_csv(StringIO(r.text), index_col=0, parse_dates=True)
    df.columns = df.columns.str.replace('[^a-zA-Z0-9]', '_', regex=True)
    return df.sort_index()

# Load macro
macro_df = load_csv(GITHUB_BASE + MACRO_FILE)
macro_df = macro_df[~macro_df.index.duplicated(keep='first')].sort_index()
print(f"macro_df shape : {macro_df.shape}")
print(f"Columns        : {macro_df.columns.tolist()}")

# Load individual index files
index_data = {}
for name, fname in INDEX_FILE_MAP.items():
    try:
        df = load_csv(GITHUB_BASE + fname)
        index_data[name] = df
        print(f"✓ {name:15s} | shape={df.shape} | cols={df.columns.tolist()[:4]}")
    except Exception as e:
        print(f"✗ {name}: {e}")

# ── 2. EXTRACT PRICE SERIES ───────────────────────────────────────────────────
PRICE_COLS = ['Close', 'close', 'Price', 'price', 'Adj_Close', 'Last']

def get_price_series(df, name):
    col = next((c for c in PRICE_COLS if c in df.columns), df.columns[0])
    s   = pd.to_numeric(df[col], errors='coerce').dropna()
    print(f"  {name}: using column '{col}'")
    return s

index_prices = {}
for name, df in index_data.items():
    index_prices[name] = get_price_series(df, name)

# ── 3. MODIFIED DETERMINANT SIGNAL MODEL ─────────────────────────────────────
DET_WINDOWS    = [3, 7, 28]
WINDOW_WEIGHTS = {3: 0.50, 7: 0.0, 28: 0.5}

def compute_modified_determinant(price_series_dict, windows=DET_WINDOWS):
    prices = pd.DataFrame(price_series_dict).sort_index()
    rets   = prices.pct_change().dropna()
    det_results = {}

    for window in windows:
        det_series = []
        for i in range(window, len(rets)):
            window_rets = rets.iloc[i - window: i]
            valid_cols  = window_rets.columns[window_rets.std() > 1e-10]
            w           = window_rets[valid_cols]

            if len(valid_cols) < 2:
                det_series.append(np.nan)
                continue
            try:
                R   = w.corr().values
                I   = np.eye(len(R))
                M   = R - I
                det = np.linalg.det(M)
            except Exception:
                det = np.nan
            det_series.append(det)

        idx = rets.index[window:]
        det_results[window] = pd.Series(det_series, index=idx, name=f'det_{window}d')
        print(f"  Window {window:2d}d | computed {len(det_series)} observations")

    return det_results

print("\n── Computing Modified Determinant ──")
det_results = compute_modified_determinant(index_prices)

# ── 4. COMPOSITE SCORE ────────────────────────────────────────────────────────
def compute_composite_score(det_results, window_weights=WINDOW_WEIGHTS):
    """
    Returns raw weighted-average of normalised determinant values (not signs).
    Normalised per window via rolling z-score so windows are comparable.
    """
    all_series = []
    for window, series in det_results.items():
        # z-score normalise so scale is comparable across windows
        mu    = series.rolling(252, min_periods=60).mean()
        sigma = series.rolling(252, min_periods=60).std()
        z     = (series - mu) / sigma.replace(0, np.nan)
        z.name = f'z_{window}d'
        all_series.append(z)

    z_df      = pd.concat(all_series, axis=1).dropna()
    composite = sum(
        z_df[f'z_{w}d'] * window_weights[w]
        for w in window_weights if f'z_{w}d' in z_df.columns
    )
    composite = composite / sum(window_weights.values())
    return composite, z_df

composite_score, z_df = compute_composite_score(det_results)

print(f"\nComposite score shape  : {composite_score.shape}")
print(f"Date range             : {composite_score.index[0].date()} → "
      f"{composite_score.index[-1].date()}")
print(f"Score range            : {composite_score.min():.3f} → "
      f"{composite_score.max():.3f}")

# ── 5. THREE-STAGE SIGNAL CLASSIFICATION ─────────────────────────────────────
# |score| > threshold  → RISK_ON  (+1): high det magnitude = abnormal regime
#                         → short equity, long country bonds + FX
# |score| <= threshold → RISK_OFF (-1): low det magnitude  = calm regime
#                         → long equity

THRESHOLDS = [0.3, 0.5, 0.7]   # tune here

def classify_signal(score, threshold):
    """
    Returns +1 (RISK_ON / defensive) or -1 (RISK_OFF / long equity)
    """
    if abs(score) > threshold:
        return +1   # high |det| → abnormal → defensive
    else:
        return -1   # low  |det| → calm     → long equity

def build_signal_df(composite_score, equity_assets, macro_dates, threshold=0.5):
    """
    Apply three-stage classification and reindex to portfolio dates.
    """
    binary = composite_score.apply(lambda x: classify_signal(x, threshold))
    signal_df = pd.DataFrame(index=macro_dates)
    for asset in equity_assets:
        signal_df[asset] = binary.reindex(macro_dates).ffill().bfill()
    return signal_df

# ── 6. ASSET UNIVERSE ─────────────────────────────────────────────────────────
EQUITY_ASSETS = list(index_prices.keys())
SAFE_BONDS    = ['US_10Y', 'UK_10Y', 'AU_10Y', 'ZA_10Y']
FX            = ['AUDUSD', 'BRLUSD', 'EURUSD', 'GBPUSD', 'JPYUSD', 'ZARUSD']
COMMODITIES   = ['Oil', 'Gold', 'Bitcoin']

ALL_ASSETS        = EQUITY_ASSETS + FX + COMMODITIES + SAFE_BONDS
ALL_ASSETS        = [a for a in ALL_ASSETS if a in macro_df.columns]
equity_assets     = [a for a in EQUITY_ASSETS if a in ALL_ASSETS]
non_equity_assets = [a for a in ALL_ASSETS if a not in equity_assets]

N           = len(ALL_ASSETS)
base_weight = 1.0 / N

print(f"\nTotal assets              : {N}")
print(f"Equity (signal-driven)    : {equity_assets}")
print(f"Non-equity (always long)  : {non_equity_assets}")
print(f"base_weight = 1/{N} = {base_weight:.4f}")

# ── 7. COUNTRY SAFE-HAVEN MAP ─────────────────────────────────────────────────
# When an equity index is shorted (RISK_ON), proceeds are redistributed
# to that country's bond + FX where available in the universe

EQUITY_SAFE_HAVEN_MAP = {
    'ASX50':       ['AU_10Y', 'AUDUSD'],
    'EUROSTOXX50': ['EURUSD'],            # no EU bond in universe
    'FTSE100':     ['UK_10Y', 'GBPUSD'],
    'Ibovespa':    ['BRLUSD'],            # no Brazil bond in universe
    'JSE40':       ['ZA_10Y', 'ZARUSD'],
    'NIKKEI225':   ['JPYUSD'],            # no JP bond in universe
    'SP500':       ['US_10Y'],            # USD is base currency
}

# Filter to assets actually in universe
UNIVERSE_SET = set(ALL_ASSETS)
for eq, havens in EQUITY_SAFE_HAVEN_MAP.items():
    EQUITY_SAFE_HAVEN_MAP[eq] = [h for h in havens if h in UNIVERSE_SET]

print("\nSafe-haven routing (filtered to universe):")
for eq, havens in EQUITY_SAFE_HAVEN_MAP.items():
    print(f"  {eq:15s} → {havens}")

# ── 8. BUILD RETURNS ──────────────────────────────────────────────────────────
returns_raw = macro_df[ALL_ASSETS].pct_change(fill_method=None)

for col in SAFE_BONDS:
    if col in returns_raw.columns:
        returns_raw[col] = -returns_raw[col]   # invert yield → price

if 'Oil' in macro_df.columns:
    bad_oil = macro_df['Oil'][macro_df['Oil'] <= 0].index
    for d in bad_oil:
        loc = returns_raw.index.get_loc(d)
        returns_raw.iloc[
            max(0, loc - 1): loc + 2,
            returns_raw.columns.get_loc('Oil')
        ] = 0.0

returns = returns_raw.clip(lower=-0.50, upper=1.00).fillna(0).iloc[1:]

assert not returns.isnull().any().any(), "NaNs in returns"
assert not np.isinf(returns.values).any(),  "Inf in returns"
print(f"\nReturns shape : {returns.shape}")
print("✓ Returns clean")

# ── 9. THREE-STAGE WEIGHT BUILDER ────────────────────────────────────────────
def build_weights_3stage(sig_row, equity_assets, non_equity_assets,
                         base_weight, date, safe_haven_map):
    """
    RISK_OFF (-1) : long equity at base_weight
    RISK_ON  (+1) : short equity, redistribute proceeds to country safe havens
                    or equally across all non-equity if no safe haven available
    Non-equity assets always start at base_weight; safe-haven assets receive
    additional weight from shorted equities.
    """
    raw = {a: base_weight for a in non_equity_assets}

    safe_haven_extra = {}   # accumulate extra weight per safe-haven asset

    for a in equity_assets:
        if sig_row is not None and a in sig_row.index and not pd.isna(sig_row[a]):
            sig = sig_row[a]
        else:
            sig = -1   # default: long equity

        if sig > 0:
            # RISK_ON: short equity, redirect base_weight to safe havens
            raw[a]   = -base_weight
            havens   = safe_haven_map.get(a, [])
            havens   = [h for h in havens if h in raw]   # only if in universe

            if havens:
                per_haven = base_weight / len(havens)
                for h in havens:
                    safe_haven_extra[h] = safe_haven_extra.get(h, 0) + per_haven
            else:
                # No safe haven available → spread across all non-equity
                per_ne = base_weight / len(non_equity_assets)
                for ne in non_equity_assets:
                    safe_haven_extra[ne] = safe_haven_extra.get(ne, 0) + per_ne
        else:
            # RISK_OFF: long equity
            raw[a] = base_weight

    # Apply safe-haven extras
    for asset, extra in safe_haven_extra.items():
        raw[asset] = raw.get(asset, 0) + extra

    # Normalise to sum = 1 (handles edge cases)
    total = sum(raw.values())
    if total <= 0:
        for a in raw:
            raw[a] = 1.0 / len(raw)
    else:
        for a in raw:
            raw[a] /= total

    return raw

# ── 10. EQUAL-WEIGHT BENCHMARK ────────────────────────────────────────────────
def run_equal_weight(returns, all_assets, tc_rate=0.005,
                     rebal_freq='monthly', initial_capital=1e9):
    N           = len(all_assets)
    base_weight = 1.0 / N
    ew          = {a: base_weight for a in all_assets}

    freq_code     = 'MS' if rebal_freq == 'monthly' else 'QS'
    reb_dates_raw = returns.resample(freq_code).first().index
    actual_dates  = returns.index

    reb_set = set()
    for d in reb_dates_raw:
        future = actual_dates[actual_dates >= d]
        if len(future) > 0:
            reb_set.add(future[0])

    pv              = float(initial_capital)
    current_weights = None
    records         = []

    for date in actual_dates:
        if date in reb_set:
            if current_weights is not None:
                turnover = sum(abs(ew.get(a, 0) - current_weights.get(a, 0))
                               for a in set(ew) | set(current_weights)) / 2.0
                pv *= (1 - turnover * tc_rate)
            else:
                pv *= (1 - tc_rate)
            current_weights = ew.copy()

        if current_weights is not None:
            dr  = sum(current_weights.get(a, 0) * returns.loc[date, a]
                      for a in all_assets)
            pv *= (1 + dr)

        records.append({'date': date, 'portfolio_value': pv})

    return pd.DataFrame(records).set_index('date')

# ── 11. THREE-STAGE SIGNAL PORTFOLIO RUNNER ───────────────────────────────────
def run_signal_portfolio_3stage(returns, signal_df, equity_assets,
                                non_equity_assets, all_assets,
                                freq='monthly', tc_rate=0.005,
                                safe_haven_map=None,
                                initial_capital=1e9):

    if safe_haven_map is None:
        safe_haven_map = {}

    N           = len(all_assets)
    base_weight = 1.0 / N

    freq_code     = 'MS' if freq == 'monthly' else 'QS'
    reb_dates_raw = returns.resample(freq_code).first().index
    actual_dates  = returns.index

    reb_set = set()
    for d in reb_dates_raw:
        future = actual_dates[actual_dates >= d]
        if len(future) > 0:
            reb_set.add(future[0])

    sig = signal_df.copy()
    sig.index = pd.to_datetime(sig.index)

    pv              = float(initial_capital)
    current_weights = None
    records         = []

    for date in actual_dates:
        if date in reb_set:
            past    = sig[sig.index <= date]
            sig_row = past.iloc[-1] if len(past) > 0 else None

            raw = build_weights_3stage(
                sig_row, equity_assets, non_equity_assets,
                base_weight, date, safe_haven_map
            )

            if current_weights is not None:
                all_keys = set(raw) | set(current_weights)
                turnover = sum(
                    abs(raw.get(a, 0) - current_weights.get(a, 0))
                    for a in all_keys
                ) / 2.0
                pv *= (1 - turnover * tc_rate)
            else:
                pv *= (1 - tc_rate)

            current_weights = raw.copy()

        if current_weights is not None:
            dr  = sum(current_weights.get(a, 0) * returns.loc[date, a]
                      for a in all_assets)
            pv *= (1 + dr)

        records.append({'date': date, 'portfolio_value': pv})

    return pd.DataFrame(records).set_index('date')

# ── 12. RUN ALL PORTFOLIOS ────────────────────────────────────────────────────
TC_LEVELS = {'0%': 0.00, '0.5%': 0.005, '1%': 0.01, '2%': 0.02}
FREQS     = ['monthly', 'quarterly']

# Equal-weight benchmarks
ew_results = {}
for freq in FREQS:
    for tc_label, tc_rate in TC_LEVELS.items():
        label = f"EW_{freq}_TC{tc_label}"
        ew_results[label] = run_equal_weight(
            returns, ALL_ASSETS, tc_rate=tc_rate,
            rebal_freq=freq, initial_capital=1e9
        )
        print(f"✓ {label}")

# Three-stage signal portfolios across all thresholds
signal_results = {}
signal_distributions = {}

for thresh in THRESHOLDS:
    sig_df_t = build_signal_df(composite_score, equity_assets,
                               macro_df.index, threshold=thresh)

    # Print signal distribution for this threshold
    n_risk_on  = (sig_df_t[equity_assets[0]] == +1).sum()
    n_risk_off = (sig_df_t[equity_assets[0]] == -1).sum()
    total      = len(sig_df_t)
    print(f"\nThreshold={thresh} | RISK_ON(+1): {n_risk_on} days "
          f"({n_risk_on/total*100:.1f}%) | "
          f"RISK_OFF(-1): {n_risk_off} days ({n_risk_off/total*100:.1f}%)")

    signal_distributions[thresh] = {'RISK_ON': n_risk_on, 'RISK_OFF': n_risk_off}

    for freq in FREQS:
        for tc_label, tc_rate in TC_LEVELS.items():
            label = f"Signal3S_t{thresh}_{freq}_TC{tc_label}"
            signal_results[label] = run_signal_portfolio_3stage(
                returns, sig_df_t, equity_assets, non_equity_assets,
                ALL_ASSETS, freq=freq, tc_rate=tc_rate,
                safe_haven_map=EQUITY_SAFE_HAVEN_MAP,
                initial_capital=1e9
            )
            print(f"  ✓ {label}")

print("\n✓ All portfolios computed")

# ── 13. PERFORMANCE METRICS ───────────────────────────────────────────────────
def compute_metrics(pv_df, label=''):
    pv        = pv_df['portfolio_value']
    daily_ret = pv.pct_change().dropna()
    n_years   = len(daily_ret) / 252
    total_ret = (pv.iloc[-1] / pv.iloc[0]) - 1
    cagr      = (1 + total_ret) ** (1 / n_years) - 1 if n_years > 0 else np.nan
    vol       = daily_ret.std() * np.sqrt(252)
    sharpe    = cagr / vol if vol > 0 else np.nan
    roll_max  = pv.cummax()
    max_dd    = ((pv - roll_max) / roll_max).min()
    calmar    = cagr / abs(max_dd) if max_dd != 0 else np.nan
    var_95    = daily_ret.quantile(0.05)

    return {
        'CAGR':         f"{cagr*100:.2f}%",
        'Volatility':   f"{vol*100:.2f}%",
        'Sharpe':       f"{sharpe:.2f}",
        'Max Drawdown': f"{max_dd*100:.2f}%",
        'Calmar':       f"{calmar:.2f}",
        'VaR 95%':      f"{var_95*100:.2f}%",
        'Total Return': f"{total_ret*100:.2f}%",
        'Final ($B)':   f"{pv.iloc[-1]/1e9:.3f}",
    }

# Print results grouped by threshold
ew_metrics = {k: compute_metrics(v, k) for k, v in ew_results.items()}

print("\n" + "="*90)
print("EQUAL-WEIGHT BENCHMARK")
print("="*90)
print(pd.DataFrame(ew_metrics).T.to_string())

sig_metrics = {k: compute_metrics(v, k) for k, v in signal_results.items()}

for thresh in THRESHOLDS:
    sub = {k: v for k, v in sig_metrics.items() if f"_t{thresh}_" in k}
    print(f"\n{'='*90}")
    print(f"THREE-STAGE SIGNAL — Threshold = {thresh} | "
          f"RISK_ON: {signal_distributions[thresh]['RISK_ON']} days | "
          f"RISK_OFF: {signal_distributions[thresh]['RISK_OFF']} days")
    print("="*90)
    print(pd.DataFrame(sub).T.to_string())

# ── 14. PLOTS ─────────────────────────────────────────────────────────────────
# One figure per threshold + one overview figure

colors_tc   = {'0%': 'black', '0.5%': 'steelblue', '1%': 'darkorange', '2%': 'crimson'}
colors_freq = {'monthly': 'steelblue', 'quarterly': 'darkorange'}

# ── Figure 1: Composite signal + safe-haven routing overview ──────────────────
fig, axes = plt.subplots(2, 1, figsize=(16, 10))
fig.suptitle("Modified Determinant — Composite Score & Threshold Signals",
             fontsize=14, fontweight='bold')

ax = axes[0]
ax.plot(composite_score.index, composite_score.values,
        color='navy', lw=1.0, alpha=0.8, label='Composite Z-score')
for thresh, col in zip(THRESHOLDS, ['green', 'orange', 'red']):
    ax.axhline( thresh, color=col, ls='--', lw=1.0, label=f'+{thresh}')
    ax.axhline(-thresh, color=col, ls='--', lw=1.0, label=f'-{thresh}')
ax.axhline(0, color='black', lw=0.6)
ax.fill_between(composite_score.index, composite_score.values, 0,
                where=(composite_score > 0), color='green', alpha=0.1)
ax.fill_between(composite_score.index, composite_score.values, 0,
                where=(composite_score <= 0), color='red', alpha=0.1)
ax.set_title("Composite Determinant Score with Thresholds", fontsize=11)
ax.set_ylabel("Z-score")
ax.legend(fontsize=8, ncol=4)
ax.grid(True, alpha=0.3)

ax = axes[1]
for window, series in det_results.items():
    ax.plot(series.index, np.sign(series.values), alpha=0.6,
            label=f"{window}d window", lw=1.0)
ax.axhline(0, color='black', lw=0.8)
ax.set_title("Sign of Det(R-I) per Window", fontsize=11)
ax.set_ylabel("+1 / -1")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("composite_signal_thresholds.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Figure 2: Portfolio values per threshold (TC=0.5%, monthly) ───────────────
fig, axes = plt.subplots(1, 3, figsize=(20, 6), sharey=False)
fig.suptitle("Three-Stage Signal Portfolio — Value by Threshold (TC=0.5%, Monthly)",
             fontsize=13, fontweight='bold')

for ax, thresh in zip(axes, THRESHOLDS):
    # EW reference
    ew_pv = ew_results['EW_monthly_TC0.5%']['portfolio_value']
    ax.plot(ew_pv.index, ew_pv / 1e9, '--', color='grey',
            lw=1.5, label='EW Benchmark')

    label = f"Signal3S_t{thresh}_monthly_TC0.5%"
    pv    = signal_results[label]['portfolio_value']
    ax.plot(pv.index, pv / 1e9, color='steelblue', lw=1.8,
            label=f'Signal (t={thresh})')

    ax.axhline(1.0, color='black', ls=':', lw=0.8)
    ax.set_title(f"Threshold = {thresh}", fontsize=11)
    ax.set_ylabel("Value ($B)")
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("signal_3stage_by_threshold.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Figure 3: Drawdowns per threshold ────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(20, 5), sharey=True)
fig.suptitle("Three-Stage Signal Portfolio — Drawdowns by Threshold (TC=0.5%, Monthly)",
             fontsize=13, fontweight='bold')

for ax, thresh in zip(axes, THRESHOLDS):
    label  = f"Signal3S_t{thresh}_monthly_TC0.5%"
    pv     = signal_results[label]['portfolio_value']
    dd     = (pv - pv.cummax()) / pv.cummax() * 100
    ew_pv  = ew_results['EW_monthly_TC0.5%']['portfolio_value']
    ew_dd  = (ew_pv - ew_pv.cummax()) / ew_pv.cummax() * 100

    ax.fill_between(dd.index,  dd.values,  0, alpha=0.3, color='steelblue')
    ax.fill_between(ew_dd.index, ew_dd.values, 0, alpha=0.2, color='grey')
    ax.plot(dd.index,    dd.values,    color='steelblue', lw=1.0,
            label=f'Signal (t={thresh})')
    ax.plot(ew_dd.index, ew_dd.values, color='grey',      lw=1.0, ls='--',
            label='EW Benchmark')
    ax.set_title(f"Threshold = {thresh}", fontsize=11)
    ax.set_ylabel("Drawdown (%)")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("signal_3stage_drawdowns.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Figure 4: TC sensitivity per threshold ────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.suptitle("Three-Stage Signal — TC Sensitivity (Monthly)",
             fontsize=13, fontweight='bold')

for ax, thresh in zip(axes, THRESHOLDS):
    for tc_label, col in colors_tc.items():
        label = f"Signal3S_t{thresh}_monthly_TC{tc_label}"
        pv    = signal_results[label]['portfolio_value']
        ax.plot(pv.index, pv / 1e9, color=col, lw=1.3,
                label=f'TC={tc_label}')
    ew_pv = ew_results['EW_monthly_TC0.5%']['portfolio_value']
    ax.plot(ew_pv.index, ew_pv / 1e9, '--', color='grey',
            lw=1.5, label='EW Benchmark')
    ax.set_title(f"Threshold = {thresh}", fontsize=11)
    ax.set_ylabel("Value ($B)")
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("signal_3stage_tc_sensitivity.png", dpi=150, bbox_inches='tight')
plt.show()

# ── 15. SAVE ──────────────────────────────────────────────────────────────────
for label, df in {**ew_results, **signal_results}.items():
    df.to_csv(f"{label}_portfolio.csv")

all_metrics = pd.DataFrame({**ew_metrics, **sig_metrics}).T
all_metrics.to_csv("all_portfolio_metrics.csv")

composite_score.to_csv("composite_signal.csv")

for thresh in THRESHOLDS:
    sig_df_t = build_signal_df(composite_score, equity_assets,
                               macro_df.index, threshold=thresh)
    sig_df_t.to_csv(f"signal_df_threshold_{thresh}.csv")

print("✓ All files saved.")


In [ ]:
# @title
# ============================================================
# SIGNAL PORTFOLIO — THREE-STAGE THRESHOLD VERSION
# Signals generated from Modified Determinant Model
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import requests
from io import StringIO

# ── 1. LOAD ALL DATA FROM GITHUB ─────────────────────────────────────────────
GITHUB_BASE = "https://raw.githubusercontent.com/kboroz/MSFE_Capstone_Project/main/01_Data/Streamlined/"

INDEX_FILE_MAP = {
    'ASX50':       'streamlined_asx_50.csv',
    'EUROSTOXX50': 'streamlined_euro_stoxx_50.csv',
    'FTSE100':     'streamlined_ftse_100.csv',
    'Ibovespa':    'streamlined_ibovespa.csv',
    'JSE40':       'streamlined_jse_top_40.csv',
    'NIKKEI225':   'streamlined_nikkei_225.csv',
    'SP500':       'streamlined_s&p_500.csv',
}

MACRO_FILE = 'streamlined_macro.csv'

def load_csv(url):
    r = requests.get(url)
    r.raise_for_status()
    df = pd.read_csv(StringIO(r.text), index_col=0, parse_dates=True)
    df.columns = df.columns.str.replace('[^a-zA-Z0-9]', '_', regex=True)
    return df.sort_index()

# Load macro
macro_df = load_csv(GITHUB_BASE + MACRO_FILE)
macro_df = macro_df[~macro_df.index.duplicated(keep='first')].sort_index()
print(f"macro_df shape : {macro_df.shape}")
print(f"Columns        : {macro_df.columns.tolist()}")

# Load individual index files
index_data = {}
for name, fname in INDEX_FILE_MAP.items():
    try:
        df = load_csv(GITHUB_BASE + fname)
        index_data[name] = df
        print(f"✓ {name:15s} | shape={df.shape} | cols={df.columns.tolist()[:4]}")
    except Exception as e:
        print(f"✗ {name}: {e}")

# ── 2. EXTRACT PRICE SERIES ───────────────────────────────────────────────────
PRICE_COLS = ['Close', 'close', 'Price', 'price', 'Adj_Close', 'Last']

def get_price_series(df, name):
    col = next((c for c in PRICE_COLS if c in df.columns), df.columns[0])
    s   = pd.to_numeric(df[col], errors='coerce').dropna()
    print(f"  {name}: using column '{col}'")
    return s

index_prices = {}
for name, df in index_data.items():
    index_prices[name] = get_price_series(df, name)

# ── 3. MODIFIED DETERMINANT SIGNAL MODEL ─────────────────────────────────────
DET_WINDOWS    = [3, 7, 28]
WINDOW_WEIGHTS = {3: 0.50, 7: 0.0, 28: 0.5}

def compute_modified_determinant(price_series_dict, windows=DET_WINDOWS):
    prices = pd.DataFrame(price_series_dict).sort_index()
    rets   = prices.pct_change().dropna()
    det_results = {}

    for window in windows:
        det_series = []
        for i in range(window, len(rets)):
            window_rets = rets.iloc[i - window: i]
            valid_cols  = window_rets.columns[window_rets.std() > 1e-10]
            w           = window_rets[valid_cols]

            if len(valid_cols) < 2:
                det_series.append(np.nan)
                continue
            try:
                R   = w.corr().values
                I   = np.eye(len(R))
                M   = R - I
                det = np.linalg.det(M)
            except Exception:
                det = np.nan
            det_series.append(det)

        idx = rets.index[window:]
        det_results[window] = pd.Series(det_series, index=idx, name=f'det_{window}d')
        print(f"  Window {window:2d}d | computed {len(det_series)} observations")

    return det_results

print("\n── Computing Modified Determinant ──")
det_results = compute_modified_determinant(index_prices)

# ── 4. COMPOSITE SCORE ────────────────────────────────────────────────────────
def compute_composite_score(det_results, window_weights=WINDOW_WEIGHTS):
    """
    Returns raw weighted-average of normalised determinant values (not signs).
    Normalised per window via rolling z-score so windows are comparable.
    """
    all_series = []
    for window, series in det_results.items():
        # z-score normalise so scale is comparable across windows
        mu    = series.rolling(252, min_periods=60).mean()
        sigma = series.rolling(252, min_periods=60).std()
        z     = (series - mu) / sigma.replace(0, np.nan)
        z.name = f'z_{window}d'
        all_series.append(z)

    z_df      = pd.concat(all_series, axis=1).dropna()
    composite = sum(
        z_df[f'z_{w}d'] * window_weights[w]
        for w in window_weights if f'z_{w}d' in z_df.columns
    )
    composite = composite / sum(window_weights.values())
    return composite, z_df

composite_score, z_df = compute_composite_score(det_results)

print(f"\nComposite score shape  : {composite_score.shape}")
print(f"Date range             : {composite_score.index[0].date()} → "
      f"{composite_score.index[-1].date()}")
print(f"Score range            : {composite_score.min():.3f} → "
      f"{composite_score.max():.3f}")

# ── 5. THREE-STAGE SIGNAL CLASSIFICATION ─────────────────────────────────────
# |score| > threshold  → RISK_ON  (+1): high det magnitude = abnormal regime
#                         → short equity, long country bonds + FX
# |score| <= threshold → RISK_OFF (-1): low det magnitude  = calm regime
#                         → long equity

THRESHOLDS = [0.2, 0.7, 1.2]   # tune here

def classify_signal(score, threshold):
    """
    Returns +1 (RISK_ON / defensive) or -1 (RISK_OFF / long equity)
    """
    if abs(score) > threshold:
        return +1   # high |det| → abnormal → defensive
    else:
        return -1   # low  |det| → calm     → long equity

def build_signal_df(composite_score, equity_assets, macro_dates, threshold=0.5):
    """
    Apply three-stage classification and reindex to portfolio dates.
    """
    binary = composite_score.apply(lambda x: classify_signal(x, threshold))
    signal_df = pd.DataFrame(index=macro_dates)
    for asset in equity_assets:
        signal_df[asset] = binary.reindex(macro_dates).ffill().bfill()
    return signal_df

# ── 6. ASSET UNIVERSE ─────────────────────────────────────────────────────────
EQUITY_ASSETS = list(index_prices.keys())
SAFE_BONDS    = ['US_10Y', 'UK_10Y', 'AU_10Y', 'ZA_10Y']
FX            = ['AUDUSD', 'BRLUSD', 'EURUSD', 'GBPUSD', 'JPYUSD', 'ZARUSD']
COMMODITIES   = ['Oil', 'Gold', 'Bitcoin']

ALL_ASSETS        = EQUITY_ASSETS + FX + COMMODITIES + SAFE_BONDS
ALL_ASSETS        = [a for a in ALL_ASSETS if a in macro_df.columns]
equity_assets     = [a for a in EQUITY_ASSETS if a in ALL_ASSETS]
non_equity_assets = [a for a in ALL_ASSETS if a not in equity_assets]

N           = len(ALL_ASSETS)
base_weight = 1.0 / N

print(f"\nTotal assets              : {N}")
print(f"Equity (signal-driven)    : {equity_assets}")
print(f"Non-equity (always long)  : {non_equity_assets}")
print(f"base_weight = 1/{N} = {base_weight:.4f}")

# ── 7. COUNTRY SAFE-HAVEN MAP ─────────────────────────────────────────────────
# When an equity index is shorted (RISK_ON), proceeds are redistributed
# to that country's bond + FX where available in the universe

EQUITY_SAFE_HAVEN_MAP = {
    'ASX50':       ['AU_10Y', 'AUDUSD'],
    'EUROSTOXX50': ['EURUSD'],            # no EU bond in universe
    'FTSE100':     ['UK_10Y', 'GBPUSD'],
    'Ibovespa':    ['BRLUSD'],            # no Brazil bond in universe
    'JSE40':       ['ZA_10Y', 'ZARUSD'],
    'NIKKEI225':   ['JPYUSD'],            # no JP bond in universe
    'SP500':       ['US_10Y'],            # USD is base currency
}

# Filter to assets actually in universe
UNIVERSE_SET = set(ALL_ASSETS)
for eq, havens in EQUITY_SAFE_HAVEN_MAP.items():
    EQUITY_SAFE_HAVEN_MAP[eq] = [h for h in havens if h in UNIVERSE_SET]

print("\nSafe-haven routing (filtered to universe):")
for eq, havens in EQUITY_SAFE_HAVEN_MAP.items():
    print(f"  {eq:15s} → {havens}")

# ── 8. BUILD RETURNS ──────────────────────────────────────────────────────────
returns_raw = macro_df[ALL_ASSETS].pct_change(fill_method=None)

for col in SAFE_BONDS:
    if col in returns_raw.columns:
        returns_raw[col] = -returns_raw[col]   # invert yield → price

if 'Oil' in macro_df.columns:
    bad_oil = macro_df['Oil'][macro_df['Oil'] <= 0].index
    for d in bad_oil:
        loc = returns_raw.index.get_loc(d)
        returns_raw.iloc[
            max(0, loc - 1): loc + 2,
            returns_raw.columns.get_loc('Oil')
        ] = 0.0

returns = returns_raw.clip(lower=-0.50, upper=1.00).fillna(0).iloc[1:]

assert not returns.isnull().any().any(), "NaNs in returns"
assert not np.isinf(returns.values).any(),  "Inf in returns"
print(f"\nReturns shape : {returns.shape}")
print("✓ Returns clean")

# ── 9. THREE-STAGE WEIGHT BUILDER ────────────────────────────────────────────
def build_weights_3stage(sig_row, equity_assets, non_equity_assets,
                         base_weight, date, safe_haven_map):
    """
    RISK_OFF (-1) : long equity at base_weight
    RISK_ON  (+1) : short equity, redistribute proceeds to country safe havens
                    or equally across all non-equity if no safe haven available
    Non-equity assets always start at base_weight; safe-haven assets receive
    additional weight from shorted equities.
    """
    raw = {a: base_weight for a in non_equity_assets}

    safe_haven_extra = {}   # accumulate extra weight per safe-haven asset

    for a in equity_assets:
        if sig_row is not None and a in sig_row.index and not pd.isna(sig_row[a]):
            sig = sig_row[a]
        else:
            sig = -1   # default: long equity

        if sig > 0:
            # RISK_ON: short equity, redirect base_weight to safe havens
            raw[a]   = -base_weight
            havens   = safe_haven_map.get(a, [])
            havens   = [h for h in havens if h in raw]   # only if in universe

            if havens:
                per_haven = base_weight / len(havens)
                for h in havens:
                    safe_haven_extra[h] = safe_haven_extra.get(h, 0) + per_haven
            else:
                # No safe haven available → spread across all non-equity
                per_ne = base_weight / len(non_equity_assets)
                for ne in non_equity_assets:
                    safe_haven_extra[ne] = safe_haven_extra.get(ne, 0) + per_ne
        else:
            # RISK_OFF: long equity
            raw[a] = base_weight

    # Apply safe-haven extras
    for asset, extra in safe_haven_extra.items():
        raw[asset] = raw.get(asset, 0) + extra

    # Normalise to sum = 1 (handles edge cases)
    total = sum(raw.values())
    if total <= 0:
        for a in raw:
            raw[a] = 1.0 / len(raw)
    else:
        for a in raw:
            raw[a] /= total

    return raw

# ── 10. EQUAL-WEIGHT BENCHMARK ────────────────────────────────────────────────
def run_equal_weight(returns, all_assets, tc_rate=0.005,
                     rebal_freq='monthly', initial_capital=1e9):
    N           = len(all_assets)
    base_weight = 1.0 / N
    ew          = {a: base_weight for a in all_assets}

    freq_code     = 'MS' if rebal_freq == 'monthly' else 'QS'
    reb_dates_raw = returns.resample(freq_code).first().index
    actual_dates  = returns.index

    reb_set = set()
    for d in reb_dates_raw:
        future = actual_dates[actual_dates >= d]
        if len(future) > 0:
            reb_set.add(future[0])

    pv              = float(initial_capital)
    current_weights = None
    records         = []

    for date in actual_dates:
        if date in reb_set:
            if current_weights is not None:
                turnover = sum(abs(ew.get(a, 0) - current_weights.get(a, 0))
                               for a in set(ew) | set(current_weights)) / 2.0
                pv *= (1 - turnover * tc_rate)
            else:
                pv *= (1 - tc_rate)
            current_weights = ew.copy()

        if current_weights is not None:
            dr  = sum(current_weights.get(a, 0) * returns.loc[date, a]
                      for a in all_assets)
            pv *= (1 + dr)

        records.append({'date': date, 'portfolio_value': pv})

    return pd.DataFrame(records).set_index('date')

# ── 11. THREE-STAGE SIGNAL PORTFOLIO RUNNER ───────────────────────────────────
def run_signal_portfolio_3stage(returns, signal_df, equity_assets,
                                non_equity_assets, all_assets,
                                freq='monthly', tc_rate=0.005,
                                safe_haven_map=None,
                                initial_capital=1e9):

    if safe_haven_map is None:
        safe_haven_map = {}

    N           = len(all_assets)
    base_weight = 1.0 / N

    freq_code     = 'MS' if freq == 'monthly' else 'QS'
    reb_dates_raw = returns.resample(freq_code).first().index
    actual_dates  = returns.index

    reb_set = set()
    for d in reb_dates_raw:
        future = actual_dates[actual_dates >= d]
        if len(future) > 0:
            reb_set.add(future[0])

    sig = signal_df.copy()
    sig.index = pd.to_datetime(sig.index)

    pv              = float(initial_capital)
    current_weights = None
    records         = []

    for date in actual_dates:
        if date in reb_set:
            past    = sig[sig.index <= date]
            sig_row = past.iloc[-1] if len(past) > 0 else None

            raw = build_weights_3stage(
                sig_row, equity_assets, non_equity_assets,
                base_weight, date, safe_haven_map
            )

            if current_weights is not None:
                all_keys = set(raw) | set(current_weights)
                turnover = sum(
                    abs(raw.get(a, 0) - current_weights.get(a, 0))
                    for a in all_keys
                ) / 2.0
                pv *= (1 - turnover * tc_rate)
            else:
                pv *= (1 - tc_rate)

            current_weights = raw.copy()

        if current_weights is not None:
            dr  = sum(current_weights.get(a, 0) * returns.loc[date, a]
                      for a in all_assets)
            pv *= (1 + dr)

        records.append({'date': date, 'portfolio_value': pv})

    return pd.DataFrame(records).set_index('date')

# ── 12. RUN ALL PORTFOLIOS ────────────────────────────────────────────────────
TC_LEVELS = {'0%': 0.00, '0.5%': 0.005, '1%': 0.01, '2%': 0.02}
FREQS     = ['monthly', 'quarterly']

# Equal-weight benchmarks
ew_results = {}
for freq in FREQS:
    for tc_label, tc_rate in TC_LEVELS.items():
        label = f"EW_{freq}_TC{tc_label}"
        ew_results[label] = run_equal_weight(
            returns, ALL_ASSETS, tc_rate=tc_rate,
            rebal_freq=freq, initial_capital=1e9
        )
        print(f"✓ {label}")

# Three-stage signal portfolios across all thresholds
signal_results = {}
signal_distributions = {}

for thresh in THRESHOLDS:
    sig_df_t = build_signal_df(composite_score, equity_assets,
                               macro_df.index, threshold=thresh)

    # Print signal distribution for this threshold
    n_risk_on  = (sig_df_t[equity_assets[0]] == +1).sum()
    n_risk_off = (sig_df_t[equity_assets[0]] == -1).sum()
    total      = len(sig_df_t)
    print(f"\nThreshold={thresh} | RISK_ON(+1): {n_risk_on} days "
          f"({n_risk_on/total*100:.1f}%) | "
          f"RISK_OFF(-1): {n_risk_off} days ({n_risk_off/total*100:.1f}%)")

    signal_distributions[thresh] = {'RISK_ON': n_risk_on, 'RISK_OFF': n_risk_off}

    for freq in FREQS:
        for tc_label, tc_rate in TC_LEVELS.items():
            label = f"Signal3S_t{thresh}_{freq}_TC{tc_label}"
            signal_results[label] = run_signal_portfolio_3stage(
                returns, sig_df_t, equity_assets, non_equity_assets,
                ALL_ASSETS, freq=freq, tc_rate=tc_rate,
                safe_haven_map=EQUITY_SAFE_HAVEN_MAP,
                initial_capital=1e9
            )
            print(f"  ✓ {label}")

print("\n✓ All portfolios computed")

# ── 13. PERFORMANCE METRICS ───────────────────────────────────────────────────
def compute_metrics(pv_df, label=''):
    pv        = pv_df['portfolio_value']
    daily_ret = pv.pct_change().dropna()
    n_years   = len(daily_ret) / 252
    total_ret = (pv.iloc[-1] / pv.iloc[0]) - 1
    cagr      = (1 + total_ret) ** (1 / n_years) - 1 if n_years > 0 else np.nan
    vol       = daily_ret.std() * np.sqrt(252)
    sharpe    = cagr / vol if vol > 0 else np.nan
    roll_max  = pv.cummax()
    max_dd    = ((pv - roll_max) / roll_max).min()
    calmar    = cagr / abs(max_dd) if max_dd != 0 else np.nan
    var_95    = daily_ret.quantile(0.05)

    return {
        'CAGR':         f"{cagr*100:.2f}%",
        'Volatility':   f"{vol*100:.2f}%",
        'Sharpe':       f"{sharpe:.2f}",
        'Max Drawdown': f"{max_dd*100:.2f}%",
        'Calmar':       f"{calmar:.2f}",
        'VaR 95%':      f"{var_95*100:.2f}%",
        'Total Return': f"{total_ret*100:.2f}%",
        'Final ($B)':   f"{pv.iloc[-1]/1e9:.3f}",
    }

# Print results grouped by threshold
ew_metrics = {k: compute_metrics(v, k) for k, v in ew_results.items()}

print("\n" + "="*90)
print("EQUAL-WEIGHT BENCHMARK")
print("="*90)
print(pd.DataFrame(ew_metrics).T.to_string())

sig_metrics = {k: compute_metrics(v, k) for k, v in signal_results.items()}

for thresh in THRESHOLDS:
    sub = {k: v for k, v in sig_metrics.items() if f"_t{thresh}_" in k}
    print(f"\n{'='*90}")
    print(f"THREE-STAGE SIGNAL — Threshold = {thresh} | "
          f"RISK_ON: {signal_distributions[thresh]['RISK_ON']} days | "
          f"RISK_OFF: {signal_distributions[thresh]['RISK_OFF']} days")
    print("="*90)
    print(pd.DataFrame(sub).T.to_string())

# ── 14. PLOTS ─────────────────────────────────────────────────────────────────
# One figure per threshold + one overview figure

colors_tc   = {'0%': 'black', '0.5%': 'steelblue', '1%': 'darkorange', '2%': 'crimson'}
colors_freq = {'monthly': 'steelblue', 'quarterly': 'darkorange'}

# ── Figure 1: Composite signal + safe-haven routing overview ──────────────────
fig, axes = plt.subplots(2, 1, figsize=(16, 10))
fig.suptitle("Modified Determinant — Composite Score & Threshold Signals",
             fontsize=14, fontweight='bold')

ax = axes[0]
ax.plot(composite_score.index, composite_score.values,
        color='navy', lw=1.0, alpha=0.8, label='Composite Z-score')
for thresh, col in zip(THRESHOLDS, ['green', 'orange', 'red']):
    ax.axhline( thresh, color=col, ls='--', lw=1.0, label=f'+{thresh}')
    ax.axhline(-thresh, color=col, ls='--', lw=1.0, label=f'-{thresh}')
ax.axhline(0, color='black', lw=0.6)
ax.fill_between(composite_score.index, composite_score.values, 0,
                where=(composite_score > 0), color='green', alpha=0.1)
ax.fill_between(composite_score.index, composite_score.values, 0,
                where=(composite_score <= 0), color='red', alpha=0.1)
ax.set_title("Composite Determinant Score with Thresholds", fontsize=11)
ax.set_ylabel("Z-score")
ax.legend(fontsize=8, ncol=4)
ax.grid(True, alpha=0.3)

ax = axes[1]
for window, series in det_results.items():
    ax.plot(series.index, np.sign(series.values), alpha=0.6,
            label=f"{window}d window", lw=1.0)
ax.axhline(0, color='black', lw=0.8)
ax.set_title("Sign of Det(R-I) per Window", fontsize=11)
ax.set_ylabel("+1 / -1")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("composite_signal_thresholds.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Figure 2: Portfolio values per threshold (TC=0.5%, monthly) ───────────────
fig, axes = plt.subplots(1, 3, figsize=(20, 6), sharey=False)
fig.suptitle("Three-Stage Signal Portfolio — Value by Threshold (TC=0.5%, Monthly)",
             fontsize=13, fontweight='bold')

for ax, thresh in zip(axes, THRESHOLDS):
    # EW reference
    ew_pv = ew_results['EW_monthly_TC0.5%']['portfolio_value']
    ax.plot(ew_pv.index, ew_pv / 1e9, '--', color='grey',
            lw=1.5, label='EW Benchmark')

    label = f"Signal3S_t{thresh}_monthly_TC0.5%"
    pv    = signal_results[label]['portfolio_value']
    ax.plot(pv.index, pv / 1e9, color='steelblue', lw=1.8,
            label=f'Signal (t={thresh})')

    ax.axhline(1.0, color='black', ls=':', lw=0.8)
    ax.set_title(f"Threshold = {thresh}", fontsize=11)
    ax.set_ylabel("Value ($B)")
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("signal_3stage_by_threshold.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Figure 3: Drawdowns per threshold ────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(20, 5), sharey=True)
fig.suptitle("Three-Stage Signal Portfolio — Drawdowns by Threshold (TC=0.5%, Monthly)",
             fontsize=13, fontweight='bold')

for ax, thresh in zip(axes, THRESHOLDS):
    label  = f"Signal3S_t{thresh}_monthly_TC0.5%"
    pv     = signal_results[label]['portfolio_value']
    dd     = (pv - pv.cummax()) / pv.cummax() * 100
    ew_pv  = ew_results['EW_monthly_TC0.5%']['portfolio_value']
    ew_dd  = (ew_pv - ew_pv.cummax()) / ew_pv.cummax() * 100

    ax.fill_between(dd.index,  dd.values,  0, alpha=0.3, color='steelblue')
    ax.fill_between(ew_dd.index, ew_dd.values, 0, alpha=0.2, color='grey')
    ax.plot(dd.index,    dd.values,    color='steelblue', lw=1.0,
            label=f'Signal (t={thresh})')
    ax.plot(ew_dd.index, ew_dd.values, color='grey',      lw=1.0, ls='--',
            label='EW Benchmark')
    ax.set_title(f"Threshold = {thresh}", fontsize=11)
    ax.set_ylabel("Drawdown (%)")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("signal_3stage_drawdowns.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Figure 4: TC sensitivity per threshold ────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.suptitle("Three-Stage Signal — TC Sensitivity (Monthly)",
             fontsize=13, fontweight='bold')

for ax, thresh in zip(axes, THRESHOLDS):
    for tc_label, col in colors_tc.items():
        label = f"Signal3S_t{thresh}_monthly_TC{tc_label}"
        pv    = signal_results[label]['portfolio_value']
        ax.plot(pv.index, pv / 1e9, color=col, lw=1.3,
                label=f'TC={tc_label}')
    ew_pv = ew_results['EW_monthly_TC0.5%']['portfolio_value']
    ax.plot(ew_pv.index, ew_pv / 1e9, '--', color='grey',
            lw=1.5, label='EW Benchmark')
    ax.set_title(f"Threshold = {thresh}", fontsize=11)
    ax.set_ylabel("Value ($B)")
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("signal_3stage_tc_sensitivity.png", dpi=150, bbox_inches='tight')
plt.show()

# ── 15. SAVE ──────────────────────────────────────────────────────────────────
for label, df in {**ew_results, **signal_results}.items():
    df.to_csv(f"{label}_portfolio.csv")

all_metrics = pd.DataFrame({**ew_metrics, **sig_metrics}).T
all_metrics.to_csv("all_portfolio_metrics.csv")

composite_score.to_csv("composite_signal.csv")

for thresh in THRESHOLDS:
    sig_df_t = build_signal_df(composite_score, equity_assets,
                               macro_df.index, threshold=thresh)
    sig_df_t.to_csv(f"signal_df_threshold_{thresh}.csv")

print("✓ All files saved.")


In [ ]:
# @title
# ============================================================
# SIGNAL PORTFOLIO — THREE-STAGE THRESHOLD VERSION
# Signals generated from Modified Determinant Model
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import requests
from io import StringIO

# ── 1. LOAD ALL DATA FROM GITHUB ─────────────────────────────────────────────
GITHUB_BASE = "https://raw.githubusercontent.com/kboroz/MSFE_Capstone_Project/main/01_Data/Streamlined/"

INDEX_FILE_MAP = {
    'ASX50':       'streamlined_asx_50.csv',
    'EUROSTOXX50': 'streamlined_euro_stoxx_50.csv',
    'FTSE100':     'streamlined_ftse_100.csv',
    'Ibovespa':    'streamlined_ibovespa.csv',
    'JSE40':       'streamlined_jse_top_40.csv',
    'NIKKEI225':   'streamlined_nikkei_225.csv',
    'SP500':       'streamlined_s&p_500.csv',
}

MACRO_FILE = 'streamlined_macro.csv'

def load_csv(url):
    r = requests.get(url)
    r.raise_for_status()
    df = pd.read_csv(StringIO(r.text), index_col=0, parse_dates=True)
    df.columns = df.columns.str.replace('[^a-zA-Z0-9]', '_', regex=True)
    return df.sort_index()

# Load macro
macro_df = load_csv(GITHUB_BASE + MACRO_FILE)
macro_df = macro_df[~macro_df.index.duplicated(keep='first')].sort_index()
print(f"macro_df shape : {macro_df.shape}")
print(f"Columns        : {macro_df.columns.tolist()}")

# Load individual index files
index_data = {}
for name, fname in INDEX_FILE_MAP.items():
    try:
        df = load_csv(GITHUB_BASE + fname)
        index_data[name] = df
        print(f"✓ {name:15s} | shape={df.shape} | cols={df.columns.tolist()[:4]}")
    except Exception as e:
        print(f"✗ {name}: {e}")

# ── 2. EXTRACT PRICE SERIES ───────────────────────────────────────────────────
PRICE_COLS = ['Close', 'close', 'Price', 'price', 'Adj_Close', 'Last']

def get_price_series(df, name):
    col = next((c for c in PRICE_COLS if c in df.columns), df.columns[0])
    s   = pd.to_numeric(df[col], errors='coerce').dropna()
    print(f"  {name}: using column '{col}'")
    return s

index_prices = {}
for name, df in index_data.items():
    index_prices[name] = get_price_series(df, name)

# ── 3. MODIFIED DETERMINANT SIGNAL MODEL ─────────────────────────────────────
DET_WINDOWS    = [3, 7, 28]
WINDOW_WEIGHTS = {3: 0.15, 7: 0.25, 28: 0.6}

def compute_modified_determinant(price_series_dict, windows=DET_WINDOWS):
    prices = pd.DataFrame(price_series_dict).sort_index()
    rets   = prices.pct_change().dropna()
    det_results = {}

    for window in windows:
        det_series = []
        for i in range(window, len(rets)):
            window_rets = rets.iloc[i - window: i]
            valid_cols  = window_rets.columns[window_rets.std() > 1e-10]
            w           = window_rets[valid_cols]

            if len(valid_cols) < 2:
                det_series.append(np.nan)
                continue
            try:
                R   = w.corr().values
                I   = np.eye(len(R))
                M   = R - I
                det = np.linalg.det(M)
            except Exception:
                det = np.nan
            det_series.append(det)

        idx = rets.index[window:]
        det_results[window] = pd.Series(det_series, index=idx, name=f'det_{window}d')
        print(f"  Window {window:2d}d | computed {len(det_series)} observations")

    return det_results

print("\n── Computing Modified Determinant ──")
det_results = compute_modified_determinant(index_prices)

# ── 4. COMPOSITE SCORE ────────────────────────────────────────────────────────
def compute_composite_score(det_results, window_weights=WINDOW_WEIGHTS):
    """
    Returns raw weighted-average of normalised determinant values (not signs).
    Normalised per window via rolling z-score so windows are comparable.
    """
    all_series = []
    for window, series in det_results.items():
        # z-score normalise so scale is comparable across windows
        mu    = series.rolling(252, min_periods=60).mean()
        sigma = series.rolling(252, min_periods=60).std()
        z     = (series - mu) / sigma.replace(0, np.nan)
        z.name = f'z_{window}d'
        all_series.append(z)

    z_df      = pd.concat(all_series, axis=1).dropna()
    composite = sum(
        z_df[f'z_{w}d'] * window_weights[w]
        for w in window_weights if f'z_{w}d' in z_df.columns
    )
    composite = composite / sum(window_weights.values())
    return composite, z_df

composite_score, z_df = compute_composite_score(det_results)

print(f"\nComposite score shape  : {composite_score.shape}")
print(f"Date range             : {composite_score.index[0].date()} → "
      f"{composite_score.index[-1].date()}")
print(f"Score range            : {composite_score.min():.3f} → "
      f"{composite_score.max():.3f}")

# ── 5. THREE-STAGE SIGNAL CLASSIFICATION ─────────────────────────────────────
# |score| > threshold  → RISK_ON  (+1): high det magnitude = abnormal regime
#                         → short equity, long country bonds + FX
# |score| <= threshold → RISK_OFF (-1): low det magnitude  = calm regime
#                         → long equity

THRESHOLDS = [0.2, 0.7, 1.2]   # tune here

def classify_signal(score, threshold):
    """
    Returns +1 (RISK_ON / defensive) or -1 (RISK_OFF / long equity)
    """
    if abs(score) > threshold:
        return +1   # high |det| → abnormal → defensive
    else:
        return -1   # low  |det| → calm     → long equity

def build_signal_df(composite_score, equity_assets, macro_dates, threshold=0.5):
    """
    Apply three-stage classification and reindex to portfolio dates.
    """
    binary = composite_score.apply(lambda x: classify_signal(x, threshold))
    signal_df = pd.DataFrame(index=macro_dates)
    for asset in equity_assets:
        signal_df[asset] = binary.reindex(macro_dates).ffill().bfill()
    return signal_df

# ── 6. ASSET UNIVERSE ─────────────────────────────────────────────────────────
EQUITY_ASSETS = list(index_prices.keys())
SAFE_BONDS    = ['US_10Y', 'UK_10Y', 'AU_10Y', 'ZA_10Y']
FX            = ['AUDUSD', 'BRLUSD', 'EURUSD', 'GBPUSD', 'JPYUSD', 'ZARUSD']
COMMODITIES   = ['Oil', 'Gold', 'Bitcoin']

ALL_ASSETS        = EQUITY_ASSETS + FX + COMMODITIES + SAFE_BONDS
ALL_ASSETS        = [a for a in ALL_ASSETS if a in macro_df.columns]
equity_assets     = [a for a in EQUITY_ASSETS if a in ALL_ASSETS]
non_equity_assets = [a for a in ALL_ASSETS if a not in equity_assets]

N           = len(ALL_ASSETS)
base_weight = 1.0 / N

print(f"\nTotal assets              : {N}")
print(f"Equity (signal-driven)    : {equity_assets}")
print(f"Non-equity (always long)  : {non_equity_assets}")
print(f"base_weight = 1/{N} = {base_weight:.4f}")

# ── 7. COUNTRY SAFE-HAVEN MAP ─────────────────────────────────────────────────
# When an equity index is shorted (RISK_ON), proceeds are redistributed
# to that country's bond + FX where available in the universe

EQUITY_SAFE_HAVEN_MAP = {
    'ASX50':       ['AU_10Y', 'AUDUSD'],
    'EUROSTOXX50': ['EURUSD'],            # no EU bond in universe
    'FTSE100':     ['UK_10Y', 'GBPUSD'],
    'Ibovespa':    ['BRLUSD'],            # no Brazil bond in universe
    'JSE40':       ['ZA_10Y', 'ZARUSD'],
    'NIKKEI225':   ['JPYUSD'],            # no JP bond in universe
    'SP500':       ['US_10Y'],            # USD is base currency
}

# Filter to assets actually in universe
UNIVERSE_SET = set(ALL_ASSETS)
for eq, havens in EQUITY_SAFE_HAVEN_MAP.items():
    EQUITY_SAFE_HAVEN_MAP[eq] = [h for h in havens if h in UNIVERSE_SET]

print("\nSafe-haven routing (filtered to universe):")
for eq, havens in EQUITY_SAFE_HAVEN_MAP.items():
    print(f"  {eq:15s} → {havens}")

# ── 8. BUILD RETURNS ──────────────────────────────────────────────────────────
returns_raw = macro_df[ALL_ASSETS].pct_change(fill_method=None)

for col in SAFE_BONDS:
    if col in returns_raw.columns:
        returns_raw[col] = -returns_raw[col]   # invert yield → price

if 'Oil' in macro_df.columns:
    bad_oil = macro_df['Oil'][macro_df['Oil'] <= 0].index
    for d in bad_oil:
        loc = returns_raw.index.get_loc(d)
        returns_raw.iloc[
            max(0, loc - 1): loc + 2,
            returns_raw.columns.get_loc('Oil')
        ] = 0.0

returns = returns_raw.clip(lower=-0.50, upper=1.00).fillna(0).iloc[1:]

assert not returns.isnull().any().any(), "NaNs in returns"
assert not np.isinf(returns.values).any(),  "Inf in returns"
print(f"\nReturns shape : {returns.shape}")
print("✓ Returns clean")

# ── 9. THREE-STAGE WEIGHT BUILDER ────────────────────────────────────────────
def build_weights_3stage(sig_row, equity_assets, non_equity_assets,
                         base_weight, date, safe_haven_map):
    """
    RISK_OFF (-1) : long equity at base_weight
    RISK_ON  (+1) : short equity, redistribute proceeds to country safe havens
                    or equally across all non-equity if no safe haven available
    Non-equity assets always start at base_weight; safe-haven assets receive
    additional weight from shorted equities.
    """
    raw = {a: base_weight for a in non_equity_assets}

    safe_haven_extra = {}   # accumulate extra weight per safe-haven asset

    for a in equity_assets:
        if sig_row is not None and a in sig_row.index and not pd.isna(sig_row[a]):
            sig = sig_row[a]
        else:
            sig = -1   # default: long equity

        if sig > 0:
            # RISK_ON: short equity, redirect base_weight to safe havens
            raw[a]   = -base_weight
            havens   = safe_haven_map.get(a, [])
            havens   = [h for h in havens if h in raw]   # only if in universe

            if havens:
                per_haven = base_weight / len(havens)
                for h in havens:
                    safe_haven_extra[h] = safe_haven_extra.get(h, 0) + per_haven
            else:
                # No safe haven available → spread across all non-equity
                per_ne = base_weight / len(non_equity_assets)
                for ne in non_equity_assets:
                    safe_haven_extra[ne] = safe_haven_extra.get(ne, 0) + per_ne
        else:
            # RISK_OFF: long equity
            raw[a] = base_weight

    # Apply safe-haven extras
    for asset, extra in safe_haven_extra.items():
        raw[asset] = raw.get(asset, 0) + extra

    # Normalise to sum = 1 (handles edge cases)
    total = sum(raw.values())
    if total <= 0:
        for a in raw:
            raw[a] = 1.0 / len(raw)
    else:
        for a in raw:
            raw[a] /= total

    return raw

# ── 10. EQUAL-WEIGHT BENCHMARK ────────────────────────────────────────────────
def run_equal_weight(returns, all_assets, tc_rate=0.005,
                     rebal_freq='monthly', initial_capital=1e9):
    N           = len(all_assets)
    base_weight = 1.0 / N
    ew          = {a: base_weight for a in all_assets}

    freq_code     = 'MS' if rebal_freq == 'monthly' else 'QS'
    reb_dates_raw = returns.resample(freq_code).first().index
    actual_dates  = returns.index

    reb_set = set()
    for d in reb_dates_raw:
        future = actual_dates[actual_dates >= d]
        if len(future) > 0:
            reb_set.add(future[0])

    pv              = float(initial_capital)
    current_weights = None
    records         = []

    for date in actual_dates:
        if date in reb_set:
            if current_weights is not None:
                turnover = sum(abs(ew.get(a, 0) - current_weights.get(a, 0))
                               for a in set(ew) | set(current_weights)) / 2.0
                pv *= (1 - turnover * tc_rate)
            else:
                pv *= (1 - tc_rate)
            current_weights = ew.copy()

        if current_weights is not None:
            dr  = sum(current_weights.get(a, 0) * returns.loc[date, a]
                      for a in all_assets)
            pv *= (1 + dr)

        records.append({'date': date, 'portfolio_value': pv})

    return pd.DataFrame(records).set_index('date')

# ── 11. THREE-STAGE SIGNAL PORTFOLIO RUNNER ───────────────────────────────────
def run_signal_portfolio_3stage(returns, signal_df, equity_assets,
                                non_equity_assets, all_assets,
                                freq='monthly', tc_rate=0.005,
                                safe_haven_map=None,
                                initial_capital=1e9):

    if safe_haven_map is None:
        safe_haven_map = {}

    N           = len(all_assets)
    base_weight = 1.0 / N

    freq_code     = 'MS' if freq == 'monthly' else 'QS'
    reb_dates_raw = returns.resample(freq_code).first().index
    actual_dates  = returns.index

    reb_set = set()
    for d in reb_dates_raw:
        future = actual_dates[actual_dates >= d]
        if len(future) > 0:
            reb_set.add(future[0])

    sig = signal_df.copy()
    sig.index = pd.to_datetime(sig.index)

    pv              = float(initial_capital)
    current_weights = None
    records         = []

    for date in actual_dates:
        if date in reb_set:
            past    = sig[sig.index <= date]
            sig_row = past.iloc[-1] if len(past) > 0 else None

            raw = build_weights_3stage(
                sig_row, equity_assets, non_equity_assets,
                base_weight, date, safe_haven_map
            )

            if current_weights is not None:
                all_keys = set(raw) | set(current_weights)
                turnover = sum(
                    abs(raw.get(a, 0) - current_weights.get(a, 0))
                    for a in all_keys
                ) / 2.0
                pv *= (1 - turnover * tc_rate)
            else:
                pv *= (1 - tc_rate)

            current_weights = raw.copy()

        if current_weights is not None:
            dr  = sum(current_weights.get(a, 0) * returns.loc[date, a]
                      for a in all_assets)
            pv *= (1 + dr)

        records.append({'date': date, 'portfolio_value': pv})

    return pd.DataFrame(records).set_index('date')

# ── 12. RUN ALL PORTFOLIOS ────────────────────────────────────────────────────
TC_LEVELS = {'0%': 0.00, '0.5%': 0.005, '1%': 0.01, '2%': 0.02}
FREQS     = ['monthly', 'quarterly']

# Equal-weight benchmarks
ew_results = {}
for freq in FREQS:
    for tc_label, tc_rate in TC_LEVELS.items():
        label = f"EW_{freq}_TC{tc_label}"
        ew_results[label] = run_equal_weight(
            returns, ALL_ASSETS, tc_rate=tc_rate,
            rebal_freq=freq, initial_capital=1e9
        )
        print(f"✓ {label}")

# Three-stage signal portfolios across all thresholds
signal_results = {}
signal_distributions = {}

for thresh in THRESHOLDS:
    sig_df_t = build_signal_df(composite_score, equity_assets,
                               macro_df.index, threshold=thresh)

    # Print signal distribution for this threshold
    n_risk_on  = (sig_df_t[equity_assets[0]] == +1).sum()
    n_risk_off = (sig_df_t[equity_assets[0]] == -1).sum()
    total      = len(sig_df_t)
    print(f"\nThreshold={thresh} | RISK_ON(+1): {n_risk_on} days "
          f"({n_risk_on/total*100:.1f}%) | "
          f"RISK_OFF(-1): {n_risk_off} days ({n_risk_off/total*100:.1f}%)")

    signal_distributions[thresh] = {'RISK_ON': n_risk_on, 'RISK_OFF': n_risk_off}

    for freq in FREQS:
        for tc_label, tc_rate in TC_LEVELS.items():
            label = f"Signal3S_t{thresh}_{freq}_TC{tc_label}"
            signal_results[label] = run_signal_portfolio_3stage(
                returns, sig_df_t, equity_assets, non_equity_assets,
                ALL_ASSETS, freq=freq, tc_rate=tc_rate,
                safe_haven_map=EQUITY_SAFE_HAVEN_MAP,
                initial_capital=1e9
            )
            print(f"  ✓ {label}")

print("\n✓ All portfolios computed")

# ── 13. PERFORMANCE METRICS ───────────────────────────────────────────────────
def compute_metrics(pv_df, label=''):
    pv        = pv_df['portfolio_value']
    daily_ret = pv.pct_change().dropna()
    n_years   = len(daily_ret) / 252
    total_ret = (pv.iloc[-1] / pv.iloc[0]) - 1
    cagr      = (1 + total_ret) ** (1 / n_years) - 1 if n_years > 0 else np.nan
    vol       = daily_ret.std() * np.sqrt(252)
    sharpe    = cagr / vol if vol > 0 else np.nan
    roll_max  = pv.cummax()
    max_dd    = ((pv - roll_max) / roll_max).min()
    calmar    = cagr / abs(max_dd) if max_dd != 0 else np.nan
    var_95    = daily_ret.quantile(0.05)

    return {
        'CAGR':         f"{cagr*100:.2f}%",
        'Volatility':   f"{vol*100:.2f}%",
        'Sharpe':       f"{sharpe:.2f}",
        'Max Drawdown': f"{max_dd*100:.2f}%",
        'Calmar':       f"{calmar:.2f}",
        'VaR 95%':      f"{var_95*100:.2f}%",
        'Total Return': f"{total_ret*100:.2f}%",
        'Final ($B)':   f"{pv.iloc[-1]/1e9:.3f}",
    }

# Print results grouped by threshold
ew_metrics = {k: compute_metrics(v, k) for k, v in ew_results.items()}

print("\n" + "="*90)
print("EQUAL-WEIGHT BENCHMARK")
print("="*90)
print(pd.DataFrame(ew_metrics).T.to_string())

sig_metrics = {k: compute_metrics(v, k) for k, v in signal_results.items()}

for thresh in THRESHOLDS:
    sub = {k: v for k, v in sig_metrics.items() if f"_t{thresh}_" in k}
    print(f"\n{'='*90}")
    print(f"THREE-STAGE SIGNAL — Threshold = {thresh} | "
          f"RISK_ON: {signal_distributions[thresh]['RISK_ON']} days | "
          f"RISK_OFF: {signal_distributions[thresh]['RISK_OFF']} days")
    print("="*90)
    print(pd.DataFrame(sub).T.to_string())

# ── 14. PLOTS ─────────────────────────────────────────────────────────────────
# One figure per threshold + one overview figure

colors_tc   = {'0%': 'black', '0.5%': 'steelblue', '1%': 'darkorange', '2%': 'crimson'}
colors_freq = {'monthly': 'steelblue', 'quarterly': 'darkorange'}

# ── Figure 1: Composite signal + safe-haven routing overview ──────────────────
fig, axes = plt.subplots(2, 1, figsize=(16, 10))
fig.suptitle("Modified Determinant — Composite Score & Threshold Signals",
             fontsize=14, fontweight='bold')

ax = axes[0]
ax.plot(composite_score.index, composite_score.values,
        color='navy', lw=1.0, alpha=0.8, label='Composite Z-score')
for thresh, col in zip(THRESHOLDS, ['green', 'orange', 'red']):
    ax.axhline( thresh, color=col, ls='--', lw=1.0, label=f'+{thresh}')
    ax.axhline(-thresh, color=col, ls='--', lw=1.0, label=f'-{thresh}')
ax.axhline(0, color='black', lw=0.6)
ax.fill_between(composite_score.index, composite_score.values, 0,
                where=(composite_score > 0), color='green', alpha=0.1)
ax.fill_between(composite_score.index, composite_score.values, 0,
                where=(composite_score <= 0), color='red', alpha=0.1)
ax.set_title("Composite Determinant Score with Thresholds", fontsize=11)
ax.set_ylabel("Z-score")
ax.legend(fontsize=8, ncol=4)
ax.grid(True, alpha=0.3)

ax = axes[1]
for window, series in det_results.items():
    ax.plot(series.index, np.sign(series.values), alpha=0.6,
            label=f"{window}d window", lw=1.0)
ax.axhline(0, color='black', lw=0.8)
ax.set_title("Sign of Det(R-I) per Window", fontsize=11)
ax.set_ylabel("+1 / -1")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("composite_signal_thresholds.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Figure 2: Portfolio values per threshold (TC=0.5%, monthly) ───────────────
fig, axes = plt.subplots(1, 3, figsize=(20, 6), sharey=False)
fig.suptitle("Three-Stage Signal Portfolio — Value by Threshold (TC=0.5%, Monthly)",
             fontsize=13, fontweight='bold')

for ax, thresh in zip(axes, THRESHOLDS):
    # EW reference
    ew_pv = ew_results['EW_monthly_TC0.5%']['portfolio_value']
    ax.plot(ew_pv.index, ew_pv / 1e9, '--', color='grey',
            lw=1.5, label='EW Benchmark')

    label = f"Signal3S_t{thresh}_monthly_TC0.5%"
    pv    = signal_results[label]['portfolio_value']
    ax.plot(pv.index, pv / 1e9, color='steelblue', lw=1.8,
            label=f'Signal (t={thresh})')

    ax.axhline(1.0, color='black', ls=':', lw=0.8)
    ax.set_title(f"Threshold = {thresh}", fontsize=11)
    ax.set_ylabel("Value ($B)")
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("signal_3stage_by_threshold.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Figure 3: Drawdowns per threshold ────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(20, 5), sharey=True)
fig.suptitle("Three-Stage Signal Portfolio — Drawdowns by Threshold (TC=0.5%, Monthly)",
             fontsize=13, fontweight='bold')

for ax, thresh in zip(axes, THRESHOLDS):
    label  = f"Signal3S_t{thresh}_monthly_TC0.5%"
    pv     = signal_results[label]['portfolio_value']
    dd     = (pv - pv.cummax()) / pv.cummax() * 100
    ew_pv  = ew_results['EW_monthly_TC0.5%']['portfolio_value']
    ew_dd  = (ew_pv - ew_pv.cummax()) / ew_pv.cummax() * 100

    ax.fill_between(dd.index,  dd.values,  0, alpha=0.3, color='steelblue')
    ax.fill_between(ew_dd.index, ew_dd.values, 0, alpha=0.2, color='grey')
    ax.plot(dd.index,    dd.values,    color='steelblue', lw=1.0,
            label=f'Signal (t={thresh})')
    ax.plot(ew_dd.index, ew_dd.values, color='grey',      lw=1.0, ls='--',
            label='EW Benchmark')
    ax.set_title(f"Threshold = {thresh}", fontsize=11)
    ax.set_ylabel("Drawdown (%)")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("signal_3stage_drawdowns.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Figure 4: TC sensitivity per threshold ────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.suptitle("Three-Stage Signal — TC Sensitivity (Monthly)",
             fontsize=13, fontweight='bold')

for ax, thresh in zip(axes, THRESHOLDS):
    for tc_label, col in colors_tc.items():
        label = f"Signal3S_t{thresh}_monthly_TC{tc_label}"
        pv    = signal_results[label]['portfolio_value']
        ax.plot(pv.index, pv / 1e9, color=col, lw=1.3,
                label=f'TC={tc_label}')
    ew_pv = ew_results['EW_monthly_TC0.5%']['portfolio_value']
    ax.plot(ew_pv.index, ew_pv / 1e9, '--', color='grey',
            lw=1.5, label='EW Benchmark')
    ax.set_title(f"Threshold = {thresh}", fontsize=11)
    ax.set_ylabel("Value ($B)")
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("signal_3stage_tc_sensitivity.png", dpi=150, bbox_inches='tight')
plt.show()

# ── 15. SAVE ──────────────────────────────────────────────────────────────────
for label, df in {**ew_results, **signal_results}.items():
    df.to_csv(f"{label}_portfolio.csv")

all_metrics = pd.DataFrame({**ew_metrics, **sig_metrics}).T
all_metrics.to_csv("all_portfolio_metrics.csv")

composite_score.to_csv("composite_signal.csv")

for thresh in THRESHOLDS:
    sig_df_t = build_signal_df(composite_score, equity_assets,
                               macro_df.index, threshold=thresh)
    sig_df_t.to_csv(f"signal_df_threshold_{thresh}.csv")

print("✓ All files saved.")


In [ ]:
# @title
# ============================================================
# LSTM APPROACH: PREDICTING EQUAL-WEIGHT PORTFOLIO RETURNS
# Train: 2015-2019 | Test: 2020+
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import requests
from io import StringIO
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
import warnings
warnings.filterwarnings('ignore')

print("="*90)
print("LSTM APPROACH: PREDICTING EW PORTFOLIO RETURNS")
print("="*90)

# ── 1. LOAD ALL DATA (REUSE FROM NAIVE APPROACH) ──────────────────────────────
GITHUB_BASE = "https://raw.githubusercontent.com/kboroz/MSFE_Capstone_Project/main/01_Data/Streamlined/"

INDEX_FILE_MAP = {
    'ASX50':       'streamlined_asx_50.csv',
    'EUROSTOXX50': 'streamlined_euro_stoxx_50.csv',
    'FTSE100':     'streamlined_ftse_100.csv',
    'Ibovespa':    'streamlined_ibovespa.csv',
    'JSE40':       'streamlined_jse_top_40.csv',
    'NIKKEI225':   'streamlined_nikkei_225.csv',
    'SP500':       'streamlined_s&p_500.csv',
}

MACRO_FILE = 'streamlined_macro.csv'

def load_csv(url):
    r = requests.get(url)
    r.raise_for_status()
    df = pd.read_csv(StringIO(r.text), index_col=0, parse_dates=True)
    df.columns = df.columns.str.replace('[^a-zA-Z0-9]', '_', regex=True)
    return df.sort_index()

macro_df = load_csv(GITHUB_BASE + MACRO_FILE)
macro_df = macro_df[~macro_df.index.duplicated(keep='first')].sort_index()

index_data = {}
for name, fname in INDEX_FILE_MAP.items():
    try:
        df = load_csv(GITHUB_BASE + fname)
        index_data[name] = df
    except Exception as e:
        print(f"✗ {name}: {e}")

PRICE_COLS = ['Close', 'close', 'Price', 'price', 'Adj_Close', 'Last']

def get_price_series(df, name):
    col = next((c for c in PRICE_COLS if c in df.columns), df.columns[0])
    s   = pd.to_numeric(df[col], errors='coerce').dropna()
    return s

index_prices = {}
for name, df in index_data.items():
    index_prices[name] = get_price_series(df, name)

# ── 2. BUILD RETURNS & EQUAL-WEIGHT PORTFOLIO ─────────────────────────────────
EQUITY_ASSETS = list(index_prices.keys())
SAFE_BONDS    = ['US_10Y', 'UK_10Y', 'AU_10Y', 'ZA_10Y']
FX            = ['AUDUSD', 'BRLUSD', 'EURUSD', 'GBPUSD', 'JPYUSD', 'ZARUSD']
COMMODITIES   = ['Oil', 'Gold', 'Bitcoin']

ALL_ASSETS        = EQUITY_ASSETS + FX + COMMODITIES + SAFE_BONDS
ALL_ASSETS        = [a for a in ALL_ASSETS if a in macro_df.columns]
equity_assets     = [a for a in EQUITY_ASSETS if a in ALL_ASSETS]
non_equity_assets = [a for a in ALL_ASSETS if a not in equity_assets]

N           = len(ALL_ASSETS)
base_weight = 1.0 / N

print(f"\nTotal assets: {N}")
print(f"Equity assets: {len(equity_assets)}")

# Build returns
returns_raw = macro_df[ALL_ASSETS].pct_change(fill_method=None)

for col in SAFE_BONDS:
    if col in returns_raw.columns:
        returns_raw[col] = -returns_raw[col]

if 'Oil' in macro_df.columns:
    bad_oil = macro_df['Oil'][macro_df['Oil'] <= 0].index
    for d in bad_oil:
        try:
            loc = returns_raw.index.get_loc(d)
            returns_raw.iloc[
                max(0, loc - 1): loc + 2,
                returns_raw.columns.get_loc('Oil')
            ] = 0.0
        except:
            pass

returns = returns_raw.clip(lower=-0.50, upper=1.00).fillna(0).iloc[1:]

# Compute equal-weight portfolio returns
ew_weights = np.array([base_weight] * N)
ew_portfolio_returns = (returns[ALL_ASSETS].values @ ew_weights).flatten()
ew_returns_series = pd.Series(ew_portfolio_returns, index=returns.index)

print(f"\nReturns shape: {returns.shape}")
print(f"EW Portfolio returns shape: {ew_returns_series.shape}")
print(f"EW Returns range: [{ew_returns_series.min():.4f}, {ew_returns_series.max():.4f}]")

# ── 3. PREPARE FEATURES FOR LSTM ──────────────────────────────────────────────
# Features: rolling statistics of individual asset returns
# Goal: LSTM learns to predict EW portfolio returns from asset-level volatility/momentum

def build_lstm_features(returns_df, lookback_window=20):
    """
    Build feature matrix from rolling statistics.
    For each asset, compute: rolling mean, rolling std, momentum
    """
    features_list = []
    feature_names = []

    for asset in returns_df.columns:
        ret = returns_df[asset].values

        # Rolling mean (momentum)
        rolling_mean = pd.Series(ret).rolling(lookback_window, min_periods=1).mean().values
        features_list.append(rolling_mean)
        feature_names.append(f"{asset}_momentum_{lookback_window}d")

        # Rolling std (volatility)
        rolling_std = pd.Series(ret).rolling(lookback_window, min_periods=1).std().values
        rolling_std = np.nan_to_num(rolling_std, nan=0)
        features_list.append(rolling_std)
        feature_names.append(f"{asset}_volatility_{lookback_window}d")

        # Exponential moving average (smoother momentum)
        ema = pd.Series(ret).ewm(span=lookback_window, adjust=False).mean().values
        features_list.append(ema)
        feature_names.append(f"{asset}_ema_{lookback_window}d")

    X = np.column_stack(features_list)
    return X, feature_names

LOOKBACK_WINDOW = 20
X_features, feature_names = build_lstm_features(returns[ALL_ASSETS], lookback_window=LOOKBACK_WINDOW)
y_target = ew_returns_series.values

print(f"\nFeature matrix shape: {X_features.shape}")
print(f"Number of features: {len(feature_names)}")
print(f"Target (EW returns) shape: {y_target.shape}")

# ── 4. DEFINE TRAIN/TEST SPLIT: 2015-2019 (train), 2020+ (test) ────────────────
train_start = pd.Timestamp('2015-01-01')
train_end   = pd.Timestamp('2019-12-31')
test_start  = pd.Timestamp('2020-01-01')

train_mask = (returns.index >= train_start) & (returns.index <= train_end)
test_mask  = (returns.index >= test_start)

train_idx = np.where(train_mask)[0]
test_idx  = np.where(test_mask)[0]

X_train = X_features[train_idx]
y_train = y_target[train_idx]
X_test  = X_features[test_idx]
y_test  = y_target[test_idx]

print(f"\n{'='*90}")
print("TRAIN/TEST SPLIT")
print(f"{'='*90}")
print(f"Training period   : {returns.index[train_idx[0]].date()} → {returns.index[train_idx[-1]].date()}")
print(f"Training samples  : {len(train_idx)}")
print(f"Testing period    : {returns.index[test_idx[0]].date()} → {returns.index[test_idx[-1]].date()}")
print(f"Testing samples   : {len(test_idx)}")

# ── 5. NORMALIZE FEATURES ─────────────────────────────────────────────────────
scaler_X = MinMaxScaler()
X_train_scaled = scaler_X.fit_transform(X_train)
X_test_scaled  = scaler_X.transform(X_test)

scaler_y = MinMaxScaler()
y_train_scaled = scaler_y.fit_transform(y_train.reshape(-1, 1)).flatten()
y_test_scaled  = scaler_y.transform(y_test.reshape(-1, 1)).flatten()

print(f"\nX_train_scaled shape: {X_train_scaled.shape}")
print(f"y_train_scaled shape: {y_train_scaled.shape}")

# ── 6. CREATE SEQUENCES FOR LSTM ──────────────────────────────────────────────
# LSTM requires 3D input: (samples, timesteps, features)
# We'll create sequences of length `seq_length` to predict next return

SEQ_LENGTH = 60  # Use 60 past days to predict next day

def create_sequences(X, y, seq_length):
    """Create sequences for LSTM"""
    X_seq, y_seq = [], []
    for i in range(len(X) - seq_length):
        X_seq.append(X[i:i + seq_length])
        y_seq.append(y[i + seq_length])
    return np.array(X_seq), np.array(y_seq)

X_train_seq, y_train_seq = create_sequences(X_train_scaled, y_train_scaled, SEQ_LENGTH)
X_test_seq, y_test_seq   = create_sequences(X_test_scaled, y_test_scaled, SEQ_LENGTH)

print(f"\n{'='*90}")
print("LSTM SEQUENCE CREATION")
print(f"{'='*90}")
print(f"Sequence length: {SEQ_LENGTH} days")
print(f"X_train_seq shape: {X_train_seq.shape}")  # (samples, timesteps, features)
print(f"y_train_seq shape: {y_train_seq.shape}")
print(f"X_test_seq shape: {X_test_seq.shape}")
print(f"y_test_seq shape: {y_test_seq.shape}")

# ── 7. BUILD & TRAIN LSTM MODEL ───────────────────────────────────────────────
print(f"\n{'='*90}")
print("LSTM MODEL ARCHITECTURE & TRAINING")
print(f"{'='*90}")

model = Sequential([
    LSTM(units=64, activation='relu', input_shape=(SEQ_LENGTH, X_train_seq.shape[2]),
         return_sequences=True),
    Dropout(0.2),
    LSTM(units=32, activation='relu', return_sequences=False),
    Dropout(0.2),
    Dense(units=16, activation='relu'),
    Dense(units=1, activation='linear')  # Linear for regression
])

model.compile(optimizer=Adam(learning_rate=0.001), loss='mse', metrics=['mae'])
print("\nModel Summary:")
model.summary()

# Early stopping to prevent overfitting
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

print("\nTraining LSTM...")
history = model.fit(
    X_train_seq, y_train_seq,
    validation_split=0.2,
    epochs=100,
    batch_size=32,
    callbacks=[early_stop],
    verbose=0
)

print(f"✓ Training complete. Best epoch: {len(history.history['loss']) - 10}")

# ── 8. MAKE PREDICTIONS ───────────────────────────────────────────────────────
y_train_pred_scaled = model.predict(X_train_seq, verbose=0)
y_test_pred_scaled  = model.predict(X_test_seq, verbose=0)

# Inverse transform to original scale
y_train_pred = scaler_y.inverse_transform(y_train_pred_scaled).flatten()
y_test_pred  = scaler_y.inverse_transform(y_test_pred_scaled).flatten()

# Align with original series (account for sequence offset)
y_train_actual = y_train[SEQ_LENGTH:]
y_test_actual  = y_test[SEQ_LENGTH:]

print(f"\n{'='*90}")
print("PREDICTION SHAPES (after accounting for sequence offset)")
print(f"{'='*90}")
print(f"y_train_actual: {y_train_actual.shape}")
print(f"y_train_pred: {y_train_pred.shape}")
print(f"y_test_actual: {y_test_actual.shape}")
print(f"y_test_pred: {y_test_pred.shape}")

# ── 9. COMPUTE METRICS ────────────────────────────────────────────────────────
def compute_prediction_metrics(y_true, y_pred, label=''):
    mse = mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)

    # Direction accuracy (sign of prediction matches sign of actual)
    dir_correct = np.sum(np.sign(y_pred) == np.sign(y_true))
    dir_accuracy = dir_correct / len(y_true)

    return {
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'R2': r2,
        'Direction Accuracy': dir_accuracy
    }

train_metrics = compute_prediction_metrics(y_train_actual, y_train_pred, 'Train')
test_metrics  = compute_prediction_metrics(y_test_actual, y_test_pred, 'Test')

print(f"\n{'='*90}")
print("LSTM PREDICTION METRICS")
print(f"{'='*90}")
print("\nTRAINING SET:")
for k, v in train_metrics.items():
    if k == 'Direction Accuracy':
        print(f"  {k:25s}: {v*100:.2f}%")
    else:
        print(f"  {k:25s}: {v:.6f}")

print("\nTEST SET:")
for k, v in test_metrics.items():
    if k == 'Direction Accuracy':
        print(f"  {k:25s}: {v*100:.2f}%")
    else:
        print(f"  {k:25s}: {v:.6f}")

# ── 10. BUILD LSTM-BASED PORTFOLIO ────────────────────────────────────────────
# Strategy: If LSTM predicts next day return > 0, go long EW; else go to cash/short

def build_lstm_signal_series(y_pred_series):
    """
    Convert LSTM predictions to trading signals:
    +1 if predicted return > 0 (long EW)
    -1 if predicted return <= 0 (neutral/cash)
    """
    signals = np.where(y_pred_series > 0, 1.0, -1.0)
    return signals

# Align predictions with portfolio dates
train_dates = returns.index[train_idx][SEQ_LENGTH:]
test_dates  = returns.index[test_idx][SEQ_LENGTH:]

lstm_train_signals = build_lstm_signal_series(y_train_pred)
lstm_test_signals  = build_lstm_signal_series(y_test_pred)

# Create signal series for entire test period
lstm_signal_series = pd.Series(lstm_test_signals, index=test_dates)

print(f"\n{'='*90}")
print("LSTM TRADING SIGNALS (TEST SET)")
print(f"{'='*90}")
n_long  = (lstm_test_signals == 1).sum()
n_short = (lstm_test_signals == -1).sum()
print(f"Long signals  (+1): {n_long} ({n_long/len(lstm_test_signals)*100:.1f}%)")
print(f"Neutral/Cash (-1): {n_short} ({n_short/len(lstm_test_signals)*100:.1f}%)")

## ── 11. RUN LSTM-BASED PORTFOLIO ──────────────────────────────────────────────
def run_equal_weight(returns_df, all_assets, tc_rate=0.005,
                     rebal_freq='monthly', initial_capital=1e9):
    """
    Simple equal-weight buy-and-hold portfolio with monthly rebalancing.
    """
    N = len(all_assets)
    base_weight = 1.0 / N

    pv = float(initial_capital)
    records = []
    last_rebal_date = None

    for date in returns_df.index:
        # Determine if rebalance is needed
        should_rebal = False
        if last_rebal_date is None:
            should_rebal = True
        elif rebal_freq == 'monthly':
            if date.month != last_rebal_date.month or date.year != last_rebal_date.year:
                should_rebal = True
        elif rebal_freq == 'daily':
            should_rebal = True

        if should_rebal:
            pv *= (1 - tc_rate)
            last_rebal_date = date

        # Apply equal-weight returns
        daily_ret = (returns_df.loc[date, all_assets].values @
                    np.array([base_weight] * N))
        pv *= (1 + daily_ret)

        records.append({'date': date, 'portfolio_value': pv})

    return pd.DataFrame(records).set_index('date')


def run_lstm_portfolio(returns_df, signal_series, all_assets, tc_rate=0.005,
                       rebal_freq='daily', initial_capital=1e9):
    """
    Portfolio that goes long/short equal-weight based on LSTM signals.
    Signal = +1 (full long EW), Signal = -1 (cash / no exposure)
    """
    N = len(all_assets)
    base_weight = 1.0 / N

    pv = float(initial_capital)
    records = []
    prev_signal = None

    for date in returns_df.index:
        # Get signal for this date
        if date in signal_series.index:
            current_signal = signal_series.loc[date]
        else:
            current_signal = prev_signal if prev_signal is not None else -1

        # Rebalance if signal changed or first day
        if current_signal != prev_signal:
            if prev_signal is not None:
                turnover = 1.0  # 100% turnover on signal change
                pv *= (1 - turnover * tc_rate)
            else:
                pv *= (1 - tc_rate)
            prev_signal = current_signal

        # Apply signal: +1 = long EW, -1 = cash
        if current_signal > 0:
            # Long equal-weight
            daily_ret = (returns_df.loc[date, all_assets].values @
                        np.array([base_weight] * N))
        else:
            # Cash (0% return)
            daily_ret = 0.0

        pv *= (1 + daily_ret)
        records.append({'date': date, 'portfolio_value': pv})

    return pd.DataFrame(records).set_index('date')


# ── 11b. RUN PORTFOLIOS ───────────────────────────────────────────────────────
# Run LSTM portfolio on test set
lstm_pv = run_lstm_portfolio(
    returns.loc[test_dates[0]:],
    lstm_signal_series,
    ALL_ASSETS,
    tc_rate=0.005,
    rebal_freq='daily',
    initial_capital=1e9
)

# Also run EW benchmark for comparison
ew_pv = run_equal_weight(returns.loc[test_dates[0]:], ALL_ASSETS,
                         tc_rate=0.005, rebal_freq='monthly',
                         initial_capital=1e9)

print(f"\n✓ LSTM portfolio and EW benchmark computed")

# ── 12. COMPUTE PORTFOLIO METRICS ─────────────────────────────────────────────
def compute_portfolio_metrics(pv_df, label=''):
    pv        = pv_df['portfolio_value']
    daily_ret = pv.pct_change().dropna()
    n_years   = len(daily_ret) / 252
    total_ret = (pv.iloc[-1] / pv.iloc[0]) - 1
    cagr      = (1 + total_ret) ** (1 / n_years) - 1 if n_years > 0 else np.nan
    vol       = daily_ret.std() * np.sqrt(252)
    sharpe    = cagr / vol if vol > 0 else np.nan
    roll_max  = pv.cummax()
    max_dd    = ((pv - roll_max) / roll_max).min()
    calmar    = cagr / abs(max_dd) if max_dd != 0 else np.nan

    return {
        'CAGR': f"{cagr*100:.2f}%",
        'Volatility': f"{vol*100:.2f}%",
        'Sharpe': f"{sharpe:.2f}",
        'Max Drawdown': f"{max_dd*100:.2f}%",
        'Calmar': f"{calmar:.2f}",
        'Total Return': f"{total_ret*100:.2f}%",
        'Final Value ($B)': f"{pv.iloc[-1]/1e9:.3f}",
    }

lstm_port_metrics = compute_portfolio_metrics(lstm_pv, 'LSTM')
ew_port_metrics   = compute_portfolio_metrics(ew_pv, 'EW')

print(f"\n{'='*90}")
print("LSTM-BASED PORTFOLIO METRICS (TEST SET)")
print(f"{'='*90}")

metrics_comparison = pd.DataFrame({
    'LSTM Portfolio': lstm_port_metrics,
    'EW Benchmark': ew_port_metrics
}).T

print(metrics_comparison.to_string())

# ── 13. VISUALIZATIONS ────────────────────────────────────────────────────────

# Figure 1: Training History
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

ax = axes[0]
ax.plot(history.history['loss'], label='Training Loss', lw=1.5)
ax.plot(history.history['val_loss'], label='Validation Loss', lw=1.5)
ax.set_xlabel('Epoch', fontsize=11)
ax.set_ylabel('MSE Loss', fontsize=11)
ax.set_title('LSTM Training History', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(history.history['mae'], label='Training MAE', lw=1.5)
ax.plot(history.history['val_mae'], label='Validation MAE', lw=1.5)
ax.set_xlabel('Epoch', fontsize=11)
ax.set_ylabel('MAE', fontsize=11)
ax.set_title('LSTM MAE History', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("lstm_training_history.png", dpi=150, bbox_inches='tight')
plt.show()

# Figure 2: Predictions vs Actuals (Test Set)
fig, axes = plt.subplots(2, 1, figsize=(16, 10))

# Full test period
ax = axes[0]
ax.plot(test_dates, y_test_actual[:len(test_dates)], 'o-',
        alpha=0.6, label='Actual EW Returns', color='navy', ms=3)
ax.plot(test_dates, y_test_pred[:len(test_dates)], 's-',
        alpha=0.6, label='LSTM Predicted Returns', color='crimson', ms=3)
ax.axhline(0, color='black', ls='--', lw=0.8, alpha=0.5)
ax.set_xlabel('Date', fontsize=11)
ax.set_ylabel('Daily Return', fontsize=11)
ax.set_title('LSTM Predictions vs Actual EW Returns (Test Set: 2020+)',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Zoomed view: first year of test period
ax = axes[1]
zoom_end = min(len(test_dates), 252)
ax.plot(test_dates[:zoom_end], y_test_actual[:zoom_end], 'o-',
        alpha=0.7, label='Actual EW Returns', color='navy', ms=4)
ax.plot(test_dates[:zoom_end], y_test_pred[:zoom_end], 's-',
        alpha=0.7, label='LSTM Predicted Returns', color='crimson', ms=4)
ax.axhline(0, color='black', ls='--', lw=0.8, alpha=0.5)
ax.set_xlabel('Date', fontsize=11)
ax.set_ylabel('Daily Return', fontsize=11)
ax.set_title('LSTM Predictions vs Actual EW Returns (Test Set: 2020 — First Year)',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("lstm_predictions_vs_actuals.png", dpi=150, bbox_inches='tight')
plt.show()

# Figure 3: LSTM Portfolio Value vs EW Benchmark
fig, ax = plt.subplots(figsize=(16, 7))

ax.plot(lstm_pv.index, lstm_pv['portfolio_value'] / 1e9,
        color='steelblue', lw=2.0, label='LSTM Portfolio')
ax.plot(ew_pv.index, ew_pv['portfolio_value'] / 1e9,
        color='grey', lw=2.0, ls='--', label='EW Benchmark')

ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Portfolio Value ($B)', fontsize=12)
ax.set_title('LSTM-Based Portfolio vs Equal-Weight Benchmark (Test: 2020+)',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))

plt.tight_layout()
plt.savefig("lstm_portfolio_vs_ew.png", dpi=150, bbox_inches='tight')
plt.show()

# Figure 4: Drawdowns Comparison
fig, ax = plt.subplots(figsize=(16, 7))

lstm_dd = (lstm_pv['portfolio_value'] - lstm_pv['portfolio_value'].cummax()) / \
          lstm_pv['portfolio_value'].cummax() * 100
ew_dd = (ew_pv['portfolio_value'] - ew_pv['portfolio_value'].cummax()) / \
        ew_pv['portfolio_value'].cummax() * 100

ax.fill_between(lstm_dd.index, lstm_dd.values, 0, alpha=0.3, color='steelblue',
                label='LSTM Portfolio')
ax.fill_between(ew_dd.index, ew_dd.values, 0, alpha=0.2, color='grey',
                label='EW Benchmark')
ax.plot(lstm_dd.index, lstm_dd.values, color='steelblue', lw=1.5)
ax.plot(ew_dd.index, ew_dd.values, color='grey', lw=1.5, ls='--')

ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Drawdown (%)', fontsize=12)
ax.set_title('Drawdown Comparison: LSTM vs Equal-Weight (Test: 2020+)',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("lstm_drawdowns_comparison.png", dpi=150, bbox_inches='tight')
plt.show()

# ── 14. SAVE RESULTS ──────────────────────────────────────────────────────────
print(f"\n{'='*90}")
print("SAVING RESULTS")
print(f"{'='*90}")

# Save portfolio values
lstm_pv.to_csv("LSTM_portfolio_test.csv")
ew_pv.to_csv("EW_benchmark_test.csv")

# Save predictions
predictions_df = pd.DataFrame({
    'date': test_dates,
    'actual_ew_return': y_test_actual[:len(test_dates)],
    'lstm_predicted_return': y_test_pred[:len(test_dates)],
    'lstm_signal': lstm_test_signals[:len(test_dates)]
})
predictions_df = predictions_df.set_index('date')
predictions_df.to_csv("LSTM_predictions.csv")

# Save metrics
metrics_comparison.to_csv("LSTM_vs_EW_metrics.csv")

# Save model
model.save("lstm_ew_prediction_model.h5")

print("✓ LSTM_portfolio_test.csv")
print("✓ EW_benchmark_test.csv")
print("✓ LSTM_predictions.csv")
print("✓ LSTM_vs_EW_metrics.csv")
print("✓ lstm_ew_prediction_model.h5")

# ── 15. COMPARISON TABLE ──────────────────────────────────────────────────────
print(f"\n{'='*90}")
print("FINAL COMPARISON: NAIVE BASELINE vs LSTM")
print(f"{'='*90}")
print("\nLSTM Approach:")
print(f"  - Learns from historical EW portfolio returns (2015-2019)")
print(f"  - Predicts daily EW returns based on rolling feature statistics")
print(f"  - Direction accuracy (test): {test_metrics['Direction Accuracy']*100:.2f}%")
print(f"\nNext: Compare with Determinant-Based Signals (Naive) and Hybrid LSTM+Determinant")


In [ ]:
# @title
# ============================================================
# HYBRID APPROACH: LSTM + DETERMINANT SIGNALS
# Combines LSTM predictions with Modified Determinant features
# Train: 2015-2019 | Test: 2020+
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import requests
from io import StringIO
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
import warnings
warnings.filterwarnings('ignore')

print("="*90)
print("HYBRID APPROACH: LSTM + DETERMINANT SIGNALS")
print("="*90)

# ── 1. LOAD DATA (REUSE FROM PREVIOUS APPROACHES) ────────────────────────────
GITHUB_BASE = "https://raw.githubusercontent.com/kboroz/MSFE_Capstone_Project/main/01_Data/Streamlined/"

INDEX_FILE_MAP = {
    'ASX50':       'streamlined_asx_50.csv',
    'EUROSTOXX50': 'streamlined_euro_stoxx_50.csv',
    'FTSE100':     'streamlined_ftse_100.csv',
    'Ibovespa':    'streamlined_ibovespa.csv',
    'JSE40':       'streamlined_jse_top_40.csv',
    'NIKKEI225':   'streamlined_nikkei_225.csv',
    'SP500':       'streamlined_s&p_500.csv',
}

MACRO_FILE = 'streamlined_macro.csv'

def load_csv(url):
    r = requests.get(url)
    r.raise_for_status()
    df = pd.read_csv(StringIO(r.text), index_col=0, parse_dates=True)
    df.columns = df.columns.str.replace('[^a-zA-Z0-9]', '_', regex=True)
    return df.sort_index()

macro_df = load_csv(GITHUB_BASE + MACRO_FILE)
macro_df = macro_df[~macro_df.index.duplicated(keep='first')].sort_index()

index_data = {}
for name, fname in INDEX_FILE_MAP.items():
    try:
        df = load_csv(GITHUB_BASE + fname)
        index_data[name] = df
    except Exception as e:
        print(f"✗ {name}: {e}")

PRICE_COLS = ['Close', 'close', 'Price', 'price', 'Adj_Close', 'Last']

def get_price_series(df, name):
    col = next((c for c in PRICE_COLS if c in df.columns), df.columns[0])
    s   = pd.to_numeric(df[col], errors='coerce').dropna()
    return s

index_prices = {}
for name, df in index_data.items():
    index_prices[name] = get_price_series(df, name)

# ── 2. COMPUTE DETERMINANT SIGNALS (FROM NAIVE APPROACH) ─────────────────────
DET_WINDOWS    = [3, 7, 28]
WINDOW_WEIGHTS = {3: 0.15, 7: 0.25, 28: 0.6}

def compute_modified_determinant(price_series_dict, windows=DET_WINDOWS):
    """
    Compute modified determinant from covariance matrix of returns.
    """
    returns_dict = {name: price.pct_change().dropna()
                   for name, price in price_series_dict.items()}

    det_scores = {}

    for window in windows:
        det_list = []
        dates_list = []

        all_returns = pd.concat(returns_dict.values(), axis=1)
        all_returns.columns = returns_dict.keys()

        for i in range(window, len(all_returns)):
            window_data = all_returns.iloc[i-window:i]
            cov_matrix = window_data.cov().values
            det_val = np.linalg.det(cov_matrix)
            det_list.append(det_val)
            dates_list.append(all_returns.index[i])

        det_scores[window] = pd.Series(det_list, index=dates_list)

    return det_scores

print("\n" + "="*90)
print("COMPUTING DETERMINANT SIGNALS")
print("="*90)

det_scores = compute_modified_determinant(index_prices, DET_WINDOWS)

for window, series in det_scores.items():
    print(f"Window {window}d: {len(series)} observations, "
          f"range [{series.min():.4e}, {series.max():.4e}]")

# ── 3. GENERATE COMPOSITE DETERMINANT SIGNAL ──────────────────────────────────
# Find common dates across all windows
common_det_dates = det_scores[3].index.intersection(det_scores[7].index).intersection(det_scores[28].index)

print(f"\nAligning determinant signals to common dates: {len(common_det_dates)} observations")

# Align all determinant scores to common dates
det_scores_aligned = {}
for window in DET_WINDOWS:
    det_scores_aligned[window] = det_scores[window].loc[common_det_dates]

# Now combine with equal length
composite_det_raw = pd.Series(0.0, index=common_det_dates)

for window, weight in WINDOW_WEIGHTS.items():
    # Normalize each window's determinant to [0, 1]
    scaler = MinMaxScaler()
    normalized = scaler.fit_transform(
        det_scores_aligned[window].values.reshape(-1, 1)
    ).flatten()
    composite_det_raw += weight * pd.Series(normalized, index=common_det_dates)

composite_det_signal = composite_det_raw / sum(WINDOW_WEIGHTS.values())

print(f"Composite Determinant Signal shape: {composite_det_signal.shape}")
print(f"Range: [{composite_det_signal.min():.4f}, {composite_det_signal.max():.4f}]")


# ── 4. PREPARE DATA FOR LSTM (REUSE FROM LSTM APPROACH) ──────────────────────
ALL_ASSETS = list(index_prices.keys())
print(f"\n{'='*90}")
print("PREPARING DATA")
print(f"{'='*90}")
print(f"Total assets: {len(ALL_ASSETS)}")

# Calculate returns
returns = pd.DataFrame({name: price.pct_change()
                       for name, price in index_prices.items()})
returns = returns.dropna()

# Align with determinant signal
common_dates = returns.index.intersection(composite_det_signal.index)
returns = returns.loc[common_dates]
composite_det_signal = composite_det_signal.loc[common_dates]

# Compute EW portfolio returns
ew_returns = returns.mean(axis=1)

print(f"Returns shape: {returns.shape}")
print(f"EW Portfolio returns shape: {ew_returns.shape}")
print(f"Determinant Signal shape: {composite_det_signal.shape}")

# ── 5. CREATE HYBRID FEATURE MATRIX ───────────────────────────────────────────
print(f"\n{'='*90}")
print("CREATING HYBRID FEATURE MATRIX (LSTM + DETERMINANT)")
print(f"{'='*90}")

def create_hybrid_features(returns_df, det_signal, lookback=20):
    """
    Create features combining:
    - Rolling momentum (each asset)
    - Rolling volatility (each asset)
    - Rolling correlation (assets)
    - Determinant signal (normalized)
    - EMA of determinant signal
    """
    N_assets = len(returns_df.columns)
    features = []
    dates = []

    for i in range(lookback, len(returns_df)):
        window = returns_df.iloc[i-lookback:i]
        current_date = returns_df.index[i]

        # 1. Momentum: cumulative return over lookback
        momentum = (1 + window).prod(axis=0).values - 1  # (N_assets,)

        # 2. Volatility: std of returns over lookback
        volatility = window.std().values  # (N_assets,)

        # 3. Pairwise correlations (upper triangle)
        corr_matrix = window.corr().values
        corr_flat = corr_matrix[np.triu_indices_from(corr_matrix, k=1)]

        # 4. Determinant signal (current value)
        if current_date in det_signal.index:
            det_val = det_signal.loc[current_date]
        else:
            det_val = 0.5  # Default to neutral

        # 5. EMA of determinant signal (20-day)
        if i >= 20:
            det_window = det_signal.iloc[max(0, i-20):i]
            det_ema = det_window.ewm(span=10).mean().iloc[-1] if len(det_window) > 0 else 0.5
        else:
            det_ema = 0.5

        # Combine all features
        feature_vector = np.concatenate([
            momentum,           # N_assets
            volatility,         # N_assets
            corr_flat,          # N_assets*(N_assets-1)/2
            [det_val, det_ema]  # 2
        ])

        features.append(feature_vector)
        dates.append(current_date)

    return np.array(features), dates

lookback_window = 20
X_hybrid, hybrid_dates = create_hybrid_features(returns, composite_det_signal,
                                                lookback=lookback_window)
y_hybrid = ew_returns.loc[hybrid_dates].values

print(f"Hybrid feature matrix shape: {X_hybrid.shape}")
print(f"  - Momentum features: {len(ALL_ASSETS)}")
print(f"  - Volatility features: {len(ALL_ASSETS)}")
print(f"  - Correlation features: {len(ALL_ASSETS)*(len(ALL_ASSETS)-1)//2}")
print(f"  - Determinant features: 2 (value + EMA)")
print(f"  - Total: {X_hybrid.shape[1]}")

# ── 6. TRAIN/TEST SPLIT ───────────────────────────────────────────────────────
train_end_date = pd.Timestamp('2017-12-31')
test_start_date = pd.Timestamp('2018-01-01')

train_mask = np.array([d <= train_end_date for d in hybrid_dates])
test_mask = np.array([d >= test_start_date for d in hybrid_dates])

X_train_hybrid = X_hybrid[train_mask]
y_train_hybrid = y_hybrid[train_mask]
train_dates_hybrid = [d for d, m in zip(hybrid_dates, train_mask) if m]

X_test_hybrid = X_hybrid[test_mask]
y_test_hybrid = y_hybrid[test_mask]
test_dates_hybrid = [d for d, m in zip(hybrid_dates, test_mask) if m]

print(f"\n{'='*90}")
print("TRAIN/TEST SPLIT")
print(f"{'='*90}")
print(f"Training period   : {train_dates_hybrid[0].date()} → {train_dates_hybrid[-1].date()}")
print(f"Training samples  : {len(X_train_hybrid)}")
print(f"Testing period    : {test_dates_hybrid[0].date()} → {test_dates_hybrid[-1].date()}")
print(f"Testing samples   : {len(X_test_hybrid)}")

# ── 7. SCALE FEATURES ─────────────────────────────────────────────────────────
scaler_hybrid = MinMaxScaler()
X_train_hybrid_scaled = scaler_hybrid.fit_transform(X_train_hybrid)
X_test_hybrid_scaled = scaler_hybrid.transform(X_test_hybrid)

scaler_y_hybrid = MinMaxScaler()
y_train_hybrid_scaled = scaler_y_hybrid.fit_transform(y_train_hybrid.reshape(-1, 1)).flatten()
y_test_hybrid_scaled = scaler_y_hybrid.transform(y_test_hybrid.reshape(-1, 1)).flatten()

print(f"\nX_train_hybrid_scaled shape: {X_train_hybrid_scaled.shape}")
print(f"y_train_hybrid_scaled shape: {y_train_hybrid_scaled.shape}")

# ── 8. CREATE SEQUENCES FOR LSTM ──────────────────────────────────────────────
seq_length = 60

def create_sequences(X, y, seq_len):
    X_seq, y_seq = [], []
    for i in range(len(X) - seq_len + 1):
        X_seq.append(X[i:i+seq_len])
        y_seq.append(y[i+seq_len-1])
    return np.array(X_seq), np.array(y_seq)

X_train_seq_hybrid, y_train_seq_hybrid = create_sequences(
    X_train_hybrid_scaled, y_train_hybrid_scaled, seq_length)
X_test_seq_hybrid, y_test_seq_hybrid = create_sequences(
    X_test_hybrid_scaled, y_test_hybrid_scaled, seq_length)

print(f"\n{'='*90}")
print("LSTM SEQUENCE CREATION")
print(f"{'='*90}")
print(f"Sequence length: {seq_length} days")
print(f"X_train_seq_hybrid shape: {X_train_seq_hybrid.shape}")
print(f"y_train_seq_hybrid shape: {y_train_seq_hybrid.shape}")
print(f"X_test_seq_hybrid shape: {X_test_seq_hybrid.shape}")
print(f"y_test_seq_hybrid shape: {y_test_seq_hybrid.shape}")

# ── 9. BUILD HYBRID LSTM MODEL ────────────────────────────────────────────────
print(f"\n{'='*90}")
print("HYBRID LSTM MODEL ARCHITECTURE & TRAINING")
print(f"{'='*90}")

n_features = X_train_seq_hybrid.shape[2]

model_hybrid = Sequential([
    LSTM(128, activation='relu', return_sequences=True,
         input_shape=(seq_length, n_features)),
    Dropout(0.2),
    LSTM(64, activation='relu', return_sequences=False),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(1, activation='linear')
])

model_hybrid.compile(optimizer=Adam(learning_rate=0.001), loss='mse', metrics=['mae'])

print("\nModel Summary:")
model_hybrid.summary()

# Early stopping
early_stop_hybrid = EarlyStopping(monitor='val_loss', patience=15,
                                  restore_best_weights=True)

print("\nTraining Hybrid LSTM...")
history_hybrid = model_hybrid.fit(
    X_train_seq_hybrid, y_train_seq_hybrid,
    validation_split=0.2,
    epochs=100,
    batch_size=32,
    callbacks=[early_stop_hybrid],
    verbose=0
)

best_epoch_hybrid = len(history_hybrid.history['loss']) - 15
print(f"✓ Training complete. Best epoch: {best_epoch_hybrid}")

# ── 10. PREDICTIONS ───────────────────────────────────────────────────────────
y_train_pred_hybrid = model_hybrid.predict(X_train_seq_hybrid, verbose=0)
y_test_pred_hybrid = model_hybrid.predict(X_test_seq_hybrid, verbose=0)

# Inverse transform
y_train_pred_hybrid = scaler_y_hybrid.inverse_transform(y_train_pred_hybrid).flatten()
y_test_pred_hybrid = scaler_y_hybrid.inverse_transform(y_test_pred_hybrid).flatten()

y_train_actual_hybrid = y_train_hybrid[seq_length-1:]
y_test_actual_hybrid = y_test_hybrid[seq_length-1:]

# ── 11. EVALUATION METRICS ────────────────────────────────────────────────────
def compute_metrics(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    direction_acc = np.mean((np.sign(y_true) == np.sign(y_pred)) |
                            ((y_true == 0) & (y_pred == 0)))
    return {'MSE': mse, 'RMSE': rmse, 'MAE': mae, 'R2': r2,
            'Direction Accuracy': direction_acc}

metrics_train_hybrid = compute_metrics(y_train_actual_hybrid, y_train_pred_hybrid)
metrics_test_hybrid = compute_metrics(y_test_actual_hybrid, y_test_pred_hybrid)

print(f"\n{'='*90}")
print("HYBRID LSTM + DETERMINANT PREDICTION METRICS")
print(f"{'='*90}")
print("\nTRAINING SET:")
for metric, value in metrics_train_hybrid.items():
    if metric == 'Direction Accuracy':
        print(f"  {metric:25s}: {value*100:7.2f}%")
    else:
        print(f"  {metric:25s}: {value:12.6f}")

print("\nTEST SET:")
for metric, value in metrics_test_hybrid.items():
    if metric == 'Direction Accuracy':
        print(f"  {metric:25s}: {value*100:7.2f}%")
    else:
        print(f"  {metric:25s}: {value:12.6f}")

# ── 12. GENERATE TRADING SIGNALS ──────────────────────────────────────────────
# Combine LSTM predictions with determinant signal
test_dates_seq_hybrid = test_dates_hybrid[seq_length-1:]

lstm_pred_strength = y_test_pred_hybrid  # Raw predictions

# Determinant signal on test dates
det_signal_test = composite_det_signal.loc[test_dates_seq_hybrid].values

# Hybrid signal: weight LSTM predictions and determinant signal
alpha_lstm = 0.6  # Weight for LSTM
alpha_det = 0.4   # Weight for determinant

# Normalize LSTM predictions to [-1, 1]
lstm_norm = (lstm_pred_strength - lstm_pred_strength.mean()) / (lstm_pred_strength.std() + 1e-6)
lstm_norm = np.tanh(lstm_norm)  # Bound to [-1, 1]

# Normalize determinant signal to [-1, 1]
det_norm = 2 * det_signal_test - 1  # [0, 1] -> [-1, 1]

# Combine
hybrid_signal_raw = alpha_lstm * lstm_norm + alpha_det * det_norm

# Convert to trading signals (+1 = long, -1 = cash)
hybrid_threshold = 0.0
hybrid_signals = np.where(hybrid_signal_raw > hybrid_threshold, 1, -1)

# Count signals
long_count = np.sum(hybrid_signals == 1)
neutral_count = np.sum(hybrid_signals == -1)

print(f"\n{'='*90}")
print("HYBRID TRADING SIGNALS (TEST SET)")
print(f"{'='*90}")
print(f"Long signals  (+1): {long_count} ({long_count/len(hybrid_signals)*100:.1f}%)")
print(f"Neutral/Cash (-1): {neutral_count} ({neutral_count/len(hybrid_signals)*100:.1f}%)")

hybrid_signal_series = pd.Series(hybrid_signals, index=test_dates_seq_hybrid)

# ── 13. RUN HYBRID PORTFOLIO ──────────────────────────────────────────────────
def run_hybrid_portfolio(returns_df, signal_series, all_assets, tc_rate=0.005,
                        initial_capital=1e9):
    """
    Portfolio based on hybrid LSTM + Determinant signals.
    """
    N = len(all_assets)
    base_weight = 1.0 / N

    pv = float(initial_capital)
    records = []
    prev_signal = None

    for date in returns_df.index:
        if date in signal_series.index:
            current_signal = signal_series.loc[date]
        else:
            current_signal = prev_signal if prev_signal is not None else -1

        # Rebalance if signal changed
        if current_signal != prev_signal:
            if prev_signal is not None:
                turnover = 1.0
                pv *= (1 - turnover * tc_rate)
            else:
                pv *= (1 - tc_rate)
            prev_signal = current_signal

        # Apply signal
        if current_signal > 0:
            daily_ret = (returns_df.loc[date, all_assets].values @
                        np.array([base_weight] * N))
        else:
            daily_ret = 0.0

        pv *= (1 + daily_ret)
        records.append({'date': date, 'portfolio_value': pv})

    return pd.DataFrame(records).set_index('date')


hybrid_pv = run_hybrid_portfolio(
    returns.loc[test_dates_seq_hybrid[0]:],
    hybrid_signal_series,
    ALL_ASSETS,
    tc_rate=0.005,
    initial_capital=1e9
)

print(f"\n✓ Hybrid (LSTM + Determinant) portfolio computed")

# ── 14. COMPARISON: NAIVE vs LSTM vs HYBRID ───────────────────────────────────
print(f"\n{'='*90}")
print("PORTFOLIO COMPARISON")
print(f"{'='*90}")

# For this comparison, we need all three portfolios on same test dates
# Assuming you have: ew_pv, lstm_pv, hybrid_pv

# Align portfolios to common dates
common_test_dates = hybrid_pv.index

if len(hybrid_pv) > 0:
    # Compute metrics for each portfolio
    def compute_portfolio_metrics(pv_series):
        returns_pf = pv_series.pct_change().dropna()

        annual_ret = (pv_series.iloc[-1] / pv_series.iloc[0]) ** (252 / len(pv_series)) - 1
        annual_vol = returns_pf.std() * np.sqrt(252)
        sharpe = annual_ret / (annual_vol + 1e-6) if annual_vol > 0 else 0

        cummax = pv_series.expanding().max()
        dd = (pv_series - cummax) / cummax
        max_dd = dd.min()

        return {
            'Final Value': pv_series.iloc[-1],
            'Total Return': (pv_series.iloc[-1] / pv_series.iloc[0] - 1) * 100,
            'Annual Return': annual_ret * 100,
            'Annual Volatility': annual_vol * 100,
            'Sharpe Ratio': sharpe,
            'Max Drawdown': max_dd * 100
        }

    hybrid_metrics = compute_portfolio_metrics(hybrid_pv['portfolio_value'])

    print("\nHYBRID PORTFOLIO (LSTM + Determinant):")
    for metric, value in hybrid_metrics.items():
        if 'Return' in metric or 'Volatility' in metric or 'Max Drawdown' in metric:
            print(f"  {metric:20s}: {value:8.2f}%")
        else:
            print(f"  {metric:20s}: {value:12.2f}")

# ── 15. VISUALIZATIONS ────────────────────────────────────────────────────────
# Plot 1: Training history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history_hybrid.history['loss'], label='Training Loss')
axes[0].plot(history_hybrid.history['val_loss'], label='Validation Loss')
axes[0].set_xlabel('Epoch', fontsize=11)
axes[0].set_ylabel('Loss (MSE)', fontsize=11)
axes[0].set_title('Hybrid LSTM Training History', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

axes[1].plot(history_hybrid.history['mae'], label='Training MAE')
axes[1].plot(history_hybrid.history['val_mae'], label='Validation MAE')
axes[1].set_xlabel('Epoch', fontsize=11)
axes[1].set_ylabel('MAE', fontsize=11)
axes[1].set_title('Hybrid LSTM MAE', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("hybrid_lstm_training_history.png", dpi=150, bbox_inches='tight')
plt.show()

# Plot 2: Predictions vs Actual (Test Set)
fig, ax = plt.subplots(figsize=(16, 6))

test_seq_dates = test_dates_seq_hybrid
ax.plot(test_seq_dates, y_test_actual_hybrid, label='Actual EW Returns',
        color='steelblue', lw=2, alpha=0.7)
ax.plot(test_seq_dates, y_test_pred_hybrid, label='Hybrid LSTM Prediction',
        color='coral', lw=1.5, alpha=0.8, linestyle='--')
ax.fill_between(test_seq_dates, y_test_actual_hybrid, y_test_pred_hybrid,
                alpha=0.2, color='grey')

ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Daily Return', fontsize=12)
ax.set_title('Hybrid LSTM: Predicted vs Actual EW Portfolio Returns (Test Set)',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("hybrid_lstm_predictions_vs_actual.png", dpi=150, bbox_inches='tight')
plt.show()

# Plot 3: Components of Hybrid Signal
fig, axes = plt.subplots(3, 1, figsize=(16, 10))

# LSTM component
axes[0].plot(test_seq_dates, lstm_norm, label='Normalized LSTM Predictions',
            color='steelblue', lw=1.5)
axes[0].axhline(0, color='black', lw=0.8, linestyle='--')
axes[0].set_ylabel('LSTM Signal', fontsize=11)
axes[0].set_title('Hybrid Signal Components', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Determinant component
axes[1].plot(test_seq_dates, det_norm, label='Normalized Determinant Signal',
            color='green', lw=1.5)
axes[1].axhline(0, color='black', lw=0.8, linestyle='--')
axes[1].set_ylabel('Determinant Signal', fontsize=11)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

# Combined signal
axes[2].plot(test_seq_dates, hybrid_signal_raw, label='Combined Hybrid Signal',
            color='purple', lw=1.5)
axes[2].axhline(0, color='black', lw=0.8, linestyle='--')
axes[2].fill_between(test_seq_dates, hybrid_signal_raw, 0,
                     where=(hybrid_signal_raw > 0), alpha=0.3, color='green',
                     label='Long Signal')
axes[2].fill_between(test_seq_dates, hybrid_signal_raw, 0,
                     where=(hybrid_signal_raw <= 0), alpha=0.3, color='red',
                     label='Neutral/Cash')
axes[2].set_xlabel('Date', fontsize=12)
axes[2].set_ylabel('Hybrid Signal', fontsize=11)
axes[2].legend(fontsize=10)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("hybrid_signal_components.png", dpi=150, bbox_inches='tight')
plt.show()

# Plot 4: Portfolio Performance
fig, ax = plt.subplots(figsize=(16, 7))

ax.plot(hybrid_pv.index, hybrid_pv['portfolio_value'] / 1e9,
        label='Hybrid (LSTM + Determinant)', lw=2, color='purple')

ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Portfolio Value ($B)', fontsize=12)
ax.set_title('Hybrid Portfolio Performance (Test: 2020+)', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:.2f}B"))

plt.tight_layout()
plt.savefig("hybrid_portfolio_performance.png", dpi=150, bbox_inches='tight')
plt.show()

# ── 16. SAVE RESULTS ──────────────────────────────────────────────────────────
print(f"\n{'='*90}")
print("SAVING HYBRID RESULTS")
print(f"{'='*90}")

hybrid_pv.to_csv("Hybrid_LSTM_Determinant_portfolio_test.csv")

hybrid_predictions_df = pd.DataFrame({
    'date': test_seq_dates,
    'actual_ew_return': y_test_actual_hybrid,
    'lstm_pred_return': y_test_pred_hybrid,
    'lstm_normalized': lstm_norm,
    'determinant_signal': det_signal_test,
    'determinant_normalized': det_norm,
    'hybrid_signal_raw': hybrid_signal_raw,
    'hybrid_trading_signal': hybrid_signals
})
hybrid_predictions_df = hybrid_predictions_df.set_index('date')
hybrid_predictions_df.to_csv("Hybrid_LSTM_Determinant_predictions.csv")

model_hybrid.save("hybrid_lstm_determinant_model.h5")

print("✓ Hybrid_LSTM_Determinant_portfolio_test.csv")
print("✓ Hybrid_LSTM_Determinant_predictions.csv")
print("✓ hybrid_lstm_determinant_model.h5")

# ── 17. FINAL COMPARISON TABLE ────────────────────────────────────────────────
print(f"\n{'='*90}")
print("APPROACH SUMMARY")
print(f"{'='*90}")
print("\n1. NAIVE APPROACH:")
print("   - Equal-weight buy-and-hold portfolio")
print("   - Baseline benchmark\n")

print("2. LSTM APPROACH:")
print("   - LSTM predicts EW portfolio returns")
print("   - Uses rolling momentum, volatility, correlations as features")
print(f"   - Direction Accuracy (test): {metrics_test_hybrid['Direction Accuracy']*100:.2f}%\n")

print("3. HYBRID APPROACH (LSTM + Determinant):")
print("   - LSTM features + Determinant signal features")
print(f"   - Test period direction accuracy: {metrics_test_hybrid['Direction Accuracy']*100:.2f}%")
print(f"   - Alpha: {alpha_lstm:.1%} LSTM, {alpha_det:.1%} Determinant")
print(f"   - Trading signals: {long_count} long, {neutral_count} neutral days\n")

print("Next: Compare all three approaches on same test set with CPCV")


In [ ]:
# @title
# ============================================================
# HYBRID APPROACH: LSTM + DETERMINANT SIGNALS
# WITH BENCHMARK COMPARISONS (B&H, EW)
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import requests
from io import StringIO
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
import warnings
warnings.filterwarnings('ignore')

# [PREVIOUS CODE: Load data, compute determinant, create features, train LSTM...]
# [Keep everything from the original code up to section 13]

# ── 13. RUN BENCHMARKS ────────────────────────────────────────────────────────
print(f"\n{'='*90}")
print("RUNNING BENCHMARK PORTFOLIOS")
print(f"{'='*90}")

def run_equal_weight_portfolio(returns_df, all_assets, tc_rate=0.005,
                               rebal_freq='monthly', initial_capital=1e9):
    """
    Equal-weight rebalancing portfolio.
    Rebalances periodically (monthly by default).
    """
    N = len(all_assets)
    base_weight = 1.0 / N

    pv = float(initial_capital)
    records = []
    last_rebal_date = None

    for date in returns_df.index:
        # Check if rebalance is needed
        should_rebal = False
        if last_rebal_date is None:
            should_rebal = True
        elif rebal_freq == 'monthly':
            if date.month != last_rebal_date.month or date.year != last_rebal_date.year:
                should_rebal = True
        elif rebal_freq == 'quarterly':
            if date.quarter != last_rebal_date.quarter or date.year != last_rebal_date.year:
                should_rebal = True

        if should_rebal:
            pv *= (1 - tc_rate)
            last_rebal_date = date

        # Equal-weight portfolio return
        daily_ret = (returns_df.loc[date, all_assets].values @
                    np.array([base_weight] * N))
        pv *= (1 + daily_ret)
        records.append({'date': date, 'portfolio_value': pv})

    return pd.DataFrame(records).set_index('date')


def run_buy_and_hold_portfolio(returns_df, all_assets, tc_rate=0.005,
                               initial_capital=1e9):
    """
    Buy-and-hold equal-weight portfolio (no rebalancing).
    Initial allocation: 1/N to each asset.
    Drift weights over time based on asset performance.
    """
    N = len(all_assets)
    base_weight = 1.0 / N

    pv = float(initial_capital)
    records = []

    # Initial transaction cost (one-time purchase)
    pv *= (1 - tc_rate)

    for date in returns_df.index:
        # Daily return from equal-weight allocation (no rebalancing)
        daily_ret = (returns_df.loc[date, all_assets].values @
                    np.array([base_weight] * N))
        pv *= (1 + daily_ret)
        records.append({'date': date, 'portfolio_value': pv})

    return pd.DataFrame(records).set_index('date')


def run_hybrid_portfolio(returns_df, signal_series, all_assets, tc_rate=0.005,
                        initial_capital=1e9):
    """
    Portfolio based on hybrid LSTM + Determinant signals.
    Signal = +1 (long EW), -1 (cash)
    """
    N = len(all_assets)
    base_weight = 1.0 / N

    pv = float(initial_capital)
    records = []
    prev_signal = None

    for date in returns_df.index:
        if date in signal_series.index:
            current_signal = signal_series.loc[date]
        else:
            current_signal = prev_signal if prev_signal is not None else -1

        # Rebalance if signal changed
        if current_signal != prev_signal:
            if prev_signal is not None:
                turnover = 1.0
                pv *= (1 - turnover * tc_rate)
            else:
                pv *= (1 - tc_rate)
            prev_signal = current_signal

        # Apply signal
        if current_signal > 0:
            daily_ret = (returns_df.loc[date, all_assets].values @
                        np.array([base_weight] * N))
        else:
            daily_ret = 0.0

        pv *= (1 + daily_ret)
        records.append({'date': date, 'portfolio_value': pv})

    return pd.DataFrame(records).set_index('date')


# Run all three portfolios on same test period
test_start = test_dates_seq_hybrid[0]
returns_test = returns.loc[test_start:]

# 1. Buy & Hold
bh_pv = run_buy_and_hold_portfolio(
    returns_test, ALL_ASSETS,
    tc_rate=0.005,
    initial_capital=1e9
)

# 2. Equal Weight (Monthly Rebalancing)
ew_pv = run_equal_weight_portfolio(
    returns_test, ALL_ASSETS,
    tc_rate=0.005,
    rebal_freq='monthly',
    initial_capital=1e9
)

# 3. Hybrid (LSTM + Determinant)
hybrid_pv = run_hybrid_portfolio(
    returns_test,
    hybrid_signal_series,
    ALL_ASSETS,
    tc_rate=0.005,
    initial_capital=1e9
)

print(f"✓ Buy & Hold portfolio computed")
print(f"✓ Equal Weight (monthly rebal) portfolio computed")
print(f"✓ Hybrid (LSTM + Determinant) portfolio computed")

# ── 14. COMPUTE PORTFOLIO METRICS ─────────────────────────────────────────────
def compute_portfolio_metrics(pv_series, name="Portfolio"):
    """
    Compute comprehensive performance metrics.
    """
    returns_pf = pv_series.pct_change().dropna()

    # Total return
    total_ret = (pv_series.iloc[-1] / pv_series.iloc[0] - 1)

    # Annualized return
    n_days = len(pv_series)
    annual_ret = (pv_series.iloc[-1] / pv_series.iloc[0]) ** (252 / n_days) - 1

    # Annualized volatility
    annual_vol = returns_pf.std() * np.sqrt(252)

    # Sharpe ratio (assuming 0% risk-free rate)
    sharpe = annual_ret / (annual_vol + 1e-6) if annual_vol > 0 else 0

    # Maximum drawdown
    cummax = pv_series.expanding().max()
    dd = (pv_series - cummax) / cummax
    max_dd = dd.min()

    # Calmar ratio
    calmar = annual_ret / (abs(max_dd) + 1e-6) if max_dd != 0 else 0

    # Win rate (% of positive return days)
    win_rate = np.mean(returns_pf > 0)

    return {
        'Final Value': pv_series.iloc[-1],
        'Total Return (%)': total_ret * 100,
        'Annual Return (%)': annual_ret * 100,
        'Annual Volatility (%)': annual_vol * 100,
        'Sharpe Ratio': sharpe,
        'Max Drawdown (%)': max_dd * 100,
        'Calmar Ratio': calmar,
        'Win Rate (%)': win_rate * 100
    }


bh_metrics = compute_portfolio_metrics(bh_pv['portfolio_value'], "Buy & Hold")
ew_metrics = compute_portfolio_metrics(ew_pv['portfolio_value'], "Equal Weight")
hybrid_metrics = compute_portfolio_metrics(hybrid_pv['portfolio_value'], "Hybrid")

# Display metrics
print(f"\n{'='*90}")
print("PORTFOLIO PERFORMANCE METRICS (Test Period: 2020+)")
print(f"{'='*90}")

metrics_comparison = pd.DataFrame({
    'Buy & Hold': bh_metrics,
    'Equal Weight': ew_metrics,
    'Hybrid (LSTM+Det)': hybrid_metrics
}).T

print("\n", metrics_comparison.to_string())

# ── 15. VISUALIZATIONS ────────────────────────────────────────────────────────
# Plot 1: Training history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history_hybrid.history['loss'], label='Training Loss', lw=2)
axes[0].plot(history_hybrid.history['val_loss'], label='Validation Loss', lw=2)
axes[0].set_xlabel('Epoch', fontsize=11)
axes[0].set_ylabel('Loss (MSE)', fontsize=11)
axes[0].set_title('Hybrid LSTM Training History', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

axes[1].plot(history_hybrid.history['mae'], label='Training MAE', lw=2)
axes[1].plot(history_hybrid.history['val_mae'], label='Validation MAE', lw=2)
axes[1].set_xlabel('Epoch', fontsize=11)
axes[1].set_ylabel('MAE', fontsize=11)
axes[1].set_title('Hybrid LSTM MAE', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("01_hybrid_lstm_training_history.png", dpi=150, bbox_inches='tight')
plt.show()

# Plot 2: Predictions vs Actual (Test Set)
fig, ax = plt.subplots(figsize=(16, 6))

test_seq_dates = test_dates_seq_hybrid
ax.plot(test_seq_dates, y_test_actual_hybrid, label='Actual EW Returns',
        color='steelblue', lw=2, alpha=0.7)
ax.plot(test_seq_dates, y_test_pred_hybrid, label='Hybrid LSTM Prediction',
        color='coral', lw=1.5, alpha=0.8, linestyle='--')
ax.fill_between(test_seq_dates, y_test_actual_hybrid, y_test_pred_hybrid,
                alpha=0.2, color='grey')

ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Daily Return', fontsize=12)
ax.set_title('Hybrid LSTM: Predicted vs Actual EW Portfolio Returns (Test Set)',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("02_hybrid_lstm_predictions_vs_actual.png", dpi=150, bbox_inches='tight')
plt.show()

# Plot 3: Components of Hybrid Signal
fig, axes = plt.subplots(3, 1, figsize=(16, 10))

# LSTM component
axes[0].plot(test_seq_dates, lstm_norm, label='Normalized LSTM Predictions',
            color='steelblue', lw=1.5)
axes[0].axhline(0, color='black', lw=0.8, linestyle='--')
axes[0].set_ylabel('LSTM Signal', fontsize=11)
axes[0].set_title('Hybrid Signal Components', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)
axes[0].fill_between(test_seq_dates, lstm_norm, 0,
                     where=(lstm_norm > 0), alpha=0.2, color='blue')
axes[0].fill_between(test_seq_dates, lstm_norm, 0,
                     where=(lstm_norm <= 0), alpha=0.2, color='red')

# Determinant component
axes[1].plot(test_seq_dates, det_norm, label='Normalized Determinant Signal',
            color='green', lw=1.5)
axes[1].axhline(0, color='black', lw=0.8, linestyle='--')
axes[1].set_ylabel('Determinant Signal', fontsize=11)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)
axes[1].fill_between(test_seq_dates, det_norm, 0,
                     where=(det_norm > 0), alpha=0.2, color='green')
axes[1].fill_between(test_seq_dates, det_norm, 0,
                     where=(det_norm <= 0), alpha=0.2, color='orange')

# Combined signal
axes[2].plot(test_seq_dates, hybrid_signal_raw, label='Combined Hybrid Signal',
            color='purple', lw=1.5)
axes[2].axhline(0, color='black', lw=0.8, linestyle='--')
axes[2].fill_between(test_seq_dates, hybrid_signal_raw, 0,
                     where=(hybrid_signal_raw > 0), alpha=0.3, color='green',
                     label='Long Signal')
axes[2].fill_between(test_seq_dates, hybrid_signal_raw, 0,
                     where=(hybrid_signal_raw <= 0), alpha=0.3, color='red',
                     label='Neutral/Cash')
axes[2].set_xlabel('Date', fontsize=12)
axes[2].set_ylabel('Hybrid Signal', fontsize=11)
axes[2].legend(fontsize=10)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("03_hybrid_signal_components.png", dpi=150, bbox_inches='tight')
plt.show()

# Plot 4: Portfolio Performance Comparison (ALL THREE APPROACHES)
fig, ax = plt.subplots(figsize=(16, 7))

# Normalize to start at 1.0 for comparison
bh_norm = bh_pv['portfolio_value'] / bh_pv['portfolio_value'].iloc[0]
ew_norm = ew_pv['portfolio_value'] / ew_pv['portfolio_value'].iloc[0]
hybrid_norm = hybrid_pv['portfolio_value'] / hybrid_pv['portfolio_value'].iloc[0]

ax.plot(bh_norm.index, bh_norm.values, label='Buy & Hold', lw=2.5, color='steelblue')
ax.plot(ew_norm.index, ew_norm.values, label='Equal Weight (Monthly Rebal)', lw=2.5, color='green')
ax.plot(hybrid_norm.index, hybrid_norm.values, label='Hybrid (LSTM + Determinant)', lw=2.5, color='purple')

ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Normalized Portfolio Value', fontsize=12)
ax.set_title('Portfolio Performance Comparison (Test Period: 2020+)', fontsize=13, fontweight='bold')
ax.legend(fontsize=11, loc='best')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("04_portfolio_performance_comparison.png", dpi=150, bbox_inches='tight')
plt.show()

# Plot 5: Cumulative Returns Comparison
fig, ax = plt.subplots(figsize=(16, 7))

bh_cumrets = (1 + bh_pv['portfolio_value'].pct_change()).cumprod() - 1
ew_cumrets = (1 + ew_pv['portfolio_value'].pct_change()).cumprod() - 1
hybrid_cumrets = (1 + hybrid_pv['portfolio_value'].pct_change()).cumprod() - 1

ax.plot(bh_cumrets.index, bh_cumrets.values * 100, label='Buy & Hold', lw=2.5, color='steelblue')
ax.plot(ew_cumrets.index, ew_cumrets.values * 100, label='Equal Weight (Monthly Rebal)', lw=2.5, color='green')
ax.plot(hybrid_cumrets.index, hybrid_cumrets.values * 100, label='Hybrid (LSTM + Determinant)', lw=2.5, color='purple')

ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Cumulative Return (%)', fontsize=12)
ax.set_title('Cumulative Returns Comparison (Test Period: 2020+)', fontsize=13, fontweight='bold')
ax.legend(fontsize=11, loc='best')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("05_cumulative_returns_comparison.png", dpi=150, bbox_inches='tight')
plt.show()

# Plot 6: Drawdown Analysis
fig, ax = plt.subplots(figsize=(16, 7))

def compute_drawdown(pv_series):
    cummax = pv_series.expanding().max()
    return (pv_series - cummax) / cummax * 100

bh_dd = compute_drawdown(bh_pv['portfolio_value'])
ew_dd = compute_drawdown(ew_pv['portfolio_value'])
hybrid_dd = compute_drawdown(hybrid_pv['portfolio_value'])

ax.fill_between(bh_dd.index, 0, bh_dd.values, alpha=0.4, label='Buy & Hold', color='steelblue')
ax.fill_between(ew_dd.index, 0, ew_dd.values, alpha=0.4, label='Equal Weight', color='green')
ax.fill_between(hybrid_dd.index, 0, hybrid_dd.values, alpha=0.4, label='Hybrid', color='purple')

ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Drawdown (%)', fontsize=12)
ax.set_title('Drawdown Analysis - All Approaches (Test Period: 2020+)', fontsize=13, fontweight='bold')
ax.legend(fontsize=11, loc='best')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("06_drawdown_analysis.png", dpi=150, bbox_inches='tight')
plt.show()

# Plot 7: Risk-Return Scatter
fig, ax = plt.subplots(figsize=(10, 8))

approaches = ['Buy & Hold', 'Equal Weight', 'Hybrid (LSTM+Det)']
returns = [bh_metrics['Annual Return (%)'], ew_metrics['Annual Return (%)'],
           hybrid_metrics['Annual Return (%)']]
volatilities = [bh_metrics['Annual Volatility (%)'], ew_metrics['Annual Volatility (%)'],
                hybrid_metrics['Annual Volatility (%)']]
sharpes = [bh_metrics['Sharpe Ratio'], ew_metrics['Sharpe Ratio'],
           hybrid_metrics['Sharpe Ratio']]

colors = ['steelblue', 'green', 'purple']

for i, (ret, vol, sharpe, name, color) in enumerate(zip(returns, volatilities, sharpes, approaches, colors)):
    ax.scatter(vol, ret, s=500, alpha=0.7, color=color, label=f'{name} (SR: {sharpe:.2f})')
    ax.annotate(name, (vol, ret), fontsize=11, ha='center', va='bottom', fontweight='bold')

ax.set_xlabel('Annual Volatility (%)', fontsize=12)
ax.set_ylabel('Annual Return (%)', fontsize=12)
ax.set_title('Risk-Return Profile Comparison', fontsize=13, fontweight='bold')
ax.legend(fontsize=10, loc='best')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("07_risk_return_scatter.png", dpi=150, bbox_inches='tight')
plt.show()

# ── 16. SAVE RESULTS ──────────────────────────────────────────────────────────
print(f"\n{'='*90}")
print("SAVING ALL RESULTS")
print(f"{'='*90}")

# Save portfolio values
bh_pv.to_csv("BuyHold_portfolio_test.csv")
ew_pv.to_csv("EqualWeight_portfolio_test.csv")
hybrid_pv.to_csv("Hybrid_LSTM_Determinant_portfolio_test.csv")

# Save predictions
hybrid_predictions_df = pd.DataFrame({
    'date': test_seq_dates,
    'actual_ew_return': y_test_actual_hybrid,
    'lstm_pred_return': y_test_pred_hybrid,
    'lstm_normalized': lstm_norm,
    'determinant_signal': det_signal_test,
    'determinant_normalized': det_norm,
    'hybrid_signal_raw': hybrid_signal_raw,
    'hybrid_trading_signal': hybrid_signals
})
hybrid_predictions_df = hybrid_predictions_df.set_index('date')
hybrid_predictions_df.to_csv("Hybrid_LSTM_Determinant_predictions.csv")

# Save metrics comparison
metrics_comparison.to_csv("Portfolio_Metrics_Comparison.csv")

# Save model
model_hybrid.save("hybrid_lstm_determinant_model.h5")

print("✓ BuyHold_portfolio_test.csv")
print("✓ EqualWeight_portfolio_test.csv")
print("✓ Hybrid_LSTM_Determinant_portfolio_test.csv")
print("✓ Hybrid_LSTM_Determinant_predictions.csv")
print("✓ Portfolio_Metrics_Comparison.csv")
print("✓ hybrid_lstm_determinant_model.h5")

# ── 17. FINAL SUMMARY ─────────────────────────────────────────────────────────
print(f"\n{'='*90}")
print("FINAL APPROACH SUMMARY")
print(f"{'='*90}")

print("\n1. BUY & HOLD:")
print("   - Buy initial EW allocation (1/N per asset)")
print("   - Hold indefinitely (no rebalancing)")
print(f"   - Annual Return: {bh_metrics['Annual Return (%)']:.2f}%")
print(f"   - Sharpe Ratio: {bh_metrics['Sharpe Ratio']:.3f}")
print(f"   - Max Drawdown: {bh_metrics['Max Drawdown (%)']:.2f}%\n")

print("2. EQUAL WEIGHT (Monthly Rebalancing):")
print("   - Maintain 1/N weights")
print("   - Rebalance monthly (5 bps per transaction)")
print(f"   - Annual Return: {ew_metrics['Annual Return (%)']:.2f}%")
print(f"   - Sharpe Ratio: {ew_metrics['Sharpe Ratio']:.3f}")
print(f"   - Max Drawdown: {ew_metrics['Max Drawdown (%)']:.2f}%\n")

print("3. HYBRID (LSTM + Determinant Signals):")
print("   - LSTM predicts EW returns (60-feature set)")
print("   - Determinant signal captures correlation regime (3d/7d/28d)")
print("   - Combined signal: 60% LSTM + 40% Determinant")
print("   - Long when hybrid_signal > 0, Cash otherwise")
print(f"   - Annual Return: {hybrid_metrics['Annual Return (%)']:.2f}%")
print(f"   - Sharpe Ratio: {hybrid_metrics['Sharpe Ratio']:.3f}")
print(f"   - Max Drawdown: {hybrid_metrics['Max Drawdown (%)']:.2f}%")
print(f"   - Trading signals: {long_count} long, {neutral_count} neutral days\n")

print("Next Step: Apply CPCV (Combinatorial Purged Cross-Validation)")
